# DIMER Notebook: Sound Event Classification — Learning to Recognize What We Hear

**What sound is present, and how much should we trust the model when the recording changes?**

**Profile:** `E2E` · **Mode:** `WORKSHOP` · **Notebook specification:** `2.2` · **Status:** Candidate; fresh hosted GPU and real labelled BYOD qualification pending.

This modular, self-paced **Level 2 — Applied tasks** unit complements Whisper. Whisper transcribes words; this unit classifies environmental sounds. Basic Python and Colab familiarity are sufficient; no prior ML experience is required. Completion records are optional personal notes, **not required submissions**.

**Input → Model → Output:** a short audio recording → a frozen Audio Spectrogram Transformer (AST; `MIT/ast-finetuned-audioset-10-10-0.4593` at immutable revision `f826b80d28226b62986cc218e5cec390b1096902`) with a new trainable classifier head → one of ten named sound categories and its scores. A confident score does not establish correctness or the absence of an unknown sound.

You will inspect waveforms and spectrograms, compare simple baselines, train only the task head, select with validation recordings, evaluate held-out recordings, verify a saved artifact in a fresh process, and change background noise while holding other conditions fixed.

**How to use this notebook — fresh Run all:** choose Runtime → Change runtime type → T4 GPU, then Runtime → Run all. Section 1 is infrastructure; learner-facing work starts in Section 2, and the optional sections at the end are form-controlled and off by default. No token, upload, restart, runtime Git clone, DIMER service or inference API is required. Public data and weights download anonymously; an isolated Python 3.12.12 worker preserves the notebook kernel. Optional own-data routes are disabled by default. The 400 pinned WAV files total **176,417,600 bytes**, plus the model and environment. Allow several GiB; full runtime and peak memory are measured during qualification, not promised here.

**Roadmap:** inspect real sounds → understand output semantics and metrics → train and select → evaluate and reload → add controlled noise → conclude → optional own recordings.

## 1. Prepare the isolated runtime

**Input:** a fresh Linux x86_64 Colab GPU runtime. **System:** embedded reference code and a verified, hash-locked environment. **Output:** an isolated worker with pinned model/data identities.

**Infrastructure:** the next three cells set up the runtime. You may run them without studying their implementation, and they may be collapsed. Public defaults need no Hugging Face token. Integrity failures stop execution; do not bypass them. Stage receipts invalidate downstream results when prerequisites change.

In [ ]:
import hashlib
import json
import os
import platform
from pathlib import Path
import subprocess
import time
import urllib.error
import urllib.request
import uuid
import zipfile

SESSION_START = time.perf_counter()
if platform.system() != 'Linux' or platform.machine() != 'x86_64':
    raise RuntimeError('The canonical environment is Linux x86_64 Google Colab with a T4 GPU.')
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
except FileNotFoundError as exc:
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU before Run all.') from exc
if gpu.returncode:
    raise RuntimeError('Select a fresh T4 GPU runtime; CUDA was not detected.')
print(gpu.stdout.strip())
ROOT = Path.cwd() / 'outputs' / 'sound_events' / uuid.uuid4().hex[:12]
ROOT.mkdir(parents=True)

In [ ]:
CARRIED_FILES = {'ast_reference/__init__.py': 'from .metrics import (\n    confusion_matrix,\n    evaluate_classification,\n    macro_f1_score,\n    majority_class_baseline,\n    multiclass_accuracy,\n    per_class_metrics,\n)\nfrom .pipeline import (\n    ARTIFACT_FORMAT,\n    ARTIFACT_FORMAT_VERSION,\n    DEFAULT_TOP_K,\n    DEFAULT_WEIGHTS_DIR,\n    INPUT_SCHEMA,\n    MAX_AUDIO_SECONDS,\n    MAX_INPUT_SECONDS,\n    MIN_AUDIO_SECONDS,\n    MODEL_ID,\n    MODEL_KEY,\n    MODEL_LICENSE,\n    MODEL_REVISION,\n    NUM_LABELS,\n    SAMPLE_RATE,\n    ASTAudioClassificationPipeline,\n    evaluation_report,\n    freeze_backbone,\n    rehead_model,\n    stage_missing_files,\n    validate_inputs,\n    verify_snapshot,\n)\nfrom .samples import (\n    ADAPT_CLASSES,\n    DATASET_REPRESENTATION,\n    MAX_DATASET_AUDIO_SECONDS,\n    MIN_DATASET_AUDIO_SECONDS,\n    generate_audio_clip,\n    split_dataset,\n    synthetic_audio_dataset,\n    tutorial_tone,\n    validate_dataset,\n)\n\n__all__ = [\n    "ADAPT_CLASSES",\n    "ARTIFACT_FORMAT",\n    "ARTIFACT_FORMAT_VERSION",\n    "DATASET_REPRESENTATION",\n    "DEFAULT_TOP_K",\n    "DEFAULT_WEIGHTS_DIR",\n    "INPUT_SCHEMA",\n    "MAX_AUDIO_SECONDS",\n    "MAX_DATASET_AUDIO_SECONDS",\n    "MAX_INPUT_SECONDS",\n    "MIN_AUDIO_SECONDS",\n    "MIN_DATASET_AUDIO_SECONDS",\n    "MODEL_ID",\n    "MODEL_KEY",\n    "MODEL_LICENSE",\n    "MODEL_REVISION",\n    "NUM_LABELS",\n    "SAMPLE_RATE",\n    "ASTAudioClassificationPipeline",\n    "confusion_matrix",\n    "evaluate_classification",\n    "evaluation_report",\n    "freeze_backbone",\n    "generate_audio_clip",\n    "macro_f1_score",\n    "majority_class_baseline",\n    "multiclass_accuracy",\n    "per_class_metrics",\n    "rehead_model",\n    "split_dataset",\n    "stage_missing_files",\n    "synthetic_audio_dataset",\n    "tutorial_tone",\n    "validate_dataset",\n    "validate_inputs",\n    "verify_snapshot",\n]\n', 'ast_reference/metrics.py': '"""Evaluation metrics for multiclass audio classification."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom collections.abc import Sequence\nfrom typing import Any\n\n\ndef _validate_num_classes(num_classes: int) -> None:\n    if isinstance(num_classes, bool) or not isinstance(num_classes, int) or num_classes < 1:\n        raise ValueError("num_classes must be a positive int")\n\n\ndef _validate_class_indices(values: Sequence[int], num_classes: int, *, name: str) -> None:\n    for value in values:\n        if isinstance(value, bool) or not isinstance(value, int):\n            raise TypeError(f"{name} must contain ints, got {type(value).__name__}")\n        if not 0 <= value < num_classes:\n            raise ValueError(f"{name} class index {value} outside [0, {num_classes - 1}]")\n\n\ndef multiclass_accuracy(predictions: Sequence[int], targets: Sequence[int]) -> float:\n    """Compute overall accuracy as the fraction of correctly classified clips."""\n    if len(predictions) != len(targets):\n        raise ValueError(f"length mismatch: predictions={len(predictions)}, targets={len(targets)}")\n    if len(targets) == 0:\n        raise ValueError("cannot compute accuracy over empty targets")\n    correct = sum(1 for p, t in zip(predictions, targets, strict=True) if p == t)\n    return correct / len(targets)\n\n\ndef confusion_matrix(\n    predictions: Sequence[int], targets: Sequence[int], num_classes: int\n) -> list[list[int]]:\n    """Compute confusion matrix where rows are ground truth classes and columns are predicted classes."""\n    _validate_num_classes(num_classes)\n    if len(predictions) != len(targets):\n        raise ValueError(f"length mismatch: predictions={len(predictions)}, targets={len(targets)}")\n    _validate_class_indices(predictions, num_classes, name="predictions")\n    _validate_class_indices(targets, num_classes, name="targets")\n    matrix = [[0] * num_classes for _ in range(num_classes)]\n    for p, t in zip(predictions, targets, strict=True):\n        matrix[t][p] += 1\n    return matrix\n\n\ndef per_class_metrics(\n    predictions: Sequence[int], targets: Sequence[int], num_classes: int\n) -> dict[int, dict[str, float]]:\n    """Compute per-class precision, recall, and F1-score."""\n    if len(predictions) != len(targets):\n        raise ValueError(f"length mismatch: predictions={len(predictions)}, targets={len(targets)}")\n    matrix = confusion_matrix(predictions, targets, num_classes)\n    results: dict[int, dict[str, float]] = {}\n    for c in range(num_classes):\n        tp = matrix[c][c]\n        fp = sum(matrix[other][c] for other in range(num_classes) if other != c)\n        fn = sum(matrix[c][other] for other in range(num_classes) if other != c)\n        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0\n        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0\n        f1 = (2.0 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0\n        results[c] = {\n            "precision": round(float(precision), 4),\n            "recall": round(float(recall), 4),\n            "f1": round(float(f1), 4),\n            "support": sum(matrix[c]),\n        }\n    return results\n\n\ndef macro_f1_score(\n    predictions: Sequence[int], targets: Sequence[int], num_classes: int\n) -> float:\n    """Compute unweighted macro-averaged F1 score across all classes."""\n    pcm = per_class_metrics(predictions, targets, num_classes)\n    f1_sum = sum(metrics["f1"] for metrics in pcm.values())\n    return round(float(f1_sum / num_classes), 4) if num_classes > 0 else 0.0\n\n\ndef majority_class_baseline(targets: Sequence[int], num_classes: int) -> dict[str, Any]:\n    """Compute the accuracy of a constant trivial classifier predicting the majority class."""\n    _validate_num_classes(num_classes)\n    if len(targets) == 0:\n        raise ValueError("cannot compute baseline over empty targets")\n    _validate_class_indices(targets, num_classes, name="targets")\n    counts = Counter(targets)\n    majority_class, count = counts.most_common(1)[0]\n    majority_acc = count / len(targets)\n    return {\n        "majority_class_index": int(majority_class),\n        "majority_class_count": int(count),\n        "majority_class_accuracy": round(float(majority_acc), 4),\n        "num_classes": num_classes,\n        "n_samples": len(targets),\n    }\n\n\ndef evaluate_classification(\n    predictions: Sequence[int],\n    targets: Sequence[int],\n    class_names: Sequence[str],\n) -> dict[str, Any]:\n    """Comprehensive multi-class classification evaluation against baselines."""\n    num_classes = len(class_names)\n    if num_classes < 1:\n        raise ValueError("class_names must not be empty")\n    if any(not isinstance(name, str) or not name.strip() for name in class_names):\n        raise ValueError("class_names must contain non-empty strings")\n    if len(set(class_names)) != num_classes:\n        raise ValueError("class_names must be unique")\n    acc = multiclass_accuracy(predictions, targets)\n    pcm = per_class_metrics(predictions, targets, num_classes)\n    macro_f1 = macro_f1_score(predictions, targets, num_classes)\n    baseline = majority_class_baseline(targets, num_classes)\n    cm = confusion_matrix(predictions, targets, num_classes)\n\n    named_per_class = {class_names[c]: pcm[c] for c in range(num_classes)}\n\n    return {\n        "accuracy": round(float(acc), 4),\n        "macro_f1": macro_f1,\n        "per_class": named_per_class,\n        "confusion_matrix": cm,\n        "baseline": baseline,\n        "accuracy_delta_vs_baseline": round(float(acc - baseline["majority_class_accuracy"]), 4),\n        "num_samples": len(targets),\n        "num_classes": num_classes,\n    }\n', 'ast_reference/pipeline.py': 'from __future__ import annotations\n\nimport hashlib\nimport json\nfrom collections.abc import Callable, Mapping, Sequence\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nMODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"\nMODEL_REVISION = "f826b80d28226b62986cc218e5cec390b1096902"\nMODEL_LICENSE = "bsd-3-clause"\nMODEL_KEY = "ast-audioset"\nDEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / "weights" / MODEL_KEY\nMANIFEST_NAME = "dimer-base-manifest.json"\n\n# The pinned checkpoint expects 16 kHz mono float32. Its feature extractor computes a 128-bin Kaldi\n# fbank with a 10 ms hop and pads or crops to 1024 frames, so only the first 10.24 s of audio reach\n# the model; anything longer is cropped and reported as `truncated`.\nSAMPLE_RATE = 16_000\nWINDOW_FRAMES = 1024\nMAX_AUDIO_SECONDS = WINDOW_FRAMES * 0.010  # 10.24 s model window\nMAX_INPUT_SECONDS = 120.0  # hard ceiling: longer input is rejected, the caller must chunk\nMIN_AUDIO_SECONDS = 0.025  # one 25 ms fbank frame\nNUM_LABELS = 527  # AudioSet ontology classes in the pinned config.json\nDEFAULT_TOP_K = 5\nACTIVATION = "sigmoid"\nARTIFACT_FORMAT = "org.valcorza.ast-audio.adapter.v1"\nARTIFACT_FORMAT_VERSION = "1.0"\n\n\ndef verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n    """Check the local snapshot against its DIMER manifest; raise naming the first mismatch."""\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n    if manifest.get("modelId") != MODEL_ID:\n        raise ValueError(f"manifest modelId {manifest.get(\'modelId\')!r} != {MODEL_ID!r}")\n    if manifest.get("revision") != MODEL_REVISION:\n        raise ValueError(f"manifest revision {manifest.get(\'revision\')!r} != {MODEL_REVISION!r}")\n    for entry in manifest["files"]:\n        file_path = root / entry["path"]\n        if not file_path.is_file():\n            raise FileNotFoundError(f"snapshot file missing: {file_path}")\n        size = file_path.stat().st_size\n        if size != entry["bytes"]:\n            raise ValueError(f"{entry[\'path\']}: size {size} != manifest {entry[\'bytes\']}")\n        digest = hashlib.sha256()\n        with open(file_path, "rb") as fh:\n            for chunk in iter(lambda: fh.read(1 << 20), b""):\n                digest.update(chunk)\n        if digest.hexdigest() != entry["sha256"]:\n            raise ValueError(f"{entry[\'path\']}: sha256 {digest.hexdigest()} != manifest {entry[\'sha256\']}")\n    return manifest\n\n\ndef _hub_download(relative_path: str, root: Path) -> None:\n    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""\n    from huggingface_hub import hf_hub_download\n\n    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))\n\n\ndef stage_missing_files(\n    path: str | Path | None = None,\n    *,\n    allow_download: bool = False,\n    downloader: Callable[[str, Path], None] | None = None,\n) -> list[str]:\n    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but\n    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n    with open(manifest_path, encoding="utf-8") as fh:\n        manifest = json.load(fh)\n    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:\n        raise ValueError(\n            f"manifest names {manifest.get(\'modelId\')}@{manifest.get(\'revision\')}, "\n            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"\n        )\n    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]\n    if not missing:\n        return []\n    if not allow_download:\n        raise FileNotFoundError(\n            f"snapshot at {root} is missing {missing}; "\n            f"pass allow_download=True to fetch them at {MODEL_REVISION}"\n        )\n    fetch = downloader or _hub_download\n    for relative_path in missing:\n        fetch(relative_path, root)\n    return missing\n\n\ndef _resample(waveform: np.ndarray, sample_rate: int) -> np.ndarray:\n    import torch\n    import torchaudio.functional as af\n\n    resampled = af.resample(torch.from_numpy(waveform), orig_freq=sample_rate, new_freq=SAMPLE_RATE)\n    return resampled.numpy().astype(np.float32)\n\n\nINPUT_SCHEMA: dict[str, Any] = {\n    "input": "1-D float numpy array of mono samples in [-1, 1], plus the sample_rate it was captured at",\n    "sample_rate_hz": f"any positive int; resampled to SAMPLE_RATE={SAMPLE_RATE} when it differs",\n    "duration_seconds": [MIN_AUDIO_SECONDS, MAX_INPUT_SECONDS],\n    "model_window_seconds": MAX_AUDIO_SECONDS,\n    "top_k": [1, NUM_LABELS],\n    "preprocessing": (\n        f"resample to {SAMPLE_RATE} Hz when needed, 128-bin Kaldi fbank with a 10 ms hop, "\n        f"pad or crop to {WINDOW_FRAMES} frames ({MAX_AUDIO_SECONDS} s)"\n    ),\n}\n\n\ndef _check_inputs(audio: Any, sample_rate: Any, top_k: Any) -> float:\n    """Raise TypeError/ValueError naming the first violated ceiling; return the clip duration in seconds."""\n    if not isinstance(audio, np.ndarray):\n        raise TypeError(f"audio must be a numpy.ndarray, got {type(audio).__name__}")\n    if audio.ndim != 1:\n        raise ValueError(f"audio must be 1-D mono, got shape {audio.shape}")\n    if not np.issubdtype(audio.dtype, np.floating):\n        raise TypeError(f"audio must be a float array in [-1, 1], got dtype {audio.dtype}")\n    if not isinstance(sample_rate, int) or isinstance(sample_rate, bool) or sample_rate <= 0:\n        raise TypeError("sample_rate must be a positive int (the rate the audio was captured at)")\n    if not isinstance(top_k, int) or isinstance(top_k, bool) or not 1 <= top_k <= NUM_LABELS:\n        raise ValueError(f"top_k must be an int in [1, {NUM_LABELS}]")\n    duration = audio.shape[0] / sample_rate\n    if duration < MIN_AUDIO_SECONDS:\n        raise ValueError(f"audio is {duration:.4f} s; minimum is {MIN_AUDIO_SECONDS} s")\n    if duration > MAX_INPUT_SECONDS:\n        raise ValueError(f"audio is {duration:.2f} s; ceiling is {MAX_INPUT_SECONDS} s (chunk it first)")\n    if not np.all(np.isfinite(audio)):\n        raise ValueError("audio contains NaN or inf samples")\n    if np.max(np.abs(audio)) > 1.0:\n        raise ValueError("audio samples must be in [-1, 1]")\n    return duration\n\n\ndef validate_inputs(\n    waveforms: Any,\n    sample_rate: Any,\n    *,\n    top_k: int = DEFAULT_TOP_K,\n    names: Sequence[str] | None = None,\n) -> dict[str, Any]:\n    """Validation stage: return the input manifest (schema, per-clip observations, verdict).\n\n    The checks are the ones ``predict`` applies, through the same private ``_check_inputs``, so a\n    rejection here raises exactly what ``predict`` would; a caller that wants the finding recorded\n    catches the exception and stores ``str(exc)`` under ``findings``.\n    """\n    batch = [waveforms] if isinstance(waveforms, np.ndarray) else waveforms\n    if not isinstance(batch, Sequence) or isinstance(batch, str | bytes):\n        raise TypeError("waveforms must be a 1-D numpy.ndarray or a sequence of them")\n    if len(batch) < 1:\n        raise ValueError("at least one waveform is required")\n    if names is not None and len(names) != len(batch):\n        raise ValueError("names must have one entry per waveform")\n    inputs = []\n    for index, waveform in enumerate(batch):\n        duration = _check_inputs(waveform, sample_rate, top_k)\n        inputs.append(\n            {\n                "id": names[index] if names else f"clip-{index}",\n                "samples": int(waveform.shape[0]),\n                "dtype": str(waveform.dtype),\n                "duration_seconds": round(duration, 4),\n                "peak_amplitude": round(float(np.max(np.abs(waveform))), 6),\n                "will_resample": sample_rate != SAMPLE_RATE,\n                "will_truncate": duration > MAX_AUDIO_SECONDS,\n            }\n        )\n    return {\n        "schema": dict(INPUT_SCHEMA),\n        "inputs": inputs,\n        "sample_rate": sample_rate,\n        "top_k": top_k,\n        "n_clips": len(inputs),\n        "verdict": "accepted",\n        "findings": [],\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n    }\n\n\ndef evaluation_report(\n    result: Mapping[str, Any],\n    targets: Sequence[Any] | None = None,\n    *,\n    sample_kind: str = "synthetic",\n) -> dict[str, Any]:\n    """Describe the unadapted AudioSet inference result without inventing ground truth."""\n    activation = result.get("activation", ACTIVATION)\n    if activation != ACTIVATION:\n        raise ValueError(\n            "evaluation_report only supports unadapted AudioSet sigmoid results; "\n            f"got activation={activation!r}"\n        )\n    predictions = result["predictions"]\n    reason = "no AudioSet-labelled ground truth exists for the evaluated clip"\n    if targets is not None:\n        reason = (\n            "targets were supplied, but they are not established as labels from the 527-class "\n            "AudioSet ontology; "\n            "score only ontology-aligned labels with an AudioSet-convention mAP implementation"\n        )\n    return {\n        "task": "multi-label audio event classification over the 527 AudioSet labels",\n        "score_semantics": (\n            f"independent {activation} score per label: the scores do not sum "\n            "to one, several labels can be high at once, none is a calibrated probability, and no "\n            "threshold is shipped"\n        ),\n        "sample_kind": sample_kind,\n        "n_clips": 1,\n        "n_scored_labels": len(predictions),\n        "metrics": [],\n        "baselines": [],\n        "verdict": "not-measurable",\n        "reason": reason,\n        "needs": (\n            f"clips labelled against the same {NUM_LABELS}-label AudioSet ontology, scored over the full "\n            "score vector (predict(..., top_k=527)) with mean average precision plus per-label precision "\n            "and recall at a threshold chosen on your own labelled clips; the \'0.4593\' in the checkpoint "\n            "name is the upstream-reported AudioSet mAP and is not measured here"\n        ),\n        "truncated": bool(result.get("truncated", False)),\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n    }\n\n\ndef _build_reheaded_classifier(\n    model: Any,\n    class_names: Sequence[str],\n    seed: int = 42,\n) -> tuple[Any, list[str]]:\n    """Build a re-headed classifier without mutating the live model."""\n    import copy\n\n    import torch\n    import torch.nn as nn\n\n    names = [str(name).strip() for name in class_names]\n    if len(names) < 2 or any(not name for name in names):\n        raise ValueError("class_names must contain at least 2 non-empty names")\n    if len(set(names)) != len(names):\n        raise ValueError("class_names must be unique")\n    classifier = copy.deepcopy(model.classifier)\n    previous_dense = classifier.dense\n    with torch.random.fork_rng(devices=[]):\n        torch.manual_seed(seed)\n        new_dense = nn.Linear(previous_dense.in_features, len(names), bias=True)\n        nn.init.kaiming_normal_(new_dense.weight, nonlinearity="linear")\n        nn.init.zeros_(new_dense.bias)\n    new_dense = new_dense.to(device=previous_dense.weight.device, dtype=previous_dense.weight.dtype)\n    classifier.dense = new_dense\n    return classifier, names\n\n\ndef _set_class_metadata(model: Any, names: Sequence[str]) -> None:\n    """Update model label metadata after a classifier replacement succeeds."""\n    model.num_labels = len(names)\n    model.config.num_labels = len(names)\n    model.config.id2label = dict(enumerate(names))\n    model.config.label2id = {name: i for i, name in enumerate(names)}\n    model.config.problem_type = "single_label_classification"\n\n\ndef rehead_model(\n    model: Any,\n    class_names: Sequence[str],\n    seed: int = 42,\n) -> None:\n    """Dynamically replace ASTMLPHead dense layer with a new linear classifier sized to class_names."""\n    classifier, names = _build_reheaded_classifier(model, class_names, seed=seed)\n    model.classifier = classifier\n    _set_class_metadata(model, names)\n\n\ndef freeze_backbone(model: Any) -> int:\n    """Freeze all parameters in the audio spectrogram transformer backbone, returning frozen count."""\n    frozen = 0\n    for param in model.audio_spectrogram_transformer.parameters():\n        param.requires_grad = False\n        frozen += param.numel()\n    return frozen\n\n\n@dataclass\nclass ASTAudioClassificationPipeline:\n    """Audio Spectrogram Transformer classifier for AudioSet and adapted audio tasks."""\n\n    _runner: Callable[[np.ndarray], np.ndarray]\n    labels: list[str]\n    device: str\n    model: Any | None = None\n    extractor: Any | None = None\n    activation: str = ACTIVATION\n    adaptation_config: dict[str, Any] = field(default_factory=dict)\n\n    @classmethod\n    def from_pretrained(\n        cls,\n        device: str | None = None,\n        weights_dir: str | Path | None = None,\n        allow_download: bool = False,\n    ) -> ASTAudioClassificationPipeline:\n        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR\n        if (root / MANIFEST_NAME).is_file():\n            stage_missing_files(root, allow_download=allow_download)\n            verify_snapshot(root)\n            source, kwargs = str(root), dict(local_files_only=True)\n        elif allow_download:\n            source, kwargs = MODEL_ID, dict(revision=MODEL_REVISION)\n        else:\n            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")\n        # Refuse invalid snapshots before importing model libraries.\n        import torch\n        from transformers import ASTFeatureExtractor, ASTForAudioClassification\n\n        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")\n        extractor = ASTFeatureExtractor.from_pretrained(source, trust_remote_code=False, **kwargs)\n        model = ASTForAudioClassification.from_pretrained(\n            source, trust_remote_code=False, dtype=torch.float32, **kwargs\n        )\n        model = model.to(resolved_device).eval()\n        labels = [model.config.id2label[i] for i in range(model.config.num_labels)]\n\n        def runner(waveform: np.ndarray) -> np.ndarray:\n            inputs = extractor(waveform, sampling_rate=SAMPLE_RATE, return_tensors="pt")\n            with torch.inference_mode():\n                logits = model(inputs["input_values"].to(resolved_device)).logits\n            return logits[0].float().cpu().numpy()\n\n        return cls(runner, labels, resolved_device, model=model, extractor=extractor)\n\n    def rehead(self, class_names: Sequence[str], seed: int = 42) -> None:\n        """Dynamically rehead the underlying model and update labels and runner."""\n        if self.model is None or self.extractor is None:\n            raise RuntimeError("cannot rehead a pipeline instance without an underlying torch model")\n        rehead_model(self.model, class_names, seed=seed)\n        self.labels = [self.model.config.id2label[i] for i in range(self.model.config.num_labels)]\n        self.activation = "softmax"\n        self.adaptation_config = {\n            "mode": "classifier-head-gradient-adaptation",\n            "class_names": list(self.labels),\n            "rehead_seed": seed,\n        }\n        self.model.to(self.device)\n        self._refresh_runner()\n\n    def _refresh_runner(self) -> None:\n        """Bind inference to the current model, extractor, and device."""\n        if self.model is None or self.extractor is None:\n            raise RuntimeError("cannot bind a runner without an underlying model and extractor")\n        resolved_device = self.device\n        model = self.model\n        extractor = self.extractor\n\n        def runner(waveform: np.ndarray) -> np.ndarray:\n            import torch\n\n            inputs = extractor(waveform, sampling_rate=SAMPLE_RATE, return_tensors="pt")\n            with torch.inference_mode():\n                logits = model(inputs["input_values"].to(resolved_device)).logits\n            return logits[0].float().cpu().numpy()\n\n        self._runner = runner\n\n    def freeze_backbone(self) -> int:\n        """Freeze the underlying backbone parameters."""\n        if self.model is None:\n            raise RuntimeError(\n                "cannot freeze backbone on a pipeline instance without an underlying torch model"\n            )\n        frozen = freeze_backbone(self.model)\n        self.adaptation_config["frozen_backbone_parameters"] = frozen\n        return frozen\n\n    def finetune(\n        self,\n        train_records: Sequence[dict[str, Any]],\n        val_records: Sequence[dict[str, Any]] | None = None,\n        *,\n        epochs: int = 3,\n        batch_size: int = 4,\n        learning_rate: float = 1e-4,\n        seed: int = 42,\n    ) -> list[dict[str, Any]]:\n        """In-process supervised fine-tuning using AdamW optimizer over trainable classifier parameters."""\n        import random\n\n        import torch\n        import torch.nn.functional as F\n        from torch.optim import AdamW\n\n        from .metrics import evaluate_classification\n        from .samples import validate_dataset\n\n        if self.model is None or self.extractor is None:\n            raise RuntimeError("cannot fine-tune a pipeline without underlying model and extractor")\n        if isinstance(epochs, bool) or not isinstance(epochs, int) or epochs < 1:\n            raise ValueError("epochs must be a positive int")\n        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or batch_size < 1:\n            raise ValueError("batch_size must be a positive int")\n        valid_learning_rate = isinstance(learning_rate, int | float) and not isinstance(learning_rate, bool)\n        if not valid_learning_rate or learning_rate <= 0:\n            raise ValueError("learning_rate must be positive")\n        validate_dataset(train_records, self.labels)\n        if val_records is not None:\n            validate_dataset(val_records, self.labels)\n        if any(param.requires_grad for param in self.model.audio_spectrogram_transformer.parameters()):\n            raise RuntimeError("backbone is trainable; call freeze_backbone() before finetune()")\n\n        torch.manual_seed(seed)\n        device = torch.device(self.device)\n        self.model.to(device)\n\n        trainable_params = [p for p in self.model.parameters() if p.requires_grad]\n        if not trainable_params:\n            raise RuntimeError("model has no trainable parameters")\n        optimizer = AdamW(trainable_params, lr=learning_rate)\n\n        def cached_features(\n            records: Sequence[dict[str, Any]],\n        ) -> tuple[torch.Tensor, torch.Tensor]:\n            feature_batches: list[torch.Tensor] = []\n            self.model.audio_spectrogram_transformer.eval()\n            with torch.inference_mode():\n                for start in range(0, len(records), batch_size):\n                    batch = records[start : start + batch_size]\n                    inputs = self.extractor(\n                        [record["waveform"] for record in batch],\n                        sampling_rate=SAMPLE_RATE,\n                        return_tensors="pt",\n                    )\n                    backbone_output = self.model.audio_spectrogram_transformer(\n                        inputs["input_values"].to(device)\n                    )\n                    feature_batches.append(backbone_output.pooler_output.detach())\n            targets = torch.tensor(\n                [record["label"] for record in records],\n                dtype=torch.long,\n                device=device,\n            )\n            return torch.cat(feature_batches), targets\n\n        train_features, train_targets = cached_features(train_records)\n        val_features: torch.Tensor | None = None\n        val_targets: torch.Tensor | None = None\n        if val_records:\n            val_features, val_targets = cached_features(val_records)\n\n        history: list[dict[str, Any]] = []\n        n_samples = len(train_records)\n\n        for epoch in range(1, epochs + 1):\n            self.model.classifier.train()\n            indices = list(range(n_samples))\n            rng = random.Random(seed + epoch * 13)\n            rng.shuffle(indices)\n\n            running_loss = 0.0\n            steps = 0\n\n            for i in range(0, n_samples, batch_size):\n                batch_indices = indices[i : i + batch_size]\n                index_tensor = torch.tensor(batch_indices, dtype=torch.long, device=device)\n\n                optimizer.zero_grad(set_to_none=True)\n                logits = self.model.classifier(train_features.index_select(0, index_tensor))\n                loss = F.cross_entropy(logits, train_targets.index_select(0, index_tensor))\n                loss.backward()\n                optimizer.step()\n\n                running_loss += float(loss.item())\n                steps += 1\n\n            epoch_train_loss = running_loss / steps if steps > 0 else 0.0\n            epoch_data: dict[str, Any] = {\n                "epoch": epoch,\n                "train_loss": round(float(epoch_train_loss), 4),\n                "steps": steps,\n            }\n\n            if val_features is not None and val_targets is not None:\n                self.model.classifier.eval()\n                with torch.inference_mode():\n                    predictions = self.model.classifier(val_features).argmax(dim=-1)\n                eval_metrics = evaluate_classification(\n                    predictions.tolist(), val_targets.tolist(), self.labels\n                )\n                epoch_data["val_accuracy"] = eval_metrics["accuracy"]\n                epoch_data["val_macro_f1"] = eval_metrics["macro_f1"]\n\n            history.append(epoch_data)\n\n        self.model.eval()\n        self.adaptation_config.update(\n            {\n                "epochs": epochs,\n                "batch_size": batch_size,\n                "learning_rate": float(learning_rate),\n                "training_seed": seed,\n                "train_records": len(train_records),\n                "validation_records": len(val_records or []),\n                "feature_cache": "frozen-backbone-pooler-output",\n                "history": history,\n            }\n        )\n        return history\n\n    def evaluate(self, records: Sequence[dict[str, Any]]) -> dict[str, Any]:\n        """Evaluate the pipeline on a sequence of labelled audio records."""\n        from .metrics import evaluate_classification\n        from .samples import validate_dataset\n\n        validate_dataset(records, self.labels)\n        if self.model is not None:\n            self.model.eval()\n        predictions: list[int] = []\n        targets: list[int] = []\n\n        for record in records:\n            waveform = record["waveform"]\n            sample_rate = record.get("sample_rate", SAMPLE_RATE)\n            result = self.predict(waveform, sample_rate=sample_rate, top_k=1)\n            pred_idx = result["predictions"][0]["index"]\n            predictions.append(pred_idx)\n            targets.append(record["label"])\n\n        return evaluate_classification(predictions, targets, self.labels)\n\n    def save_artifact(self, output_path: str | Path) -> Path:\n        """Export adapted weights, vocabulary, base model lineage, and artifact metadata."""\n        import torch\n\n        if self.model is None:\n            raise RuntimeError("cannot save artifact without an underlying torch model")\n\n        out = Path(output_path)\n        out.parent.mkdir(parents=True, exist_ok=True)\n\n        if self.activation != "softmax" or len(self.labels) == NUM_LABELS:\n            raise RuntimeError("artifact export requires an adapted single-label classifier")\n\n        payload = {\n            "format": ARTIFACT_FORMAT,\n            "format_version": ARTIFACT_FORMAT_VERSION,\n            "artifact_kind": "classifier-head-adapter",\n            "base_model_id": MODEL_ID,\n            "base_model_revision": MODEL_REVISION,\n            "class_names": list(self.labels),\n            "num_classes": len(self.labels),\n            "activation": self.activation,\n            "adaptation": dict(self.adaptation_config),\n            "classifier_state_dict": self.model.classifier.state_dict(),\n        }\n        torch.save(payload, out)\n        return out\n\n    def load_artifact(self, artifact_path: str | Path) -> None:\n        """Load an adapted artifact into the existing pipeline safely using weights_only=True."""\n        import torch\n\n        if self.model is None or self.extractor is None:\n            raise RuntimeError("cannot load artifact into a pipeline without an underlying torch model")\n\n        payload = torch.load(artifact_path, map_location=self.device, weights_only=True)\n        if not isinstance(payload, dict):\n            raise ValueError(f"expected artifact dict, got {type(payload).__name__}")\n        if payload.get("format") != ARTIFACT_FORMAT:\n            raise ValueError(f"unrecognized artifact format: {payload.get(\'format\')}")\n        if payload.get("format_version") != ARTIFACT_FORMAT_VERSION:\n            raise ValueError(f"unsupported artifact format_version: {payload.get(\'format_version\')}")\n        if payload.get("artifact_kind") != "classifier-head-adapter":\n            raise ValueError(f"unsupported artifact_kind: {payload.get(\'artifact_kind\')}")\n        if payload.get("base_model_id") != MODEL_ID:\n            raise ValueError(f"base_model_id mismatch: {payload.get(\'base_model_id\')} != {MODEL_ID}")\n        if payload.get("base_model_revision") != MODEL_REVISION:\n            raise ValueError(\n                f"base_model_revision mismatch: {payload.get(\'base_model_revision\')} != {MODEL_REVISION}"\n            )\n        if payload.get("activation") != "softmax":\n            raise ValueError(f"unsupported artifact activation: {payload.get(\'activation\')}")\n\n        class_names = payload.get("class_names")\n        if not isinstance(class_names, list) or payload.get("num_classes") != len(class_names):\n            raise ValueError("artifact class_names and num_classes are inconsistent")\n        if any(not isinstance(name, str) or not name.strip() for name in class_names):\n            raise ValueError("artifact class_names must contain non-empty strings")\n        if len(class_names) < 2 or len(set(class_names)) != len(class_names):\n            raise ValueError("artifact class_names must contain at least 2 unique names")\n        classifier_state = payload.get("classifier_state_dict")\n        if not isinstance(classifier_state, dict):\n            raise ValueError("artifact classifier_state_dict must be a dict")\n        adaptation = payload.get("adaptation", {})\n        if not isinstance(adaptation, dict):\n            raise ValueError("artifact adaptation metadata must be a dict")\n\n        # Build and validate off to the side so a rejected artifact cannot leave\n        # the live pipeline with a partially replaced classifier.\n        classifier, normalized_names = _build_reheaded_classifier(self.model, class_names)\n        classifier.load_state_dict(classifier_state, strict=True)\n\n        self.model.classifier = classifier\n        _set_class_metadata(self.model, normalized_names)\n        self.labels = normalized_names\n        self.activation = "softmax"\n        self.adaptation_config = dict(adaptation)\n        self.model.to(self.device).eval()\n        self._refresh_runner()\n\n    @classmethod\n    def from_artifact(\n        cls,\n        artifact_path: str | Path,\n        weights_dir: str | Path | None = None,\n        device: str | None = None,\n        allow_download: bool = False,\n    ) -> ASTAudioClassificationPipeline:\n        """Instantiate a base pipeline and load an adapted artifact."""\n        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)\n        pipe.load_artifact(artifact_path)\n        return pipe\n\n    def predict(\n        self,\n        audio: np.ndarray,\n        sample_rate: int,\n        top_k: int | None = None,\n    ) -> dict[str, Any]:\n        """Classify one clip. `audio` is a 1-D float array; `sample_rate` is the rate it was captured at."""\n        resolved_top_k = min(DEFAULT_TOP_K, len(self.labels)) if top_k is None else top_k\n        duration = _check_inputs(audio, sample_rate, resolved_top_k)\n        if resolved_top_k > len(self.labels):\n            raise ValueError(\n                f"top_k={resolved_top_k} exceeds the active label count ({len(self.labels)})"\n            )\n        waveform = audio.astype(np.float32, copy=False)\n        resampled = sample_rate != SAMPLE_RATE\n        if resampled:\n            waveform = _resample(waveform, sample_rate)\n        logits = np.asarray(self._runner(waveform), dtype=np.float32)\n        if logits.shape != (len(self.labels),):\n            raise RuntimeError(f"backend returned logits {logits.shape}, expected ({len(self.labels)},)")\n        if self.activation == "sigmoid":\n            scores = 1.0 / (1.0 + np.exp(-logits))\n        elif self.activation == "softmax":\n            shifted = logits - np.max(logits)\n            exp_scores = np.exp(shifted)\n            scores = exp_scores / np.sum(exp_scores)\n        else:\n            raise RuntimeError(f"unsupported activation: {self.activation}")\n        order = np.argsort(-scores)[:resolved_top_k]\n        return {\n            "predictions": [\n                {"label": self.labels[int(i)], "index": int(i), "score": float(scores[i])} for i in order\n            ],\n            "activation": self.activation,\n            "truncated": duration > MAX_AUDIO_SECONDS,\n            "duration_seconds": duration,\n            "window_seconds": MAX_AUDIO_SECONDS,\n            "resampled": resampled,\n            "input_sample_rate": sample_rate,\n            "model_id": MODEL_ID,\n            "model_revision": MODEL_REVISION,\n        }\n', 'ast_reference/samples.py': '"""Deterministic in-code sample audio data: pretrained demonstration tone and acoustic ecology dataset.\n\nNothing here is downloaded and nothing requires external dependencies beyond numpy,\nensuring zero network dataset dependencies and deterministic execution across clean-room environments.\n\nTwo separate label contexts live here and must not be conflated:\n1. ``tutorial_tone``: 440 Hz sinusoidal waveform demonstrating pretrained 527-class AudioSet inference.\n2. ``synthetic_audio_dataset``: 24 audio clips labelled with ``ADAPT_CLASSES``, the canonical\n   Krause/Pijanowski tripartite acoustic ecology vocabulary:\n   - ``geophony``: non-biological natural sounds (rain, wind, flowing water).\n   - ``biophony``: non-human biological vocalizations (birdsong, insect chirps).\n   - ``anthrophony``: human-generated mechanical sounds (engines, drones, machinery).\n"""\n\nfrom __future__ import annotations\n\nimport random\nfrom collections.abc import Sequence\nfrom typing import Any\n\nimport numpy as np\n\nSAMPLE_RATE = 16_000\nSAMPLE_DURATION = 3.0  # seconds; comfortably inside AST\'s 10.24 s window\nMIN_DATASET_AUDIO_SECONDS = 0.025\nMAX_DATASET_AUDIO_SECONDS = 10.24\nTONE_HZ = 440.0\nTONE_AMPLITUDE = 0.5\n\n# Canonical Krause/Pijanowski acoustic ecology classification vocabulary\nADAPT_CLASSES: tuple[str, ...] = ("geophony", "biophony", "anthrophony")\nDATASET_REPRESENTATION = "io.github.kurtvalcorza.dataset.audio.waveform-classification.v1"\n\n\ndef tutorial_tone(\n    duration: float = SAMPLE_DURATION,\n    sample_rate: int = SAMPLE_RATE,\n    frequency: float = TONE_HZ,\n    amplitude: float = TONE_AMPLITUDE,\n) -> np.ndarray:\n    """Deterministic single-tone audio waveform demonstrating pretrained AudioSet inference."""\n    if not isinstance(sample_rate, int) or isinstance(sample_rate, bool) or sample_rate <= 0:\n        raise ValueError("sample_rate must be a positive int")\n    if not isinstance(duration, int | float) or isinstance(duration, bool) or duration <= 0:\n        raise ValueError("duration must be positive")\n    if not isinstance(frequency, int | float) or isinstance(frequency, bool) or frequency <= 0:\n        raise ValueError("frequency must be positive")\n    if not isinstance(amplitude, int | float) or isinstance(amplitude, bool) or not 0 <= amplitude <= 1:\n        raise ValueError("amplitude must be in [0, 1]")\n    t = np.linspace(0.0, duration, int(sample_rate * duration), endpoint=False, dtype=np.float32)\n    return (amplitude * np.sin(2.0 * np.pi * frequency * t)).astype(np.float32)\n\n\ndef generate_audio_clip(\n    index: int,\n    class_idx: int,\n    duration: float = SAMPLE_DURATION,\n    sample_rate: int = SAMPLE_RATE,\n    seed: int = 42,\n) -> np.ndarray:\n    """Generate a deterministic synthetic 1-D audio waveform characteristic of its acoustic class."""\n    if isinstance(index, bool) or not isinstance(index, int) or index < 0:\n        raise ValueError("index must be a non-negative int")\n    valid_class_idx = (\n        isinstance(class_idx, int)\n        and not isinstance(class_idx, bool)\n        and 0 <= class_idx < len(ADAPT_CLASSES)\n    )\n    if not valid_class_idx:\n        raise ValueError(f"class_idx must be an int in [0, {len(ADAPT_CLASSES) - 1}]")\n    if sample_rate != SAMPLE_RATE:\n        raise ValueError(f"sample_rate must be {SAMPLE_RATE}")\n    if not isinstance(duration, int | float) or isinstance(duration, bool):\n        raise ValueError("duration must be numeric")\n    if not MIN_DATASET_AUDIO_SECONDS <= duration <= MAX_DATASET_AUDIO_SECONDS:\n        raise ValueError(\n            f"duration must be in [{MIN_DATASET_AUDIO_SECONDS}, {MAX_DATASET_AUDIO_SECONDS}]"\n        )\n    if isinstance(seed, bool) or not isinstance(seed, int):\n        raise ValueError("seed must be an int")\n    n_samples = int(sample_rate * duration)\n    t = np.linspace(0.0, duration, n_samples, endpoint=False, dtype=np.float32)\n    rng = np.random.default_rng(seed=seed + index * 101 + class_idx * 17)\n\n    if class_idx == 0:\n        # Geophony: broadband turbulent noise with slow gentle envelope (simulating rain/wind)\n        raw_noise = rng.standard_normal(n_samples).astype(np.float32)\n        # 1-pole low-pass smooth filter\n        alpha = 0.15\n        filtered = np.zeros_like(raw_noise)\n        acc = 0.0\n        for i in range(n_samples):\n            acc = alpha * raw_noise[i] + (1.0 - alpha) * acc\n            filtered[i] = acc\n        # Gentle swell envelope\n        envelope = 0.7 + 0.3 * np.sin(2.0 * np.pi * 0.5 * t)\n        signal = filtered * envelope\n    elif class_idx == 1:\n        # Biophony: rapid frequency-modulated chirp whistles (simulating bird song / biophonic calls)\n        chirp_rate = 3.0 + (index % 3) * 1.5  # chirps per second\n        f_center = 2800.0 + (index % 4) * 400.0\n        f_dev = 1200.0\n        # Phase modulation\n        instantaneous_freq = f_center + f_dev * np.sin(2.0 * np.pi * chirp_rate * t)\n        phase = 2.0 * np.pi * np.cumsum(instantaneous_freq) / sample_rate\n        fundamental = np.sin(phase)\n        harmonic = 0.35 * np.sin(2.0 * phase)\n        # Pulsed amplitude gate\n        gate = np.clip(np.sin(2.0 * np.pi * chirp_rate * t), 0.0, 1.0) ** 2\n        signal = (fundamental + harmonic) * gate\n    elif class_idx == 2:\n        # Anthrophony: low-frequency harmonic drone with motor buzz (simulating engine/machinery)\n        f0 = 120.0 + (index % 3) * 20.0  # 120 Hz fundamental\n        harmonics = [\n            (1.0, 0.45),\n            (2.0, 0.30),\n            (3.0, 0.20),\n            (4.0, 0.15),\n            (5.0, 0.10),\n        ]\n        drone = np.zeros(n_samples, dtype=np.float32)\n        for mult, weight in harmonics:\n            drone += weight * np.sin(2.0 * np.pi * (f0 * mult) * t)\n        # Add motor stroke amplitude pulsation (15 Hz)\n        engine_pulse = 0.75 + 0.25 * np.sin(2.0 * np.pi * 15.0 * t)\n        signal = drone * engine_pulse\n    # Normalize peak amplitude safely to [-0.85, 0.85]\n    peak = np.max(np.abs(signal))\n    if peak > 1e-6:\n        signal = (signal / peak) * 0.85\n    return signal.astype(np.float32)\n\n\ndef synthetic_audio_dataset(\n    n_samples: int = 24,\n    seed: int = 42,\n    duration: float = SAMPLE_DURATION,\n    sample_rate: int = SAMPLE_RATE,\n) -> list[dict[str, Any]]:\n    """Generate a balanced deterministic dataset across ADAPT_CLASSES."""\n    if isinstance(n_samples, bool) or not isinstance(n_samples, int) or n_samples < len(ADAPT_CLASSES) * 2:\n        raise ValueError(f"n_samples must be an int >= {len(ADAPT_CLASSES) * 2}")\n    if n_samples % len(ADAPT_CLASSES):\n        raise ValueError(f"n_samples must be divisible by {len(ADAPT_CLASSES)} for class balance")\n    if isinstance(seed, bool) or not isinstance(seed, int):\n        raise ValueError("seed must be an int")\n    num_classes = len(ADAPT_CLASSES)\n    records: list[dict[str, Any]] = []\n    for idx in range(n_samples):\n        class_idx = idx % num_classes\n        waveform = generate_audio_clip(\n            index=idx,\n            class_idx=class_idx,\n            duration=duration,\n            sample_rate=sample_rate,\n            seed=seed,\n        )\n        records.append(\n            {\n                "id": f"clip-{idx:03d}",\n                "waveform": waveform,\n                "sample_rate": sample_rate,\n                "label": class_idx,\n                "class_name": ADAPT_CLASSES[class_idx],\n                "duration": round(float(duration), 4),\n            }\n        )\n    return records\n\n\ndef split_dataset(\n    records: Sequence[dict[str, Any]],\n    val_fraction: float = 0.25,\n    seed: int = 42,\n) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:\n    """Stratified train/validation split preserving balanced class distribution."""\n    if not 0.0 < val_fraction < 1.0:\n        raise ValueError("val_fraction must be strictly between 0 and 1")\n    by_class: dict[int, list[dict[str, Any]]] = {}\n    for r in records:\n        by_class.setdefault(r["label"], []).append(r)\n\n    rng = random.Random(seed)\n    train_records: list[dict[str, Any]] = []\n    val_records: list[dict[str, Any]] = []\n\n    for label in sorted(by_class.keys()):\n        class_list = list(by_class[label])\n        if len(class_list) < 2:\n            raise ValueError(f"class {label} requires at least 2 records for a disjoint split")\n        rng.shuffle(class_list)\n        n_val = min(len(class_list) - 1, max(1, round(len(class_list) * val_fraction)))\n        val_records.extend(class_list[:n_val])\n        train_records.extend(class_list[n_val:])\n\n    rng.shuffle(train_records)\n    rng.shuffle(val_records)\n    return train_records, val_records\n\n\ndef validate_dataset(\n    records: Sequence[dict[str, Any]],\n    class_names: Sequence[str] = ADAPT_CLASSES,\n) -> dict[str, Any]:\n    """Validate that audio records strictly conform to the classification contract."""\n    if not isinstance(records, Sequence) or isinstance(records, str | bytes):\n        raise TypeError(f"records must be a sequence, got {type(records).__name__}")\n    if len(records) < 2:\n        raise ValueError(f"dataset requires at least 2 records, got {len(records)}")\n\n    num_classes = len(class_names)\n    if num_classes < 2:\n        raise ValueError("class_names must contain at least 2 classes")\n    if any(not isinstance(name, str) or not name.strip() for name in class_names):\n        raise ValueError("class_names must contain non-empty strings")\n    if len(set(class_names)) != num_classes:\n        raise ValueError("class_names must be unique")\n    class_counts = [0] * num_classes\n    seen_ids: set[str] = set()\n\n    for i, r in enumerate(records):\n        if not isinstance(r, dict):\n            raise TypeError(f"record {i} must be a dict, got {type(r).__name__}")\n        if "waveform" not in r:\n            raise KeyError(f"record {i} missing required key \'waveform\'")\n        if "label" not in r:\n            raise KeyError(f"record {i} missing required key \'label\'")\n        if "id" not in r:\n            raise KeyError(f"record {i} missing required key \'id\'")\n        if "sample_rate" not in r:\n            raise KeyError(f"record {i} missing required key \'sample_rate\'")\n        if "class_name" not in r:\n            raise KeyError(f"record {i} missing required key \'class_name\'")\n\n        record_id = r["id"]\n        if not isinstance(record_id, str) or not record_id.strip():\n            raise ValueError(f"record {i} id must be a non-empty string")\n        if record_id in seen_ids:\n            raise ValueError(f"record {i} duplicates id {record_id!r}")\n        seen_ids.add(record_id)\n\n        sample_rate = r["sample_rate"]\n        if sample_rate != SAMPLE_RATE:\n            raise ValueError(f"record {i} sample_rate must be {SAMPLE_RATE}, got {sample_rate!r}")\n\n        waveform = r["waveform"]\n        if not isinstance(waveform, np.ndarray):\n            raise TypeError(f"record {i} waveform must be a np.ndarray, got {type(waveform).__name__}")\n        if waveform.ndim != 1:\n            raise ValueError(f"record {i} waveform must be 1-D mono, got shape {waveform.shape}")\n        if waveform.dtype != np.float32:\n            raise TypeError(f"record {i} waveform must be float32, got {waveform.dtype}")\n        if not np.all(np.isfinite(waveform)):\n            raise ValueError(f"record {i} waveform contains NaN or inf values")\n        if waveform.size == 0:\n            raise ValueError(f"record {i} waveform must not be empty")\n        if np.max(np.abs(waveform)) > 1.0:\n            raise ValueError(f"record {i} waveform samples must be in [-1, 1]")\n        duration = waveform.size / sample_rate\n        if not MIN_DATASET_AUDIO_SECONDS <= duration <= MAX_DATASET_AUDIO_SECONDS:\n            raise ValueError(\n                f"record {i} duration {duration:.4f} s outside "\n                f"[{MIN_DATASET_AUDIO_SECONDS}, {MAX_DATASET_AUDIO_SECONDS}]"\n            )\n\n        label = r["label"]\n        if isinstance(label, bool) or not isinstance(label, int):\n            raise TypeError(f"record {i} label must be an int, got {type(label).__name__}")\n        if not 0 <= label < num_classes:\n            raise ValueError(f"record {i} label {label} outside valid range [0, {num_classes - 1}]")\n        if r["class_name"] != class_names[label]:\n            raise ValueError(\n                f"record {i} class_name {r[\'class_name\']!r} does not match "\n                f"label {label} ({class_names[label]!r})"\n            )\n\n        class_counts[label] += 1\n\n    missing_classes = [class_names[i] for i, count in enumerate(class_counts) if count == 0]\n    if missing_classes:\n        raise ValueError(f"dataset has no records for classes: {missing_classes}")\n\n    return {\n        "representation": DATASET_REPRESENTATION,\n        "n_records": len(records),\n        "num_classes": num_classes,\n        "class_counts": {class_names[c]: class_counts[c] for c in range(num_classes)},\n        "verdict": "accepted",\n    }\n', 'workshop.py': '"""Standalone sound-event workshop worker; heavy libraries load only in model stages."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport re\nimport struct\nimport subprocess\nimport sys\nimport time\nimport urllib.request\nimport uuid\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\n\nSEED = 42\nROLES = ("train", "validation", "test")\nSTAGES = ("prepare", "train", "reload", "activity", "report")\nRECIPE = {\n    "seed": 42,\n    "epochs": 10,\n    "batch_size": 4,\n    "learning_rate": 0.001,\n    "weight_decay": 0.01,\n    "precision": "float32",\n    "shuffle": "random.Random(42 + epoch * 13)",\n}\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for block in iter(lambda: f.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef array_hash(a):\n    return hashlib.sha256(np.asarray(a, dtype="<f4").tobytes()).hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write_json(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n\n\ndef contained(root, relative):\n    if not isinstance(relative, str) or not relative or "\\\\" in relative or ":" in relative:\n        raise ValueError("audio must be a contained relative path")\n    p = Path(relative)\n    if p.is_absolute() or ".." in p.parts:\n        raise ValueError("audio path traversal")\n    path = (root / p).resolve()\n    if not path.is_relative_to(root.resolve()):\n        raise ValueError("audio path escapes dataset")\n    return path\n\n\ndef audio_header(path):\n    """Bound RIFF chunks before scipy allocates decoded samples."""\n    path = Path(path)\n    if not 44 <= path.stat().st_size <= 8 * 1024**2:\n        raise ValueError("WAV must be 44 bytes to 8 MiB")\n    with path.open("rb") as f:\n        header = f.read(12)\n        if header[:4] != b"RIFF" or header[8:] != b"WAVE":\n            raise ValueError("expected RIFF WAV")\n        if struct.unpack("<I", header[4:8])[0] + 8 != path.stat().st_size:\n            raise ValueError("truncated or trailing WAV bytes")\n        fmt = None\n        size = None\n        while f.tell() < path.stat().st_size:\n            chunk = f.read(8)\n            if len(chunk) != 8:\n                raise ValueError("truncated chunk")\n            kind, length = struct.unpack("<4sI", chunk)\n            if f.tell() + length > path.stat().st_size:\n                raise ValueError("truncated WAV data")\n            if kind == b"fmt ":\n                if length < 16 or length > 128 or fmt is not None:\n                    raise ValueError("invalid WAV format chunk")\n                fmt = struct.unpack("<HHIIHH", f.read(16))\n                f.seek(length - 16, 1)\n            elif kind == b"data":\n                if size is not None:\n                    raise ValueError("multiple data chunks")\n                size = length\n                f.seek(length, 1)\n            else:\n                f.seek(length, 1)\n            if length % 2:\n                f.seek(1, 1)\n    if fmt is None or size is None:\n        raise ValueError("missing WAV format/data")\n    encoding, channels, rate, byte_rate, align, bits = fmt\n    if (encoding, bits) not in ((1, 16), (3, 32)) or channels not in (1, 2):\n        raise ValueError("only mono/stereo PCM16 or IEEE float32 WAV accepted")\n    if not 8000 <= rate <= 48000 or align != channels * bits // 8 or byte_rate != rate * align:\n        raise ValueError("invalid rate/block alignment")\n    if size % align or not 0.5 <= size / align / rate <= 10:\n        raise ValueError("WAV duration must be 0.5-10 seconds")\n    return {\n        "sample_rate": rate,\n        "channels": channels,\n        "frames": size // align,\n        "seconds": size / align / rate,\n        "encoding": encoding,\n        "bits": bits,\n    }\n\n\ndef read_audio(path):\n    info = audio_header(path)\n    from scipy.io import wavfile\n\n    rate, raw = wavfile.read(path)\n    if raw.shape[0] != info["frames"] or rate != info["sample_rate"]:\n        raise ValueError("decoded WAV disagrees with header")\n    x = raw.astype(np.float32) / (32768.0 if raw.dtype == np.int16 else 1.0)\n    if not np.isfinite(x).all() or np.max(np.abs(x)) > 1:\n        raise ValueError("nonfinite or out-of-range WAV samples")\n    info["decoded_sha256"] = array_hash(x)\n    if x.ndim == 2:\n        x = x.mean(axis=1, dtype=np.float32)\n    info["mono_sha256"] = array_hash(x)\n    if np.sqrt(np.mean(x.astype(np.float64) ** 2)) <= 1e-8:\n        raise ValueError("silent audio is not a labelled sound example")\n    info["original_mono_peak"] = float(np.max(np.abs(x)))\n    info["fixed_input_gain"] = 0.5\n    x = x * np.float32(0.5)\n    if rate != 16000:\n        import torch\n        import torchaudio.functional as af\n\n        x = af.resample(torch.from_numpy(x), rate, 16000).numpy().astype(np.float32)\n    if not np.isfinite(x).all() or np.max(np.abs(x)) > 1:\n        raise ValueError("resampling exceeded normalized amplitude; no silent clipping")\n    info.update(\n        {\n            "waveform_sha256": array_hash(x),\n            "resampled_frames": len(x),\n            "resampler": "torchaudio.functional.resample default sinc_interp_hann",\n            "resampled_peak": float(np.max(np.abs(x))),\n            "channel_policy": "arithmetic mean" if info["channels"] == 2 else "mono unchanged",\n        }\n    )\n    return x, info\n\n\ndef validate_records(records, class_names, byod=False):\n    if not 2 <= len(class_names) <= 10 or len(set(class_names)) != len(class_names):\n        raise ValueError("vocabulary needs 2-10 unique classes")\n    if any(not isinstance(n, str) or not n.strip() or n != n.strip() for n in class_names):\n        raise ValueError("invalid class names")\n    if not records or len(records) > 400:\n        raise ValueError("dataset needs 1-400 records")\n    ids, hashes, mono, groups = set(), set(), set(), {}\n    total = 0.0\n    for r in records:\n        if not re.fullmatch(r"[A-Za-z0-9_.-]{1,160}", r["id"]) or r["id"] in ids:\n            raise ValueError("invalid or duplicate record ID")\n        ids.add(r["id"])\n        if r["role"] not in ROLES or type(r["label"]) is not int or not 0 <= r["label"] < len(class_names):\n            raise ValueError("invalid role or vocabulary label")\n        g = r["group_id"]\n        if not isinstance(g, str) or not g:\n            raise ValueError("group_id required")\n        if g in groups and groups[g] != r["role"]:\n            raise ValueError("recording group crosses roles")\n        groups[g] = r["role"]\n        for key, seen in (("waveform_sha256", hashes), ("mono_sha256", mono)):\n            if r[key] in seen:\n                raise ValueError("duplicate decoded/resampled waveform")\n            seen.add(r[key])\n        total += r["seconds"]\n    if total > 4000:\n        raise ValueError("dataset exceeds 4000 seconds")\n    for role in ROLES:\n        for label in range(len(class_names)):\n            subset = [r for r in records if r["role"] == role and r["label"] == label]\n            minimum = (4 if role == "train" else 2) if byod else (24 if role == "train" else 8)\n            if len(subset) < minimum or (not byod and len(subset) != minimum):\n                raise ValueError("class support violates frozen split")\n            if byod and len({r["group_id"] for r in subset}) < 2:\n                raise ValueError("BYOD needs two recording groups per class per role")\n    return {\n        "roles": {role: sum(r["role"] == role for r in records) for role in ROLES},\n        "groups": {role: len({r["group_id"] for r in records if r["role"] == role}) for role in ROLES},\n        "seconds": total,\n        "class_names": class_names,\n        "private_byod": byod,\n    }\n\n\ndef metrics(targets, predictions, class_names):\n    n = len(class_names)\n    t, p = np.asarray(targets), np.asarray(predictions)\n    if len(t) == 0 or t.shape != p.shape or t.ndim != 1:\n        raise ValueError("metric inputs must be nonempty aligned vectors")\n    if (\n        t.dtype.kind not in "iu"\n        or p.dtype.kind not in "iu"\n        or min(t.min(), p.min()) < 0\n        or max(t.max(), p.max()) >= n\n    ):\n        raise ValueError("metric indices outside vocabulary")\n    cm = np.zeros((n, n), dtype=int)\n    np.add.at(cm, (t, p), 1)\n    if np.any(cm.sum(1) == 0):\n        raise ValueError("every declared class requires evaluation support")\n    rows = {}\n    for i, name in enumerate(class_names):\n        tp, support, predicted = int(cm[i, i]), int(cm[i].sum()), int(cm[:, i].sum())\n        rows[name] = {\n            "precision": tp / predicted if predicted else None,\n            "precision_reason": None if predicted else "no predicted positives",\n            "recall": tp / support,\n            "f1": 2 * tp / (support + predicted),\n            "support": support,\n        }\n    return {\n        "accuracy": float(np.trace(cm) / len(t)),\n        "macro_f1": sum(r["f1"] for r in rows.values()) / n,\n        "per_class": rows,\n        "confusion_matrix": cm.tolist(),\n        "row_normalized_confusion": (cm / cm.sum(1, keepdims=True)).tolist(),\n        "clips": len(t),\n    }\n\n\ndef noise_conditions(waveform, clip_id):\n    x = np.asarray(waveform, dtype=np.float32)\n    rms = float(np.sqrt(np.mean(x.astype(np.float64) ** 2)))\n    if not np.isfinite(x).all() or rms <= 1e-8:\n        raise ValueError("SNR undefined for silent/nonfinite signal")\n    seed = SEED + int(hashlib.sha256(clip_id.encode()).hexdigest()[:16], 16)\n    z = np.random.default_rng(seed).normal(size=len(x))\n    z -= z.mean()\n    z /= np.sqrt(np.mean(z**2))\n    signals = {"clean": x.astype(np.float64)}\n    for snr in (20, 10, 0):\n        signals[str(snr)] = x + z * rms * 10 ** (-snr / 20)\n    gain = min(1.0, 0.99 / max(float(np.max(np.abs(y))) for y in signals.values()))\n    result = []\n    for condition, y in signals.items():\n        audio = (gain * y).astype(np.float32)\n        noise = audio.astype(np.float64) - gain * x.astype(np.float64)\n        measured = None if condition == "clean" else 20 * math.log10(gain * rms / np.sqrt(np.mean(noise**2)))\n        result.append(\n            {\n                "condition": condition,\n                "audio": audio,\n                "gain": gain,\n                "actual_snr_db": measured,\n                "peak": float(np.max(np.abs(audio))),\n                "clipped_samples": int(np.sum(np.abs(audio) >= 1)),\n                "noise_sha256": array_hash(z),\n                "waveform_sha256": array_hash(audio),\n                "seed": seed,\n            }\n        )\n    return result\n\n\ndef activity_ids(records, names):\n    chosen = []\n    for label in range(len(names)):\n        pool = sorted(\n            (r for r in records if r["role"] == "validation" and r["label"] == label), key=lambda r: r["id"]\n        )\n        first = pool[0]\n        second = next((r for r in pool[1:] if r["group_id"] != first["group_id"]), pool[1])\n        chosen.extend([first["id"], second["id"]])\n    return chosen\n\n\ndef plot_audio(x, path, title):\n    import matplotlib\n\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    import torch\n    import torchaudio.compliance.kaldi as kaldi\n    from scipy.io import wavfile\n\n    path.parent.mkdir(parents=True, exist_ok=True)\n    wavfile.write(path.with_suffix(".wav"), 16000, x.astype(np.float32))\n    mel = kaldi.fbank(\n        torch.from_numpy(x).unsqueeze(0),\n        sample_frequency=16000,\n        num_mel_bins=128,\n        dither=0.0,\n        window_type="hanning",\n        use_energy=False,\n        htk_compat=True,\n    ).numpy()\n    fig, ax = plt.subplots(2, 1, figsize=(9, 5))\n    ax[0].plot(np.arange(len(x)) / 16000, x, linewidth=0.5)\n    ax[0].set(xlabel="Seconds", ylabel="Amplitude", title=title)\n    ax[1].imshow(mel.T, aspect="auto", origin="lower", extent=(0, len(mel) * 0.01, 0, 128))\n    ax[1].set(\n        xlabel="Seconds",\n        ylabel="Mel-frequency bin (low to high)",\n        title="Log-mel filterbank before AST normalization/padding",\n    )\n    fig.tight_layout()\n    fig.savefig(path)\n    plt.close(fig)\n\n\ndef prepare(root, byod=None):\n    if byod and (byod / "manifest.json").stat().st_size > 1024**2:\n        raise ValueError("BYOD manifest exceeds 1 MiB")\n    manifest = read_json((byod / "manifest.json") if byod else root / "sample.json")\n    names = manifest["class_names"]\n    if (\n        not isinstance(names, list)\n        or not 2 <= len(names) <= 10\n        or any(not isinstance(n, str) or not n.strip() or n != n.strip() for n in names)\n        or len(set(names)) != len(names)\n    ):\n        raise ValueError("invalid class vocabulary")\n    source_records = manifest["records"]\n    if not 1 <= len(source_records) <= 400:\n        raise ValueError("maximum 400 clips")\n    data = root / "data"\n    data.mkdir(parents=True, exist_ok=True)\n    cache = root / "cache"\n    cache.mkdir(exist_ok=True)\n    records = []\n    for entry in source_records:\n        r = dict(entry)\n        if not re.fullmatch(r"[A-Za-z0-9_.-]{1,160}", r["id"]):\n            raise ValueError("unsafe record ID")\n        if byod:\n            path = contained(byod, r["audio"])\n        else:\n            path = cache / (r["id"] + ".wav")\n            if not path.exists():\n                with urllib.request.urlopen(r["url"], timeout=60) as response:\n                    payload = response.read(r["bytes"] + 1)\n                if len(payload) != r["bytes"] or hashlib.sha256(payload).hexdigest() != r["sha256"]:\n                    raise ValueError("download integrity mismatch")\n                path.write_bytes(payload)\n            if path.stat().st_size != r["bytes"] or digest(path) != r["sha256"]:\n                raise ValueError("cached data integrity mismatch")\n        x, info = read_audio(path)\n        if not byod and (info["channels"] != 1 or info["sample_rate"] != 44100 or info["frames"] != 220500):\n            raise ValueError("default ESC10 format changed")\n        label = r.get("label", r.get("category"))\n        label = names.index(label) if isinstance(label, str) else label\n        if type(label) is not int or not 0 <= label < len(names):\n            raise ValueError("label outside vocabulary")\n        group = r.get("group_id", r.get("src_file", r.get("source_id", "")))\n        if not isinstance(group, str) or not group:\n            raise ValueError("group_id must be a nonempty string")\n        if "decoded_sha256" in r:\n            r["source_pcm_decoded_sha256"] = r["decoded_sha256"]\n        r.update(info)\n        r.update(\n            {\n                "label": label,\n                "category": names[label],\n                "group_id": group,\n                "audio": f"data/{r[\'id\']}.npy",\n                "original_sample_rate": info["sample_rate"],\n                "original_frames": info["frames"],\n                "sample_rate": 16000,\n                "frames": len(x),\n                "channels": 1,\n                "original_channels": info["channels"],\n            }\n        )\n        np.save(root / r["audio"], x, allow_pickle=False)\n        records.append(r)\n    summary = validate_records(records, names, byod is not None)\n    write_json(root / "outputs/data_manifest.json", manifest)\n    write_json(root / "outputs/records.json", records)\n    write_json(root / "outputs/prepare.json", summary)\n    write_json(\n        root / "outputs/configuration.json",\n        {"recipe": RECIPE, "class_names": names, "activity_ids": activity_ids(records, names)},\n    )\n    train_labels = [r["label"] for r in records if r["role"] == "train"]\n    majority = int(np.bincount(train_labels, minlength=len(names)).argmax())\n    baseline = {\n        role: metrics(\n            [r["label"] for r in records if r["role"] == role],\n            [majority] * sum(r["role"] == role for r in records),\n            names,\n        )\n        for role in ("validation", "test")\n    }\n    write_json(\n        root / "outputs/baseline.json",\n        {"majority_label": majority, "tie_rule": "lowest vocabulary index", "metrics": baseline},\n    )\n    for label, name in enumerate(names):\n        r = min((r for r in records if r["role"] == "train" and r["label"] == label), key=lambda r: r["id"])\n        plot_audio(np.load(root / r["audio"]), root / f"outputs/examples/{r[\'id\']}.png", name)\n    return summary\n\n\ndef cache_identity(root, records, extractor_config):\n    identity = {\n        "inputs": [(r["id"], r["waveform_sha256"]) for r in records],\n        "extractor": extractor_config,\n        "dtype": "float32",\n        "source": read_json(root / "source.json"),\n        "worker_sha256": digest(__file__),\n        "weights": read_json(root / "weights/ast/dimer-base-manifest.json"),\n    }\n    return hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()\n\n\ndef require_gpu():\n    import torch\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Model stages require a hosted CUDA GPU; local validation is CPU-only")\n    torch.cuda.reset_peak_memory_stats()\n    torch.manual_seed(SEED)\n    torch.backends.cudnn.benchmark = False\n    return "cuda:0"\n\n\ndef load_pipe(root):\n    from ast_reference.pipeline import ASTAudioClassificationPipeline\n\n    return ASTAudioClassificationPipeline.from_pretrained(\n        device=require_gpu(), weights_dir=root / "weights/ast", allow_download=True\n    )\n\n\ndef backbone_hash(model):\n    h = hashlib.sha256()\n    for name, tensor in model.audio_spectrogram_transformer.state_dict().items():\n        h.update(name.encode())\n        h.update(tensor.detach().cpu().contiguous().numpy().tobytes())\n    return h.hexdigest()\n\n\ndef extract_features(pipe, waveforms):\n    import torch\n\n    result = []\n    pipe.model.audio_spectrogram_transformer.eval()\n    for start in range(0, len(waveforms), 4):\n        batch = pipe.extractor(waveforms[start : start + 4], sampling_rate=16000, return_tensors="pt")\n        with torch.inference_mode():\n            feature = pipe.model.audio_spectrogram_transformer(\n                batch["input_values"].to(pipe.device)\n            ).pooler_output\n            array = feature.float().cpu().numpy().copy()\n        # Ordinary tensors outside inference_mode can be saved for classifier backward.\n        result.append(torch.from_numpy(array).to(pipe.device))\n    return torch.cat(result)\n\n\ndef head_logits(pipe, features):\n    import torch\n\n    pipe.model.classifier.eval()\n    with torch.inference_mode():\n        return pipe.model.classifier(features).float().cpu().numpy()\n\n\ndef softmax(logits):\n    x = np.asarray(logits, dtype=np.float64)\n    ex = np.exp(x - x.max(axis=-1, keepdims=True))\n    return ex / ex.sum(axis=-1, keepdims=True)\n\n\ndef train_head(classifier, train_features, targets, val_features, val_targets, names):\n    """One optimizer, exact macro-F1, epoch-zero snapshot and earliest winning tie."""\n    import torch\n    import torch.nn.functional as F\n\n    torch.manual_seed(SEED)\n    initial = {k: v.detach().cpu().clone() for k, v in classifier.state_dict().items()}\n    best = initial\n    best_score = -1.0\n    selected = 0\n    history = []\n    optimizer = torch.optim.AdamW(classifier.parameters(), lr=0.001, weight_decay=0.01)\n    steps = 0\n    maximum_delta = 0.0\n    for epoch in range(11):\n        loss_total = 0.0\n        count = 0\n        if epoch:\n            classifier.train()\n            order = list(range(len(targets)))\n            random.Random(SEED + epoch * 13).shuffle(order)\n            for start in range(0, len(order), 4):\n                indices = torch.tensor(order[start : start + 4], device=train_features.device)\n                optimizer.zero_grad(set_to_none=True)\n                loss = F.cross_entropy(classifier(train_features[indices]), targets[indices])\n                if not torch.isfinite(loss):\n                    raise ValueError("nonfinite training loss")\n                loss.backward()\n                optimizer.step()\n                steps += 1\n                loss_total += float(loss.detach()) * len(indices)\n                count += len(indices)\n            maximum_delta = max(\n                maximum_delta,\n                max(\n                    float((v.detach().cpu() - initial[k]).abs().max())\n                    for k, v in classifier.state_dict().items()\n                ),\n            )\n        classifier.eval()\n        with torch.inference_mode():\n            pred = classifier(val_features).argmax(-1).cpu().tolist()\n        report = metrics(val_targets.cpu().tolist(), pred, names)\n        if report["macro_f1"] > best_score:\n            best_score = report["macro_f1"]\n            selected = epoch\n            best = {k: v.detach().cpu().clone() for k, v in classifier.state_dict().items()}\n        history.append(\n            {\n                "epoch": epoch,\n                "train_loss": loss_total / count if count else None,\n                "validation_macro_f1": report["macro_f1"],\n                "validation_accuracy": report["accuracy"],\n                "optimizer_steps": steps,\n            }\n        )\n        print(f"Epoch {epoch}: validation macro-F1={report[\'macro_f1\']:.6f}, updates={steps}", flush=True)\n    if steps != 10 * math.ceil(len(targets) / 4) or maximum_delta <= 0:\n        raise RuntimeError("actual optimizer updates or parameter changes missing")\n    classifier.load_state_dict(best)\n    return (\n        initial,\n        history,\n        {\n            "selected_epoch": selected,\n            "validation_macro_f1": best_score,\n            "optimizer_steps": steps,\n            "max_head_delta_during_training": maximum_delta,\n            "selected_max_head_delta": max(float((v - initial[k]).abs().max()) for k, v in best.items()),\n            "tie_rule": "strict improvement, earliest epoch including zero",\n        },\n    )\n\n\ndef score_rows(records, logits, model, names):\n    probabilities = softmax(logits)\n    return [\n        {\n            "id": r["id"],\n            "role": r["role"],\n            "group_id": r["group_id"],\n            "fold": r.get("fold"),\n            "label": r["label"],\n            "category": names[r["label"]],\n            "model": model,\n            "predicted": int(np.argmax(row)),\n            "logits": row.tolist(),\n            "probabilities": prob.tolist(),\n        }\n        for r, row, prob in zip(records, logits, probabilities, strict=True)\n    ]\n\n\ndef train(root):\n    import torch\n\n    records = read_json(root / "outputs/records.json")\n    names = read_json(root / "outputs/configuration.json")["class_names"]\n    setup_start = time.perf_counter()\n    pipe = load_pipe(root)\n    torch.cuda.synchronize()\n    setup_seconds = time.perf_counter() - setup_start\n    extractor = pipe.extractor.to_dict()\n    write_json(root / "outputs/extractor.json", extractor)\n    example = next(r for r in records if r["role"] == "train")\n    x = np.load(root / example["audio"])\n    inputs = pipe.extractor(x, sampling_rate=16000, return_tensors="pt")["input_values"][0].numpy()\n    import matplotlib.pyplot as plt\n\n    fig, ax = plt.subplots(figsize=(10, 4))\n    ax.imshow(inputs.T, aspect="auto", origin="lower")\n    valid = max(0, (len(x) - 400) // 160 + 1)\n    ax.axvline(valid - 0.5, color="red", label=f"Observed frames end ({valid}); padding follows")\n    ax.set(xlabel="10 ms frame index", ylabel="Mel bin", title="Actual normalized AST input")\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(root / "outputs/model_input.png")\n    plt.close(fig)\n    pipe.rehead(names, seed=SEED)\n    pipe.freeze_backbone()\n    before = backbone_hash(pipe.model)\n    trainable = {name: p.numel() for name, p in pipe.model.named_parameters() if p.requires_grad}\n    if not trainable or any(not n.startswith("classifier.") for n in trainable):\n        raise RuntimeError("unexpected trainable backbone parameters")\n    subsets = {role: [r for r in records if r["role"] == role] for role in ROLES}\n    features = {}\n    cache_start = time.perf_counter()\n    for role in ("train", "validation"):\n        subset = subsets[role]\n        features[role] = extract_features(pipe, [np.load(root / r["audio"]) for r in subset])\n        np.save(root / f"data/features_{role}.npy", features[role].cpu().numpy())\n    torch.cuda.synchronize()\n    cache_seconds = time.perf_counter() - cache_start\n    identity = cache_identity(root, records, extractor)\n    write_json(\n        root / "outputs/cache_identity.json",\n        {\n            "identity": identity,\n            "feature_cache_seconds": cache_seconds,\n            "files": {role: digest(root / f"data/features_{role}.npy") for role in features},\n        },\n    )\n    targets = {\n        role: torch.tensor([r["label"] for r in subset], device=pipe.device)\n        for role, subset in subsets.items()\n        if role in ("train", "validation")\n    }\n    torch.cuda.synchronize()\n    training_start = time.perf_counter()\n    initial, history, selection = train_head(\n        pipe.model.classifier,\n        features["train"],\n        targets["train"],\n        features["validation"],\n        targets["validation"],\n        names,\n    )\n    torch.cuda.synchronize()\n    selection["training_seconds"] = time.perf_counter() - training_start\n    selection["setup_including_model_download_seconds"] = setup_seconds\n    selection["feature_cache_seconds"] = cache_seconds\n    selected = {k: v.detach().cpu().clone() for k, v in pipe.model.classifier.state_dict().items()}\n    if before != backbone_hash(pipe.model):\n        raise RuntimeError("frozen backbone changed")\n    selection.update(\n        {\n            "backbone_sha256_before": before,\n            "backbone_sha256_after": before,\n            "trainable_parameters": trainable,\n            "pid": os.getpid(),\n            "precision": "float32",\n            "peak_gpu_bytes": torch.cuda.max_memory_allocated(),\n        }\n    )\n    write_json(root / "outputs/training_history.json", history)\n    write_json(root / "outputs/selection.json", selection)\n    # Test extraction and label evaluation occur only after selection is irrevocably recorded.\n    features["test"] = extract_features(pipe, [np.load(root / r["audio"]) for r in subsets["test"]])\n    rows = []\n    for model, state in (("initial", initial), ("selected", selected)):\n        pipe.model.classifier.load_state_dict(state)\n        for role in ("validation", "test"):\n            rows.extend(score_rows(subsets[role], head_logits(pipe, features[role]), model, names))\n    write_json(root / "outputs/scores.json", rows)\n    reports = {"majority": read_json(root / "outputs/baseline.json")["metrics"]}\n    for model in ("initial", "selected"):\n        reports[model] = {}\n        for role in ("validation", "test"):\n            subset = [r for r in rows if r["model"] == model and r["role"] == role]\n            reports[model][role] = metrics(\n                [r["label"] for r in subset], [r["predicted"] for r in subset], names\n            )\n    write_json(root / "outputs/metrics.json", reports)\n    pipe.adaptation_config.update({"recipe": RECIPE, "selection": selection})\n    path = pipe.save_artifact(root / "outputs/selected_head.pt")\n    from ast_reference.pipeline import MODEL_ID, MODEL_REVISION\n\n    write_json(\n        root / "outputs/adapter_manifest.json",\n        {\n            "sha256": digest(path),\n            "base_model_id": MODEL_ID,\n            "base_model_revision": MODEL_REVISION,\n            "class_names": names,\n            "activation": "softmax",\n            "extractor_sha256": digest(root / "outputs/extractor.json"),\n            "recipe": RECIPE,\n            "selection": selection,\n            "tensor_schema": {\n                k: {"shape": list(v.shape), "dtype": str(v.dtype)} for k, v in selected.items()\n            },\n        },\n    )\n\n\ndef verify_parity(live, reloaded):\n    a, b = np.asarray(live), np.asarray(reloaded)\n    if a.shape != b.shape or not np.isfinite(a).all() or not np.isfinite(b).all():\n        raise ValueError("reload shape/finite mismatch")\n    pa, pb = softmax(a), softmax(b)\n    if (\n        not np.allclose(a, b, atol=1e-5, rtol=1e-4)\n        or not np.allclose(pa, pb, atol=1e-6, rtol=1e-4)\n        or not np.array_equal(a.argmax(-1), b.argmax(-1))\n    ):\n        raise ValueError("reload parity failed")\n    return {\n        "passed": True,\n        "max_logit_delta": float(np.max(np.abs(a - b))),\n        "max_probability_delta": float(np.max(np.abs(pa - pb))),\n    }\n\n\ndef strict_load(root, pipe):\n    import torch\n    from ast_reference.pipeline import MODEL_ID, MODEL_REVISION\n\n    manifest = read_json(root / "outputs/adapter_manifest.json")\n    names = read_json(root / "outputs/configuration.json")["class_names"]\n    path = root / "outputs/selected_head.pt"\n    if (\n        digest(path) != manifest["sha256"]\n        or manifest["class_names"] != names\n        or manifest["activation"] != "softmax"\n        or manifest["base_model_id"] != MODEL_ID\n        or manifest["base_model_revision"] != MODEL_REVISION\n    ):\n        raise ValueError("adapter digest, vocabulary or lineage mismatch")\n    if digest(root / "outputs/extractor.json") != manifest[\n        "extractor_sha256"\n    ] or pipe.extractor.to_dict() != read_json(root / "outputs/extractor.json"):\n        raise ValueError("extractor identity changed")\n    payload = torch.load(path, map_location="cpu", weights_only=True)\n    if payload["class_names"] != names or payload["activation"] != "softmax":\n        raise ValueError("payload vocabulary or activation mismatch")\n    pipe.rehead(names, seed=SEED)\n    expected = pipe.model.classifier.state_dict()\n    state = payload["classifier_state_dict"]\n    if set(state) != set(expected):\n        raise ValueError("adapter tensor names mismatch")\n    for key, tensor in state.items():\n        if (\n            not isinstance(tensor, torch.Tensor)\n            or tensor.shape != expected[key].shape\n            or tensor.dtype != expected[key].dtype\n            or not torch.isfinite(tensor).all()\n        ):\n            raise ValueError("adapter tensor schema or finiteness mismatch")\n    # Real repository loader used after stricter checks; no unrestricted pickle fallback.\n    pipe.load_artifact(path)\n    return pipe\n\n\ndef reload(root):\n    selection = read_json(root / "outputs/selection.json")\n    if selection["pid"] == os.getpid():\n        raise RuntimeError("reload must run in a fresh process")\n    pipe = strict_load(root, load_pipe(root))\n    names = pipe.labels\n    records = read_json(root / "outputs/records.json")\n    live = read_json(root / "outputs/scores.json")\n    probes = sorted((r for r in records if r["role"] == "validation"), key=lambda r: r["id"])[:4]\n    tested = probes + [r for r in records if r["role"] == "test"]\n    logits = head_logits(pipe, extract_features(pipe, [np.load(root / r["audio"]) for r in tested]))\n    live_by_id = {r["id"]: r for r in live if r["model"] == "selected"}\n    parity = verify_parity([live_by_id[r["id"]]["logits"] for r in tested], logits)\n    rows = score_rows(tested, logits, "reloaded", names)\n    test = [r for r in rows if r["role"] == "test"]\n    report = metrics([r["label"] for r in test], [r["predicted"] for r in test], names)\n    previous = read_json(root / "outputs/metrics.json")["selected"]["test"]\n    if report["confusion_matrix"] != previous["confusion_matrix"] or any(\n        abs(report[k] - previous[k]) > 1e-12 for k in ("accuracy", "macro_f1")\n    ):\n        raise ValueError("reload metric parity failed")\n    parity.update({"pid": os.getpid(), "probes": len(probes), "test_clips": len(test), "metric_parity": True})\n    write_json(root / "outputs/reload_scores.json", rows)\n    write_json(root / "outputs/reload_parity.json", parity)\n\n\ndef activity(root):\n    pipe = strict_load(root, load_pipe(root))\n    names = pipe.labels\n    records = {r["id"]: r for r in read_json(root / "outputs/records.json")}\n    ids = read_json(root / "outputs/configuration.json")["activity_ids"]\n    rows = []\n    for number, clip_id in enumerate(ids):\n        record = records[clip_id]\n        conditions = noise_conditions(np.load(root / record["audio"]), clip_id)\n        logits = head_logits(pipe, extract_features(pipe, [c["audio"] for c in conditions]))\n        for condition, row in zip(\n            conditions, score_rows([record] * 4, logits, "selected", names), strict=True\n        ):\n            if number == 0:\n                plot_audio(\n                    condition["audio"],\n                    root / f"outputs/noise_examples/{clip_id}_{condition[\'condition\']}.png",\n                    f"Gain-matched {condition[\'condition\']} dB",\n                )\n            row.update({k: v for k, v in condition.items() if k != "audio"})\n            rows.append(row)\n    summary = []\n    clean = {r["id"]: r["predicted"] for r in rows if r["condition"] == "clean"}\n    for condition in ("clean", "20", "10", "0"):\n        subset = [r for r in rows if r["condition"] == condition]\n        report = metrics([r["label"] for r in subset], [r["predicted"] for r in subset], names)\n        summary.append(\n            {\n                "condition": condition,\n                **report,\n                "changed_predictions": sum(r["predicted"] != clean[r["id"]] for r in subset),\n            }\n        )\n    write_json(root / "outputs/noise_records.json", rows)\n    write_json(root / "outputs/noise_summary.json", summary)\n\n\ndef report(root):\n    import csv\n    import importlib.metadata\n\n    import matplotlib.pyplot as plt\n\n    output = root / "outputs"\n    reports = read_json(output / "metrics.json")\n    records = read_json(output / "records.json")\n    rows = read_json(output / "scores.json")\n    initial = {r["id"]: r for r in rows if r["model"] == "initial" and r["role"] == "test"}\n    selected = [r for r in rows if r["model"] == "selected" and r["role"] == "test"]\n    transitions = {"corrected": 0, "new_errors": 0, "unchanged_correct": 0, "unchanged_incorrect": 0}\n    for r in selected:\n        old = initial[r["id"]]["predicted"] == r["label"]\n        new = r["predicted"] == r["label"]\n        transitions[\n            ("unchanged_correct" if old else "corrected")\n            if new\n            else ("new_errors" if old else "unchanged_incorrect")\n        ] += 1\n    mistakes = sorted(\n        (r for r in selected if r["predicted"] != r["label"]),\n        key=lambda r: (-max(r["probabilities"]), r["id"]),\n    )[:5]\n    cm = np.array(reports["selected"]["test"]["confusion_matrix"])\n    offdiag = cm.copy()\n    np.fill_diagonal(offdiag, 0)\n    pair = list(map(int, np.unravel_index(offdiag.argmax(), offdiag.shape))) if offdiag.max() else None\n    write_json(\n        output / "paired_errors.json",\n        {\n            **transitions,\n            "highest_confidence_mistakes": mistakes,\n            "representative_confusion_pair": pair,\n            "fixed_example": min(selected, key=lambda r: r["id"]),\n            "selection_rule": (\n                "test filename minimum; largest off-diagonal count row-major tie; "\n                "error softmax descending then filename"\n            ),\n        },\n    )\n    names = read_json(output / "configuration.json")["class_names"]\n    fig, axes = plt.subplots(1, 2, figsize=(14, 6))\n    for ax, matrix, title in zip(\n        axes, (cm, cm / cm.sum(1, keepdims=True)), ("Counts", "Row-normalized"), strict=True\n    ):\n        im = ax.imshow(matrix, cmap="Blues")\n        fig.colorbar(im, ax=ax)\n        ax.set(\n            xticks=range(len(names)),\n            yticks=range(len(names)),\n            xticklabels=names,\n            yticklabels=names,\n            xlabel="Predicted",\n            ylabel="Reference",\n            title=title,\n        )\n        ax.tick_params(axis="x", rotation=90)\n    fig.tight_layout()\n    fig.savefig(output / "confusion.png")\n    plt.close(fig)\n    with (output / "scores.csv").open("w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    with (output / "scores.csv").open(encoding="utf-8", newline="") as f:\n        loaded = list(csv.DictReader(f))\n    if len(loaded) != len(rows) or any(\n        a["id"] != b["id"] or json.loads(a["logits"]) != b["logits"]\n        for a, b in zip(loaded, rows, strict=True)\n    ):\n        raise ValueError("exported score table round trip failed")\n    lines = [\n        "# Sound event classification results",\n        "",\n        "| System | Test accuracy | Test macro-F1 |",\n        "|---|---:|---:|",\n    ]\n    for system in ("majority", "initial", "selected"):\n        score = reports[system]["test"]\n        lines.append(f"| {system} | {score[\'accuracy\']:.6f} | {score[\'macro_f1\']:.6f} |")\n    lines += [\n        "",\n        (\n            f"Selected epoch: {read_json(output / \'selection.json\')[\'selected_epoch\']}; "\n            f"actual updates: {read_json(output / \'selection.json\')[\'optimizer_steps\']}."\n        ),\n        "",\n        (\n            "Multiple clips from one source are correlated. This split is not the five-fold ESC benchmark. "\n            "AudioSet pretraining overlap has not been ruled out."\n        ),\n        "Softmax is uncalibrated; synthetic additive noise is not evidence of real-world noise robustness.",\n        "Candidate: this run is evidence for maintainer review, not automatic release approval.",\n    ]\n    (output / "summary.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    write_json(\n        output / "run_summary.json",\n        {\n            "status": "completed_candidate",\n            "records": len(records),\n            "class_names": names,\n            "test_metrics": reports["selected"]["test"],\n            "reload": read_json(output / "reload_parity.json"),\n            "private_byod": read_json(output / "prepare.json")["private_byod"],\n            "release_approved": False,\n        },\n    )\n    write_json(\n        output / "provenance.json",\n        {\n            "python": sys.version,\n            "packages": {\n                p: importlib.metadata.version(p)\n                for p in ("torch", "torchaudio", "transformers", "numpy", "scipy")\n            },\n            "source": read_json(root / "source.json"),\n            "requirements_sha256": digest(root / "requirements.txt"),\n            "stage_receipts_before_report_completion": read_json(root / "stage_receipts.json"),\n            "base_manifest": read_json(root / "weights/ast/dimer-base-manifest.json"),\n        },\n    )\n    allow = [\n        "data_manifest.json",\n        "records.json",\n        "prepare.json",\n        "configuration.json",\n        "baseline.json",\n        "extractor.json",\n        "cache_identity.json",\n        "training_history.json",\n        "selection.json",\n        "scores.json",\n        "scores.csv",\n        "metrics.json",\n        "selected_head.pt",\n        "adapter_manifest.json",\n        "reload_scores.json",\n        "reload_parity.json",\n        "noise_records.json",\n        "noise_summary.json",\n        "paired_errors.json",\n        "summary.md",\n        "run_summary.json",\n        "provenance.json",\n        "confusion.png",\n    ]\n    checksums = {name: digest(output / name) for name in allow}\n    write_json(output / "checksums.json", checksums)\n    with zipfile.ZipFile(output / "sound_results.zip", "w", zipfile.ZIP_DEFLATED) as z:\n        for name in allow + ["checksums.json"]:\n            z.write(output / name, name)\n    with zipfile.ZipFile(output / "sound_results.zip") as z:\n        if any(hashlib.sha256(z.read(name)).hexdigest() != value for name, value in checksums.items()):\n            raise ValueError("archive hash mismatch")\n\n\ndef input_identity(root):\n    paths = [\n        root / "source.json",\n        root / "requirements.txt",\n        root / "sample.json",\n        root / "weights/ast/dimer-base-manifest.json",\n        Path(__file__),\n    ]\n    paths += [root / "outputs/records.json", root / "outputs/configuration.json"]\n    result = {\n        str(p.relative_to(root)) if p.is_relative_to(root) else p.name: digest(p) for p in paths if p.exists()\n    }\n    if (root / "outputs/records.json").exists():\n        for r in read_json(root / "outputs/records.json"):\n            result[r["audio"]] = digest(root / r["audio"])\n    return result\n\n\ndef begin_stage(root, stage):\n    path = root / "stage_receipts.json"\n    receipts = read_json(path) if path.exists() else {"run_id": uuid.uuid4().hex, "stages": {}}\n    idx = STAGES.index(stage)\n    if stage == "prepare":\n        receipts = {"run_id": uuid.uuid4().hex, "stages": {}}\n    else:\n        current = input_identity(root)\n        for previous in STAGES[:idx]:\n            receipt = receipts["stages"].get(previous, {})\n            if receipt.get("status") != "complete" or receipt["identity"] != current:\n                raise RuntimeError("missing, failed or stale prerequisite stage")\n            if any(\n                not (root / p).exists() or digest(root / p) != sha\n                for p, sha in receipt.get("outputs", {}).items()\n            ):\n                raise RuntimeError("prerequisite output changed")\n    for descendant in STAGES[idx:]:\n        receipts["stages"].pop(descendant, None)\n    receipts["stages"][stage] = {"status": "running", "pid": os.getpid()}\n    write_json(path, receipts)\n    return time.perf_counter()\n\n\nSTAGE_FILES = {\n    "prepare": ["data_manifest.json", "records.json", "prepare.json", "configuration.json", "baseline.json"],\n    "train": [\n        "training_history.json",\n        "selection.json",\n        "scores.json",\n        "metrics.json",\n        "selected_head.pt",\n        "adapter_manifest.json",\n        "extractor.json",\n    ],\n    "reload": ["reload_scores.json", "reload_parity.json"],\n    "activity": ["noise_records.json", "noise_summary.json"],\n    "report": ["sound_results.zip", "checksums.json", "run_summary.json"],\n}\n\n\ndef finish_stage(root, stage, start):\n    receipts = read_json(root / "stage_receipts.json")\n    rss = None\n    if sys.platform != "win32":\n        import resource\n\n        rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024\n    gpu_peak = None\n    torch_module = sys.modules.get("torch")\n    if torch_module is not None and torch_module.cuda.is_initialized():\n        torch_module.cuda.synchronize()\n        gpu_peak = torch_module.cuda.max_memory_allocated()\n    receipts["stages"][stage] = {\n        "status": "complete",\n        "pid": os.getpid(),\n        "seconds": time.perf_counter() - start,\n        "peak_host_rss_bytes": rss,\n        "peak_gpu_bytes": gpu_peak,\n        "identity": input_identity(root),\n        "outputs": {f"outputs/{name}": digest(root / "outputs" / name) for name in STAGE_FILES[stage]},\n    }\n    write_json(root / "stage_receipts.json", receipts)\n\n\ndef byod_run(root, directory):\n    child = root / "byod_runs" / uuid.uuid4().hex\n    child.mkdir(parents=True)\n    import shutil\n\n    for name in ("source.json", "requirements.txt", "sample.json"):\n        shutil.copyfile(root / name, child / name)\n    (child / "weights").symlink_to((root / "weights").resolve(), target_is_directory=True)\n    for stage in STAGES:\n        command = [sys.executable, __file__, "--root", str(child), "--stage", stage]\n        if stage == "prepare":\n            command += ["--byod", str(directory)]\n        subprocess.run(command, check=True)\n    write_json(\n        root / "outputs/byod_output.json",\n        {"root": str(child), "archive": str(child / "outputs/sound_results.zip")},\n    )\n\n\ndef infer(root, audio_path):\n    x, info = read_audio(audio_path)\n    pipe = strict_load(root, load_pipe(root))\n    logits = head_logits(pipe, extract_features(pipe, [x]))[0]\n    write_json(\n        root / "outputs/inference.json",\n        {\n            "class_names": pipe.labels,\n            "logits": logits.tolist(),\n            "probabilities": softmax(logits).tolist(),\n            "predicted": pipe.labels[int(logits.argmax())],\n            "accuracy": None,\n            "macro_f1": None,\n            "reason": "unlabelled audio; no unknown-class rejection",\n            "audio": info,\n        },\n    )\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--stage", choices=(*STAGES, "byod", "infer"), required=True)\n    parser.add_argument("--byod", type=Path)\n    parser.add_argument("--audio", type=Path)\n    args = parser.parse_args()\n    root = args.root.resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    (root / "outputs").mkdir(exist_ok=True)\n    if args.stage == "byod":\n        if args.byod is None:\n            parser.error("--byod directory required")\n        byod_run(root, args.byod.resolve())\n        return\n    if args.stage == "infer":\n        if args.audio is None:\n            parser.error("--audio WAV required")\n        infer(root, args.audio)\n        return\n    start = begin_stage(root, args.stage)\n    try:\n        if args.stage == "prepare":\n            prepare(root, args.byod.resolve() if args.byod else None)\n        else:\n            globals()[args.stage](root)\n        finish_stage(root, args.stage, start)\n    except Exception as error:\n        receipts = read_json(root / "stage_receipts.json")\n        receipts["stages"][args.stage].update({"status": "failed", "error_type": type(error).__name__})\n        write_json(root / "stage_receipts.json", receipts)\n        raise\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements.txt': '# This file was autogenerated by uv via the following command:\n#    uv pip compile tools/sound-workshop-requirements.in --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --output-file tools/sound-workshop-requirements.lock\ncertifi==2026.7.22 \\\n    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \\\n    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55\n    # via requests\ncharset-normalizer==3.5.1 \\\n    --hash=sha256:00668ebb0609751758682eb0b5857e7c35b9f00e84dfdef062e103244ec94d45 \\\n    --hash=sha256:012a22b88a77ca2e59b98ac5889b0deb604147666032f45e6d6e217634d2550d \\\n    --hash=sha256:01e93745f7f219b703b60ba7afead36cfc4242782be5af484673fc500df12da5 \\\n    --hash=sha256:04368edf83514385ffc3e1cfd4546e595f4f1272dd23ba437a93a9cc3741d47b \\\n    --hash=sha256:0722590aabf9dc6a6c0343d523c05458fa2b5047dbe6302fd526bb570600753f \\\n    --hash=sha256:07ffd07412fc5d5e84cd8952acf9ff7e4ed7a708e69d1bada19d8ba91711353f \\\n    --hash=sha256:09a7bba9f739468c8e78c36a75c33768e53cb1959fc638f510454c14683f00d5 \\\n    --hash=sha256:0b2b1b3fa5670c127b246df1d0c059defd41f689a868a3b9d79df9b1cac42d22 \\\n    --hash=sha256:0c6dfb5ca6723eeed15aa8e564a014d69fcb8812f94eef11fe3631e0508199f5 \\\n    --hash=sha256:0d929fc574b4d6fd9e7c0f5c2ede8716a41911923aa7fa5fce38e0818aa4a1ac \\\n    --hash=sha256:13e3afe97712e8887cd516e960c63f0b93122971e5b5e4b2622fe7701771e838 \\\n    --hash=sha256:15f024313246a4ed976c60f440bb8d257815513a681d212ff74fd46f7d715a90 \\\n    --hash=sha256:195ce897c6153c0700078142cf8efe3e6454ca4cf4357499e4078dfd83396626 \\\n    --hash=sha256:19a3dd5aa73cef1c99687c4fc57db016a9c17104ae1185da88ba566a5d3bebe4 \\\n    --hash=sha256:1d1c7a53a6c2103925cdd6d7229f8c567379f211c869793df679f2e9f738c369 \\\n    --hash=sha256:1f5883d77fd409a261abb5dc8ccbe335720d798b1de4abb3b1d47ccbbc76b53b \\\n    --hash=sha256:21b82d8082f6f5e7f456ef0bd16323d08de1266efbfeb476e64b2a91d1471a4e \\\n    --hash=sha256:252d099029bcbea642f2a06c4ed5046bdf8b5a8150b64afa5e027e88b106e5ee \\\n    --hash=sha256:256dd4d85d9e4dc595e2bc983c980e73f62ddeb3165c58b4c3dfe78c5c8548c1 \\\n    --hash=sha256:26422d45fd13551cf564c58932f7d72b4f58b93b0fcf18c35ba6be12b46bb102 \\\n    --hash=sha256:2679de311c7946dde5d3b6f44941844133ff5c7cb86099c0061ab1e8901c20a8 \\\n    --hash=sha256:29880d17a8eb0b5cfdfd8944b468322928059aa35f1f5fa8ff22b149ec0b42f8 \\\n    --hash=sha256:2bced4061f000f7187254a02ad3433ae17eaf991747ceea2f478422590a5bba9 \\\n    --hash=sha256:2e9cf9253119d8e5d111f05d71626786fd3d6193817316eab1ca088cdb8593cf \\\n    --hash=sha256:2f06b7eae9dbe77fe1d644ca244dad508de8d302870a43f3c559b521270938a0 \\\n    --hash=sha256:2f293479cce755c75f1697e87c409b7ae4c555c7dfecb6e988ad13abba943031 \\\n    --hash=sha256:329fc3ccb63ad22d867d84c2adea759a64079a37ba4a343433b02c7a2816871e \\\n    --hash=sha256:343fb4f2821043bd87095f7b08a1a181febc8e36ac64212143bbfd0a0e1bc235 \\\n    --hash=sha256:3588e376b3ea2eea84976f67273d679f229e24c66dce7b82ae45aef04ff6e072 \\\n    --hash=sha256:35aea775dc2bd5f54cd84a1cd2696cc3207c479cb9cf0bd346f0d343e4300ddb \\\n    --hash=sha256:35fe081843b35aad20ffeccec3eeffbe637b15d14f3fb22cc1b59cd8ec17e93c \\\n    --hash=sha256:36047af20e17097c3bb9476c2b7655f2f7aa51322c0ba58c07695bedf755a950 \\\n    --hash=sha256:3617ac3cfd8b9888f145ad89dd6e692285834b0201c6074a5eeaad3fd4d668c2 \\\n    --hash=sha256:366ec70f5547c640d3ce1985722490f23faf4eb5216a7eeba78277490e78dacb \\\n    --hash=sha256:394fea06235c8543390050ed5f529187074b029fb027213f6c46ac11ab5d950e \\\n    --hash=sha256:3d27167433c0d5f18dc850f07d0b3816221984fecdc405d6c157a6f0b8f8e9e6 \\\n    --hash=sha256:3e5e1224c0a6a90e05843e07adfec669edebec17801c67072f51e59561d63c0b \\\n    --hash=sha256:41876ee62a3dddf48ff1121ad8f0798032aa03f2fd35f21f34a4cab14f18d8d2 \\\n    --hash=sha256:433c5a81eade63b47e522303bad236f59dba55ea6951746f5558355eeed8c75d \\\n    --hash=sha256:4582c27e8c889d64811987b5967fbd3ae0c823fe1fd933b543d55ac20bb475fa \\\n    --hash=sha256:485a0d363cafefcd2538a73c7c838daa2035f09b2c9f9b5e3133f80c6aeb84c2 \\\n    --hash=sha256:494b70049a4d69aec6e8137c13af4cf8db8c9f9820a1392ac293b0dd2987a818 \\\n    --hash=sha256:496846868fea80e479324862fa877f02411f2fd0f83b79ccee2607aa68b2a032 \\\n    --hash=sha256:4abdc5f9ad448c1ecbfae2974b820535d6bc6e7eef63babbab3d81cf46968c71 \\\n    --hash=sha256:4b599739b93b2cbeded49645ae3c8d1405c29ddfbceac1545c87a3f9580a9e96 \\\n    --hash=sha256:4bea7f8ebe90bbd7f0e4a2de42ca6924ba23e3e76418c408ff82f1d46fabd687 \\\n    --hash=sha256:4c4fb141a727957c93edfe5c32a26ceb6b5f6461d67146e2d39f51e16170bea8 \\\n    --hash=sha256:4c9548dc78002099910abaebc0a72ac58b7d30931869e0351c09b507dff4ece3 \\\n    --hash=sha256:4d26f14f041e83dd8edfd61f4cd4fa7285d31798b5bf1f28e70c367ba6c41d61 \\\n    --hash=sha256:4f298bdadb8f0b9e5672877f647d1be9373ef5320c9e2f049795e26cad28b6a9 \\\n    --hash=sha256:52ec005752a56ae79547a05c0139ca2501a0c866390b6115008456b9f0e7cde1 \\\n    --hash=sha256:55261ac0d2941c42f196dd576f543d87a8ee03cd6f5e30dfb4d807b2e3b9121a \\\n    --hash=sha256:56490c595a28b1bb27dfc583e816152a9767721ef58b2c03b13f954d2f707420 \\\n    --hash=sha256:58d3e12c88e0950bca850ae1f7c256055c097639c2edb9eb123af9807d8b15e4 \\\n    --hash=sha256:58d4aa13a59c969dbfdf9e6a9560e242cbfd9e8a8f50c2747714df1a423adf65 \\\n    --hash=sha256:59171c6e45bf07d0d5cab3b0bf81d945035530f6873398b3b531c31184d46663 \\\n    --hash=sha256:5b6d1386bf0096d26d3a863dc0a487a5b4eb9aa93cf5ba69683d29dde6b9d60f \\\n    --hash=sha256:5c0ea61a470e070686aa30892fed79e297d2c8d0ab46b8bcdf027d38c51da591 \\\n    --hash=sha256:5c84bec0ab5ae0c64bfe73a7d2adcb5ce73b467523fc27fd6a28ab2aa6cbe35a \\\n    --hash=sha256:5ca0555312ae2fe82715cada7fac375530c2f3349e1eaa1bcb33d0283ac79a18 \\\n    --hash=sha256:5d8531a6569d025f68e2321e7638fb7978f23db58e5f69f56913837aae03816e \\\n    --hash=sha256:5e2d0e146dcb57034f8b97dc58d2d512cb90aba253960ce449f695fec6a82c6f \\\n    --hash=sha256:5fc45d653ea8c9a20479167e11d4a0f8cb2fa3470737ab6f9c827532313187b7 \\\n    --hash=sha256:6117b84ea48435e5356dc737f5121485c30920ba43375fa7b434fd753df0eac3 \\\n    --hash=sha256:6199d5606e2bbf2b096cf64d03f8b6790c91081d5ac866b8e7bb6422738cc60c \\\n    --hash=sha256:62b55f6722735a6c472f88361cde6640608773d9443cebdbb51abf436a1fcdd3 \\\n    --hash=sha256:687c9ca3035544b113bea2055e180af96fb63c0c476e22a9180f51925186e7b7 \\\n    --hash=sha256:6b7430cf5728e68f6c462254009a6ef4086e1bea43cf2f57aa9c55fb4f50ff96 \\\n    --hash=sha256:6ba32c4d2abf1d2fe7cf27d280f4cca5664233b0f885549c7761719eb977f486 \\\n    --hash=sha256:6c9cdde8becb25a7fde49924511aa2644d6f8081cc8df8e9452724303348d8e3 \\\n    --hash=sha256:6df0ec430f9a831772c23ca5a224cba36517a58a84bb32c32bb59a9fa67c47f6 \\\n    --hash=sha256:6e2912d4babbc65196ac13c2f53468dc57fb8b9c25ef913e8c59ddf7c6dc0e1b \\\n    --hash=sha256:6e5e4d73d588ca5ed09df1b7dcd1b203d1df3c542e3f50d126c947d432b10731 \\\n    --hash=sha256:70055ff39b97c99e7ae40ea3e393fb62aa2e44dbd9b29f8d14f42fb0025c3959 \\\n    --hash=sha256:706bfd38730a5ac7a365793269a00f4e988178cec121391f4248d84ad8c972e9 \\\n    --hash=sha256:7235dc28fc6dd9d832ac7c7bce95367dedb85929f17368a0c2bee1e080b9acbf \\\n    --hash=sha256:774d157f112367ff4abd29019f38f023c24e00e56edc7829c20e358a5a913ad8 \\\n    --hash=sha256:77efcff2b23071c349402ac1066667a3d011f62398d81408c9b88ad991747c9e \\\n    --hash=sha256:789b8982559ae28dad2356519f841655756cdcd96616410590ae0b17454ee64f \\\n    --hash=sha256:7ac76cf9afd34929d76eb7fcb63be476a4853d8a96f0dcf2d0db68a0cbdf9885 \\\n    --hash=sha256:7c0c10730342b0c9b35dd1d619beb8214e520bd96a1f870f452680b238aab3e0 \\\n    --hash=sha256:823f82903d189af463d7df250ef1f7f696f3cee08cc8d91deb565e8d425f6506 \\\n    --hash=sha256:838648accb3a7fd9803fd45c87bce8509648eb0c11bc34e216141300977244f2 \\\n    --hash=sha256:854066be00447fa8de2ccbbe893e2ffc4b123ef16d897af794c1e18bd4a714b0 \\\n    --hash=sha256:85d5855daafc240cc045c026d7a15fd198a09b0fc8ff6f5ecbb5297b509cb11e \\\n    --hash=sha256:85de3134b5379856e323ba37c19c9256d39425f7b76a63af52b09fb4664c2e8f \\\n    --hash=sha256:87e4f41d375c0b9be2fb5251aee4b8a689169e134535aed81bf085c3b647451e \\\n    --hash=sha256:88ca277405c2d3b71c4e1c2ee0e7966e807bcba86a69d11e19ba199d18ae4491 \\\n    --hash=sha256:88e85ab89cb822c1e635f51d6d32e488f94e002e70e2f492bdb8b945543f345a \\\n    --hash=sha256:8ac8c94b6539074e0f40899301273ac8402b9b3e01c7b7ba269ff30340aaaf20 \\\n    --hash=sha256:8fe532b3c966d1fb794e0698e4589d0444017ae77fc0b31edea13c0e35bcc449 \\\n    --hash=sha256:9085f87b0e38a2b92b8923059b4e8789fe40d9279712d15dcc670048d77079af \\\n    --hash=sha256:90b7481fb62fbe172c558bc6fd1c4c98d82004a54a7551f20e11ac9bf0b8708c \\\n    --hash=sha256:92caef967d287a407085d61176fce4012b1dd62daed4eb6d5ceb26d3d2538712 \\\n    --hash=sha256:9362dd90aa7dab48c0054a21187791ccf05473f7dba5d92b8033ae62164675e7 \\\n    --hash=sha256:94d78ecec2605a8d0398b0f365d5f12a63248438516f5dac536a5eff7337df4a \\\n    --hash=sha256:94fbf1c0c6cc0d3d5e50f9a9313a8cdca90dd696d34b381cd1704f8c9e939f20 \\\n    --hash=sha256:950f23cb393f85543777b0433f082cddd25b51ab398eac7971146495679efe5f \\\n    --hash=sha256:96eefc178f8636b9c760c5829345307fd81cfae9ab1e80997dbddeb0f54ee9a3 \\\n    --hash=sha256:96fef3e886d6a9874b14f27fc193fbdc69d5d8035783d86aa4e1cea594e695f9 \\\n    --hash=sha256:977cdbd483a9cff38179bea4fd754289a6f2195c7abd414aba85410b3e66cc5e \\\n    --hash=sha256:978eab16f55b4ab2c2a745be9a0a840bf8f09a7f227d9c76eb30214d078865a5 \\\n    --hash=sha256:994e883d17c559cdfd38c84003c8b27d25424a1077272a17e7cd27bfe0bf57b2 \\\n    --hash=sha256:9ac4444d8d4fd4c4bd08bf451ed3167aa9e7ec6cdb41b648794f1d1103652e36 \\\n    --hash=sha256:9b5db6052055d34d41230fb78d7c439c23dc536a9896f6cb039e8dd92cfc1263 \\\n    --hash=sha256:9d9a0dc7cbe9bec24c3f767c9122c41fe5a1bc43f47cd099d00d393e09769de4 \\\n    --hash=sha256:9dbdd9205662134957cf0c324f639bdc5031c0ca056e2369e238db75187c0f11 \\\n    --hash=sha256:9eea3ab2597a5e65fe65296e2d6a84570845a6b55532d90333d740d48bbc850a \\\n    --hash=sha256:a2028475ba855475b8b4d3cfeb4994269c967aea8b9892dfba907f4263a863a3 \\\n    --hash=sha256:a3a370082ce34d0612f421e15fe011c53bb1feff21a26d06ad4fb244dab5a375 \\\n    --hash=sha256:a545775cfe815855ea32d7c27731d79da358ef2055b4a25830231b1622dd18aa \\\n    --hash=sha256:a5cbd90ecf0fc62e64726917ad083b73001f0563657a87ec3c0b504e277dc90d \\\n    --hash=sha256:a6d095662e73e74f0a49988e0593373e243e3a52e27bfeea0a859e88acf4a0f5 \\\n    --hash=sha256:a6dac12ff6b846103483683f60c5f8fee205121adc58ffd87e90a90a3af69e99 \\\n    --hash=sha256:a951ad59cad9145664a730d3036b40b844e74d2d3683da40111463cd3a83845d \\\n    --hash=sha256:aa1099b956fb795e686d073568f6dc002a0bb89765ea6d5b055dd7d9bf1b116c \\\n    --hash=sha256:aa2bb0b37202dca27175591f761108b5d34096ade1191ffe4808bdf6b1571488 \\\n    --hash=sha256:aae2ee51122d3ae968a3837d97dc24a0aeebb0dea23694422cd172bd30017cd6 \\\n    --hash=sha256:ab743e9bc90c1f73552ec33e10e3331315acd2c397b36065b591b0181de533cc \\\n    --hash=sha256:ac00177c4831ffa650f8609e4bdddd5fe09c03b1c0c47acece7e6ea20421598b \\\n    --hash=sha256:ac13b004224fb341e1e25a1ed5e19d32f57cdb2a403e01f003b46f051a550f6f \\\n    --hash=sha256:acaf604462bf330b0d07e7a07c1d6e4adac79e5fb13e9c5140590542cafacc00 \\\n    --hash=sha256:ae31a1a1db2ee6cc2942fccaf695c934bc7f3db9f2133a3fef1f367cf1a4ab10 \\\n    --hash=sha256:ae4a097991662cd4fff0ddc74e0fe7874f82e00042fa0ea00855645ed0c79598 \\\n    --hash=sha256:aea996a6aba25260827c9ea511d1addfde2da9eb686ac961838509086188b7e6 \\\n    --hash=sha256:b39b69b347e5e47a3b5b8cfc005c68c1ba347474e3960236c4944a8ecd174962 \\\n    --hash=sha256:b54e7e13267d49ffbfe68e25b3cbd774dab38fa37238f71265e91b36146eb21c \\\n    --hash=sha256:b9af956078716df40d985fb0dfeb2c2120c5ca92ba4ff4b388acfd01cdc14d08 \\\n    --hash=sha256:ba2f37ee79e6338845261a3c5b1784e5d1acdff2c0785b284f1b633033d136ab \\\n    --hash=sha256:ba501e667c17d8411f98e67a022d9604ef179aff0e459b7e292c796837c13573 \\\n    --hash=sha256:baf3775a2635e5a11fbd5e4e64ee69c7e86875d224a5c72aca4c141064589a90 \\\n    --hash=sha256:bb57753e36e4855b8ca375069482250a6246372331a3e4f3407eaebb007443f5 \\\n    --hash=sha256:bd6c173f04743d483881bffa1478d5a4624475b8cd1d2194956a75548e191c18 \\\n    --hash=sha256:be47f99644b208bff7766314013f9acf57b056b04191d570d68ad14022cf5b1d \\\n    --hash=sha256:c010f5581d9c612804cc59fcf7b524b707fbcb72828551237ab545bb5c7034af \\\n    --hash=sha256:c1dcc36dcb96abc02236e182d17e0f71430152a6c2c7447421da2d2dc144edea \\\n    --hash=sha256:c428c6c31eb5f4277d7f8eccaf767fbd548ddd5ce3c8b4f4cbbfab3d96b5904c \\\n    --hash=sha256:c658c50ac0c98cd755a2dd50b7977d3bca7df401dcc47fbdfa87db53ef7d4e8b \\\n    --hash=sha256:c71fb0d56c920c269cd3e2e3fe7c610e3f1fdb21a6ce60efa6430ff63676cea6 \\\n    --hash=sha256:c7b742bf31c88566b4bb6335a7f393bb322e580b6bb98df7bd0c25e6e3519ce8 \\\n    --hash=sha256:cc0329df4caaceb950d2f580b5ac716a377f7059624a0bafaeaf8a218c6ed774 \\\n    --hash=sha256:cc5d36d96478aa9c60654bd932525bf32964c62a7281eafdf16d85003a8d6004 \\\n    --hash=sha256:ce854f5f478050ade5a238731c4ca985a7d3b3cb53ff600a9b5c3b689b5f0a7a \\\n    --hash=sha256:ced3fdd71aaa83ce593746c2edb42b7a59cb4c19c8b5c407781c72e493aae55a \\\n    --hash=sha256:cee5dd7c6fb5dd52a0fe2a740f9bc6e3593f5f8b1788bde49de02086f30182b2 \\\n    --hash=sha256:cfa1c0cc3a8f9f53f1243a5a99ac36fd003880199383b37672e86ddda9cb07e2 \\\n    --hash=sha256:d1ee1e296209fdce05b81b663250eefa02213a2da7b41bf26f7829b8ba3545aa \\\n    --hash=sha256:d59b75732e9b6f27388e10c14b0259cc5f2e48c78627d185e6a177b58ad3cffe \\\n    --hash=sha256:d63600d620ad0064c3a748b950ac5ea38a80190e5498532efefa4b7b3f1da1f3 \\\n    --hash=sha256:dd732602a7009217f658d5863d12d79d373a4de0eebc111094bcdd3bb8e0a6cc \\\n    --hash=sha256:e06efa066f7dbadbc84ebc126a97c452a6451dfcf589d89d788484949e1cf795 \\\n    --hash=sha256:e199fb99720074809a7720f1c0b4d919eea8b87e88713e0f8f602f7bef543d9d \\\n    --hash=sha256:e4b018dc5a0eee4676e38fe84a47a427816c590b93b55d9025274ec4d6ffc2dc \\\n    --hash=sha256:e6621fb2a4988d6e53eedc455e5903e2679f3967b8acb3d639f1b63c14a2e893 \\\n    --hash=sha256:e71c909f353863b2b89c83de2ebed71ea6d0df8a6ef65a128193c5e650766bef \\\n    --hash=sha256:e90251c0c7bdd54a100a0dce3c07b7e637278c93af29dbf78ebb89a58c4bac7d \\\n    --hash=sha256:e9fbdce1e47394b09bc9f26ab117dfc8d6491977a11d86f592bb42c779db2fda \\\n    --hash=sha256:eb12fb2ba69ffa05f8695f61c69e591dc4b4a12ac3757ac8af8adb259bf56d17 \\\n    --hash=sha256:eda059b6bc8bc0812d626fd91a7ce01bf583df0a61296eff390fd94141a34e30 \\\n    --hash=sha256:f03ac127268b43ef4fe9e6ab6794a6794b49485a0cc0c1db79876d2f33f75bc7 \\\n    --hash=sha256:f298e218441525d3794428b4c8b8fb8662c6d3ea79925d4807ee6b9a96a3bca5 \\\n    --hash=sha256:f5542f9b941279d82d41eb0aa9f98eba36fe4df5c7086c651df7944935b37182 \\\n    --hash=sha256:f6f7deae3feb4edfa2efaf7c574fe88cbf055038a6abdb40188e4fff66d5699f \\\n    --hash=sha256:f9b1e28d0e8dbfa858abdba91d6b547beaf2df1a59bec6da6faae7b96a4991a9 \\\n    --hash=sha256:f9f8405c2c758532c74fed975dbee57be1f31a6e865c031870c79a6ed3212ada \\\n    --hash=sha256:fa48b1b63d639f9483e0633e092f5851e2348c352f1f9bb6c8182f87884ef876 \\\n    --hash=sha256:fb78f6e7fcd8ad785d28cd577168bc1aaee827b25bb8755638f694794ea98f0a \\\n    --hash=sha256:fbc597639158fd7c14d55e808718848319540f51b0e6746e3eefa59723a4a348 \\\n    --hash=sha256:fce8cbd4997efeb450bd298b54f755dcdff18d496f7a5ddbb4867c6d7c88fdc3 \\\n    --hash=sha256:fd0350afdc3aabd5576f60ea109228bd5538139713c7b094c5cd27c73a98bc6f \\\n    --hash=sha256:fd0a274c0e5f9a21565cd9d3dd749b61f96b7aa1e20a93aa1ba4029518f2e5c0 \\\n    --hash=sha256:fdb8a068947befafba9952162645dc2fecaeb400e64584829ed5e9b2fbe21a7f\n    # via requests\ncontourpy==1.4.0 \\\n    --hash=sha256:018227e134b73090b06f911e773a526c77c9af426be29065500ce096a5accd4c \\\n    --hash=sha256:072a702e2e178f4fcf96f04775d0c059e4c917925abf0d3208ebb6865df3c23d \\\n    --hash=sha256:0c7a4c2716a4e98342221954416a836cca77c996c14ddbf22b5f01d5d93ca09c \\\n    --hash=sha256:107ec46f7aa1664266d69b1181aadc953d58d39f7d8d648fa5b795d4a060b0de \\\n    --hash=sha256:181bea01bc742734ae672fa00c717d855dd35e1f029536406538c52e7b0cd73d \\\n    --hash=sha256:186ba929df36d61b6127da2e89cd1357e4cb79aca647381ab1a6feb0b152877b \\\n    --hash=sha256:196759a3e4db60e1546167e361090ebb6d199aa3aafc9e34ea17a5ac3b23e814 \\\n    --hash=sha256:20156f5a1ac4f8ce02656e39a61e82164a3d359796dc8026f75b062783d500e1 \\\n    --hash=sha256:20da2f86bfaed60dbba729fe738c8241b1dd22b6cf7e8cbbf30df290bd04ba28 \\\n    --hash=sha256:2deb580178ca19437a84bd77e4bc2cd91a8ccad212413a83c273900868b5978c \\\n    --hash=sha256:2e741a39dfc96babe1722561e81351aeec104526c57dbe91c167c1db606c2d55 \\\n    --hash=sha256:3863ef2e2b13fe93f8c0ebb08ee400cb07153b8b0e91c5acb26e4537f283634f \\\n    --hash=sha256:3e0a2392533e5e2abdf277c951047516acc8d2f3c42b2dd9dc34c907c9ebabf2 \\\n    --hash=sha256:404dbcd9233513dfd1323b66ea593fef90e899f658f6b8a9ed8e93bd0ca669db \\\n    --hash=sha256:417be048f7e122cefbe2a34c51a1f2b0410f8eb35796f794d45475ddb7872d9d \\\n    --hash=sha256:423bd5f4b3f11d54a8c597234e513382e3454bb106b8ec7ae32ba4ed63f8d99b \\\n    --hash=sha256:432e89f835cf01f8d2123a130a3126b17ee6e0348e4d62c0973872b30aa873bf \\\n    --hash=sha256:43c3ccbb32c6294b183dcc8e8c46dacf5ecef809497e3d48a5be298eef185ad0 \\\n    --hash=sha256:43d80072a32299bf945de0a6dd4e034ea0162e832261de2e07a274a5adbba9c9 \\\n    --hash=sha256:4bffcb2e8cc5c0e837631acab59dd04265fdac376550e7c3e310523877c68cb7 \\\n    --hash=sha256:510f7d93d94cf6ebf4e2cd0640bea16ab8affac7547230175b6745b596ce0599 \\\n    --hash=sha256:5450f091ac1be0be3ad3a2a3b3f23b5e443e78c670ced4fd347d626f92a28fd2 \\\n    --hash=sha256:5545ff38229c15d11d822fc6f7415b932da9f4975f53e305b7b96ebfb01f889c \\\n    --hash=sha256:5a4c89c7f38a0d7a94356d74e3391073c4325c4d47ad2fd065c3fe7dbae16c0c \\\n    --hash=sha256:61624f6722480aa7e168fd746164e7bfdf48f8ccb6542f1613741200dc37e2b1 \\\n    --hash=sha256:618137ec5778fb76d494c9ec854b104a377fddba0d128bef7136cce2c3bf0df8 \\\n    --hash=sha256:630d32f06cedf37b7f1dd2b6a12e4492826bd477a4c9c7b2e2fee6ce3a2cea76 \\\n    --hash=sha256:64039341e2d8804f1a13bda8c69083eab935167bbd7e856cf096241f17d5bd45 \\\n    --hash=sha256:6507a016976a75a15aee47809a34605de6244ccc056e6164bf9fb14b27eecad4 \\\n    --hash=sha256:6ced1670fafee703b8277ab1645072a226e74f167166eae146fb743916119b67 \\\n    --hash=sha256:6e697d94e69f499ff6bebb899cae97a58d5d14f0e1fe9568b43a0248d2f9af8c \\\n    --hash=sha256:6f963ea9f0d68d2b6a02d861e3c0aae09ae25ba21f243fcf281e9fed468b078d \\\n    --hash=sha256:738c44fa71735a617f36da58e32810512407d283d5d1bb5b1cecb00d7eeba7cf \\\n    --hash=sha256:758e7496cec3195fc28a28945eab3ceef348c47fb95c7afc2eca2f3901209c8c \\\n    --hash=sha256:78ed5c5f962b3e156109f0531b174a65ba80d13cc681a70227b12524c162e184 \\\n    --hash=sha256:79b06c60d5e569ce12c5da8f0c51217cdc484386e2349fa717c30fafb56d2871 \\\n    --hash=sha256:7f861af508fca2384eef392bc1d8377cfa349b2b854cb11b45d89c75780e2561 \\\n    --hash=sha256:80c7fc8e7ac217777ce7ba2db3cf8478652b5a4b335283111012ff9eaac9d842 \\\n    --hash=sha256:86d05cec773c9507a3950122e0e40ce77c23c75ceeb2fc189514e71893cbb34b \\\n    --hash=sha256:875f42444c9cf48d56f724f2637e60d0f73b3b12c9041e1484580a233edf9591 \\\n    --hash=sha256:90feb8da006803a95ba573fddc9536357d0c3faddc3a4c9e22816ca49af52878 \\\n    --hash=sha256:912c6afaa106e2f74ba22b30416b77ef7eb94e3bdc5a4df51ab147845dba9b6d \\\n    --hash=sha256:96019f059bcb3774acc0104be3360a57b36d33eaaa2e1d7f77c800ca8e07618b \\\n    --hash=sha256:9c0e07c691f3b3321913ed9b8161c50ea5f77fadd006f52f5e755359b0dcbfc5 \\\n    --hash=sha256:a03b32c2e7eda8b17757c022162c13c86f5c3d6f937dddf9ba3c3ff7b513953b \\\n    --hash=sha256:a1c8a74744fa746eeaa12f0f9ed7f8b4add9d8f1b14d95e3b5e133675eac888f \\\n    --hash=sha256:a3e67bc1a6a4618a1dac7c3053a9ffece5ddfa2046b2670b342cf430bb0b87d1 \\\n    --hash=sha256:a58b5f130afec61a093d743cdda435cc4047748b6791627e34a68f41ec3a9b07 \\\n    --hash=sha256:a5e2bb871b90cd7a52bdee63bf80ef7acb46398d27efc66d7679240016efc5aa \\\n    --hash=sha256:abd0f7e51ecc52bf8c95713bd80b1c17c516a71fa391c54475308a4377903c2b \\\n    --hash=sha256:b7794ea07cab575633daad8d6e963b662053391022294d3aacc1d9ba9ef54114 \\\n    --hash=sha256:b9dc493e7924e5aa32d89d0b8e50986280f3d4b284968a792787bd7bcffd3cad \\\n    --hash=sha256:be85d160e7c795113c2a93254dde2fc7e86b375322570eee78b420969426973c \\\n    --hash=sha256:bee6ff96653e0807cd0f8eaef00174a031e3a2effa5c20f49a5a8d574b05af23 \\\n    --hash=sha256:c76f3a5318164db7d9401132fc1b5364b784c613c93fa506e3b5e6d1bf353ec8 \\\n    --hash=sha256:c8be3392b84cf43373a488874dc1ebf46f60784a70977f92586e18019d71c7ed \\\n    --hash=sha256:c927ec747633e68c9920bcaad48b0bcefe648812b822f18c34e2d0e2270a3d2a \\\n    --hash=sha256:cc87f2ffa84a49a77d76cb792f28a48ff3b05a222aa0b2bddbd839ba78e1d5ca \\\n    --hash=sha256:dd59df9e2fb0aff8bd7fd1adb9f349b7c835245a9d0f18c2f2deeb537190f2b1 \\\n    --hash=sha256:ddf5a2596d716fd3793434844caf31abc7a40b1e8718431420c89858268fb909 \\\n    --hash=sha256:de503609fdb71f597634ca64fb4bd2f13d27c97f140e912b1e8ed93544840df0 \\\n    --hash=sha256:e21d1d4a9db5b3a793649c7ab0ddb4697f2818929eec871a7b9c45fce7b2a1ea \\\n    --hash=sha256:e439ab450c93455feb0218532ded3e946ec7a9b5f459069f122cfa44b89229f6 \\\n    --hash=sha256:ea09dc704029930cbd097f75cb0cc1db9852adbcf1a151fac7e9559f7bcbfd19 \\\n    --hash=sha256:ef47fef9a912c77e3d84b014f701e5a9340f25703e9ed3bcebe37f91fb69dc49 \\\n    --hash=sha256:ef9440f6f8506246269a82734f5ff9e2e4c5e775b3996fc883cc491c5257eca6 \\\n    --hash=sha256:f1219a8898523cba821085f2da8a1b962cc696325763f09d7f93538ba43b2d70 \\\n    --hash=sha256:f863c6100bf926cf47d13f3cd75f9bb8ebd98aaab230eeb4468b91f98f39a6b3 \\\n    --hash=sha256:fa1b787362a3856e63dd2b89449f6c384d55ee1beb8f94f4bcc871b40f02462c \\\n    --hash=sha256:fc9feef8f1f001c5b87decadc67c4a5d1eebb62ca39c4763d1237ff62cf2b707\n    # via matplotlib\ncuda-bindings==13.4.3 \\\n    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \\\n    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \\\n    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \\\n    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \\\n    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \\\n    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \\\n    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \\\n    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \\\n    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \\\n    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \\\n    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \\\n    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \\\n    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \\\n    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \\\n    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \\\n    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \\\n    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \\\n    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \\\n    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \\\n    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \\\n    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \\\n    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \\\n    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368\n    # via torch\ncuda-pathfinder==1.8.2 \\\n    --hash=sha256:4e65059febdb4d19d5cbc4798677e19db2b582f2f702f457b609e571690d357e\n    # via cuda-bindings\ncuda-toolkit==13.0.2 \\\n    --hash=sha256:b198824cf2f54003f50d64ada3a0f184b42ca0846c1c94192fa269ecd97a66eb\n    # via torch\ncycler==0.12.1 \\\n    --hash=sha256:85cef7cff222d8644161529808465972e51340599459b8ac3ccbac5a854e0d30 \\\n    --hash=sha256:88bb128f02ba341da8ef447245a9e138fae777f6a23943da4540077d3601eb1c\n    # via matplotlib\nfilelock==4.0.4 \\\n    --hash=sha256:0df72be195ca7892216d16f2edce8d9b93a571f02402972020a8cff84c594c7b \\\n    --hash=sha256:90999ed63a26ccf86b93b959ab10cf1017f422d816be454ed54cbed263e71ab5\n    # via\n    #   huggingface-hub\n    #   torch\n    #   transformers\nfonttools==4.66.0 \\\n    --hash=sha256:02117d05dddb51e39b5c5a6eed0702ed8e0e1c341cb138101b2446eb479858f8 \\\n    --hash=sha256:03922992966a1830b94a750961d1ccfe1a7375d792ca61077a5afb719811788a \\\n    --hash=sha256:056641fcedeaad24b92e343e35e9fd1337200c102450678894b17928482d010d \\\n    --hash=sha256:0640e69b00e089d6b82e0a54e3a03af84b775f1f0b2cbecc1b0d7fe3980a194a \\\n    --hash=sha256:0cf33d1041364ebb7a9958db757e213c6d2a238eb41755fbf935db028c2db20f \\\n    --hash=sha256:107aae7c38f05561fef5103baa98a44400b5f4c69d87839c9a5f8125405e7d9a \\\n    --hash=sha256:135ce8738a6341fbefeedc061e20c2889350e84405abea7e61b1c9d87e1a1f2f \\\n    --hash=sha256:15315302c620e6ce8582934e07932dcf442fc3f59ab208d30564200b9a2f7ea4 \\\n    --hash=sha256:1d120ea0f5260b04e9b5ac0d9239efde4c23d990347d5d5889cdd37221726fec \\\n    --hash=sha256:221defad3b1949c1fdde2558d1fe780d42f926b683a71a46a4495a0e6a6614bd \\\n    --hash=sha256:2b278e2abe596872b3c9fe611f956ded85588537974e48e2adbe2d6ee159e099 \\\n    --hash=sha256:35684b562df7154d7a0ebfa512db9199c7fb0a93b5abfb59dac9d022dbee7aaa \\\n    --hash=sha256:379b94fe10651d26caf398eb646b3ecde886a9c504d3cce5e46164d58a78e32d \\\n    --hash=sha256:3889fdbe63e85dbf2a0d9d7c9d90746827ce60af2cdd24f0bb4738380a9540eb \\\n    --hash=sha256:398bfc5ce9055e0b91a6189f8cf462d4230b38f522b2ceaecb4f34cbf4fb5480 \\\n    --hash=sha256:3ea59aaec135c96c4cff20d79fdf1d10623993712a77990585a42cfd6a42bc5c \\\n    --hash=sha256:4513753259f06316bdc2039c9bf3129e74923c1cfad0722b4728a9ce19beeeb2 \\\n    --hash=sha256:4bdaf80590fcccb1b9d8e3b8ca72225defb14d03b79788f59781ce1bb66049b9 \\\n    --hash=sha256:5dda4087a7f8f0b985995489b3c0115bd117f18c0fc236013a0e78e837bb9ac2 \\\n    --hash=sha256:6371e7ed43cc9e23550fe4a1b72af3205275f28dcecac8c09b80a4582e326802 \\\n    --hash=sha256:6627f48130cdbe04c7e1189bff4192886ecda757f575626ee8468f6580eaecf3 \\\n    --hash=sha256:67ea5af3ca60e1e5c9b2841b1f6dba5ef16aab581bf463abb68515447154ab61 \\\n    --hash=sha256:6b11180c4166a4fc353bb02b147322b7c454927bba4f5040ccdef109a73f9051 \\\n    --hash=sha256:700c448f190cc7bbd9f059bd96131f4770e90938c9692550b142153bea73b14c \\\n    --hash=sha256:8009e736e1491c569392b2e9782c30dcb6ac879def102c8c2e06add8d2adc10f \\\n    --hash=sha256:850d50b347f31667d3a277a6e66a4219236db9d98afadb2993b8addf9bd4f593 \\\n    --hash=sha256:85e2960a29e6882567b981ba408b5a5ca2d8865b8f67240e1b420348997a13be \\\n    --hash=sha256:8739d77810aeb6d2ecd55cc2e4864d98a00c317fb6131a6d22ca58b3f50cf22a \\\n    --hash=sha256:886f3029ea10362c592d2e1a5326a44645b85a2a0be6fdb7c9edbdb60cedb0ab \\\n    --hash=sha256:8af85dfc2ad564f95b3e650024578bb2268bc4d827d418aab9650bbde4b6926a \\\n    --hash=sha256:8ea6265535b25f8254868f5a212a1c9dbd88c2c4050f3df18d9174949df0d6d9 \\\n    --hash=sha256:9b0198962bbba64a88bf15dc5d95e4bab8c5208dede22a10fb06280406ed9874 \\\n    --hash=sha256:a5a0624aaefb0a29e4ed7500a98732a1b0ec2d3040fe86cc8837f82e3fe4f446 \\\n    --hash=sha256:a9a45a23b020e667499fb2dbce0f1373b1b1b8cf88e47f3031e69ef3ea21f3ee \\\n    --hash=sha256:ada87643b20a8fa5763e6dd4fa0f5e2901cbdbe0833deb6d5c5a9c3186782b94 \\\n    --hash=sha256:aea0cc5609a5f2a2500f91bd6e1989fdd22da0f225be00dc1c4cec775838a7d0 \\\n    --hash=sha256:b02686b2e47115b2378ab6156abd984c776c6faa4f56898fbcc1c30ee387937d \\\n    --hash=sha256:b432b6b54a0f3be118c6243312c826aa086aa9bcb150160be4f0bc8a21ea3aaa \\\n    --hash=sha256:b46500e4d6f5708a9127ea12902ba54cb0ec594fc6bec649afc53154a1125b2b \\\n    --hash=sha256:ba65eb3e2c46ad0922d84ef78e287c2f48b7a8cae4420728bf3b8e583282d6d7 \\\n    --hash=sha256:bc7b7ddc1a1f46c363354304e9a8dd93722e4a6a24f785015650898dacf40df9 \\\n    --hash=sha256:c144e68572af1dfcb05fe065827d0920d4118b265504928e1d2262e3701a25a1 \\\n    --hash=sha256:c3a66c749b69e9abd92e519ccbdea7210522b0fffdba3f6492a8fab461d4708a \\\n    --hash=sha256:c519433e8632284dbe64ee4c81b609a17e205bc1afff285553723964880d1c89 \\\n    --hash=sha256:ce818581070527883e3678b92b1ac7ed3569e48c80f51b89b264f97d84be2608 \\\n    --hash=sha256:cf0033f343cb1592fb24ae3eaf8b58156f208884ac16991094293ee94c585654 \\\n    --hash=sha256:cffa168522a0e057d3a5628c8ebd43ddf66a34613f42e116bffb85b0b32ce5a0 \\\n    --hash=sha256:d2d10f89c4bf7cd42b84695da617f9130fa594647ee74673bf8c8d001fa04ef9 \\\n    --hash=sha256:dbee639a23c4e067aedfbcf84d2e466a71a1a1156ad04d6d09023791ed167495 \\\n    --hash=sha256:e4677025ee40b3b1420387ca85f55545b8095f7596c3d5d59552dbee19c91f07 \\\n    --hash=sha256:ee4f85d1341af630d787514eef1a0e1ad883f9d6c11a4c1465faeb9cd0bb5ced \\\n    --hash=sha256:ee7b9f6c835ea1a9beb5a35c8eb0c62b6a3290f8105d3d0c29a0a18c90f9b8ba \\\n    --hash=sha256:ef0610dfe7bb5bf574d9bdad6f597403ebc9807d124ac6f148d7604b2609be98 \\\n    --hash=sha256:f014bbf05f30731bf8b9f5c7d2b4c0e4f28f375926f95b187f85e0bbceb16029 \\\n    --hash=sha256:f2032ac005e14c56e774fd19f2bc1c22c796c0c3b39d91741c0e3f9575944d38 \\\n    --hash=sha256:fa995d96ced196388d8ed7a67ee001b181459d230e24bb68b193ae859f5609da \\\n    --hash=sha256:fd3dcb69d65c05a2fffa5d9b743b77e0cdc13832665ba435065493dad2b57f83 \\\n    --hash=sha256:fd9297a5f670f3e59ae697289ddad5902d3c4a86188e83e7823bde06334a560c\n    # via matplotlib\nfsspec==2026.9.0 \\\n    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \\\n    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f\n    # via\n    #   huggingface-hub\n    #   torch\nhf-xet==1.6.0 \\\n    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \\\n    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \\\n    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \\\n    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \\\n    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \\\n    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \\\n    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \\\n    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \\\n    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \\\n    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \\\n    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \\\n    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \\\n    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \\\n    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \\\n    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \\\n    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \\\n    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b\n    # via huggingface-hub\nhuggingface-hub==0.36.2 \\\n    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \\\n    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270\n    # via\n    #   -r tools/sound-workshop-requirements.in\n    #   tokenizers\n    #   transformers\nidna==3.20 \\\n    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \\\n    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c\n    # via requests\njinja2==3.1.6 \\\n    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \\\n    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67\n    # via torch\nkiwisolver==1.5.1 \\\n    --hash=sha256:007a5553dfc4f4e8d184f588a0200e2cd4b63a59cc8796df3c39909e679dc7a0 \\\n    --hash=sha256:0324cd2567259b7a095f6cf18a52b0ffc6f3de9e69528ff1bc0e7a37bd43ff1a \\\n    --hash=sha256:0627b9bceb9c3cdcf12b8a18655eedfed2692b038df27423383c120d0b7dc2d6 \\\n    --hash=sha256:06a6917674de9e0fe3f66f5430787f59a9f2ddb64af9b714eaec547e29ef5c19 \\\n    --hash=sha256:072bdb15a3c19a5b5dbc8f8fb1f4e1884bf4f3507eeb4cc6334401274d37a5c0 \\\n    --hash=sha256:0a4faea5c6db201c6a21391d2ac926ea97acf7dacdbc3c417189e1adb1a00837 \\\n    --hash=sha256:0ba9527afc80ae3d7814ed98b6572d02bf85eaf48065678342c5f0c6dab7a8c7 \\\n    --hash=sha256:0d8924877ce22e17326a99a418c3c82037da078df3c6a260b13eca677444e6e7 \\\n    --hash=sha256:0ebdef3eae5336568147c39a55be6a2036ffde53faa9ca2d978989ae7c2da12c \\\n    --hash=sha256:1209042a623ddfda5497e4066c7b77651dde8e1d3a9dd97599dc7e97f3b9b78c \\\n    --hash=sha256:16895f553ee6620a827d2da56b871f835fb70b9216cca5d188e885caf6e3bd23 \\\n    --hash=sha256:17851e5dad4484be0cbccbde3b15331deae036de9aebd45eed964487802b172f \\\n    --hash=sha256:1798e83840c3f627246104c4d8a9639c60fa068adf9ce92b61791781fa8a68c1 \\\n    --hash=sha256:18170a77ddfecf40ec60d0928268dc95880c881864e015a8f34094ed18b9b9ad \\\n    --hash=sha256:186884a58486651e3c217b6acea0a53eaa9498fdd472057c46f2f0fb5c25aad5 \\\n    --hash=sha256:18a0cfb124546a4c2e6087c5f3029c7f44b37c85b142e0ced71f73a7599ac208 \\\n    --hash=sha256:1983f0974a750a6f6556f368ba11105d1d8369c735b944747c9f12ae5aea7aae \\\n    --hash=sha256:1a7587dc335f2c0f5bd577fd0540bd16c66006bdb60f759a1059f025e6c4f071 \\\n    --hash=sha256:1acc7e5b7ef05e9da8bb70cd6c7c4513090213d2e1ad9720f599f0bf6c52aec5 \\\n    --hash=sha256:1d852545c4d0e35a72728d072cbaa59e2fa7dd84bdf01e068d670dd0ceb58eb6 \\\n    --hash=sha256:1ed0f5e49d0ceff8b72190824d9e59c062fbbc02c231b853112c78474b3f5ec2 \\\n    --hash=sha256:1fff05e239575b1481b6ed1a782f6fad616efbf1f0b1f44e6e85c4dfe426e483 \\\n    --hash=sha256:21e46b23a2da695c364124817bc01d970effd5483147f8d66a6a7167e3f6b851 \\\n    --hash=sha256:22d5e5aaad6be121f2515765e3b1c444352cb8eb4c86510801db8f2e50757316 \\\n    --hash=sha256:2551cf9917af48ee7c4b29cc82320489508cf96fd26a51f6fc124de661cd44c7 \\\n    --hash=sha256:255605693a483db7bd5c79f60437f7bf658f7f520d61aa42722e32257c941951 \\\n    --hash=sha256:26e8268480be5061d509e29669d59103c067a26377a56491630ece11762e3858 \\\n    --hash=sha256:27add358abe374ebaa3b8763ef380bc99051b5a4b18d94878366a9e4f59efef0 \\\n    --hash=sha256:2ae70bc59790d2af72a3f76f24b272403e135070340281108b447cb77ea70819 \\\n    --hash=sha256:2e10ae1bba1899188b33557c10d73affcc12033edd18adddb57d209039976a4c \\\n    --hash=sha256:3221f78211074f561c44ca42eac0619828171bec15a2c4cf6f7747d07df76e8e \\\n    --hash=sha256:34633ecf50d16187ab8e5528b7a2530f2feb4e23f300db4672538b51cfc5cd38 \\\n    --hash=sha256:34ec467940442c9943016fb2d4c81d1ba84351eeca2f1a78f8bc87f1ba0d414c \\\n    --hash=sha256:37f801b5d7cc0e5a548921308e059fd2b057bb42972b591cfa3049f95423c4ed \\\n    --hash=sha256:38f6e0deb4d0a4615efe0c4efc5990b06ae450ab50a0b321c0b078b6d238c083 \\\n    --hash=sha256:3c24cd69455e1b00ddf770c13b6e2c33e07d6dc3f2d34add0bf9277c5c6bbd46 \\\n    --hash=sha256:3cc210010fd2f438a3ed430b45f1b501fd13a8618bf984dc2c5ce5b69b78752e \\\n    --hash=sha256:3fa5855898f6d3d01b72ccd48a2d65cbdee301251603fefe34e2025bddba219c \\\n    --hash=sha256:416ba7ff9f233b7036689bb5a3783537e838ad483f63558d2a800f75afe738b1 \\\n    --hash=sha256:431dc224a1a92a5c8f582d96e505196a3b5997a7271076678da2dfde67b77e9a \\\n    --hash=sha256:43844c1a7ad6d723d5b5b4c4fc7f5bd399c40e288120d16257c7c9e8765c6e85 \\\n    --hash=sha256:44b8faef94f1857e77fa0238f3390ff1ac51d2ea20a487e2e452a59fd2b5f5ca \\\n    --hash=sha256:470d420f98d368d6f010633a20659b544c5fdfa5329e6b70219f2ef08fd4a7ef \\\n    --hash=sha256:482676e5bd48d70ac99d9fc78863469845421e01184fa83f1f9366dc49f7e974 \\\n    --hash=sha256:4d4ca09bf13cff792b1884f64b98ee6c2467930d632233be25c56b442d99f10e \\\n    --hash=sha256:5025e36fb4fb275cef0a4e30dbb11cb4ae61d1c83deb90189cb5d7e4cafd6b55 \\\n    --hash=sha256:509735237ae0d849e8a843551d423d2500d2e0a9ac1611a145658b29c0fb9f85 \\\n    --hash=sha256:534f02c1abb31ed6dbd3515545285c330b2f12d00fdb1fdb71658b9ca5a13a6a \\\n    --hash=sha256:5978c3340f16a35c30f8ab2fa7bcf559973c55f1a5ef6970e1f621acf3c4db13 \\\n    --hash=sha256:5b973887ff782cfd6b67c9904ad8ca542e0bc5e4961503408b423b5a688b4d38 \\\n    --hash=sha256:5c490db2168a508088f59140dd392556a54b8bd1048fc6383c8baff13c359673 \\\n    --hash=sha256:5d142e352eb13facc7dd047489aebdff6ba78576c239f1ea04931979caaf0567 \\\n    --hash=sha256:5daa1f19e097050b9c4d9a78fcc9263cb96c9dfae08037ddc1b7c4ad1889f2a2 \\\n    --hash=sha256:61e9a64c7635095a6bfe483e2ff055d437c59bd45f3617a228b37277f0185d62 \\\n    --hash=sha256:63fb7294b768f444eb4b068965f2662f28c2fd4161e23bd60fcf3ff27b74c046 \\\n    --hash=sha256:685929988b208a911f1285e2f8ed54210b0d681a3dc0f03e00d599d291986e7e \\\n    --hash=sha256:6a797a1cefc8b9c93170db580337e1fe3d011ad18b1299943231279406342048 \\\n    --hash=sha256:6b92f60017dda7d877fdc546438b5e28f31c523264f49cf5a48c1d0ce1a0dfbc \\\n    --hash=sha256:70ed9a45c7484d2b30cdacf60d220f494a1763b9fec1ad03285c6553fa0889f2 \\\n    --hash=sha256:719a35fa1156db3640555f95ebb94f60a444e64d1c69626b0edef5df78eba225 \\\n    --hash=sha256:74ad5c3dad54a4641b4c28cd15ded70899d04459c6c7aeacafea716be97cce6d \\\n    --hash=sha256:74ea337e0ec3f6f342a36a4f1b5cd94dd9affddcd28ba9aae2905af932ee8c6b \\\n    --hash=sha256:75d9b1cf8258462dbdc1eeda718c96ea7f079324c09067f6daabfcf37712b7fe \\\n    --hash=sha256:77a4c8187a5948d7f8795adb765a3c7b553d07d86d88e43038fc32fc1fb9a3f3 \\\n    --hash=sha256:7824b5e8bdbf0bccb4ccd37bbb115849a1dc45437fb4de8351385ed07c437ee0 \\\n    --hash=sha256:7d38b0c279c3032e8c9cc013b405c6df8e1668dbf15465779aa7f15f61201812 \\\n    --hash=sha256:7e9c01d3dd7ceba4d1d436cc021d40d592466e40b9bc7f5d83dc4e98a5c9cd8c \\\n    --hash=sha256:7fd82debf43c6acd0a94359d232f6bb516ee13f269a7993736a9ac9f988bb5d9 \\\n    --hash=sha256:824c3d763a05ea9e9003610145186b0e9848c7584a5575c79bac5a8e7cd80bad \\\n    --hash=sha256:828f75af2b0080c8a972e75f649ab46af008e92c6104a57a759157200b835b75 \\\n    --hash=sha256:83f78128fa28705fa85d01c59771c72fe81c11bd0e6155edbb9f818983a7d761 \\\n    --hash=sha256:876bbfd276473d3daffe30e8c975df4ed9429967b41a6cb362dbb5155b6f13ad \\\n    --hash=sha256:886fc26012f0e8b5f69d1cfe6d711f6b11f194621539bf8e6bb1c25c5dc82724 \\\n    --hash=sha256:8a34616dc2521cc8dc1d7d081734da63539f021ac0450ce950908340c6e7aa2f \\\n    --hash=sha256:8a708a47ade1fe19e8371d5da076bac0dd4b0a5a7985ad6c637f7f7e361b6baa \\\n    --hash=sha256:8af9b142ad719ae3a911ebf616bc4b78b32bbab84d6a40d3ad2f129670509957 \\\n    --hash=sha256:8bf4df63592c2a66b4f8edc5df2544998c288aa02f96ce0acd880cd1de8c8127 \\\n    --hash=sha256:8de6f2a4ce7e7bd27d23dd94abf0ccafe0e0e5cc9c764b0577191f2c25f08f26 \\\n    --hash=sha256:8f8fddb8e323bd6eee4e54e69a39243beab22689070f4c66b472c4cc88bb89d8 \\\n    --hash=sha256:8fca690b00c4c48f6c2a547b0160ed511357093a4e4c9b47e0fadf3128066d89 \\\n    --hash=sha256:9506e892bcc3b409831d363c6f53e5985e1c8d1f6f6b0256d00358684ff85378 \\\n    --hash=sha256:958254518717542d02d0688d0d20cbf771da5e415e6f49543f92481c850a4540 \\\n    --hash=sha256:95a02752aa032eef4aed01cda6d9b687c669bd0396bf4519eef8bba22a286720 \\\n    --hash=sha256:96c30002424670b5e1e46495c2b8cbffef39cf77c1d79e76462029d50339785b \\\n    --hash=sha256:98b208a7cc42c803445ef551d6753cc42a5ea13e9cab1ee66cd8b9cb70195330 \\\n    --hash=sha256:9b3092d8992a1d69b7a59c3e39f35e1b9be327a17f68a7c35fc17329e337d6f2 \\\n    --hash=sha256:9e51c119992ea8820706871c30a4642ec76de20ae82f9b50b9a45517d8e9f810 \\\n    --hash=sha256:a5716a33bfabb2c6ce27b6cf03253467b3804f83e215f4d202685cf93c6c9874 \\\n    --hash=sha256:a5a00665d1a0e26763a7338d7e911d4598fbc1d50dd0d6b7919b7dc6c5d6569f \\\n    --hash=sha256:a5ca5aebae78a0bc13c1943af4af615d4966c5b650b05d5aa83b50e427196fee \\\n    --hash=sha256:a7b85b2cc6ea45e5f7e8c9a30bc9fabd47cda09106cbb4b967335c3e6c43b69d \\\n    --hash=sha256:a83ee7107df13abe42a54a6654670eef9bb39425cf2e27f65e0007465e1286ab \\\n    --hash=sha256:aa7d00b1700966d2917e54d278aba86897890ca9276dd8b76cf6446b6c181b92 \\\n    --hash=sha256:ab620eb663952455271ac37f9aaad86b73c969c02f11f53cea405b38e96a4300 \\\n    --hash=sha256:ad8b9671348d7c8716715652ae11f85ed0eb99e265a2df2ca490577d69860b2c \\\n    --hash=sha256:aefe930d113798330e9462f7874542977869c0613cba3262e2de3a8d5dee8f3a \\\n    --hash=sha256:b03af77d77e50edba2030fd5f7c352ff209314b09030a3cba7c14edf9a09a444 \\\n    --hash=sha256:b390aec180a7c054919c04898835e1c77bced23ea8383eb2c570213bf25d1a86 \\\n    --hash=sha256:b3d78f7bb2b9d9a30345be1474b9aaa8685430b54afb51ba3639b5c6c11e9ed6 \\\n    --hash=sha256:b5664603a253efd3a75716d793d1d3a6a82723b61dc6db767b2460bbbeec4c0f \\\n    --hash=sha256:b69602970994a2ed8bbfa78c2f0394a7435226c6040489702d9f0a0ad0c07052 \\\n    --hash=sha256:b6ae6a0328f0bc035741820fdeecdcd67bf4694eee03972e843663107122f450 \\\n    --hash=sha256:bad20d4c69c851c982a1e3606f4c293edfd5a87885786c50082412240c4b1ffd \\\n    --hash=sha256:bb7c99f0673c03017a3ee01e54a5c2617a05468b11eabe513b0080e063ed95b1 \\\n    --hash=sha256:bebb89489b279b2f5661bbbb2abcc87bcd4a46607bb4a5c966f04f1db6b8df9a \\\n    --hash=sha256:bfd1de989b3330420e29de39352f5c049905c9e3ee67233a50d550e3d652c148 \\\n    --hash=sha256:c2306e8bb53601979fcb3fa09cc65e031876d9ae01eff2fcbcd7a84ef94d5bc1 \\\n    --hash=sha256:c3a4e41e3096bf1f0f1b76e2ffd6d828d6547f574f702d59bdbef7acfa59db9c \\\n    --hash=sha256:c6834b92dd2428e2dd85ef3d85f723d3c12f20aaf43a2ddd4f944ca25d833408 \\\n    --hash=sha256:c90d3022d8a94778939cda8638c6c8da8fa757b8958dad7ec868ce29c87681b8 \\\n    --hash=sha256:ca307d6c259e5c98d3cb9ade55342b47a6839762caf2536f3d7b46ee660cc82e \\\n    --hash=sha256:ca7f6fe0f37ca978a1e5eb7a3a68e6413f417e78e838324947ffd420202b198b \\\n    --hash=sha256:cb6fae641357ed2f6e533c0d3c6504a4a5703621a50c89459e46051d56b61140 \\\n    --hash=sha256:cdaeeb6c350106df6bf9d873395973e5f066a9713200b72cd64f55d0a3eafab6 \\\n    --hash=sha256:cea20da04494e662b83c872683bf4ff2345206043d036315ed0e924b652e7294 \\\n    --hash=sha256:cea90547bfd93807e0013a004dc76552be44fad3bc1cc2b38610a9e889ed098f \\\n    --hash=sha256:d09037ca068d784ebc4aec290ef952ca27ac15dd9c0b5801a88c6e1096b83e6b \\\n    --hash=sha256:d27c2123977cb9269c30a49ba45f03a4323017ef693e19db4ec9dbe1299a3002 \\\n    --hash=sha256:d50de98e8d807dc31822fff96f50293163a62418eb65487a21b42713d72ed0b7 \\\n    --hash=sha256:d66a64dd5dec136040ec2ae94aa026a912ee60fdd45bc28d3db30037fd809e88 \\\n    --hash=sha256:d79308fa689fac89cbcfbd4dbfc80b5f95c54c5a7fd4d194be221f9d33d026e6 \\\n    --hash=sha256:da3275833be0edbaf4830fae08bae3dc7219f40ce0c37eaa6c25825957e06612 \\\n    --hash=sha256:dc1a26b8e53395a01c2c611e58602fa47461f136fba7cd5542e6db6d64be1839 \\\n    --hash=sha256:dc23390afe9f4ef9ac3bcc72a03a56eebbde03f4c571a32cb38f859cff9a6524 \\\n    --hash=sha256:e05c2f7925f1d88778e53cb44f14e0223204a3bdd09a41664750363acfb1f2ef \\\n    --hash=sha256:e12dfea7f5fc2a34a9080efbf79c4c44eb380ec5b9c6fea09407e08f0d1e941d \\\n    --hash=sha256:e4e4523d6f336708d732516e6cfca7796cf3d96c9474eb5aecf6165f2f1fefc3 \\\n    --hash=sha256:e4e49f7e1a4e7191bdf9dc67a974db714501b1fc52c24324103d06a86abd5c08 \\\n    --hash=sha256:e68e151428b5384f766cd25739bf77c7e4a3dc93b5ded7a12118d9fbfdf78ab6 \\\n    --hash=sha256:e8e4d953faaded9ec7ede36824e9814082d22d4c7b1eafbfa079ecba8cd0d076 \\\n    --hash=sha256:ee9df1f0d77b9c6e94f4ac0fec533fbddd5ea3a327807f18d7b069ae019ded80 \\\n    --hash=sha256:f0a887b6565bbfe80efde2b7f6e8890d7d9bbdb11bdb17028a3690c32fe0621f \\\n    --hash=sha256:f0f4a42db92d6ec7677ab9d12830a2a8ec145a9c6d15db2b593466bc875c78d7 \\\n    --hash=sha256:f1303ef2eec81262a4b708c3e858afe58d7c75ad91c1c05266eda7673369859a \\\n    --hash=sha256:f1d56ec54d257d05e0b50f5780d967540cd07beeaf9e5f645b26d50cce79f4d8 \\\n    --hash=sha256:f4167e87b397f273dc2356fcf1eaf50a6bac51e6105f45103ef7129c8efb0255 \\\n    --hash=sha256:f76fc85bd054c806960f917ec0f329e24e436f1712267d90588e4c39890caa63 \\\n    --hash=sha256:f942903fde7363d1d879057ec5de01310efda2597161784d752fa9953a01a71a \\\n    --hash=sha256:f9b1c4900736e489a812c529100de4b8fb617d4db075e931e213c57424b83d9b \\\n    --hash=sha256:fc271a6f0a2126958f4090e5507b9da5848927dae331f8f763bd4aa642b3d2cd \\\n    --hash=sha256:febcce10f2bcdbb80b4ea919238a6a4ac13dbc4c7cadbe8d5d75c3682f8b5404\n    # via matplotlib\nmarkupsafe==3.0.3 \\\n    --hash=sha256:0303439a41979d9e74d18ff5e2dd8c43ed6c6001fd40e5bf2e43f7bd9bbc523f \\\n    --hash=sha256:068f375c472b3e7acbe2d5318dea141359e6900156b5b2ba06a30b169086b91a \\\n    --hash=sha256:0bf2a864d67e76e5c9a34dc26ec616a66b9888e25e7b9460e1c76d3293bd9dbf \\\n    --hash=sha256:0db14f5dafddbb6d9208827849fad01f1a2609380add406671a26386cdf15a19 \\\n    --hash=sha256:0eb9ff8191e8498cca014656ae6b8d61f39da5f95b488805da4bb029cccbfbaf \\\n    --hash=sha256:0f4b68347f8c5eab4a13419215bdfd7f8c9b19f2b25520968adfad23eb0ce60c \\\n    --hash=sha256:1085e7fbddd3be5f89cc898938f42c0b3c711fdcb37d75221de2666af647c175 \\\n    --hash=sha256:116bb52f642a37c115f517494ea5feb03889e04df47eeff5b130b1808ce7c219 \\\n    --hash=sha256:12c63dfb4a98206f045aa9563db46507995f7ef6d83b2f68eda65c307c6829eb \\\n    --hash=sha256:133a43e73a802c5562be9bbcd03d090aa5a1fe899db609c29e8c8d815c5f6de6 \\\n    --hash=sha256:1353ef0c1b138e1907ae78e2f6c63ff67501122006b0f9abad68fda5f4ffc6ab \\\n    --hash=sha256:15d939a21d546304880945ca1ecb8a039db6b4dc49b2c5a400387cdae6a62e26 \\\n    --hash=sha256:177b5253b2834fe3678cb4a5f0059808258584c559193998be2601324fdeafb1 \\\n    --hash=sha256:1872df69a4de6aead3491198eaf13810b565bdbeec3ae2dc8780f14458ec73ce \\\n    --hash=sha256:1b4b79e8ebf6b55351f0d91fe80f893b4743f104bff22e90697db1590e47a218 \\\n    --hash=sha256:1b52b4fb9df4eb9ae465f8d0c228a00624de2334f216f178a995ccdcf82c4634 \\\n    --hash=sha256:1ba88449deb3de88bd40044603fafffb7bc2b055d626a330323a9ed736661695 \\\n    --hash=sha256:1cc7ea17a6824959616c525620e387f6dd30fec8cb44f649e31712db02123dad \\\n    --hash=sha256:218551f6df4868a8d527e3062d0fb968682fe92054e89978594c28e642c43a73 \\\n    --hash=sha256:26a5784ded40c9e318cfc2bdb30fe164bdb8665ded9cd64d500a34fb42067b1c \\\n    --hash=sha256:2713baf880df847f2bece4230d4d094280f4e67b1e813eec43b4c0e144a34ffe \\\n    --hash=sha256:2a15a08b17dd94c53a1da0438822d70ebcd13f8c3a95abe3a9ef9f11a94830aa \\\n    --hash=sha256:2f981d352f04553a7171b8e44369f2af4055f888dfb147d55e42d29e29e74559 \\\n    --hash=sha256:32001d6a8fc98c8cb5c947787c5d08b0a50663d139f1305bac5885d98d9b40fa \\\n    --hash=sha256:3524b778fe5cfb3452a09d31e7b5adefeea8c5be1d43c4f810ba09f2ceb29d37 \\\n    --hash=sha256:3537e01efc9d4dccdf77221fb1cb3b8e1a38d5428920e0657ce299b20324d758 \\\n    --hash=sha256:35add3b638a5d900e807944a078b51922212fb3dedb01633a8defc4b01a3c85f \\\n    --hash=sha256:38664109c14ffc9e7437e86b4dceb442b0096dfe3541d7864d9cbe1da4cf36c8 \\\n    --hash=sha256:3a7e8ae81ae39e62a41ec302f972ba6ae23a5c5396c8e60113e9066ef893da0d \\\n    --hash=sha256:3b562dd9e9ea93f13d53989d23a7e775fdfd1066c33494ff43f5418bc8c58a5c \\\n    --hash=sha256:457a69a9577064c05a97c41f4e65148652db078a3a509039e64d3467b9e7ef97 \\\n    --hash=sha256:4bd4cd07944443f5a265608cc6aab442e4f74dff8088b0dfc8238647b8f6ae9a \\\n    --hash=sha256:4e885a3d1efa2eadc93c894a21770e4bc67899e3543680313b09f139e149ab19 \\\n    --hash=sha256:4faffd047e07c38848ce017e8725090413cd80cbc23d86e55c587bf979e579c9 \\\n    --hash=sha256:509fa21c6deb7a7a273d629cf5ec029bc209d1a51178615ddf718f5918992ab9 \\\n    --hash=sha256:5678211cb9333a6468fb8d8be0305520aa073f50d17f089b5b4b477ea6e67fdc \\\n    --hash=sha256:591ae9f2a647529ca990bc681daebdd52c8791ff06c2bfa05b65163e28102ef2 \\\n    --hash=sha256:5a7d5dc5140555cf21a6fefbdbf8723f06fcd2f63ef108f2854de715e4422cb4 \\\n    --hash=sha256:69c0b73548bc525c8cb9a251cddf1931d1db4d2258e9599c28c07ef3580ef354 \\\n    --hash=sha256:6b5420a1d9450023228968e7e6a9ce57f65d148ab56d2313fcd589eee96a7a50 \\\n    --hash=sha256:722695808f4b6457b320fdc131280796bdceb04ab50fe1795cd540799ebe1698 \\\n    --hash=sha256:729586769a26dbceff69f7a7dbbf59ab6572b99d94576a5592625d5b411576b9 \\\n    --hash=sha256:77f0643abe7495da77fb436f50f8dab76dbc6e5fd25d39589a0f1fe6548bfa2b \\\n    --hash=sha256:795e7751525cae078558e679d646ae45574b47ed6e7771863fcc079a6171a0fc \\\n    --hash=sha256:7be7b61bb172e1ed687f1754f8e7484f1c8019780f6f6b0786e76bb01c2ae115 \\\n    --hash=sha256:7c3fb7d25180895632e5d3148dbdc29ea38ccb7fd210aa27acbd1201a1902c6e \\\n    --hash=sha256:7e68f88e5b8799aa49c85cd116c932a1ac15caaa3f5db09087854d218359e485 \\\n    --hash=sha256:83891d0e9fb81a825d9a6d61e3f07550ca70a076484292a70fde82c4b807286f \\\n    --hash=sha256:8485f406a96febb5140bfeca44a73e3ce5116b2501ac54fe953e488fb1d03b12 \\\n    --hash=sha256:8709b08f4a89aa7586de0aadc8da56180242ee0ada3999749b183aa23df95025 \\\n    --hash=sha256:8f71bc33915be5186016f675cd83a1e08523649b0e33efdb898db577ef5bb009 \\\n    --hash=sha256:915c04ba3851909ce68ccc2b8e2cd691618c4dc4c4232fb7982bca3f41fd8c3d \\\n    --hash=sha256:949b8d66bc381ee8b007cd945914c721d9aba8e27f71959d750a46f7c282b20b \\\n    --hash=sha256:94c6f0bb423f739146aec64595853541634bde58b2135f27f61c1ffd1cd4d16a \\\n    --hash=sha256:9a1abfdc021a164803f4d485104931fb8f8c1efd55bc6b748d2f5774e78b62c5 \\\n    --hash=sha256:9b79b7a16f7fedff2495d684f2b59b0457c3b493778c9eed31111be64d58279f \\\n    --hash=sha256:a320721ab5a1aba0a233739394eb907f8c8da5c98c9181d1161e77a0c8e36f2d \\\n    --hash=sha256:a4afe79fb3de0b7097d81da19090f4df4f8d3a2b3adaa8764138aac2e44f3af1 \\\n    --hash=sha256:ad2cf8aa28b8c020ab2fc8287b0f823d0a7d8630784c31e9ee5edea20f406287 \\\n    --hash=sha256:b8512a91625c9b3da6f127803b166b629725e68af71f8184ae7e7d54686a56d6 \\\n    --hash=sha256:bc51efed119bc9cfdf792cdeaa4d67e8f6fcccab66ed4bfdd6bde3e59bfcbb2f \\\n    --hash=sha256:bdc919ead48f234740ad807933cdf545180bfbe9342c2bb451556db2ed958581 \\\n    --hash=sha256:bdd37121970bfd8be76c5fb069c7751683bdf373db1ed6c010162b2a130248ed \\\n    --hash=sha256:be8813b57049a7dc738189df53d69395eba14fb99345e0a5994914a3864c8a4b \\\n    --hash=sha256:c0c0b3ade1c0b13b936d7970b1d37a57acde9199dc2aecc4c336773e1d86049c \\\n    --hash=sha256:c47a551199eb8eb2121d4f0f15ae0f923d31350ab9280078d1e5f12b249e0026 \\\n    --hash=sha256:c4ffb7ebf07cfe8931028e3e4c85f0357459a3f9f9490886198848f4fa002ec8 \\\n    --hash=sha256:ccfcd093f13f0f0b7fdd0f198b90053bf7b2f02a3927a30e63f3ccc9df56b676 \\\n    --hash=sha256:d2ee202e79d8ed691ceebae8e0486bd9a2cd4794cec4824e1c99b6f5009502f6 \\\n    --hash=sha256:d53197da72cc091b024dd97249dfc7794d6a56530370992a5e1a08983ad9230e \\\n    --hash=sha256:d6dd0be5b5b189d31db7cda48b91d7e0a9795f31430b7f271219ab30f1d3ac9d \\\n    --hash=sha256:d88b440e37a16e651bda4c7c2b930eb586fd15ca7406cb39e211fcff3bf3017d \\\n    --hash=sha256:de8a88e63464af587c950061a5e6a67d3632e36df62b986892331d4620a35c01 \\\n    --hash=sha256:df2449253ef108a379b8b5d6b43f4b1a8e81a061d6537becd5582fba5f9196d7 \\\n    --hash=sha256:e1c1493fb6e50ab01d20a22826e57520f1284df32f2d8601fdd90b6304601419 \\\n    --hash=sha256:e1cf1972137e83c5d4c136c43ced9ac51d0e124706ee1c8aa8532c1287fa8795 \\\n    --hash=sha256:e2103a929dfa2fcaf9bb4e7c091983a49c9ac3b19c9061b6d5427dd7d14d81a1 \\\n    --hash=sha256:e56b7d45a839a697b5eb268c82a71bd8c7f6c94d6fd50c3d577fa39a9f1409f5 \\\n    --hash=sha256:e8afc3f2ccfa24215f8cb28dcf43f0113ac3c37c2f0f0806d8c70e4228c5cf4d \\\n    --hash=sha256:e8fc20152abba6b83724d7ff268c249fa196d8259ff481f3b1476383f8f24e42 \\\n    --hash=sha256:eaa9599de571d72e2daf60164784109f19978b327a3910d3e9de8c97b5b70cfe \\\n    --hash=sha256:ec15a59cf5af7be74194f7ab02d0f59a62bdcf1a537677ce67a2537c9b87fcda \\\n    --hash=sha256:f190daf01f13c72eac4efd5c430a8de82489d9cff23c364c3ea822545032993e \\\n    --hash=sha256:f34c41761022dd093b4b6896d4810782ffbabe30f2d443ff5f083e0cbbb8c737 \\\n    --hash=sha256:f3e98bb3798ead92273dc0e5fd0f31ade220f59a266ffd8a4f6065e0a3ce0523 \\\n    --hash=sha256:f42d0984e947b8adf7dd6dde396e720934d12c506ce84eea8476409563607591 \\\n    --hash=sha256:f71a396b3bf33ecaa1626c255855702aca4d3d9fea5e051b41ac59a9c1c41edc \\\n    --hash=sha256:f9e130248f4462aaa8e2552d547f36ddadbeaa573879158d721bbd33dfe4743a \\\n    --hash=sha256:fed51ac40f757d41b7c48425901843666a6677e3e8eb0abcff09e4ba6e664f50\n    # via jinja2\nmatplotlib==3.10.8 \\\n    --hash=sha256:00270d217d6b20d14b584c521f810d60c5c78406dc289859776550df837dcda7 \\\n    --hash=sha256:0a33deb84c15ede243aead39f77e990469fff93ad1521163305095b77b72ce4a \\\n    --hash=sha256:113bb52413ea508ce954a02c10ffd0d565f9c3bc7f2eddc27dfe1731e71c7b5f \\\n    --hash=sha256:12d90df9183093fcd479f4172ac26b322b1248b15729cb57f42f71f24c7e37a3 \\\n    --hash=sha256:15d30132718972c2c074cd14638c7f4592bd98719e2308bccea40e0538bc0cb5 \\\n    --hash=sha256:18821ace09c763ec93aef5eeff087ee493a24051936d7b9ebcad9662f66501f9 \\\n    --hash=sha256:1ae029229a57cd1e8fe542485f27e7ca7b23aa9e8944ddb4985d0bc444f1eca2 \\\n    --hash=sha256:2299372c19d56bcd35cf05a2738308758d32b9eaed2371898d8f5bd33f084aa3 \\\n    --hash=sha256:238b7ce5717600615c895050239ec955d91f321c209dd110db988500558e70d6 \\\n    --hash=sha256:24d50994d8c5816ddc35411e50a86ab05f575e2530c02752e02538122613371f \\\n    --hash=sha256:25d380fe8b1dc32cf8f0b1b448470a77afb195438bafdf1d858bfb876f3edf7b \\\n    --hash=sha256:2c1998e92cd5999e295a731bcb2911c75f597d937341f3030cc24ef2733d78a8 \\\n    --hash=sha256:2cf5bd12cecf46908f286d7838b2abc6c91cda506c0445b8223a7c19a00df008 \\\n    --hash=sha256:32f8dce744be5569bebe789e46727946041199030db8aeb2954d26013a0eb26b \\\n    --hash=sha256:37b3c1cc42aa184b3f738cfa18c1c1d72fd496d85467a6cf7b807936d39aa656 \\\n    --hash=sha256:3a48a78d2786784cc2413e57397981fb45c79e968d99656706018d6e62e57958 \\\n    --hash=sha256:3ab4aabc72de4ff77b3ec33a6d78a68227bf1123465887f9905ba79184a1cc04 \\\n    --hash=sha256:3c624e43ed56313651bc18a47f838b60d7b8032ed348911c54906b130b20071b \\\n    --hash=sha256:3f2e409836d7f5ac2f1c013110a4d50b9f7edc26328c108915f9075d7d7a91b6 \\\n    --hash=sha256:3f5c3e4da343bba819f0234186b9004faba952cc420fbc522dc4e103c1985908 \\\n    --hash=sha256:41703cc95688f2516b480f7f339d8851a6035f18e100ee6a32bc0b8536a12a9c \\\n    --hash=sha256:495672de149445ec1b772ff2c9ede9b769e3cb4f0d0aa7fa730d7f59e2d4e1c1 \\\n    --hash=sha256:4cf267add95b1c88300d96ca837833d4112756045364f5c734a2276038dae27d \\\n    --hash=sha256:56271f3dac49a88d7fca5060f004d9d22b865f743a12a23b1e937a0be4818ee1 \\\n    --hash=sha256:595ba4d8fe983b88f0eec8c26a241e16d6376fe1979086232f481f8f3f67494c \\\n    --hash=sha256:5f62550b9a30afde8c1c3ae450e5eb547d579dd69b25c2fc7a1c67f934c1717a \\\n    --hash=sha256:646d95230efb9ca614a7a594d4fcacde0ac61d25e37dd51710b36477594963ce \\\n    --hash=sha256:64fcc24778ca0404ce0cb7b6b77ae1f4c7231cdd60e6778f999ee05cbd581b9a \\\n    --hash=sha256:6be43b667360fef5c754dda5d25a32e6307a03c204f3c0fc5468b78fa87b4160 \\\n    --hash=sha256:6da7c2ce169267d0d066adcf63758f0604aa6c3eebf67458930f9d9b79ad1db1 \\\n    --hash=sha256:83d282364ea9f3e52363da262ce32a09dfe241e4080dcedda3c0db059d3c1f11 \\\n    --hash=sha256:9153c3292705be9f9c64498a8872118540c3f4123d1a1c840172edf262c8be4a \\\n    --hash=sha256:99eefd13c0dc3b3c1b4d561c1169e65fe47aab7b8158754d7c084088e2329466 \\\n    --hash=sha256:a0a7f52498f72f13d4a25ea70f35f4cb60642b466cbb0a9be951b5bc3f45a486 \\\n    --hash=sha256:a2b336e2d91a3d7006864e0990c83b216fcdca64b5a6484912902cef87313d78 \\\n    --hash=sha256:a48f2b74020919552ea25d222d5cc6af9ca3f4eb43a93e14d068457f545c2a17 \\\n    --hash=sha256:ad3d9833a64cf48cc4300f2b406c3d0f4f4724a91c0bd5640678a6ba7c102077 \\\n    --hash=sha256:b44d07310e404ba95f8c25aa5536f154c0a8ec473303535949e52eb71d0a1565 \\\n    --hash=sha256:b53285e65d4fa4c86399979e956235deb900be5baa7fc1218ea67fbfaeaadd6f \\\n    --hash=sha256:b5a2b97dbdc7d4f353ebf343744f1d1f1cca8aa8bfddb4262fcf4306c3761d50 \\\n    --hash=sha256:b9a5ca4ac220a0cdd1ba6bcba3608547117d30468fefce49bb26f55c1a3d5c58 \\\n    --hash=sha256:bab485bcf8b1c7d2060b4fcb6fc368a9e6f4cd754c9c2fea281f4be21df394a2 \\\n    --hash=sha256:c108a1d6fa78a50646029cb6d49808ff0fc1330fda87fa6f6250c6b5369b6645 \\\n    --hash=sha256:d56a1efd5bfd61486c8bc968fa18734464556f0fb8e51690f4ac25d85cbbbbc2 \\\n    --hash=sha256:d9050fee89a89ed57b4fb2c1bfac9a3d0c57a0d55aed95949eedbc42070fea39 \\\n    --hash=sha256:dd80ecb295460a5d9d260df63c43f4afbdd832d725a531f008dad1664f458adf \\\n    --hash=sha256:e8ea3e2d4066083e264e75c829078f9e149fa119d27e19acd503de65e0b13149 \\\n    --hash=sha256:eb3823f11823deade26ce3b9f40dcb4a213da7a670013929f31d5f5ed1055b22 \\\n    --hash=sha256:ee40c27c795bda6a5292e9cff9890189d32f7e3a0bf04e0e3c9430c4a00c37df \\\n    --hash=sha256:efb30e3baaea72ce5928e32bab719ab4770099079d66726a62b11b1ef7273be4 \\\n    --hash=sha256:f254d118d14a7f99d616271d6c3c27922c092dac11112670b157798b89bf4933 \\\n    --hash=sha256:f89c151aab2e2e23cb3fe0acad1e8b82841fd265379c4cecd0f3fcb34c15e0f6 \\\n    --hash=sha256:f97aeb209c3d2511443f8797e3e5a569aebb040d4f8bc79aa3ee78a8fb9e3dd8 \\\n    --hash=sha256:f9b587c9c7274c1613a30afabf65a272114cd6cdbe67b3406f818c79d7ab2e2a \\\n    --hash=sha256:fb061f596dad3a0f52b60dc6a5dec4a0c300dec41e058a7efe09256188d170b7\n    # via -r tools/sound-workshop-requirements.in\nmpmath==1.3.0 \\\n    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \\\n    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\n    # via sympy\nnetworkx==3.7 \\\n    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \\\n    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a\n    # via torch\nnumpy==2.5.3 \\\n    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \\\n    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \\\n    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \\\n    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \\\n    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \\\n    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \\\n    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \\\n    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \\\n    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \\\n    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \\\n    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \\\n    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \\\n    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \\\n    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \\\n    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \\\n    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \\\n    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \\\n    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \\\n    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \\\n    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \\\n    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \\\n    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \\\n    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \\\n    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \\\n    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \\\n    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \\\n    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \\\n    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \\\n    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \\\n    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \\\n    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \\\n    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \\\n    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \\\n    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \\\n    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \\\n    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \\\n    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \\\n    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \\\n    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \\\n    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \\\n    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \\\n    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \\\n    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \\\n    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \\\n    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \\\n    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \\\n    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \\\n    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \\\n    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \\\n    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \\\n    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \\\n    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \\\n    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \\\n    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \\\n    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \\\n    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \\\n    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \\\n    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \\\n    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \\\n    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \\\n    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \\\n    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \\\n    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \\\n    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \\\n    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \\\n    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab\n    # via\n    #   -r tools/sound-workshop-requirements.in\n    #   contourpy\n    #   matplotlib\n    #   scipy\n    #   transformers\nnvidia-cublas==13.1.0.3 \\\n    --hash=sha256:2a3b94a37def342471c59fad7856caee4926809a72dd5270155d6a31b5b277be \\\n    --hash=sha256:c86fc7f7ae36d7528288c5d88098edcb7b02c633d262e7ddbb86b0ad91be5df2 \\\n    --hash=sha256:ee8722c1f0145ab246bccb9e452153b5e0515fd094c3678df50b2a0888b8b171\n    # via\n    #   cuda-toolkit\n    #   nvidia-cudnn-cu13\n    #   nvidia-cusolver\nnvidia-cuda-cupti==13.0.85 \\\n    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \\\n    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \\\n    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151\n    # via cuda-toolkit\nnvidia-cuda-nvrtc==13.0.88 \\\n    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \\\n    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \\\n    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b\n    # via cuda-toolkit\nnvidia-cuda-runtime==13.0.96 \\\n    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \\\n    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \\\n    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492\n    # via cuda-toolkit\nnvidia-cudnn-cu13==9.19.0.56 \\\n    --hash=sha256:40d8c375005bcb01495f8edf375230b203a411a0c05fb6dc92a3781edcb23eac \\\n    --hash=sha256:6ed29ffaee1176c612daf442e4dd6cfeb6a0caa43ddcbeb59da94953030b1be4 \\\n    --hash=sha256:d20e1734305e9d68889a96e3f35094d733ff1f83932ebe462753973e53a572bf\n    # via torch\nnvidia-cufft==12.0.0.61 \\\n    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \\\n    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \\\n    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3\n    # via cuda-toolkit\nnvidia-cufile==1.15.1.6 \\\n    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \\\n    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1\n    # via cuda-toolkit\nnvidia-curand==10.4.0.35 \\\n    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \\\n    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \\\n    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f\n    # via cuda-toolkit\nnvidia-cusolver==12.0.4.66 \\\n    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \\\n    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \\\n    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65\n    # via cuda-toolkit\nnvidia-cusparse==12.6.3.3 \\\n    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \\\n    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \\\n    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79\n    # via\n    #   cuda-toolkit\n    #   nvidia-cusolver\nnvidia-cusparselt-cu13==0.8.0 \\\n    --hash=sha256:25e30a8a7323935d4ad0340b95a0b69926eee755767e8e0b1cf8dd85b197d3fd \\\n    --hash=sha256:400c6ed1cf6780fc6efedd64ec9f1345871767e6a1a0a552a1ea0578117ea77c \\\n    --hash=sha256:e80212ed7b1afc97102fbb2b5c82487aa73f6a0edfa6d26c5a152593e520bb8f\n    # via torch\nnvidia-nccl-cu13==2.28.9 \\\n    --hash=sha256:01c873ba1626b54caa12272ed228dc5b2781545e0ae8ba3f432a8ef1c6d78643 \\\n    --hash=sha256:e4553a30f34195f3fa1da02a6da3d6337d28f2003943aa0a3d247bbc25fefc42\n    # via torch\nnvidia-nvjitlink==13.0.88 \\\n    --hash=sha256:13a74f429e23b921c1109976abefacc69835f2f433ebd323d3946e11d804e47b \\\n    --hash=sha256:634e96e3da9ef845ae744097a1f289238ecf946ce0b82e93cdce14b9782e682f \\\n    --hash=sha256:e931536ccc7d467a98ba1d8b89ff7fa7f1fa3b13f2b0069118cd7f47bff07d0c\n    # via\n    #   cuda-toolkit\n    #   nvidia-cufft\n    #   nvidia-cusolver\n    #   nvidia-cusparse\nnvidia-nvshmem-cu13==3.4.5 \\\n    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \\\n    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9\n    # via torch\nnvidia-nvtx==13.0.85 \\\n    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \\\n    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \\\n    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519\n    # via cuda-toolkit\npackaging==26.3 \\\n    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \\\n    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\n    # via\n    #   huggingface-hub\n    #   matplotlib\n    #   transformers\npillow==11.3.0 \\\n    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \\\n    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \\\n    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \\\n    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \\\n    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \\\n    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \\\n    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \\\n    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \\\n    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \\\n    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \\\n    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \\\n    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \\\n    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \\\n    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \\\n    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \\\n    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \\\n    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \\\n    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \\\n    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \\\n    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \\\n    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \\\n    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \\\n    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \\\n    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \\\n    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \\\n    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \\\n    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \\\n    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \\\n    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \\\n    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \\\n    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \\\n    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \\\n    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \\\n    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \\\n    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \\\n    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \\\n    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \\\n    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \\\n    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \\\n    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \\\n    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \\\n    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \\\n    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \\\n    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \\\n    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \\\n    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \\\n    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \\\n    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \\\n    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \\\n    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \\\n    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \\\n    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \\\n    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \\\n    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \\\n    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \\\n    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \\\n    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \\\n    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \\\n    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \\\n    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \\\n    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \\\n    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \\\n    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \\\n    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \\\n    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \\\n    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \\\n    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \\\n    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \\\n    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \\\n    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \\\n    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \\\n    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \\\n    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \\\n    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \\\n    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \\\n    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \\\n    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \\\n    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \\\n    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \\\n    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \\\n    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \\\n    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \\\n    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \\\n    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \\\n    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \\\n    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \\\n    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \\\n    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \\\n    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \\\n    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \\\n    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \\\n    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \\\n    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \\\n    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \\\n    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \\\n    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \\\n    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \\\n    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \\\n    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \\\n    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \\\n    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \\\n    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \\\n    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \\\n    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \\\n    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \\\n    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3\n    # via\n    #   -r tools/sound-workshop-requirements.in\n    #   matplotlib\npyparsing==3.3.3 \\\n    --hash=sha256:928ae7e20211f3b6f3915a72f06a0cfd29ab9d24279dd6346b6b1a7146397d36 \\\n    --hash=sha256:ece8c00a69cf01b45d0b1dedabb469c90d8caf996d4fda40f147627a122849a4\n    # via matplotlib\npython-dateutil==2.9.0.post0 \\\n    --hash=sha256:37dd54208da7e1cd875388217d5e00ebd4179249f90fb72437e91a35459a0ad3 \\\n    --hash=sha256:a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427\n    # via matplotlib\npyyaml==6.0.3 \\\n    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \\\n    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \\\n    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \\\n    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \\\n    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \\\n    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \\\n    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \\\n    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \\\n    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \\\n    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \\\n    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \\\n    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \\\n    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \\\n    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \\\n    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \\\n    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \\\n    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \\\n    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \\\n    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \\\n    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \\\n    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \\\n    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \\\n    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \\\n    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \\\n    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \\\n    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \\\n    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \\\n    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \\\n    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \\\n    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \\\n    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \\\n    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \\\n    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \\\n    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \\\n    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \\\n    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \\\n    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \\\n    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \\\n    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \\\n    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \\\n    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \\\n    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \\\n    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \\\n    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \\\n    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \\\n    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \\\n    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \\\n    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \\\n    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \\\n    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \\\n    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \\\n    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \\\n    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \\\n    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \\\n    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \\\n    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \\\n    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \\\n    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \\\n    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \\\n    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \\\n    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \\\n    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \\\n    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \\\n    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \\\n    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \\\n    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \\\n    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \\\n    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \\\n    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \\\n    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \\\n    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \\\n    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \\\n    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0\n    # via\n    #   huggingface-hub\n    #   transformers\nregex==2026.9.10 \\\n    --hash=sha256:030fa9e23624e39b3b94e46b90a5abd1a1678eb2f58fcdd3fd6c27526bf91c7e \\\n    --hash=sha256:032da15431c890d376f53547f0a6219f4f4cd19f3e4f11bdc321453b5bd207e4 \\\n    --hash=sha256:044bd4639b6bb409ec9e5d8b7accd57e02b4c4a4e2eafde916f8ae8006b3e40b \\\n    --hash=sha256:048a89ee797db10160bd2bd519286577a6b43a100279bd4b7d8456a3d69c80a0 \\\n    --hash=sha256:05fb018cfe7144585fc83882405906ff84994a2d154afc2509ecc7752c51f864 \\\n    --hash=sha256:07b45ba5c94b8fcb30cb6c56a11f715c57533a3017964504322ea52690a27b72 \\\n    --hash=sha256:0aa7589394230e0f0a422ab6b90841ff12c87e855e7aaf75d192a54a5f124548 \\\n    --hash=sha256:0acee94b480dd853e39434aa9a575f95385b1b4b8fa3feae56db363ca5cad782 \\\n    --hash=sha256:0b9ba3b2765cdfe18f0f561a69f78a69701f2896654a81c711108d35d14e5099 \\\n    --hash=sha256:0c32480f3371b75068decaf9e5da72c224e953830dd71e36e06cf80e30ea39d8 \\\n    --hash=sha256:1270cdec69248592bbe38a0b263ed58d907b891bd2b93703e225c317e421bda1 \\\n    --hash=sha256:13c52fc377792675f604a207a2ae5958c080f6854f7698d40d9ff034d95b1e76 \\\n    --hash=sha256:14caa05ce39ec70437af5aac8814c50ee6628f4a90353871c059692f448a164f \\\n    --hash=sha256:1562aabd9d4eb09bd88a62ad97ed06800094b529ac43419e43020b9cefec79b0 \\\n    --hash=sha256:175cf49ce7a994c88b8f15e3cb17cdb66a48ebb2d36de736b8205033db950f89 \\\n    --hash=sha256:1aa309ab7ba89a62d6cf70dbd38d4176440bce3c7001ab86256704cf4c18c6eb \\\n    --hash=sha256:1ad10a135fa0b4e4a462a61d07c6654d7518cfdb5cb8da08f9ff7d61384af1fe \\\n    --hash=sha256:1b891f77554bff991804cee24b78b40789f7d5993a24c7907bc7025fd2a70c8d \\\n    --hash=sha256:1e321e2c84f0e52c457f5ea5944f796d6e8e09cb99738ea98dcc1bfe402a128d \\\n    --hash=sha256:1e954e246466d5a1a78f563ce8364b5d7cb19e7adb0ccdec8f9c9610083187bc \\\n    --hash=sha256:1f0a8b4928823bc8b217a1ab7bf3d90598909dec9a70fbbfe9a52cc4eca55990 \\\n    --hash=sha256:1fbc8314436353e097c050e11b01a6c11433579437ed0579730157676ef59e2f \\\n    --hash=sha256:20e8bfb07ad79a282f8b95b56fe67f9750b1b7f775724e4ba1f23cb296115ce4 \\\n    --hash=sha256:217e98ba5fc8908ed8ffd4ebac04753a0c831067cbfb495b9821b94cc61eaa76 \\\n    --hash=sha256:239620b0e0681669367c0e218c8eb2551d9f8fe3b9fccfc8d0003377804e8348 \\\n    --hash=sha256:23ac9a28180f274d7dd7651fa131ad5b02d343b75df4b040737f0356223895dd \\\n    --hash=sha256:2479171edccced52ef02b899558f88ab2c235fe05b93180fdcae1670aacd89e1 \\\n    --hash=sha256:24d12a625a37c89c2b09303402a06942f55f071b95a7916a49c17034c3d47cd5 \\\n    --hash=sha256:2dd9286093c71afc8f55ef035c5b9d2776641fd72c6535f1febc92d0b0be9666 \\\n    --hash=sha256:2e67f8843f0e4b931f1fa860bf3bbe4134b714c0155cc5c7c0d7ea450230aae0 \\\n    --hash=sha256:31e4df2b11d48f61d511019bc1ee9b477055f17c352b68fe72db7a98b14d603c \\\n    --hash=sha256:3264132d576847ab5f88bb83e7debe67854bf165b3ea613bd467312b6099536a \\\n    --hash=sha256:3540734dbe241ebb3b87d5713781f6749a3e4d45480f506aa5fb5cbb0c37d249 \\\n    --hash=sha256:35ba3bab0c45079735f55ac61526774de1d84bc4a0333cc554e1a4ab74913924 \\\n    --hash=sha256:3a66e40a1a20de96a2fee00ed67e11012b62d85b277688258677fd19997addb7 \\\n    --hash=sha256:3bdeed3318a8eb2bbadc9c56347e0ff651639e934a47e168d05a3b12929fd0e7 \\\n    --hash=sha256:3fb4ae8cf83ef4e9addd43b2da31a9f45be816a8036fae8af59c8998b72718e2 \\\n    --hash=sha256:4971776b4f2bd7fd9a83eceb2cb2592cbe2924f639fe8045e6a9de5ba4bfcf25 \\\n    --hash=sha256:4a761ea45f2ad74c575ef5850ea514cef97302a552d3c7c9d1a1a870d4661d6c \\\n    --hash=sha256:4c66d54042a14a503907d81861b8a5235e6d1f03d4fbc1d8767f652eaf957ac1 \\\n    --hash=sha256:4db7d00c4afbfbb55b8e17b1e371da11418ea9389b030acec63c1fa4c7ad4b86 \\\n    --hash=sha256:4f0407474ffac8e5e89d93ca41d60891e29f0ab8423eb66ff292d850a86a0843 \\\n    --hash=sha256:53e182b6b04d0011909b47d51a2d72d908de07c7b1c7f16b3adda2204d723bc1 \\\n    --hash=sha256:5847e22bbf959764d776937d791d034cc2d19b787e361c88d97e859e8dc68502 \\\n    --hash=sha256:58c01f7b81079cf0817ba831ff4d9eff5d28be4a3ac76c353e6f09bd63f4c386 \\\n    --hash=sha256:58da726d3e766c0b3f5a3997dfaf0275898a1107b8191cdd6b0437fe45fd817d \\\n    --hash=sha256:5bef622850cf760154719d4e0d74b0a855962432995168e250069899ae12fe8f \\\n    --hash=sha256:5ccd139b2061132e7b265cfb4b4721baeb9f8928b81415304abf1ec7e3181c26 \\\n    --hash=sha256:5cef9f3d14796500ea834c41dbe688f1f6b23c7024dc23e8a794d7ebaf5d71d0 \\\n    --hash=sha256:63bb62cf62217dc38c8a6b2b61b165b0e4eb8fa93b0aba12139251c0986a8fa3 \\\n    --hash=sha256:681ed38664b64c6617d3c3c332018d1948c77e139c5ea667c1886efa671e426f \\\n    --hash=sha256:6888065672b341e5246f391ec16dc258a29218ac784172fd67c30d941544755b \\\n    --hash=sha256:6aebdd9a946de328b3f6f61dbf48dd064a36eb6dddf96e34ae6651d37f6e9383 \\\n    --hash=sha256:6afcad14310f1311d077553ed374b42a5e538f85a8c884b4e38e52de091c8077 \\\n    --hash=sha256:6b34a778c695d24e77c140e3b4c95da69282e34f2f6b02b55656aa4a0379f643 \\\n    --hash=sha256:6fd555fc9abef50c530869690b2daca054c8811a7aff632d11f9a7b2590b2742 \\\n    --hash=sha256:71879292c9c7ac67b1680345b16daba1be937cb027362cfa04e68f65db2dcfdd \\\n    --hash=sha256:75242f44a3e283106077be4ab717bc535e4701c9d54ad69e195945c22f137a1d \\\n    --hash=sha256:75aa39d3f4f1650eea84e46b0d8cefe77dd5478c10e3d0aaf0b0f00493475a7a \\\n    --hash=sha256:75f9297b16fcb588a1f8d8a55dabef3c0c20b0c7bac43c87ceaaaf1a825c12f4 \\\n    --hash=sha256:79e9432995e14c749d34209413de5e621ec8e67789bf4f46dbfabea9d06a2406 \\\n    --hash=sha256:7abb38b8c40f3a235235a44da452c64b7b5c1d650ec6351027db0e090804f2e5 \\\n    --hash=sha256:7dcad477c49c4c626a6c4fcd71b39a971aa217060cc40a6569fd24edcc0fa509 \\\n    --hash=sha256:7e6c0b5ec6ddee4032247585dc491b0fa58627745b66a705728703a3f0331231 \\\n    --hash=sha256:7f8f10015866608fe4c043cec2e4fe4c39a94bb50e45091de4cdf4004b9ae4b0 \\\n    --hash=sha256:866de9f98df0611d7b62b3a8729d3284a64c0cc6edd90bb95a533e443a4939cb \\\n    --hash=sha256:87f5f75c109f08f5c602d68e1af54cead8165189c727b6ac946b30b9833a3ba4 \\\n    --hash=sha256:880ac684c27176464c00c3fdc456116364f5ebc70da07aad0c2d4a7ba45e98db \\\n    --hash=sha256:88b02aa8d0ec9b6189fe933d425775882271c23700ac11fd26d1779b0f56fde3 \\\n    --hash=sha256:8ba1f78bd4fef2d8f84b894ec28ac3481afe6cc07aaa253ad4717ef7b3fe6bcb \\\n    --hash=sha256:8c07021a4faa3f092869adbd1f35cdc7a592276c807aeebc3ceb8ff1a638f0b4 \\\n    --hash=sha256:8d5c4518235a2ec1611e57af85fa488d529c1106aacff12adadcedf8687012cd \\\n    --hash=sha256:8e127d9a80cbf1c3276bb465c6d047e8705e97b58c2b8f2f0c0a69c336b44b37 \\\n    --hash=sha256:94c5ce3bc41d226b4eb89ca3f842b2e28c031487fb1f34eb2153d98235831325 \\\n    --hash=sha256:94d096369b7cd96d15343fef5257fe39eff9d0e8758b92a0e15e358b92cdb2fc \\\n    --hash=sha256:968c1e33edd9a104d1bf24c8d476c72de7e3839ae7f894b37e9e4f4739fdeeca \\\n    --hash=sha256:990797e765d89a423880052c68b61c31afe701de94a8c060f61c40605ca6c727 \\\n    --hash=sha256:9ce239acb15843ab03976626af810a4424b0409689ec2bbc52088ab5479ab487 \\\n    --hash=sha256:9d772586951d7d6a5d162d48f414065e483b1c81ab38fd8ed97c78b05883421a \\\n    --hash=sha256:9fbd2e5d8002dc49a6129fb321ec51c57a025e752ed525ddce0ba9223c4350a7 \\\n    --hash=sha256:a41693eb3fc4b92e6127d113813c6c395237f7edd3224abf67609af48c690d11 \\\n    --hash=sha256:abbfc1c33bf8efddcc43844aba61e036d74a918680dc3ce8ce2538b004eda0f9 \\\n    --hash=sha256:b298cdc33c5cc6969ff07f0fba19cc73e0fd8576373c50935feadaca2f6b4405 \\\n    --hash=sha256:b43456de605c8ee77eb75f07bc1ee44ba27f9cee22207deb77d495e954b7d953 \\\n    --hash=sha256:b71649169a9fcf30b395ee01047fa7ad6654a4c900ca75b23c04dedcce6a1f8c \\\n    --hash=sha256:b91c37551bf39d75116c02b146956f65b9aa0337a4a652f4ae186983789d4001 \\\n    --hash=sha256:b9d36b03dc362aa40ffaaec9d9bd75e87763529563ec008c43b0e07782f5be7a \\\n    --hash=sha256:bafa41b0dd63669e5c0f8adf3d24819efeb73c847f492eb011212eb352e69041 \\\n    --hash=sha256:bb7774924f8cd69f49cba0b3c2d679a6326f777e0e67d130ad5203e4df53f0d3 \\\n    --hash=sha256:bf29611e5376fec8f795879bb5c6153a76c3a292573d173c26784042b01eb840 \\\n    --hash=sha256:c014641157e9049b0603b8daa5343bd408d9b757b709aaa0f373cd3fab2d7944 \\\n    --hash=sha256:c103b3b14e011774af4fb7e4617ad4d72b9171905cd3b231a70a4efd76e477d7 \\\n    --hash=sha256:c22df8dd6373bbe3898e77429ffc85594300e39d752fd0e68a31e59d37899376 \\\n    --hash=sha256:c25a754bb81a2edcfc3b65eda50f017d736f818112ed43e8aafd595cb00678ae \\\n    --hash=sha256:c32818b28bcd153b25b63038348a9fe9b9fbcddb60df43f204c3ab55eeb57f77 \\\n    --hash=sha256:c37fa93bf18bf4f90b01c0fa9f11ea567ee4b7dd8bf96e63663e5edc37aa38cf \\\n    --hash=sha256:c3d95d7d9538b5b726dd6fcd7b6117a71e6565202f6d64f5845fb4d8f203f533 \\\n    --hash=sha256:c8fbd9cb30c68c1686b94029b9ef845d5870d3d65baf66cb126b676849b9d72b \\\n    --hash=sha256:cb76a9c4e07a6a47849726af0ed14c41741a182f097f134a8cf29c1bc0f4dde8 \\\n    --hash=sha256:ce7c118cb102975f974585688357a717ffbf9dddd64ab0bb1bc93eb5b367cf95 \\\n    --hash=sha256:cf377960d2ac37d987394a9dbaa75e91338c41a46d41e1d25e90125e7b3ee2dc \\\n    --hash=sha256:d278ad30ec83b6b9202685b0f80b741a51ea3ca7f0595ebda96e7628b6398876 \\\n    --hash=sha256:d2d377fd1cad611b806cdd732d86b65f536c768209890cb442556548daa65a23 \\\n    --hash=sha256:d414c411c06fe0009eac33488fb1591c66b5c2673e342e452e7bb2fe63da8194 \\\n    --hash=sha256:d8c668af8f7bdb1d18739c27d30cd9f4b371495a883f75a002fb7a39d740fecd \\\n    --hash=sha256:dce932f8e3ba936475ea3d0d8b59f7b050a9e206e994f53f8fd80299871e87da \\\n    --hash=sha256:debc629e98b95abaea1cf3057ca296151f348c697c9b8a59d18013adb302c0dd \\\n    --hash=sha256:e0dc78251154b66dc60211563fc115345da332eaa881e4e2523fb1edae3772f4 \\\n    --hash=sha256:e5e4a6e0734a685d13b9685622bb503bdbb2927f8b0df025a5085f0ea067475b \\\n    --hash=sha256:e6b99181d184d0f5c7b36b8d12b94d1e9499cce6246594331f9edc5d2ea9fceb \\\n    --hash=sha256:e7327795089ddb44912dce1434e1d7244be2e9fb48fcc2d6782936af7a3062db \\\n    --hash=sha256:ebb2ba68e4641a994061f70bf44ed448fba0b9b1d18c94ffb9efc1cca805b39b \\\n    --hash=sha256:ec8855f08c17895a26fbf5f19ed829722e19b34a96629e49a43c92974924026b \\\n    --hash=sha256:ecb2e7acb18f8cc4a67f0ad986c0af291ea4dd385d0614ba9bc09d7f8bbb478c \\\n    --hash=sha256:ef4c0a9dfdc90581b90b1b95a8c3d1557f8ff8f5a2a53536d26314de699d1468 \\\n    --hash=sha256:ef4ce69ff97fbb44b46751cfea5e859ad0b66d1a50abf34954f0645f51e81671 \\\n    --hash=sha256:ef5a059ea1c6ee5d1c7e99a2484e628608d010921efe876c6f0e2029d2f35eca \\\n    --hash=sha256:f0e2e5d23448b660d60a6ed85c46cc03b4b48bd276b8f4041d4a5fe2a4a0626b \\\n    --hash=sha256:f2374c27deb189b282ec7e16106752c22ad39b056bbd8018960b1e4cc95d67a1 \\\n    --hash=sha256:f2f43bf4e47ff7ce9e585558706d698c6204d0f80bf2207766382ed817c8e9f4 \\\n    --hash=sha256:f5c629df03adec31ee505dda3c8988f106c9390e4cbd343600036eb8b3d6724f \\\n    --hash=sha256:f70b9f0e39c2dba1d9da6bf7ef7c377cad7277f8440e9a69be05ede529ff024c \\\n    --hash=sha256:f7d4656e17ab736e9415a6442a345bfc97bb8b7dcce47884bb74a37f70f08d0c \\\n    --hash=sha256:f8bdec659a8fa7af51a32b224b3b7c02bc415d54ffd35187b1d224176b17d607 \\\n    --hash=sha256:faa911fbbcf8ac90bda0e0657d60768e3390954ef0588211d63a22add1cb1cd1 \\\n    --hash=sha256:fbc4e2f3cb7ce8436154e6483079e7d35eeb321a952fa936e180300630d8b873 \\\n    --hash=sha256:fd6bd89b9fc06018d35851cab0240adb7dd84d51941b19f6574ac90cd54e3ae5 \\\n    --hash=sha256:ff4d7b14ea19e50c8d9d6d83f45bd9b45cbb624c07ac1fa54db0a019049abed7 \\\n    --hash=sha256:ff6b3267318661dfddf6b3628663e00e5946bd0a5c8fa678537a1401f0388f91 \\\n    --hash=sha256:ffc2da104e43db716ce30cef9f28049a1faa6aca385dd8771b033268d0730b07\n    # via transformers\nrequests==2.34.2 \\\n    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \\\n    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed\n    # via\n    #   huggingface-hub\n    #   transformers\nsafetensors==0.8.0 \\\n    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \\\n    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \\\n    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \\\n    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \\\n    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \\\n    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \\\n    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \\\n    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \\\n    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \\\n    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \\\n    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \\\n    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \\\n    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \\\n    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \\\n    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \\\n    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \\\n    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774\n    # via\n    #   -r tools/sound-workshop-requirements.in\n    #   transformers\nscipy==1.17.1 \\\n    --hash=sha256:010f4333c96c9bb1a4516269e33cb5917b08ef2166d5556ca2fd9f082a9e6ea0 \\\n    --hash=sha256:02ae3b274fde71c5e92ac4d54bc06c42d80e399fec704383dcd99b301df37458 \\\n    --hash=sha256:08b900519463543aa604a06bec02461558a6e1cef8fdbb8098f77a48a83c8118 \\\n    --hash=sha256:131f5aaea57602008f9822e2115029b55d4b5f7c070287699fe45c661d051e39 \\\n    --hash=sha256:158dd96d2207e21c966063e1635b1063cd7787b627b6f07305315dd73d9c679e \\\n    --hash=sha256:1cc682cea2ae55524432f3cdff9e9a3be743d52a7443d0cba9017c23c87ae2f6 \\\n    --hash=sha256:1f95b894f13729334fb990162e911c9e5dc1ab390c58aa6cbecb389c5b5e28ec \\\n    --hash=sha256:200e1050faffacc162be6a486a984a0497866ec54149a01270adc8a59b7c7d21 \\\n    --hash=sha256:2040ad4d1795a0ae89bfc7e8429677f365d45aa9fd5e4587cf1ea737f927b4a1 \\\n    --hash=sha256:2b64ca7d4aee0102a97f3ba22124052b4bd2152522355073580bf4845e2550b6 \\\n    --hash=sha256:2ceb2d3e01c5f1d83c4189737a42d9cb2fc38a6eeed225e7515eef71ad301dce \\\n    --hash=sha256:35c3a56d2ef83efc372eaec584314bd0ef2e2f0d2adb21c55e6ad5b344c0dcb8 \\\n    --hash=sha256:37425bc9175607b0268f493d79a292c39f9d001a357bebb6b88fdfaff13f6448 \\\n    --hash=sha256:3877ac408e14da24a6196de0ddcace62092bfc12a83823e92e49e40747e52c19 \\\n    --hash=sha256:3fd1fcdab3ea951b610dc4cef356d416d5802991e7e32b5254828d342f7b7e0b \\\n    --hash=sha256:41b71f4a3a4cab9d366cd9065b288efc4d4f3c0b37a91a8e0947fb5bd7f31d87 \\\n    --hash=sha256:43af8d1f3bea642559019edfe64e9b11192a8978efbd1539d7bc2aaa23d92de4 \\\n    --hash=sha256:45abad819184f07240d8a696117a7aacd39787af9e0b719d00285549ed19a1e9 \\\n    --hash=sha256:4b400bdc6f79fa02a4d86640310dde87a21fba0c979efff5248908c6f15fad1b \\\n    --hash=sha256:4eb6c25dd62ee8d5edf68a8e1c171dd71c292fdae95d8aeb3dd7d7de4c364082 \\\n    --hash=sha256:581b2264fc0aa555f3f435a5944da7504ea3a065d7029ad60e7c3d1ae09c5464 \\\n    --hash=sha256:5cf36e801231b6a2059bf354720274b7558746f3b1a4efb43fcf557ccd484a87 \\\n    --hash=sha256:5e3c5c011904115f88a39308379c17f91546f77c1667cea98739fe0fccea804c \\\n    --hash=sha256:6609bc224e9568f65064cfa72edc0f24ee6655b47575954ec6339534b2798369 \\\n    --hash=sha256:6e3dcd57ab780c741fde8dc68619de988b966db759a3c3152e8e9142c26295ad \\\n    --hash=sha256:6fac755ca3d2c3edcb22f479fceaa241704111414831ddd3bc6056e18516892f \\\n    --hash=sha256:744b2bf3640d907b79f3fd7874efe432d1cf171ee721243e350f55234b4cec4c \\\n    --hash=sha256:74cbb80d93260fe2ffa334efa24cb8f2f0f622a9b9febf8b483c0b865bfb3475 \\\n    --hash=sha256:766e0dc5a616d026a3a1cffa379af959671729083882f50307e18175797b3dfd \\\n    --hash=sha256:7bdf2da170b67fdf10bca777614b1c7d96ae3ca5794fd9587dce41eb2966e866 \\\n    --hash=sha256:7ff200bf9d24f2e4d5dc6ee8c3ac64d739d3a89e2326ba68aaf6c4a2b838fd7d \\\n    --hash=sha256:844e165636711ef41f80b4103ed234181646b98a53c8f05da12ca5ca289134f6 \\\n    --hash=sha256:8a604bae87c6195d8b1045eddece0514d041604b14f2727bbc2b3020172045eb \\\n    --hash=sha256:94055a11dfebe37c656e70317e1996dc197e1a15bbcc351bcdd4610e128fe1ca \\\n    --hash=sha256:95d8e012d8cb8816c226aef832200b1d45109ed4464303e997c5b13122b297c0 \\\n    --hash=sha256:9cdc1a2fcfd5c52cfb3045feb399f7b3ce822abdde3a193a6b9a60b3cb5854ca \\\n    --hash=sha256:9ecb4efb1cd6e8c4afea0daa91a87fbddbce1b99d2895d151596716c0b2e859d \\\n    --hash=sha256:a3472cfbca0a54177d0faa68f697d8ba4c80bbdc19908c3465556d9f7efce9ee \\\n    --hash=sha256:a4328d245944d09fd639771de275701ccadf5f781ba0ff092ad141e017eccda4 \\\n    --hash=sha256:a48a72c77a310327f6a3a920092fa2b8fd03d7deaa60f093038f22d98e096717 \\\n    --hash=sha256:a720477885a9d2411f94a93d16f9d89bad0f28ca23c3f8daa521e2dcc3f44d49 \\\n    --hash=sha256:a77cbd07b940d326d39a1d1b37817e2ee4d79cb30e7338f3d0cddffae70fcaa2 \\\n    --hash=sha256:a9956e4d4f4a301ebf6cde39850333a6b6110799d470dbbb1e25326ac447f52a \\\n    --hash=sha256:adb2642e060a6549c343603a3851ba76ef0b74cc8c079a9a58121c7ec9fe2350 \\\n    --hash=sha256:beeda3d4ae615106d7094f7e7cef6218392e4465cc95d25f900bebabfded0950 \\\n    --hash=sha256:c80be5ede8f3f8eded4eff73cc99a25c388ce98e555b17d31da05287015ffa5b \\\n    --hash=sha256:cc90d2e9c7e5c7f1a482c9875007c095c3194b1cfedca3c2f3291cdc2bc7c086 \\\n    --hash=sha256:cd96a1898c0a47be4520327e01f874acfd61fb48a9420f8aa9f6483412ffa444 \\\n    --hash=sha256:d2650c1fb97e184d12d8ba010493ee7b322864f7d3d00d3f9bb97d9c21de4068 \\\n    --hash=sha256:d30e57c72013c2a4fe441c2fcb8e77b14e152ad48b5464858e07e2ad9fbfceff \\\n    --hash=sha256:d59c30000a16d8edc7e64152e30220bfbd724c9bbb08368c054e24c651314f0a \\\n    --hash=sha256:dbc12c9f3d185f5c737d801da555fb74b3dcfa1a50b66a1a93e09190f41fab50 \\\n    --hash=sha256:e18f12c6b0bc5a592ed23d3f7b891f68fd7f8241d69b7883769eb5d5dfb52696 \\\n    --hash=sha256:e19ebea31758fac5893a2ac360fedd00116cbb7628e650842a6691ba7ca28a21 \\\n    --hash=sha256:e30bdeaa5deed6bc27b4cc490823cd0347d7dae09119b8803ae576ea0ce52e4c \\\n    --hash=sha256:eb092099205ef62cd1782b006658db09e2fed75bffcae7cc0d44052d8aa0f484 \\\n    --hash=sha256:eee2cfda04c00a857206a4330f0c5e3e56535494e30ca445eb19ec624ae75118 \\\n    --hash=sha256:f4115102802df98b2b0db3cce5cb9b92572633a1197c77b7553e5203f284a5b3 \\\n    --hash=sha256:f590cd684941912d10becc07325a3eeb77886fe981415660d9265c4c418d0bea \\\n    --hash=sha256:f8885db0bc2bffa59d5c1b72fad7a6a92d3e80e7257f967dd81abb553a90d293 \\\n    --hash=sha256:fcb310ddb270a06114bb64bbe53c94926b943f5b7f0842194d585c65eb4edd76\n    # via -r tools/sound-workshop-requirements.in\nsetuptools==81.0.0 \\\n    --hash=sha256:487b53915f52501f0a79ccfd0c02c165ffe06631443a886740b91af4b7a5845a \\\n    --hash=sha256:fdd925d5c5d9f62e4b74b30d6dd7828ce236fd6ed998a08d81de62ce5a6310d6\n    # via torch\nsix==1.17.0 \\\n    --hash=sha256:4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274 \\\n    --hash=sha256:ff70335d468e7eb6ec65b95b99d3a2836546063f63acc5171de367e834932a81\n    # via python-dateutil\nsympy==1.14.0 \\\n    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \\\n    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5\n    # via torch\ntokenizers==0.22.2 \\\n    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \\\n    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \\\n    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \\\n    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \\\n    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \\\n    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \\\n    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \\\n    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \\\n    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \\\n    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \\\n    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \\\n    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \\\n    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \\\n    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \\\n    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \\\n    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \\\n    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \\\n    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \\\n    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \\\n    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \\\n    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \\\n    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \\\n    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \\\n    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5\n    # via transformers\ntorch==2.11.0 \\\n    --hash=sha256:01018087326984a33b64e04c8cb5c2795f9120e0d775ada1f6638840227b04d7 \\\n    --hash=sha256:0f68f4ac6d95d12e896c3b7a912b5871619542ec54d3649cf48cc1edd4dd2756 \\\n    --hash=sha256:1b32ceda909818a03b112006709b02be1877240c31750a8d9c6b7bf5f2d8a6e5 \\\n    --hash=sha256:1e6debd97ccd3205bbb37eb806a9d8219e1139d15419982c09e23ef7d4369d18 \\\n    --hash=sha256:2658f34ce7e2dabf4ec73b45e2ca68aedad7a5be87ea756ad656eaf32bf1e1ea \\\n    --hash=sha256:2b4e811728bd0cc58fb2b0948fe939a1ee2bf1422f6025be2fca4c7bd9d79718 \\\n    --hash=sha256:2bb3cc54bd0dea126b0060bb1ec9de0f9c7f7342d93d436646516b0330cd5be7 \\\n    --hash=sha256:2c0d7fcfbc0c4e8bb5ebc3907cbc0c6a0da1b8f82b1fc6e14e914fa0b9baf74e \\\n    --hash=sha256:4b5866312ee6e52ea625cd211dcb97d6a2cdc1131a5f15cc0d87eec948f6dd34 \\\n    --hash=sha256:4cf8687f4aec3900f748d553483ef40e0ac38411c3c48d0a86a438f6d7a99b18 \\\n    --hash=sha256:4dc8b3809469b6c30b411bb8c4cad3828efd26236153d9beb6a3ec500f211a60 \\\n    --hash=sha256:4dda3b3f52d121063a731ddb835f010dc137b920d7fec2778e52f60d8e4bf0cd \\\n    --hash=sha256:563ed3d25542d7e7bbc5b235ccfacfeb97fb470c7fee257eae599adb8005c8a2 \\\n    --hash=sha256:63a68fa59de8f87acc7e85a5478bb2dddbb3392b7593ec3e78827c793c4b73fd \\\n    --hash=sha256:73e24aaf8f36ab90d95cd1761208b2eb70841c2a9ca1a3f9061b39fc5331b708 \\\n    --hash=sha256:7aa2f9bbc6d4595ba72138026b2074be1233186150e9292865e04b7a63b8c67a \\\n    --hash=sha256:7b6a60d48062809f58595509c524b88e6ddec3ebe25833d6462eeab81e5f2ce4 \\\n    --hash=sha256:8245477871c3700d4370352ffec94b103cfcb737229445cf9946cddb7b2ca7cd \\\n    --hash=sha256:8b394322f49af4362d4f80e424bcaca7efcd049619af03a4cf4501520bdf0fb4 \\\n    --hash=sha256:98bb213c3084cfe176302949bdc360074b18a9da7ab59ef2edc9d9f742504778 \\\n    --hash=sha256:a97b94bbf62992949b4730c6cd2cc9aee7b335921ee8dc207d930f2ed09ae2db \\\n    --hash=sha256:ab9a8482f475f9ba20e12db84b0e55e2f58784bdca43a854a6ccd3fd4b9f75e6 \\\n    --hash=sha256:b2a43985ff5ef6ddd923bbcf99943e5f58059805787c5c9a2622bf05ca2965b0 \\\n    --hash=sha256:b3c712ae6fb8e7a949051a953fc412fe0a6940337336c3b6f905e905dac5157f \\\n    --hash=sha256:cc89b9b173d9adfab59fd227f0ab5e5516d9a52b658ae41d64e59d2e55a418db \\\n    --hash=sha256:d91aac77f24082809d2c5a93f52a5f085032740a1ebc9252a7b052ef5a4fddc6 \\\n    --hash=sha256:f99924682ef0aa6a4ab3b1b76f40dc6e273fca09f367d15a524266db100a723f \\\n    --hash=sha256:fbf39280699d1b869f55eac536deceaa1b60bd6788ba74f399cc67e60a5fab10\n    # via -r tools/sound-workshop-requirements.in\ntorchaudio==2.11.0 \\\n    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \\\n    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \\\n    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \\\n    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \\\n    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \\\n    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \\\n    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \\\n    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \\\n    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \\\n    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \\\n    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \\\n    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \\\n    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \\\n    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \\\n    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \\\n    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \\\n    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \\\n    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \\\n    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \\\n    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \\\n    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \\\n    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \\\n    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \\\n    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \\\n    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \\\n    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \\\n    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \\\n    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84\n    # via -r tools/sound-workshop-requirements.in\ntqdm==4.70.1 \\\n    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \\\n    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4\n    # via\n    #   huggingface-hub\n    #   transformers\ntransformers==4.57.6 \\\n    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \\\n    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3\n    # via -r tools/sound-workshop-requirements.in\ntriton==3.6.0 \\\n    --hash=sha256:0b3a97e8ed304dfa9bd23bb41ca04cdf6b2e617d5e782a8653d616037a5d537d \\\n    --hash=sha256:10c7f76c6e72d2ef08df639e3d0d30729112f47a56b0c81672edc05ee5116ac9 \\\n    --hash=sha256:1722e172d34e32abc3eb7711d0025bb69d7959ebea84e3b7f7a341cd7ed694d6 \\\n    --hash=sha256:374f52c11a711fd062b4bfbb201fd9ac0a5febd28a96fb41b4a0f51dde3157f4 \\\n    --hash=sha256:448e02fe6dc898e9e5aa89cf0ee5c371e99df5aa5e8ad976a80b93334f3494fd \\\n    --hash=sha256:46bd1c1af4b6704e554cad2eeb3b0a6513a980d470ccfa63189737340c7746a7 \\\n    --hash=sha256:49df5ef37379c0c2b5c0012286f80174fcf0e073e5ade1ca9a86c36814553651 \\\n    --hash=sha256:6c723cfb12f6842a0ae94ac307dba7e7a44741d720a40cf0e270ed4a4e3be781 \\\n    --hash=sha256:74caf5e34b66d9f3a429af689c1c7128daba1d8208df60e81106b115c00d6fca \\\n    --hash=sha256:a17a5d5985f0ac494ed8a8e54568f092f7057ef60e1b0fa09d3fd1512064e803 \\\n    --hash=sha256:a6550fae429e0667e397e5de64b332d1e5695b73650ee75a6146e2e902770bea \\\n    --hash=sha256:d002e07d7180fd65e622134fbd980c9a3d4211fb85224b56a0a0efbd422ab72f \\\n    --hash=sha256:e8e323d608e3a9bfcc2d9efcc90ceefb764a82b99dea12a86d643c72539ad5d3 \\\n    --hash=sha256:ef5523241e7d1abca00f1d240949eebdd7c673b005edbbce0aca95b8191f1d43\n    # via torch\ntyping-extensions==4.16.0 \\\n    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \\\n    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5\n    # via\n    #   huggingface-hub\n    #   torch\nurllib3==2.8.0 \\\n    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \\\n    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63\n    # via requests\n', 'sample.json': '{\n  "schema_version": 1,\n  "repository": "karolpiczak/ESC-50",\n  "revision": "33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6",\n  "license_statement": "ESC-10 subset: CC BY 3.0; retain original per-clip terms and attribution",\n  "metadata_pins": {\n    "csv": {\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/meta/esc50.csv",\n      "bytes": 93742,\n      "sha256": "ca660da60191a97de289983a05821c9382d852a38a2ba8428980816b68cf6246"\n    },\n    "license": {\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/LICENSE",\n      "bytes": 284523,\n      "sha256": "f6aec3a3283a2efdcbb1feff334adad7ab17c097bdec0cbb4eebdb9e16886074"\n    }\n  },\n  "class_names": [\n    "chainsaw",\n    "clock_tick",\n    "crackling_fire",\n    "crying_baby",\n    "dog",\n    "helicopter",\n    "rain",\n    "rooster",\n    "sea_waves",\n    "sneezing"\n  ],\n  "counts": {\n    "train": 240,\n    "validation": 80,\n    "test": 80\n  },\n  "source_groups": 304,\n  "total_audio_bytes": 176417600,\n  "original_license_counts": {\n    "CC-BY": 165,\n    "CC-Sampling+": 2,\n    "CC0": 233\n  },\n  "records": [\n    {\n      "id": "1-100032-A-0",\n      "filename": "1-100032-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-100032-A-0.wav",\n      "bytes": 441044,\n      "sha256": "f40a849a2375c8c63312a73dd2dd6c74007301fcc21b4be2ece29a642831e3d8",\n      "decoded_sha256": "ce93f9b12df9b8780dd80c365fbfadb1b48e8c491c057b9c7f5011901151ff47",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "100032",\n      "source_id": "100032",\n      "attribution": "- [1-100032-A.ogg]: clip derived from rose_bark.wav (http://www.freesound.org/people/nfrae/sounds/100032/) by nfrae [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-110389-A-0",\n      "filename": "1-110389-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-110389-A-0.wav",\n      "bytes": 441044,\n      "sha256": "4a18f3f834eb184b9f02595ac74c7de37ef8048a3da92f58f27018fec580b36a",\n      "decoded_sha256": "adeddad989a374eb83e7dd8856b74860240838cb549de9066fae264a529f5907",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "110389",\n      "source_id": "110389",\n      "attribution": "- [1-110389-A.ogg]: clip derived from animals_dog_bark_springer_spaniel_001.wav (http://www.freesound.org/people/soundscalpel.com/sounds/110389/) by soundscalpel.com [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-116765-A-41",\n      "filename": "1-116765-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-116765-A-41.wav",\n      "bytes": 441044,\n      "sha256": "1199ff6d4f5040bf20f06a441276bcd6448b1e1de4f098ce3e47d856ee97ce33",\n      "decoded_sha256": "a413d3477970fafdfa919d1c40c7e831e659858f702321d29418ccd5b1ee88f7",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "116765",\n      "source_id": "116765",\n      "attribution": "- [1-116765-A.ogg]: clip derived from chainsaw.wav (http://www.freesound.org/people/UncleSigmund/sounds/116765/) by UncleSigmund [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-17150-A-12",\n      "filename": "1-17150-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-17150-A-12.wav",\n      "bytes": 441044,\n      "sha256": "3d278bc2034dfb83881491a8eec97de6f1239f0a5b8a7c02be46b5aab0a51609",\n      "decoded_sha256": "67cefb7ce299a80fe358ab0271a21f956ff7fc66d5200f607b04b89942cc21b9",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "17150",\n      "source_id": "17150",\n      "attribution": "- [1-17150-A.ogg]: clip derived from lg fire5.wav (http://www.freesound.org/people/lgarrett/sounds/17150/) by lgarrett [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-172649-A-40",\n      "filename": "1-172649-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-172649-A-40.wav",\n      "bytes": 441044,\n      "sha256": "23ad50d792907e1caa894e6e31bb623938ae0417606b830b433da11ca2ecb2f4",\n      "decoded_sha256": "13343b98b5248f4baa271ab6bb52092254f16df530f8fbae18aad436149a1d37",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "172649",\n      "source_id": "172649",\n      "attribution": "- [1-172649-A.ogg]: clip derived from Small Helicopter Takes Off (http://www.freesound.org/people/senorstudy/sounds/172649/) by senorstudy [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-172649-B-40",\n      "filename": "1-172649-B-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-172649-B-40.wav",\n      "bytes": 441044,\n      "sha256": "ca32a4cd05246a83b4ede5ecd48d05de0fd026169002f786f28de7eaa3693630",\n      "decoded_sha256": "dcf4cfeaa4f4d3b9411b7bf385b2773fe9eb620ed916e033ccacad55ec003efb",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "172649",\n      "source_id": "172649",\n      "attribution": "- [1-172649-B.ogg]: clip derived from Small Helicopter Takes Off (http://www.freesound.org/people/senorstudy/sounds/172649/) by senorstudy [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-172649-C-40",\n      "filename": "1-172649-C-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-172649-C-40.wav",\n      "bytes": 441044,\n      "sha256": "9bc5e8a84cc1beb275a81d8041faaf0c6b0681c6a6962972c85e5c8e54a03a89",\n      "decoded_sha256": "63457cab82bcdd656d0d8b9b99edc3b767e3cac4c67f6794a2d6797aeeb33687",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "172649",\n      "source_id": "172649",\n      "attribution": "- [1-172649-C.ogg]: clip derived from Small Helicopter Takes Off (http://www.freesound.org/people/senorstudy/sounds/172649/) by senorstudy [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-172649-D-40",\n      "filename": "1-172649-D-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-172649-D-40.wav",\n      "bytes": 441044,\n      "sha256": "0c742933bef2f80b0bde223bbd00bf92013b0130fc0ebfb139d873774118af6f",\n      "decoded_sha256": "0b82030508ea192f5d4643919c0635b3a8b73e240c578498242f88fe689f77da",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "172649",\n      "source_id": "172649",\n      "attribution": "- [1-172649-D.ogg]: clip derived from Small Helicopter Takes Off (http://www.freesound.org/people/senorstudy/sounds/172649/) by senorstudy [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-172649-E-40",\n      "filename": "1-172649-E-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-172649-E-40.wav",\n      "bytes": 441044,\n      "sha256": "2fa49f370f70849805b2467339431682860ac025c2beccfd4afaaa9939dc63c7",\n      "decoded_sha256": "9aa31be51d765f5579f3b32c2fec767b9d9e769c31e31e7d084a34e4d9ff61dd",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "172649",\n      "source_id": "172649",\n      "attribution": "- [1-172649-E.ogg]: clip derived from Small Helicopter Takes Off (http://www.freesound.org/people/senorstudy/sounds/172649/) by senorstudy [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-172649-F-40",\n      "filename": "1-172649-F-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-172649-F-40.wav",\n      "bytes": 441044,\n      "sha256": "02635aac12a85ba006c6558dd11246a2afed7e123cc42c3e14a5ab4081a22d71",\n      "decoded_sha256": "4e3fc1fb3f9f503ce97c7fa47061acef86a99ea588e4a4ab9b0fd4554773cce0",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "172649",\n      "source_id": "172649",\n      "attribution": "- [1-172649-F.ogg]: clip derived from Small Helicopter Takes Off (http://www.freesound.org/people/senorstudy/sounds/172649/) by senorstudy [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-17367-A-10",\n      "filename": "1-17367-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-17367-A-10.wav",\n      "bytes": 441044,\n      "sha256": "d732acd2e0c7c40405968010f49f250740c86998fc416eff2ed5330fc8c89dea",\n      "decoded_sha256": "2f43bc1f0217e8ec6ec528dcdb1ea0f2f12085ead08c3d8673467b6a5faf8e6a",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "17367",\n      "source_id": "17367",\n      "attribution": "- [1-17367-A.ogg]: clip derived from unlocki door to mod rain.wav (http://www.freesound.org/people/cognito%20perceptu/sounds/17367/) by cognito perceptu [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-17565-A-12",\n      "filename": "1-17565-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-17565-A-12.wav",\n      "bytes": 441044,\n      "sha256": "bf16e25363a2d5824c07e96dc05defcb87e1d9dcd47bfd241a9d25e01b411209",\n      "decoded_sha256": "f7d1b447a5323e732c9c7655efbd575037a3a5fb8378ef3859652811a81fd5f7",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "17565",\n      "source_id": "17565",\n      "attribution": "- [1-17565-A.ogg]: clip derived from fire.mp3 (http://www.freesound.org/people/markfrancombe/sounds/17565/) by markfrancombe [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-17742-A-12",\n      "filename": "1-17742-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-17742-A-12.wav",\n      "bytes": 441044,\n      "sha256": "57c4d44ed80ad163ac63353ea98a7b286cfbe5ceb844cc57959b70c08ce5dde9",\n      "decoded_sha256": "0c2da8ea830d4994732de4bc24a156ea687b02095d6113e0d90c0f41d5e78e31",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "17742",\n      "source_id": "17742",\n      "attribution": "- [1-17742-A.ogg]: clip derived from Fire_Crackles(No Room).wav (http://www.freesound.org/people/Krisboruff/sounds/17742/) by Krisboruff [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-17808-A-12",\n      "filename": "1-17808-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-17808-A-12.wav",\n      "bytes": 441044,\n      "sha256": "e6a90f134fe2f2eaa759b2b01ec831d6f86e300522fb0a5d19e050df16c606af",\n      "decoded_sha256": "073295221a444e8555e78946c498703feed5e24c12d459bd829c9ea559033ec3",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "17808",\n      "source_id": "17808",\n      "attribution": "- [1-17808-A.ogg]: clip derived from Foc.WAV (http://www.freesound.org/people/Everard/sounds/17808/) by Everard [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-17808-B-12",\n      "filename": "1-17808-B-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-17808-B-12.wav",\n      "bytes": 441044,\n      "sha256": "f28b0f8158b6a5adfa4bca5df5c71963c839dd22815440c8b3e220ef2016a365",\n      "decoded_sha256": "8fb0b396e8011fad2dfcd361ba0be5e68b293d0eba4add652a8b869f624baf7c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "17808",\n      "source_id": "17808",\n      "attribution": "- [1-17808-B.ogg]: clip derived from Foc.WAV (http://www.freesound.org/people/Everard/sounds/17808/) by Everard [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-181071-A-40",\n      "filename": "1-181071-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-181071-A-40.wav",\n      "bytes": 441044,\n      "sha256": "5faf5c216d7573ba4f0a07c5495d3666925c5287b97752341661614ea68d87dd",\n      "decoded_sha256": "829a6a153377a936f981033050514f6eeaa4a13fe2b7cb1d08ceb22adebc7dbd",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "181071",\n      "source_id": "181071",\n      "attribution": "- [1-181071-A.ogg]: clip derived from helicopter1.wav (http://www.freesound.org/people/dwareing/sounds/181071/) by dwareing [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-181071-B-40",\n      "filename": "1-181071-B-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-181071-B-40.wav",\n      "bytes": 441044,\n      "sha256": "6727248f265e7f9d179c6b72e85f8d2cbc4a31edcac27492036422116bb1673d",\n      "decoded_sha256": "3e1c7d6ea08e9a20a213a234e555afdc59d2f9d3117e003dc81a24929f7c6c56",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "181071",\n      "source_id": "181071",\n      "attribution": "- [1-181071-B.ogg]: clip derived from helicopter1.wav (http://www.freesound.org/people/dwareing/sounds/181071/) by dwareing [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-187207-A-20",\n      "filename": "1-187207-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-187207-A-20.wav",\n      "bytes": 441044,\n      "sha256": "ddf5bf45f73bc73d1838bf0cb6af530584f324aef513112ee15c210e83ecbd6b",\n      "decoded_sha256": "df05466749532c4a99a5c0d4f9050ab055519a8337197ac7ef839725ccc37845",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "187207",\n      "source_id": "187207",\n      "attribution": "- [1-187207-A.ogg]: clip derived from More baby noises... (http://www.freesound.org/people/k3pp/sounds/187207/) by k3pp [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-19898-A-41",\n      "filename": "1-19898-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-19898-A-41.wav",\n      "bytes": 441044,\n      "sha256": "02afa3102978b5fb16f9c6237d90231abb74318873309600f01caf4776746944",\n      "decoded_sha256": "97c3591c344191037f60ae5fbac651c07412a23e844ae34ed7ab47b8da4b71eb",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "19898",\n      "source_id": "19898",\n      "attribution": "- [1-19898-A.ogg]: clip derived from CHAINSAW.wav (http://www.freesound.org/people/JFBSAUVE/sounds/19898/) by JFBSAUVE [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-19898-B-41",\n      "filename": "1-19898-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-19898-B-41.wav",\n      "bytes": 441044,\n      "sha256": "cab4735712a7727e16cc1946308f80034c926c59ed467dccf539f39bebe8d61d",\n      "decoded_sha256": "cd201ed2f8bbb871a3ef97981172376e32825bd4bbeb422bcb564421c3843a18",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "19898",\n      "source_id": "19898",\n      "attribution": "- [1-19898-B.ogg]: clip derived from CHAINSAW.wav (http://www.freesound.org/people/JFBSAUVE/sounds/19898/) by JFBSAUVE [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-19898-C-41",\n      "filename": "1-19898-C-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-19898-C-41.wav",\n      "bytes": 441044,\n      "sha256": "04f11bf73d8b4df3c64b523a334f42f44053045478810093bf028fb02a82448e",\n      "decoded_sha256": "0bd4e34e3fc4d09d040dabd8b054a83b434faaf9b9a5e2809c9c3f72cf914134",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "19898",\n      "source_id": "19898",\n      "attribution": "- [1-19898-C.ogg]: clip derived from CHAINSAW.wav (http://www.freesound.org/people/JFBSAUVE/sounds/19898/) by JFBSAUVE [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-211527-A-20",\n      "filename": "1-211527-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-211527-A-20.wav",\n      "bytes": 441044,\n      "sha256": "3cbcd59a0db21f0ed8c252c74ea1bc193117d2c13ed99821885c46c564010b3e",\n      "decoded_sha256": "f29ff8fab712b93eed724433d92422bebc1ba706bb59e6b76e83eaada5fc68b3",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "211527",\n      "source_id": "211527",\n      "attribution": "- [1-211527-A.ogg]: clip derived from Crying newborn baby child 3.WAV (http://www.freesound.org/people/the_yura/sounds/211527/) by the_yura [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-211527-B-20",\n      "filename": "1-211527-B-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-211527-B-20.wav",\n      "bytes": 441044,\n      "sha256": "935be6bfdd4e24f767f5af092bb88c9a6ac000b926f1fe9075b66ebed907c8c0",\n      "decoded_sha256": "c51f8276bd424b7c98579e1e1ed69caf0512a4afc9444c9579520f7916ff8a87",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "211527",\n      "source_id": "211527",\n      "attribution": "- [1-211527-B.ogg]: clip derived from Crying newborn baby child 3.WAV (http://www.freesound.org/people/the_yura/sounds/211527/) by the_yura [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-211527-C-20",\n      "filename": "1-211527-C-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-211527-C-20.wav",\n      "bytes": 441044,\n      "sha256": "165fa383fbfd24ea4455a84802a78970f02c3461677c23fabecd647c3c675dbc",\n      "decoded_sha256": "b03b29619ece52716a4daf59ef0f097c217c1ee43f7dbbdc529efb279a86f480",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "211527",\n      "source_id": "211527",\n      "attribution": "- [1-211527-C.ogg]: clip derived from Crying newborn baby child 3.WAV (http://www.freesound.org/people/the_yura/sounds/211527/) by the_yura [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-21189-A-10",\n      "filename": "1-21189-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-21189-A-10.wav",\n      "bytes": 441044,\n      "sha256": "5f15ab2d8f6ef9932909b1b75c8ac254cbea46c93b4b2f0f3542abdf09b9abc0",\n      "decoded_sha256": "8bcabced864843a07ac4f311fdc5e5560dda8bcf6d3fccdac7da2a82cb56e314",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "21189",\n      "source_id": "21189",\n      "attribution": "- [1-21189-A.ogg]: clip derived from Louisiana Rain 1.wav (http://www.freesound.org/people/uberhuberman/sounds/21189/) by uberhuberman [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-21934-A-38",\n      "filename": "1-21934-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-21934-A-38.wav",\n      "bytes": 441044,\n      "sha256": "8d26e40e0e382b0d2d7f6dafcb132cfe703ee3f7f32780e878fdb5a1aaffcc28",\n      "decoded_sha256": "783329baa2d2d6c66fde645b8bc6f217ad7e88c0d952aaea9e0d59f416d19d2c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "21934",\n      "source_id": "21934",\n      "attribution": "- [1-21934-A.ogg]: clip derived from 20060810.peters.clock.01.flac (http://www.freesound.org/people/dobroide/sounds/21934/) by dobroide [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-21935-A-38",\n      "filename": "1-21935-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-21935-A-38.wav",\n      "bytes": 441044,\n      "sha256": "4d7ad80cf2733e2d9260061afbbdb0bd3d64cfcbc3df5c8a0258639716a7329d",\n      "decoded_sha256": "b8797576301341d80c32c0ceb8254a14fdf278c59905ac96dea024b4d977ba38",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "21935",\n      "source_id": "21935",\n      "attribution": "- [1-21935-A.ogg]: clip derived from 20060810.peters.clock.02.flac (http://www.freesound.org/people/dobroide/sounds/21935/) by dobroide [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-22694-A-20",\n      "filename": "1-22694-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-22694-A-20.wav",\n      "bytes": 441044,\n      "sha256": "e97893bf8a391f769576ee70497ba51640d3bc646cbf20e719f6d22cc033b2e3",\n      "decoded_sha256": "de416215c1f6b9cfc92a60205e2880f85888604a5e8936865e7c8c2fc3cf99f3",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "22694",\n      "source_id": "22694",\n      "attribution": "- [1-22694-A.ogg]: clip derived from baby2.mp3 (http://www.freesound.org/people/Erdie/sounds/22694/) by Erdie [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-22694-B-20",\n      "filename": "1-22694-B-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-22694-B-20.wav",\n      "bytes": 441044,\n      "sha256": "cf7f480ac6fe0f61ed436d230200ca160f0c936c70009f3622b71c432af8b5ce",\n      "decoded_sha256": "d2372c0172a991b0845bf9a7d44080bd5140c43c7fdcace6c2bdc9d30e0ffa05",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "22694",\n      "source_id": "22694",\n      "attribution": "- [1-22694-B.ogg]: clip derived from baby2.mp3 (http://www.freesound.org/people/Erdie/sounds/22694/) by Erdie [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-26143-A-21",\n      "filename": "1-26143-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-26143-A-21.wav",\n      "bytes": 441044,\n      "sha256": "8e32762c8c48e5414dbecc2979833748cb838ba476bac75d2a512262ef842eb0",\n      "decoded_sha256": "6a153197ca327a8c84c826669f9c7a8262a77ad80c2f1fcdaa5e0130ae8f8eb0",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "26143",\n      "source_id": "26143",\n      "attribution": "- [1-26143-A.ogg]: clip derived from hatsjoe.wav (http://www.freesound.org/people/WIM/sounds/26143/) by WIM [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-26222-A-10",\n      "filename": "1-26222-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-26222-A-10.wav",\n      "bytes": 441044,\n      "sha256": "7a03695543a56d61464ea2bd87e0947bb994d38c914492c02be686693b6e1dac",\n      "decoded_sha256": "5e896369478c0fe43f04e8573b0bb032108ae1c65e01028f184053afd29c779c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "26222",\n      "source_id": "26222",\n      "attribution": "- [1-26222-A.ogg]: clip derived from medium heavey rain.wav (http://www.freesound.org/people/Walter_Odington/sounds/26222/) by Walter_Odington [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-26806-A-1",\n      "filename": "1-26806-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-26806-A-1.wav",\n      "bytes": 441044,\n      "sha256": "9021db2a8218814e1f90957a179589c7f3167fd1f235f58019890064e23ee5be",\n      "decoded_sha256": "379469b4292f9190742d1bc583b2fed8d1273b943524440ccadbd5371fbcbd26",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "26806",\n      "source_id": "26806",\n      "attribution": "- [1-26806-A.ogg]: clip derived from crows.aiff (http://www.freesound.org/people/jimblodget/sounds/26806/) by jimblodget [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-27724-A-1",\n      "filename": "1-27724-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-27724-A-1.wav",\n      "bytes": 441044,\n      "sha256": "3bb615f722d94caa848f64bf4da365fde52dc93a732b0d017d38fe9e1474649f",\n      "decoded_sha256": "c5c2e84ef3a4f934746a27c941bb3a80adf9291dfe769ba356a48e8ba61d7093",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "27724",\n      "source_id": "27724",\n      "attribution": "- [1-27724-A.ogg]: clip derived from citadel park gand.wav (http://www.freesound.org/people/philippe%20b/sounds/27724/) by philippe b [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-28135-A-11",\n      "filename": "1-28135-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-28135-A-11.wav",\n      "bytes": 441044,\n      "sha256": "4d04b84d490c4d9ab28a52c4c96d1b14c2f8b132fc4be38166197ae814920742",\n      "decoded_sha256": "1c68d20130e686a702858f19a5975254b8beba7f9b77fb534457e91a3b6ea751",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "28135",\n      "source_id": "28135",\n      "attribution": "- [1-28135-A.ogg]: clip derived from Branding_kort.wav (http://www.freesound.org/people/HerbertBoland/sounds/28135/) by HerbertBoland [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-28135-B-11",\n      "filename": "1-28135-B-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-28135-B-11.wav",\n      "bytes": 441044,\n      "sha256": "806548ec876df9f91ced5503f76794a42dca6b8790324f87464de5ed780189cb",\n      "decoded_sha256": "6941c7c70bb3c8c19a66cc2bca3fb2e2b24da2cf9b572203fcc93f6ab815d572",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "28135",\n      "source_id": "28135",\n      "attribution": "- [1-28135-B.ogg]: clip derived from Branding_kort.wav (http://www.freesound.org/people/HerbertBoland/sounds/28135/) by HerbertBoland [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-29561-A-10",\n      "filename": "1-29561-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-29561-A-10.wav",\n      "bytes": 441044,\n      "sha256": "4dc600962b53709664b27073f46ef2ad7d766ddad39c3ff04fcaf98442aae65b",\n      "decoded_sha256": "9a6137919cf0348160e714208eab9dd86843b384a92827c52ab58cd5b4d05ed9",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "29561",\n      "source_id": "29561",\n      "attribution": "- [1-29561-A.ogg]: clip derived from rain.aif (http://www.freesound.org/people/palem/sounds/29561/) by palem [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-29680-A-21",\n      "filename": "1-29680-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-29680-A-21.wav",\n      "bytes": 441044,\n      "sha256": "4c1cdb30db5fd43eb62c17c051efaf7dde38938c0449e6bfcfb998dba92cc443",\n      "decoded_sha256": "040626a60e41ffd40ea3e4a92c1269b57e2da8f0c75548504c5037897cb8fd99",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "29680",\n      "source_id": "29680",\n      "attribution": "- [1-29680-A.ogg]: clip derived from Sneeze_.wav (http://www.freesound.org/people/HerbertBoland/sounds/29680/) by HerbertBoland [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-30226-A-0",\n      "filename": "1-30226-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-30226-A-0.wav",\n      "bytes": 441044,\n      "sha256": "ca2626d2b49c7c20a3e0f9f8d7aa56f4497643c6bc264ff37764392be2b15753",\n      "decoded_sha256": "0102eaa49137eae4fe144c5220bb817817b2ce4e5c430695d787bb034eab3cec",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "30226",\n      "source_id": "30226",\n      "attribution": "- [1-30226-A.ogg]: clip derived from dogs.aiff (http://www.freesound.org/people/Heigh-hoo/sounds/30226/) by Heigh-hoo [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-30344-A-0",\n      "filename": "1-30344-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-30344-A-0.wav",\n      "bytes": 441044,\n      "sha256": "52a876cff473fe9b6deb4feea9545a2b976e63e03744921afc72f9800758ce30",\n      "decoded_sha256": "7bf2b2cbc1487f58999dab54bbcd85dfed903676a33a511d9e33638e414397ea",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "30344",\n      "source_id": "30344",\n      "attribution": "- [1-30344-A.ogg]: clip derived from My Dog George.wav (http://www.freesound.org/people/ronfont/sounds/30344/) by ronfont [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-31748-A-21",\n      "filename": "1-31748-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-31748-A-21.wav",\n      "bytes": 441044,\n      "sha256": "865652b27db628cd353379f222c01a48711f383d16b3d80bcd8493e6e8432a9e",\n      "decoded_sha256": "80199a624b65995c84b77e89c1890ba0ec55ba025b7d0fcf1c603f8ff18ffa68",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "31748",\n      "source_id": "31748",\n      "attribution": "- [1-31748-A.ogg]: clip derived from Sneezing.wav (http://www.freesound.org/people/digifishmusic/sounds/31748/) by digifishmusic [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-32318-A-0",\n      "filename": "1-32318-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-32318-A-0.wav",\n      "bytes": 441044,\n      "sha256": "0e96f0bab8bbba81c98a8d7741cf258c8381c6481bac36961b8603a600a6c321",\n      "decoded_sha256": "fa403a260752b47f99edd10e90d80e712af8daf558e15f2282aeb989da74a067",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "32318",\n      "source_id": "32318",\n      "attribution": "- [1-32318-A.ogg]: clip derived from MABEL 1.aif (http://www.freesound.org/people/Bobby%20Beck/sounds/32318/) by Bobby Beck [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-34119-A-1",\n      "filename": "1-34119-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-34119-A-1.wav",\n      "bytes": 441044,\n      "sha256": "4085bfea267b1ecfcdd95b666e2f0c6a5fdf152794740834a24c54c75967f5fe",\n      "decoded_sha256": "80e59df303ac419b2a2156532b79e6042caf77f76cf5a7eb2aab025ee28fa423",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "34119",\n      "source_id": "34119",\n      "attribution": "- [1-34119-A.ogg]: clip derived from cockeril.wav (http://www.freesound.org/people/Charliefarley/sounds/34119/) by Charliefarley [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-34119-B-1",\n      "filename": "1-34119-B-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-34119-B-1.wav",\n      "bytes": 441044,\n      "sha256": "55c3e6b3ce6299d1c1675fc9655327aa2dbc2130d87c08f5f38cac7d33e336cc",\n      "decoded_sha256": "1161390d0fac73368208c6b7498c8dc1a1a37ee48a3e1e80401b6fef42d5794b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "34119",\n      "source_id": "34119",\n      "attribution": "- [1-34119-B.ogg]: clip derived from cockeril.wav (http://www.freesound.org/people/Charliefarley/sounds/34119/) by Charliefarley [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-35687-A-38",\n      "filename": "1-35687-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-35687-A-38.wav",\n      "bytes": 441044,\n      "sha256": "48f0b86e08a4349f66efed2b56405ba5b2b04cc01b8b5701465d9f283d4b5d8b",\n      "decoded_sha256": "15bdb92ae73ea1a070f5c5191588ec50e811cc9f530396afa48de770067ee649",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "35687",\n      "source_id": "35687",\n      "attribution": "- [1-35687-A.ogg]: clip derived from Clock-grandfather-ticks & striking once.MP3 (http://www.freesound.org/people/Bansemer/sounds/35687/) by Bansemer [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-39901-A-11",\n      "filename": "1-39901-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-39901-A-11.wav",\n      "bytes": 441044,\n      "sha256": "b1e33d5fe73bc31317fa9fa4e70d0fcaf761346f8a908c89cec50c6c23f41603",\n      "decoded_sha256": "4342f295aceaa8cbacba890f43dc2923b33a51d85611983b6244d117e4f9f93d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "39901",\n      "source_id": "39901",\n      "attribution": "- [1-39901-A.ogg]: clip derived from sea_water_passing_through_pier_hole_01.flac (http://www.freesound.org/people/aesqe/sounds/39901/) by aesqe [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-39901-B-11",\n      "filename": "1-39901-B-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-39901-B-11.wav",\n      "bytes": 441044,\n      "sha256": "582d4c541bb4bdbc300ccbbaa52f120931a849b1616a4eb6135cf32c5ba8c517",\n      "decoded_sha256": "184dd9f50161ae9532007a941bf0fe6824d3ffbb000efb862f1bd1cf8b5ca5d6",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "39901",\n      "source_id": "39901",\n      "attribution": "- [1-39901-B.ogg]: clip derived from sea_water_passing_through_pier_hole_01.flac (http://www.freesound.org/people/aesqe/sounds/39901/) by aesqe [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-39923-A-1",\n      "filename": "1-39923-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-39923-A-1.wav",\n      "bytes": 441044,\n      "sha256": "62379ace4cf58d0625a09fa8cf943ce8ab5a99602c20c6f96bacd522e6903e06",\n      "decoded_sha256": "2e5b1bca46a2c255b8287b8a254c938bbf800da41a1f6f3ab3e1d09c3fdd8651",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "39923",\n      "source_id": "39923",\n      "attribution": "- [1-39923-A.ogg]: clip derived from 20070812.rooster.wav (http://www.freesound.org/people/dobroide/sounds/39923/) by dobroide [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-40730-A-1",\n      "filename": "1-40730-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-40730-A-1.wav",\n      "bytes": 441044,\n      "sha256": "5b6f2f3795ac7b2711c39742a77ae7b81017aad43d0a226896cc5d78c87c1b92",\n      "decoded_sha256": "6255f55c686ded274a6759ce0122f8c0d000d31d883680d6efc6c68bf6c5ad1f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "40730",\n      "source_id": "40730",\n      "attribution": "- [1-40730-A.ogg]: clip derived from rooster2.wav (http://www.freesound.org/people/pionium/sounds/40730/) by pionium [CC-Sampling+]",\n      "original_license": "CC-Sampling+"\n    },\n    {\n      "id": "1-4211-A-12",\n      "filename": "1-4211-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-4211-A-12.wav",\n      "bytes": 441044,\n      "sha256": "6e8d15a855d1622c1f7dcd12e00ba2bfc3b65a316947e1bbdcaa0be0157175db",\n      "decoded_sha256": "0367d4dc424628b853414254836d875e283f56442d61e93ec8b8d1996991d4b0",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "4211",\n      "source_id": "4211",\n      "attribution": "- [1-4211-A.ogg]:  clip derived from fire.crackling.mp3 (http://www.freesound.org/people/dobroide/sounds/4211/) by dobroide [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-42139-A-38",\n      "filename": "1-42139-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-42139-A-38.wav",\n      "bytes": 441044,\n      "sha256": "169e283f52931f7de0c752e59a5b5149c96fc33ed7323031fe22392e1abab283",\n      "decoded_sha256": "9afde3b4c10946e3e15de9b51a2fedfbb7307bc344773712a6034ec7db09eef1",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "42139",\n      "source_id": "42139",\n      "attribution": "- [1-42139-A.ogg]: clip derived from Old Scots Clock - Ticking.wav (http://www.freesound.org/people/fauxpress/sounds/42139/) by fauxpress [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-43382-A-1",\n      "filename": "1-43382-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-43382-A-1.wav",\n      "bytes": 441044,\n      "sha256": "95fff21655b9945a12e73e69c1d1e5015b2f2802bde43bc7f76351b695cf9901",\n      "decoded_sha256": "5875e1342d2e4c1ca5a9c678661729585c26fd65577c3dd255ead5aabde3013d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "43382",\n      "source_id": "43382",\n      "attribution": "- [1-43382-A.ogg]: clip derived from Rooster chicken calls_2.wav (http://www.freesound.org/people/AGFX/sounds/43382/) by AGFX [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-43760-A-11",\n      "filename": "1-43760-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-43760-A-11.wav",\n      "bytes": 441044,\n      "sha256": "efd808d11d8df8f75f46b7d4183baf082abce9948985c4fca02e191d3204be10",\n      "decoded_sha256": "cab383ad3e91cb3dac2b8d5c2c51f03d2659ec6de680ab6afc2f38ec5afced6b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "43760",\n      "source_id": "43760",\n      "attribution": "- [1-43760-A.ogg]: clip derived from Gentle Sea On Flat Beach.wav (http://www.freesound.org/people/digifishmusic/sounds/43760/) by digifishmusic [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-44831-A-1",\n      "filename": "1-44831-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-44831-A-1.wav",\n      "bytes": 441044,\n      "sha256": "f8958a100e0b005c1421eac9b114d4dff6b0d7d58caf24ba933a2c6ae5343d26",\n      "decoded_sha256": "2153b1328fb9ec6b3de20288d34f0ab1590a6cedb6a47fa7e194f63ff882cf63",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "44831",\n      "source_id": "44831",\n      "attribution": "- [1-44831-A.ogg]: clip derived from Animal Conversation.mp3 (http://www.freesound.org/people/fauxpress/sounds/44831/) by fauxpress [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-46272-A-12",\n      "filename": "1-46272-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-46272-A-12.wav",\n      "bytes": 441044,\n      "sha256": "d48f5f4994b0ac7bbd4983f7c4cb3bdee0a3fe15bdbd0ac8f0fba7450d75c678",\n      "decoded_sha256": "316375e140b73d91c7da8c82580c1dfca3fcd5e75c846c441fba1e92ea82cccc",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "46272",\n      "source_id": "46272",\n      "attribution": "- [1-46272-A.ogg]: clip derived from fire_medium_loop.wav (http://www.freesound.org/people/PhreaKsAccount/sounds/46272/) by PhreaKsAccount [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-47250-A-41",\n      "filename": "1-47250-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-47250-A-41.wav",\n      "bytes": 441044,\n      "sha256": "81312e10f426ae6312c29a97fd4227ef36bcae323c520ddff4b8e890d458772d",\n      "decoded_sha256": "c673965a34ab11a1be024d07a6591e07076f69fd0e3ab1901126ed4b5bcb9fac",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "47250",\n      "source_id": "47250",\n      "attribution": "- [1-47250-A.ogg]: clip derived from chainsaw.WAV (http://www.freesound.org/people/stomachache/sounds/47250/) by stomachache [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-47250-B-41",\n      "filename": "1-47250-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-47250-B-41.wav",\n      "bytes": 441044,\n      "sha256": "25cd4254bfe2be5d1d4674161a8dbc16635fd658cdd1b7bac80510099911f3c5",\n      "decoded_sha256": "b7d27f2e9074f84079a484a265b594edc39692aa3e99059d8002c746fef3484b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "47250",\n      "source_id": "47250",\n      "attribution": "- [1-47250-B.ogg]: clip derived from chainsaw.WAV (http://www.freesound.org/people/stomachache/sounds/47250/) by stomachache [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-47273-A-21",\n      "filename": "1-47273-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-47273-A-21.wav",\n      "bytes": 441044,\n      "sha256": "a77b4c35c9d4a66bdbe2a677a43b8f646579ce5773c59cf721bcb9db419d1b2d",\n      "decoded_sha256": "7f1959f7f444a14f377d7cf3b98e7bf0a71f1ecc7bc5dda457ec1a80d6a4feb9",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "47273",\n      "source_id": "47273",\n      "attribution": "- [1-47273-A.ogg]: clip derived from baby_sneeze.wav (http://www.freesound.org/people/reinsamba/sounds/47273/) by reinsamba [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-47274-A-21",\n      "filename": "1-47274-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-47274-A-21.wav",\n      "bytes": 441044,\n      "sha256": "c7114b4b84b83598dfacbf7d1774f34fb61ba87af426944db4ad8ade82c7779b",\n      "decoded_sha256": "a26329948c2bd648685e6c71ae9ac27a0e718acc3f1f3bf150c03ad1a9ad5c25",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "47274",\n      "source_id": "47274",\n      "attribution": "- [1-47274-A.ogg]: clip derived from male_sneeze.wav (http://www.freesound.org/people/reinsamba/sounds/47274/) by reinsamba [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-48413-A-38",\n      "filename": "1-48413-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-48413-A-38.wav",\n      "bytes": 441044,\n      "sha256": "e8aebc842de3e00fb0ba00f085e482fa8e9e43a4398d646341434e6c0f501bc3",\n      "decoded_sha256": "3838aa494707c2f37cf2b222b0bf7bfbf25ce9338a9aea278340a68b9a283750",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "48413",\n      "source_id": "48413",\n      "attribution": "- [1-48413-A.ogg]: clip derived from clock.wav (http://www.freesound.org/people/cynical/sounds/48413/) by cynical [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-50060-A-10",\n      "filename": "1-50060-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-50060-A-10.wav",\n      "bytes": 441044,\n      "sha256": "cc8119be878eafdc69f7408dad614493a238174372eebf75e4c8314d1bb9807d",\n      "decoded_sha256": "6e9c96094cbd529a2726d00e88ebdc83181276f85b5bc3e51fe770d979b360bf",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "50060",\n      "source_id": "50060",\n      "attribution": "- [1-50060-A.ogg]: clip derived from rain_session_27sec_2006.wav (http://www.freesound.org/people/vibe_crc/sounds/50060/) by vibe_crc [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-54505-A-21",\n      "filename": "1-54505-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-54505-A-21.wav",\n      "bytes": 441044,\n      "sha256": "94c62d8761cb68e3a7ee6c7c64ef5bdb4f14b5b32a5ae0d792cd7eebfdcb38c0",\n      "decoded_sha256": "e5140249ac8a5fc55cfbc38526b5b4c09633fcd2e5ea3661d446c5e33bd5721c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "54505",\n      "source_id": "54505",\n      "attribution": "- [1-54505-A.ogg]: clip derived from sneeze.wav (http://www.freesound.org/people/sagetyrtle/sounds/54505/) by sagetyrtle [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-54958-A-10",\n      "filename": "1-54958-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-54958-A-10.wav",\n      "bytes": 441044,\n      "sha256": "d93233d42dc57a3e81cb9ca446145f50cdc806f0174393b1055e5d9a327cef86",\n      "decoded_sha256": "0ac9ab7fadf569e2d90430b31e1c1cc9958e7e9c0d88ac08ab36823f9a6d8025",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "54958",\n      "source_id": "54958",\n      "attribution": "- [1-54958-A.ogg]: clip derived from rain.ogg (http://www.freesound.org/people/thammo/sounds/54958/) by thammo [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-56311-A-10",\n      "filename": "1-56311-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-56311-A-10.wav",\n      "bytes": 441044,\n      "sha256": "cf0dcbd49d979bf73992068b51db46cf3098995e66b3fb8ceb6ec276009aadc7",\n      "decoded_sha256": "79f76d0f69c6c3c1c6b12df0666c7e8e5d5786848428df32d41fcd121ed8d978",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "56311",\n      "source_id": "56311",\n      "attribution": "- [1-56311-A.ogg]: clip derived from Rain_06.wav (http://www.freesound.org/people/Q.K./sounds/56311/) by Q.K. [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-57163-A-38",\n      "filename": "1-57163-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-57163-A-38.wav",\n      "bytes": 441044,\n      "sha256": "be4ee5f3dbf6cfdf65a4afa6b0dc98914e5e36728d1d3778039de70f21579303",\n      "decoded_sha256": "1a1815be1a174005ff8f9f4bed9c9b475fa8c246275bef58b851cc3b71a50d29",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "57163",\n      "source_id": "57163",\n      "attribution": "- [1-57163-A.ogg]: clip derived from delphis CLOCK FX LOOP 5.wav (http://www.freesound.org/people/delphidebrain/sounds/57163/) by delphidebrain [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-59324-A-21",\n      "filename": "1-59324-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-59324-A-21.wav",\n      "bytes": 441044,\n      "sha256": "fe73df75398008f02288e9b4dfeb267814e357d9290339e5cea3dfaf03fdce74",\n      "decoded_sha256": "8e681dfa2cf09e35f6caa358a6d6cf5b45a5f04d6dd8030263d3a42763860c89",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "59324",\n      "source_id": "59324",\n      "attribution": "- [1-59324-A.ogg]: clip derived from Sneeze.wav (http://www.freesound.org/people/transitking/sounds/59324/) by transitking [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-59513-A-0",\n      "filename": "1-59513-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-59513-A-0.wav",\n      "bytes": 441044,\n      "sha256": "b573fcbb4d3a8a05a857dc1ffdb09c687253ff3a03baa326460e291ce4cbfee7",\n      "decoded_sha256": "798cda7c53c8e03acf1eadcd39afef1f66594ba3ff83c276c303f24d751913aa",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "59513",\n      "source_id": "59513",\n      "attribution": "- [1-59513-A.ogg]: clip derived from Blaffende hond2.wav (http://www.freesound.org/people/zeshoog/sounds/59513/) by zeshoog [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-60997-A-20",\n      "filename": "1-60997-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-60997-A-20.wav",\n      "bytes": 441044,\n      "sha256": "6e225fdade2a26e8477eab8f3d9058537cd919de78fd1a2dcb029dcc014ce744",\n      "decoded_sha256": "dce32dae0453b3ecb5bd883808c782b448f88c1e6bd38a35a9060fbf96bd1fb7",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "60997",\n      "source_id": "60997",\n      "attribution": "- [1-60997-A.ogg]: clip derived from 01-Baby_Crying_small-room.wav (http://www.freesound.org/people/poissonmort/sounds/60997/) by poissonmort [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-60997-B-20",\n      "filename": "1-60997-B-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-60997-B-20.wav",\n      "bytes": 441044,\n      "sha256": "2de04b4d958ab53a2f66653f66bc640757bb07cacbb1a614c28777f505c9aa20",\n      "decoded_sha256": "0ddddc43d9ce1fce9530f3c2141204b382663a8ed3006f567ed4a65a40f4338c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "60997",\n      "source_id": "60997",\n      "attribution": "- [1-60997-B.ogg]: clip derived from 01-Baby_Crying_small-room.wav (http://www.freesound.org/people/poissonmort/sounds/60997/) by poissonmort [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-61252-A-11",\n      "filename": "1-61252-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-61252-A-11.wav",\n      "bytes": 441044,\n      "sha256": "fb76a05332ed37fa76ef09b878d28831d5c433554d317893a9911384bd84082b",\n      "decoded_sha256": "1061d05c71ea73fc8bd104eae2af1755334f6a26c5108343d54603b27d82b465",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "61252",\n      "source_id": "61252",\n      "attribution": "- [1-61252-A.ogg]: clip derived from mediterranean_sea_porticcio.flac (http://www.freesound.org/people/Corsica_S/sounds/61252/) by Corsica_S [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-62849-A-38",\n      "filename": "1-62849-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-62849-A-38.wav",\n      "bytes": 441044,\n      "sha256": "a78412e271e9bac017201211a97f8e67278d157f980c62a9680bdab90d444785",\n      "decoded_sha256": "d7a150df044b8ccac8a05a6f00280fe3e2c5216548e13076e5ef9068640f4f86",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "62849",\n      "source_id": "62849",\n      "attribution": "- [1-62849-A.ogg]: clip derived from clock_tick_close.wav (http://www.freesound.org/people/life_interconnected/sounds/62849/) by life_interconnected [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-62850-A-38",\n      "filename": "1-62850-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-62850-A-38.wav",\n      "bytes": 441044,\n      "sha256": "62077d3dbf18bb92558998085d84778a6dd7390574c3080ef7b600158f2fd984",\n      "decoded_sha256": "47a70c44a89800042ae65f7f57be64d73f06accbc8bdb1aee5706ecfbdc6e0d8",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "62850",\n      "source_id": "62850",\n      "attribution": "- [1-62850-A.ogg]: clip derived from clock_tick_realistic.wav (http://www.freesound.org/people/life_interconnected/sounds/62850/) by life_interconnected [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-63871-A-10",\n      "filename": "1-63871-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-63871-A-10.wav",\n      "bytes": 441044,\n      "sha256": "9ae1461223eaecf2df3818b51e0d7b5987ef497b17038594455bbcee6673d350",\n      "decoded_sha256": "0753efdfbb7c2cc6ebda625a0f5ef9ca4e3450271d0c49167d7c00d51c20bca5",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "63871",\n      "source_id": "63871",\n      "attribution": "- [1-63871-A.ogg]: clip derived from rain.mp3 (http://www.freesound.org/people/liliangorini/sounds/63871/) by liliangorini [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "1-64398-A-41",\n      "filename": "1-64398-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-64398-A-41.wav",\n      "bytes": 441044,\n      "sha256": "0bb8b8988b6d398e67e0f896d5acbf28adf1d4bec9d00cb45fb00d430389ad95",\n      "decoded_sha256": "c11b7ecc0d218af907f34c16d660dfbbe85bfb7be0566650f1ddca293a47ecd0",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "64398",\n      "source_id": "64398",\n      "attribution": "- [1-64398-A.ogg]: clip derived from Chainsaw Crosscutting  3.wav (http://www.freesound.org/people/Benboncan/sounds/64398/) by Benboncan [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-64398-B-41",\n      "filename": "1-64398-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-64398-B-41.wav",\n      "bytes": 441044,\n      "sha256": "fc460220cde3dc9a080da0c1742d38755a25870463bc34d66aec65eb16050fb5",\n      "decoded_sha256": "4a389501d8cafebed92c911683c204fe1c91be55ce2c74ad807a6f589d605370",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "64398",\n      "source_id": "64398",\n      "attribution": "- [1-64398-B.ogg]: clip derived from Chainsaw Crosscutting  3.wav (http://www.freesound.org/people/Benboncan/sounds/64398/) by Benboncan [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-7057-A-12",\n      "filename": "1-7057-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-7057-A-12.wav",\n      "bytes": 441044,\n      "sha256": "f92e52f9ab78d700dc10bcbc36020082613e669db245961b3205e8394c075da6",\n      "decoded_sha256": "f4ee1d7e71c03941239b659e765832a3b1ca6396fcc67ef42acd62d5af25ed95",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "7057",\n      "source_id": "7057",\n      "attribution": "- [1-7057-A.ogg]:  clip derived from fire.wav (http://www.freesound.org/people/mystiscool/sounds/7057/) by mystiscool [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-81883-A-21",\n      "filename": "1-81883-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-81883-A-21.wav",\n      "bytes": 441044,\n      "sha256": "1d3237552ea143411a563b9e886f870df5870d31cb3272ae33f2588ba5076fe7",\n      "decoded_sha256": "82e931c0141c4f6ef789b0b25897030d28fa724ed7b8c1ca8f2b85896e5883a0",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "81883",\n      "source_id": "81883",\n      "attribution": "- [1-81883-A.ogg]: clip derived from sneeze.wav (http://www.freesound.org/people/stevenpam/sounds/81883/) by stevenpam [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-85362-A-0",\n      "filename": "1-85362-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-85362-A-0.wav",\n      "bytes": 441044,\n      "sha256": "2f7742933d677b0dc8d77b802e90ee0fb94b5828183ca96f7a8dd8d24ab57b7b",\n      "decoded_sha256": "331733b7eea0a3cfd5f441eba15c1b3a807c465c0ada884ff3ffb1b82e9f7b99",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "85362",\n      "source_id": "85362",\n      "attribution": "- [1-85362-A.ogg]: clip derived from 20091211.barking.stairs.wav (http://www.freesound.org/people/dobroide/sounds/85362/) by dobroide [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-91359-A-11",\n      "filename": "1-91359-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-91359-A-11.wav",\n      "bytes": 441044,\n      "sha256": "dc68614d5082a595bce514b13304d91451cc6d570dddfb84a9283e5d77fc3148",\n      "decoded_sha256": "23f840b30cb34c62cec965dcc44d4a828abcc0ce5d88f2f1ad2176ee642b419f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "91359",\n      "source_id": "91359",\n      "attribution": "- [1-91359-A.ogg]: clip derived from sea.aif (http://www.freesound.org/people/ariat/sounds/91359/) by ariat [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-91359-B-11",\n      "filename": "1-91359-B-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-91359-B-11.wav",\n      "bytes": 441044,\n      "sha256": "119e12872a9ec159c255ddaab2c8382439a97815ce90552273529d53f86ee8ff",\n      "decoded_sha256": "f8a0beeeea9cf4c6cd30aae2af9fec6dfedb3900f4afd1e7c5fd9bf207e6b637",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "91359",\n      "source_id": "91359",\n      "attribution": "- [1-91359-B.ogg]: clip derived from sea.aif (http://www.freesound.org/people/ariat/sounds/91359/) by ariat [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "1-97392-A-0",\n      "filename": "1-97392-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/1-97392-A-0.wav",\n      "bytes": 441044,\n      "sha256": "9ec2eb304d2a1ab3a50150a2a558dce0771e4ffc920d074e107fd76cb7b5d28c",\n      "decoded_sha256": "1b302c4d60b728b419f35943245e93ffe35954717ece813620b6ee215c9ca028",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 1,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "97392",\n      "source_id": "97392",\n      "attribution": "- [1-97392-A.ogg]: clip derived from dog barking.wav (http://www.freesound.org/people/Tomlija/sounds/97392/) by Tomlija [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-100786-A-1",\n      "filename": "2-100786-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-100786-A-1.wav",\n      "bytes": 441044,\n      "sha256": "53704d16503042581837335ba937d399c028f788903f103abf44af1c2d2d5d71",\n      "decoded_sha256": "4f66d55bcb1197df803b7303860128527ab334e1bb562077713ce148bd4c7c33",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "100786",\n      "source_id": "100786",\n      "attribution": "- [2-100786-A.ogg]: clip derived from rooster 2.aif (http://www.freesound.org/people/Tomlija/sounds/100786/) by Tomlija [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-101676-A-10",\n      "filename": "2-101676-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-101676-A-10.wav",\n      "bytes": 441044,\n      "sha256": "ef456a67e12d2e2c84513e9eb8bea769e343cd5e0bfe65a34dc96c526f260838",\n      "decoded_sha256": "7c637012efd07ba81fa4354aab3ff9569973f973376c28ffbd8196fb40de9f63",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "101676",\n      "source_id": "101676",\n      "attribution": "- [2-101676-A.ogg]: clip derived from rain, sm57.wav (http://www.freesound.org/people/nialldoran/sounds/101676/) by nialldoran [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-102852-A-11",\n      "filename": "2-102852-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-102852-A-11.wav",\n      "bytes": 441044,\n      "sha256": "88e893f0db59886dd96f9b2c0f839eaad0364e9d2ec948eca3ef6af1a66e64b5",\n      "decoded_sha256": "d320adcd48679bdeae5fe0749891955e4b46497236c708890d9fda72db9da8c1",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "102852",\n      "source_id": "102852",\n      "attribution": "- [2-102852-A.ogg]: clip derived from 20100803.surf.seagulls.wav (http://www.freesound.org/people/dobroide/sounds/102852/) by dobroide [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-107351-A-20",\n      "filename": "2-107351-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-107351-A-20.wav",\n      "bytes": 441044,\n      "sha256": "f89a24c9494eab238d80013b9409e6da4058b8097c665a789e48de1aef1bcef5",\n      "decoded_sha256": "a3a87a577cd519e59e32554d1c4f456cf331728e122d179e778501aade2b9401",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "107351",\n      "source_id": "107351",\n      "attribution": "- [2-107351-A.ogg]: clip derived from baby-crying-1_(sounds-AT-ermfilm_pl).wav (http://www.freesound.org/people/ermfilm/sounds/107351/) by ermfilm [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-107351-B-20",\n      "filename": "2-107351-B-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-107351-B-20.wav",\n      "bytes": 441044,\n      "sha256": "ebb3365010813cec4b951d1294d3376fc598116a1f86115a3e279ea013160a6a",\n      "decoded_sha256": "91c53426f414d17940f1094560733dfff198a605003fad3f827bcfc1969e339f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "107351",\n      "source_id": "107351",\n      "attribution": "- [2-107351-B.ogg]: clip derived from baby-crying-1_(sounds-AT-ermfilm_pl).wav (http://www.freesound.org/people/ermfilm/sounds/107351/) by ermfilm [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-109505-A-21",\n      "filename": "2-109505-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-109505-A-21.wav",\n      "bytes": 441044,\n      "sha256": "93f6ffc756ec82bf5e6be83ccc070415138b60fa00408734a78753089567e489",\n      "decoded_sha256": "b8a484601e0b1704a83f247af5e246711c4b4e228b0e875995aa4cf66dab281d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "109505",\n      "source_id": "109505",\n      "attribution": "- [2-109505-A.ogg]: clip derived from atishoo.wav (http://www.freesound.org/people/jeanorwin/sounds/109505/) by jeanorwin [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-114280-A-0",\n      "filename": "2-114280-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-114280-A-0.wav",\n      "bytes": 441044,\n      "sha256": "23cd06e6c8b2e054d13ad6392d34d91588cd8572041d6f4c1c337048fd5deb69",\n      "decoded_sha256": "5ae8adfe58dcf832b286572c350bca2c4c006303357e0d7f3c3020b9fafb31d7",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "114280",\n      "source_id": "114280",\n      "attribution": "- [2-114280-A.ogg]: clip derived from dogbarks.wav (http://www.freesound.org/people/plingativator/sounds/114280/) by plingativator [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-114587-A-0",\n      "filename": "2-114587-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-114587-A-0.wav",\n      "bytes": 441044,\n      "sha256": "34d7f9c32c2cda9b6b68483028bb85326259d1c5f2956eac2ecc48b18b06f999",\n      "decoded_sha256": "fcacb40bd72ed1477517f096f680cf886af33599f28263b3344dbb767ed5280e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "114587",\n      "source_id": "114587",\n      "attribution": "- [2-114587-A.ogg]: clip derived from Buddy Bark.wav (http://www.freesound.org/people/InDaHouse20/sounds/114587/) by InDaHouse20 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-116400-A-0",\n      "filename": "2-116400-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-116400-A-0.wav",\n      "bytes": 441044,\n      "sha256": "9830a9ccefad59e7ac9313b86913a6a15e5b4cb281fd479fc6fb076a06ff2a6e",\n      "decoded_sha256": "34967a9c9c9f0d25d252a8e84e5463a2f69149cde82cacc686697686ddaedfcc",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "116400",\n      "source_id": "116400",\n      "attribution": "- [2-116400-A.ogg]: clip derived from 2011-03-09-dog-in-the-night.flac (http://www.freesound.org/people/ayamahambho/sounds/116400/) by ayamahambho [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-117271-A-0",\n      "filename": "2-117271-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-117271-A-0.wav",\n      "bytes": 441044,\n      "sha256": "636eb41093489304bb85802fdf2e8ac8b911500516e210e01f3b0b92f156c01d",\n      "decoded_sha256": "e080f5e0eae638597e970d16fb0c8fcf24e7e4bb32a9316dcdcf32b8f2e17f08",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "117271",\n      "source_id": "117271",\n      "attribution": "- [2-117271-A.ogg]: clip derived from angry_dogs.wav (http://www.freesound.org/people/polipa/sounds/117271/) by polipa [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-117625-A-10",\n      "filename": "2-117625-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-117625-A-10.wav",\n      "bytes": 441044,\n      "sha256": "26c3b6b455aeb91f6f3d9a2b00f7e06bd1fc79609606ea2ae86272657dc02e88",\n      "decoded_sha256": "ee9e0217361e76b44df24d8ecbcabc3478af388dce3bc841002252884e1ae085",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "117625",\n      "source_id": "117625",\n      "attribution": "- [2-117625-A.ogg]: clip derived from Tent - int - rain on canvas.wav (http://www.freesound.org/people/soundmary/sounds/117625/) by soundmary [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-118072-A-0",\n      "filename": "2-118072-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-118072-A-0.wav",\n      "bytes": 441044,\n      "sha256": "c2003f5ce159e77c3a0e52ff042567d159ea8eab6cd3081e64c0b709556dcb4e",\n      "decoded_sha256": "2209c82a213071186bb6ca426fb951305ccd3748585626384afbb90a2069d7f2",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "118072",\n      "source_id": "118072",\n      "attribution": "- [2-118072-A.ogg]: clip derived from dogbark.wav (http://www.freesound.org/people/ipears1/sounds/118072/) by ipears1 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-118104-A-21",\n      "filename": "2-118104-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-118104-A-21.wav",\n      "bytes": 441044,\n      "sha256": "c32ccc6f408d6546132b177036d7a1881bb8f3d9d42f9a2c2a1708c1afc84317",\n      "decoded_sha256": "a1a83cb624b77a65b07a478c1bc1088febb5d957a80c0b97590b46563dd4fd96",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "118104",\n      "source_id": "118104",\n      "attribution": "- [2-118104-A.ogg]: clip derived from Female Sneezing -- Two Sneezes Spaced Out.WAV (http://www.freesound.org/people/Ndheger/sounds/118104/) by Ndheger [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-118964-A-0",\n      "filename": "2-118964-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-118964-A-0.wav",\n      "bytes": 441044,\n      "sha256": "c684bc7d6ff7d8b9dbb7b033760319c4b4aac2ec0ae221df9ba684790a502d87",\n      "decoded_sha256": "8e93e77e7b558a097f0346e48e2bb9580eca249932c8d756c842925eaa141cc4",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "118964",\n      "source_id": "118964",\n      "attribution": "- [2-118964-A.ogg]: clip derived from Dog Bark 4.wav (http://www.freesound.org/people/esperri/sounds/118964/) by esperri [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-119102-A-21",\n      "filename": "2-119102-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-119102-A-21.wav",\n      "bytes": 441044,\n      "sha256": "38fbd04161595637bc147c898db64461961794931a7f1e7202430926a5cb8d0b",\n      "decoded_sha256": "b3f5dfc7f83d4f3c5290d79e50f10e5bf92c9bd8ea852f53017f6945e3c89ad6",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "119102",\n      "source_id": "119102",\n      "attribution": "- [2-119102-A.ogg]: clip derived from Sneeze; male_1-2.aif (http://www.freesound.org/people/domrodrig/sounds/119102/) by domrodrig [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-119748-A-38",\n      "filename": "2-119748-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-119748-A-38.wav",\n      "bytes": 441044,\n      "sha256": "7bb645903e539845643c4162d619f51b3648307fbe1966b19d15f01ceb8ecde5",\n      "decoded_sha256": "b5d3025c87f56f8b37e7a6e3a6c37bda8a3886afd530283fe61cafb35382c53b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "119748",\n      "source_id": "119748",\n      "attribution": "- [2-119748-A.ogg]: clip derived from Clock Ticking.wav (http://www.freesound.org/people/jesabat/sounds/119748/) by jesabat [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-122104-A-0",\n      "filename": "2-122104-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-122104-A-0.wav",\n      "bytes": 441044,\n      "sha256": "0ed7ff66a9d9a570768815d0027803283f091201ec8bf742f466441ad051f736",\n      "decoded_sha256": "0062da9b173c41c83d3685ac5cdf0059eddf9798684fca4305f11ea72de37502",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "122104",\n      "source_id": "122104",\n      "attribution": "- [2-122104-A.ogg]: clip derived from Dog barking at night (nice reflection).wav (http://www.freesound.org/people/rickyjezz/sounds/122104/) by rickyjezz [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-122104-B-0",\n      "filename": "2-122104-B-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-122104-B-0.wav",\n      "bytes": 441044,\n      "sha256": "3cbf9d8b919210c37a2c74706cdeacf99073a83e98ae6a797d31b34e35db9a59",\n      "decoded_sha256": "bbc8fc62b5d230e4691fee96772f0bcc230194528434d4b773cedf88d9ca473b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "122104",\n      "source_id": "122104",\n      "attribution": "- [2-122104-B.ogg]: clip derived from Dog barking at night (nice reflection).wav (http://www.freesound.org/people/rickyjezz/sounds/122104/) by rickyjezz [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-124662-A-11",\n      "filename": "2-124662-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-124662-A-11.wav",\n      "bytes": 441044,\n      "sha256": "60c6883343535fcf14bb030d30c7b79c81fad89fdc7f4e6f34d81c1d98c1e645",\n      "decoded_sha256": "8e04c065bc550c213bdeae1298a5a6d1675b15aac5e34714077bcf1337a68871",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "124662",\n      "source_id": "124662",\n      "attribution": "- [2-124662-A.ogg]: clip derived from Water sea shore.mp3 (http://www.freesound.org/people/Slapbass/sounds/124662/) by Slapbass [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-125966-A-11",\n      "filename": "2-125966-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-125966-A-11.wav",\n      "bytes": 441044,\n      "sha256": "86643b7df512575db786b21b70092f27050b81e01f459f7fcbcb862314827e51",\n      "decoded_sha256": "c4723b0f2a5f899258b9ba0f1b3b56407038ecacc3f9c783ad60790830fcad00",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "125966",\n      "source_id": "125966",\n      "attribution": "- [2-125966-A.ogg]: clip derived from Waves in sea.wav (http://www.freesound.org/people/Ryding/sounds/125966/) by Ryding [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-127108-A-38",\n      "filename": "2-127108-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-127108-A-38.wav",\n      "bytes": 441044,\n      "sha256": "23f4cb882b12e8da2ab06a9b8f2f58affb715007b1846d6a041b0717aeb04ec0",\n      "decoded_sha256": "0b552a179d9aab80590c2155fa5f891340f9d2eca6a4f209da6e4c48ddc3f0b4",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "127108",\n      "source_id": "127108",\n      "attribution": "- [2-127108-A.ogg]: clip derived from grandfather_clock_ticking_60_seconds.wav (http://www.freesound.org/people/joedeshon/sounds/127108/) by joedeshon [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-128631-A-21",\n      "filename": "2-128631-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-128631-A-21.wav",\n      "bytes": 441044,\n      "sha256": "f188f4f7bfe6ade616fcecb2f0c0245a4c6c08996a6714c04c7702edf417e31a",\n      "decoded_sha256": "005cd941747aa2dc6e79a905a533e2f984da9485ed5281b72fee61d4358b8796",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "128631",\n      "source_id": "128631",\n      "attribution": "- [2-128631-A.ogg]: clip derived from sneeze (http://www.freesound.org/people/bigjoedrummer/sounds/128631/) by bigjoedrummer [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-130978-A-21",\n      "filename": "2-130978-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-130978-A-21.wav",\n      "bytes": 441044,\n      "sha256": "c11d9e1ea6af05182568e915c8e237ab6fee643ffb054fcbaa7d2961c1aa584a",\n      "decoded_sha256": "d9814752f7442d9f88966ae06007b310e0738e00f561c192a9d9541cc398941a",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "130978",\n      "source_id": "130978",\n      "attribution": "- [2-130978-A.ogg]: clip derived from Sneeze AC 2.wav (http://www.freesound.org/people/oldedgar/sounds/130978/) by oldedgar [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-130979-A-21",\n      "filename": "2-130979-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-130979-A-21.wav",\n      "bytes": 441044,\n      "sha256": "6d8820a0d6362e67817faa206672b7599ab808877b7df702530b69aa22651813",\n      "decoded_sha256": "1dd31b7af83a0ab3f772e9376e0296d9e901404a56151bd2130bc47356ea985c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "130979",\n      "source_id": "130979",\n      "attribution": "- [2-130979-A.ogg]: clip derived from Sneeze AC 1.wav (http://www.freesound.org/people/oldedgar/sounds/130979/) by oldedgar [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-131943-A-38",\n      "filename": "2-131943-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-131943-A-38.wav",\n      "bytes": 441044,\n      "sha256": "069b85098b561472e16a15929fd1774827a13e0373feae1dc07d0d70d791d581",\n      "decoded_sha256": "2741912a17ad4f37e7d1e57bdb5416157f47530079c6e24027a43ddac6af1756",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "131943",\n      "source_id": "131943",\n      "attribution": "- [2-131943-A.ogg]: clip derived from Wind up clock with twin bell alarm (http://www.freesound.org/people/AngryAdam/sounds/131943/) by AngryAdam [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-132157-A-11",\n      "filename": "2-132157-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-132157-A-11.wav",\n      "bytes": 441044,\n      "sha256": "1925d1067111dd06970446f46a786f261efa68b121039bb5cea4f546f9600685",\n      "decoded_sha256": "494a3a23cbac8243b43205a311e9860c4adede9f9af9736b7e2a1ac738920178",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "132157",\n      "source_id": "132157",\n      "attribution": "- [2-132157-A.ogg]: clip derived from beach sea ocean waves crush breakwater 2.wav (http://www.freesound.org/people/bulbastre/sounds/132157/) by bulbastre [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-132157-B-11",\n      "filename": "2-132157-B-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-132157-B-11.wav",\n      "bytes": 441044,\n      "sha256": "4158aa78341d58d23771deb264c7db18f9a3784b42fc838c77a002cb28e3f784",\n      "decoded_sha256": "6da2cb40be27b7b29be78e50d569aec3baf74453fa209325f9b0a264a812d618",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "132157",\n      "source_id": "132157",\n      "attribution": "- [2-132157-B.ogg]: clip derived from beach sea ocean waves crush breakwater 2.wav (http://www.freesound.org/people/bulbastre/sounds/132157/) by bulbastre [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-133863-A-11",\n      "filename": "2-133863-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-133863-A-11.wav",\n      "bytes": 441044,\n      "sha256": "a5dc90f79b7bfbe1b1b8279d5f5542be82af3fa32502ba57518d1302e9e38a9f",\n      "decoded_sha256": "f579806bdcc9f868febf6048dcf4e724d2bce0789f8d85338e73b60e65c65ce4",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "133863",\n      "source_id": "133863",\n      "attribution": "- [2-133863-A.ogg]: clip derived from EXT RED SEA.wav (http://www.freesound.org/people/vhlam/sounds/133863/) by vhlam [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-134700-A-38",\n      "filename": "2-134700-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-134700-A-38.wav",\n      "bytes": 441044,\n      "sha256": "568eb59736a3e7a59e588f0876a6acc81ecf9eff88179e42a995ff82f572a8db",\n      "decoded_sha256": "75db26fb9e4bb91a2f0fbcdade1932230fa8c2a95d38a4e280266028dbb9dfdc",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "134700",\n      "source_id": "134700",\n      "attribution": "- [2-134700-A.ogg]: clip derived from wall clock tw2.WAV (http://www.freesound.org/people/Setuniman/sounds/134700/) by Setuniman [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-135728-A-38",\n      "filename": "2-135728-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-135728-A-38.wav",\n      "bytes": 441044,\n      "sha256": "3c5b88ddfd575a07515060508e33c810ead374401b0ab16429263fd22fdb3911",\n      "decoded_sha256": "5d4d30ea6fe89813fc043780d217e4618cb9850c66c3a34f8b26f19b154c48f1",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "135728",\n      "source_id": "135728",\n      "attribution": "- [2-135728-A.ogg]: clip derived from tickende-uhr-fewo-leipzig.wav (http://www.freesound.org/people/massenbachhausen/sounds/135728/) by massenbachhausen [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-137162-A-11",\n      "filename": "2-137162-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-137162-A-11.wav",\n      "bytes": 441044,\n      "sha256": "4d9d5bd94694230cc4d40e3c1e1bfc9a91228741fc516946852c0bb8dac6c2de",\n      "decoded_sha256": "d6cda165498261bfcefc7966c05684fa712762db966cd7ab6bf0600b21d3ed1b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "137162",\n      "source_id": "137162",\n      "attribution": "- [2-137162-A.ogg]: clip derived from Sea inside a room (http://www.freesound.org/people/Akoustikos/sounds/137162/) by Akoustikos [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-140147-A-38",\n      "filename": "2-140147-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-140147-A-38.wav",\n      "bytes": 441044,\n      "sha256": "b3a54dc34ab83fa96fc06604e0852973bce5ed9ed872cdc9be92d389e0d5a1ef",\n      "decoded_sha256": "ef9e91ea49ebbc7a1aecdeafaef759ebc827184693c34d079cab066821b64858",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "140147",\n      "source_id": "140147",\n      "attribution": "- [2-140147-A.ogg]: clip derived from Mantel Clock Ticking.wav (http://www.freesound.org/people/Tewkesound/sounds/140147/) by Tewkesound [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-141584-A-38",\n      "filename": "2-141584-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-141584-A-38.wav",\n      "bytes": 441044,\n      "sha256": "b87b57fe9099b0eaf893274145d5237d2da3228dd9b641f473273dcd6759f3a3",\n      "decoded_sha256": "548292fcd8bee844921ce26be5ce5ff8bd1b192c24e9c6e0d771db3ff162dc46",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "141584",\n      "source_id": "141584",\n      "attribution": "- [2-141584-A.ogg]: clip derived from Plastic Quartz Clock Tick (http://www.freesound.org/people/MentalSanityOff/sounds/141584/) by MentalSanityOff [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-151079-A-20",\n      "filename": "2-151079-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-151079-A-20.wav",\n      "bytes": 441044,\n      "sha256": "8a13243f77b12668061f6fb18e820d8398c75d35eca09f9640efdd9df6703499",\n      "decoded_sha256": "850ba3799d28d297dca2a45ca161b819cac599a41cead7a333001daebefd33da",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "151079",\n      "source_id": "151079",\n      "attribution": "- [2-151079-A.ogg]: clip derived from 01 baby with hiccups.wav (http://www.freesound.org/people/PickleJones/sounds/151079/) by PickleJones [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-155801-A-11",\n      "filename": "2-155801-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-155801-A-11.wav",\n      "bytes": 441044,\n      "sha256": "1851566021e7ad54163515236574452dd86cadc4657317f6bc285c9e06a9b436",\n      "decoded_sha256": "387b4a767e110988017811bfd30dead1afc976e8ac31cba7a74e10d3f8aef098",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "155801",\n      "source_id": "155801",\n      "attribution": "- [2-155801-A.ogg]: clip derived from sea, rocky beach 1 (http://freesound.org/people/mario1298/sounds/155801/) by mario1298 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-18766-A-12",\n      "filename": "2-18766-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-18766-A-12.wav",\n      "bytes": 441044,\n      "sha256": "10a98a61a73baee8028c19b9de330261f2dae87f57b2792457d81e46a544697a",\n      "decoded_sha256": "a4c276540fea882a51d80ae6c9acfacc461796c70113b1465498f053cabd40f8",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "18766",\n      "source_id": "18766",\n      "attribution": "- [2-18766-A.ogg]: clip derived from chimney fire.wav (http://www.freesound.org/people/reinsamba/sounds/18766/) by reinsamba [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-18766-B-12",\n      "filename": "2-18766-B-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-18766-B-12.wav",\n      "bytes": 441044,\n      "sha256": "ce9d67f6f6b8d093d3dc2d14088251d76b2772210733908d01b0ecc08c349fe4",\n      "decoded_sha256": "90b1bce7e6a37a6516caa3c3fddf15b8a6f61e5825083284cdb7ea9dbec7f544",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "18766",\n      "source_id": "18766",\n      "attribution": "- [2-18766-B.ogg]: clip derived from chimney fire.wav (http://www.freesound.org/people/reinsamba/sounds/18766/) by reinsamba [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-188822-A-40",\n      "filename": "2-188822-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-188822-A-40.wav",\n      "bytes": 441044,\n      "sha256": "b5a49dab9e1ba8bf7c694ae863d3f73747102cd73b74d3a2060fe66e9796d91d",\n      "decoded_sha256": "62604784b724e4e3c235578c816ae8bed61771d847ae25eedfc001ea69f1fc65",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "188822",\n      "source_id": "188822",\n      "attribution": "- [2-188822-A.ogg]: clip derived from Helicopter landing (http://www.freesound.org/people/peridactyloptrix/sounds/188822/) by peridactyloptrix [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-188822-B-40",\n      "filename": "2-188822-B-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-188822-B-40.wav",\n      "bytes": 441044,\n      "sha256": "2c6928f5af242d0da9bfa41749ff6d606785ab179754a275c2be9d7c3b64461f",\n      "decoded_sha256": "f0086a283fa6da048394870bcb26f5880d7030199d38bd866527e8c36980422c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "188822",\n      "source_id": "188822",\n      "attribution": "- [2-188822-B.ogg]: clip derived from Helicopter landing (http://www.freesound.org/people/peridactyloptrix/sounds/188822/) by peridactyloptrix [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-188822-C-40",\n      "filename": "2-188822-C-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-188822-C-40.wav",\n      "bytes": 441044,\n      "sha256": "df124f48361127f4aa51cfe34078634e047c19af3a63d29826cb5a1def0c743f",\n      "decoded_sha256": "1e0bfb13d9ae14c2d01e2e5bc210adf575cdf08848417adb32bb1c71663e8d9a",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "188822",\n      "source_id": "188822",\n      "attribution": "- [2-188822-C.ogg]: clip derived from Helicopter landing (http://www.freesound.org/people/peridactyloptrix/sounds/188822/) by peridactyloptrix [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-188822-D-40",\n      "filename": "2-188822-D-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-188822-D-40.wav",\n      "bytes": 441044,\n      "sha256": "7ffae2dcd7611c77a1d57de223433e1e70fa6ffe2422b9e882e815ccaedc62b8",\n      "decoded_sha256": "f2fdcf52f24e4df3d0c62e90270321e30959a8da8dba6ddc275e09fc00af5152",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "188822",\n      "source_id": "188822",\n      "attribution": "- [2-188822-D.ogg]: clip derived from Helicopter landing (http://www.freesound.org/people/peridactyloptrix/sounds/188822/) by peridactyloptrix [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-28314-A-12",\n      "filename": "2-28314-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-28314-A-12.wav",\n      "bytes": 441044,\n      "sha256": "f77a729e093baeeba246d00c4875e6fd7702c3635e0e5e58aca2357f62d16c92",\n      "decoded_sha256": "c29a4510c5de63547a6e2a5ff189836def90bb6f01f5b285801b4321529cf8ec",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "28314",\n      "source_id": "28314",\n      "attribution": "- [2-28314-A.ogg]: clip derived from FireBurning.wav (http://www.freesound.org/people/pcaeldries/sounds/28314/) by pcaeldries [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-28314-B-12",\n      "filename": "2-28314-B-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-28314-B-12.wav",\n      "bytes": 441044,\n      "sha256": "6c5f3f8b6af384c0b553741b51e7348c24537d686e7abaa6eadf4badd2da2168",\n      "decoded_sha256": "01ef843100afe5ab0c66a28e0ea7c47a4b0ff5ada60f060833a62d7166bcc461",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "28314",\n      "source_id": "28314",\n      "attribution": "- [2-28314-B.ogg]: clip derived from FireBurning.wav (http://www.freesound.org/people/pcaeldries/sounds/28314/) by pcaeldries [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-30322-A-12",\n      "filename": "2-30322-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-30322-A-12.wav",\n      "bytes": 441044,\n      "sha256": "894526cbb7e9ca1150893b332a69cdc2cffaede5a29a881b4b7fe7dab4f1156e",\n      "decoded_sha256": "a70ed982ed6ec1fe33cbeedc5aae220a281d2fe849c7052ca42c89679c1b926f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "30322",\n      "source_id": "30322",\n      "attribution": "- [2-30322-A.ogg]: clip derived from FireBurning_v2.wav (http://www.freesound.org/people/pcaeldries/sounds/30322/) by pcaeldries [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-30322-B-12",\n      "filename": "2-30322-B-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-30322-B-12.wav",\n      "bytes": 441044,\n      "sha256": "b0a16101ed58292d6105d0f694adfb885578c56292e04e6c64a53ca3d5c2e884",\n      "decoded_sha256": "271cb0afe264c31be75841dc60e481e6f6d415e5602066b640def2e179228aee",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "30322",\n      "source_id": "30322",\n      "attribution": "- [2-30322-B.ogg]: clip derived from FireBurning_v2.wav (http://www.freesound.org/people/pcaeldries/sounds/30322/) by pcaeldries [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-37806-A-40",\n      "filename": "2-37806-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-37806-A-40.wav",\n      "bytes": 441044,\n      "sha256": "df941887d080de8487f8018f09091c040e82eae7a3a0cb7d532fb64ba64e04ef",\n      "decoded_sha256": "23dd65a9bbe7b98fe05af603ac6512803eca84f4cc84c496f8e702d53be0c4e3",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "37806",\n      "source_id": "37806",\n      "attribution": "- [2-37806-A.ogg]: clip derived from helicopters2.mp3 (http://www.freesound.org/people/mw_1984/sounds/37806/) by mw_1984 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-37806-B-40",\n      "filename": "2-37806-B-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-37806-B-40.wav",\n      "bytes": 441044,\n      "sha256": "bf730207adaed816833dc254f33b6f67b62b88b309a3ad9d54b70c2af310c9d7",\n      "decoded_sha256": "917fce95412b555b5d42f30b9d6d7fe3ccfc87acac743e814feb67e21baf0143",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "37806",\n      "source_id": "37806",\n      "attribution": "- [2-37806-B.ogg]: clip derived from helicopters2.mp3 (http://www.freesound.org/people/mw_1984/sounds/37806/) by mw_1984 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-37806-C-40",\n      "filename": "2-37806-C-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-37806-C-40.wav",\n      "bytes": 441044,\n      "sha256": "512833e0ad721ba5c7bbb699403b50dd878ebd29de4cac2ca247e5315da5e987",\n      "decoded_sha256": "732733a37b57f8c7e43e1dd00b8b337a1cdecb436854705e033c36de501daf11",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "37806",\n      "source_id": "37806",\n      "attribution": "- [2-37806-C.ogg]: clip derived from helicopters2.mp3 (http://www.freesound.org/people/mw_1984/sounds/37806/) by mw_1984 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-37806-D-40",\n      "filename": "2-37806-D-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-37806-D-40.wav",\n      "bytes": 441044,\n      "sha256": "d1691db93768e4e6ea6d2b20e4aeaccd383b3e2d28b8ba476ad86526bd7fb452",\n      "decoded_sha256": "dac07da26e818740aeeb4497ce9855c3497f7f6e46b2cd92f4b9bb5a876dc92e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "37806",\n      "source_id": "37806",\n      "attribution": "- [2-37806-D.ogg]: clip derived from helicopters2.mp3 (http://www.freesound.org/people/mw_1984/sounds/37806/) by mw_1984 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-50665-A-20",\n      "filename": "2-50665-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-50665-A-20.wav",\n      "bytes": 441044,\n      "sha256": "8179d295f531f165ee89e92c3fb9f496ca6e13cff1fae3bb76067ee84da49078",\n      "decoded_sha256": "31831def595c8e0e3532b1640b2e385ffdf8e76a82e48b9da259f80090424356",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "50665",\n      "source_id": "50665",\n      "attribution": "- [2-50665-A.ogg]: clip derived from toddler cry 1 0803xx.wav (http://www.freesound.org/people/BoilingSand/sounds/50665/) by BoilingSand [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-50666-A-20",\n      "filename": "2-50666-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-50666-A-20.wav",\n      "bytes": 441044,\n      "sha256": "1f15be2b6c7bc156ccb5174b205b03eace593d728ea227a889ecd17375d4f5ea",\n      "decoded_sha256": "287e11f6be69f8566fd48350cb73f34f52f054439c2298e23bcad836d8612c17",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "50666",\n      "source_id": "50666",\n      "attribution": "- [2-50666-A.ogg]: clip derived from toddler cry 2 0803xx.wav (http://www.freesound.org/people/BoilingSand/sounds/50666/) by BoilingSand [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-50667-A-41",\n      "filename": "2-50667-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-50667-A-41.wav",\n      "bytes": 441044,\n      "sha256": "b099e90db935a98b053d2a13db180ed783d35c89d600c07fa4f3aa8261cbf1ef",\n      "decoded_sha256": "1ea8edfb09bbf2020c4a12d0a0cf1e80dd5534fb24664d67cd89ad28b4b65aae",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "50667",\n      "source_id": "50667",\n      "attribution": "- [2-50667-A.ogg]: clip derived from 080320 Chainsaw start, branches falling and engine cut.wav (http://www.freesound.org/people/BoilingSand/sounds/50667/) by BoilingSand [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-50667-B-41",\n      "filename": "2-50667-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-50667-B-41.wav",\n      "bytes": 441044,\n      "sha256": "c8d73fbda3f8b2ed27df38e09ced22fe2d59c46160a92ba985c37d9fdcb6f474",\n      "decoded_sha256": "293f5fcd657bf8166adc92d1b9377c6721e826a4fd4354fafddd2e68320e96b9",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "50667",\n      "source_id": "50667",\n      "attribution": "- [2-50667-B.ogg]: clip derived from 080320 Chainsaw start, branches falling and engine cut.wav (http://www.freesound.org/people/BoilingSand/sounds/50667/) by BoilingSand [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-50668-A-41",\n      "filename": "2-50668-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-50668-A-41.wav",\n      "bytes": 441044,\n      "sha256": "b3b5b151092231382feea724dc38188638814a7824592c7d80967d1bad59669f",\n      "decoded_sha256": "22673c3793fcb5b278641c0e37e8f667ee149aea4abf2771154a4fa25ab650f8",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "50668",\n      "source_id": "50668",\n      "attribution": "- [2-50668-A.ogg]: clip derived from Chainsaw with Bobcat idling and tree bits falling 080320.wav (http://www.freesound.org/people/BoilingSand/sounds/50668/) by BoilingSand [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-50668-B-41",\n      "filename": "2-50668-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-50668-B-41.wav",\n      "bytes": 441044,\n      "sha256": "11a783b1f9a74db022ac654722d21aef2e374cbf6b68ecb8fa20c576662edefc",\n      "decoded_sha256": "d088a4bc532bf4eee2fa46396ffe7ab234866f5170c5b530e804a7bdd1176d29",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "50668",\n      "source_id": "50668",\n      "attribution": "- [2-50668-B.ogg]: clip derived from Chainsaw with Bobcat idling and tree bits falling 080320.wav (http://www.freesound.org/people/BoilingSand/sounds/50668/) by BoilingSand [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-61311-A-12",\n      "filename": "2-61311-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-61311-A-12.wav",\n      "bytes": 441044,\n      "sha256": "e0c27e6498d90f6e3a70346ebd43669478a92a170d453ebae6afbda2af1954e2",\n      "decoded_sha256": "d9323631e1bf5c11fcdab037becf35728ed9795769f62356635a9fc71262ce7b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "61311",\n      "source_id": "61311",\n      "attribution": "- [2-61311-A.ogg]: clip derived from Fire In Woodburning Stove.wav (http://www.freesound.org/people/Benboncan/sounds/61311/) by Benboncan [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-65747-A-12",\n      "filename": "2-65747-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-65747-A-12.wav",\n      "bytes": 441044,\n      "sha256": "14a098365b21366eeec932f9cadfb368144fbd7d84e0066f81992afc3fac19a5",\n      "decoded_sha256": "02e9c19f3353313931f98b1f46c9657a76d35ef16473fef495fa3104d41b720c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "65747",\n      "source_id": "65747",\n      "attribution": "- [2-65747-A.ogg]: clip derived from 20081231.fire.01.wav (http://www.freesound.org/people/dobroide/sounds/65747/) by dobroide [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-65750-A-1",\n      "filename": "2-65750-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-65750-A-1.wav",\n      "bytes": 441044,\n      "sha256": "665283baf62e962de29cbd5ffa44457c52b98ff6fa6db9469b82f33fab488d4c",\n      "decoded_sha256": "68478a5babaa0bcb09bd4d674d81abfd9f432cd59925a434c4116a7bda3b88f3",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "65750",\n      "source_id": "65750",\n      "attribution": "- [2-65750-A.ogg]: clip derived from 20081231.rooster.flac (http://www.freesound.org/people/dobroide/sounds/65750/) by dobroide [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-66637-A-20",\n      "filename": "2-66637-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-66637-A-20.wav",\n      "bytes": 441044,\n      "sha256": "7956582f54756ada936b22c5d2e810d7237ddc95897a1557a59417e3b1be0f99",\n      "decoded_sha256": "085036905a1f22b8b84dd6ca327cbb0e38ee8efcd22c8b2498a37c259e4dd4d6",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "66637",\n      "source_id": "66637",\n      "attribution": "- [2-66637-A.ogg]: clip derived from crying baby merged.wav (http://www.freesound.org/people/bonehead/sounds/66637/) by bonehead [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-66637-B-20",\n      "filename": "2-66637-B-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-66637-B-20.wav",\n      "bytes": 441044,\n      "sha256": "9ef70dfc68879c756405662282e61e32aa88b3716455b13205936c4f7a135bb8",\n      "decoded_sha256": "e4113bf37fc17c1b299adb4dd21886a1492765c6097b77b8630d61a0af3d00ec",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "66637",\n      "source_id": "66637",\n      "attribution": "- [2-66637-B.ogg]: clip derived from crying baby merged.wav (http://www.freesound.org/people/bonehead/sounds/66637/) by bonehead [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-68391-A-41",\n      "filename": "2-68391-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-68391-A-41.wav",\n      "bytes": 441044,\n      "sha256": "fd9981a2adcd14a9b297e1bae44b268cfbbb9586a57219e1297c3fc470ec525b",\n      "decoded_sha256": "f085dc6460ec1ad49079c7ec5f8f019054e4979bcbcc859e9a203230437d78e2",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "68391",\n      "source_id": "68391",\n      "attribution": "- [2-68391-A.ogg]: clip derived from Chainsaw - Tree cases.WAV (http://www.freesound.org/people/Ohrwurm/sounds/68391/) by Ohrwurm [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-68391-B-41",\n      "filename": "2-68391-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-68391-B-41.wav",\n      "bytes": 441044,\n      "sha256": "7c098d4477e3ac8cbf8a66d4ff144ef3706526c159eb54ede68f6b39257db425",\n      "decoded_sha256": "a42d4ea37fad136b0b0dc0c6b44ca74d5ff0b42a494ec03a8fe01a37763de49b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "68391",\n      "source_id": "68391",\n      "attribution": "- [2-68391-B.ogg]: clip derived from Chainsaw - Tree cases.WAV (http://www.freesound.org/people/Ohrwurm/sounds/68391/) by Ohrwurm [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-71162-A-1",\n      "filename": "2-71162-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-71162-A-1.wav",\n      "bytes": 441044,\n      "sha256": "0e66be15bafedfc302c7a14003083c8296388bc983aa93f257c709e780da6ac8",\n      "decoded_sha256": "1d792cccdc2e24f11094cdeb2f59878af1d502136c3a80bc986f50e2ec42b989",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "71162",\n      "source_id": "71162",\n      "attribution": "- [2-71162-A.ogg]: clip derived from bees buzzing.aif (http://www.freesound.org/people/simongray/sounds/71162/) by simongray [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-72970-A-10",\n      "filename": "2-72970-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-72970-A-10.wav",\n      "bytes": 441044,\n      "sha256": "f6d91324daa8863100c7f922b20727c86d46c27249cce9e15d66305e623e56b8",\n      "decoded_sha256": "a9a7159c4fbe88b09a33e736d7a5b397cd00e624c7db968bcfa35706c6e824a1",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "72970",\n      "source_id": "72970",\n      "attribution": "- [2-72970-A.ogg]: clip derived from RAIN indoors.ogg (http://www.freesound.org/people/bsyapril88/sounds/72970/) by bsyapril88 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-73027-A-10",\n      "filename": "2-73027-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-73027-A-10.wav",\n      "bytes": 441044,\n      "sha256": "42eba8ebc797e9fa843e8dbfb907c58ea8c8976277997d5f6e726b1bd3fe4111",\n      "decoded_sha256": "0a73dcbef9da378ccb7b69995a8e7c620c9136f7d21090433ff02afb9f2def0c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "73027",\n      "source_id": "73027",\n      "attribution": "- [2-73027-A.ogg]: clip derived from rain_on_greenhouse.mp3 (http://www.freesound.org/people/j1987/sounds/73027/) by j1987 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-73260-A-10",\n      "filename": "2-73260-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-73260-A-10.wav",\n      "bytes": 441044,\n      "sha256": "206b445eaf57f739c09185695e82bc0c3e7b9f68215ad9f7cb4374f7648e9acc",\n      "decoded_sha256": "9b3a8d9ee5e279a7696b174d3c9015e78de7bdb2cee12613016449c7ac089089",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "73260",\n      "source_id": "73260",\n      "attribution": "- [2-73260-A.ogg]: clip derived from Rain on car roof.wav (http://www.freesound.org/people/jakobhandersen/sounds/73260/) by jakobhandersen [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-77945-A-41",\n      "filename": "2-77945-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-77945-A-41.wav",\n      "bytes": 441044,\n      "sha256": "cf77f72a73605adbcea2a0d61bdd666b4335e190edf722b8f67a216392ebd224",\n      "decoded_sha256": "49ec8daf3fd905a5fd59cc4a7c08966c3c3dc6a57db219e4354e6bb08133d0b0",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "77945",\n      "source_id": "77945",\n      "attribution": "- [2-77945-A.ogg]: clip derived from chainsaw_Husqvarna_385XPG.wav (http://www.freesound.org/people/theTone/sounds/77945/) by theTone [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-77945-B-41",\n      "filename": "2-77945-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-77945-B-41.wav",\n      "bytes": 441044,\n      "sha256": "7a7c9cb12b77609346548ae861044137910b0e1dddc8da116fdc258fff13cd6c",\n      "decoded_sha256": "e9a51c04bc64b485f8fbbf44bb316901bd77309bbdcd9eb6ae6ac7441448628e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "77945",\n      "source_id": "77945",\n      "attribution": "- [2-77945-B.ogg]: clip derived from chainsaw_Husqvarna_385XPG.wav (http://www.freesound.org/people/theTone/sounds/77945/) by theTone [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-80482-A-20",\n      "filename": "2-80482-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-80482-A-20.wav",\n      "bytes": 441044,\n      "sha256": "1bc0b84be46c90e74f19f4704f488cb17a4e34686fd76bc4454007f13bf64f33",\n      "decoded_sha256": "6873f180067d8261e7324f73a385579bf180b2d553f0352c234b08cf8555252a",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "80482",\n      "source_id": "80482",\n      "attribution": "- [2-80482-A.ogg]: clip derived from bebe_32_Iz.aif (http://www.freesound.org/people/stevelalonde/sounds/80482/) by stevelalonde [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-81270-A-1",\n      "filename": "2-81270-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-81270-A-1.wav",\n      "bytes": 441044,\n      "sha256": "ffbbe318c72dca7a09901a60088fd1cd1337ebde994ef32c4fe40fa829c2ca14",\n      "decoded_sha256": "d2c98620a6afbb2f7211258afaa41bd8611fabe98d07a270809313c725ace658",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "81270",\n      "source_id": "81270",\n      "attribution": "- [2-81270-A.ogg]: clip derived from animals-hen-rooster-Blaafarveverket-090721.wav (http://www.freesound.org/people/ra_gun/sounds/81270/) by ra_gun [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-81731-A-10",\n      "filename": "2-81731-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-81731-A-10.wav",\n      "bytes": 441044,\n      "sha256": "25d3a915368db5e4175b1d51dea34bfad02d909039a051cbf72c0f39a0f34667",\n      "decoded_sha256": "4cb1dca4b9c1887a2546e95cf7c0bd5bd92dc5484cf3b5d9abec0e4dd249cb21",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "81731",\n      "source_id": "81731",\n      "attribution": "- [2-81731-A.ogg]: clip derived from Silencyo_RAIN_NIGHT_Village_Drips_Courtyard.aif (http://www.freesound.org/people/silencyo/sounds/81731/) by silencyo [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-82367-A-10",\n      "filename": "2-82367-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-82367-A-10.wav",\n      "bytes": 441044,\n      "sha256": "2cce637e5f60368db490c8ec980eb0cc7dbdd623c077bcd456a2fb60afdd4820",\n      "decoded_sha256": "6bc1dc6699aece17f20002be4e47fa21179e5c385b224007c02622d0b81398fb",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "82367",\n      "source_id": "82367",\n      "attribution": "- [2-82367-A.ogg]: clip derived from Rain on Plastic Roof.WAV (http://www.freesound.org/people/gynation/sounds/82367/) by gynation [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-82538-A-21",\n      "filename": "2-82538-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-82538-A-21.wav",\n      "bytes": 441044,\n      "sha256": "c6a37c234e9c3333de6fc9b16e93db76060e9eee9b1e989970a3cbc0bfc1cb5a",\n      "decoded_sha256": "8db02f82eb159e9b627ba5bcd53940fe0e61188927976a6b5cd4719ebe768797",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "82538",\n      "source_id": "82538",\n      "attribution": "- [2-82538-A.ogg]: clip derived from Mark sneezing.wav (http://www.freesound.org/people/Electron%20John/sounds/82538/) by Electron John [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-87781-A-10",\n      "filename": "2-87781-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-87781-A-10.wav",\n      "bytes": 441044,\n      "sha256": "81fe7ce5da401a69a542439a5c8277864468a0bf09bf1378b0accac15c033638",\n      "decoded_sha256": "b7553905229fcccdeef434b6cb1c156338c0e0b422bc90276cb77ce134d82d19",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "87781",\n      "source_id": "87781",\n      "attribution": "- [2-87781-A.ogg]: clip derived from rain.wav (http://www.freesound.org/people/sagetyrtle/sounds/87781/) by sagetyrtle [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-88724-A-38",\n      "filename": "2-88724-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-88724-A-38.wav",\n      "bytes": 441044,\n      "sha256": "ba63b4175b60e54464f7bbec4b91f1f9345323221fe48655dc31c48be72312d6",\n      "decoded_sha256": "5553f270eb364caba88675c0ae95ef7dac1b75336421093c56e82d8627f34c1b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "88724",\n      "source_id": "88724",\n      "attribution": "- [2-88724-A.ogg]: clip derived from Cuckoo Clock-Twelve Noon.wav (http://www.freesound.org/people/lonemonk/sounds/88724/) by lonemonk [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-93030-A-21",\n      "filename": "2-93030-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-93030-A-21.wav",\n      "bytes": 441044,\n      "sha256": "e56f2b85d767caab2bf9ade0a4049bee154919d9650166a2c05c430face89347",\n      "decoded_sha256": "9e87577d3c04ecc3532b5edfb4b4e5487f3d76d47cba23d3567c317167f95876",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "93030",\n      "source_id": "93030",\n      "attribution": "- [2-93030-A.ogg]: clip derived from Sneeze and Sniffle 44hzStereo.wav (http://www.freesound.org/people/higginsdj/sounds/93030/) by higginsdj [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "2-95035-A-1",\n      "filename": "2-95035-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-95035-A-1.wav",\n      "bytes": 441044,\n      "sha256": "21fae7e7daf06262ec9230a4f2f2bb013fbbaeb126eb50ca62ab364d52c84759",\n      "decoded_sha256": "4fbca1f87795025084761c9c1c387606127140caa4ccb21f94bee006d7e00d6e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "95035",\n      "source_id": "95035",\n      "attribution": "- [2-95035-A.ogg]: clip derived from 20100418.marshes.storm.03.wav (http://www.freesound.org/people/dobroide/sounds/95035/) by dobroide [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-95258-A-1",\n      "filename": "2-95258-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-95258-A-1.wav",\n      "bytes": 441044,\n      "sha256": "d28ca7b6710feb0ec50272e5a77f924e80fa83290bdec6ec4c0db82ce6576f07",\n      "decoded_sha256": "dd25e5909e1cc1fd7cd1f0fa7123f39cd905537f4807807112bb559c99ebd837",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "95258",\n      "source_id": "95258",\n      "attribution": "- [2-95258-A.ogg]: clip derived from Rural Back Yard Whitehill Nova Scotia April 21 2010.mp3 (http://www.freesound.org/people/wildear1/sounds/95258/) by wildear1 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-95258-B-1",\n      "filename": "2-95258-B-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-95258-B-1.wav",\n      "bytes": 441044,\n      "sha256": "f3d02c689e4003bf4800de2af77dc773ccb4de9049ce4cb81104696d17ba1956",\n      "decoded_sha256": "67d0674c18e3c937f867ae7137f33b8ea3d1512146c60af7eb18598da26ee410",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "95258",\n      "source_id": "95258",\n      "attribution": "- [2-95258-B.ogg]: clip derived from Rural Back Yard Whitehill Nova Scotia April 21 2010.mp3 (http://www.freesound.org/people/wildear1/sounds/95258/) by wildear1 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "2-96460-A-1",\n      "filename": "2-96460-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/2-96460-A-1.wav",\n      "bytes": 441044,\n      "sha256": "21a1734ac6f4894f2575e87721b3347fc97b856e63d49ac2fd0a4745a34b6b88",\n      "decoded_sha256": "e87b38acf518cf96dda985eab1344f4c9373e85346c86e5c14a9f710164e5ca9",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 2,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "96460",\n      "source_id": "96460",\n      "attribution": "- [2-96460-A.ogg]: clip derived from ambience_pigeons_roosters_hens.mp3 (http://www.freesound.org/people/123a4567/sounds/96460/) by 123a4567 [CC-Sampling+]",\n      "original_license": "CC-Sampling+"\n    },\n    {\n      "id": "3-104632-A-12",\n      "filename": "3-104632-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-104632-A-12.wav",\n      "bytes": 441044,\n      "sha256": "415e6d92371447377a916760fdda326f55f6f88cf428dc45b3edfe45b1d2b0a5",\n      "decoded_sha256": "ca0c20ea69ae952a9da98434ad82f50f54805fbf422ea524aa65758f82bac155",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "104632",\n      "source_id": "104632",\n      "attribution": "- [3-104632-A.ogg]: clip derived from greve feu.wav (http://www.freesound.org/people/bogus/sounds/104632/) by bogus [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-104958-A-12",\n      "filename": "3-104958-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-104958-A-12.wav",\n      "bytes": 441044,\n      "sha256": "d538428b1a51d450809757d169ba16f77c85721c267a46d29f5d538dfc846563",\n      "decoded_sha256": "377ccd07d25201eef92ce882fb3018c0844b540f5f694b5fafa17a917e3f232a",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "104958",\n      "source_id": "104958",\n      "attribution": "- [3-104958-A.ogg]: clip derived from petit feu-little fire (2).wav (http://www.freesound.org/people/Glaneur%20de%20sons/sounds/104958/) by Glaneur de sons [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-107219-A-1",\n      "filename": "3-107219-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-107219-A-1.wav",\n      "bytes": 441044,\n      "sha256": "18c72ca3f004fa0367a401f65ea375e46a81ce715e1993e4c67c7a8ffe364d20",\n      "decoded_sha256": "78c2da0ad43f2f914945a372bcf3ffca40923e1db04339807de27aa88a873fde",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "107219",\n      "source_id": "107219",\n      "attribution": "- [3-107219-A.ogg]: clip derived from birmingham-nechells-rooster.wav (http://www.freesound.org/people/keithpeter/sounds/107219/) by keithpeter [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-116135-A-1",\n      "filename": "3-116135-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-116135-A-1.wav",\n      "bytes": 441044,\n      "sha256": "0415467964520c0268ea0b5beb9465ff6a2458f508d409a9b6d039f247f74352",\n      "decoded_sha256": "2b5a979f09ac2b3469bddeedb85aaccfc7d8921e5ffb5fe5af6d775db92a2e42",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "116135",\n      "source_id": "116135",\n      "attribution": "- [3-116135-A.ogg]: clip derived from Rooster and chicks.mp3 (http://www.freesound.org/people/Barbamiz/sounds/116135/) by Barbamiz [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-118656-A-41",\n      "filename": "3-118656-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-118656-A-41.wav",\n      "bytes": 441044,\n      "sha256": "c8b03e975750a6c264f839e6adc68a5450a5d44d4db808b443522d918fe3185f",\n      "decoded_sha256": "fd6ee0989bc59e79101a3129c297a2b47921f8607c74c5d00b391c98aba18d9b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "118656",\n      "source_id": "118656",\n      "attribution": "- [3-118656-A.ogg]: clip derived from Chainsaw in the garden.wav (http://www.freesound.org/people/Kyster/sounds/118656/) by Kyster [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-118657-A-41",\n      "filename": "3-118657-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-118657-A-41.wav",\n      "bytes": 441044,\n      "sha256": "e3ea73b1d6c8390853acf5e390eae7d90b314d749c03cdab3a4560d924687aba",\n      "decoded_sha256": "9e113e7bcecf1039ffa574e8e6cbce83042054e2b0050b7061a87fe337d2874d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "118657",\n      "source_id": "118657",\n      "attribution": "- [3-118657-A.ogg]: clip derived from chainsaw and little tree.wav (http://www.freesound.org/people/Kyster/sounds/118657/) by Kyster [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-118657-B-41",\n      "filename": "3-118657-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-118657-B-41.wav",\n      "bytes": 441044,\n      "sha256": "a1f94069bc51bb5ef19075a4a746610d98f9e05405750fa30d9818832286c07f",\n      "decoded_sha256": "89f392e5a221bd902fc7970d896f3612b03d840143715d66ca31e8055142e0c5",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "118657",\n      "source_id": "118657",\n      "attribution": "- [3-118657-B.ogg]: clip derived from chainsaw and little tree.wav (http://www.freesound.org/people/Kyster/sounds/118657/) by Kyster [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-118658-A-41",\n      "filename": "3-118658-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-118658-A-41.wav",\n      "bytes": 441044,\n      "sha256": "55f4073b9095db0ad4d20230badbb7decc9549e5f080df29717d7d379648405e",\n      "decoded_sha256": "963c5820d1bfc2053d005e6fb220ab64efd076028cf83d95be8eb8bc1c519200",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "118658",\n      "source_id": "118658",\n      "attribution": "- [3-118658-A.ogg]: clip derived from chainsaw vs chestnut tree.wav (http://www.freesound.org/people/Kyster/sounds/118658/) by Kyster [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-118658-B-41",\n      "filename": "3-118658-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-118658-B-41.wav",\n      "bytes": 441044,\n      "sha256": "f26428971e55d517dbca42ad5b5678fd0fce166931f3e1c581121a93b346cf61",\n      "decoded_sha256": "2716fecec44a79c783d03926d0a87f4e3eaa6546c2ad3d7a78bced7a9878e06d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "118658",\n      "source_id": "118658",\n      "attribution": "- [3-118658-B.ogg]: clip derived from chainsaw vs chestnut tree.wav (http://www.freesound.org/people/Kyster/sounds/118658/) by Kyster [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-118972-A-41",\n      "filename": "3-118972-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-118972-A-41.wav",\n      "bytes": 441044,\n      "sha256": "c98d58cf863f4b16e9c9416f4a3b253f69eed3f4fc3b0ce22ae3f7f2dedf43d2",\n      "decoded_sha256": "5870e7ff95e1d94cd67959f512b206a8bfd0f4844643fbbb1bc7eaa3d46a96c8",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "118972",\n      "source_id": "118972",\n      "attribution": "- [3-118972-A.ogg]: clip derived from Chainsaw.wav (http://www.freesound.org/people/esperri/sounds/118972/) by esperri [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-118972-B-41",\n      "filename": "3-118972-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-118972-B-41.wav",\n      "bytes": 441044,\n      "sha256": "354e5e850c5a474d4e822af1ed35ce304780ee4a9494f3d5edfee7a9fe859cfe",\n      "decoded_sha256": "d098631885e6496b0bbbd41448cbc8698db60d49c0d5c913ffef2642a5b112e3",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "118972",\n      "source_id": "118972",\n      "attribution": "- [3-118972-B.ogg]: clip derived from Chainsaw.wav (http://www.freesound.org/people/esperri/sounds/118972/) by esperri [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-120644-A-12",\n      "filename": "3-120644-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-120644-A-12.wav",\n      "bytes": 441044,\n      "sha256": "44acca39a9f884a0ba916ea8f5865079006d42acd31a8498f4e5bb0d5bd10146",\n      "decoded_sha256": "c57e2ebe34ae151e9aba74488550758e1c3cce70e913406d3650bd027b0ff615",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "120644",\n      "source_id": "120644",\n      "attribution": "- [3-120644-A.ogg]: clip derived from 20110503_fireplace.00.wav (http://www.freesound.org/people/dobroide/sounds/120644/) by dobroide [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-132852-A-10",\n      "filename": "3-132852-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-132852-A-10.wav",\n      "bytes": 441044,\n      "sha256": "5d4960883ceabe28c008721ea3b5f5206a6f0234cacd4c1af81dd7594b4d3503",\n      "decoded_sha256": "a720f5cfec1038e72a2c8730401fcdeb96da69a6f20c8744dff7d06fcadb56df",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "132852",\n      "source_id": "132852",\n      "attribution": "- [3-132852-A.ogg]: clip derived from rain from a car.WAV (http://www.freesound.org/people/inchadney/sounds/132852/) by inchadney [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-134049-A-1",\n      "filename": "3-134049-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-134049-A-1.wav",\n      "bytes": 441044,\n      "sha256": "2ccd86599da9a402b116d814d765b0fd1a568ee11923394bfc0a3af4db1581fa",\n      "decoded_sha256": "3a545b0bd277351ed136b25d699fc0de2eae7ae6680a404be9e94693f41ecd60",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "134049",\n      "source_id": "134049",\n      "attribution": "- [3-134049-A.ogg]: clip derived from cock and hen.WAV (http://www.freesound.org/people/Setuniman/sounds/134049/) by Setuniman [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-136288-A-0",\n      "filename": "3-136288-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-136288-A-0.wav",\n      "bytes": 441044,\n      "sha256": "8293b4fe3c40cf9bbe243fa27d153bbef86217f855e4295e89f41411267c9f8d",\n      "decoded_sha256": "f8182f6f3efebaad7040ae807c89dea75c82179fc897b44c92e95206cba3b8d8",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "136288",\n      "source_id": "136288",\n      "attribution": "- [3-136288-A.ogg]: clip derived from Dogs barking and birds singing in the Red Wood Forest, close to San Francisco. Road in the far background. (http://www.freesound.org/people/felix.blume/sounds/136288/) by felix.blume [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-137152-A-1",\n      "filename": "3-137152-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-137152-A-1.wav",\n      "bytes": 441044,\n      "sha256": "55944b46e8d399218da6b57cc1b3359d58e9da5db30cb377c3c8ae46d692d1fc",\n      "decoded_sha256": "433e110db001c8308c5cc194ff0b80b36f80d5d02d504866534ee0693ce70f5a",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "137152",\n      "source_id": "137152",\n      "attribution": "- [3-137152-A.ogg]: clip derived from 20111206_02.rooster.wav (http://www.freesound.org/people/dobroide/sounds/137152/) by dobroide [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-140774-A-10",\n      "filename": "3-140774-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-140774-A-10.wav",\n      "bytes": 441044,\n      "sha256": "1dd994fb071b52e4232b0cbd06558dd8d179008be2e4cf1f41d4abdca4f390fe",\n      "decoded_sha256": "ee764150a9680d62f6c6dd44bc6081beff6b101ce837e990397e5cc417e38bd2",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "140774",\n      "source_id": "140774",\n      "attribution": "- [3-140774-A.ogg]: clip derived from Rain over a car (http://www.freesound.org/people/Sam80/sounds/140774/) by Sam80 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-141684-A-21",\n      "filename": "3-141684-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-141684-A-21.wav",\n      "bytes": 441044,\n      "sha256": "6046ae5559d7ce126b25406ff8538214d66c73c77ba15cd9ecb4849513dd3d87",\n      "decoded_sha256": "b154a33557b35bccec68db7a5d1e597507b8523345ac022fa9f068d7eefb38a0",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "141684",\n      "source_id": "141684",\n      "attribution": "- [3-141684-A.ogg]: clip derived from Two Sneezes (http://www.freesound.org/people/thomasevd/sounds/141684/) by thomasevd [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-142005-A-10",\n      "filename": "3-142005-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-142005-A-10.wav",\n      "bytes": 441044,\n      "sha256": "784441f1c74be9dc64ec77aa9628df5b4557711b5f1435286dff9ea9317f02e0",\n      "decoded_sha256": "c467bef6c1f3eb3d224b6ad53faa1c1fa4cf0821922bb1a0e4bb4c540cd39199",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "142005",\n      "source_id": "142005",\n      "attribution": "- [3-142005-A.ogg]: clip derived from Heavy rain - some people 1 (http://www.freesound.org/people/jop9798/sounds/142005/) by jop9798 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-142006-A-10",\n      "filename": "3-142006-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-142006-A-10.wav",\n      "bytes": 441044,\n      "sha256": "bd24c750af1d7ed49a3d19ecc4401cd10368a18e02a07c3a5524fdb7a576d5af",\n      "decoded_sha256": "b1b81ddbea8de3a3a6166f67c97cd0601d9a2906b2f3323fba3f58125365ffb3",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "142006",\n      "source_id": "142006",\n      "attribution": "- [3-142006-A.ogg]: clip derived from Heavy rain - some people 2 (http://www.freesound.org/people/jop9798/sounds/142006/) by jop9798 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-142593-A-38",\n      "filename": "3-142593-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-142593-A-38.wav",\n      "bytes": 441044,\n      "sha256": "8eb0915dab65e09fd489c3ccac93e620892cb795abe4b16e7a103c5bbc92446b",\n      "decoded_sha256": "6f7361108a3095c485c84f288d8fe12d02ad859f678d4a4812f26843db03d950",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "142593",\n      "source_id": "142593",\n      "attribution": "- [3-142593-A.ogg]: clip derived from Ticking.aif (http://www.freesound.org/people/timgormly/sounds/142593/) by timgormly [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-142601-A-21",\n      "filename": "3-142601-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-142601-A-21.wav",\n      "bytes": 441044,\n      "sha256": "ef15ba95b34a2a12a0b7f272d5b26422e828ceffa16613e3f89df1890d5f1d75",\n      "decoded_sha256": "7cb49190af8576780fb19b0931ca2898942fe55c9c5261158e83b3e41f6851c5",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "142601",\n      "source_id": "142601",\n      "attribution": "- [3-142601-A.ogg]: clip derived from sneeze2.mp3 (http://www.freesound.org/people/loisgwenllian/sounds/142601/) by loisgwenllian [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-142605-A-21",\n      "filename": "3-142605-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-142605-A-21.wav",\n      "bytes": 441044,\n      "sha256": "444b8bc04cef9c0ec40d33e3c163c54a98185e23bc76eab68d02ce8c8cdf7e21",\n      "decoded_sha256": "0b81beb37b6d074e1bc0dd7b030c2469bbab4fe96771cb0dd04a380016f5a155",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "142605",\n      "source_id": "142605",\n      "attribution": "- [3-142605-A.ogg]: clip derived from sneeze3.mp3 (http://www.freesound.org/people/loisgwenllian/sounds/142605/) by loisgwenllian [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-143119-A-21",\n      "filename": "3-143119-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-143119-A-21.wav",\n      "bytes": 441044,\n      "sha256": "bd5ca9e66497ab6b82c8c158d657011240040a7f9b5434fa73d40314b66433b2",\n      "decoded_sha256": "fbedacfdc8b6d40876672f6598d1302c80f4e9f5088368d4e6430d9fa5a63969",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "143119",\n      "source_id": "143119",\n      "attribution": "- [3-143119-A.ogg]: clip derived from Sneezer.wav (http://www.freesound.org/people/1pjladd2/sounds/143119/) by 1pjladd2 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-143929-A-10",\n      "filename": "3-143929-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-143929-A-10.wav",\n      "bytes": 441044,\n      "sha256": "5510934acbb493aee8e975d5b1c1b34b1a39ef0b870e6091778909995fef9004",\n      "decoded_sha256": "58abb867e92fc6839376c7ce4101a3263a5e86116fb15a5bca8855286f1be5a5",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "143929",\n      "source_id": "143929",\n      "attribution": "- [3-143929-A.ogg]: clip derived from Driving in rain.wav (http://www.freesound.org/people/Polmantz/sounds/143929/) by Polmantz [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-143933-A-38",\n      "filename": "3-143933-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-143933-A-38.wav",\n      "bytes": 441044,\n      "sha256": "02809701aa82027a54a463637255b4e26ef3ccd1a47d69b59289e185b28ed675",\n      "decoded_sha256": "02f60acd37c4c3b3de84e8c3549aa4cfcacf7fb26aeeb8632888c8fd3fc81657",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "143933",\n      "source_id": "143933",\n      "attribution": "- [3-143933-A.ogg]: clip derived from clock.wav (http://www.freesound.org/people/wentzelj/sounds/143933/) by wentzelj [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-144028-A-0",\n      "filename": "3-144028-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-144028-A-0.wav",\n      "bytes": 441044,\n      "sha256": "2828c40f81b8bdfded734b1ea057444f780d1ada4241a2dc2ab70c646f963b47",\n      "decoded_sha256": "d714edace42f5dde1977e61633fffdfbed04ef73dfd6bcde2e116df56a523b44",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "144028",\n      "source_id": "144028",\n      "attribution": "- [3-144028-A.ogg]: clip derived from dog.aif (http://www.freesound.org/people/djym/sounds/144028/) by djym [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-144692-A-21",\n      "filename": "3-144692-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-144692-A-21.wav",\n      "bytes": 441044,\n      "sha256": "c7c434ea67b02a56ea63337e9d7a823deca517a038a80a0cf01daed61d18dba5",\n      "decoded_sha256": "351d945ebf0f2235d81f5e36434aded0b4049428adc2d19ede8e1d31e360a548",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "144692",\n      "source_id": "144692",\n      "attribution": "- [3-144692-A.ogg]: clip derived from Sneeze.wav (http://www.freesound.org/people/tekgnosis/sounds/144692/) by tekgnosis [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-144827-A-11",\n      "filename": "3-144827-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-144827-A-11.wav",\n      "bytes": 441044,\n      "sha256": "553ceeb0be8984ea22e9ae1b4c892e76c19867e9240d5f9367c62b08ee7eba4e",\n      "decoded_sha256": "a736a11147649aeddb50d607e9139e12828adbec8820791fa7e223570eeff38b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "144827",\n      "source_id": "144827",\n      "attribution": "- [3-144827-A.ogg]: clip derived from sea waves (http://www.freesound.org/people/Eelke/sounds/144827/) by Eelke [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-144827-B-11",\n      "filename": "3-144827-B-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-144827-B-11.wav",\n      "bytes": 441044,\n      "sha256": "977a5b671cb9b57aea049bce2b1c139c039aa745c5c46c1f84964d8098381aa8",\n      "decoded_sha256": "796bdc6ab29ff9f6f31770dbbee732bf2daba767624ee362279461267b223aa8",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "144827",\n      "source_id": "144827",\n      "attribution": "- [3-144827-B.ogg]: clip derived from sea waves (http://www.freesound.org/people/Eelke/sounds/144827/) by Eelke [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-145382-A-1",\n      "filename": "3-145382-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-145382-A-1.wav",\n      "bytes": 441044,\n      "sha256": "20a3a8bbeec30ee012d8241acc02752d5f80877d2d3b58847b23bd5a99f7ee56",\n      "decoded_sha256": "3993edc9b43c04644ea1ba0500811ac1df5ca33eb03ea27137317004deb9d2bf",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "145382",\n      "source_id": "145382",\n      "attribution": "- [3-145382-A.ogg]: clip derived from Morning start.mp3 (http://www.freesound.org/people/NikosDemetriou/sounds/145382/) by NikosDemetriou [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-145774-A-12",\n      "filename": "3-145774-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-145774-A-12.wav",\n      "bytes": 441044,\n      "sha256": "126163ee53c9cfc620a9963303a56e5e8fb2e1a47da02ac0a8849f35244e0c8e",\n      "decoded_sha256": "a27fb967185382b2e1bcd8227f6ab0ec38b403aa496b4ad179a26eb53fdf511e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "145774",\n      "source_id": "145774",\n      "attribution": "- [3-145774-A.ogg]: clip derived from crackling_fire_021012.wav (http://www.freesound.org/people/sagetyrtle/sounds/145774/) by sagetyrtle [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-147965-A-12",\n      "filename": "3-147965-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-147965-A-12.wav",\n      "bytes": 441044,\n      "sha256": "e5de3d144c438512347ce7ebad8a7d69b0f1a28e47dc1c2ee86da44a6ef30a03",\n      "decoded_sha256": "ae45ea0f5fc673da7b76959dbc15a0524219dff5b2bc90d75ffd86b7d0ef2984",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "147965",\n      "source_id": "147965",\n      "attribution": "- [3-147965-A.ogg]: clip derived from Fire Burning (http://www.freesound.org/people/andydonaldson6/sounds/147965/) by andydonaldson6 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-148330-A-21",\n      "filename": "3-148330-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-148330-A-21.wav",\n      "bytes": 441044,\n      "sha256": "bfd24723af5d665600f2eef92307ab266f444465a231e6cc2276783bcc05f4af",\n      "decoded_sha256": "298e381c4b88bd8c1a701fc14d54334f6c7e0052388075edc3b66c9b770c9c47",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "148330",\n      "source_id": "148330",\n      "attribution": "- [3-148330-A.ogg]: clip derived from Sneeze.wav (http://www.freesound.org/people/Shapeshifter242/sounds/148330/) by Shapeshifter242 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-149189-A-1",\n      "filename": "3-149189-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-149189-A-1.wav",\n      "bytes": 441044,\n      "sha256": "0ab042724404b7b43a3f0b31188f57e6acf04f531dc3fa0f1f09ab68ad096d4f",\n      "decoded_sha256": "a8a9b9a27fab0e989d8fdbcdd1d882760dea4f8afa8896bc85d33e336b4c99ed",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "149189",\n      "source_id": "149189",\n      "attribution": "- [3-149189-A.ogg]: clip derived from rooster cock-a-doodle-doo (http://www.freesound.org/people/videog/sounds/149189/) by videog [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-150231-A-21",\n      "filename": "3-150231-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-150231-A-21.wav",\n      "bytes": 441044,\n      "sha256": "9e21a595275419e08b419011167efbe40de0ee2a610adbcaa0b0951c61230622",\n      "decoded_sha256": "19f6533027744f3638573dbb07c1eb8a2579eb6e70759f4aff702d5f7a061c68",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "150231",\n      "source_id": "150231",\n      "attribution": "- [3-150231-A.ogg]: clip derived from Female Sneeze (http://www.freesound.org/people/domvoice/sounds/150231/) by domvoice [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-150363-A-38",\n      "filename": "3-150363-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-150363-A-38.wav",\n      "bytes": 441044,\n      "sha256": "4d1d86984a7d2408c0719015f37ca12054b97cd2e8b9e7853462908122e6d424",\n      "decoded_sha256": "64c7987726a9c6ac7ca06f4e81c12e87a8a3e993030fa953f69b6428ba06319b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "150363",\n      "source_id": "150363",\n      "attribution": "- [3-150363-A.ogg]: clip derived from GB_wall.flac (http://www.freesound.org/people/Zbylut/sounds/150363/) by Zbylut [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-150979-A-40",\n      "filename": "3-150979-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-150979-A-40.wav",\n      "bytes": 441044,\n      "sha256": "41e05ff06cd2d710f748b0d10bf136fd6e11d489327cbeafed690918b417fc42",\n      "decoded_sha256": "a98d283427f1b6993f7303c124d2eeeba20fb31217fdaffeeddd781ecac7167e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "150979",\n      "source_id": "150979",\n      "attribution": "- [3-150979-A.ogg]: clip derived from helicopter.wav (http://www.freesound.org/people/soundingoffagain/sounds/150979/) by soundingoffagain [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-150979-B-40",\n      "filename": "3-150979-B-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-150979-B-40.wav",\n      "bytes": 441044,\n      "sha256": "60f34984d5387e02be66cd64b3170c1b2c1d953defe874930c15f8fd775dc34d",\n      "decoded_sha256": "1ed8f6512e4c4055f3093b40bdbf7d546f400411a39c6d53453243f026697f9e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "150979",\n      "source_id": "150979",\n      "attribution": "- [3-150979-B.ogg]: clip derived from helicopter.wav (http://www.freesound.org/people/soundingoffagain/sounds/150979/) by soundingoffagain [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-150979-C-40",\n      "filename": "3-150979-C-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-150979-C-40.wav",\n      "bytes": 441044,\n      "sha256": "38ee55ad9ddb3090622e41f502c5097214517b2a47c161f8d9e0d57c26636cd7",\n      "decoded_sha256": "a8f517885b35433d3613834e128de97e5183294987371c6025b1b8cb4995b352",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "150979",\n      "source_id": "150979",\n      "attribution": "- [3-150979-C.ogg]: clip derived from helicopter.wav (http://www.freesound.org/people/soundingoffagain/sounds/150979/) by soundingoffagain [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-151080-A-20",\n      "filename": "3-151080-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-151080-A-20.wav",\n      "bytes": 441044,\n      "sha256": "e673e80074c99c9a8426ff708ca40d5f7a146605cf7b1625532276a7b95fd801",\n      "decoded_sha256": "c184264f17aae1b5acc43fc2b992c53528849654b37c1ff4ebcdc4f8091b5fa8",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "151080",\n      "source_id": "151080",\n      "attribution": "- [3-151080-A.ogg]: clip derived from 01 baby crying hard and long.wav (http://www.freesound.org/people/PickleJones/sounds/151080/) by PickleJones [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-151081-A-20",\n      "filename": "3-151081-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-151081-A-20.wav",\n      "bytes": 441044,\n      "sha256": "48a0009a2b2f7e97d19c020c1292e17ee0775dc7537bca3109c4128fb17ac7de",\n      "decoded_sha256": "603c1680b94917323792ed80385dd91cfc90a91d8e4883a54f3dd6d15d96478e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "151081",\n      "source_id": "151081",\n      "attribution": "- [3-151081-A.ogg]: clip derived from 01 baby crying winding down.wav (http://www.freesound.org/people/PickleJones/sounds/151081/) by PickleJones [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-151081-B-20",\n      "filename": "3-151081-B-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-151081-B-20.wav",\n      "bytes": 441044,\n      "sha256": "922250bbeca13111c955a83958f013c3a50b48e240705c57a645faf8b47a35d6",\n      "decoded_sha256": "6a581e17d4f7a406baac1f9ea7dc97ccc6e85d0bc728414e7d424f226985c536",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "151081",\n      "source_id": "151081",\n      "attribution": "- [3-151081-B.ogg]: clip derived from 01 baby crying winding down.wav (http://www.freesound.org/people/PickleJones/sounds/151081/) by PickleJones [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-152007-A-20",\n      "filename": "3-152007-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-152007-A-20.wav",\n      "bytes": 441044,\n      "sha256": "dff786672fb70a78c6ddcde9935d1327fcb6704251844753c3a28b73ad2eef44",\n      "decoded_sha256": "ba114c02de65b8b38d1b7d623809d91ed837d491a8362c836e3148d35138ed02",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "152007",\n      "source_id": "152007",\n      "attribution": "- [3-152007-A.ogg]: clip derived from babys crying.mp3 (http://www.freesound.org/people/winsx87/sounds/152007/) by winsx87 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-152007-B-20",\n      "filename": "3-152007-B-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-152007-B-20.wav",\n      "bytes": 441044,\n      "sha256": "39f6c0417c9bc1154f1cc292a593a33caec51eb985482c89d57e5fede76cdb9f",\n      "decoded_sha256": "e5907e905045f9564bbf55a4283b7721163beda484b53a1464c98ce2152b96cb",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "152007",\n      "source_id": "152007",\n      "attribution": "- [3-152007-B.ogg]: clip derived from babys crying.mp3 (http://www.freesound.org/people/winsx87/sounds/152007/) by winsx87 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-152007-C-20",\n      "filename": "3-152007-C-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-152007-C-20.wav",\n      "bytes": 441044,\n      "sha256": "ccd8f855a799a8d21f3234ba81cce8a4eccbbfdef8d141b938f2c5e0ea85aa44",\n      "decoded_sha256": "9b78c4c7021787857c4a95ecc8f76fcf4c469600ecd9b4ba5d0c2233b370c89a",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "152007",\n      "source_id": "152007",\n      "attribution": "- [3-152007-C.ogg]: clip derived from babys crying.mp3 (http://www.freesound.org/people/winsx87/sounds/152007/) by winsx87 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-152007-D-20",\n      "filename": "3-152007-D-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-152007-D-20.wav",\n      "bytes": 441044,\n      "sha256": "c47feeedb4dbf4681b595318dfc3ca0aa555af0a2ef87022715c55ff47922999",\n      "decoded_sha256": "033f5ff0d4499ed4e0b221ba99b638146d4b7f29ff07c5a87b32dd2bb83be729",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "152007",\n      "source_id": "152007",\n      "attribution": "- [3-152007-D.ogg]: clip derived from babys crying.mp3 (http://www.freesound.org/people/winsx87/sounds/152007/) by winsx87 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-152007-E-20",\n      "filename": "3-152007-E-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-152007-E-20.wav",\n      "bytes": 441044,\n      "sha256": "2369f99bd8ec01a9eb92d1be9a418143ff6f2298791d7811b614ba6cf92ef420",\n      "decoded_sha256": "89d720c47a15a930535144f2830a5a0c5274d675134e5548dc85750f41082b7d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "152007",\n      "source_id": "152007",\n      "attribution": "- [3-152007-E.ogg]: clip derived from babys crying.mp3 (http://www.freesound.org/people/winsx87/sounds/152007/) by winsx87 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-154926-A-40",\n      "filename": "3-154926-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-154926-A-40.wav",\n      "bytes": 441044,\n      "sha256": "c21a6fadacf782bac2e8efab3815d2fe18ff0130239e671fb666acd2d52dfac5",\n      "decoded_sha256": "7eee8538d52512badc538d5074a64febb8e6db65246d136c424ac0b11561b935",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "154926",\n      "source_id": "154926",\n      "attribution": "- [3-154926-A.ogg]: clip derived from London Exterior  helicopter passes overhead.mp3 (http://www.freesound.org/people/kerouac21/sounds/154926/) by kerouac21 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-154926-B-40",\n      "filename": "3-154926-B-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-154926-B-40.wav",\n      "bytes": 441044,\n      "sha256": "b919f24982343c8aaeb58472f23ac370b0ea7e343f4925fc6cca81714086f215",\n      "decoded_sha256": "2789df99ea89031611b37f7ac574ba61d5e9e0b2e05fc3bb024f818c2b50b49d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "154926",\n      "source_id": "154926",\n      "attribution": "- [3-154926-B.ogg]: clip derived from London Exterior  helicopter passes overhead.mp3 (http://www.freesound.org/people/kerouac21/sounds/154926/) by kerouac21 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-154957-A-1",\n      "filename": "3-154957-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-154957-A-1.wav",\n      "bytes": 441044,\n      "sha256": "6ffb2cb87baaaadf4833dc967e2c5b0006efed54c30cd1723bd546266cd0f089",\n      "decoded_sha256": "e75d650c86fb7b36c59980c9f9d83a75d2703baeba90bde559f56fbd61c6d733",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "154957",\n      "source_id": "154957",\n      "attribution": "- [3-154957-A.ogg]: clip derived from brahma-rooster.wav (http://www.freesound.org/people/signorvaso/sounds/154957/) by signorvaso [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-155312-A-0",\n      "filename": "3-155312-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-155312-A-0.wav",\n      "bytes": 441044,\n      "sha256": "cd91728dada57b9083122f0991dad5de4010287f91847bca4ac828d6402080e4",\n      "decoded_sha256": "ba032274d2b0ccb25abcc1967e904d0113448ca9b7e8c6b344b46dc25e7239bb",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "155312",\n      "source_id": "155312",\n      "attribution": "- [3-155312-A.ogg]: clip derived from Dog Barking (http://www.freesound.org/people/Jace/sounds/155312/) by Jace [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-155642-A-11",\n      "filename": "3-155642-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-155642-A-11.wav",\n      "bytes": 441044,\n      "sha256": "2b3168d0ece00768528c830841cd6edd683367bfb1ae0da1788912713a8a895d",\n      "decoded_sha256": "817b6fa9871c10e688b092339050b4ab6ae16361010c7e0194c0b689314b922c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "155642",\n      "source_id": "155642",\n      "attribution": "- [3-155642-A.ogg]: clip derived from sea \\\\u200b\\\\u200bon the road, night, Rhodes (http://www.freesound.org/people/mario1298/sounds/155642/) by mario1298 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-155642-B-11",\n      "filename": "3-155642-B-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-155642-B-11.wav",\n      "bytes": 441044,\n      "sha256": "76de2c046bd684e4788f852b5c3bc56df90b97660627b197619d156eccd3f306",\n      "decoded_sha256": "c0ff95b98c58bccece2417ab1449608da4109b1efea6a111f10ecee0681f51a2",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "155642",\n      "source_id": "155642",\n      "attribution": "- [3-155642-B.ogg]: clip derived from sea \\\\u200b\\\\u200bon the road, night, Rhodes (http://www.freesound.org/people/mario1298/sounds/155642/) by mario1298 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-156558-A-21",\n      "filename": "3-156558-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-156558-A-21.wav",\n      "bytes": 441044,\n      "sha256": "6d28e8680e1e333d864bdf850f3e4964429e4f4f3d55e50f39f2894050bd0f56",\n      "decoded_sha256": "c651960a1978bdb99d008e7defd0e8e0b6c469593ce586e5b5c316e8139eed9b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "156558",\n      "source_id": "156558",\n      "attribution": "- [3-156558-A.ogg]: clip derived from sneeze001.wav (http://www.freesound.org/people/bennyConsulting/sounds/156558/) by bennyConsulting [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-157149-A-10",\n      "filename": "3-157149-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-157149-A-10.wav",\n      "bytes": 441044,\n      "sha256": "83d1a7c62ffbf4f2241775502ec2fe3cf5c32fd69b4c107d8ca66b21d08bcda9",\n      "decoded_sha256": "e116f4b627860a8f87332b137dc6471af6c00a6aed6ecd844ed90698ac0578bc",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "157149",\n      "source_id": "157149",\n      "attribution": "- [3-157149-A.ogg]: clip derived from raininwoods.aiff (http://www.freesound.org/people/klangfabrik/sounds/157149/) by klangfabrik [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-157187-A-12",\n      "filename": "3-157187-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-157187-A-12.wav",\n      "bytes": 441044,\n      "sha256": "732379827a29ef372d098553de376a66adaac8c32b357221295a9bc7ebba84f0",\n      "decoded_sha256": "9957e55091aee03836cab9be184ad8bec50b11b043f7501805af939cba4cbdcd",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "157187",\n      "source_id": "157187",\n      "attribution": "- [3-157187-A.ogg]: clip derived from Campfire Crackle 3 (http://www.freesound.org/people/rbirdwise/sounds/157187/) by rbirdwise [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-157487-A-10",\n      "filename": "3-157487-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-157487-A-10.wav",\n      "bytes": 441044,\n      "sha256": "2acadf44d110c13b39bf17d6db9afa13bb4db19c8b3147b1a61b4f492caf1e97",\n      "decoded_sha256": "05ac3c746ef9484fc4011e4cc8b3d308898083a044e68ee95891b509b23c31bf",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "157487",\n      "source_id": "157487",\n      "attribution": "- [3-157487-A.ogg]: clip derived from rain_near_smooth.aif (http://www.freesound.org/people/loopbasedmusic/sounds/157487/) by loopbasedmusic [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-157615-A-10",\n      "filename": "3-157615-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-157615-A-10.wav",\n      "bytes": 441044,\n      "sha256": "5d3ac19904b4f7f0545f20b5a4d31d0cc9d97c5f555efc42b0484b41418052fa",\n      "decoded_sha256": "8a4d1d918998676034425b4db0497f91d5b4aa91f50854bacbb5124ca83d04c0",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "157615",\n      "source_id": "157615",\n      "attribution": "- [3-157615-A.ogg]: clip derived from rain recording.mp3 (http://www.freesound.org/people/Keygenerator/sounds/157615/) by Keygenerator [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-157695-A-0",\n      "filename": "3-157695-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-157695-A-0.wav",\n      "bytes": 441044,\n      "sha256": "7a6099b6ef5f58af8b6f9a9deda41cc0d8efe5a0fd60fdd638a711523674903a",\n      "decoded_sha256": "0d6fed3fcb6f1d74dafc7c9f8ea73ca85f1128e8a82db21c32bffe321db55260",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "157695",\n      "source_id": "157695",\n      "attribution": "- [3-157695-A.ogg]: clip derived from Dog on a chain (http://www.freesound.org/people/urupin/sounds/157695/) by urupin [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-158476-A-12",\n      "filename": "3-158476-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-158476-A-12.wav",\n      "bytes": 441044,\n      "sha256": "6d5eb5cbb48457fddc3647fa4750e1c5d3c8f5121756182206adfbf9c9eedc50",\n      "decoded_sha256": "6d7a02a5e93ade3368961a33861e07ca897df8eea4e2692a74585ed79c39cf70",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "158476",\n      "source_id": "158476",\n      "attribution": "- [3-158476-A.ogg]: clip derived from fire burning.mp3 (http://www.freesound.org/people/SoundCollectah/sounds/158476/) by SoundCollectah [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-163288-A-1",\n      "filename": "3-163288-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-163288-A-1.wav",\n      "bytes": 441044,\n      "sha256": "941201c0fc36b34f7b84ed0b89342c7d8a66576a526c68d4992265f4b81646fe",\n      "decoded_sha256": "801410296a2a317e748a1a026b4139379bc8019d7021014f5115a4bc84565424",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "163288",\n      "source_id": "163288",\n      "attribution": "- [3-163288-A.ogg]: clip derived from Rooster_Mono.wav (http://www.freesound.org/people/cclaretc/sounds/163288/) by cclaretc [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-163459-A-0",\n      "filename": "3-163459-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-163459-A-0.wav",\n      "bytes": 441044,\n      "sha256": "fede0d9dcbf6c0252880acf39d411d62e0060e9c9ff3bda9135b3dc2bccb41fe",\n      "decoded_sha256": "1a4766e8594c34b48a998a5fcf4d55369ac62a85785b991a55f9b87f86543917",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "163459",\n      "source_id": "163459",\n      "attribution": "- [3-163459-A.ogg]: clip derived from LBS_FX DOG Small Alert Bark001.wav (http://www.freesound.org/people/LittleBigSounds/sounds/163459/) by LittleBigSounds [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-164120-A-11",\n      "filename": "3-164120-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-164120-A-11.wav",\n      "bytes": 441044,\n      "sha256": "adcd03bf5b6a5c220cfbc4382ba6aa6873bedef53d0034a4dc5e2f0f6a183cb4",\n      "decoded_sha256": "90944ad20f7fdaccaff7812da17c47f8283d5047c9aed4c9af960dd6c7536277",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "164120",\n      "source_id": "164120",\n      "attribution": "- [3-164120-A.ogg]: clip derived from waves003.wav (http://www.freesound.org/people/sonitusetaudio/sounds/164120/) by sonitusetaudio [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-164630-A-11",\n      "filename": "3-164630-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-164630-A-11.wav",\n      "bytes": 441044,\n      "sha256": "f51da90204e7194f92090de1872a27968ffda90c53fc694ca9d4781233fb6dad",\n      "decoded_sha256": "e23d7e3a4e3f7ff45d99c981ca5e3f2e68361f89d23af3aee70e46ff9da2f735",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "164630",\n      "source_id": "164630",\n      "attribution": "- [3-164630-A.ogg]: clip derived from sea2.wav (http://www.freesound.org/people/schafferdavid/sounds/164630/) by schafferdavid [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-164688-A-38",\n      "filename": "3-164688-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-164688-A-38.wav",\n      "bytes": 441044,\n      "sha256": "9ab49e668972a4d80ce6afdc17b03e8093d9a63a151e4f07b14cf6322d5bf68c",\n      "decoded_sha256": "c13657e89a6233ed2fe5104673a15de0d06395075016e5f47ddcac9859fc10c4",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "164688",\n      "source_id": "164688",\n      "attribution": "- [3-164688-A.ogg]: clip derived from Ticking_clock.wav (http://www.freesound.org/people/Adam_N/sounds/164688/) by Adam_N [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-165856-A-41",\n      "filename": "3-165856-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-165856-A-41.wav",\n      "bytes": 441044,\n      "sha256": "4382daa9057a7ccd87814f783f8cc72260a243e989cab1680977711cc2117d6b",\n      "decoded_sha256": "85feae18b869496d7bbfecf32f052177ffd8efc9293d8b656774ea8f792eb33f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "165856",\n      "source_id": "165856",\n      "attribution": "- [3-165856-A.ogg]: clip derived from Chainsawing.wav (http://www.freesound.org/people/Puniho/sounds/165856/) by Puniho [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-166422-A-11",\n      "filename": "3-166422-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-166422-A-11.wav",\n      "bytes": 441044,\n      "sha256": "3ecc909c603753417b812b66a4846f3159bf854cc22b695f5018948bd1035fc9",\n      "decoded_sha256": "109a768b56c87292de24a835e81014871f009003111405012fe3c621b0e20cd0",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "166422",\n      "source_id": "166422",\n      "attribution": "- [3-166422-A.ogg]: clip derived from wave sound (http://www.freesound.org/people/GrooveOption/sounds/166422/) by GrooveOption [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-170015-A-0",\n      "filename": "3-170015-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-170015-A-0.wav",\n      "bytes": 441044,\n      "sha256": "7aff1d04c4c381636e0bfc7d9dfbf76509d0f30674fc542bb8f8ed757f29e29d",\n      "decoded_sha256": "14fd7d065257e1d4a1349c6faddae8d00ae6a34631782b197868605ab5264694",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "170015",\n      "source_id": "170015",\n      "attribution": "- [3-170015-A.ogg]: clip derived from one bark of a poodle dog (http://www.freesound.org/people/fabiopx/sounds/170015/) by fabiopx [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-170377-A-38",\n      "filename": "3-170377-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-170377-A-38.wav",\n      "bytes": 441044,\n      "sha256": "19eb30064eb159ebaa25a91a2435ff5f3eea28a536c9e11ca983dcbd8ce6d93b",\n      "decoded_sha256": "130e49be03e68c745364bff23ad760dc07cfb9843b65a74cd48badd48a81c835",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "170377",\n      "source_id": "170377",\n      "attribution": "- [3-170377-A.ogg]: clip derived from TicTac of a wall clock (http://www.freesound.org/people/felix.blume/sounds/170377/) by felix.blume [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-170383-A-38",\n      "filename": "3-170383-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-170383-A-38.wav",\n      "bytes": 441044,\n      "sha256": "04e08776e964a521d01760abfced1577572fc873f43a93a0039f41dcf61f9e68",\n      "decoded_sha256": "ccc2dd3ac39a4f7de13d608d2fbd3f302ec4c0fc625414d972ae88b03a5a74b7",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "170383",\n      "source_id": "170383",\n      "attribution": "- [3-170383-A.ogg]: clip derived from Cuckoo clock (http://www.freesound.org/people/finalCrystine/sounds/170383/) by finalCrystine [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-171012-A-38",\n      "filename": "3-171012-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-171012-A-38.wav",\n      "bytes": 441044,\n      "sha256": "caec02091ae6803d4dec5136f0aa6d539a8788609d83063409bc3b534ae21024",\n      "decoded_sha256": "8ef125b19f04aea3d30763016a0b2520ef21818ed1e07ce845bc99d1a40123d2",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "171012",\n      "source_id": "171012",\n      "attribution": "- [3-171012-A.ogg]: clip derived from Tick 3.aif (http://www.freesound.org/people/Goup_1/sounds/171012/) by Goup_1 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-171041-A-38",\n      "filename": "3-171041-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-171041-A-38.wav",\n      "bytes": 441044,\n      "sha256": "2101ae519c1e5ac0fb0cd6a516346fc1361fb277077b2998c9666c12465811c7",\n      "decoded_sha256": "055df692ab73265047a813f902e90e3d6ea3003dca77b8602820bfe8e41b2d84",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "171041",\n      "source_id": "171041",\n      "attribution": "- [3-171041-A.ogg]: clip derived from Old Electromechanical Clock Ticking \\"Yantar\\".wav (http://www.freesound.org/people/ST303/sounds/171041/) by ST303 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-180256-A-0",\n      "filename": "3-180256-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-180256-A-0.wav",\n      "bytes": 441044,\n      "sha256": "39ed8c8b60974843511f0f53e33e37f670f9cf22eebcb8b23798d52e0309127b",\n      "decoded_sha256": "f394216eec59166b47b298fafe0f86e73eaebd92b3cc85251c4bf50bedf4c868",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "180256",\n      "source_id": "180256",\n      "attribution": "- [3-180256-A.ogg]: clip derived from Dog Barking - Shih Tzu Pug (http://www.freesound.org/people/bspiller5/sounds/180256/) by bspiller5 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-180977-A-0",\n      "filename": "3-180977-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-180977-A-0.wav",\n      "bytes": 441044,\n      "sha256": "d5d290a12f7aac1e5a5cee44e7a4d6033ee9f9999846f84b2278f7fbd6516df2",\n      "decoded_sha256": "44c6e676114019c6f8c9c45d273ee7c71e92542569c3d682d58f43f6834c3e95",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "180977",\n      "source_id": "180977",\n      "attribution": "- [3-180977-A.ogg]: clip derived from Barking Dog (http://www.freesound.org/people/Suprasummun/sounds/180977/) by Suprasummun [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-187710-A-11",\n      "filename": "3-187710-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-187710-A-11.wav",\n      "bytes": 441044,\n      "sha256": "97714f535ad104a9eac0b85552f37b97e05b1d18f67223d502d201503b02dbaa",\n      "decoded_sha256": "b549b819c7af308d073038a50f835bbd9cefe751974200dbb860bccfa7b97498",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "187710",\n      "source_id": "187710",\n      "attribution": "- [3-187710-A.ogg]: clip derived from beach_short02.flac (http://www.freesound.org/people/Aiyumi/sounds/187710/) by Aiyumi [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-65748-A-12",\n      "filename": "3-65748-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-65748-A-12.wav",\n      "bytes": 441044,\n      "sha256": "e829acb3fa5e83e70139e275435c6bf63b2400d85456d34490633d4e16f4309e",\n      "decoded_sha256": "62a04ea1afeab092d515844269c05d8f892460066e16383ec32f336ae7b44c0d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "65748",\n      "source_id": "65748",\n      "attribution": "- [3-65748-A.ogg]: clip derived from 20081231.fire.02.wav (http://www.freesound.org/people/dobroide/sounds/65748/) by dobroide [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "3-68630-A-40",\n      "filename": "3-68630-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-68630-A-40.wav",\n      "bytes": 441044,\n      "sha256": "f085e649b44c0ff59ce93228fd367da81c68f53e5f932fb3c2dbbcc66c997a66",\n      "decoded_sha256": "6edbf2ddb1317654db83be6ab517a00608417590321424df8cc2caa81eeafbbc",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "68630",\n      "source_id": "68630",\n      "attribution": "- [3-68630-A.ogg]: clip derived from helicopter take off distance.wav (http://www.freesound.org/people/bwav/sounds/68630/) by bwav [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-68630-B-40",\n      "filename": "3-68630-B-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-68630-B-40.wav",\n      "bytes": 441044,\n      "sha256": "fc18f3d9663d8f180bf65e1747543816ceebc6a710de319de74017a8eb2e4b15",\n      "decoded_sha256": "522f899b5f618233216b043bf3070557a31b113a8848008655496772470773c4",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "68630",\n      "source_id": "68630",\n      "attribution": "- [3-68630-B.ogg]: clip derived from helicopter take off distance.wav (http://www.freesound.org/people/bwav/sounds/68630/) by bwav [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "3-68630-C-40",\n      "filename": "3-68630-C-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/3-68630-C-40.wav",\n      "bytes": 441044,\n      "sha256": "0358b8428bc77a109345b4f36f92fcdae1795efdf1928e74a4bbf35d58b607ee",\n      "decoded_sha256": "d443613b4375869609a3227759442171de14580581d4943557f7f45a99a8b7dd",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "train",\n      "fold": 3,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "68630",\n      "source_id": "68630",\n      "attribution": "- [3-68630-C.ogg]: clip derived from helicopter take off distance.wav (http://www.freesound.org/people/bwav/sounds/68630/) by bwav [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-125929-A-40",\n      "filename": "4-125929-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-125929-A-40.wav",\n      "bytes": 441044,\n      "sha256": "adfb82d86a3b2a7ac8fcf85c98f6b222ef5b0e47991cf6c0047e3d99e2f11443",\n      "decoded_sha256": "4dbb51fac54ea569651d3c488b01a9ce4361810ee3493e367ae5d3c9947d15b4",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "125929",\n      "source_id": "125929",\n      "attribution": "- [4-125929-A.ogg]: clip derived from helicopter-construction.aif (http://www.freesound.org/people/alienistcog/sounds/125929/) by alienistcog [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-149294-A-41",\n      "filename": "4-149294-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-149294-A-41.wav",\n      "bytes": 441044,\n      "sha256": "6f17cda330dc854816ff6fa810a4535882077f1db34f7da8f6a046661164b958",\n      "decoded_sha256": "6ca5e26fa5183af7bc623c3f779f5d7c2f37992e6061bfd7d9da5a10dd998870",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "149294",\n      "source_id": "149294",\n      "attribution": "- [4-149294-A.ogg]: clip derived from chainsaw_cutting01.wav (http://www.freesound.org/people/Vocalphobic/sounds/149294/) by Vocalphobic [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-149294-B-41",\n      "filename": "4-149294-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-149294-B-41.wav",\n      "bytes": 441044,\n      "sha256": "93988074f02db3b15f57181ee8bf2707a2151190dd7b674d110bd88e32a57506",\n      "decoded_sha256": "5797c6a9cba19e17f27ad806428d0891115f4361b59cd0f220c7a4865460a64d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "149294",\n      "source_id": "149294",\n      "attribution": "- [4-149294-B.ogg]: clip derived from chainsaw_cutting01.wav (http://www.freesound.org/people/Vocalphobic/sounds/149294/) by Vocalphobic [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-156843-A-21",\n      "filename": "4-156843-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-156843-A-21.wav",\n      "bytes": 441044,\n      "sha256": "915d649c4fbc6df93b6b6a94f45732e02e4a15728531ac114f867c969e989cbd",\n      "decoded_sha256": "70a20c494ff568978b5a035ab4b54382f01f471f5250314cd4537464b58c3f7a",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "156843",\n      "source_id": "156843",\n      "attribution": "- [4-156843-A.ogg]: clip derived from Sneeze Male JGray.wav (http://www.freesound.org/people/JGrayArt/sounds/156843/) by JGrayArt [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-156844-A-21",\n      "filename": "4-156844-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-156844-A-21.wav",\n      "bytes": 441044,\n      "sha256": "6fbfe7a57440e68e7a3c1e46dc039df88cdf64f765c9eb2402e8352382078f78",\n      "decoded_sha256": "6051f4ea72029d80bb041dcc06e6eb11dccb43c706ad810dab1d560d0286cf2a",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "156844",\n      "source_id": "156844",\n      "attribution": "- [4-156844-A.ogg]: clip derived from Sneeze Female x3 JGray.wav (http://www.freesound.org/people/JGrayArt/sounds/156844/) by JGrayArt [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-157297-A-21",\n      "filename": "4-157297-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-157297-A-21.wav",\n      "bytes": 441044,\n      "sha256": "81500a8a8c1c56a714db75c93abb4a040a8e0bdd07ee920076a32c715452bc89",\n      "decoded_sha256": "6ab426d6e87e0db9c4b7e50d8e327aa7fb10930362483ec002186cfa075b542f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "157297",\n      "source_id": "157297",\n      "attribution": "- [4-157297-A.ogg]: clip derived from Sneeze.wav (http://www.freesound.org/people/husky70/sounds/157297/) by husky70 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-157611-A-41",\n      "filename": "4-157611-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-157611-A-41.wav",\n      "bytes": 441044,\n      "sha256": "9a71c693e78177287023ad26d3192535104f8a5fa45290150a3f4db647ae5226",\n      "decoded_sha256": "ad70a5801f000c2a67d13e8fb4ce3cb079731abe6a328d9185dd4d303aaf7001",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "157611",\n      "source_id": "157611",\n      "attribution": "- [4-157611-A.ogg]: clip derived from Exterior_Chainsaw.wav (http://www.freesound.org/people/DJillcom/sounds/157611/) by DJillcom [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-157611-B-41",\n      "filename": "4-157611-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-157611-B-41.wav",\n      "bytes": 441044,\n      "sha256": "e2c2b22cd1b32e80cb5bd0cb5d4e5e3f593fddb609b32ce13e92ee1be0409a7d",\n      "decoded_sha256": "44f1808e6ba8b13760eb5a941720f7124667ed379ae260b8d7bb9042536f9dd3",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "157611",\n      "source_id": "157611",\n      "attribution": "- [4-157611-B.ogg]: clip derived from Exterior_Chainsaw.wav (http://www.freesound.org/people/DJillcom/sounds/157611/) by DJillcom [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-160999-A-10",\n      "filename": "4-160999-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-160999-A-10.wav",\n      "bytes": 441044,\n      "sha256": "c85ef3d5cbd86ba80a7f089277d66f165e36b8dcf8070643ddc7ec73eb4f5dcb",\n      "decoded_sha256": "fd62bee54887d51807ba7b689b2117844e00a55aac66ba220201c3a6e69372b5",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "160999",\n      "source_id": "160999",\n      "attribution": "- [4-160999-A.ogg]: clip derived from rain.MP3 (http://www.freesound.org/people/Mafon2/sounds/160999/) by Mafon2 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-161127-A-10",\n      "filename": "4-161127-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-161127-A-10.wav",\n      "bytes": 441044,\n      "sha256": "3ea9076dd2ee68b67bd0707652a0c0ca77811b0664426e10e6860c187401af89",\n      "decoded_sha256": "de272d1a03333d5416743262350c02aced357b849629673400e2e1fa956a5f9d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "161127",\n      "source_id": "161127",\n      "attribution": "- [4-161127-A.ogg]: clip derived from Heavy rain.wav (http://www.freesound.org/people/reelworldstudio/sounds/161127/) by reelworldstudio [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-161579-A-40",\n      "filename": "4-161579-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-161579-A-40.wav",\n      "bytes": 441044,\n      "sha256": "74bfa5c0c1d5926aa20fa25d4b57fc3cd445208a23200e282f5f603dc2d0fdc8",\n      "decoded_sha256": "d9110828a2527cba3c1dfce72729e864fbbea0c3599819d495151550b66606d6",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "161579",\n      "source_id": "161579",\n      "attribution": "- [4-161579-A.ogg]: clip derived from Helicopter Flyover (http://www.freesound.org/people/doctor567/sounds/161579/) by doctor567 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-161579-B-40",\n      "filename": "4-161579-B-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-161579-B-40.wav",\n      "bytes": 441044,\n      "sha256": "4f110f1d5c7d4185c7ed5f5a678f9233871126a826698b9eec1e4b2f3768aa1d",\n      "decoded_sha256": "61f379af3613998193fee0a162a39a0f8dafe476d3f110a208deed3b8293ee07",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "161579",\n      "source_id": "161579",\n      "attribution": "- [4-161579-B.ogg]: clip derived from Helicopter Flyover (http://www.freesound.org/people/doctor567/sounds/161579/) by doctor567 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-163264-A-10",\n      "filename": "4-163264-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-163264-A-10.wav",\n      "bytes": 441044,\n      "sha256": "c2f5737bfafb1a3cb2f06a3d97d291e33901051ae4ef5b25c6f19093453c0ead",\n      "decoded_sha256": "0fec1b568a4527d72d8bf58e84b2c65531adf36ffdc76df15d37fa095df1a911",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "163264",\n      "source_id": "163264",\n      "attribution": "- [4-163264-A.ogg]: clip derived from Light rain (http://www.freesound.org/people/Zepp2010/sounds/163264/) by Zepp2010 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-164021-A-1",\n      "filename": "4-164021-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-164021-A-1.wav",\n      "bytes": 441044,\n      "sha256": "95b7f782e42027588c92db9665f009b042c59436dcee563a01d6d71af6e1f0e8",\n      "decoded_sha256": "6d6a1695957d3e28f1d0355767f3f4ab00188012c62ea08a51e37deb21be80aa",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "164021",\n      "source_id": "164021",\n      "attribution": "- [4-164021-A.ogg]: clip derived from Cock-a-doodle-doo (http://www.freesound.org/people/kangaroovindaloo/sounds/164021/) by kangaroovindaloo [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-164064-A-1",\n      "filename": "4-164064-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-164064-A-1.wav",\n      "bytes": 441044,\n      "sha256": "08b3564898604da9f5699f4dee35e79ee06c9f7025c85f98b5d35d3efdc42b0d",\n      "decoded_sha256": "ef33d9cde20f04a82cb88a2e73d0f6f03c54b62f15c09e171f4d1f5eb72e3f91",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "164064",\n      "source_id": "164064",\n      "attribution": "- [4-164064-A.ogg]: clip derived from Rooster_Ste.wav (http://www.freesound.org/people/cclaretc/sounds/164064/) by cclaretc [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-164064-B-1",\n      "filename": "4-164064-B-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-164064-B-1.wav",\n      "bytes": 441044,\n      "sha256": "60ddaf0fd58578373fe5f2227753c1363b95fcf5b63bceaeeb08f54e483b540b",\n      "decoded_sha256": "cf226bcdb3e98855c7034af618d236c078b9fc3b022b5eaada3925467f3fdb15",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "164064",\n      "source_id": "164064",\n      "attribution": "- [4-164064-B.ogg]: clip derived from Rooster_Ste.wav (http://www.freesound.org/people/cclaretc/sounds/164064/) by cclaretc [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-164064-C-1",\n      "filename": "4-164064-C-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-164064-C-1.wav",\n      "bytes": 441044,\n      "sha256": "34e4f9b7454361e903c43ea59cd69df272c9b308084598a4108c575a41a80113",\n      "decoded_sha256": "141bc2c7c351481f973d8c639f40009952b2d009a1f49774e30d1942f7908d10",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "164064",\n      "source_id": "164064",\n      "attribution": "- [4-164064-C.ogg]: clip derived from Rooster_Ste.wav (http://www.freesound.org/people/cclaretc/sounds/164064/) by cclaretc [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-164206-A-10",\n      "filename": "4-164206-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-164206-A-10.wav",\n      "bytes": 441044,\n      "sha256": "75f81596617c19cb01f4e0a0422798f100d1bedd796cd13aa59687b2bac13d16",\n      "decoded_sha256": "8229c5d734ac81e64d5e257ad5f7bb0e35101c68df33813f6ca062161ec93a4c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "164206",\n      "source_id": "164206",\n      "attribution": "- [4-164206-A.ogg]: clip derived from Rain very heavy with thunder.wav (http://www.freesound.org/people/Danjocross/sounds/164206/) by Danjocross [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-164661-A-12",\n      "filename": "4-164661-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-164661-A-12.wav",\n      "bytes": 441044,\n      "sha256": "5dba3d3689ad470168a93c321674fb1d0e152a939bc034dcfe88c7a055e6d5f5",\n      "decoded_sha256": "abb644f2526f6950fdadc47e01d644d2f160d25c28959f5dfae63670319e367c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "164661",\n      "source_id": "164661",\n      "attribution": "- [4-164661-A.ogg]: clip derived from Fire.wav (http://www.freesound.org/people/Adam_N/sounds/164661/) by Adam_N [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-164661-B-12",\n      "filename": "4-164661-B-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-164661-B-12.wav",\n      "bytes": 441044,\n      "sha256": "e8db809460aaf1d4cad3458762e5b60c31e88efdc00b33a6d6a7c3bd83306ec4",\n      "decoded_sha256": "6029efc9e7a9cd2893e53019a8a56ab8a3261f0bf6fd2d0b7c217fb42a923db1",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "164661",\n      "source_id": "164661",\n      "attribution": "- [4-164661-B.ogg]: clip derived from Fire.wav (http://www.freesound.org/people/Adam_N/sounds/164661/) by Adam_N [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-164859-A-1",\n      "filename": "4-164859-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-164859-A-1.wav",\n      "bytes": 441044,\n      "sha256": "96f91f5dc4b8a14ef17926b0adc7229e2161b4e61e1d079ac45bed2a085a6b08",\n      "decoded_sha256": "4a0cd8360a1ce3a2030fbdb0637a61fb87f7e5879b6ac81db6568f6d632fb760",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "164859",\n      "source_id": "164859",\n      "attribution": "- [4-164859-A.ogg]: clip derived from rooster.wav (http://www.freesound.org/people/bob58/sounds/164859/) by bob58 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-165823-A-41",\n      "filename": "4-165823-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-165823-A-41.wav",\n      "bytes": 441044,\n      "sha256": "59bb8e0b1a116bbea7ea48ff6898f37a22a2e875437d480a8ae990fec37ab5ee",\n      "decoded_sha256": "f3c4d7e5ac4e9fcce4234b7a2b476701409bf2020b5139922a988d52b6b6bebc",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "165823",\n      "source_id": "165823",\n      "attribution": "- [4-165823-A.ogg]: clip derived from Chainsaw Rampage in the Forest (http://www.freesound.org/people/unfa/sounds/165823/) by unfa [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-165823-B-41",\n      "filename": "4-165823-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-165823-B-41.wav",\n      "bytes": 441044,\n      "sha256": "b65207c334c37b5ce88da303197f15d1b2e21ee3002b6d644d06c9f9cd5c0a6e",\n      "decoded_sha256": "8bd1245ef50f0e3ed231ee2b3a0ca14d12e9d5ee2ea7b5e54f7c824b3b84ff7e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "165823",\n      "source_id": "165823",\n      "attribution": "- [4-165823-B.ogg]: clip derived from Chainsaw Rampage in the Forest (http://www.freesound.org/people/unfa/sounds/165823/) by unfa [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-166661-A-10",\n      "filename": "4-166661-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-166661-A-10.wav",\n      "bytes": 441044,\n      "sha256": "a958fd56bfff6c34c896c8ca4e5f69705b47a446edcb57060c492f65007d277d",\n      "decoded_sha256": "f18ce55ee13fa588cee1ab8f29691dee5c42abf2a1649e3a9a6d35ee83143507",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "166661",\n      "source_id": "166661",\n      "attribution": "- [4-166661-A.ogg]: clip derived from pheiland (http://www.freesound.org/people/pheiland/sounds/166661/) by pheiland [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-167063-A-11",\n      "filename": "4-167063-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-167063-A-11.wav",\n      "bytes": 441044,\n      "sha256": "3cfc8edd94c4efc892a80151ed83aff7febcab1219d5c1e41abe5e4bf6c7d8ad",\n      "decoded_sha256": "0e3d661b191032693d4364a312ba6b44c9972f1507cf132f46700aadecb58f38",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "167063",\n      "source_id": "167063",\n      "attribution": "- [4-167063-A.ogg]: clip derived from Close sea waves (http://www.freesound.org/people/slag777/sounds/167063/) by slag777 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-167063-B-11",\n      "filename": "4-167063-B-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-167063-B-11.wav",\n      "bytes": 441044,\n      "sha256": "0dd8330a8d102d1192dca1e0c43e83602f3f308994aaf8f9f68bb8b1c682cb4a",\n      "decoded_sha256": "c07f9a4ae4cc8be043bb5b4f1e1ba9320070c8e4e72803b3365f4919b837436d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "167063",\n      "source_id": "167063",\n      "attribution": "- [4-167063-B.ogg]: clip derived from Close sea waves (http://www.freesound.org/people/slag777/sounds/167063/) by slag777 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-167063-C-11",\n      "filename": "4-167063-C-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-167063-C-11.wav",\n      "bytes": 441044,\n      "sha256": "0c11db409fe2f395dbd8d34d83b9d96cb10d8e971cda562120d7fea19d61d62d",\n      "decoded_sha256": "2e70f3cf770f52b8ed11c510667858686916a93a5378b5c393449fe2b66cf1c4",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "167063",\n      "source_id": "167063",\n      "attribution": "- [4-167063-C.ogg]: clip derived from Close sea waves (http://www.freesound.org/people/slag777/sounds/167063/) by slag777 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-167077-A-20",\n      "filename": "4-167077-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-167077-A-20.wav",\n      "bytes": 441044,\n      "sha256": "43a894a23a4a1bed053e2ca040f7029a5d9029269173d8bac1c650c52c03e937",\n      "decoded_sha256": "6796b69af0d63d9844cb6eb0dc5dc4f996c24d3c2b0d5c49cd7d92b3c3f2785a",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "167077",\n      "source_id": "167077",\n      "attribution": "- [4-167077-A.ogg]: clip derived from JM_HUMAN_Baby 01c - Crying.wav (http://www.freesound.org/people/Julien%20Matthey/sounds/167077/) by Julien Matthey [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-167077-B-20",\n      "filename": "4-167077-B-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-167077-B-20.wav",\n      "bytes": 441044,\n      "sha256": "0dd0122251e1b989c8309596e92ea95d06989db8b37d28fee3c485842f9f0014",\n      "decoded_sha256": "fde6f4398e7d00c9b2a60fad7109f0c9077d2dbe3b8a44a6b31855ab1364c619",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "167077",\n      "source_id": "167077",\n      "attribution": "- [4-167077-B.ogg]: clip derived from JM_HUMAN_Baby 01c - Crying.wav (http://www.freesound.org/people/Julien%20Matthey/sounds/167077/) by Julien Matthey [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-167077-C-20",\n      "filename": "4-167077-C-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-167077-C-20.wav",\n      "bytes": 441044,\n      "sha256": "c9ec416d8fb525a66d084250b4fbbf041e8bd5f31cc1c7c809e381a45902cc06",\n      "decoded_sha256": "9eabf0620af2743a0f24ef515e285641cac304c8f20920b850a0495621ce2972",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "167077",\n      "source_id": "167077",\n      "attribution": "- [4-167077-C.ogg]: clip derived from JM_HUMAN_Baby 01c - Crying.wav (http://www.freesound.org/people/Julien%20Matthey/sounds/167077/) by Julien Matthey [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-167642-A-21",\n      "filename": "4-167642-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-167642-A-21.wav",\n      "bytes": 441044,\n      "sha256": "38591180e9da943cd3e472e1ad80826a56308bf473f48ee6ebc909cf8584d543",\n      "decoded_sha256": "3186f9b925fb145f7515646c6dd9bd9d8ad3822c9f032afceaf04249245f49e8",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "167642",\n      "source_id": "167642",\n      "attribution": "- [4-167642-A.ogg]: clip derived from Sneeze 01.aiff (http://www.freesound.org/people/SoundCollectah/sounds/167642/) by SoundCollectah [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-169127-A-41",\n      "filename": "4-169127-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-169127-A-41.wav",\n      "bytes": 441044,\n      "sha256": "039a8b4ff220493783bc34bbf5f08126f28ea061a50f723f5f334437669484ee",\n      "decoded_sha256": "f2b9a68940e21abe57eb1d27980a4fa3feb3942acf84f85aeba482120877453e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "169127",\n      "source_id": "169127",\n      "attribution": "- [4-169127-A.ogg]: clip derived from cutting_tree.wav (http://www.freesound.org/people/fkurz/sounds/169127/) by fkurz [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-169127-B-41",\n      "filename": "4-169127-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-169127-B-41.wav",\n      "bytes": 441044,\n      "sha256": "6417b5f63d03cd517df89d1bb598b9e991debfd43a64d3b168d76e75866d6a42",\n      "decoded_sha256": "bb5409c4ebc2b90569ac5152e6b0fda388b355bbe67a3a097b807ff347ac98eb",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "169127",\n      "source_id": "169127",\n      "attribution": "- [4-169127-B.ogg]: clip derived from cutting_tree.wav (http://www.freesound.org/people/fkurz/sounds/169127/) by fkurz [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-170078-A-1",\n      "filename": "4-170078-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-170078-A-1.wav",\n      "bytes": 441044,\n      "sha256": "542476574580295ef7a930f25d115cd26e77c7e9a386088908ad5953ea456228",\n      "decoded_sha256": "a4a7d0e959afdce1d03b4fa48b94db8adc7763cc3f682d8667f59bc6564dc54f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "170078",\n      "source_id": "170078",\n      "attribution": "- [4-170078-A.ogg]: clip derived from Rooster at Max\'s.mp3 (http://www.freesound.org/people/dudermensch/sounds/170078/) by dudermensch [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-170247-A-12",\n      "filename": "4-170247-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-170247-A-12.wav",\n      "bytes": 441044,\n      "sha256": "19a2e37237b10d02f8c54281ecacd17d4e7af1c4b69df5c1db3831fe55d996a7",\n      "decoded_sha256": "1f6e5688d620dd150f6d4746be95e4a735ee3ffaaf93ab213f5e58597da4a905",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "170247",\n      "source_id": "170247",\n      "attribution": "- [4-170247-A.ogg]: clip derived from campfire_01.wav (http://www.freesound.org/people/matucha/sounds/170247/) by matucha [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-170247-B-12",\n      "filename": "4-170247-B-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-170247-B-12.wav",\n      "bytes": 441044,\n      "sha256": "ff4985fe31123d45350cfc29e048d6817a1cdfe27059b5ccb55546d67c668bee",\n      "decoded_sha256": "8478a7f624c62f6f275bb6a3c8bac4fead68ee8708dc54f936f3d586c2ffcf9e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "170247",\n      "source_id": "170247",\n      "attribution": "- [4-170247-B.ogg]: clip derived from campfire_01.wav (http://www.freesound.org/people/matucha/sounds/170247/) by matucha [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-171207-A-12",\n      "filename": "4-171207-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-171207-A-12.wav",\n      "bytes": 441044,\n      "sha256": "c2b852693789c4d835c9e6e5584788298401fb2cae183353f6388cfc99f30027",\n      "decoded_sha256": "c41b9e8c6f2934d05f25079c412bae1138a96ab3d26c37a6d9eaad6569dcd197",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "171207",\n      "source_id": "171207",\n      "attribution": "- [4-171207-A.ogg]: clip derived from Wood-burning stove (http://www.freesound.org/people/MrAuralization/sounds/171207/) by MrAuralization [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-171519-A-21",\n      "filename": "4-171519-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-171519-A-21.wav",\n      "bytes": 441044,\n      "sha256": "e639fa6e14d378ea3514bba8f0957b74174c88a9cec508d8925a5d8a138334db",\n      "decoded_sha256": "a885372400e9ab7055b16e705d27732b9214ee04d9f0041cfaa1cf8ddf105e84",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "171519",\n      "source_id": "171519",\n      "attribution": "- [4-171519-A.ogg]: clip derived from sneeze.wav (http://www.freesound.org/people/Zajjman/sounds/171519/) by Zajjman [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-175000-A-40",\n      "filename": "4-175000-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-175000-A-40.wav",\n      "bytes": 441044,\n      "sha256": "1955ca9fc02340cef8d66b125b0e2cdaabea757b1ee38fe9dc3df9646a50657e",\n      "decoded_sha256": "8944d1f3d6a0297efcfd32879d2ea214c02d062e28c52f105a296587fc478e06",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "175000",\n      "source_id": "175000",\n      "attribution": "- [4-175000-A.ogg]: clip derived from Helicopter Fly By (http://www.freesound.org/people/altfuture/sounds/175000/) by altfuture [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-175000-B-40",\n      "filename": "4-175000-B-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-175000-B-40.wav",\n      "bytes": 441044,\n      "sha256": "84b0d5dd09ad0fdca162eebabaacdf07f396a13df8402ab1d6909f348abd25c3",\n      "decoded_sha256": "080fc1969de0c73de3e144eb6210fd30b7be67f0f3baa4850502d63e8f55a1ba",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "175000",\n      "source_id": "175000",\n      "attribution": "- [4-175000-B.ogg]: clip derived from Helicopter Fly By (http://www.freesound.org/people/altfuture/sounds/175000/) by altfuture [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-175000-C-40",\n      "filename": "4-175000-C-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-175000-C-40.wav",\n      "bytes": 441044,\n      "sha256": "ef2f54b5d05ee09cd5b6e76af1e0465cd6f07a1f2a06d49eca912a995c0e48cd",\n      "decoded_sha256": "7916882a987513db69f5e719a310a7f16d6a111aa439aa237f0bbfb8b13ead48",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "175000",\n      "source_id": "175000",\n      "attribution": "- [4-175000-C.ogg]: clip derived from Helicopter Fly By (http://www.freesound.org/people/altfuture/sounds/175000/) by altfuture [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-175945-A-38",\n      "filename": "4-175945-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-175945-A-38.wav",\n      "bytes": 441044,\n      "sha256": "e21894f4cd41eeb1423fe7d6eadda4dfd32cbd686c639bbadbf3ab3bbb90bc28",\n      "decoded_sha256": "6a9d63223e2ee49e5993ef5b585ff298e82898cc6113e051f4e1fc329372963c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "175945",\n      "source_id": "175945",\n      "attribution": "- [4-175945-A.ogg]: clip derived from TimerRecording.wav (http://www.freesound.org/people/Ovaltine%20Vortex/sounds/175945/) by Ovaltine Vortex [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-177250-A-10",\n      "filename": "4-177250-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-177250-A-10.wav",\n      "bytes": 441044,\n      "sha256": "daae54d9bcfb4f31adfac8adbca4ab54c3543b8ed68c555b5a98c3fc20283358",\n      "decoded_sha256": "fcdeb2b29e9d2d6147de00098137c2e3573e87285eae3098ec6bc2ebc55d760a",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "177250",\n      "source_id": "177250",\n      "attribution": "- [4-177250-A.ogg]: clip derived from Rain and Storm (http://www.freesound.org/people/josephsardin/sounds/177250/) by josephsardin [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-179984-A-38",\n      "filename": "4-179984-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-179984-A-38.wav",\n      "bytes": 441044,\n      "sha256": "c39eef951ebd527500e4c8bae4e7d5c6638017dc093125f0d46d462573a8fea5",\n      "decoded_sha256": "a738ab5b71521c6dd1694a4814af55b6fa473c69daf22d838ae2bf08cc6750c6",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "179984",\n      "source_id": "179984",\n      "attribution": "- [4-179984-A.ogg]: clip derived from Ansonia Clock Strikes 11 (http://www.freesound.org/people/kangaroovindaloo/sounds/179984/) by kangaroovindaloo [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-180380-A-10",\n      "filename": "4-180380-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-180380-A-10.wav",\n      "bytes": 441044,\n      "sha256": "d73fdc45016ab102ed49683509d93eccb40b985ed9492fe17c3aecf50963a35b",\n      "decoded_sha256": "3f4e3fefaec39e37f1e41e5328328c9c180f8226d9c331b91c9833cc24470129",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "180380",\n      "source_id": "180380",\n      "attribution": "- [4-180380-A.ogg]: clip derived from rain.wav (http://www.freesound.org/people/aumguy/sounds/180380/) by aumguy [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-181035-A-38",\n      "filename": "4-181035-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-181035-A-38.wav",\n      "bytes": 441044,\n      "sha256": "6bd3751b298e3d567982a965519cf89c6fd6fd491d2d60e4271b7784015a0fd7",\n      "decoded_sha256": "f105be6324f8b4da596182eb312997a540c1059253479575d287b8299ea8e5ac",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "181035",\n      "source_id": "181035",\n      "attribution": "- [4-181035-A.ogg]: clip derived from Tick Tock Clock - Close Mic.wav (http://www.freesound.org/people/Toby_Crane/sounds/181035/) by Toby_Crane [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-181286-A-10",\n      "filename": "4-181286-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-181286-A-10.wav",\n      "bytes": 441044,\n      "sha256": "f4adde0e7a70a466a161325824b020a28ae3aaf76568436116c82b6b25d76fdf",\n      "decoded_sha256": "eb051f8456ade022871e8f01dadd7907eac98eeee458157fcdce643e683c149b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "181286",\n      "source_id": "181286",\n      "attribution": "- [4-181286-A.ogg]: clip derived from Wind and Sprinkling Rain (http://www.freesound.org/people/asbestos%20bill/sounds/181286/) by asbestos bill [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-181563-A-12",\n      "filename": "4-181563-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-181563-A-12.wav",\n      "bytes": 441044,\n      "sha256": "ddb4f826b26093badfbec99c52ae78fdf83b8f86a633864f220f7097660d7132",\n      "decoded_sha256": "17132527825a475d8de4372cff2878d788c9766d7ff63720299dd07f3f3eb399",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "181563",\n      "source_id": "181563",\n      "attribution": "- [4-181563-A.ogg]: clip derived from Fire Crackling 01.wav (http://www.freesound.org/people/kingsrow/sounds/181563/) by kingsrow [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-181865-A-38",\n      "filename": "4-181865-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-181865-A-38.wav",\n      "bytes": 441044,\n      "sha256": "0321714c36d8186de7b7089f372b853d76ffdd68f84c42872adb8ea3ba6c8f61",\n      "decoded_sha256": "7af63d73ba713a9e05947fce34cf7550f07a1a77b56ede7cfd5ba53da2698149",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "181865",\n      "source_id": "181865",\n      "attribution": "- [4-181865-A.ogg]: clip derived from Ticking Clock (http://www.freesound.org/people/LeoDeSafo/sounds/181865/) by LeoDeSafo [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-182368-A-12",\n      "filename": "4-182368-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-182368-A-12.wav",\n      "bytes": 441044,\n      "sha256": "c3ba5793bb74cbf7e92b678bc41f7ad4ab925b175c77e8320a755d92e4d1bad8",\n      "decoded_sha256": "fcbe465f4e46c75567f10e1fb0e267ef735052870d15a9b30a0d1f07bc6b5504",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "182368",\n      "source_id": "182368",\n      "attribution": "- [4-182368-A.ogg]: clip derived from Fire Crackling 02.mp3 (http://www.freesound.org/people/kingsrow/sounds/182368/) by kingsrow [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-182369-A-12",\n      "filename": "4-182369-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-182369-A-12.wav",\n      "bytes": 441044,\n      "sha256": "886fca3ecfcbd5fbfb52cb83d4c490f9e865b58f7992282220354e1731a9f2a7",\n      "decoded_sha256": "d8790527b31011006ea482f49e85298eb57d51316476a02778f8dcfa17812878",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "182369",\n      "source_id": "182369",\n      "attribution": "- [4-182369-A.ogg]: clip derived from Fire Crackling 01.mp3 (http://www.freesound.org/people/kingsrow/sounds/182369/) by kingsrow [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-182395-A-0",\n      "filename": "4-182395-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-182395-A-0.wav",\n      "bytes": 441044,\n      "sha256": "c249bcb8ef81d12516ad17c8a77f88c01952cdf70b9413732af5fb8b803f9c7d",\n      "decoded_sha256": "42c34448a4eb19a98a901ec83c5b29e5dffff8a3618f4bb85f8ab78b33b28e14",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "182395",\n      "source_id": "182395",\n      "attribution": "- [4-182395-A.ogg]: clip derived from Big_dog_bark_01.aiff (http://www.freesound.org/people/pgonsilva/sounds/182395/) by pgonsilva [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-182613-A-11",\n      "filename": "4-182613-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-182613-A-11.wav",\n      "bytes": 441044,\n      "sha256": "c8b796cb2da15e66ee4c6611f82c02a13f7eadd3629fc8f4b000d4574864c8e6",\n      "decoded_sha256": "316ac5d829c4882da2a4b1c6cc915a8a8d0b7cfbd168fefd36fa25b5a03dee19",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "182613",\n      "source_id": "182613",\n      "attribution": "- [4-182613-A.ogg]: clip derived from CFX-20130331-UK-DorsetSeaBeach02.wav (http://www.freesound.org/people/Carlvus/sounds/182613/) by Carlvus [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-182613-B-11",\n      "filename": "4-182613-B-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-182613-B-11.wav",\n      "bytes": 441044,\n      "sha256": "4cb8e67374474186ba13ae2f6ee19f748c05d805603240154dd1b658cbbf45d4",\n      "decoded_sha256": "40dfdfbacb84409a1bc9f1c081baa72cf1495bfcf728b7666dff2a63d15f277c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "182613",\n      "source_id": "182613",\n      "attribution": "- [4-182613-B.ogg]: clip derived from CFX-20130331-UK-DorsetSeaBeach02.wav (http://www.freesound.org/people/Carlvus/sounds/182613/) by Carlvus [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-183487-A-1",\n      "filename": "4-183487-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-183487-A-1.wav",\n      "bytes": 441044,\n      "sha256": "0cb40e860a07d9ef3843375df34033fcc7a655ffd01f1d693ae58f9ecfa0e1a3",\n      "decoded_sha256": "5642900b68bcb8380e63e8665a02719ada6cbc9a5bb895030ac4fbb67441e754",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "183487",\n      "source_id": "183487",\n      "attribution": "- [4-183487-A.ogg]: clip derived from Chickens.wav (http://www.freesound.org/people/ecfike/sounds/183487/) by ecfike [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-183992-A-0",\n      "filename": "4-183992-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-183992-A-0.wav",\n      "bytes": 441044,\n      "sha256": "96bf2e0efce166f347db6ae02b3c17d24acadffc45d67ad6015de7b1162f54dc",\n      "decoded_sha256": "4c16751d6ff23d825eabfd4ef110477e619bc7c4b6ca50dfd7e44534e2280b9b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "183992",\n      "source_id": "183992",\n      "attribution": "- [4-183992-A.ogg]: clip derived from Supermarket Dog.flac (http://www.freesound.org/people/qubodup/sounds/183992/) by qubodup [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-184434-A-21",\n      "filename": "4-184434-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-184434-A-21.wav",\n      "bytes": 441044,\n      "sha256": "3df4211c5afd73ca0e85d282f84313b73ccb8d3993af59a9f368e91da274d020",\n      "decoded_sha256": "6a3612d486dc93cd31020debb66c74607c76c8a07e8e4082c4ad1c0b257397b7",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "184434",\n      "source_id": "184434",\n      "attribution": "- [4-184434-A.ogg]: clip derived from Keyboard Sneeze (http://www.freesound.org/people/unfa/sounds/184434/) by unfa [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-184575-A-0",\n      "filename": "4-184575-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-184575-A-0.wav",\n      "bytes": 441044,\n      "sha256": "417a923837951edc374447528f1387b6bfb961a9605b165cfcb3d9d63735d179",\n      "decoded_sha256": "22a417b4197c982b5104619b5f9c55dfcb28b705d416de46bbc832ee8799f52d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "184575",\n      "source_id": "184575",\n      "attribution": "- [4-184575-A.ogg]: clip derived from Dog barking outside (http://www.freesound.org/people/magnus589/sounds/184575/) by magnus589 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-185415-A-21",\n      "filename": "4-185415-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-185415-A-21.wav",\n      "bytes": 441044,\n      "sha256": "3520064ea13f5685a83a24231a417615b153a374c954302abae0c063961b8649",\n      "decoded_sha256": "5290d3e5b5008d83831509a88f95d8376249c31d14cd8def4bb394eaa7915d73",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "185415",\n      "source_id": "185415",\n      "attribution": "- [4-185415-A.ogg]: clip derived from Sneeze Shot (http://www.freesound.org/people/unfa/sounds/185415/) by unfa [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-185575-A-20",\n      "filename": "4-185575-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-185575-A-20.wav",\n      "bytes": 441044,\n      "sha256": "9f40eaeef4e23e134a64b010d8a42636c7a9337e066539ebb183117744745f48",\n      "decoded_sha256": "aeb2833a0f5082b74c9ab9f038597d89dc37aa797f5a274aea81645e06c3e61e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "185575",\n      "source_id": "185575",\n      "attribution": "- [4-185575-A.ogg]: clip derived from Crying baby.wav (http://www.freesound.org/people/ciccarelli/sounds/185575/) by ciccarelli [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-185575-B-20",\n      "filename": "4-185575-B-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-185575-B-20.wav",\n      "bytes": 441044,\n      "sha256": "630641ffb71938bd943b0ac6c5c2523fea1d7aa1e34a93de6f7268183a25ee5a",\n      "decoded_sha256": "480a38bd129cca254b3dd32a4c3c58f85fb1e044312d607367c60b3bb5ba2374",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "185575",\n      "source_id": "185575",\n      "attribution": "- [4-185575-B.ogg]: clip derived from Crying baby.wav (http://www.freesound.org/people/ciccarelli/sounds/185575/) by ciccarelli [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-185575-C-20",\n      "filename": "4-185575-C-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-185575-C-20.wav",\n      "bytes": 441044,\n      "sha256": "7cdd46d701197c99ab637a10bc37e61a9859fd88218d339d4699a46f748c0200",\n      "decoded_sha256": "bfdf74778eabced41e86075cd32de30e8b0c2a4f75d6ddb74c54ce0a616611da",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "185575",\n      "source_id": "185575",\n      "attribution": "- [4-185575-C.ogg]: clip derived from Crying baby.wav (http://www.freesound.org/people/ciccarelli/sounds/185575/) by ciccarelli [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-185619-A-21",\n      "filename": "4-185619-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-185619-A-21.wav",\n      "bytes": 441044,\n      "sha256": "3c07f7c7150ca66a8f690bfc56ccbcdab63d94b870c2e0e9c6ec150f56316c91",\n      "decoded_sha256": "669c85f56bc9ad37f93089ca1c4df609df4b2aeec67a05e2deff383db8d1ba57",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "185619",\n      "source_id": "185619",\n      "attribution": "- [4-185619-A.ogg]: clip derived from sneeze (http://www.freesound.org/people/organicmanpl/sounds/185619/) by organicmanpl [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-188033-A-38",\n      "filename": "4-188033-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-188033-A-38.wav",\n      "bytes": 441044,\n      "sha256": "7a76b052b1344dfdebd234ed10b1e4a7d167e4bc7575dc2938fef7f9e1ad1a06",\n      "decoded_sha256": "0c76472892a2fe96227a917f730a75b7b025e653b739d21cab611a9ecd063ec3",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "188033",\n      "source_id": "188033",\n      "attribution": "- [4-188033-A.ogg]: clip derived from Ticking Clock (http://www.freesound.org/people/AntumDeluge/sounds/188033/) by AntumDeluge [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-191327-A-38",\n      "filename": "4-191327-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-191327-A-38.wav",\n      "bytes": 441044,\n      "sha256": "f68408749999319178d1dba0d2f107bf91ffd92262c18610429ce74ed16cc25e",\n      "decoded_sha256": "b65a6b766c7d94554a93d7adb97e86601a34582421dd9571b3ba3045b0c0c2f8",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "191327",\n      "source_id": "191327",\n      "attribution": "- [4-191327-A.ogg]: clip derived from TICKTOCK.wav (http://www.freesound.org/people/needforsuv/sounds/191327/) by needforsuv [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-191687-A-0",\n      "filename": "4-191687-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-191687-A-0.wav",\n      "bytes": 441044,\n      "sha256": "65bcaf4cce64177bb0dcc139c63dc9bee48d86c559b3019248d7c79db6a8b2ef",\n      "decoded_sha256": "fd51be25f50d63ed0a464b5068b37bcc6703ac5d4e555ddcc392003f9162b729",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "191687",\n      "source_id": "191687",\n      "attribution": "- [4-191687-A.ogg]: clip derived from Dog Barks.wav (http://www.freesound.org/people/UnderlinedDesigns/sounds/191687/) by UnderlinedDesigns [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-192236-A-0",\n      "filename": "4-192236-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-192236-A-0.wav",\n      "bytes": 441044,\n      "sha256": "cea20645e18a4ad56ad53905a6185e891b53f65989e4208f3014aba838096f08",\n      "decoded_sha256": "5bc0dc824d50b1a1842204c9095d46cf5d55ce84afe47e520b2beaaea67d055c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "192236",\n      "source_id": "192236",\n      "attribution": "- [4-192236-A.ogg]: clip derived from Miniature Dachshund Bark - Indoors.wav (http://www.freesound.org/people/Ligidium/sounds/192236/) by Ligidium [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-193480-A-40",\n      "filename": "4-193480-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-193480-A-40.wav",\n      "bytes": 441044,\n      "sha256": "1cf1759e082bf6f1ff9387cefa9cd7050f05154ed8c5e0dc490f43ee9a6dc1b5",\n      "decoded_sha256": "85077f310efca2f645cce2aacf2b0a16f838288c7dcd8628ffff2e81dadd8e6f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "193480",\n      "source_id": "193480",\n      "attribution": "- [4-193480-A.ogg]: clip derived from Helicopter circles three times overhead (http://www.freesound.org/people/trgame/sounds/193480/) by trgame [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-193480-B-40",\n      "filename": "4-193480-B-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-193480-B-40.wav",\n      "bytes": 441044,\n      "sha256": "70a164e3d5cbb8735afd80b6507de1bc4234d30d70efa036ae1f5424573f02df",\n      "decoded_sha256": "1671f7505afdee8cba665df2c76aaf3c73406b6258454e684611b0b56c373f55",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "193480",\n      "source_id": "193480",\n      "attribution": "- [4-193480-B.ogg]: clip derived from Helicopter circles three times overhead (http://www.freesound.org/people/trgame/sounds/193480/) by trgame [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-194711-A-38",\n      "filename": "4-194711-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-194711-A-38.wav",\n      "bytes": 441044,\n      "sha256": "909357b7e612f4d93399b8d3723410213546640147997e5c81a10c6995f966ed",\n      "decoded_sha256": "b2c4e96f13d59d8794708745f06a4561c36c6eadfe7f61dc1eaff65bad625d1f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "194711",\n      "source_id": "194711",\n      "attribution": "- [4-194711-A.ogg]: clip derived from Old 60s soviet clock ticking (http://www.freesound.org/people/doxent/sounds/194711/) by doxent [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-194754-A-0",\n      "filename": "4-194754-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-194754-A-0.wav",\n      "bytes": 441044,\n      "sha256": "89e569fc4f5659de5a5c70fc5b196270b218d8f51cb6d1235d9568540061b309",\n      "decoded_sha256": "b66ae16c7b01b27d25b544022854fbf9bc7163cd2244ebeb50e83cd3586fcb36",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "194754",\n      "source_id": "194754",\n      "attribution": "- [4-194754-A.ogg]: clip derived from Dog Barking in Brussels Street (http://www.freesound.org/people/qubodup/sounds/194754/) by qubodup [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-195497-A-11",\n      "filename": "4-195497-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-195497-A-11.wav",\n      "bytes": 441044,\n      "sha256": "754ff6b164f369e9bcda1b0d2ea28552a8097f8042a450f4987b79d916872dea",\n      "decoded_sha256": "efd2ab3d815808caf091a4148f3e0ee68dead846b20bd8ed9610c45de0444a59",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "195497",\n      "source_id": "195497",\n      "attribution": "- [4-195497-A.ogg]: clip derived from Sea tide Waves At Robin Hoods Bay 24/07/2013 (http://www.freesound.org/people/m.newlove/sounds/195497/) by m.newlove [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-195497-B-11",\n      "filename": "4-195497-B-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-195497-B-11.wav",\n      "bytes": 441044,\n      "sha256": "e227f80e99d397ec87eeb3d6d773c89238ddd10d6aae8d6cba0cd6988b054099",\n      "decoded_sha256": "72144eed80bcb5f80de6e8aed6426cbeab98e83f47acbe1884bc46331f45842e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "195497",\n      "source_id": "195497",\n      "attribution": "- [4-195497-B.ogg]: clip derived from Sea tide Waves At Robin Hoods Bay 24/07/2013 (http://www.freesound.org/people/m.newlove/sounds/195497/) by m.newlove [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-198965-A-38",\n      "filename": "4-198965-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-198965-A-38.wav",\n      "bytes": 441044,\n      "sha256": "0beb5ea6f39bbd91bcca7e433ba2b96f0dbcfb8f5f8b988ee98a3370a7be795d",\n      "decoded_sha256": "966b6c065e6f965221d698b16a7792b51af1c6e45e779f368e06b38439475a8b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "198965",\n      "source_id": "198965",\n      "attribution": "- [4-198965-A.ogg]: clip derived from mantelclock.wav (http://www.freesound.org/people/photog123/sounds/198965/) by photog123 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-199261-A-0",\n      "filename": "4-199261-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-199261-A-0.wav",\n      "bytes": 441044,\n      "sha256": "527a52b5cd50b5171b45ea0dd3c5efe5d11fb75a0c29cdd777c6f17255ac8262",\n      "decoded_sha256": "9230e0ae67b96e14b85ef448751708cc415bc4cd49fff4368e3a7944f09350c4",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "199261",\n      "source_id": "199261",\n      "attribution": "- [4-199261-A.ogg]: clip derived from A dog barking (close recording) (http://www.freesound.org/people/felix.blume/sounds/199261/) by felix.blume [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-204618-A-11",\n      "filename": "4-204618-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-204618-A-11.wav",\n      "bytes": 441044,\n      "sha256": "5df6ab900126bb95c8ea660cb15a668d3325f1770a081dfbe28e3bd37adf2dc0",\n      "decoded_sha256": "8454b9252db35396793a4d431daab961ac8528f9000c122ec9f7c3bfc4d142ce",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "204618",\n      "source_id": "204618",\n      "attribution": "- [4-204618-A.ogg]: clip derived from 8am.wav (http://www.freesound.org/people/jareilly/sounds/204618/) by jareilly [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-207124-A-0",\n      "filename": "4-207124-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-207124-A-0.wav",\n      "bytes": 441044,\n      "sha256": "4172efa07482e36b67f50bc730d9ecaf95da20c81f79242d0439ac6bae21f53c",\n      "decoded_sha256": "631e5559a3b99598a90f53cb662bb4df2db959b4edb789894f1c470dd7236372",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "207124",\n      "source_id": "207124",\n      "attribution": "- [4-207124-A.ogg]: clip derived from DogBARKING.wav (http://www.freesound.org/people/ahill86/sounds/207124/) by ahill86 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-208021-A-1",\n      "filename": "4-208021-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-208021-A-1.wav",\n      "bytes": 441044,\n      "sha256": "b3daaeacca340f005fe7a337c1ab2b22f28c8ea1f477dc5f6aeb06e5a3a1a582",\n      "decoded_sha256": "0b263b686488e02c308b2c4b214b68c165bfe0b04d5416ccdc7197715e30954f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "208021",\n      "source_id": "208021",\n      "attribution": "- [4-208021-A.ogg]: clip derived from Cock-a-doodle-doo (http://www.freesound.org/people/cocoricosound/sounds/208021/) by cocoricosound [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "4-59579-A-20",\n      "filename": "4-59579-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-59579-A-20.wav",\n      "bytes": 441044,\n      "sha256": "9c21cb5f694f7c375c0d9d33f2f618b58460b7d1b39681dd2d3fbb493f0b0052",\n      "decoded_sha256": "6eb30dab29a8cda22469e037bae70cf60f291b655102252c567cdaddd7b36adb",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "59579",\n      "source_id": "59579",\n      "attribution": "- [4-59579-A.ogg]: clip derived from babycrying2.mp3 (http://www.freesound.org/people/morgantj/sounds/59579/) by morgantj [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "4-59579-B-20",\n      "filename": "4-59579-B-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/4-59579-B-20.wav",\n      "bytes": 441044,\n      "sha256": "8129eb08025521fbb11f273c6c506ebed89becab8ccb31ec6c0954162b2242c1",\n      "decoded_sha256": "28bcbd7a7a9b5c21b9f48d30d494447d7296e80f2b29c2626b0aeeff4289ba3f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "validation",\n      "fold": 4,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "59579",\n      "source_id": "59579",\n      "attribution": "- [4-59579-B.ogg]: clip derived from babycrying2.mp3 (http://www.freesound.org/people/morgantj/sounds/59579/) by morgantj [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-151085-A-20",\n      "filename": "5-151085-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-151085-A-20.wav",\n      "bytes": 441044,\n      "sha256": "e46d87b3673136a4d97b42f0ac6411ed299075bb2c88d61f7c456c0afe6804db",\n      "decoded_sha256": "21896e7750df9988e578a501a80d4a2f1f85fc142d7532d924d450eca0346a60",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "151085",\n      "source_id": "151085",\n      "attribution": "- [5-151085-A.ogg]: clip derived from 01 baby crying hard.wav (http://www.freesound.org/people/PickleJones/sounds/151085/) by PickleJones [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-170338-A-41",\n      "filename": "5-170338-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-170338-A-41.wav",\n      "bytes": 441044,\n      "sha256": "db6e46d327b67ba496c758b58e18dcbef90ab832ed84596985ff84ad5bce8538",\n      "decoded_sha256": "1d111c3d52994bab90f14de5fb3703eb8ac252fdaf22a31afb3e4fffed806b6c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "170338",\n      "source_id": "170338",\n      "attribution": "- [5-170338-A.ogg]: clip derived from Chainsaws (distant) (http://www.freesound.org/people/micadoe/sounds/170338/) by micadoe [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-170338-B-41",\n      "filename": "5-170338-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-170338-B-41.wav",\n      "bytes": 441044,\n      "sha256": "cd356d4499a30a75b976d45500bff121d0c13d68fcb3864a76d44731ac87f9aa",\n      "decoded_sha256": "6b122b988570f6aa9063a057041bb7ac1e192eb9967accf0bf932d42a69b9567",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "170338",\n      "source_id": "170338",\n      "attribution": "- [5-170338-B.ogg]: clip derived from Chainsaws (distant) (http://www.freesound.org/people/micadoe/sounds/170338/) by micadoe [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-171653-A-41",\n      "filename": "5-171653-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-171653-A-41.wav",\n      "bytes": 441044,\n      "sha256": "2fcbbc97d0ac376a8219ec709ee6d90ccfca3df7bcbd2abbbf9a60491cddbb5d",\n      "decoded_sha256": "2dae9871e05a4039c4a505ecb141b8cad56d4c55038ed2e1d47aee656fbf41a7",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "171653",\n      "source_id": "171653",\n      "attribution": "- [5-171653-A.ogg]: clip derived from Chainsaw cutting through flesh (http://www.freesound.org/people/Audionautics/sounds/171653/) by Audionautics [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-177957-A-40",\n      "filename": "5-177957-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-177957-A-40.wav",\n      "bytes": 441044,\n      "sha256": "923786a6b83adcee780dc8afa78e3c4c1329ab6c679543bb08a0f56821f21f60",\n      "decoded_sha256": "6ce21cbf2de1bfca44cf9cdc3f501271c7c5b2841dc21129455961803e8fb56b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "177957",\n      "source_id": "177957",\n      "attribution": "- [5-177957-A.ogg]: clip derived from Helicopter - inside - Takeoff (http://www.freesound.org/people/senorstudy/sounds/177957/) by senorstudy [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-177957-B-40",\n      "filename": "5-177957-B-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-177957-B-40.wav",\n      "bytes": 441044,\n      "sha256": "0328636d7592af9a974c4939c54150aaad7b383c1d1e80793eb71af209b7c0ca",\n      "decoded_sha256": "144790dbd1d200ae3a0aed98acb9b5790944f2ea362f81c8c7551dc47ec86554",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "177957",\n      "source_id": "177957",\n      "attribution": "- [5-177957-B.ogg]: clip derived from Helicopter - inside - Takeoff (http://www.freesound.org/people/senorstudy/sounds/177957/) by senorstudy [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-177957-C-40",\n      "filename": "5-177957-C-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-177957-C-40.wav",\n      "bytes": 441044,\n      "sha256": "5b8598501cf0737d37118d1c80582d7338fcd4f3a61f339aaec85f892a3117cd",\n      "decoded_sha256": "0c5fb1b9741bd61b7483c55b883f2e10e1d3cfcc3fed6baa504a2172ba033b73",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "177957",\n      "source_id": "177957",\n      "attribution": "- [5-177957-C.ogg]: clip derived from Helicopter - inside - Takeoff (http://www.freesound.org/people/senorstudy/sounds/177957/) by senorstudy [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-177957-D-40",\n      "filename": "5-177957-D-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-177957-D-40.wav",\n      "bytes": 441044,\n      "sha256": "8d88a04034bfaf1d279d25097d59e6205c0c3ddc7757ee35b691768a9a2c0aaa",\n      "decoded_sha256": "afbcec9ef5fe82b1869f3cd0d0ef98d6f4d6232f7d47fda16cc9b4cd94f8a6c6",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "177957",\n      "source_id": "177957",\n      "attribution": "- [5-177957-D.ogg]: clip derived from Helicopter - inside - Takeoff (http://www.freesound.org/people/senorstudy/sounds/177957/) by senorstudy [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-177957-E-40",\n      "filename": "5-177957-E-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-177957-E-40.wav",\n      "bytes": 441044,\n      "sha256": "d42b641fe36d07e0b74d1b684ad46c1dfd597f5a537c005bb4b37eda0b3d9e34",\n      "decoded_sha256": "14d4a63b19a86254fcc0daf07b7b6c63cb171263a5e2823b6ef6236e45465bdd",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "177957",\n      "source_id": "177957",\n      "attribution": "- [5-177957-E.ogg]: clip derived from Helicopter - inside - Takeoff (http://www.freesound.org/people/senorstudy/sounds/177957/) by senorstudy [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-181766-A-10",\n      "filename": "5-181766-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-181766-A-10.wav",\n      "bytes": 441044,\n      "sha256": "608fcc0bef32512e2a984c7d3d2cac8ea161f6337ec0056850a6d34f07f8db35",\n      "decoded_sha256": "e040f03efe75e9d6af99d7a40eb74d26322c5438dc63feac0a56b355b5eded9b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "181766",\n      "source_id": "181766",\n      "attribution": "- [5-181766-A.ogg]: clip derived from rain.aiff (http://www.freesound.org/people/laura222/sounds/181766/) by laura222 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-185579-A-41",\n      "filename": "5-185579-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-185579-A-41.wav",\n      "bytes": 441044,\n      "sha256": "aa779381c3411b478b46c61cc46fd0ac9bac4ecfd4f37c0ac412cec1421a8578",\n      "decoded_sha256": "01ec008935a997ba03ccd9e0b7dd589815d6152bc5a951f412555e40bda39e68",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "185579",\n      "source_id": "185579",\n      "attribution": "- [5-185579-A.ogg]: clip derived from Chainsaw Cutting.wav (http://www.freesound.org/people/lonemonk/sounds/185579/) by lonemonk [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-185579-B-41",\n      "filename": "5-185579-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-185579-B-41.wav",\n      "bytes": 441044,\n      "sha256": "853d5ae507977df944cba7775cbdd2a3e262b3f7e99729d5ec465c51dc692a6d",\n      "decoded_sha256": "d7b78b10d8af524e44f8c68f14c0029e9c1153a3f3f7352f45e28f2a2b15ec91",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "185579",\n      "source_id": "185579",\n      "attribution": "- [5-185579-B.ogg]: clip derived from Chainsaw Cutting.wav (http://www.freesound.org/people/lonemonk/sounds/185579/) by lonemonk [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-186924-A-12",\n      "filename": "5-186924-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-186924-A-12.wav",\n      "bytes": 441044,\n      "sha256": "963758fc0a9bb51fbc0659ac0ab75f2aede097519f46b92efecac22baae215cc",\n      "decoded_sha256": "736e46054716458dd19e94ba45abc003cdb6c44b4a9bd60c21bb811d3e323ebf",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "186924",\n      "source_id": "186924",\n      "attribution": "- [5-186924-A.ogg]: clip derived from Fire_Loop.wav (http://www.freesound.org/people/ReadeOnly/sounds/186924/) by ReadeOnly [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-187979-A-21",\n      "filename": "5-187979-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-187979-A-21.wav",\n      "bytes": 441044,\n      "sha256": "f9b1c8dcaebec0e9d7bba56142e831a6530ee06bd969e0b80c91bf7cd57e4b4f",\n      "decoded_sha256": "fb9cb63ed725d3749ce436518b4c9d444e28f77329ded20487f8b37df19309c5",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "187979",\n      "source_id": "187979",\n      "attribution": "- [5-187979-A.ogg]: clip derived from sneeze.MP3 (http://www.freesound.org/people/fliegermatze/sounds/187979/) by fliegermatze [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-188655-A-10",\n      "filename": "5-188655-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-188655-A-10.wav",\n      "bytes": 441044,\n      "sha256": "9ed876436c72f36c230c275744818829912a2299bcbd8ccca6eccce9df6f8a57",\n      "decoded_sha256": "b62634f6415bd4177f761a210446c9193f7c73e2f53b5bc03501a491ffa946fd",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "188655",\n      "source_id": "188655",\n      "attribution": "- [5-188655-A.ogg]: clip derived from 1 Minute Rain (Medium) (http://www.freesound.org/people/yzhamua/sounds/188655/) by yzhamua [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-189212-A-12",\n      "filename": "5-189212-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-189212-A-12.wav",\n      "bytes": 441044,\n      "sha256": "bd9983a6e739ea7df03c9363b17834a71ba0b74cbb59d8766d48ccb0fedd2138",\n      "decoded_sha256": "cc8a7654594e969c619ef8ff39fa1cd3a5821f7ac469d10d5426ca2b27de9554",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "189212",\n      "source_id": "189212",\n      "attribution": "- [5-189212-A.ogg]: clip derived from Burning Fire (http://www.freesound.org/people/Vurca/sounds/189212/) by Vurca [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-189237-A-12",\n      "filename": "5-189237-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-189237-A-12.wav",\n      "bytes": 441044,\n      "sha256": "41ab28c22910b7fd2834728fd4819d8a0d4a9268e5d1b8b434a5705473fb373a",\n      "decoded_sha256": "0a5c7698f9f634dec219c36652f6ec5cd45d94fbaf4edb18b06e008a4e94d6f6",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "189237",\n      "source_id": "189237",\n      "attribution": "- [5-189237-A.ogg]: clip derived from Bonfire (http://www.freesound.org/people/amether/sounds/189237/) by amether [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-191131-A-40",\n      "filename": "5-191131-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-191131-A-40.wav",\n      "bytes": 441044,\n      "sha256": "498f75725d397a2253c6fdc687f48773f2ed19a0e44c22463767f4ed186c3cc1",\n      "decoded_sha256": "5a8f36c3072ba5ffee8780bda5fa458d652154891b9f3c19de13cc3bc26f14c1",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "191131",\n      "source_id": "191131",\n      "attribution": "- [5-191131-A.ogg]: clip derived from helicopter and thunder (http://www.freesound.org/people/yacou/sounds/191131/) by yacou [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-193339-A-10",\n      "filename": "5-193339-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-193339-A-10.wav",\n      "bytes": 441044,\n      "sha256": "aec7a17013e61faedb111983e0699a8b63c0e1af9569bb4a819ffba86a28f778",\n      "decoded_sha256": "9b9d4d1445f58681435a868814658c889f954d01183246ef55a41844667a8b29",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "193339",\n      "source_id": "193339",\n      "attribution": "- [5-193339-A.ogg]: clip derived from rain.mp3 (http://www.freesound.org/people/soundman9826/sounds/193339/) by soundman9826 [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-193473-A-12",\n      "filename": "5-193473-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-193473-A-12.wav",\n      "bytes": 441044,\n      "sha256": "0c34b4945be60984588bf5b3048ff62974eaaf69b0fa8ef4f06408debb0d7fc9",\n      "decoded_sha256": "5a1efcc6b18690ef19319612cac7d3b29a8080b256f9a6cda6a4f81d10b68af5",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "193473",\n      "source_id": "193473",\n      "attribution": "- [5-193473-A.ogg]: clip derived from RRMX SMALL FIRE.wav (http://www.freesound.org/people/skyklan47/sounds/193473/) by skyklan47 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-193473-B-12",\n      "filename": "5-193473-B-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-193473-B-12.wav",\n      "bytes": 441044,\n      "sha256": "0ab553a2ffee37391bd4ae8aa56961c64d5e54e69a2733857b53fbd251dcb216",\n      "decoded_sha256": "5897414c3d40d688ce0ae8b9f2616df1d897cc00fed300fead9dbbd5b37f4ac2",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "193473",\n      "source_id": "193473",\n      "attribution": "- [5-193473-B.ogg]: clip derived from RRMX SMALL FIRE.wav (http://www.freesound.org/people/skyklan47/sounds/193473/) by skyklan47 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-194533-A-21",\n      "filename": "5-194533-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-194533-A-21.wav",\n      "bytes": 441044,\n      "sha256": "ecb5cab5084d452afe2cf0d65405ee8f38dcdddae32622b44cdd553d8e7975ec",\n      "decoded_sha256": "48b2d81f3fc5ae4242587c25d225cd377c0814f3b242c6b3b501f69e29ea5935",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "194533",\n      "source_id": "194533",\n      "attribution": "- [5-194533-A.ogg]: clip derived from stan_baby_sneezing.flac (http://www.freesound.org/people/wjoojoo/sounds/194533/) by wjoojoo [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-194892-A-10",\n      "filename": "5-194892-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-194892-A-10.wav",\n      "bytes": 441044,\n      "sha256": "12df0d996db614fff7148063232490f76413448b6b193034b5f9655ec5172a37",\n      "decoded_sha256": "a5fe3cbf424e18d266e713ced5925272c7a30c4104ae646dfe540bd0ac2c2848",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "194892",\n      "source_id": "194892",\n      "attribution": "- [5-194892-A.ogg]: clip derived from lo-fi-rain-and-thunder.wav (http://www.freesound.org/people/arseniiv/sounds/194892/) by arseniiv [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-194930-A-1",\n      "filename": "5-194930-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-194930-A-1.wav",\n      "bytes": 441044,\n      "sha256": "785615790c00a2ac8d5f4d579873a8b1bd1f5a77fc990eafbd05725974d63659",\n      "decoded_sha256": "3d19b16757bfb3dcad67af05102b9cd14de312944c106408e3f6f58bf99000cb",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "194930",\n      "source_id": "194930",\n      "attribution": "- [5-194930-A.ogg]: clip derived from cock crows near and far .mp3 (http://www.freesound.org/people/soundmary/sounds/194930/) by soundmary [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-194930-B-1",\n      "filename": "5-194930-B-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-194930-B-1.wav",\n      "bytes": 441044,\n      "sha256": "b263df738b703f0001765438cd95cdd83c8926b281ce36edce49ee91df5eb82f",\n      "decoded_sha256": "1d5a17b58813c2939392bcbf3478b889f357e178ef85906199d513febc319512",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "194930",\n      "source_id": "194930",\n      "attribution": "- [5-194930-B.ogg]: clip derived from cock crows near and far .mp3 (http://www.freesound.org/people/soundmary/sounds/194930/) by soundmary [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-195710-A-10",\n      "filename": "5-195710-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-195710-A-10.wav",\n      "bytes": 441044,\n      "sha256": "939959e3cb6df01e3a67d47cf1b1d0429785a1906562dd143805e8fb8db909c0",\n      "decoded_sha256": "c9d50131bae40f16f0c63dc1955339b5b80193c026461ccc57525e8fd6414ded",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "195710",\n      "source_id": "195710",\n      "attribution": "- [5-195710-A.ogg]: clip derived from Ambient_Summer Rain_Storm_Urban_Dogs_People_Water Drops.wav (http://www.freesound.org/people/janstumpa/sounds/195710/) by janstumpa [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-198321-A-10",\n      "filename": "5-198321-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-198321-A-10.wav",\n      "bytes": 441044,\n      "sha256": "a16dbc1f4320ade2754c13c9540e4cfdbdfdc44008c3ba699f00476d9a70fe63",\n      "decoded_sha256": "03f1c2bfdf0a03ddcf0a9021af75e5fb6743ba2a58867b1f94b87edd65447b4e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "198321",\n      "source_id": "198321",\n      "attribution": "- [5-198321-A.ogg]: clip derived from rain.wav (http://www.freesound.org/people/schafferdavid/sounds/198321/) by schafferdavid [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-198411-A-20",\n      "filename": "5-198411-A-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-198411-A-20.wav",\n      "bytes": 441044,\n      "sha256": "efa9989fa053f1bcb7916965791c7bf55c05e4e5acf0bd6d61c649d833659494",\n      "decoded_sha256": "4fb7955bf44418e0d98d33cb33b6a1ba5dc0aa20f3dcf4a8f7c397aaf941d3b7",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "198411",\n      "source_id": "198411",\n      "attribution": "- [5-198411-A.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-198411-B-20",\n      "filename": "5-198411-B-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-198411-B-20.wav",\n      "bytes": 441044,\n      "sha256": "0a322dd25c0a694b06732a57f9e153f4615679c02fd5ec274aedf0bee3102b7d",\n      "decoded_sha256": "1348971221b101a6745afd882b46ba0c4176fbb7de74c13399bb032058912573",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "198411",\n      "source_id": "198411",\n      "attribution": "- [5-198411-B.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-198411-C-20",\n      "filename": "5-198411-C-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-198411-C-20.wav",\n      "bytes": 441044,\n      "sha256": "23cf024e102435298b16a8138129f411326a239a2486fc49ca61b4cc15f24875",\n      "decoded_sha256": "73c852c78934a09c4a46ebbb8ca05e2cab312e9b023071813b3ad01bfec2fd63",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "198411",\n      "source_id": "198411",\n      "attribution": "- [5-198411-C.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-198411-D-20",\n      "filename": "5-198411-D-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-198411-D-20.wav",\n      "bytes": 441044,\n      "sha256": "ea7b47c6eb1383ec5b8ba4147aa63f7adaff0b218d02731e24bcb3903549abd9",\n      "decoded_sha256": "c2e57af29f04d93d7ca29e0570a321b191839d95ed2f833d075c2b7a0d273996",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "198411",\n      "source_id": "198411",\n      "attribution": "- [5-198411-D.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-198411-E-20",\n      "filename": "5-198411-E-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-198411-E-20.wav",\n      "bytes": 441044,\n      "sha256": "d473b14efed04cdc8cfa37dff146f2d861e744414b86a8b113d42a506df598ad",\n      "decoded_sha256": "e8f6deab69c141a200e6e497217466daf7dd8f97e289ca46e186ed2ea6c9c74f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "198411",\n      "source_id": "198411",\n      "attribution": "- [5-198411-E.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-198411-F-20",\n      "filename": "5-198411-F-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-198411-F-20.wav",\n      "bytes": 441044,\n      "sha256": "c7a8af6048461639c8931e6c0622a79ccfde881ec5b0ce0c902cb446eb699ec1",\n      "decoded_sha256": "2de6b8e8352910e68a1f90c8369e07742af29795c97813d2336b807b1faccb08",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "198411",\n      "source_id": "198411",\n      "attribution": "- [5-198411-F.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-198411-G-20",\n      "filename": "5-198411-G-20.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-198411-G-20.wav",\n      "bytes": 441044,\n      "sha256": "a5172b7c92be5d7663f1e97b629dda2e9020ee4caf4b626db1b05817b1ee7f1c",\n      "decoded_sha256": "543733a74a728ad1aa3f90b11a8fef948f6a63626ff63eb53ae687325a822da5",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 3,\n      "category": "crying_baby",\n      "target": 20,\n      "group_id": "198411",\n      "source_id": "198411",\n      "attribution": "- [5-198411-G.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-200334-A-1",\n      "filename": "5-200334-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-200334-A-1.wav",\n      "bytes": 441044,\n      "sha256": "6c5e64aad993a023cf4748b96759f97793b4c9d55db0fcf3d42754912cdabdbd",\n      "decoded_sha256": "81e5c7407711d479e5fed322ea9a6d276f859f033cd1b45c7195413b78a8a02c",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "200334",\n      "source_id": "200334",\n      "attribution": "- [5-200334-A.ogg]: clip derived from rooster_crows_inside.aif (http://www.freesound.org/people/jsbarrett/sounds/200334/) by jsbarrett [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-200334-B-1",\n      "filename": "5-200334-B-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-200334-B-1.wav",\n      "bytes": 441044,\n      "sha256": "e98e4171fea916bcf7de6c6803ff7ee518bc0728f49e59d6c04ae0070ddfbbb4",\n      "decoded_sha256": "e55c093ecc2a786fbdf3280ac24e875fbf411961e8590a017e5cd3afbb9d6698",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "200334",\n      "source_id": "200334",\n      "attribution": "- [5-200334-B.ogg]: clip derived from rooster_crows_inside.aif (http://www.freesound.org/people/jsbarrett/sounds/200334/) by jsbarrett [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-200339-A-1",\n      "filename": "5-200339-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-200339-A-1.wav",\n      "bytes": 441044,\n      "sha256": "a29c86e2ebae1084f18a17ff73f6de34a3470a974de6c44f714801fc3eb49fde",\n      "decoded_sha256": "46896bd3314fb6a63b6f6ee49c5b7b2cd9127248c8fb5988dcb7d0be137384c8",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "200339",\n      "source_id": "200339",\n      "attribution": "- [5-200339-A.ogg]: clip derived from rooster_crows_outside.aif (http://www.freesound.org/people/jsbarrett/sounds/200339/) by jsbarrett [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-200461-A-11",\n      "filename": "5-200461-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-200461-A-11.wav",\n      "bytes": 441044,\n      "sha256": "f66d4da342b3ae9096593084915f57d2763bd5e90ff28fbb7c60a446d6c76a6f",\n      "decoded_sha256": "03b4b6aa332ecf16dba7fc5e37fa1cd8e333d7fdfd75d5dbe5bf447ca346300e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "200461",\n      "source_id": "200461",\n      "attribution": "- [5-200461-A.ogg]: clip derived from Sea gushes into seacave close.aif (http://www.freesound.org/people/ikbenraar/sounds/200461/) by ikbenraar [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-200461-B-11",\n      "filename": "5-200461-B-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-200461-B-11.wav",\n      "bytes": 441044,\n      "sha256": "772ed35d4a5617deb375b6532e830071b21e9b3d5d16b6b33b16c9535ef8e1a7",\n      "decoded_sha256": "c9f62d7156ee9748a8b8e8ce1056aed2e0514ec7719089a190c1efd4e6358d2d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "200461",\n      "source_id": "200461",\n      "attribution": "- [5-200461-B.ogg]: clip derived from Sea gushes into seacave close.aif (http://www.freesound.org/people/ikbenraar/sounds/200461/) by ikbenraar [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-201194-A-38",\n      "filename": "5-201194-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-201194-A-38.wav",\n      "bytes": 441044,\n      "sha256": "61599b9d388e887b0318d4b008ca3f545b28f327ba3ecb12afdff7a128162c28",\n      "decoded_sha256": "07f168f506858228ebf25d97a87ac23c2cfe80287f27068cab18eac1cc811697",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "201194",\n      "source_id": "201194",\n      "attribution": "- [5-201194-A.ogg]: clip derived from Mechanical clock ticking (http://www.freesound.org/people/opticalnoise/sounds/201194/) by opticalnoise [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-201274-A-21",\n      "filename": "5-201274-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-201274-A-21.wav",\n      "bytes": 441044,\n      "sha256": "272a632b7c6f25802d6646ffba3b24f72e69345c1e9bdfad7634f6b90967f149",\n      "decoded_sha256": "c3264af86751d0cf1a896811b8c3605e47dd69fed763a9b70d27be08e6652f78",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "201274",\n      "source_id": "201274",\n      "attribution": "- [5-201274-A.ogg]: clip derived from Person Sneezing Sound Bit.wav (http://www.freesound.org/people/chrisbrowne626/sounds/201274/) by chrisbrowne626 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-202220-A-21",\n      "filename": "5-202220-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-202220-A-21.wav",\n      "bytes": 441044,\n      "sha256": "2c46cb7a03294173f92a577297bccfb8c1843ccbfbe7b1aacee02f8d29706859",\n      "decoded_sha256": "b2ff71ca399a12c916510d09f7ef31ed67b5d83f72b62ab593b5c8f5011e81bb",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "202220",\n      "source_id": "202220",\n      "attribution": "- [5-202220-A.ogg]: clip derived from Sneeze (http://www.freesound.org/people/Huminaatio/sounds/202220/) by Huminaatio [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-202898-A-10",\n      "filename": "5-202898-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-202898-A-10.wav",\n      "bytes": 441044,\n      "sha256": "06ba576df5efc793523c7adebfca21094e1d3cbdd924c0655a16ac5aa5785b78",\n      "decoded_sha256": "82fbdd730843eb42a612e8375bff9232129b80fb85f0a1b2c571ce8d0f84d5c4",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "202898",\n      "source_id": "202898",\n      "attribution": "- [5-202898-A.ogg]: clip derived from EXT_RainInPuddle.wav (http://www.freesound.org/people/jongrubbs/sounds/202898/) by jongrubbs [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-203128-A-0",\n      "filename": "5-203128-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-203128-A-0.wav",\n      "bytes": 441044,\n      "sha256": "435e476d5d429aa1cc51c23c9b5e5f8f472faf022846996ade18a457621d0552",\n      "decoded_sha256": "3ec46d97e9c35257fda42729701d4fd69162ab30fa2095cf794afc30f28fc6fb",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "203128",\n      "source_id": "203128",\n      "attribution": "- [5-203128-A.ogg]: clip derived from Toupie1.mp3 (http://www.freesound.org/people/Pierre%20Grandjean/sounds/203128/) by Pierre Grandjean [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-203128-B-0",\n      "filename": "5-203128-B-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-203128-B-0.wav",\n      "bytes": 441044,\n      "sha256": "2a253659c2c3c86a0fbaf1e928d8a30933e90c7e7d4be43a3e4b4217e5be0c26",\n      "decoded_sha256": "97b5b22318f96a663a28fe4220816206131d30777bebbea8dd16b617f4e816ed",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "203128",\n      "source_id": "203128",\n      "attribution": "- [5-203128-B.ogg]: clip derived from Toupie1.mp3 (http://www.freesound.org/people/Pierre%20Grandjean/sounds/203128/) by Pierre Grandjean [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-203739-A-10",\n      "filename": "5-203739-A-10.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-203739-A-10.wav",\n      "bytes": 441044,\n      "sha256": "226ca7d5b50f0b45fed093765ac1eb3d243812c7940cec7c7d696aa65e67f447",\n      "decoded_sha256": "aaf38a4ed61b4d107ede6c478e819acd2c067b3b9e22d39e50f6abe8c03bc1e7",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 6,\n      "category": "rain",\n      "target": 10,\n      "group_id": "203739",\n      "source_id": "203739",\n      "attribution": "- [5-203739-A.ogg]: clip derived from rain with occasional thunder.wav (http://www.freesound.org/people/MrBoing/sounds/203739/) by MrBoing [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-205898-A-40",\n      "filename": "5-205898-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-205898-A-40.wav",\n      "bytes": 441044,\n      "sha256": "8d688e82f3ff6f075be0479c57c8a5cdffa09c2ed46dd3950f3682208b94dd21",\n      "decoded_sha256": "d0e2be3329eac95e18f58f21f8242405946978c19401b4f782b5b32953137ccf",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "205898",\n      "source_id": "205898",\n      "attribution": "- [5-205898-A.ogg]: clip derived from Helicopter Flight recorded from inside.mp3 (http://www.freesound.org/people/Spankous/sounds/205898/) by Spankous [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-208030-A-0",\n      "filename": "5-208030-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-208030-A-0.wav",\n      "bytes": 441044,\n      "sha256": "8b3f5052dad5383ea171aa8feaead47879bf7b5ae3acaf5a26c5167838a49494",\n      "decoded_sha256": "bae101708bc110c750c21307b505d1d45e4bd507bd6421fa0f6dc6515e84ef91",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "208030",\n      "source_id": "208030",\n      "attribution": "- [5-208030-A.ogg]: clip derived from Chihuahua Barks (http://www.freesound.org/people/Mewsel/sounds/208030/) by Mewsel [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-208624-A-38",\n      "filename": "5-208624-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-208624-A-38.wav",\n      "bytes": 441044,\n      "sha256": "536288f95505f82775b97445bfb95839520ebd4e11fb28d4be175f99adcd5060",\n      "decoded_sha256": "f8fda2a994ad86f357cc4b26cf05a977285018c6dfb95197eb7225aed5584f69",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "208624",\n      "source_id": "208624",\n      "attribution": "- [5-208624-A.ogg]: clip derived from Old Clock Ticking (http://www.freesound.org/people/LanDub/sounds/208624/) by LanDub [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-208810-A-11",\n      "filename": "5-208810-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-208810-A-11.wav",\n      "bytes": 441044,\n      "sha256": "ab3b06f58d545c4d7e4f543f5a1cc1b193f57be9bc76b3a5b1537ee0098a6bcb",\n      "decoded_sha256": "29d74dfd498fb1ad0af12de8e5376f9e27bd8be851afae8bd7d74ca84da5e01f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "208810",\n      "source_id": "208810",\n      "attribution": "- [5-208810-A.ogg]: clip derived from sea waves atmo.wav (http://www.freesound.org/people/monotraum/sounds/208810/) by monotraum [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-208810-B-11",\n      "filename": "5-208810-B-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-208810-B-11.wav",\n      "bytes": 441044,\n      "sha256": "1c0139a66c9d36e7f6ff0bf8b8d13a03600bd224092c72050b95c89d0ac28c09",\n      "decoded_sha256": "f2fc9c682a81b174b00780e21beda0fe5f8593d5af2b135bd8e072fbfea501f6",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "208810",\n      "source_id": "208810",\n      "attribution": "- [5-208810-B.ogg]: clip derived from sea waves atmo.wav (http://www.freesound.org/people/monotraum/sounds/208810/) by monotraum [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-209698-A-38",\n      "filename": "5-209698-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-209698-A-38.wav",\n      "bytes": 441044,\n      "sha256": "a1b535c73dfcdba86fdf04c44a10fde902698ca6c84d2ce0d823651c6356fb24",\n      "decoded_sha256": "277ac7c46ac33df91c5fa0bcfda597eacbce0b54a47dd6a5d0d8960f33be9ae5",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "209698",\n      "source_id": "209698",\n      "attribution": "- [5-209698-A.ogg]: clip derived from Analog Timer (http://www.freesound.org/people/EagleStealthTeam/sounds/209698/) by EagleStealthTeam [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-209833-A-38",\n      "filename": "5-209833-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-209833-A-38.wav",\n      "bytes": 441044,\n      "sha256": "30ad821df0ab0cdc69a6aadd99861f02a087901dfac2727c86106d096cc994c2",\n      "decoded_sha256": "1bf2d3551505085510c4b3863d428973a86df8dcd3e8483cb4578ecc79179900",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "209833",\n      "source_id": "209833",\n      "attribution": "- [5-209833-A.ogg]: clip derived from Old Clock Tick Tock (http://www.freesound.org/people/spaceemotion/sounds/209833/) by spaceemotion [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-210571-A-38",\n      "filename": "5-210571-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-210571-A-38.wav",\n      "bytes": 441044,\n      "sha256": "740bb48c0f990654d90ad56c54b74a9fdbf1617c0092dd9dbf72f890d7860fac",\n      "decoded_sha256": "8bb2d374000e2725891d856383bd2950bc963383541a98dbd1be8f2f3b0c8471",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "210571",\n      "source_id": "210571",\n      "attribution": "- [5-210571-A.ogg]: clip derived from wall_clock.wav (http://www.freesound.org/people/Sclolex/sounds/210571/) by Sclolex [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-212181-A-38",\n      "filename": "5-212181-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-212181-A-38.wav",\n      "bytes": 441044,\n      "sha256": "b920b5a43853cfe138924b3e9278b09f1c45af9e7ee065598605c7d589d57ce7",\n      "decoded_sha256": "b61a1c33fd245da350460e40bf67d77b4c4282b5825e3a07a01274d4aaa5be9a",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "212181",\n      "source_id": "212181",\n      "attribution": "- [5-212181-A.ogg]: clip derived from Loopable Ticking Clock (http://www.freesound.org/people/OwlStorm/sounds/212181/) by OwlStorm [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-212454-A-0",\n      "filename": "5-212454-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-212454-A-0.wav",\n      "bytes": 441044,\n      "sha256": "a3f7089d31d506d7a5c80f0af79afebfdb8fd61759eab03891b01c809d210e5a",\n      "decoded_sha256": "f9fe2413cdfe5dc538476f1b7bc7115b1f4d1f5354311bfe8471357c00b7991f",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "212454",\n      "source_id": "212454",\n      "attribution": "- [5-212454-A.ogg]: clip derived from Barking (http://www.freesound.org/people/unfa/sounds/212454/) by unfa [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-213077-A-11",\n      "filename": "5-213077-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-213077-A-11.wav",\n      "bytes": 441044,\n      "sha256": "394b71055793aa111d953b9c33ecd226e215a4221eefb248f3e5e43bc711626c",\n      "decoded_sha256": "2b38586eabffe2da79201f0e2ab235e75a8619b9c07f39d2c7f0805286725919",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "213077",\n      "source_id": "213077",\n      "attribution": "- [5-213077-A.ogg]: clip derived from 3 jan 2014, waves breaking over sea wall at crosby, during high spring tide in winter storm, with Applied EQ.wav (http://www.freesound.org/people/crosbychris/sounds/213077/) by crosbychris [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-213802-A-12",\n      "filename": "5-213802-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-213802-A-12.wav",\n      "bytes": 441044,\n      "sha256": "eb7d0d7bd9ccbfc6134370a8fbe776ea4cde5fcfa2f7b8fbf0959e686f70716b",\n      "decoded_sha256": "52093102f79b4fc97ec99de4239bc86e81469a278a97a692076def3dab4e6090",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "213802",\n      "source_id": "213802",\n      "attribution": "- [5-213802-A.ogg]: clip derived from burning-fire-steam.wav (http://www.freesound.org/people/laribum/sounds/213802/) by laribum [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-213855-A-0",\n      "filename": "5-213855-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-213855-A-0.wav",\n      "bytes": 441044,\n      "sha256": "1013f545442642bd8a7788279d75d131754127a4bdee39dc9c5299c04c4962c8",\n      "decoded_sha256": "f87e7cfdb328ebf8d41646fdfbcbeaebbec57c7776c080423bd9a0996dc0724e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "213855",\n      "source_id": "213855",\n      "attribution": "- [5-213855-A.ogg]: clip derived from Tiny Dog Barking in Park (http://www.freesound.org/people/qubodup/sounds/213855/) by qubodup [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-215658-A-12",\n      "filename": "5-215658-A-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-215658-A-12.wav",\n      "bytes": 441044,\n      "sha256": "d55ff3e5f491a0d5a137632596d89c4db5c4e9b516de9636ec363e83e9b1e518",\n      "decoded_sha256": "a0c22e8658ebd30d03c6675efc0fc79f367d96b3bec1667086dd7606fa89fdb7",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "215658",\n      "source_id": "215658",\n      "attribution": "- [5-215658-A.ogg]: clip derived from Fireplace.wav (http://www.freesound.org/people/juskiddink/sounds/215658/) by juskiddink [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-215658-B-12",\n      "filename": "5-215658-B-12.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-215658-B-12.wav",\n      "bytes": 441044,\n      "sha256": "ca7f0180a8d1747fd135700f672cc91f7fa976ed952f76a677b080453db8b3f4",\n      "decoded_sha256": "822aab8c85e33ecac17ce30dc7b60565219ecbebb2bc7e52b52888864c1191df",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 2,\n      "category": "crackling_fire",\n      "target": 12,\n      "group_id": "215658",\n      "source_id": "215658",\n      "attribution": "- [5-215658-B.ogg]: clip derived from Fireplace.wav (http://www.freesound.org/people/juskiddink/sounds/215658/) by juskiddink [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-216370-A-41",\n      "filename": "5-216370-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-216370-A-41.wav",\n      "bytes": 441044,\n      "sha256": "429453805c51b7fe554997e8ec90680545ba375eea3d18869ed141be72872dd8",\n      "decoded_sha256": "8eacdc170773e40ffcf62a51b45237cc3839d4cdfa3df15ed7a22fcb12207223",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "216370",\n      "source_id": "216370",\n      "attribution": "- [5-216370-A.ogg]: clip derived from Chainsaw.wav (http://www.freesound.org/people/RSilveira_88/sounds/216370/) by RSilveira_88 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-216370-B-41",\n      "filename": "5-216370-B-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-216370-B-41.wav",\n      "bytes": 441044,\n      "sha256": "0c641aafddc7f25a6f10e5f2a34578bef750c3ffc09f9caa195060162703fc6d",\n      "decoded_sha256": "e37e0c0b4abb418e18f0aacaf01e116dfed0a164a83ad4c67dc1cd446c555323",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "216370",\n      "source_id": "216370",\n      "attribution": "- [5-216370-B.ogg]: clip derived from Chainsaw.wav (http://www.freesound.org/people/RSilveira_88/sounds/216370/) by RSilveira_88 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-217158-A-0",\n      "filename": "5-217158-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-217158-A-0.wav",\n      "bytes": 441044,\n      "sha256": "170df4aad8dd13df6e418c8ff4e5a8cf8bec6fece5e3d0150b94d391e91b1356",\n      "decoded_sha256": "334013ec045dd8475d12818ebb989fe79c8bea7cb393cdc4e9b58fa8ff810025",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "217158",\n      "source_id": "217158",\n      "attribution": "- [5-217158-A.ogg]: clip derived from barkingdog_in_dream.mp3 (http://www.freesound.org/people/Taira%20Komori/sounds/217158/) by Taira Komori [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-219342-A-38",\n      "filename": "5-219342-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-219342-A-38.wav",\n      "bytes": 441044,\n      "sha256": "08c0a65319a5bf24b1fe37bb3ed9dcbf3f384f6185f292d8fc8a532c37b6336b",\n      "decoded_sha256": "1b5f24ab0e77d33feb7a90816e3a49ac42797180bc7573034addda73ac7812a2",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "219342",\n      "source_id": "219342",\n      "attribution": "- [5-219342-A.ogg]: clip derived from clock.wav (http://www.freesound.org/people/fatbass/sounds/219342/) by fatbass [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-219379-A-11",\n      "filename": "5-219379-A-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-219379-A-11.wav",\n      "bytes": 441044,\n      "sha256": "152ec7b65431ae066b3ed6e19d695febbd641f9186fd7e549526160108b123e7",\n      "decoded_sha256": "67200e9e7b44c561c784556c6e17b705afab9dced41720ccb4c64c7c1648d0ef",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "219379",\n      "source_id": "219379",\n      "attribution": "- [5-219379-A.ogg]: clip derived from the night Baltic001_14022014.wav (http://www.freesound.org/people/miastodzwiekow/sounds/219379/) by miastodzwiekow [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-219379-B-11",\n      "filename": "5-219379-B-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-219379-B-11.wav",\n      "bytes": 441044,\n      "sha256": "b84df2ce299ed891276f0c5866dd0b7762cac19ae895ba775e5a89b6a594d63c",\n      "decoded_sha256": "a84e9e536c08ee83a74a9914442e50139aa20386a67d6f150fdae1726f87782e",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "219379",\n      "source_id": "219379",\n      "attribution": "- [5-219379-B.ogg]: clip derived from the night Baltic001_14022014.wav (http://www.freesound.org/people/miastodzwiekow/sounds/219379/) by miastodzwiekow [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-219379-C-11",\n      "filename": "5-219379-C-11.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-219379-C-11.wav",\n      "bytes": 441044,\n      "sha256": "c9e96bcc6a2781e66fed8cc3a7ac9980d3bfc18c55cc1be27a606803843b2399",\n      "decoded_sha256": "38f79b518736b16ae0842094bff1859b1b79ac513f5c1e090b6af52c465ef002",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 8,\n      "category": "sea_waves",\n      "target": 11,\n      "group_id": "219379",\n      "source_id": "219379",\n      "attribution": "- [5-219379-C.ogg]: clip derived from the night Baltic001_14022014.wav (http://www.freesound.org/people/miastodzwiekow/sounds/219379/) by miastodzwiekow [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-220026-A-21",\n      "filename": "5-220026-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-220026-A-21.wav",\n      "bytes": 441044,\n      "sha256": "01d02b64a5bbf7bf29672883699b1ecebd16ed6f5bd138ad2fc905214b65b63b",\n      "decoded_sha256": "dabb10454fa0f9ca1b9d180c089e0ce490dca27a8ff28d02929bb615681b78bc",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "220026",\n      "source_id": "220026",\n      "attribution": "- [5-220026-A.ogg]: clip derived from mysneeze2.wav (http://www.freesound.org/people/Reitanna/sounds/220026/) by Reitanna [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-220027-A-21",\n      "filename": "5-220027-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-220027-A-21.wav",\n      "bytes": 441044,\n      "sha256": "d5e90d42600c83b3b4ea0223f5afdc9aa92b931ddf432eecaeca112beac69f42",\n      "decoded_sha256": "426c01bc7f04bd0340012ccd8b40ecf0a9d624ff8b4520192f7cc8a34641774b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "220027",\n      "source_id": "220027",\n      "attribution": "- [5-220027-A.ogg]: clip derived from mysneeze.wav (http://www.freesound.org/people/Reitanna/sounds/220027/) by Reitanna [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-220955-A-40",\n      "filename": "5-220955-A-40.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-220955-A-40.wav",\n      "bytes": 441044,\n      "sha256": "0f65b8ca406c78dffd82f8e7e4deed7613154de9ee58856b90327e0cf356ad18",\n      "decoded_sha256": "6c07fc0ecf99673e5f572cfcd861cee1ea03f0d842adcb2fda0e90e20209886b",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 5,\n      "category": "helicopter",\n      "target": 40,\n      "group_id": "220955",\n      "source_id": "220955",\n      "attribution": "- [5-220955-A.ogg]: clip derived from Helicopter.wav (http://www.freesound.org/people/Islabonita/sounds/220955/) by Islabonita [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-221518-A-21",\n      "filename": "5-221518-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-221518-A-21.wav",\n      "bytes": 441044,\n      "sha256": "0015a35c6851eee3a03b145f6703f4980a8a34ca6a69b61324cc7c7cad8896b6",\n      "decoded_sha256": "cb38c79a57d7e1395a8d64b8b2d2f56dfa0f6847741f85b591a4ce3111c39cd2",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "221518",\n      "source_id": "221518",\n      "attribution": "- [5-221518-A.ogg]: clip derived from woman sneeze (http://www.freesound.org/people/Danieka/sounds/221518/) by Danieka [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-221593-A-21",\n      "filename": "5-221593-A-21.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-221593-A-21.wav",\n      "bytes": 441044,\n      "sha256": "603d9fd8af740638eaf529212f6413c1a63a771accd5970b537c5deb93da117f",\n      "decoded_sha256": "82c36389443e5122257b1f84672b76c06fd356b1d622d1fa28cc5e779db288d9",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 9,\n      "category": "sneezing",\n      "target": 21,\n      "group_id": "221593",\n      "source_id": "221593",\n      "attribution": "- [5-221593-A.ogg]: clip derived from Girly sneeze (http://www.freesound.org/people/ahjteam/sounds/221593/) by ahjteam [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-222524-A-41",\n      "filename": "5-222524-A-41.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-222524-A-41.wav",\n      "bytes": 441044,\n      "sha256": "0161df57f3e454993357e6a2082d895a9ec7dbeb83e730e63bde030b46837136",\n      "decoded_sha256": "996f3bf7a6fb1bd0ecbd48ccf308faa7a8784651b70af89533e556164ecaf2df",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 0,\n      "category": "chainsaw",\n      "target": 41,\n      "group_id": "222524",\n      "source_id": "222524",\n      "attribution": "- [5-222524-A.ogg]: clip derived from wood chipper and chainsaw (http://www.freesound.org/people/Darrenlibrarian/sounds/222524/) by Darrenlibrarian [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-231762-A-0",\n      "filename": "5-231762-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-231762-A-0.wav",\n      "bytes": 441044,\n      "sha256": "cb686222c5eeb07c8c8ee508adce7bedbabb3358aee5c46b051788075835bd96",\n      "decoded_sha256": "da81809d931f6e64442b58a441f13f9236da5e124deabd2440e12656628c71f8",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "231762",\n      "source_id": "231762",\n      "attribution": "- [5-231762-A.ogg]: clip derived from FX_Dog Barking.wav (http://www.freesound.org/people/davidmenke/sounds/231762/) by davidmenke [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-233160-A-1",\n      "filename": "5-233160-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-233160-A-1.wav",\n      "bytes": 441044,\n      "sha256": "efa062fe484e9c4ec3ac72227f630152cd9b9622631afc26129a47d62155f527",\n      "decoded_sha256": "4f040bf56c04082049838e5d392339a893ec726858526075c4d42fbfe9bc4d23",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "233160",\n      "source_id": "233160",\n      "attribution": "- [5-233160-A.ogg]: clip derived from Rooster - Crow 2 - 96kHz.wav (http://www.freesound.org/people/JarredGibb/sounds/233160/) by JarredGibb [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-234879-A-1",\n      "filename": "5-234879-A-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-234879-A-1.wav",\n      "bytes": 441044,\n      "sha256": "89cfcee14646d5f23317afd07c2ef80e590fb12471029a3cd0cfa438a473c7af",\n      "decoded_sha256": "a4eb82a01bb487744dd3b7e42b90cbb5bfa22400a2602a738c900935a5d3932a",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "234879",\n      "source_id": "234879",\n      "attribution": "- [5-234879-A.ogg]: clip derived from chickensMS.wav (http://www.freesound.org/people/leosalom/sounds/234879/) by leosalom [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-234879-B-1",\n      "filename": "5-234879-B-1.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-234879-B-1.wav",\n      "bytes": 441044,\n      "sha256": "8f0a677caef84cd230aac3377a8db8a357971816e1943a5ca646289950b54024",\n      "decoded_sha256": "180ff669b6b6b6c04bd87aa8229f65cf637ca051d75b9f4bbf4f753b3406cf2d",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 7,\n      "category": "rooster",\n      "target": 1,\n      "group_id": "234879",\n      "source_id": "234879",\n      "attribution": "- [5-234879-B.ogg]: clip derived from chickensMS.wav (http://www.freesound.org/people/leosalom/sounds/234879/) by leosalom [CC0]",\n      "original_license": "CC0"\n    },\n    {\n      "id": "5-235671-A-38",\n      "filename": "5-235671-A-38.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-235671-A-38.wav",\n      "bytes": 441044,\n      "sha256": "28a721a5db87aed186e7cd63439c798ba9c0e44cf6654340d7b1ac2e7af02eee",\n      "decoded_sha256": "3d65ab81ca8a422019f318daf6e4a0425b61194603b088d57194078b67f56b80",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 1,\n      "category": "clock_tick",\n      "target": 38,\n      "group_id": "235671",\n      "source_id": "235671",\n      "attribution": "- [5-235671-A.ogg]: clip derived from clock tick.aif (http://www.freesound.org/people/gpenn76/sounds/235671/) by gpenn76 [CC-BY]",\n      "original_license": "CC-BY"\n    },\n    {\n      "id": "5-9032-A-0",\n      "filename": "5-9032-A-0.wav",\n      "url": "https://raw.githubusercontent.com/karolpiczak/ESC-50/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6/audio/5-9032-A-0.wav",\n      "bytes": 441044,\n      "sha256": "e49492e3cd7be48c3e5f82a121e71f23687ecb047609085231e06a0cccb2b12d",\n      "decoded_sha256": "093d410f28c666c08315c858f0a46c361a522900219a89e3a99bcf15a1f17614",\n      "sample_rate": 44100,\n      "channels": 1,\n      "frames": 220500,\n      "role": "test",\n      "fold": 5,\n      "label": 4,\n      "category": "dog",\n      "target": 0,\n      "group_id": "9032",\n      "source_id": "9032",\n      "attribution": "- [5-9032-A.ogg]:  clip derived from Dog bark2.wav (http://www.freesound.org/people/MisterTood/sounds/9032/) by MisterTood [CC0]",\n      "original_license": "CC0"\n    }\n  ]\n}\n', 'licenses/esc-attribution.txt': '# ESC-50: Dataset for Environmental Sound Classification\n# https://github.com/karoldvl/ESC-50/\n# https://dx.doi.org/10.7910/DVN/YDEPUT\n\n## Dataset license\n\nThe dataset as a whole is available under the terms of the Creative Commons\nAttribution-NonCommercial license (http://creativecommons.org/licenses/by-nc/3.0/).\n\nThe ESC-10 subset is licensed as a Creative Commons Attribution 3.0 Unported\n(https://creativecommons.org/licenses/by/3.0/) dataset.\n\n## Licensing/attribution details for individual clips:\n\n101 - Dog:\n- [1-100032-A.ogg]: clip derived from rose_bark.wav (http://www.freesound.org/people/nfrae/sounds/100032/) by nfrae [CC0]\n- [1-110389-A.ogg]: clip derived from animals_dog_bark_springer_spaniel_001.wav (http://www.freesound.org/people/soundscalpel.com/sounds/110389/) by soundscalpel.com [CC-BY]\n- [1-30226-A.ogg]: clip derived from dogs.aiff (http://www.freesound.org/people/Heigh-hoo/sounds/30226/) by Heigh-hoo [CC0]\n- [1-30344-A.ogg]: clip derived from My Dog George.wav (http://www.freesound.org/people/ronfont/sounds/30344/) by ronfont [CC-BY]\n- [1-32318-A.ogg]: clip derived from MABEL 1.aif (http://www.freesound.org/people/Bobby%20Beck/sounds/32318/) by Bobby Beck [CC0]\n- [1-59513-A.ogg]: clip derived from Blaffende hond2.wav (http://www.freesound.org/people/zeshoog/sounds/59513/) by zeshoog [CC0]\n- [1-85362-A.ogg]: clip derived from 20091211.barking.stairs.wav (http://www.freesound.org/people/dobroide/sounds/85362/) by dobroide [CC-BY]\n- [1-97392-A.ogg]: clip derived from dog barking.wav (http://www.freesound.org/people/Tomlija/sounds/97392/) by Tomlija [CC-BY]\n- [2-114280-A.ogg]: clip derived from dogbarks.wav (http://www.freesound.org/people/plingativator/sounds/114280/) by plingativator [CC-BY]\n- [2-114587-A.ogg]: clip derived from Buddy Bark.wav (http://www.freesound.org/people/InDaHouse20/sounds/114587/) by InDaHouse20 [CC0]\n- [2-116400-A.ogg]: clip derived from 2011-03-09-dog-in-the-night.flac (http://www.freesound.org/people/ayamahambho/sounds/116400/) by ayamahambho [CC-BY]\n- [2-117271-A.ogg]: clip derived from angry_dogs.wav (http://www.freesound.org/people/polipa/sounds/117271/) by polipa [CC0]\n- [2-118072-A.ogg]: clip derived from dogbark.wav (http://www.freesound.org/people/ipears1/sounds/118072/) by ipears1 [CC0]\n- [2-118964-A.ogg]: clip derived from Dog Bark 4.wav (http://www.freesound.org/people/esperri/sounds/118964/) by esperri [CC0]\n- [2-122104-A.ogg]: clip derived from Dog barking at night (nice reflection).wav (http://www.freesound.org/people/rickyjezz/sounds/122104/) by rickyjezz [CC0]\n- [2-122104-B.ogg]: clip derived from Dog barking at night (nice reflection).wav (http://www.freesound.org/people/rickyjezz/sounds/122104/) by rickyjezz [CC0]\n- [3-136288-A.ogg]: clip derived from Dogs barking and birds singing in the Red Wood Forest, close to San Francisco. Road in the far background. (http://www.freesound.org/people/felix.blume/sounds/136288/) by felix.blume [CC0]\n- [3-144028-A.ogg]: clip derived from dog.aif (http://www.freesound.org/people/djym/sounds/144028/) by djym [CC0]\n- [3-155312-A.ogg]: clip derived from Dog Barking (http://www.freesound.org/people/Jace/sounds/155312/) by Jace [CC0]\n- [3-157695-A.ogg]: clip derived from Dog on a chain (http://www.freesound.org/people/urupin/sounds/157695/) by urupin [CC-BY]\n- [3-163459-A.ogg]: clip derived from LBS_FX DOG Small Alert Bark001.wav (http://www.freesound.org/people/LittleBigSounds/sounds/163459/) by LittleBigSounds [CC0]\n- [3-170015-A.ogg]: clip derived from one bark of a poodle dog (http://www.freesound.org/people/fabiopx/sounds/170015/) by fabiopx [CC0]\n- [3-180256-A.ogg]: clip derived from Dog Barking - Shih Tzu Pug (http://www.freesound.org/people/bspiller5/sounds/180256/) by bspiller5 [CC-BY]\n- [3-180977-A.ogg]: clip derived from Barking Dog (http://www.freesound.org/people/Suprasummun/sounds/180977/) by Suprasummun [CC0]\n- [4-182395-A.ogg]: clip derived from Big_dog_bark_01.aiff (http://www.freesound.org/people/pgonsilva/sounds/182395/) by pgonsilva [CC-BY]\n- [4-183992-A.ogg]: clip derived from Supermarket Dog.flac (http://www.freesound.org/people/qubodup/sounds/183992/) by qubodup [CC0]\n- [4-184575-A.ogg]: clip derived from Dog barking outside (http://www.freesound.org/people/magnus589/sounds/184575/) by magnus589 [CC0]\n- [4-191687-A.ogg]: clip derived from Dog Barks.wav (http://www.freesound.org/people/UnderlinedDesigns/sounds/191687/) by UnderlinedDesigns [CC0]\n- [4-192236-A.ogg]: clip derived from Miniature Dachshund Bark - Indoors.wav (http://www.freesound.org/people/Ligidium/sounds/192236/) by Ligidium [CC0]\n- [4-194754-A.ogg]: clip derived from Dog Barking in Brussels Street (http://www.freesound.org/people/qubodup/sounds/194754/) by qubodup [CC-BY]\n- [4-199261-A.ogg]: clip derived from A dog barking (close recording) (http://www.freesound.org/people/felix.blume/sounds/199261/) by felix.blume [CC0]\n- [4-207124-A.ogg]: clip derived from DogBARKING.wav (http://www.freesound.org/people/ahill86/sounds/207124/) by ahill86 [CC0]\n- [5-203128-A.ogg]: clip derived from Toupie1.mp3 (http://www.freesound.org/people/Pierre%20Grandjean/sounds/203128/) by Pierre Grandjean [CC0]\n- [5-203128-B.ogg]: clip derived from Toupie1.mp3 (http://www.freesound.org/people/Pierre%20Grandjean/sounds/203128/) by Pierre Grandjean [CC0]\n- [5-208030-A.ogg]: clip derived from Chihuahua Barks (http://www.freesound.org/people/Mewsel/sounds/208030/) by Mewsel [CC0]\n- [5-212454-A.ogg]: clip derived from Barking (http://www.freesound.org/people/unfa/sounds/212454/) by unfa [CC-BY]\n- [5-213855-A.ogg]: clip derived from Tiny Dog Barking in Park (http://www.freesound.org/people/qubodup/sounds/213855/) by qubodup [CC-BY]\n- [5-217158-A.ogg]: clip derived from barkingdog_in_dream.mp3 (http://www.freesound.org/people/Taira%20Komori/sounds/217158/) by Taira Komori [CC-BY]\n- [5-231762-A.ogg]: clip derived from FX_Dog Barking.wav (http://www.freesound.org/people/davidmenke/sounds/231762/) by davidmenke [CC0]\n- [5-9032-A.ogg]:  clip derived from Dog bark2.wav (http://www.freesound.org/people/MisterTood/sounds/9032/) by MisterTood [CC0]\n\n102 - Rooster:\n- [1-26806-A.ogg]: clip derived from crows.aiff (http://www.freesound.org/people/jimblodget/sounds/26806/) by jimblodget [CC-BY]\n- [1-27724-A.ogg]: clip derived from citadel park gand.wav (http://www.freesound.org/people/philippe%20b/sounds/27724/) by philippe b [CC0]\n- [1-34119-A.ogg]: clip derived from cockeril.wav (http://www.freesound.org/people/Charliefarley/sounds/34119/) by Charliefarley [CC0]\n- [1-34119-B.ogg]: clip derived from cockeril.wav (http://www.freesound.org/people/Charliefarley/sounds/34119/) by Charliefarley [CC0]\n- [1-39923-A.ogg]: clip derived from 20070812.rooster.wav (http://www.freesound.org/people/dobroide/sounds/39923/) by dobroide [CC-BY]\n- [1-40730-A.ogg]: clip derived from rooster2.wav (http://www.freesound.org/people/pionium/sounds/40730/) by pionium [CC-Sampling+]\n- [1-43382-A.ogg]: clip derived from Rooster chicken calls_2.wav (http://www.freesound.org/people/AGFX/sounds/43382/) by AGFX [CC-BY]\n- [1-44831-A.ogg]: clip derived from Animal Conversation.mp3 (http://www.freesound.org/people/fauxpress/sounds/44831/) by fauxpress [CC0]\n- [2-100786-A.ogg]: clip derived from rooster 2.aif (http://www.freesound.org/people/Tomlija/sounds/100786/) by Tomlija [CC-BY]\n- [2-65750-A.ogg]: clip derived from 20081231.rooster.flac (http://www.freesound.org/people/dobroide/sounds/65750/) by dobroide [CC-BY]\n- [2-71162-A.ogg]: clip derived from bees buzzing.aif (http://www.freesound.org/people/simongray/sounds/71162/) by simongray [CC0]\n- [2-81270-A.ogg]: clip derived from animals-hen-rooster-Blaafarveverket-090721.wav (http://www.freesound.org/people/ra_gun/sounds/81270/) by ra_gun [CC0]\n- [2-95035-A.ogg]: clip derived from 20100418.marshes.storm.03.wav (http://www.freesound.org/people/dobroide/sounds/95035/) by dobroide [CC-BY]\n- [2-95258-A.ogg]: clip derived from Rural Back Yard Whitehill Nova Scotia April 21 2010.mp3 (http://www.freesound.org/people/wildear1/sounds/95258/) by wildear1 [CC-BY]\n- [2-95258-B.ogg]: clip derived from Rural Back Yard Whitehill Nova Scotia April 21 2010.mp3 (http://www.freesound.org/people/wildear1/sounds/95258/) by wildear1 [CC-BY]\n- [2-96460-A.ogg]: clip derived from ambience_pigeons_roosters_hens.mp3 (http://www.freesound.org/people/123a4567/sounds/96460/) by 123a4567 [CC-Sampling+]\n- [3-107219-A.ogg]: clip derived from birmingham-nechells-rooster.wav (http://www.freesound.org/people/keithpeter/sounds/107219/) by keithpeter [CC0]\n- [3-116135-A.ogg]: clip derived from Rooster and chicks.mp3 (http://www.freesound.org/people/Barbamiz/sounds/116135/) by Barbamiz [CC0]\n- [3-134049-A.ogg]: clip derived from cock and hen.WAV (http://www.freesound.org/people/Setuniman/sounds/134049/) by Setuniman [CC-BY]\n- [3-137152-A.ogg]: clip derived from 20111206_02.rooster.wav (http://www.freesound.org/people/dobroide/sounds/137152/) by dobroide [CC-BY]\n- [3-145382-A.ogg]: clip derived from Morning start.mp3 (http://www.freesound.org/people/NikosDemetriou/sounds/145382/) by NikosDemetriou [CC-BY]\n- [3-149189-A.ogg]: clip derived from rooster cock-a-doodle-doo (http://www.freesound.org/people/videog/sounds/149189/) by videog [CC0]\n- [3-154957-A.ogg]: clip derived from brahma-rooster.wav (http://www.freesound.org/people/signorvaso/sounds/154957/) by signorvaso [CC0]\n- [3-163288-A.ogg]: clip derived from Rooster_Mono.wav (http://www.freesound.org/people/cclaretc/sounds/163288/) by cclaretc [CC0]\n- [4-164021-A.ogg]: clip derived from Cock-a-doodle-doo (http://www.freesound.org/people/kangaroovindaloo/sounds/164021/) by kangaroovindaloo [CC-BY]\n- [4-164064-A.ogg]: clip derived from Rooster_Ste.wav (http://www.freesound.org/people/cclaretc/sounds/164064/) by cclaretc [CC-BY]\n- [4-164064-B.ogg]: clip derived from Rooster_Ste.wav (http://www.freesound.org/people/cclaretc/sounds/164064/) by cclaretc [CC-BY]\n- [4-164064-C.ogg]: clip derived from Rooster_Ste.wav (http://www.freesound.org/people/cclaretc/sounds/164064/) by cclaretc [CC-BY]\n- [4-164859-A.ogg]: clip derived from rooster.wav (http://www.freesound.org/people/bob58/sounds/164859/) by bob58 [CC0]\n- [4-170078-A.ogg]: clip derived from Rooster at Max\'s.mp3 (http://www.freesound.org/people/dudermensch/sounds/170078/) by dudermensch [CC0]\n- [4-183487-A.ogg]: clip derived from Chickens.wav (http://www.freesound.org/people/ecfike/sounds/183487/) by ecfike [CC0]\n- [4-208021-A.ogg]: clip derived from Cock-a-doodle-doo (http://www.freesound.org/people/cocoricosound/sounds/208021/) by cocoricosound [CC0]\n- [5-194930-A.ogg]: clip derived from cock crows near and far .mp3 (http://www.freesound.org/people/soundmary/sounds/194930/) by soundmary [CC-BY]\n- [5-194930-B.ogg]: clip derived from cock crows near and far .mp3 (http://www.freesound.org/people/soundmary/sounds/194930/) by soundmary [CC-BY]\n- [5-200334-A.ogg]: clip derived from rooster_crows_inside.aif (http://www.freesound.org/people/jsbarrett/sounds/200334/) by jsbarrett [CC0]\n- [5-200334-B.ogg]: clip derived from rooster_crows_inside.aif (http://www.freesound.org/people/jsbarrett/sounds/200334/) by jsbarrett [CC0]\n- [5-200339-A.ogg]: clip derived from rooster_crows_outside.aif (http://www.freesound.org/people/jsbarrett/sounds/200339/) by jsbarrett [CC0]\n- [5-233160-A.ogg]: clip derived from Rooster - Crow 2 - 96kHz.wav (http://www.freesound.org/people/JarredGibb/sounds/233160/) by JarredGibb [CC0]\n- [5-234879-A.ogg]: clip derived from chickensMS.wav (http://www.freesound.org/people/leosalom/sounds/234879/) by leosalom [CC0]\n- [5-234879-B.ogg]: clip derived from chickensMS.wav (http://www.freesound.org/people/leosalom/sounds/234879/) by leosalom [CC0]\n\n103 - Pig:\n- [1-208757-A.ogg]: clip derived from Pigs in an intensive pigs farming, in Texas, USA. (http://www.freesound.org/people/felix.blume/sounds/208757/) by felix.blume [CC0]\n- [1-208757-B.ogg]: clip derived from Pigs in an intensive pigs farming, in Texas, USA. (http://www.freesound.org/people/felix.blume/sounds/208757/) by felix.blume [CC0]\n- [1-208757-C.ogg]: clip derived from Pigs in an intensive pigs farming, in Texas, USA. (http://www.freesound.org/people/felix.blume/sounds/208757/) by felix.blume [CC0]\n- [1-208757-D.ogg]: clip derived from Pigs in an intensive pigs farming, in Texas, USA. (http://www.freesound.org/people/felix.blume/sounds/208757/) by felix.blume [CC0]\n- [1-208757-E.ogg]: clip derived from Pigs in an intensive pigs farming, in Texas, USA. (http://www.freesound.org/people/felix.blume/sounds/208757/) by felix.blume [CC0]\n- [1-260640-A.ogg]: clip derived from Pig Squealing.mp3 (http://www.freesound.org/people/TheAcidRomance/sounds/260640/) by TheAcidRomance [CC0]\n- [1-260640-B.ogg]: clip derived from Pig Squealing.mp3 (http://www.freesound.org/people/TheAcidRomance/sounds/260640/) by TheAcidRomance [CC0]\n- [1-260640-C.ogg]: clip derived from Pig Squealing.mp3 (http://www.freesound.org/people/TheAcidRomance/sounds/260640/) by TheAcidRomance [CC0]\n- [2-158746-A.ogg]: clip derived from A pig grunting, grumbling and falling asleep (France, Limousin) (http://www.freesound.org/people/felix.blume/sounds/158746/) by felix.blume [CC0]\n- [2-158746-B.ogg]: clip derived from A pig grunting, grumbling and falling asleep (France, Limousin) (http://www.freesound.org/people/felix.blume/sounds/158746/) by felix.blume [CC0]\n- [2-158746-C.ogg]: clip derived from A pig grunting, grumbling and falling asleep (France, Limousin) (http://www.freesound.org/people/felix.blume/sounds/158746/) by felix.blume [CC0]\n- [2-158746-D.ogg]: clip derived from A pig grunting, grumbling and falling asleep (France, Limousin) (http://www.freesound.org/people/felix.blume/sounds/158746/) by felix.blume [CC0]\n- [2-166644-A.ogg]: clip derived from Pigs in a farm (10 pigs inside a French farm) (http://www.freesound.org/people/felix.blume/sounds/166644/) by felix.blume [CC0]\n- [2-166644-B.ogg]: clip derived from Pigs in a farm (10 pigs inside a French farm) (http://www.freesound.org/people/felix.blume/sounds/166644/) by felix.blume [CC0]\n- [2-166644-C.ogg]: clip derived from Pigs in a farm (10 pigs inside a French farm) (http://www.freesound.org/people/felix.blume/sounds/166644/) by felix.blume [CC0]\n- [2-37870-A.ogg]: clip derived from Pigs-01.flac (http://www.freesound.org/people/Erdie/sounds/37870/) by Erdie [CC-BY]\n- [3-233151-A.ogg]: clip derived from Pig - Scream 5 - 96kHz.wav (http://www.freesound.org/people/JarredGibb/sounds/233151/) by JarredGibb [CC0]\n- [3-253081-A.ogg]: clip derived from Sikala, siat sy\\xf6v\\xe4t ja r\\xf6hkiv\\xe4t / A piggery, pigs eating and grunting (http://www.freesound.org/people/YleArkisto/sounds/253081/) by YleArkisto [CC-BY]\n- [3-253084-A.ogg]: clip derived from Porsas r\\xf6hkii ja kiljuu / A piglet oinking and squealling in the pen (http://www.freesound.org/people/YleArkisto/sounds/253084/) by YleArkisto [CC-BY]\n- [3-253084-B.ogg]: clip derived from Porsas r\\xf6hkii ja kiljuu / A piglet oinking and squealling in the pen (http://www.freesound.org/people/YleArkisto/sounds/253084/) by YleArkisto [CC-BY]\n- [3-253084-C.ogg]: clip derived from Porsas r\\xf6hkii ja kiljuu / A piglet oinking and squealling in the pen (http://www.freesound.org/people/YleArkisto/sounds/253084/) by YleArkisto [CC-BY]\n- [3-253084-D.ogg]: clip derived from Porsas r\\xf6hkii ja kiljuu / A piglet oinking and squealling in the pen (http://www.freesound.org/people/YleArkisto/sounds/253084/) by YleArkisto [CC-BY]\n- [3-253084-E.ogg]: clip derived from Porsas r\\xf6hkii ja kiljuu / A piglet oinking and squealling in the pen (http://www.freesound.org/people/YleArkisto/sounds/253084/) by YleArkisto [CC-BY]\n- [3-257858-A.ogg]: clip derived from Pig_Grunts_Snorts_Breathing_Hackney_City_Farm.wav (http://www.freesound.org/people/tompallant/sounds/257858/) by tompallant [CC0]\n- [4-132383-A.ogg]: clip derived from Black_Pigs_Fighting_Alentejo_Portugal_2011.wav (http://www.freesound.org/people/ideiasemvideo/sounds/132383/) by ideiasemvideo [CC0]\n- [4-132383-B.ogg]: clip derived from Black_Pigs_Fighting_Alentejo_Portugal_2011.wav (http://www.freesound.org/people/ideiasemvideo/sounds/132383/) by ideiasemvideo [CC0]\n- [4-147240-A.ogg]: clip derived from Two pigs by the river, grunting and squealing. Iruya, Northwest Argentina.  (http://www.freesound.org/people/MatiasEzcurra/sounds/147240/) by MatiasEzcurra [CC0]\n- [4-147240-B.ogg]: clip derived from Two pigs by the river, grunting and squealing. Iruya, Northwest Argentina.  (http://www.freesound.org/people/MatiasEzcurra/sounds/147240/) by MatiasEzcurra [CC0]\n- [4-234644-A.ogg]: clip derived from Two pigs in conversation (http://www.freesound.org/people/Thalamus_Lab/sounds/234644/) by Thalamus_Lab [CC-BY]\n- [4-250869-A.ogg]: clip derived from Porsas kiljuu sikalassa / A piglet squealling loudly in the piggery (http://www.freesound.org/people/YleArkisto/sounds/250869/) by YleArkisto [CC-BY]\n- [4-250869-B.ogg]: clip derived from Porsas kiljuu sikalassa / A piglet squealling loudly in the piggery (http://www.freesound.org/people/YleArkisto/sounds/250869/) by YleArkisto [CC-BY]\n- [4-250869-C.ogg]: clip derived from Porsas kiljuu sikalassa / A piglet squealling loudly in the piggery (http://www.freesound.org/people/YleArkisto/sounds/250869/) by YleArkisto [CC-BY]\n- [5-103415-A.ogg]: clip derived from Pig 1 (C).wav (http://www.freesound.org/people/confusion_music/sounds/103415/) by confusion_music [CC-BY]\n- [5-103416-A.ogg]: clip derived from Pig 2 (B).wav (http://www.freesound.org/people/confusion_music/sounds/103416/) by confusion_music [CC-BY]\n- [5-103418-A.ogg]: clip derived from Pig 4 (B).wav (http://www.freesound.org/people/confusion_music/sounds/103418/) by confusion_music [CC-BY]\n- [5-103420-A.ogg]: clip derived from Pig 6 (A).wav (http://www.freesound.org/people/confusion_music/sounds/103420/) by confusion_music [CC-BY]\n- [5-103421-A.ogg]: clip derived from Pig 7 (B).wav (http://www.freesound.org/people/confusion_music/sounds/103421/) by confusion_music [CC-BY]\n- [5-103422-A.ogg]: clip derived from Pig 8 (C).wav (http://www.freesound.org/people/confusion_music/sounds/103422/) by confusion_music [CC-BY-NC]\n- [5-117250-A.ogg]: clip derived from boar.mp3 (http://www.freesound.org/people/rvinyard/sounds/117250/) by rvinyard [CC-BY]\n- [5-127990-A.ogg]: clip derived from grunting_and_munching_Pigs.wav (http://www.freesound.org/people/1pjladd2/sounds/127990/) by 1pjladd2 [CC0]\n\n104 - Cow:\n- [1-16568-A.ogg]: clip derived from TwoCows.wav (http://www.freesound.org/people/acclivity/sounds/16568/) by acclivity [CC-BY-NC]\n- [1-202111-A.ogg]: clip derived from Two bulls singing in the desert of Arizona (http://www.freesound.org/people/felix.blume/sounds/202111/) by felix.blume [CC0]\n- [1-58277-A.ogg]: clip derived from Cow.wav (http://www.freesound.org/people/Benboncan/sounds/58277/) by Benboncan [CC-BY]\n- [1-69422-A.ogg]: clip derived from Calf And Cow.wav (http://www.freesound.org/people/Benboncan/sounds/69422/) by Benboncan [CC-BY]\n- [1-69641-A.ogg]: clip derived from Cow2.wav (http://www.freesound.org/people/genghis%20attenborough/sounds/69641/) by genghis attenborough [CC-BY-NC]\n- [1-77160-A.ogg]: clip derived from 01186 village walk 3.wav (http://www.freesound.org/people/Robinhood76/sounds/77160/) by Robinhood76 [CC-BY-NC]\n- [1-77241-A.ogg]: clip derived from 01188 cow munch one shot 2.wav (http://www.freesound.org/people/Robinhood76/sounds/77241/) by Robinhood76 [CC-BY-NC]\n- [1-81269-A.ogg]: clip derived from ambience-Blaafarveverket 02-090721.wav (http://www.freesound.org/people/ra_gun/sounds/81269/) by ra_gun [CC0]\n- [2-103423-A.ogg]: clip derived from bull 1 (B).wav (http://www.freesound.org/people/confusion_music/sounds/103423/) by confusion_music [CC-BY]\n- [2-103424-A.ogg]: clip derived from bull 2 (C).wav (http://www.freesound.org/people/confusion_music/sounds/103424/) by confusion_music [CC-BY]\n- [2-103426-A.ogg]: clip derived from bull 4 (C).wav (http://www.freesound.org/people/confusion_music/sounds/103426/) by confusion_music [CC-BY]\n- [2-103427-A.ogg]: clip derived from bull 5 (B).wav (http://www.freesound.org/people/confusion_music/sounds/103427/) by confusion_music [CC-BY]\n- [2-103428-A.ogg]: clip derived from bull 6 (C).wav (http://www.freesound.org/people/confusion_music/sounds/103428/) by confusion_music [CC-BY]\n- [2-104877-A.ogg]: clip derived from Cows muing.wav (http://www.freesound.org/people/Kulanen/sounds/104877/) by Kulanen [CC0]\n- [2-117795-A.ogg]: clip derived from Cows moo at 10m in an estancia of Tierra del fuego (Argentina). (http://www.freesound.org/people/felix.blume/sounds/117795/) by felix.blume [CC0]\n- [2-117795-B.ogg]: clip derived from Cows moo at 10m in an estancia of Tierra del fuego (Argentina). (http://www.freesound.org/people/felix.blume/sounds/117795/) by felix.blume [CC0]\n- [3-124376-A.ogg]: clip derived from Cows at Casterino resting 1.flac (http://www.freesound.org/people/truflabart/sounds/124376/) by truflabart [CC-BY-NC]\n- [3-124376-B.ogg]: clip derived from Cows at Casterino resting 1.flac (http://www.freesound.org/people/truflabart/sounds/124376/) by truflabart [CC-BY-NC]\n- [3-126358-A.ogg]: clip derived from Herd Of Cows Mooing (http://www.freesound.org/people/tdes/sounds/126358/) by tdes [CC-BY]\n- [3-126358-B.ogg]: clip derived from Herd Of Cows Mooing (http://www.freesound.org/people/tdes/sounds/126358/) by tdes [CC-BY]\n- [3-152039-A.ogg]: clip derived from Uighur Animal Market.wav (http://www.freesound.org/people/RTB45/sounds/152039/) by RTB45 [CC-BY]\n- [3-152039-B.ogg]: clip derived from Uighur Animal Market.wav (http://www.freesound.org/people/RTB45/sounds/152039/) by RTB45 [CC-BY]\n- [3-160993-A.ogg]: clip derived from Stockyard 1.wav (http://www.freesound.org/people/ecfike/sounds/160993/) by ecfike [CC0]\n- [3-163727-A.ogg]: clip derived from Cow mooing in south of France (Limousin) (http://www.freesound.org/people/felix.blume/sounds/163727/) by felix.blume [CC0]\n- [4-174860-A.ogg]: clip derived from Cows herd walking in the bush in Africa (Mali) (http://www.freesound.org/people/felix.blume/sounds/174860/) by felix.blume [CC0]\n- [4-174860-B.ogg]: clip derived from Cows herd walking in the bush in Africa (Mali) (http://www.freesound.org/people/felix.blume/sounds/174860/) by felix.blume [CC0]\n- [4-181955-A.ogg]: clip derived from Felix Blume - Cows Mooing 1 min (http://www.freesound.org/people/qubodup/sounds/181955/) by qubodup [CC0]\n- [4-181955-B.ogg]: clip derived from Felix Blume - Cows Mooing 1 min (http://www.freesound.org/people/qubodup/sounds/181955/) by qubodup [CC0]\n- [4-181955-C.ogg]: clip derived from Felix Blume - Cows Mooing 1 min (http://www.freesound.org/people/qubodup/sounds/181955/) by qubodup [CC0]\n- [4-213915-A.ogg]: clip derived from Cows mooing in a stockyard in St Joseph, Missouri, USA. (http://www.freesound.org/people/felix.blume/sounds/213915/) by felix.blume [CC0]\n- [4-213915-B.ogg]: clip derived from Cows mooing in a stockyard in St Joseph, Missouri, USA. (http://www.freesound.org/people/felix.blume/sounds/213915/) by felix.blume [CC0]\n- [4-213915-C.ogg]: clip derived from Cows mooing in a stockyard in St Joseph, Missouri, USA. (http://www.freesound.org/people/felix.blume/sounds/213915/) by felix.blume [CC0]\n- [5-194899-A.ogg]: clip derived from Distressed Mother Cow and Calves.wav (http://www.freesound.org/people/lolamadeus/sounds/194899/) by lolamadeus [CC0]\n- [5-194899-B.ogg]: clip derived from Distressed Mother Cow and Calves.wav (http://www.freesound.org/people/lolamadeus/sounds/194899/) by lolamadeus [CC0]\n- [5-194899-C.ogg]: clip derived from Distressed Mother Cow and Calves.wav (http://www.freesound.org/people/lolamadeus/sounds/194899/) by lolamadeus [CC0]\n- [5-194899-D.ogg]: clip derived from Distressed Mother Cow and Calves.wav (http://www.freesound.org/people/lolamadeus/sounds/194899/) by lolamadeus [CC0]\n- [5-202795-A.ogg]: clip derived from Bulls and cows in a field in Missouri, USA. (http://www.freesound.org/people/felix.blume/sounds/202795/) by felix.blume [CC0]\n- [5-242492-A.ogg]: clip derived from Mooing cow 170713_0299.wav (http://www.freesound.org/people/klankbeeld/sounds/242492/) by klankbeeld [CC-BY]\n- [5-253085-A.ogg]: clip derived from Vasikat ammuvat /  Two calves on the pasture, now and then mooing (http://www.freesound.org/people/YleArkisto/sounds/253085/) by YleArkisto [CC-BY]\n- [5-253085-B.ogg]: clip derived from Vasikat ammuvat /  Two calves on the pasture, now and then mooing (http://www.freesound.org/people/YleArkisto/sounds/253085/) by YleArkisto [CC-BY]\n\n105 - Frog:\n- [1-15689-A.ogg]: clip derived from tree frog.wav (http://www.freesound.org/people/reinsamba/sounds/15689/) by reinsamba [CC-BY]\n- [1-15689-B.ogg]: clip derived from tree frog.wav (http://www.freesound.org/people/reinsamba/sounds/15689/) by reinsamba [CC-BY]\n- [1-17970-A.ogg]: clip derived from Water frogs.wav (http://www.freesound.org/people/reinsamba/sounds/17970/) by reinsamba [CC-BY]\n- [1-18755-A.ogg]: clip derived from frogs01.wav (http://www.freesound.org/people/WIM/sounds/18755/) by WIM [CC-BY]\n- [1-18755-B.ogg]: clip derived from frogs01.wav (http://www.freesound.org/people/WIM/sounds/18755/) by WIM [CC-BY]\n- [1-18757-A.ogg]: clip derived from frogs03.wav (http://www.freesound.org/people/WIM/sounds/18757/) by WIM [CC-BY]\n- [1-31836-A.ogg]: clip derived from 20070303.frogs.flac (http://www.freesound.org/people/dobroide/sounds/31836/) by dobroide [CC-BY]\n- [1-31836-B.ogg]: clip derived from 20070303.frogs.flac (http://www.freesound.org/people/dobroide/sounds/31836/) by dobroide [CC-BY]\n- [2-32515-A.ogg]: clip derived from 200703101950PacificChorusFrogsChorusAndTrainNR.flac (http://www.freesound.org/people/daveincamas/sounds/32515/) by daveincamas [CC-BY]\n- [2-32515-B.ogg]: clip derived from 200703101950PacificChorusFrogsChorusAndTrainNR.flac (http://www.freesound.org/people/daveincamas/sounds/32515/) by daveincamas [CC-BY]\n- [2-32515-C.ogg]: clip derived from 200703101950PacificChorusFrogsChorusAndTrainNR.flac (http://www.freesound.org/people/daveincamas/sounds/32515/) by daveincamas [CC-BY]\n- [2-32515-D.ogg]: clip derived from 200703101950PacificChorusFrogsChorusAndTrainNR.flac (http://www.freesound.org/people/daveincamas/sounds/32515/) by daveincamas [CC-BY]\n- [2-32834-A.ogg]: clip derived from 200703101950PacificChorusFrogsConfrontAirplane.wav (http://www.freesound.org/people/daveincamas/sounds/32834/) by daveincamas [CC-BY]\n- [2-52085-A.ogg]: clip derived from sherman_frogs_spring_2007.mp3 (http://www.freesound.org/people/aguasonic/sounds/52085/) by aguasonic [CC-BY]\n- [2-52085-B.ogg]: clip derived from sherman_frogs_spring_2007.mp3 (http://www.freesound.org/people/aguasonic/sounds/52085/) by aguasonic [CC-BY]\n- [2-52789-A.ogg]: clip derived from frogs.wav (http://www.freesound.org/people/alexbird/sounds/52789/) by alexbird [CC-BY]\n- [3-102908-A.ogg]: clip derived from frog in the tropical garden.wav (http://www.freesound.org/people/E330/sounds/102908/) by E330 [CC-BY]\n- [3-70962-A.ogg]: clip derived from FROGS_4824_st.wav (http://www.freesound.org/people/Sedi/sounds/70962/) by Sedi [CC-BY-NC]\n- [3-70962-B.ogg]: clip derived from FROGS_4824_st.wav (http://www.freesound.org/people/Sedi/sounds/70962/) by Sedi [CC-BY-NC]\n- [3-70962-C.ogg]: clip derived from FROGS_4824_st.wav (http://www.freesound.org/people/Sedi/sounds/70962/) by Sedi [CC-BY-NC]\n- [3-71964-A.ogg]: clip derived from FrogsFLP.wav (http://www.freesound.org/people/daveincamas/sounds/71964/) by daveincamas [CC-BY]\n- [3-71964-B.ogg]: clip derived from FrogsFLP.wav (http://www.freesound.org/people/daveincamas/sounds/71964/) by daveincamas [CC-BY]\n- [3-71964-C.ogg]: clip derived from FrogsFLP.wav (http://www.freesound.org/people/daveincamas/sounds/71964/) by daveincamas [CC-BY]\n- [3-83527-A.ogg]: clip derived from tree frog.wav (http://www.freesound.org/people/Tito%20Lahaye/sounds/83527/) by Tito Lahaye [CC-BY-NC]\n- [4-130584-A.ogg]: clip derived from Frogs Under Pier.wav (http://www.freesound.org/people/casemundy/sounds/130584/) by casemundy [CC-BY]\n- [4-154793-A.ogg]: clip derived from Heavy Frogs (http://www.freesound.org/people/unfa/sounds/154793/) by unfa [CC-BY]\n- [4-99193-A.ogg]: clip derived from swamp frogs.wav (http://www.freesound.org/people/Tomlija/sounds/99193/) by Tomlija [CC-BY]\n- [4-99193-B.ogg]: clip derived from swamp frogs.wav (http://www.freesound.org/people/Tomlija/sounds/99193/) by Tomlija [CC-BY]\n- [4-99644-A.ogg]: clip derived from Frogs.wav (http://www.freesound.org/people/juskiddink/sounds/99644/) by juskiddink [CC-BY]\n- [4-99644-B.ogg]: clip derived from Frogs.wav (http://www.freesound.org/people/juskiddink/sounds/99644/) by juskiddink [CC-BY]\n- [4-99644-C.ogg]: clip derived from Frogs.wav (http://www.freesound.org/people/juskiddink/sounds/99644/) by juskiddink [CC-BY]\n- [4-99644-D.ogg]: clip derived from Frogs.wav (http://www.freesound.org/people/juskiddink/sounds/99644/) by juskiddink [CC-BY]\n- [5-156026-A.ogg]: clip derived from Frog Concerto (http://www.freesound.org/people/Anton/sounds/156026/) by Anton [CC-BY-NC]\n- [5-156026-B.ogg]: clip derived from Frog Concerto (http://www.freesound.org/people/Anton/sounds/156026/) by Anton [CC-BY-NC]\n- [5-156026-C.ogg]: clip derived from Frog Concerto (http://www.freesound.org/people/Anton/sounds/156026/) by Anton [CC-BY-NC]\n- [5-156026-D.ogg]: clip derived from Frog Concerto (http://www.freesound.org/people/Anton/sounds/156026/) by Anton [CC-BY-NC]\n- [5-187201-A.ogg]: clip derived from file0306_pond_frog_bird_3.wav (http://www.freesound.org/people/maphill/sounds/187201/) by maphill [CC0]\n- [5-187201-B.ogg]: clip derived from file0306_pond_frog_bird_3.wav (http://www.freesound.org/people/maphill/sounds/187201/) by maphill [CC0]\n- [5-189795-A.ogg]: clip derived from Frogs Forever (http://www.freesound.org/people/magixmusic/sounds/189795/) by magixmusic [CC-BY-NC]\n- [5-237499-A.ogg]: clip derived from Edible Frog - Pelophylax kl esculentus140430_0191.wav (http://www.freesound.org/people/klankbeeld/sounds/237499/) by klankbeeld [CC-BY]\n\n106 - Cat:\n- [1-34094-A.ogg]: clip derived from 070422-cats-sample.wav (http://www.freesound.org/people/sazman/sounds/34094/) by sazman [CC-BY]\n- [1-34094-B.ogg]: clip derived from 070422-cats-sample.wav (http://www.freesound.org/people/sazman/sounds/34094/) by sazman [CC-BY]\n- [1-47819-A.ogg]: clip derived from cat_door.wav (http://www.freesound.org/people/YuriVoorhak/sounds/47819/) by YuriVoorhak [CC-Sampling+]\n- [1-47819-B.ogg]: clip derived from cat_door.wav (http://www.freesound.org/people/YuriVoorhak/sounds/47819/) by YuriVoorhak [CC-Sampling+]\n- [1-47819-C.ogg]: clip derived from cat_door.wav (http://www.freesound.org/people/YuriVoorhak/sounds/47819/) by YuriVoorhak [CC-Sampling+]\n- [1-56380-A.ogg]: clip derived from Cat.wav (http://www.freesound.org/people/oliver%20eye/sounds/56380/) by oliver eye [CC-Sampling+]\n- [1-56380-B.ogg]: clip derived from Cat.wav (http://www.freesound.org/people/oliver%20eye/sounds/56380/) by oliver eye [CC-Sampling+]\n- [1-79113-A.ogg]: clip derived from cat meow 01.wav (http://www.freesound.org/people/Kyster/sounds/79113/) by Kyster [CC-BY]\n- [2-110010-A.ogg]: clip derived from cat meow II (http://www.freesound.org/people/tuberatanka/sounds/110010/) by tuberatanka [CC0]\n- [2-110011-A.ogg]: clip derived from cat meow (http://www.freesound.org/people/tuberatanka/sounds/110011/) by tuberatanka [CC0]\n- [2-69131-A.ogg]: clip derived from cat.mp3 (http://www.freesound.org/people/milton./sounds/69131/) by milton. [CC-BY-NC]\n- [2-69131-B.ogg]: clip derived from cat.mp3 (http://www.freesound.org/people/milton./sounds/69131/) by milton. [CC-BY-NC]\n- [2-82274-A.ogg]: clip derived from garage cat 01.wav (http://www.freesound.org/people/Kyster/sounds/82274/) by Kyster [CC-BY]\n- [2-82274-B.ogg]: clip derived from garage cat 01.wav (http://www.freesound.org/people/Kyster/sounds/82274/) by Kyster [CC-BY]\n- [2-83934-A.ogg]: clip derived from CatMewing.wav (http://www.freesound.org/people/acclivity/sounds/83934/) by acclivity [CC-BY-NC]\n- [2-83934-B.ogg]: clip derived from CatMewing.wav (http://www.freesound.org/people/acclivity/sounds/83934/) by acclivity [CC-BY-NC]\n- [3-146964-A.ogg]: clip derived from catHowling1.wav (http://www.freesound.org/people/Zabuhailo/sounds/146964/) by Zabuhailo [CC0]\n- [3-146965-A.ogg]: clip derived from catHowling2.wav (http://www.freesound.org/people/Zabuhailo/sounds/146965/) by Zabuhailo [CC0]\n- [3-146972-A.ogg]: clip derived from catAttack.wav (http://www.freesound.org/people/Zabuhailo/sounds/146972/) by Zabuhailo [CC0]\n- [3-95694-A.ogg]: clip derived from 20100423.hungry.cats.01.wav (http://www.freesound.org/people/dobroide/sounds/95694/) by dobroide [CC-BY]\n- [3-95695-A.ogg]: clip derived from 20100423.hungry.cats.02.wav (http://www.freesound.org/people/dobroide/sounds/95695/) by dobroide [CC-BY]\n- [3-95695-B.ogg]: clip derived from 20100423.hungry.cats.02.wav (http://www.freesound.org/people/dobroide/sounds/95695/) by dobroide [CC-BY]\n- [3-95697-A.ogg]: clip derived from 20100423.hungry.cats.04.wav (http://www.freesound.org/people/dobroide/sounds/95697/) by dobroide [CC-BY]\n- [3-95698-A.ogg]: clip derived from 20100423.hungry.cats.05.wav (http://www.freesound.org/people/dobroide/sounds/95698/) by dobroide [CC-BY]\n- [4-120160-A.ogg]: clip derived from Cat Wants Food (http://www.freesound.org/people/Oneirophile/sounds/120160/) by Oneirophile [CC0]\n- [4-133047-A.ogg]: clip derived from MAD CAT !.wav (http://www.freesound.org/people/temawas/sounds/133047/) by temawas [CC0]\n- [4-133047-B.ogg]: clip derived from MAD CAT !.wav (http://www.freesound.org/people/temawas/sounds/133047/) by temawas [CC0]\n- [4-133047-C.ogg]: clip derived from MAD CAT !.wav (http://www.freesound.org/people/temawas/sounds/133047/) by temawas [CC0]\n- [4-149940-A.ogg]: clip derived from Cat wants something (http://www.freesound.org/people/Jeanine2012/sounds/149940/) by Jeanine2012 [CC0]\n- [4-149940-B.ogg]: clip derived from Cat wants something (http://www.freesound.org/people/Jeanine2012/sounds/149940/) by Jeanine2012 [CC0]\n- [4-161303-A.ogg]: clip derived from 03498 fake cat.wav (http://www.freesound.org/people/Robinhood76/sounds/161303/) by Robinhood76 [CC-BY-NC]\n- [4-161303-B.ogg]: clip derived from 03498 fake cat.wav (http://www.freesound.org/people/Robinhood76/sounds/161303/) by Robinhood76 [CC-BY-NC]\n- [5-169983-A.ogg]: clip derived from Hungry cats (http://www.freesound.org/people/missozzy/sounds/169983/) by missozzy [CC0]\n- [5-172299-A.ogg]: clip derived from Diddlebopp the Cat Meowing (http://www.freesound.org/people/jslarve/sounds/172299/) by jslarve [CC-BY]\n- [5-172639-A.ogg]: clip derived from Cat.mp3 (http://www.freesound.org/people/telesik/sounds/172639/) by telesik [CC-BY]\n- [5-177614-A.ogg]: clip derived from 02 Cat Miauing (http://www.freesound.org/people/aminut/sounds/177614/) by aminut [CC0]\n- [5-214759-A.ogg]: clip derived from Cat meowing x5 (http://www.freesound.org/people/peridactyloptrix/sounds/214759/) by peridactyloptrix [CC0]\n- [5-214759-B.ogg]: clip derived from Cat meowing x5 (http://www.freesound.org/people/peridactyloptrix/sounds/214759/) by peridactyloptrix [CC0]\n- [5-256452-A.ogg]: clip derived from Cat meow (http://www.freesound.org/people/philsapphire/sounds/256452/) by philsapphire [CC0]\n- [5-259169-A.ogg]: clip derived from CAT MEOW.wav (http://www.freesound.org/people/xtrgamr/sounds/259169/) by xtrgamr [CC-BY]\n\n107 - Hen:\n- [1-18074-A.ogg]: clip derived from BerberFarmyard2.wav (http://www.freesound.org/people/acclivity/sounds/18074/) by acclivity [CC-BY-NC]\n- [1-18074-B.ogg]: clip derived from BerberFarmyard2.wav (http://www.freesound.org/people/acclivity/sounds/18074/) by acclivity [CC-BY-NC]\n- [1-31251-A.ogg]: clip derived from 20070218.el.lobo.hens.wav (http://www.freesound.org/people/dobroide/sounds/31251/) by dobroide [CC-BY]\n- [1-31251-B.ogg]: clip derived from 20070218.el.lobo.hens.wav (http://www.freesound.org/people/dobroide/sounds/31251/) by dobroide [CC-BY]\n- [1-5996-A.ogg]: clip derived from hens.00.wav (http://www.freesound.org/people/dobroide/sounds/5996/) by dobroide [CC-BY]\n- [1-71030-A.ogg]: clip derived from 01011 cote chickencoop distant walk.wav (http://www.freesound.org/people/Robinhood76/sounds/71030/) by Robinhood76 [CC-BY-NC]\n- [1-72229-A.ogg]: clip derived from 20090501.henhouse.flac (http://www.freesound.org/people/dobroide/sounds/72229/) by dobroide [CC-BY]\n- [1-72229-B.ogg]: clip derived from 20090501.henhouse.flac (http://www.freesound.org/people/dobroide/sounds/72229/) by dobroide [CC-BY]\n- [2-120586-A.ogg]: clip derived from 20110503_clucking.04.wav (http://www.freesound.org/people/dobroide/sounds/120586/) by dobroide [CC-BY]\n- [2-120587-A.ogg]: clip derived from 20110503_clucking.05.wav (http://www.freesound.org/people/dobroide/sounds/120587/) by dobroide [CC-BY]\n- [2-127109-A.ogg]: clip derived from Angry Bird.wav (http://www.freesound.org/people/Grigore/sounds/127109/) by Grigore [CC-BY]\n- [2-134049-A.ogg]: clip derived from cock and hen.WAV (http://www.freesound.org/people/Setuniman/sounds/134049/) by Setuniman [CC-BY]\n- [2-157488-A.ogg]: clip derived from Chicken coop (http://www.freesound.org/people/caquet/sounds/157488/) by caquet [CC-BY]\n- [2-72268-A.ogg]: clip derived from 20090503.hens.wav (http://www.freesound.org/people/dobroide/sounds/72268/) by dobroide [CC-BY]\n- [2-75726-A.ogg]: clip derived from 01130 chicken hen 2.wav (http://www.freesound.org/people/Robinhood76/sounds/75726/) by Robinhood76 [CC-BY-NC]\n- [2-76868-A.ogg]: clip derived from 01170 village hen 4.wav (http://www.freesound.org/people/Robinhood76/sounds/76868/) by Robinhood76 [CC-BY-NC]\n- [3-164216-A.ogg]: clip derived from clucking_chicken.wav (http://www.freesound.org/people/pillonoise/sounds/164216/) by pillonoise [CC-BY]\n- [3-164216-B.ogg]: clip derived from clucking_chicken.wav (http://www.freesound.org/people/pillonoise/sounds/164216/) by pillonoise [CC-BY]\n- [3-164216-C.ogg]: clip derived from clucking_chicken.wav (http://www.freesound.org/people/pillonoise/sounds/164216/) by pillonoise [CC-BY]\n- [3-171281-A.ogg]: clip derived from hen and rooster (http://www.freesound.org/people/signorvaso/sounds/171281/) by signorvaso [CC0]\n- [3-178096-A.ogg]: clip derived from excited chickens (http://www.freesound.org/people/Setuniman/sounds/178096/) by Setuniman [CC-BY]\n- [3-187549-A.ogg]: clip derived from Chickens in the hen house, in a farm in the south of Patagonia (http://www.freesound.org/people/felix.blume/sounds/187549/) by felix.blume [CC0]\n- [3-187549-B.ogg]: clip derived from Chickens in the hen house, in a farm in the south of Patagonia (http://www.freesound.org/people/felix.blume/sounds/187549/) by felix.blume [CC0]\n- [3-188390-A.ogg]: clip derived from hen.mp3 (http://www.freesound.org/people/luis_audp/sounds/188390/) by luis_audp [CC-BY]\n- [4-197103-A.ogg]: clip derived from Chickens & Hens (http://www.freesound.org/people/musicradiocreative/sounds/197103/) by musicradiocreative [CC0]\n- [4-200330-A.ogg]: clip derived from chicken_chatter.aif (http://www.freesound.org/people/jsbarrett/sounds/200330/) by jsbarrett [CC0]\n- [4-200330-B.ogg]: clip derived from chicken_chatter.aif (http://www.freesound.org/people/jsbarrett/sounds/200330/) by jsbarrett [CC0]\n- [4-204830-A.ogg]: clip derived from H\\xf6nor.wav (http://www.freesound.org/people/fialisa/sounds/204830/) by fialisa [CC-BY-NC]\n- [4-232495-A.ogg]: clip derived from Polish hen cackling.wav (http://www.freesound.org/people/eksaa/sounds/232495/) by eksaa [CC0]\n- [4-234879-A.ogg]: clip derived from chickensMS.wav (http://www.freesound.org/people/leosalom/sounds/234879/) by leosalom [CC0]\n- [4-244318-A.ogg]: clip derived from Farm house - Chicken 4.wav (http://www.freesound.org/people/E330/sounds/244318/) by E330 [CC-BY]\n- [4-253649-A.ogg]: clip derived from Two Chickens "Talking" (http://www.freesound.org/people/SquaredGlasses/sounds/253649/) by SquaredGlasses [CC-BY]\n- [5-244315-A.ogg]: clip derived from Farm house - Chicken 1.wav (http://www.freesound.org/people/E330/sounds/244315/) by E330 [CC-BY]\n- [5-244315-B.ogg]: clip derived from Farm house - Chicken 1.wav (http://www.freesound.org/people/E330/sounds/244315/) by E330 [CC-BY]\n- [5-244315-C.ogg]: clip derived from Farm house - Chicken 1.wav (http://www.freesound.org/people/E330/sounds/244315/) by E330 [CC-BY]\n- [5-248341-A.ogg]: clip derived from Pieni kanala / Small henhouse, hens cackling (http://www.freesound.org/people/YleArkisto/sounds/248341/) by YleArkisto [CC-BY]\n- [5-248341-B.ogg]: clip derived from Pieni kanala / Small henhouse, hens cackling (http://www.freesound.org/people/YleArkisto/sounds/248341/) by YleArkisto [CC-BY]\n- [5-248341-C.ogg]: clip derived from Pieni kanala / Small henhouse, hens cackling (http://www.freesound.org/people/YleArkisto/sounds/248341/) by YleArkisto [CC-BY]\n- [5-263831-A.ogg]: clip derived from 12092014_port2000_mostki_hens.mp3 (http://www.freesound.org/people/miastodzwiekow/sounds/263831/) by miastodzwiekow [CC-BY]\n- [5-263831-B.ogg]: clip derived from 12092014_port2000_mostki_hens.mp3 (http://www.freesound.org/people/miastodzwiekow/sounds/263831/) by miastodzwiekow [CC-BY]\n\n108 - Insects (flying):\n- [1-17585-A.ogg]: clip derived from 20060328.fat.fly.flac (http://www.freesound.org/people/dobroide/sounds/17585/) by dobroide [CC-BY]\n- [1-19501-A.ogg]: clip derived from 20060524.fly.flac (http://www.freesound.org/people/dobroide/sounds/19501/) by dobroide [CC-BY]\n- [1-46938-A.ogg]: clip derived from flys-I.wav (http://www.freesound.org/people/galeku/sounds/46938/) by galeku [CC0]\n- [1-46938-B.ogg]: clip derived from flys-I.wav (http://www.freesound.org/people/galeku/sounds/46938/) by galeku [CC0]\n- [1-73585-A.ogg]: clip derived from Bees in treetop.wav (http://www.freesound.org/people/J.Zazvurek/sounds/73585/) by J.Zazvurek [CC-BY]\n- [1-75189-A.ogg]: clip derived from BunchOfFlies.wav (http://www.freesound.org/people/HerbertBoland/sounds/75189/) by HerbertBoland [CC-BY]\n- [1-7973-A.ogg]: clip derived from fly.00.wav (http://www.freesound.org/people/dobroide/sounds/7973/) by dobroide [CC-BY]\n- [1-80785-A.ogg]: clip derived from flyrawflyby2.wav (http://www.freesound.org/people/NoiseCollector/sounds/80785/) by NoiseCollector [CC-BY]\n- [2-160128-A.ogg]: clip derived from flybuzzing_fafe_ambient.wav (http://www.freesound.org/people/dio_333/sounds/160128/) by dio_333 [CC-BY]\n- [2-81970-A.ogg]: clip derived from Fly.wav (http://www.freesound.org/people/Benboncan/sounds/81970/) by Benboncan [CC-BY]\n- [2-81970-B.ogg]: clip derived from Fly.wav (http://www.freesound.org/people/Benboncan/sounds/81970/) by Benboncan [CC-BY]\n- [2-81970-C.ogg]: clip derived from Fly.wav (http://www.freesound.org/people/Benboncan/sounds/81970/) by Benboncan [CC-BY]\n- [2-82077-A.ogg]: clip derived from Fly 1.wav (http://www.freesound.org/people/Benboncan/sounds/82077/) by Benboncan [CC-BY]\n- [2-99955-A.ogg]: clip derived from Bumble bee long.wav (http://www.freesound.org/people/CGEffex/sounds/99955/) by CGEffex [CC-BY]\n- [2-99955-B.ogg]: clip derived from Bumble bee long.wav (http://www.freesound.org/people/CGEffex/sounds/99955/) by CGEffex [CC-BY]\n- [2-99955-C.ogg]: clip derived from Bumble bee long.wav (http://www.freesound.org/people/CGEffex/sounds/99955/) by CGEffex [CC-BY]\n- [3-104761-A.ogg]: clip derived from momoscas 1.flac (http://www.freesound.org/people/galeku/sounds/104761/) by galeku [CC0]\n- [3-104761-B.ogg]: clip derived from momoscas 1.flac (http://www.freesound.org/people/galeku/sounds/104761/) by galeku [CC0]\n- [3-105235-A.ogg]: clip derived from 20100923.buzz.00.wav (http://www.freesound.org/people/dobroide/sounds/105235/) by dobroide [CC-BY]\n- [3-105236-A.ogg]: clip derived from 20100923.buzz.01.wav (http://www.freesound.org/people/dobroide/sounds/105236/) by dobroide [CC-BY]\n- [3-110913-A.ogg]: clip derived from 20101206.02.flies.wav (http://www.freesound.org/people/dobroide/sounds/110913/) by dobroide [CC-BY]\n- [3-110913-B.ogg]: clip derived from 20101206.02.flies.wav (http://www.freesound.org/people/dobroide/sounds/110913/) by dobroide [CC-BY]\n- [3-110913-C.ogg]: clip derived from 20101206.02.flies.wav (http://www.freesound.org/people/dobroide/sounds/110913/) by dobroide [CC-BY]\n- [3-110913-D.ogg]: clip derived from 20101206.02.flies.wav (http://www.freesound.org/people/dobroide/sounds/110913/) by dobroide [CC-BY]\n- [4-130891-A.ogg]: clip derived from Fly LOOP-01.wav (http://www.freesound.org/people/toyoto/sounds/130891/) by toyoto [CC-BY]\n- [4-133895-A.ogg]: clip derived from FLY_STUCK_BUZZING_002.wav (http://www.freesound.org/people/JoelAudio/sounds/133895/) by JoelAudio [CC-BY]\n- [4-143118-A.ogg]: clip derived from Fly_versus_Lamp.wav (http://www.freesound.org/people/1pjladd2/sounds/143118/) by 1pjladd2 [CC0]\n- [4-143118-B.ogg]: clip derived from Fly_versus_Lamp.wav (http://www.freesound.org/people/1pjladd2/sounds/143118/) by 1pjladd2 [CC0]\n- [4-182314-A.ogg]: clip derived from A fly, a plane, a rockslide. (http://www.freesound.org/people/stomachache/sounds/182314/) by stomachache [CC-BY]\n- [4-182314-B.ogg]: clip derived from A fly, a plane, a rockslide. (http://www.freesound.org/people/stomachache/sounds/182314/) by stomachache [CC-BY]\n- [4-191246-A.ogg]: clip derived from 110703_05 blue bottle fly.WAV (http://www.freesound.org/people/toiletrolltube/sounds/191246/) by toiletrolltube [CC0]\n- [4-191246-B.ogg]: clip derived from 110703_05 blue bottle fly.WAV (http://www.freesound.org/people/toiletrolltube/sounds/191246/) by toiletrolltube [CC0]\n- [5-194932-A.ogg]: clip derived from bee fly .mp3 (http://www.freesound.org/people/soundmary/sounds/194932/) by soundmary [CC-BY]\n- [5-195517-A.ogg]: clip derived from Foley Small Fly.WAV (http://www.freesound.org/people/jamesrodavidson/sounds/195517/) by jamesrodavidson [CC0]\n- [5-195518-A.ogg]: clip derived from Foley Bluebottle Fly.WAV (http://www.freesound.org/people/jamesrodavidson/sounds/195518/) by jamesrodavidson [CC0]\n- [5-197446-A.ogg]: clip derived from A young calf is born during the night, he suck his mother pie with a lot of fly around. (http://www.freesound.org/people/felix.blume/sounds/197446/) by felix.blume [CC0]\n- [5-198278-A.ogg]: clip derived from Flies (http://www.freesound.org/people/miklovan/sounds/198278/) by miklovan [CC0]\n- [5-198278-B.ogg]: clip derived from Flies (http://www.freesound.org/people/miklovan/sounds/198278/) by miklovan [CC0]\n- [5-198278-C.ogg]: clip derived from Flies (http://www.freesound.org/people/miklovan/sounds/198278/) by miklovan [CC0]\n- [5-233787-A.ogg]: clip derived from BUMBLEBEE 16.wav (http://www.freesound.org/people/delphidebrain/sounds/233787/) by delphidebrain [CC-BY]\n\n109 - Sheep:\n- [1-121951-A.ogg]: clip derived from sheep pasture.wav (http://www.freesound.org/people/odilonmarcenaro/sounds/121951/) by odilonmarcenaro [CC0]\n- [1-196660-A.ogg]: clip derived from flock of sheep - quad bike background .mp3 (http://www.freesound.org/people/soundmary/sounds/196660/) by soundmary [CC-BY]\n- [1-196660-B.ogg]: clip derived from flock of sheep - quad bike background .mp3 (http://www.freesound.org/people/soundmary/sounds/196660/) by soundmary [CC-BY]\n- [1-49409-A.ogg]: clip derived from 20080302.sheeps.close.watermarked.wav (http://www.freesound.org/people/dobroide/sounds/49409/) by dobroide [CC-BY]\n- [1-49409-B.ogg]: clip derived from 20080302.sheeps.close.watermarked.wav (http://www.freesound.org/people/dobroide/sounds/49409/) by dobroide [CC-BY]\n- [1-57795-A.ogg]: clip derived from 1202_sheep_and_goats.wav (http://www.freesound.org/people/reinsamba/sounds/57795/) by reinsamba [CC-BY]\n- [1-75190-A.ogg]: clip derived from Sheeheep.wav (http://www.freesound.org/people/HerbertBoland/sounds/75190/) by HerbertBoland [CC-BY]\n- [1-88574-A.ogg]: clip derived from Sheep.WAV (http://www.freesound.org/people/inchadney/sounds/88574/) by inchadney [CC-BY]\n- [2-110614-A.ogg]: clip derived from animals_sheep_bleating_many_001.wav (http://www.freesound.org/people/soundscalpel.com/sounds/110614/) by soundscalpel.com [CC-BY]\n- [2-110614-B.ogg]: clip derived from animals_sheep_bleating_many_001.wav (http://www.freesound.org/people/soundscalpel.com/sounds/110614/) by soundscalpel.com [CC-BY]\n- [2-119161-A.ogg]: clip derived from 110419-000_sheep_bleating.wav (http://www.freesound.org/people/reinsamba/sounds/119161/) by reinsamba [CC-BY]\n- [2-119161-B.ogg]: clip derived from 110419-000_sheep_bleating.wav (http://www.freesound.org/people/reinsamba/sounds/119161/) by reinsamba [CC-BY]\n- [2-119161-C.ogg]: clip derived from 110419-000_sheep_bleating.wav (http://www.freesound.org/people/reinsamba/sounds/119161/) by reinsamba [CC-BY]\n- [2-182508-A.ogg]: clip derived from Sheep Herd (http://www.freesound.org/people/swiftoid/sounds/182508/) by swiftoid [CC0]\n- [2-182508-B.ogg]: clip derived from Sheep Herd (http://www.freesound.org/people/swiftoid/sounds/182508/) by swiftoid [CC0]\n- [2-196688-A.ogg]: clip derived from sheep sheering.mp3 (http://www.freesound.org/people/soundmary/sounds/196688/) by soundmary [CC-BY]\n- [3-140199-A.ogg]: clip derived from A herd of sheep are bleating, some dogs are barking and the horses are neighing... in a field in patagonia (Tierra del Fuego, Argentina). (http://www.freesound.org/people/felix.blume/sounds/140199/) by felix.blume [CC0]\n- [3-140199-B.ogg]: clip derived from A herd of sheep are bleating, some dogs are barking and the horses are neighing... in a field in patagonia (Tierra del Fuego, Argentina). (http://www.freesound.org/people/felix.blume/sounds/140199/) by felix.blume [CC0]\n- [3-140199-C.ogg]: clip derived from A herd of sheep are bleating, some dogs are barking and the horses are neighing... in a field in patagonia (Tierra del Fuego, Argentina). (http://www.freesound.org/people/felix.blume/sounds/140199/) by felix.blume [CC0]\n- [3-140199-D.ogg]: clip derived from A herd of sheep are bleating, some dogs are barking and the horses are neighing... in a field in patagonia (Tierra del Fuego, Argentina). (http://www.freesound.org/people/felix.blume/sounds/140199/) by felix.blume [CC0]\n- [3-197408-A.ogg]: clip derived from  A herd of sheep are bleating, some dogs are barking and the horses are neighing... in a field in patagonia (Tierra del Fuego, Argentina) (http://www.freesound.org/people/felix.blume/sounds/197408/) by felix.blume [CC0]\n- [3-197408-B.ogg]: clip derived from  A herd of sheep are bleating, some dogs are barking and the horses are neighing... in a field in patagonia (Tierra del Fuego, Argentina) (http://www.freesound.org/people/felix.blume/sounds/197408/) by felix.blume [CC0]\n- [3-197408-C.ogg]: clip derived from  A herd of sheep are bleating, some dogs are barking and the horses are neighing... in a field in patagonia (Tierra del Fuego, Argentina) (http://www.freesound.org/people/felix.blume/sounds/197408/) by felix.blume [CC0]\n- [3-20861-A.ogg]: clip derived from zombie_sheep1.wav (http://www.freesound.org/people/plagasul/sounds/20861/) by plagasul [CC-BY]\n- [4-188703-A.ogg]: clip derived from Sheep.wav (http://www.freesound.org/people/LolitaPerdurabo/sounds/188703/) by LolitaPerdurabo [CC-BY]\n- [4-188703-B.ogg]: clip derived from Sheep.wav (http://www.freesound.org/people/LolitaPerdurabo/sounds/188703/) by LolitaPerdurabo [CC-BY]\n- [4-188703-C.ogg]: clip derived from Sheep.wav (http://www.freesound.org/people/LolitaPerdurabo/sounds/188703/) by LolitaPerdurabo [CC-BY]\n- [4-188703-D.ogg]: clip derived from Sheep.wav (http://www.freesound.org/people/LolitaPerdurabo/sounds/188703/) by LolitaPerdurabo [CC-BY]\n- [4-196671-A.ogg]: clip derived from sheep in field.mp3 (http://www.freesound.org/people/soundmary/sounds/196671/) by soundmary [CC-BY]\n- [4-196671-B.ogg]: clip derived from sheep in field.mp3 (http://www.freesound.org/people/soundmary/sounds/196671/) by soundmary [CC-BY]\n- [4-196672-A.ogg]: clip derived from sheep shearing up close.mp3 (http://www.freesound.org/people/soundmary/sounds/196672/) by soundmary [CC-BY]\n- [4-250864-A.ogg]: clip derived from Lampaat m\\xe4\\xe4kiv\\xe4t, katras / A flock of sheep bleating on the pasture  (http://www.freesound.org/people/YleArkisto/sounds/250864/) by YleArkisto [CC-BY]\n- [5-198891-A.ogg]: clip derived from Barn Sounds (http://www.freesound.org/people/bone666138/sounds/198891/) by bone666138 [CC-BY]\n- [5-198891-B.ogg]: clip derived from Barn Sounds (http://www.freesound.org/people/bone666138/sounds/198891/) by bone666138 [CC-BY]\n- [5-198891-C.ogg]: clip derived from Barn Sounds (http://www.freesound.org/people/bone666138/sounds/198891/) by bone666138 [CC-BY]\n- [5-198891-D.ogg]: clip derived from Barn Sounds (http://www.freesound.org/people/bone666138/sounds/198891/) by bone666138 [CC-BY]\n- [5-200329-A.ogg]: clip derived from Sheep_in_field.aiff (http://www.freesound.org/people/zwier/sounds/200329/) by zwier [CC0]\n- [5-200329-B.ogg]: clip derived from Sheep_in_field.aiff (http://www.freesound.org/people/zwier/sounds/200329/) by zwier [CC0]\n- [5-200329-C.ogg]: clip derived from Sheep_in_field.aiff (http://www.freesound.org/people/zwier/sounds/200329/) by zwier [CC0]\n- [5-61635-A.ogg]: clip derived from Schafe.wav (http://www.freesound.org/people/Andune/sounds/61635/) by Andune [CC-BY]\n\n110 - Crow:\n- [1-103298-A.ogg]: clip derived from br_Woodsy_Ambiance2.mp3 (http://www.freesound.org/people/Flemur/sounds/103298/) by Flemur [CC-Sampling+]\n- [1-39835-A.ogg]: clip derived from crows.flac (http://www.freesound.org/people/aesqe/sounds/39835/) by aesqe [CC-BY]\n- [1-39835-B.ogg]: clip derived from crows.flac (http://www.freesound.org/people/aesqe/sounds/39835/) by aesqe [CC-BY]\n- [1-56233-A.ogg]: clip derived from Crows_01.wav (http://www.freesound.org/people/Q.K./sounds/56233/) by Q.K. [CC0]\n- [1-56234-A.ogg]: clip derived from Crows_02.wav (http://www.freesound.org/people/Q.K./sounds/56234/) by Q.K. [CC0]\n- [1-75162-A.ogg]: clip derived from crow.wav (http://www.freesound.org/people/nigelcoop/sounds/75162/) by nigelcoop [CC0]\n- [1-96950-A.ogg]: clip derived from Carrion Crows.wav (http://www.freesound.org/people/Benboncan/sounds/96950/) by Benboncan [CC-BY]\n- [1-96950-B.ogg]: clip derived from Carrion Crows.wav (http://www.freesound.org/people/Benboncan/sounds/96950/) by Benboncan [CC-BY]\n- [2-108763-A.ogg]: clip derived from birds_morning02.wav (http://www.freesound.org/people/soundbytez/sounds/108763/) by soundbytez [CC-BY]\n- [2-108766-A.ogg]: clip derived from crows.wav (http://www.freesound.org/people/soundbytez/sounds/108766/) by soundbytez [CC-BY]\n- [2-108767-A.ogg]: clip derived from crows_and_magpies.wav (http://www.freesound.org/people/soundbytez/sounds/108767/) by soundbytez [CC-BY]\n- [2-108767-B.ogg]: clip derived from crows_and_magpies.wav (http://www.freesound.org/people/soundbytez/sounds/108767/) by soundbytez [CC-BY]\n- [2-108767-C.ogg]: clip derived from crows_and_magpies.wav (http://www.freesound.org/people/soundbytez/sounds/108767/) by soundbytez [CC-BY]\n- [2-109231-A.ogg]: clip derived from crows and magpie.WAV (http://www.freesound.org/people/inchadney/sounds/109231/) by inchadney [CC-BY]\n- [2-109231-B.ogg]: clip derived from crows and magpie.WAV (http://www.freesound.org/people/inchadney/sounds/109231/) by inchadney [CC-BY]\n- [2-109231-C.ogg]: clip derived from crows and magpie.WAV (http://www.freesound.org/people/inchadney/sounds/109231/) by inchadney [CC-BY]\n- [3-112397-A.ogg]: clip derived from crows.WAV (http://www.freesound.org/people/inchadney/sounds/112397/) by inchadney [CC-BY]\n- [3-117293-A.ogg]: clip derived from Birds-Crow & Song Birds.wav (http://www.freesound.org/people/Bansemer/sounds/117293/) by Bansemer [CC0]\n- [3-121348-A.ogg]: clip derived from Crow.WAV (http://www.freesound.org/people/inchadney/sounds/121348/) by inchadney [CC-BY]\n- [3-124925-A.ogg]: clip derived from crow.WAV (http://www.freesound.org/people/inchadney/sounds/124925/) by inchadney [CC-BY]\n- [3-127890-A.ogg]: clip derived from crows.wav (http://www.freesound.org/people/inchadney/sounds/127890/) by inchadney [CC-BY]\n- [3-127890-B.ogg]: clip derived from crows.wav (http://www.freesound.org/people/inchadney/sounds/127890/) by inchadney [CC-BY]\n- [3-127890-C.ogg]: clip derived from crows.wav (http://www.freesound.org/people/inchadney/sounds/127890/) by inchadney [CC-BY]\n- [3-129264-A.ogg]: clip derived from Black Bird/Crow (http://www.freesound.org/people/stair/sounds/129264/) by stair [CC0]\n- [4-136381-A.ogg]: clip derived from crow.wav (http://www.freesound.org/people/inchadney/sounds/136381/) by inchadney [CC-BY]\n- [4-138344-A.ogg]: clip derived from crow.wav (http://www.freesound.org/people/UncleSigmund/sounds/138344/) by UncleSigmund [CC0]\n- [4-145081-A.ogg]: clip derived from Crows Calling (http://www.freesound.org/people/GB01/sounds/145081/) by GB01 [CC-BY]\n- [4-156227-A.ogg]: clip derived from augarten_crows.wav (http://www.freesound.org/people/akame/sounds/156227/) by akame [CC0]\n- [4-159426-A.ogg]: clip derived from Crow.WAV (http://www.freesound.org/people/inchadney/sounds/159426/) by inchadney [CC-BY]\n- [4-173865-A.ogg]: clip derived from ANML_M_Ext_Crows_Flock_Angry2.wav (http://www.freesound.org/people/conleec/sounds/173865/) by conleec [CC0]\n- [4-173865-B.ogg]: clip derived from ANML_M_Ext_Crows_Flock_Angry2.wav (http://www.freesound.org/people/conleec/sounds/173865/) by conleec [CC0]\n- [4-188287-A.ogg]: clip derived from crow 1 (http://www.freesound.org/people/pepv/sounds/188287/) by pepv [CC0]\n- [5-213836-A.ogg]: clip derived from crows.WAV (http://www.freesound.org/people/inchadney/sounds/213836/) by inchadney [CC-BY-NC]\n- [5-213836-B.ogg]: clip derived from crows.WAV (http://www.freesound.org/people/inchadney/sounds/213836/) by inchadney [CC-BY-NC]\n- [5-213836-C.ogg]: clip derived from crows.WAV (http://www.freesound.org/people/inchadney/sounds/213836/) by inchadney [CC-BY-NC]\n- [5-213836-D.ogg]: clip derived from crows.WAV (http://www.freesound.org/people/inchadney/sounds/213836/) by inchadney [CC-BY-NC]\n- [5-214869-A.ogg]: clip derived from A Murder of Crows (http://www.freesound.org/people/Frankie01234/sounds/214869/) by Frankie01234 [CC0]\n- [5-222041-A.ogg]: clip derived from Crows and Rooks 3 (http://www.freesound.org/people/Jagadamba/sounds/222041/) by Jagadamba [CC-BY]\n- [5-242711-A.ogg]: clip derived from Cuckoo and crow (noisy).wav (http://www.freesound.org/people/Martineerok/sounds/242711/) by Martineerok [CC-BY]\n- [5-261325-A.ogg]: clip derived from Crows (http://www.freesound.org/people/VlatkoBlazek/sounds/261325/) by VlatkoBlazek [CC-BY]\n\n201 - Rain:\n- [1-17367-A.ogg]: clip derived from unlocki door to mod rain.wav (http://www.freesound.org/people/cognito%20perceptu/sounds/17367/) by cognito perceptu [CC0]\n- [1-21189-A.ogg]: clip derived from Louisiana Rain 1.wav (http://www.freesound.org/people/uberhuberman/sounds/21189/) by uberhuberman [CC0]\n- [1-26222-A.ogg]: clip derived from medium heavey rain.wav (http://www.freesound.org/people/Walter_Odington/sounds/26222/) by Walter_Odington [CC0]\n- [1-29561-A.ogg]: clip derived from rain.aif (http://www.freesound.org/people/palem/sounds/29561/) by palem [CC0]\n- [1-50060-A.ogg]: clip derived from rain_session_27sec_2006.wav (http://www.freesound.org/people/vibe_crc/sounds/50060/) by vibe_crc [CC0]\n- [1-54958-A.ogg]: clip derived from rain.ogg (http://www.freesound.org/people/thammo/sounds/54958/) by thammo [CC0]\n- [1-56311-A.ogg]: clip derived from Rain_06.wav (http://www.freesound.org/people/Q.K./sounds/56311/) by Q.K. [CC0]\n- [1-63871-A.ogg]: clip derived from rain.mp3 (http://www.freesound.org/people/liliangorini/sounds/63871/) by liliangorini [CC0]\n- [2-101676-A.ogg]: clip derived from rain, sm57.wav (http://www.freesound.org/people/nialldoran/sounds/101676/) by nialldoran [CC0]\n- [2-117625-A.ogg]: clip derived from Tent - int - rain on canvas.wav (http://www.freesound.org/people/soundmary/sounds/117625/) by soundmary [CC0]\n- [2-72970-A.ogg]: clip derived from RAIN indoors.ogg (http://www.freesound.org/people/bsyapril88/sounds/72970/) by bsyapril88 [CC0]\n- [2-73027-A.ogg]: clip derived from rain_on_greenhouse.mp3 (http://www.freesound.org/people/j1987/sounds/73027/) by j1987 [CC0]\n- [2-73260-A.ogg]: clip derived from Rain on car roof.wav (http://www.freesound.org/people/jakobhandersen/sounds/73260/) by jakobhandersen [CC0]\n- [2-81731-A.ogg]: clip derived from Silencyo_RAIN_NIGHT_Village_Drips_Courtyard.aif (http://www.freesound.org/people/silencyo/sounds/81731/) by silencyo [CC0]\n- [2-82367-A.ogg]: clip derived from Rain on Plastic Roof.WAV (http://www.freesound.org/people/gynation/sounds/82367/) by gynation [CC0]\n- [2-87781-A.ogg]: clip derived from rain.wav (http://www.freesound.org/people/sagetyrtle/sounds/87781/) by sagetyrtle [CC0]\n- [3-132852-A.ogg]: clip derived from rain from a car.WAV (http://www.freesound.org/people/inchadney/sounds/132852/) by inchadney [CC0]\n- [3-140774-A.ogg]: clip derived from Rain over a car (http://www.freesound.org/people/Sam80/sounds/140774/) by Sam80 [CC0]\n- [3-142005-A.ogg]: clip derived from Heavy rain - some people 1 (http://www.freesound.org/people/jop9798/sounds/142005/) by jop9798 [CC0]\n- [3-142006-A.ogg]: clip derived from Heavy rain - some people 2 (http://www.freesound.org/people/jop9798/sounds/142006/) by jop9798 [CC0]\n- [3-143929-A.ogg]: clip derived from Driving in rain.wav (http://www.freesound.org/people/Polmantz/sounds/143929/) by Polmantz [CC0]\n- [3-157149-A.ogg]: clip derived from raininwoods.aiff (http://www.freesound.org/people/klangfabrik/sounds/157149/) by klangfabrik [CC0]\n- [3-157487-A.ogg]: clip derived from rain_near_smooth.aif (http://www.freesound.org/people/loopbasedmusic/sounds/157487/) by loopbasedmusic [CC0]\n- [3-157615-A.ogg]: clip derived from rain recording.mp3 (http://www.freesound.org/people/Keygenerator/sounds/157615/) by Keygenerator [CC0]\n- [4-160999-A.ogg]: clip derived from rain.MP3 (http://www.freesound.org/people/Mafon2/sounds/160999/) by Mafon2 [CC0]\n- [4-161127-A.ogg]: clip derived from Heavy rain.wav (http://www.freesound.org/people/reelworldstudio/sounds/161127/) by reelworldstudio [CC0]\n- [4-163264-A.ogg]: clip derived from Light rain (http://www.freesound.org/people/Zepp2010/sounds/163264/) by Zepp2010 [CC0]\n- [4-164206-A.ogg]: clip derived from Rain very heavy with thunder.wav (http://www.freesound.org/people/Danjocross/sounds/164206/) by Danjocross [CC0]\n- [4-166661-A.ogg]: clip derived from pheiland (http://www.freesound.org/people/pheiland/sounds/166661/) by pheiland [CC0]\n- [4-177250-A.ogg]: clip derived from Rain and Storm (http://www.freesound.org/people/josephsardin/sounds/177250/) by josephsardin [CC0]\n- [4-180380-A.ogg]: clip derived from rain.wav (http://www.freesound.org/people/aumguy/sounds/180380/) by aumguy [CC0]\n- [4-181286-A.ogg]: clip derived from Wind and Sprinkling Rain (http://www.freesound.org/people/asbestos%20bill/sounds/181286/) by asbestos bill [CC0]\n- [5-181766-A.ogg]: clip derived from rain.aiff (http://www.freesound.org/people/laura222/sounds/181766/) by laura222 [CC0]\n- [5-188655-A.ogg]: clip derived from 1 Minute Rain (Medium) (http://www.freesound.org/people/yzhamua/sounds/188655/) by yzhamua [CC0]\n- [5-193339-A.ogg]: clip derived from rain.mp3 (http://www.freesound.org/people/soundman9826/sounds/193339/) by soundman9826 [CC0]\n- [5-194892-A.ogg]: clip derived from lo-fi-rain-and-thunder.wav (http://www.freesound.org/people/arseniiv/sounds/194892/) by arseniiv [CC0]\n- [5-195710-A.ogg]: clip derived from Ambient_Summer Rain_Storm_Urban_Dogs_People_Water Drops.wav (http://www.freesound.org/people/janstumpa/sounds/195710/) by janstumpa [CC0]\n- [5-198321-A.ogg]: clip derived from rain.wav (http://www.freesound.org/people/schafferdavid/sounds/198321/) by schafferdavid [CC0]\n- [5-202898-A.ogg]: clip derived from EXT_RainInPuddle.wav (http://www.freesound.org/people/jongrubbs/sounds/202898/) by jongrubbs [CC0]\n- [5-203739-A.ogg]: clip derived from rain with occasional thunder.wav (http://www.freesound.org/people/MrBoing/sounds/203739/) by MrBoing [CC0]\n\n202 - Sea waves:\n- [1-28135-A.ogg]: clip derived from Branding_kort.wav (http://www.freesound.org/people/HerbertBoland/sounds/28135/) by HerbertBoland [CC-BY]\n- [1-28135-B.ogg]: clip derived from Branding_kort.wav (http://www.freesound.org/people/HerbertBoland/sounds/28135/) by HerbertBoland [CC-BY]\n- [1-39901-A.ogg]: clip derived from sea_water_passing_through_pier_hole_01.flac (http://www.freesound.org/people/aesqe/sounds/39901/) by aesqe [CC-BY]\n- [1-39901-B.ogg]: clip derived from sea_water_passing_through_pier_hole_01.flac (http://www.freesound.org/people/aesqe/sounds/39901/) by aesqe [CC-BY]\n- [1-43760-A.ogg]: clip derived from Gentle Sea On Flat Beach.wav (http://www.freesound.org/people/digifishmusic/sounds/43760/) by digifishmusic [CC-BY]\n- [1-61252-A.ogg]: clip derived from mediterranean_sea_porticcio.flac (http://www.freesound.org/people/Corsica_S/sounds/61252/) by Corsica_S [CC-BY]\n- [1-91359-A.ogg]: clip derived from sea.aif (http://www.freesound.org/people/ariat/sounds/91359/) by ariat [CC-BY]\n- [1-91359-B.ogg]: clip derived from sea.aif (http://www.freesound.org/people/ariat/sounds/91359/) by ariat [CC-BY]\n- [2-102852-A.ogg]: clip derived from 20100803.surf.seagulls.wav (http://www.freesound.org/people/dobroide/sounds/102852/) by dobroide [CC-BY]\n- [2-124662-A.ogg]: clip derived from Water sea shore.mp3 (http://www.freesound.org/people/Slapbass/sounds/124662/) by Slapbass [CC-BY]\n- [2-125966-A.ogg]: clip derived from Waves in sea.wav (http://www.freesound.org/people/Ryding/sounds/125966/) by Ryding [CC0]\n- [2-132157-A.ogg]: clip derived from beach sea ocean waves crush breakwater 2.wav (http://www.freesound.org/people/bulbastre/sounds/132157/) by bulbastre [CC-BY]\n- [2-132157-B.ogg]: clip derived from beach sea ocean waves crush breakwater 2.wav (http://www.freesound.org/people/bulbastre/sounds/132157/) by bulbastre [CC-BY]\n- [2-133863-A.ogg]: clip derived from EXT RED SEA.wav (http://www.freesound.org/people/vhlam/sounds/133863/) by vhlam [CC0]\n- [2-137162-A.ogg]: clip derived from Sea inside a room (http://www.freesound.org/people/Akoustikos/sounds/137162/) by Akoustikos [CC-BY]\n- [2-155801-A.ogg]: clip derived from sea, rocky beach 1 (http://freesound.org/people/mario1298/sounds/155801/) by mario1298 [CC-BY]\n- [3-144827-A.ogg]: clip derived from sea waves (http://www.freesound.org/people/Eelke/sounds/144827/) by Eelke [CC-BY]\n- [3-144827-B.ogg]: clip derived from sea waves (http://www.freesound.org/people/Eelke/sounds/144827/) by Eelke [CC-BY]\n- [3-155642-A.ogg]: clip derived from sea \\u200b\\u200bon the road, night, Rhodes (http://www.freesound.org/people/mario1298/sounds/155642/) by mario1298 [CC-BY]\n- [3-155642-B.ogg]: clip derived from sea \\u200b\\u200bon the road, night, Rhodes (http://www.freesound.org/people/mario1298/sounds/155642/) by mario1298 [CC-BY]\n- [3-164120-A.ogg]: clip derived from waves003.wav (http://www.freesound.org/people/sonitusetaudio/sounds/164120/) by sonitusetaudio [CC-BY]\n- [3-164630-A.ogg]: clip derived from sea2.wav (http://www.freesound.org/people/schafferdavid/sounds/164630/) by schafferdavid [CC-BY]\n- [3-166422-A.ogg]: clip derived from wave sound (http://www.freesound.org/people/GrooveOption/sounds/166422/) by GrooveOption [CC-BY]\n- [3-187710-A.ogg]: clip derived from beach_short02.flac (http://www.freesound.org/people/Aiyumi/sounds/187710/) by Aiyumi [CC-BY]\n- [4-167063-A.ogg]: clip derived from Close sea waves (http://www.freesound.org/people/slag777/sounds/167063/) by slag777 [CC-BY]\n- [4-167063-B.ogg]: clip derived from Close sea waves (http://www.freesound.org/people/slag777/sounds/167063/) by slag777 [CC-BY]\n- [4-167063-C.ogg]: clip derived from Close sea waves (http://www.freesound.org/people/slag777/sounds/167063/) by slag777 [CC-BY]\n- [4-182613-A.ogg]: clip derived from CFX-20130331-UK-DorsetSeaBeach02.wav (http://www.freesound.org/people/Carlvus/sounds/182613/) by Carlvus [CC0]\n- [4-182613-B.ogg]: clip derived from CFX-20130331-UK-DorsetSeaBeach02.wav (http://www.freesound.org/people/Carlvus/sounds/182613/) by Carlvus [CC0]\n- [4-195497-A.ogg]: clip derived from Sea tide Waves At Robin Hoods Bay 24/07/2013 (http://www.freesound.org/people/m.newlove/sounds/195497/) by m.newlove [CC-BY]\n- [4-195497-B.ogg]: clip derived from Sea tide Waves At Robin Hoods Bay 24/07/2013 (http://www.freesound.org/people/m.newlove/sounds/195497/) by m.newlove [CC-BY]\n- [4-204618-A.ogg]: clip derived from 8am.wav (http://www.freesound.org/people/jareilly/sounds/204618/) by jareilly [CC0]\n- [5-200461-A.ogg]: clip derived from Sea gushes into seacave close.aif (http://www.freesound.org/people/ikbenraar/sounds/200461/) by ikbenraar [CC-BY]\n- [5-200461-B.ogg]: clip derived from Sea gushes into seacave close.aif (http://www.freesound.org/people/ikbenraar/sounds/200461/) by ikbenraar [CC-BY]\n- [5-208810-A.ogg]: clip derived from sea waves atmo.wav (http://www.freesound.org/people/monotraum/sounds/208810/) by monotraum [CC0]\n- [5-208810-B.ogg]: clip derived from sea waves atmo.wav (http://www.freesound.org/people/monotraum/sounds/208810/) by monotraum [CC0]\n- [5-213077-A.ogg]: clip derived from 3 jan 2014, waves breaking over sea wall at crosby, during high spring tide in winter storm, with Applied EQ.wav (http://www.freesound.org/people/crosbychris/sounds/213077/) by crosbychris [CC-BY]\n- [5-219379-A.ogg]: clip derived from the night Baltic001_14022014.wav (http://www.freesound.org/people/miastodzwiekow/sounds/219379/) by miastodzwiekow [CC-BY]\n- [5-219379-B.ogg]: clip derived from the night Baltic001_14022014.wav (http://www.freesound.org/people/miastodzwiekow/sounds/219379/) by miastodzwiekow [CC-BY]\n- [5-219379-C.ogg]: clip derived from the night Baltic001_14022014.wav (http://www.freesound.org/people/miastodzwiekow/sounds/219379/) by miastodzwiekow [CC-BY]\n\n\n203 - Crackling fire:\n- [1-17150-A.ogg]: clip derived from lg fire5.wav (http://www.freesound.org/people/lgarrett/sounds/17150/) by lgarrett [CC-BY]\n- [1-17565-A.ogg]: clip derived from fire.mp3 (http://www.freesound.org/people/markfrancombe/sounds/17565/) by markfrancombe [CC-BY]\n- [1-17742-A.ogg]: clip derived from Fire_Crackles(No Room).wav (http://www.freesound.org/people/Krisboruff/sounds/17742/) by Krisboruff [CC-BY]\n- [1-17808-A.ogg]: clip derived from Foc.WAV (http://www.freesound.org/people/Everard/sounds/17808/) by Everard [CC0]\n- [1-17808-B.ogg]: clip derived from Foc.WAV (http://www.freesound.org/people/Everard/sounds/17808/) by Everard [CC0]\n- [1-4211-A.ogg]:  clip derived from fire.crackling.mp3 (http://www.freesound.org/people/dobroide/sounds/4211/) by dobroide [CC-BY]\n- [1-46272-A.ogg]: clip derived from fire_medium_loop.wav (http://www.freesound.org/people/PhreaKsAccount/sounds/46272/) by PhreaKsAccount [CC-BY]\n- [1-7057-A.ogg]:  clip derived from fire.wav (http://www.freesound.org/people/mystiscool/sounds/7057/) by mystiscool [CC-BY]\n- [2-18766-A.ogg]: clip derived from chimney fire.wav (http://www.freesound.org/people/reinsamba/sounds/18766/) by reinsamba [CC-BY]\n- [2-18766-B.ogg]: clip derived from chimney fire.wav (http://www.freesound.org/people/reinsamba/sounds/18766/) by reinsamba [CC-BY]\n- [2-28314-A.ogg]: clip derived from FireBurning.wav (http://www.freesound.org/people/pcaeldries/sounds/28314/) by pcaeldries [CC-BY]\n- [2-28314-B.ogg]: clip derived from FireBurning.wav (http://www.freesound.org/people/pcaeldries/sounds/28314/) by pcaeldries [CC-BY]\n- [2-30322-A.ogg]: clip derived from FireBurning_v2.wav (http://www.freesound.org/people/pcaeldries/sounds/30322/) by pcaeldries [CC-BY]\n- [2-30322-B.ogg]: clip derived from FireBurning_v2.wav (http://www.freesound.org/people/pcaeldries/sounds/30322/) by pcaeldries [CC-BY]\n- [2-61311-A.ogg]: clip derived from Fire In Woodburning Stove.wav (http://www.freesound.org/people/Benboncan/sounds/61311/) by Benboncan [CC-BY]\n- [2-65747-A.ogg]: clip derived from 20081231.fire.01.wav (http://www.freesound.org/people/dobroide/sounds/65747/) by dobroide [CC-BY]\n- [3-104632-A.ogg]: clip derived from greve feu.wav (http://www.freesound.org/people/bogus/sounds/104632/) by bogus [CC-BY]\n- [3-104958-A.ogg]: clip derived from petit feu-little fire (2).wav (http://www.freesound.org/people/Glaneur%20de%20sons/sounds/104958/) by Glaneur de sons [CC-BY]\n- [3-120644-A.ogg]: clip derived from 20110503_fireplace.00.wav (http://www.freesound.org/people/dobroide/sounds/120644/) by dobroide [CC-BY]\n- [3-145774-A.ogg]: clip derived from crackling_fire_021012.wav (http://www.freesound.org/people/sagetyrtle/sounds/145774/) by sagetyrtle [CC0]\n- [3-147965-A.ogg]: clip derived from Fire Burning (http://www.freesound.org/people/andydonaldson6/sounds/147965/) by andydonaldson6 [CC-BY]\n- [3-157187-A.ogg]: clip derived from Campfire Crackle 3 (http://www.freesound.org/people/rbirdwise/sounds/157187/) by rbirdwise [CC-BY]\n- [3-158476-A.ogg]: clip derived from fire burning.mp3 (http://www.freesound.org/people/SoundCollectah/sounds/158476/) by SoundCollectah [CC0]\n- [3-65748-A.ogg]: clip derived from 20081231.fire.02.wav (http://www.freesound.org/people/dobroide/sounds/65748/) by dobroide [CC-BY]\n- [4-164661-A.ogg]: clip derived from Fire.wav (http://www.freesound.org/people/Adam_N/sounds/164661/) by Adam_N [CC0]\n- [4-164661-B.ogg]: clip derived from Fire.wav (http://www.freesound.org/people/Adam_N/sounds/164661/) by Adam_N [CC0]\n- [4-170247-A.ogg]: clip derived from campfire_01.wav (http://www.freesound.org/people/matucha/sounds/170247/) by matucha [CC-BY]\n- [4-170247-B.ogg]: clip derived from campfire_01.wav (http://www.freesound.org/people/matucha/sounds/170247/) by matucha [CC-BY]\n- [4-171207-A.ogg]: clip derived from Wood-burning stove (http://www.freesound.org/people/MrAuralization/sounds/171207/) by MrAuralization [CC-BY]\n- [4-181563-A.ogg]: clip derived from Fire Crackling 01.wav (http://www.freesound.org/people/kingsrow/sounds/181563/) by kingsrow [CC0]\n- [4-182368-A.ogg]: clip derived from Fire Crackling 02.mp3 (http://www.freesound.org/people/kingsrow/sounds/182368/) by kingsrow [CC0]\n- [4-182369-A.ogg]: clip derived from Fire Crackling 01.mp3 (http://www.freesound.org/people/kingsrow/sounds/182369/) by kingsrow [CC0]\n- [5-186924-A.ogg]: clip derived from Fire_Loop.wav (http://www.freesound.org/people/ReadeOnly/sounds/186924/) by ReadeOnly [CC0]\n- [5-189212-A.ogg]: clip derived from Burning Fire (http://www.freesound.org/people/Vurca/sounds/189212/) by Vurca [CC0]\n- [5-189237-A.ogg]: clip derived from Bonfire (http://www.freesound.org/people/amether/sounds/189237/) by amether [CC-BY]\n- [5-193473-A.ogg]: clip derived from RRMX SMALL FIRE.wav (http://www.freesound.org/people/skyklan47/sounds/193473/) by skyklan47 [CC-BY]\n- [5-193473-B.ogg]: clip derived from RRMX SMALL FIRE.wav (http://www.freesound.org/people/skyklan47/sounds/193473/) by skyklan47 [CC-BY]\n- [5-213802-A.ogg]: clip derived from burning-fire-steam.wav (http://www.freesound.org/people/laribum/sounds/213802/) by laribum [CC0]\n- [5-215658-A.ogg]: clip derived from Fireplace.wav (http://www.freesound.org/people/juskiddink/sounds/215658/) by juskiddink [CC-BY]\n- [5-215658-B.ogg]: clip derived from Fireplace.wav (http://www.freesound.org/people/juskiddink/sounds/215658/) by juskiddink [CC-BY]\n\n204 - Crickets:\n- [1-57316-A.ogg]: clip derived from Grillons1.WAV (http://www.freesound.org/people/J%20-%20Luc/sounds/57316/) by J - Luc [CC-BY]\n- [1-57318-A.ogg]: clip derived from Grillons3.WAV (http://www.freesound.org/people/J%20-%20Luc/sounds/57318/) by J - Luc [CC-BY]\n- [1-59102-A.ogg]: clip derived from emmentaler crickets.wav (http://www.freesound.org/people/Zozzy/sounds/59102/) by Zozzy [CC0]\n- [1-65483-A.ogg]: clip derived from Crickets.wav (http://www.freesound.org/people/Pogotron/sounds/65483/) by Pogotron [CC-Sampling+]\n- [1-7456-A.ogg]: clip derived from crickets.wav (http://www.freesound.org/people/mystiscool/sounds/7456/) by mystiscool [CC-BY]\n- [1-80840-A.ogg]: clip derived from Night Crickets & Bullfrogs.wav (http://www.freesound.org/people/sunsinger/sounds/80840/) by sunsinger [CC-BY]\n- [1-80841-A.ogg]: clip derived from Night Crickets at Bosque Del Apache-2.wav (http://www.freesound.org/people/sunsinger/sounds/80841/) by sunsinger [CC-BY]\n- [1-9841-A.ogg]: clip derived from crickets near ftn2.mp3 (http://www.freesound.org/people/LS/sounds/9841/) by LS [CC-BY]\n- [2-106019-A.ogg]: clip derived from crickets.wav (http://www.freesound.org/people/Christoff45/sounds/106019/) by Christoff45 [CC-BY]\n- [2-110613-A.ogg]: clip derived from animals_insect_cricket_001.wav (http://www.freesound.org/people/soundscalpel.com/sounds/110613/) by soundscalpel.com [CC-BY]\n- [2-125821-A.ogg]: clip derived from Crickets in Pont de Gau.wav (http://www.freesound.org/people/truflabart/sounds/125821/) by truflabart [CC-BY-NC]\n- [2-125875-A.ogg]: clip derived from Crickets and frogs in Camargue.wav (http://www.freesound.org/people/truflabart/sounds/125875/) by truflabart [CC-BY-NC]\n- [2-80844-A.ogg]: clip derived from One Cricket Louder.wav (http://www.freesound.org/people/sunsinger/sounds/80844/) by sunsinger [CC-BY]\n- [2-83270-A.ogg]: clip derived from lonely cricket.wav (http://www.freesound.org/people/Tito%20Lahaye/sounds/83270/) by Tito Lahaye [CC-BY-NC]\n- [2-85139-A.ogg]: clip derived from crickets.wav.mp3 (http://www.freesound.org/people/becks77/sounds/85139/) by becks77 [CC-BY]\n- [2-96033-A.ogg]: clip derived from FR.AO.CricketsAmbience.3.1.mp3 (http://www.freesound.org/people/rfhache/sounds/96033/) by rfhache [CC-BY]\n- [3-129338-A.ogg]: clip derived from Cricket 5.aif (http://www.freesound.org/people/le_abbaye_Noirlac/sounds/129338/) by le_abbaye_Noirlac [CC-BY]\n- [3-129678-A.ogg]: clip derived from crickets (http://www.freesound.org/people/FreethinkerAnon/sounds/129678/) by FreethinkerAnon [CC0]\n- [3-134802-A.ogg]: clip derived from barrage_of_crickets_chirping.wav (http://www.freesound.org/people/ianjuby/sounds/134802/) by ianjuby [CC0]\n- [3-146033-A.ogg]: clip derived from Crickets (http://www.freesound.org/people/thedogryan420/sounds/146033/) by thedogryan420 [CC0]\n- [3-155766-A.ogg]: clip derived from Crickets and Other Insects (http://www.freesound.org/people/kslagerman/sounds/155766/) by kslagerman [CC0]\n- [3-162786-A.ogg]: clip derived from CricketFX.wav (http://www.freesound.org/people/Mark_Ian/sounds/162786/) by Mark_Ian [CC0]\n- [3-163607-A.ogg]: clip derived from Wind, Soft. Crickets.wav (http://www.freesound.org/people/Leandros.Ntounis/sounds/163607/) by Leandros.Ntounis [CC-BY]\n- [3-163607-B.ogg]: clip derived from Wind, Soft. Crickets.wav (http://www.freesound.org/people/Leandros.Ntounis/sounds/163607/) by Leandros.Ntounis [CC-BY]\n- [4-163697-A.ogg]: clip derived from Crickets Orchestra.flac (http://www.freesound.org/people/Znalac/sounds/163697/) by Znalac [CC0]\n- [4-171823-A.ogg]: clip derived from crickets_creek.wav (http://www.freesound.org/people/rhapsodize/sounds/171823/) by rhapsodize [CC-BY]\n- [4-172143-A.ogg]: clip derived from Crickets (http://www.freesound.org/people/frus/sounds/172143/) by frus [CC-BY-NC]\n- [4-181362-A.ogg]: clip derived from Crickets calling at night, UK (http://www.freesound.org/people/icuttv/sounds/181362/) by icuttv [CC0]\n- [4-194246-A.ogg]: clip derived from meadow, fly, horse, crickets.wav (http://www.freesound.org/people/cosmician/sounds/194246/) by cosmician [CC0]\n- [4-195707-A.ogg]: clip derived from Evening_crickets.wav (http://www.freesound.org/people/adneonlux/sounds/195707/) by adneonlux [CC0]\n- [4-195805-A.ogg]: clip derived from crickets-and-bg.wav (http://www.freesound.org/people/arseniiv/sounds/195805/) by arseniiv [CC0]\n- [4-202749-A.ogg]: clip derived from Soft, Chilled Crickets Field Recording (http://www.freesound.org/people/rayjensen/sounds/202749/) by rayjensen [CC0]\n- [5-204352-A.ogg]: clip derived from crickets.wav (http://www.freesound.org/people/tagigobaso/sounds/204352/) by tagigobaso [CC-BY-NC]\n- [5-204352-B.ogg]: clip derived from crickets.wav (http://www.freesound.org/people/tagigobaso/sounds/204352/) by tagigobaso [CC-BY-NC]\n- [5-210540-A.ogg]: clip derived from crickets.wav (http://www.freesound.org/people/Sclolex/sounds/210540/) by Sclolex [CC0]\n- [5-215172-A.ogg]: clip derived from Chip and the outdoor cricket.mp3 (http://www.freesound.org/people/john9/sounds/215172/) by john9 [CC-BY]\n- [5-215179-A.ogg]: clip derived from high park crickets.3.wav (http://www.freesound.org/people/john9/sounds/215179/) by john9 [CC-BY]\n- [5-216213-A.ogg]: clip derived from Crickets_04.wav (http://www.freesound.org/people/RSilveira_88/sounds/216213/) by RSilveira_88 [CC-BY]\n- [5-216214-A.ogg]: clip derived from Crickets_03.wav (http://www.freesound.org/people/RSilveira_88/sounds/216214/) by RSilveira_88 [CC-BY]\n- [5-216216-A.ogg]: clip derived from Crickets_01.wav (http://www.freesound.org/people/RSilveira_88/sounds/216216/) by RSilveira_88 [CC-BY]\n\n205 - Chirping birds:\n- [1-100038-A.ogg]: clip derived from saz_birds_hyena.wav (http://www.freesound.org/people/soundbytez/sounds/100038/) by soundbytez [CC-BY]\n- [1-34495-A.ogg]: clip derived from birdsWBD.A.aiff (http://www.freesound.org/people/hanstimm/sounds/34495/) by hanstimm [CC-BY-NC]\n- [1-34497-A.ogg]: clip derived from birdsWBD.C.aiff (http://www.freesound.org/people/hanstimm/sounds/34497/) by hanstimm [CC-BY-NC]\n- [1-38559-A.ogg]: clip derived from Daintree Rainforrest Birds 1.wav (http://www.freesound.org/people/digifishmusic/sounds/38559/) by digifishmusic [CC-BY]\n- [1-38560-A.ogg]: clip derived from Daintree Rainforrest Birds 2.wav (http://www.freesound.org/people/digifishmusic/sounds/38560/) by digifishmusic [CC-BY]\n- [1-46040-A.ogg]: clip derived from birds_05.flac (http://www.freesound.org/people/cajo/sounds/46040/) by cajo [CC-BY-NC]\n- [1-54918-A.ogg]: clip derived from Wren and other birds.wav (http://www.freesound.org/people/inchadney/sounds/54918/) by inchadney [CC-BY]\n- [1-54918-B.ogg]: clip derived from Wren and other birds.wav (http://www.freesound.org/people/inchadney/sounds/54918/) by inchadney [CC-BY]\n- [2-108760-A.ogg]: clip derived from birds_evening.wav (http://www.freesound.org/people/soundbytez/sounds/108760/) by soundbytez [CC-BY]\n- [2-108760-B.ogg]: clip derived from birds_evening.wav (http://www.freesound.org/people/soundbytez/sounds/108760/) by soundbytez [CC-BY]\n- [2-108761-A.ogg]: clip derived from birds_late_morning.wav (http://www.freesound.org/people/soundbytez/sounds/108761/) by soundbytez [CC-BY]\n- [2-122616-A.ogg]: clip derived from Song thrush (pevchiy drozd) (http://www.freesound.org/people/urupin/sounds/122616/) by urupin [CC0]\n- [2-72547-A.ogg]: clip derived from BirdsWBD090509.NR.mp3 (http://www.freesound.org/people/hanstimm/sounds/72547/) by hanstimm [CC-BY-NC]\n- [2-72547-B.ogg]: clip derived from BirdsWBD090509.NR.mp3 (http://www.freesound.org/people/hanstimm/sounds/72547/) by hanstimm [CC-BY-NC]\n- [2-72547-C.ogg]: clip derived from BirdsWBD090509.NR.mp3 (http://www.freesound.org/people/hanstimm/sounds/72547/) by hanstimm [CC-BY-NC]\n- [2-72547-D.ogg]: clip derived from BirdsWBD090509.NR.mp3 (http://www.freesound.org/people/hanstimm/sounds/72547/) by hanstimm [CC-BY-NC]\n- [3-155577-A.ogg]: clip derived from Birds of the Weerribben-1.aiff (http://www.freesound.org/people/ducksingel/sounds/155577/) by ducksingel [CC-BY]\n- [3-155578-A.ogg]: clip derived from Birds of the Weerribben-2.aiff (http://www.freesound.org/people/ducksingel/sounds/155578/) by ducksingel [CC-BY]\n- [3-155579-A.ogg]: clip derived from Birds around the sandpit near Melissant.aiff (http://www.freesound.org/people/ducksingel/sounds/155579/) by ducksingel [CC-BY]\n- [3-155583-A.ogg]: clip derived from Birds of Tiengemeten-1.aiff (http://www.freesound.org/people/ducksingel/sounds/155583/) by ducksingel [CC-BY]\n- [3-155584-A.ogg]: clip derived from Birds of Tiengemeten-2.aiff (http://www.freesound.org/people/ducksingel/sounds/155584/) by ducksingel [CC-BY]\n- [3-156581-A.ogg]: clip derived from Singing Birds 3 - (Kouri Forest - Salonika) 22.05.12 18:35 (http://www.freesound.org/people/Ali@k/sounds/156581/) by Ali@k [CC-BY]\n- [3-156581-B.ogg]: clip derived from Singing Birds 3 - (Kouri Forest - Salonika) 22.05.12 18:35 (http://www.freesound.org/people/Ali@k/sounds/156581/) by Ali@k [CC-BY]\n- [3-181132-A.ogg]: clip derived from bird chirps.wav (http://www.freesound.org/people/keweldog/sounds/181132/) by keweldog [CC0]\n- [4-159609-A.ogg]: clip derived from Sylvia communis (slavka gray). (http://www.freesound.org/people/urupin/sounds/159609/) by urupin [CC-BY]\n- [4-159609-B.ogg]: clip derived from Sylvia communis (slavka gray). (http://www.freesound.org/people/urupin/sounds/159609/) by urupin [CC-BY]\n- [4-182795-A.ogg]: clip derived from bird chirps4.wav (http://www.freesound.org/people/keweldog/sounds/182795/) by keweldog [CC0]\n- [4-187769-A.ogg]: clip derived from binaural-birds-06-05-2013.flac (http://www.freesound.org/people/dwareing/sounds/187769/) by dwareing [CC0]\n- [4-187769-B.ogg]: clip derived from binaural-birds-06-05-2013.flac (http://www.freesound.org/people/dwareing/sounds/187769/) by dwareing [CC0]\n- [4-216211-A.ogg]: clip derived from Bird_01.wav (http://www.freesound.org/people/RSilveira_88/sounds/216211/) by RSilveira_88 [CC-BY]\n- [4-223125-A.ogg]: clip derived from starlings2.wav (http://www.freesound.org/people/keweldog/sounds/223125/) by keweldog [CC0]\n- [4-223127-A.ogg]: clip derived from general birdsong.wav (http://www.freesound.org/people/keweldog/sounds/223127/) by keweldog [CC0]\n- [5-242490-A.ogg]: clip derived from voice of birds (http://www.freesound.org/people/moloch777/sounds/242490/) by moloch777 [CC-BY]\n- [5-242491-A.ogg]: clip derived from voice of birds recorded in the forest (http://www.freesound.org/people/moloch777/sounds/242491/) by moloch777 [CC-BY]\n- [5-243448-A.ogg]: clip derived from Port d\'Aula-Oiseaux - 1 (http://www.freesound.org/people/Julien%20Nicolas/sounds/243448/) by Julien Nicolas [CC-BY]\n- [5-243449-A.ogg]: clip derived from Port d\'Aula-Oiseaux - 2 (http://www.freesound.org/people/Julien%20Nicolas/sounds/243449/) by Julien Nicolas [CC-BY]\n- [5-243450-A.ogg]: clip derived from Port d\'Aula-Oiseaux - 3 (http://www.freesound.org/people/Julien%20Nicolas/sounds/243450/) by Julien Nicolas [CC-BY]\n- [5-243459-A.ogg]: clip derived from BIRDS (http://www.freesound.org/people/Julien%20Nicolas/sounds/243459/) by Julien Nicolas [CC-BY]\n- [5-243459-B.ogg]: clip derived from BIRDS (http://www.freesound.org/people/Julien%20Nicolas/sounds/243459/) by Julien Nicolas [CC-BY]\n- [5-257839-A.ogg]: clip derived from Blackcap and other birds.wav (http://www.freesound.org/people/inchadney/sounds/257839/) by inchadney [CC-BY-NC]\n\n206 - Water drops:\n- [1-12653-A.ogg]: clip derived from Drop_Fast_Long_02.wav (http://www.freesound.org/people/mich3d/sounds/12653/) by mich3d [CC-BY]\n- [1-12654-A.ogg]: clip derived from Drop_Slow_Long_01.wav (http://www.freesound.org/people/mich3d/sounds/12654/) by mich3d [CC-BY]\n- [1-12654-B.ogg]: clip derived from Drop_Slow_Long_01.wav (http://www.freesound.org/people/mich3d/sounds/12654/) by mich3d [CC-BY]\n- [1-16746-A.ogg]: clip derived from water drop-01.aif (http://www.freesound.org/people/kijjaz/sounds/16746/) by kijjaz [CC-BY]\n- [1-23094-A.ogg]: clip derived from 20060915.drip.wav (http://www.freesound.org/people/dobroide/sounds/23094/) by dobroide [CC-BY]\n- [1-23094-B.ogg]: clip derived from 20060915.drip.wav (http://www.freesound.org/people/dobroide/sounds/23094/) by dobroide [CC-BY]\n- [1-50623-A.ogg]: clip derived from water_splash_10shots.wav (http://www.freesound.org/people/vibe_crc/sounds/50623/) by vibe_crc [CC0]\n- [1-90797-A.ogg]: clip derived from Water_Drops.wav (http://www.freesound.org/people/kMoon/sounds/90797/) by kMoon [CC-BY]\n- [2-124564-A.ogg]: clip derived from water-drip-rhythm.aif (http://www.freesound.org/people/alienistcog/sounds/124564/) by alienistcog [CC0]\n- [2-139748-A.ogg]: clip derived from water dripping 1 (http://www.freesound.org/people/hybu/sounds/139748/) by hybu [CC0]\n- [2-139748-B.ogg]: clip derived from water dripping 1 (http://www.freesound.org/people/hybu/sounds/139748/) by hybu [CC0]\n- [2-139749-A.ogg]: clip derived from water dripping 2 (http://www.freesound.org/people/hybu/sounds/139749/) by hybu [CC0]\n- [2-64962-A.ogg]: clip derived from 1233_dripping_water_1.wav (http://www.freesound.org/people/reinsamba/sounds/64962/) by reinsamba [CC-BY]\n- [2-64963-A.ogg]: clip derived from 1233_dripping_water_2.wav (http://www.freesound.org/people/reinsamba/sounds/64963/) by reinsamba [CC-BY]\n- [2-68595-A.ogg]: clip derived from 00932 water drops in severe canals.wav (http://www.freesound.org/people/Robinhood76/sounds/68595/) by Robinhood76 [CC-BY-NC]\n- [2-68595-B.ogg]: clip derived from 00932 water drops in severe canals.wav (http://www.freesound.org/people/Robinhood76/sounds/68595/) by Robinhood76 [CC-BY-NC]\n- [3-156907-A.ogg]: clip derived from Water drop (http://www.freesound.org/people/Elfman-Rox/sounds/156907/) by Elfman-Rox [CC0]\n- [3-160119-A.ogg]: clip derived from Dropping tiny pebble in still water surface (http://www.freesound.org/people/jorickhoofd/sounds/160119/) by jorickhoofd [CC-BY]\n- [3-164592-A.ogg]: clip derived from Water splash 3.wav (http://www.freesound.org/people/Adam_N/sounds/164592/) by Adam_N [CC0]\n- [3-164593-A.ogg]: clip derived from Water splash 2.wav (http://www.freesound.org/people/Adam_N/sounds/164593/) by Adam_N [CC0]\n- [3-164594-A.ogg]: clip derived from Water splash 1.wav (http://www.freesound.org/people/Adam_N/sounds/164594/) by Adam_N [CC0]\n- [3-164595-A.ogg]: clip derived from Water splash 6.wav (http://www.freesound.org/people/Adam_N/sounds/164595/) by Adam_N [CC0]\n- [3-166324-A.ogg]: clip derived from Water dripping on metal.wav (http://www.freesound.org/people/Adam_N/sounds/166324/) by Adam_N [CC0]\n- [3-166326-A.ogg]: clip derived from Water_dripping_echoing.wav (http://www.freesound.org/people/Adam_N/sounds/166326/) by Adam_N [CC0]\n- [4-168155-A.ogg]: clip derived from Water Drops (http://www.freesound.org/people/ultradust/sounds/168155/) by ultradust [CC-BY]\n- [4-174797-A.ogg]: clip derived from 120115_05 stones drop bath.WAV (http://www.freesound.org/people/toiletrolltube/sounds/174797/) by toiletrolltube [CC0]\n- [4-188293-A.ogg]: clip derived from Water Drop (http://www.freesound.org/people/willstepp/sounds/188293/) by willstepp [CC0]\n- [4-188293-B.ogg]: clip derived from Water Drop (http://www.freesound.org/people/willstepp/sounds/188293/) by willstepp [CC0]\n- [4-197871-A.ogg]: clip derived from drops in a glass.WAV (http://www.freesound.org/people/Millavsb/sounds/197871/) by Millavsb [CC0]\n- [4-212604-A.ogg]: clip derived from Water droplets (http://www.freesound.org/people/l3ardoc/sounds/212604/) by l3ardoc [CC0]\n- [4-212604-B.ogg]: clip derived from Water droplets (http://www.freesound.org/people/l3ardoc/sounds/212604/) by l3ardoc [CC0]\n- [4-212604-C.ogg]: clip derived from Water droplets (http://www.freesound.org/people/l3ardoc/sounds/212604/) by l3ardoc [CC0]\n- [5-241846-A.ogg]: clip derived from Splashing Water.wav (http://www.freesound.org/people/rafael45/sounds/241846/) by rafael45 [CC0]\n- [5-254832-A.ogg]: clip derived from Water drops in a bathtub (http://www.freesound.org/people/doxent/sounds/254832/) by doxent [CC-BY]\n- [5-254832-B.ogg]: clip derived from Water drops in a bathtub (http://www.freesound.org/people/doxent/sounds/254832/) by doxent [CC-BY]\n- [5-257349-A.ogg]: clip derived from small water drops (http://www.freesound.org/people/arseniiv/sounds/257349/) by arseniiv [CC0]\n- [5-259180-A.ogg]: clip derived from DUBECQ_Alison_2014_2015_water.wav (http://www.freesound.org/people/iut_Paris8/sounds/259180/) by iut_Paris8 [CC-BY-NC]\n- [5-261433-A.ogg]: clip derived from Base weich.WAV (http://www.freesound.org/people/cpttatosa/sounds/261433/) by cpttatosa [CC0]\n- [5-261439-A.ogg]: clip derived from Birne.WAV (http://www.freesound.org/people/cpttatosa/sounds/261439/) by cpttatosa [CC-BY]\n- [5-262641-A.ogg]: clip derived from drops in a underground parking (http://www.freesound.org/people/heifara/sounds/262641/) by heifara [CC0]\n\n207 - Wind:\n- [1-137296-A.ogg]: clip derived from Rain And Wind On Glass.flac (http://www.freesound.org/people/Benboncan/sounds/137296/) by Benboncan [CC-BY]\n- [1-29532-A.ogg]: clip derived from wind3.wav (http://www.freesound.org/people/eliasheuninck/sounds/29532/) by eliasheuninck [CC0]\n- [1-47709-A.ogg]: clip derived from Wind 02.flac (http://www.freesound.org/people/Jovica/sounds/47709/) by Jovica [CC-BY]\n- [1-47714-A.ogg]: clip derived from Wind 07.flac (http://www.freesound.org/people/Jovica/sounds/47714/) by Jovica [CC-BY]\n- [1-51035-A.ogg]: clip derived from 20080322.wind.light.wav (http://www.freesound.org/people/dobroide/sounds/51035/) by dobroide [CC-BY]\n- [1-51036-A.ogg]: clip derived from 20080322.wind.medium.wav (http://www.freesound.org/people/dobroide/sounds/51036/) by dobroide [CC-BY]\n- [1-51037-A.ogg]: clip derived from 20080322.wind.strong.wav (http://www.freesound.org/people/dobroide/sounds/51037/) by dobroide [CC-BY]\n- [1-69760-A.ogg]: clip derived from Wind In Bamboo.wav (http://www.freesound.org/people/Benboncan/sounds/69760/) by Benboncan [CC-BY]\n- [2-104952-A.ogg]: clip derived from Vent - wind (1).wav (http://www.freesound.org/people/Glaneur%20de%20sons/sounds/104952/) by Glaneur de sons [CC-BY]\n- [2-104952-B.ogg]: clip derived from Vent - wind (1).wav (http://www.freesound.org/people/Glaneur%20de%20sons/sounds/104952/) by Glaneur de sons [CC-BY]\n- [2-109371-A.ogg]: clip derived from 20101121.wind.howl.furious.01.wav (http://www.freesound.org/people/dobroide/sounds/109371/) by dobroide [CC-BY]\n- [2-109371-B.ogg]: clip derived from 20101121.wind.howl.furious.01.wav (http://www.freesound.org/people/dobroide/sounds/109371/) by dobroide [CC-BY]\n- [2-109371-C.ogg]: clip derived from 20101121.wind.howl.furious.01.wav (http://www.freesound.org/people/dobroide/sounds/109371/) by dobroide [CC-BY]\n- [2-109371-D.ogg]: clip derived from 20101121.wind.howl.furious.01.wav (http://www.freesound.org/people/dobroide/sounds/109371/) by dobroide [CC-BY]\n- [2-109374-A.ogg]: clip derived from 20101121.wind.howl.soft.01.wav (http://www.freesound.org/people/dobroide/sounds/109374/) by dobroide [CC-BY]\n- [2-205966-A.ogg]: clip derived from Medium Wind (http://www.freesound.org/people/kangaroovindaloo/sounds/205966/) by kangaroovindaloo [CC-BY]\n- [3-117504-A.ogg]: clip derived from winter wind 01a.aiff (http://www.freesound.org/people/klangfabrik/sounds/117504/) by klangfabrik [CC0]\n- [3-117504-B.ogg]: clip derived from winter wind 01a.aiff (http://www.freesound.org/people/klangfabrik/sounds/117504/) by klangfabrik [CC0]\n- [3-134699-A.ogg]: clip derived from Wind With Rigging.flac (http://www.freesound.org/people/Benboncan/sounds/134699/) by Benboncan [CC-BY]\n- [3-134699-B.ogg]: clip derived from Wind With Rigging.flac (http://www.freesound.org/people/Benboncan/sounds/134699/) by Benboncan [CC-BY]\n- [3-134699-C.ogg]: clip derived from Wind With Rigging.flac (http://www.freesound.org/people/Benboncan/sounds/134699/) by Benboncan [CC-BY]\n- [3-136608-A.ogg]: clip derived from Wind in small Forrest_01.wav (http://www.freesound.org/people/TheSoundcatcher/sounds/136608/) by TheSoundcatcher [CC-BY]\n- [3-246513-A.ogg]: clip derived from Pt. Lonsdale Wind #3 (http://www.freesound.org/people/kangaroovindaloo/sounds/246513/) by kangaroovindaloo [CC-BY]\n- [3-246513-B.ogg]: clip derived from Pt. Lonsdale Wind #3 (http://www.freesound.org/people/kangaroovindaloo/sounds/246513/) by kangaroovindaloo [CC-BY]\n- [4-144083-A.ogg]: clip derived from Wind Houling 1 .wav (http://www.freesound.org/people/Bosk1/sounds/144083/) by Bosk1 [CC-BY-NC]\n- [4-144083-B.ogg]: clip derived from Wind Houling 1 .wav (http://www.freesound.org/people/Bosk1/sounds/144083/) by Bosk1 [CC-BY-NC]\n- [4-144083-C.ogg]: clip derived from Wind Houling 1 .wav (http://www.freesound.org/people/Bosk1/sounds/144083/) by Bosk1 [CC-BY-NC]\n- [4-144085-A.ogg]: clip derived from Wind Houling little 2.wav (http://www.freesound.org/people/Bosk1/sounds/144085/) by Bosk1 [CC-BY-NC]\n- [4-163606-A.ogg]: clip derived from Wind, strong. through power cables.wav (http://www.freesound.org/people/Leandros.Ntounis/sounds/163606/) by Leandros.Ntounis [CC-BY]\n- [4-163608-B.ogg]: clip derived from Wind, strong. through rocks 2.wav (http://www.freesound.org/people/Leandros.Ntounis/sounds/163608/) by Leandros.Ntounis [CC-BY]\n- [4-163609-A.ogg]: clip derived from Wind, strong. through rocks 1.wav (http://www.freesound.org/people/Leandros.Ntounis/sounds/163609/) by Leandros.Ntounis [CC-BY]\n- [4-163609-B.ogg]: clip derived from Wind, strong. through rocks 1.wav (http://www.freesound.org/people/Leandros.Ntounis/sounds/163609/) by Leandros.Ntounis [CC-BY]\n- [5-117773-A.ogg]: clip derived from roaring wind.aiff (http://www.freesound.org/people/klangfabrik/sounds/117773/) by klangfabrik [CC0]\n- [5-157204-A.ogg]: clip derived from howling wind.WAV (http://www.freesound.org/people/inchadney/sounds/157204/) by inchadney [CC-BY]\n- [5-157204-B.ogg]: clip derived from howling wind.WAV (http://www.freesound.org/people/inchadney/sounds/157204/) by inchadney [CC-BY]\n- [5-179496-A.ogg]: clip derived from 111224_04 wind.WAV (http://www.freesound.org/people/toiletrolltube/sounds/179496/) by toiletrolltube [CC0]\n- [5-179496-B.ogg]: clip derived from 111224_04 wind.WAV (http://www.freesound.org/people/toiletrolltube/sounds/179496/) by toiletrolltube [CC0]\n- [5-217186-A.ogg]: clip derived from Wind at door howling 4.wav (http://www.freesound.org/people/Bosk1/sounds/217186/) by Bosk1 [CC-BY]\n- [5-217186-B.ogg]: clip derived from Wind at door howling 4.wav (http://www.freesound.org/people/Bosk1/sounds/217186/) by Bosk1 [CC-BY]\n- [5-217186-C.ogg]: clip derived from Wind at door howling 4.wav (http://www.freesound.org/people/Bosk1/sounds/217186/) by Bosk1 [CC-BY]\n\n208 - Pouring water:\n- [1-118559-A.ogg]: clip derived from Drink Pouring.wav (http://www.freesound.org/people/antwash/sounds/118559/) by antwash [CC0]\n- [1-50625-A.ogg]: clip derived from pouring_water_in_glass2.wav (http://www.freesound.org/people/vibe_crc/sounds/50625/) by vibe_crc [CC0]\n- [1-50688-A.ogg]: clip derived from 00052 nalewanie wody kubek 2.wav (http://www.freesound.org/people/Robinhood76/sounds/50688/) by Robinhood76 [CC-BY-NC]\n- [1-51433-A.ogg]: clip derived from pouring_water_4.wav (http://www.freesound.org/people/vibe_crc/sounds/51433/) by vibe_crc [CC0]\n- [1-51436-A.ogg]: clip derived from pouring_water_6.wav (http://www.freesound.org/people/vibe_crc/sounds/51436/) by vibe_crc [CC0]\n- [1-61221-A.ogg]: clip derived from water in cup.wav (http://www.freesound.org/people/timtube/sounds/61221/) by timtube [CC-BY-NC]\n- [1-67152-A.ogg]: clip derived from pouring water (dif speeds).wav (http://www.freesound.org/people/jrssandoval/sounds/67152/) by jrssandoval [CC-Sampling+]\n- [1-79220-A.ogg]: clip derived from water_pouring_ceramicbowl.wav (http://www.freesound.org/people/axiyee/sounds/79220/) by axiyee [CC-Sampling+]\n- [2-102414-A.ogg]: clip derived from WATER POURING MULTIPLE.mp3 (http://www.freesound.org/people/deankonidaris/sounds/102414/) by deankonidaris [CC-BY]\n- [2-102414-B.ogg]: clip derived from WATER POURING MULTIPLE.mp3 (http://www.freesound.org/people/deankonidaris/sounds/102414/) by deankonidaris [CC-BY]\n- [2-102414-C.ogg]: clip derived from WATER POURING MULTIPLE.mp3 (http://www.freesound.org/people/deankonidaris/sounds/102414/) by deankonidaris [CC-BY]\n- [2-102414-D.ogg]: clip derived from WATER POURING MULTIPLE.mp3 (http://www.freesound.org/people/deankonidaris/sounds/102414/) by deankonidaris [CC-BY]\n- [2-102414-E.ogg]: clip derived from WATER POURING MULTIPLE.mp3 (http://www.freesound.org/people/deankonidaris/sounds/102414/) by deankonidaris [CC-BY]\n- [2-102414-F.ogg]: clip derived from WATER POURING MULTIPLE.mp3 (http://www.freesound.org/people/deankonidaris/sounds/102414/) by deankonidaris [CC-BY]\n- [2-102414-G.ogg]: clip derived from WATER POURING MULTIPLE.mp3 (http://www.freesound.org/people/deankonidaris/sounds/102414/) by deankonidaris [CC-BY]\n- [2-126433-A.ogg]: clip derived from pouring-water.wav (http://www.freesound.org/people/rocketlazor/sounds/126433/) by rocketlazor [CC-Sampling+]\n- [3-108451-A.ogg]: clip derived from Pouring Into Glass.wav (http://www.freesound.org/people/Swoboman87/sounds/108451/) by Swoboman87 [CC0]\n- [3-108451-B.ogg]: clip derived from Pouring Into Glass.wav (http://www.freesound.org/people/Swoboman87/sounds/108451/) by Swoboman87 [CC0]\n- [3-127874-A.ogg]: clip derived from Household_Water_Cooler_Pour_Cup.wav (http://www.freesound.org/people/StephenSaldanha/sounds/127874/) by StephenSaldanha [CC-BY]\n- [3-142349-A.ogg]: clip derived from Pouring sounds (http://www.freesound.org/people/BranRainey/sounds/142349/) by BranRainey [CC0]\n- [3-145719-A.ogg]: clip derived from pouring water into a cup.wav (http://www.freesound.org/people/Zabuhailo/sounds/145719/) by Zabuhailo [CC0]\n- [3-149448-A.ogg]: clip derived from Pouring Water in metallic cup.mp3 (http://www.freesound.org/people/orginaljun/sounds/149448/) by orginaljun [CC-BY]\n- [3-154439-A.ogg]: clip derived from Water on Cup_1-2.aif (http://www.freesound.org/people/lucaslara/sounds/154439/) by lucaslara [CC0]\n- [3-161500-A.ogg]: clip derived from Water 03 (http://www.freesound.org/people/unfa/sounds/161500/) by unfa [CC-BY]\n- [4-168868-A.ogg]: clip derived from Pouring water 1 (http://www.freesound.org/people/DSPena/sounds/168868/) by DSPena [CC0]\n- [4-172377-A.ogg]: clip derived from Pouring Water (http://www.freesound.org/people/runirasmussen/sounds/172377/) by runirasmussen [CC0]\n- [4-180453-A.ogg]: clip derived from KOFFIE1.wav (http://www.freesound.org/people/RTFS/sounds/180453/) by RTFS [CC0]\n- [4-181628-A.ogg]: clip derived from Pouring water in a glass.wav (http://www.freesound.org/people/ancorapazzo/sounds/181628/) by ancorapazzo [CC-BY]\n- [4-182839-A.ogg]: clip derived from Filling Teapot (http://www.freesound.org/people/LanDub/sounds/182839/) by LanDub [CC0]\n- [4-186693-A.ogg]: clip derived from Fill Cup of Tea (Stereo Mic) (http://www.freesound.org/people/DrZoom/sounds/186693/) by DrZoom [CC-BY]\n- [4-187504-A.ogg]: clip derived from ds-001-teacup-pouring-steering.flac (http://www.freesound.org/people/datSilencer/sounds/187504/) by datSilencer [CC-BY]\n- [4-187504-B.ogg]: clip derived from ds-001-teacup-pouring-steering.flac (http://www.freesound.org/people/datSilencer/sounds/187504/) by datSilencer [CC-BY]\n- [5-205589-A.ogg]: clip derived from WATER_INTO_GLASS_CLOSE_01.wav (http://www.freesound.org/people/justingregoire/sounds/205589/) by justingregoire [CC0]\n- [5-205589-B.ogg]: clip derived from WATER_INTO_GLASS_CLOSE_01.wav (http://www.freesound.org/people/justingregoire/sounds/205589/) by justingregoire [CC0]\n- [5-207781-A.ogg]: clip derived from Pouring Liquid.wav (http://www.freesound.org/people/Dvideoguy/sounds/207781/) by Dvideoguy [CC0]\n- [5-212730-A.ogg]: clip derived from pouring01.mp3 (http://www.freesound.org/people/Taira%20Komori/sounds/212730/) by Taira Komori [CC-BY]\n- [5-212734-A.ogg]: clip derived from Pouring_into_grass.mp3 (http://www.freesound.org/people/Taira%20Komori/sounds/212734/) by Taira Komori [CC-BY]\n- [5-212736-A.ogg]: clip derived from Pouring_cold_tea.mp3 (http://www.freesound.org/people/Taira%20Komori/sounds/212736/) by Taira Komori [CC-BY]\n- [5-214362-A.ogg]: clip derived from  pouring Water into a glass  (http://www.freesound.org/people/edhutschek/sounds/214362/) by edhutschek [CC0]\n- [5-215634-A.ogg]: clip derived from Pouring beer into a glass (http://www.freesound.org/people/edhutschek/sounds/215634/) by edhutschek [CC0]\n\n209 - Toilet flush:\n- [1-20736-A.ogg]: clip derived from toilet flush.wav (http://www.freesound.org/people/megamart/sounds/20736/) by megamart [CC-Sampling+]\n- [1-28005-A.ogg]: clip derived from Flush1.wav (http://www.freesound.org/people/Konamiuss/sounds/28005/) by Konamiuss [CC0]\n- [1-30214-A.ogg]: clip derived from Toilet Flush.wav (http://www.freesound.org/people/tweeterdj/sounds/30214/) by tweeterdj [CC-BY]\n- [1-46274-A.ogg]: clip derived from flush1.wav (http://www.freesound.org/people/PhreaKsAccount/sounds/46274/) by PhreaKsAccount [CC-BY]\n- [1-51170-A.ogg]: clip derived from Toilet Flushed.wav (http://www.freesound.org/people/RutgerMuller/sounds/51170/) by RutgerMuller [CC0]\n- [1-53670-A.ogg]: clip derived from toilet flush.wav (http://www.freesound.org/people/crispy527/sounds/53670/) by crispy527 [CC-Sampling+]\n- [1-54752-A.ogg]: clip derived from Toilet flush.wav (http://www.freesound.org/people/Satoration/sounds/54752/) by Satoration [CC-BY]\n- [1-56269-A.ogg]: clip derived from Toilet.wav (http://www.freesound.org/people/Q.K./sounds/56269/) by Q.K. [CC0]\n- [2-64332-A.ogg]: clip derived from toilet_flush.WAV (http://www.freesound.org/people/Sir%20Smith/sounds/64332/) by Sir Smith [CC-Sampling+]\n- [2-65484-A.ogg]: clip derived from Toilet Flush.wav (http://www.freesound.org/people/Pogotron/sounds/65484/) by Pogotron [CC-Sampling+]\n- [2-67422-A.ogg]: clip derived from Toilet_Flush.wav (http://www.freesound.org/people/RoninMastaFX/sounds/67422/) by RoninMastaFX [CC-BY]\n- [2-70280-A.ogg]: clip derived from toilet_flushing.flac (http://www.freesound.org/people/ZuluT/sounds/70280/) by ZuluT [CC-BY]\n- [2-72677-A.ogg]: clip derived from toilet_flush_twice.wav (http://www.freesound.org/people/j1987/sounds/72677/) by j1987 [CC0]\n- [2-74977-A.ogg]: clip derived from Toilet Flush.wav (http://www.freesound.org/people/CBeeching/sounds/74977/) by CBeeching [CC0]\n- [2-84943-A.ogg]: clip derived from Toilet_Flush.aif (http://www.freesound.org/people/amszala/sounds/84943/) by amszala [CC-Sampling+]\n- [2-85945-A.ogg]: clip derived from toilet flush.aiff (http://www.freesound.org/people/mwmarsh/sounds/85945/) by mwmarsh [CC-Sampling+]\n- [3-100018-A.ogg]: clip derived from flush.aif (http://www.freesound.org/people/limwei/sounds/100018/) by limwei [CC-BY]\n- [3-108677-A.ogg]: clip derived from ToiletFlush.wav (http://www.freesound.org/people/Hotlavaman/sounds/108677/) by Hotlavaman [CC0]\n- [3-108791-A.ogg]: clip derived from toilet flush.wav (http://www.freesound.org/people/Corsica_S/sounds/108791/) by Corsica_S [CC-BY]\n- [3-112356-A.ogg]: clip derived from JM_HOME&OFFICE_Toilet Flush 01.wav (http://www.freesound.org/people/Julien%20Matthey/sounds/112356/) by Julien Matthey [CC0]\n- [3-118059-A.ogg]: clip derived from RKeaton_EMF366_7_Toilet Flush.aif (http://www.freesound.org/people/rkeato/sounds/118059/) by rkeato [CC-BY]\n- [3-92637-A.ogg]: clip derived from Flushing toilet.mp3 (http://www.freesound.org/people/CGEffex/sounds/92637/) by CGEffex [CC-BY]\n- [3-93010-A.ogg]: clip derived from 4.wav (http://www.freesound.org/people/CosmicD/sounds/93010/) by CosmicD [CC-BY]\n- [3-98771-A.ogg]: clip derived from LouisToiletLongFlush.wav (http://www.freesound.org/people/the_toilet_guy/sounds/98771/) by the_toilet_guy [CC-BY]\n- [4-119720-A.ogg]: clip derived from Toilet Flush Open.wav (http://www.freesound.org/people/jesabat/sounds/119720/) by jesabat [CC-BY]\n- [4-123680-A.ogg]: clip derived from Toilet Flushaf.aiff (http://www.freesound.org/people/priestjd/sounds/123680/) by priestjd [CC-BY]\n- [4-126046-A.ogg]: clip derived from Toilet Flushing And Refilling (http://www.freesound.org/people/mhtaylor67/sounds/126046/) by mhtaylor67 [CC0]\n- [4-126532-A.ogg]: clip derived from Flush.wav (http://www.freesound.org/people/sinkhole7/sounds/126532/) by sinkhole7 [CC-BY]\n- [4-135439-A.ogg]: clip derived from TOILET FLUSH AND REFILLING ECHO HI PITCH.wav (http://www.freesound.org/people/mlsprovideos/sounds/135439/) by mlsprovideos [CC0]\n- [4-141365-A.ogg]: clip derived from Toilet Flush & Tank-Refill (http://www.freesound.org/people/Oneirophile/sounds/141365/) by Oneirophile [CC0]\n- [4-145006-A.ogg]: clip derived from Toilet flush and fill (http://www.freesound.org/people/CastleofSamples/sounds/145006/) by CastleofSamples [CC-BY]\n- [4-152958-A.ogg]: clip derived from Flushing toilet (http://www.freesound.org/people/peasoperro/sounds/152958/) by peasoperro [CC-BY]\n- [5-156698-A.ogg]: clip derived from Toilet Flush Final.wav (http://www.freesound.org/people/dyoung117x/sounds/156698/) by dyoung117x [CC0]\n- [5-177034-A.ogg]: clip derived from Toilet Flush 020213 (http://www.freesound.org/people/richardemoore/sounds/177034/) by richardemoore [CC-BY]\n- [5-182404-A.ogg]: clip derived from Toilet Flush (http://www.freesound.org/people/LanDub/sounds/182404/) by LanDub [CC0]\n- [5-185908-A.ogg]: clip derived from Toilet Flush (http://www.freesound.org/people/cocaine/sounds/185908/) by cocaine [CC0]\n- [5-197913-A.ogg]: clip derived from toilet flush (http://www.freesound.org/people/fabiopx/sounds/197913/) by fabiopx [CC0]\n- [5-201664-A.ogg]: clip derived from Toilet Flush (http://www.freesound.org/people/ashleyxxpiano/sounds/201664/) by ashleyxxpiano [CC-BY-NC]\n- [5-202020-A.ogg]: clip derived from toilet flush.wav (http://www.freesound.org/people/ryancacophony/sounds/202020/) by ryancacophony [CC0]\n- [5-202540-A.ogg]: clip derived from toilet flush.flac (http://www.freesound.org/people/Vitouliss/sounds/202540/) by Vitouliss [CC0]\n\n210 - Thunderstorm:\n- [1-101296-A.ogg]: clip derived from thunderstorm.wav (http://www.freesound.org/people/raoul_slayer/sounds/101296/) by raoul_slayer [CC0]\n- [1-101296-B.ogg]: clip derived from thunderstorm.wav (http://www.freesound.org/people/raoul_slayer/sounds/101296/) by raoul_slayer [CC0]\n- [1-115521-A.ogg]: clip derived from 20100614thunder7.wav (http://www.freesound.org/people/csengeri/sounds/115521/) by csengeri [CC-BY]\n- [1-23222-A.ogg]: clip derived from thunderstorm2.flac (http://www.freesound.org/people/Erdie/sounds/23222/) by Erdie [CC-BY]\n- [1-23222-B.ogg]: clip derived from thunderstorm2.flac (http://www.freesound.org/people/Erdie/sounds/23222/) by Erdie [CC-BY]\n- [1-24524-A.ogg]: clip derived from thunderstorm-raw-data.flac (http://www.freesound.org/people/Erdie/sounds/24524/) by Erdie [CC-BY]\n- [1-24524-B.ogg]: clip derived from thunderstorm-raw-data.flac (http://www.freesound.org/people/Erdie/sounds/24524/) by Erdie [CC-BY]\n- [1-24524-C.ogg]: clip derived from thunderstorm-raw-data.flac (http://www.freesound.org/people/Erdie/sounds/24524/) by Erdie [CC-BY]\n- [2-104105-A.ogg]: clip derived from 02139 rolling thunderstorm.wav (http://www.freesound.org/people/Robinhood76/sounds/104105/) by Robinhood76 [CC-BY-NC]\n- [2-104105-B.ogg]: clip derived from 02139 rolling thunderstorm.wav (http://www.freesound.org/people/Robinhood76/sounds/104105/) by Robinhood76 [CC-BY-NC]\n- [2-39441-A.ogg]: clip derived from Vihar2.MP3 (http://www.freesound.org/people/csengeri/sounds/39441/) by csengeri [CC-BY]\n- [2-39443-A.ogg]: clip derived from Vihar4.MP3 (http://www.freesound.org/people/csengeri/sounds/39443/) by csengeri [CC-BY]\n- [2-39443-B.ogg]: clip derived from Vihar4.MP3 (http://www.freesound.org/people/csengeri/sounds/39443/) by csengeri [CC-BY]\n- [2-39945-A.ogg]: clip derived from EQCloseLightningBolts20August2007.mp3 (http://www.freesound.org/people/csengeri/sounds/39945/) by csengeri [CC-BY]\n- [2-39945-B.ogg]: clip derived from EQCloseLightningBolts20August2007.mp3 (http://www.freesound.org/people/csengeri/sounds/39945/) by csengeri [CC-BY]\n- [2-39945-C.ogg]: clip derived from EQCloseLightningBolts20August2007.mp3 (http://www.freesound.org/people/csengeri/sounds/39945/) by csengeri [CC-BY]\n- [3-103050-A.ogg]: clip derived from Storm1408201001.mp3 (http://www.freesound.org/people/csengeri/sounds/103050/) by csengeri [CC-BY]\n- [3-103051-A.ogg]: clip derived from Storm1408201002.mp3 (http://www.freesound.org/people/csengeri/sounds/103051/) by csengeri [CC-BY]\n- [3-103051-B.ogg]: clip derived from Storm1408201002.mp3 (http://www.freesound.org/people/csengeri/sounds/103051/) by csengeri [CC-BY]\n- [3-103051-C.ogg]: clip derived from Storm1408201002.mp3 (http://www.freesound.org/people/csengeri/sounds/103051/) by csengeri [CC-BY]\n- [3-123224-A.ogg]: clip derived from Thunderstorm.WAV (http://www.freesound.org/people/inchadney/sounds/123224/) by inchadney [CC-BY]\n- [3-124600-A.ogg]: clip derived from Thunderstorm.WAV (http://www.freesound.org/people/inchadney/sounds/124600/) by inchadney [CC-BY]\n- [3-144891-A.ogg]: clip derived from 20120202 Bangkok Thunderstorm (http://www.freesound.org/people/kijjaz/sounds/144891/) by kijjaz [CC0]\n- [3-144891-B.ogg]: clip derived from 20120202 Bangkok Thunderstorm (http://www.freesound.org/people/kijjaz/sounds/144891/) by kijjaz [CC0]\n- [4-125070-A.ogg]: clip derived from 2011julius1901.wav (http://www.freesound.org/people/csengeri/sounds/125070/) by csengeri [CC-BY]\n- [4-125071-A.ogg]: clip derived from 2011julius1902.wav (http://www.freesound.org/people/csengeri/sounds/125071/) by csengeri [CC-BY]\n- [4-125072-A.ogg]: clip derived from 2011julius1903.wav (http://www.freesound.org/people/csengeri/sounds/125072/) by csengeri [CC-BY]\n- [4-125072-B.ogg]: clip derived from 2011julius1903.wav (http://www.freesound.org/people/csengeri/sounds/125072/) by csengeri [CC-BY]\n- [4-156993-A.ogg]: clip derived from Thunder.wav (http://www.freesound.org/people/chrscrwfrd18/sounds/156993/) by chrscrwfrd18 [CC0]\n- [4-161519-A.ogg]: clip derived from 20120720Thunder.wav (http://www.freesound.org/people/daveincamas/sounds/161519/) by daveincamas [CC0]\n- [4-165813-A.ogg]: clip derived from 120925_0011.mp3 (http://www.freesound.org/people/csengeri/sounds/165813/) by csengeri [CC-BY]\n- [4-165813-B.ogg]: clip derived from 120925_0011.mp3 (http://www.freesound.org/people/csengeri/sounds/165813/) by csengeri [CC-BY]\n- [5-156999-A.ogg]: clip derived from Lightening.wav (http://www.freesound.org/people/chrscrwfrd18/sounds/156999/) by chrscrwfrd18 [CC0]\n- [5-156999-B.ogg]: clip derived from Lightening.wav (http://www.freesound.org/people/chrscrwfrd18/sounds/156999/) by chrscrwfrd18 [CC0]\n- [5-156999-C.ogg]: clip derived from Lightening.wav (http://www.freesound.org/people/chrscrwfrd18/sounds/156999/) by chrscrwfrd18 [CC0]\n- [5-156999-D.ogg]: clip derived from Lightening.wav (http://www.freesound.org/people/chrscrwfrd18/sounds/156999/) by chrscrwfrd18 [CC0]\n- [5-156999-E.ogg]: clip derived from Lightening.wav (http://www.freesound.org/people/chrscrwfrd18/sounds/156999/) by chrscrwfrd18 [CC0]\n- [5-188495-A.ogg]: clip derived from Thunderstorm.WAV (http://www.freesound.org/people/inchadney/sounds/188495/) by inchadney [CC-BY-NC]\n- [5-192191-A.ogg]: clip derived from Thunderstorm.WAV (http://www.freesound.org/people/inchadney/sounds/192191/) by inchadney [CC-BY-NC]\n- [5-195557-A.ogg]: clip derived from Thunderstorm at night.WAV (http://www.freesound.org/people/inchadney/sounds/195557/) by inchadney [CC-BY-NC]\n\n301 - Crying baby:\n- [1-187207-A.ogg]: clip derived from More baby noises... (http://www.freesound.org/people/k3pp/sounds/187207/) by k3pp [CC0]\n- [1-211527-A.ogg]: clip derived from Crying newborn baby child 3.WAV (http://www.freesound.org/people/the_yura/sounds/211527/) by the_yura [CC0]\n- [1-211527-B.ogg]: clip derived from Crying newborn baby child 3.WAV (http://www.freesound.org/people/the_yura/sounds/211527/) by the_yura [CC0]\n- [1-211527-C.ogg]: clip derived from Crying newborn baby child 3.WAV (http://www.freesound.org/people/the_yura/sounds/211527/) by the_yura [CC0]\n- [1-22694-A.ogg]: clip derived from baby2.mp3 (http://www.freesound.org/people/Erdie/sounds/22694/) by Erdie [CC-BY]\n- [1-22694-B.ogg]: clip derived from baby2.mp3 (http://www.freesound.org/people/Erdie/sounds/22694/) by Erdie [CC-BY]\n- [1-60997-A.ogg]: clip derived from 01-Baby_Crying_small-room.wav (http://www.freesound.org/people/poissonmort/sounds/60997/) by poissonmort [CC-BY]\n- [1-60997-B.ogg]: clip derived from 01-Baby_Crying_small-room.wav (http://www.freesound.org/people/poissonmort/sounds/60997/) by poissonmort [CC-BY]\n- [2-107351-A.ogg]: clip derived from baby-crying-1_(sounds-AT-ermfilm_pl).wav (http://www.freesound.org/people/ermfilm/sounds/107351/) by ermfilm [CC-BY]\n- [2-107351-B.ogg]: clip derived from baby-crying-1_(sounds-AT-ermfilm_pl).wav (http://www.freesound.org/people/ermfilm/sounds/107351/) by ermfilm [CC-BY]\n- [2-151079-A.ogg]: clip derived from 01 baby with hiccups.wav (http://www.freesound.org/people/PickleJones/sounds/151079/) by PickleJones [CC0]\n- [2-50665-A.ogg]: clip derived from toddler cry 1 0803xx.wav (http://www.freesound.org/people/BoilingSand/sounds/50665/) by BoilingSand [CC-BY]\n- [2-50666-A.ogg]: clip derived from toddler cry 2 0803xx.wav (http://www.freesound.org/people/BoilingSand/sounds/50666/) by BoilingSand [CC-BY]\n- [2-66637-A.ogg]: clip derived from crying baby merged.wav (http://www.freesound.org/people/bonehead/sounds/66637/) by bonehead [CC0]\n- [2-66637-B.ogg]: clip derived from crying baby merged.wav (http://www.freesound.org/people/bonehead/sounds/66637/) by bonehead [CC0]\n- [2-80482-A.ogg]: clip derived from bebe_32_Iz.aif (http://www.freesound.org/people/stevelalonde/sounds/80482/) by stevelalonde [CC-BY]\n- [3-151080-A.ogg]: clip derived from 01 baby crying hard and long.wav (http://www.freesound.org/people/PickleJones/sounds/151080/) by PickleJones [CC0]\n- [3-151081-A.ogg]: clip derived from 01 baby crying winding down.wav (http://www.freesound.org/people/PickleJones/sounds/151081/) by PickleJones [CC0]\n- [3-151081-B.ogg]: clip derived from 01 baby crying winding down.wav (http://www.freesound.org/people/PickleJones/sounds/151081/) by PickleJones [CC0]\n- [3-152007-A.ogg]: clip derived from babys crying.mp3 (http://www.freesound.org/people/winsx87/sounds/152007/) by winsx87 [CC0]\n- [3-152007-B.ogg]: clip derived from babys crying.mp3 (http://www.freesound.org/people/winsx87/sounds/152007/) by winsx87 [CC0]\n- [3-152007-C.ogg]: clip derived from babys crying.mp3 (http://www.freesound.org/people/winsx87/sounds/152007/) by winsx87 [CC0]\n- [3-152007-D.ogg]: clip derived from babys crying.mp3 (http://www.freesound.org/people/winsx87/sounds/152007/) by winsx87 [CC0]\n- [3-152007-E.ogg]: clip derived from babys crying.mp3 (http://www.freesound.org/people/winsx87/sounds/152007/) by winsx87 [CC0]\n- [4-167077-A.ogg]: clip derived from JM_HUMAN_Baby 01c - Crying.wav (http://www.freesound.org/people/Julien%20Matthey/sounds/167077/) by Julien Matthey [CC0]\n- [4-167077-B.ogg]: clip derived from JM_HUMAN_Baby 01c - Crying.wav (http://www.freesound.org/people/Julien%20Matthey/sounds/167077/) by Julien Matthey [CC0]\n- [4-167077-C.ogg]: clip derived from JM_HUMAN_Baby 01c - Crying.wav (http://www.freesound.org/people/Julien%20Matthey/sounds/167077/) by Julien Matthey [CC0]\n- [4-185575-A.ogg]: clip derived from Crying baby.wav (http://www.freesound.org/people/ciccarelli/sounds/185575/) by ciccarelli [CC0]\n- [4-185575-B.ogg]: clip derived from Crying baby.wav (http://www.freesound.org/people/ciccarelli/sounds/185575/) by ciccarelli [CC0]\n- [4-185575-C.ogg]: clip derived from Crying baby.wav (http://www.freesound.org/people/ciccarelli/sounds/185575/) by ciccarelli [CC0]\n- [4-59579-A.ogg]: clip derived from babycrying2.mp3 (http://www.freesound.org/people/morgantj/sounds/59579/) by morgantj [CC-BY]\n- [4-59579-B.ogg]: clip derived from babycrying2.mp3 (http://www.freesound.org/people/morgantj/sounds/59579/) by morgantj [CC-BY]\n- [5-151085-A.ogg]: clip derived from 01 baby crying hard.wav (http://www.freesound.org/people/PickleJones/sounds/151085/) by PickleJones [CC0]\n- [5-198411-A.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]\n- [5-198411-B.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]\n- [5-198411-C.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]\n- [5-198411-D.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]\n- [5-198411-E.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]\n- [5-198411-F.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]\n- [5-198411-G.ogg]: clip derived from long baby cry 7 minutes.wav (http://www.freesound.org/people/klankbeeld/sounds/198411/) by klankbeeld [CC-BY]\n\n302 - Sneezing:\n- [1-26143-A.ogg]: clip derived from hatsjoe.wav (http://www.freesound.org/people/WIM/sounds/26143/) by WIM [CC-BY]\n- [1-29680-A.ogg]: clip derived from Sneeze_.wav (http://www.freesound.org/people/HerbertBoland/sounds/29680/) by HerbertBoland [CC-BY]\n- [1-31748-A.ogg]: clip derived from Sneezing.wav (http://www.freesound.org/people/digifishmusic/sounds/31748/) by digifishmusic [CC-BY]\n- [1-47273-A.ogg]: clip derived from baby_sneeze.wav (http://www.freesound.org/people/reinsamba/sounds/47273/) by reinsamba [CC-BY]\n- [1-47274-A.ogg]: clip derived from male_sneeze.wav (http://www.freesound.org/people/reinsamba/sounds/47274/) by reinsamba [CC-BY]\n- [1-54505-A.ogg]: clip derived from sneeze.wav (http://www.freesound.org/people/sagetyrtle/sounds/54505/) by sagetyrtle [CC0]\n- [1-59324-A.ogg]: clip derived from Sneeze.wav (http://www.freesound.org/people/transitking/sounds/59324/) by transitking [CC0]\n- [1-81883-A.ogg]: clip derived from sneeze.wav (http://www.freesound.org/people/stevenpam/sounds/81883/) by stevenpam [CC-BY]\n- [2-109505-A.ogg]: clip derived from atishoo.wav (http://www.freesound.org/people/jeanorwin/sounds/109505/) by jeanorwin [CC-BY]\n- [2-118104-A.ogg]: clip derived from Female Sneezing -- Two Sneezes Spaced Out.WAV (http://www.freesound.org/people/Ndheger/sounds/118104/) by Ndheger [CC-BY]\n- [2-119102-A.ogg]: clip derived from Sneeze; male_1-2.aif (http://www.freesound.org/people/domrodrig/sounds/119102/) by domrodrig [CC0]\n- [2-128631-A.ogg]: clip derived from sneeze (http://www.freesound.org/people/bigjoedrummer/sounds/128631/) by bigjoedrummer [CC-BY]\n- [2-130978-A.ogg]: clip derived from Sneeze AC 2.wav (http://www.freesound.org/people/oldedgar/sounds/130978/) by oldedgar [CC0]\n- [2-130979-A.ogg]: clip derived from Sneeze AC 1.wav (http://www.freesound.org/people/oldedgar/sounds/130979/) by oldedgar [CC0]\n- [2-82538-A.ogg]: clip derived from Mark sneezing.wav (http://www.freesound.org/people/Electron%20John/sounds/82538/) by Electron John [CC0]\n- [2-93030-A.ogg]: clip derived from Sneeze and Sniffle 44hzStereo.wav (http://www.freesound.org/people/higginsdj/sounds/93030/) by higginsdj [CC0]\n- [3-141684-A.ogg]: clip derived from Two Sneezes (http://www.freesound.org/people/thomasevd/sounds/141684/) by thomasevd [CC-BY]\n- [3-142601-A.ogg]: clip derived from sneeze2.mp3 (http://www.freesound.org/people/loisgwenllian/sounds/142601/) by loisgwenllian [CC0]\n- [3-142605-A.ogg]: clip derived from sneeze3.mp3 (http://www.freesound.org/people/loisgwenllian/sounds/142605/) by loisgwenllian [CC0]\n- [3-143119-A.ogg]: clip derived from Sneezer.wav (http://www.freesound.org/people/1pjladd2/sounds/143119/) by 1pjladd2 [CC0]\n- [3-144692-A.ogg]: clip derived from Sneeze.wav (http://www.freesound.org/people/tekgnosis/sounds/144692/) by tekgnosis [CC0]\n- [3-148330-A.ogg]: clip derived from Sneeze.wav (http://www.freesound.org/people/Shapeshifter242/sounds/148330/) by Shapeshifter242 [CC-BY]\n- [3-150231-A.ogg]: clip derived from Female Sneeze (http://www.freesound.org/people/domvoice/sounds/150231/) by domvoice [CC0]\n- [3-156558-A.ogg]: clip derived from sneeze001.wav (http://www.freesound.org/people/bennyConsulting/sounds/156558/) by bennyConsulting [CC0]\n- [4-156843-A.ogg]: clip derived from Sneeze Male JGray.wav (http://www.freesound.org/people/JGrayArt/sounds/156843/) by JGrayArt [CC0]\n- [4-156844-A.ogg]: clip derived from Sneeze Female x3 JGray.wav (http://www.freesound.org/people/JGrayArt/sounds/156844/) by JGrayArt [CC0]\n- [4-157297-A.ogg]: clip derived from Sneeze.wav (http://www.freesound.org/people/husky70/sounds/157297/) by husky70 [CC0]\n- [4-167642-A.ogg]: clip derived from Sneeze 01.aiff (http://www.freesound.org/people/SoundCollectah/sounds/167642/) by SoundCollectah [CC0]\n- [4-171519-A.ogg]: clip derived from sneeze.wav (http://www.freesound.org/people/Zajjman/sounds/171519/) by Zajjman [CC0]\n- [4-184434-A.ogg]: clip derived from Keyboard Sneeze (http://www.freesound.org/people/unfa/sounds/184434/) by unfa [CC-BY]\n- [4-185415-A.ogg]: clip derived from Sneeze Shot (http://www.freesound.org/people/unfa/sounds/185415/) by unfa [CC-BY]\n- [4-185619-A.ogg]: clip derived from sneeze (http://www.freesound.org/people/organicmanpl/sounds/185619/) by organicmanpl [CC0]\n- [5-187979-A.ogg]: clip derived from sneeze.MP3 (http://www.freesound.org/people/fliegermatze/sounds/187979/) by fliegermatze [CC-BY]\n- [5-194533-A.ogg]: clip derived from stan_baby_sneezing.flac (http://www.freesound.org/people/wjoojoo/sounds/194533/) by wjoojoo [CC-BY]\n- [5-201274-A.ogg]: clip derived from Person Sneezing Sound Bit.wav (http://www.freesound.org/people/chrisbrowne626/sounds/201274/) by chrisbrowne626 [CC-BY]\n- [5-202220-A.ogg]: clip derived from Sneeze (http://www.freesound.org/people/Huminaatio/sounds/202220/) by Huminaatio [CC0]\n- [5-220026-A.ogg]: clip derived from mysneeze2.wav (http://www.freesound.org/people/Reitanna/sounds/220026/) by Reitanna [CC0]\n- [5-220027-A.ogg]: clip derived from mysneeze.wav (http://www.freesound.org/people/Reitanna/sounds/220027/) by Reitanna [CC0]\n- [5-221518-A.ogg]: clip derived from woman sneeze (http://www.freesound.org/people/Danieka/sounds/221518/) by Danieka [CC0]\n- [5-221593-A.ogg]: clip derived from Girly sneeze (http://www.freesound.org/people/ahjteam/sounds/221593/) by ahjteam [CC0]\n\n303 - Clapping:\n- [1-104089-A.ogg]: clip derived from Clapping Small Room.wav (http://www.freesound.org/people/sorohanro/sounds/104089/) by sorohanro [CC-BY]\n- [1-104089-B.ogg]: clip derived from Clapping Small Room.wav (http://www.freesound.org/people/sorohanro/sounds/104089/) by sorohanro [CC-BY]\n- [1-105224-A.ogg]: clip derived from Crowd Claps.wav (http://www.freesound.org/people/club%20sound/sounds/105224/) by club sound [CC-BY-NC]\n- [1-110537-A.ogg]: clip derived from www.soundbyter.com_LargeAudienceClapping.wav (http://www.freesound.org/people/soundbyter.com/sounds/110537/) by soundbyter.com [CC-Sampling+]\n- [1-115920-A.ogg]: clip derived from Clapping, Medium Audience, slow to fast.wav (http://www.freesound.org/people/issalcake/sounds/115920/) by issalcake [CC0]\n- [1-115920-B.ogg]: clip derived from Clapping, Medium Audience, slow to fast.wav (http://www.freesound.org/people/issalcake/sounds/115920/) by issalcake [CC0]\n- [1-115921-A.ogg]: clip derived from Clapping, Small audience, far away.wav (http://www.freesound.org/people/issalcake/sounds/115921/) by issalcake [CC0]\n- [1-94036-A.ogg]: clip derived from audience clapping.wav (http://www.freesound.org/people/kukla/sounds/94036/) by kukla [CC-BY-NC]\n- [2-25292-A.ogg]: clip derived from Bye_Bye_Claps_ses1.wav (http://www.freesound.org/people/freesound/sounds/25292/) by freesound [CC0]\n- [2-25293-A.ogg]: clip derived from Clap_ses2.wav (http://www.freesound.org/people/freesound/sounds/25293/) by freesound [CC0]\n- [2-57733-A.ogg]: clip derived from fade clapping.wav (http://www.freesound.org/people/timtube/sounds/57733/) by timtube [CC-BY-NC]\n- [2-60900-A.ogg]: clip derived from clapping.wav (http://www.freesound.org/people/timtube/sounds/60900/) by timtube [CC-BY-NC]\n- [2-76408-A.ogg]: clip derived from Clap.mp3 (http://www.freesound.org/people/therabidfrog/sounds/76408/) by therabidfrog [CC0]\n- [2-76408-B.ogg]: clip derived from Clap.mp3 (http://www.freesound.org/people/therabidfrog/sounds/76408/) by therabidfrog [CC0]\n- [2-76408-C.ogg]: clip derived from Clap.mp3 (http://www.freesound.org/people/therabidfrog/sounds/76408/) by therabidfrog [CC0]\n- [2-76408-D.ogg]: clip derived from Clap.mp3 (http://www.freesound.org/people/therabidfrog/sounds/76408/) by therabidfrog [CC0]\n- [3-130330-A.ogg]: clip derived from Polite Clapping with no Yelling (http://www.freesound.org/people/dianadesim/sounds/130330/) by dianadesim [CC0]\n- [3-138114-A.ogg]: clip derived from Small Crowd Clapping 2 (http://www.freesound.org/people/snakebarney/sounds/138114/) by snakebarney [CC0]\n- [3-149465-A.ogg]: clip derived from clapping.wav (http://www.freesound.org/people/Clagnut/sounds/149465/) by Clagnut [CC0]\n- [3-177082-A.ogg]: clip derived from Amen Clap.wav (http://www.freesound.org/people/GB01/sounds/177082/) by GB01 [CC-BY]\n- [3-177083-A.ogg]: clip derived from Crowd Clap.wav (http://www.freesound.org/people/GB01/sounds/177083/) by GB01 [CC-BY]\n- [3-181278-A.ogg]: clip derived from applause 29.1.11.wav (http://www.freesound.org/people/toiletrolltube/sounds/181278/) by toiletrolltube [CC0]\n- [3-197435-A.ogg]: clip derived from clapping group unisono (http://www.freesound.org/people/Yuval/sounds/197435/) by Yuval [CC-BY-NC]\n- [3-197435-B.ogg]: clip derived from clapping group unisono (http://www.freesound.org/people/Yuval/sounds/197435/) by Yuval [CC-BY-NC]\n- [4-189828-A.ogg]: clip derived from audience clapping 02 130525_13.wav (http://www.freesound.org/people/klankbeeld/sounds/189828/) by klankbeeld [CC-BY]\n- [4-189830-A.ogg]: clip derived from audience ClapYell outoors 01.wav (http://www.freesound.org/people/klankbeeld/sounds/189830/) by klankbeeld [CC-BY]\n- [4-189832-A.ogg]: clip derived from audience clapping 06 130525_13.wav (http://www.freesound.org/people/klankbeeld/sounds/189832/) by klankbeeld [CC-BY]\n- [4-189832-B.ogg]: clip derived from audience clapping 06 130525_13.wav (http://www.freesound.org/people/klankbeeld/sounds/189832/) by klankbeeld [CC-BY]\n- [4-189833-A.ogg]: clip derived from audience clapping 05 130525_13.wav (http://www.freesound.org/people/klankbeeld/sounds/189833/) by klankbeeld [CC-BY]\n- [4-189836-A.ogg]: clip derived from audience clapping 08 130525_13.wav (http://www.freesound.org/people/klankbeeld/sounds/189836/) by klankbeeld [CC-BY]\n- [4-189838-A.ogg]: clip derived from audience clapping 10 130525_13.wav (http://www.freesound.org/people/klankbeeld/sounds/189838/) by klankbeeld [CC-BY]\n- [4-205738-A.ogg]: clip derived from clapping.wav (http://www.freesound.org/people/SamKolber/sounds/205738/) by SamKolber [CC-BY]\n- [5-209989-A.ogg]: clip derived from Small Clap (http://www.freesound.org/people/kellieskitchen/sounds/209989/) by kellieskitchen [CC-BY]\n- [5-218494-A.ogg]: clip derived from 04807 strong clapping - 100 people.wav (http://www.freesound.org/people/Robinhood76/sounds/218494/) by Robinhood76 [CC-BY-NC]\n- [5-221567-A.ogg]: clip derived from cheering and clapping crowd 2 (http://www.freesound.org/people/AlaskaRobotics/sounds/221567/) by AlaskaRobotics [CC0]\n- [5-221568-A.ogg]: clip derived from cheering and clapping crowd 1 (http://www.freesound.org/people/AlaskaRobotics/sounds/221568/) by AlaskaRobotics [CC0]\n- [5-221950-A.ogg]: clip derived from Soft clapping (http://www.freesound.org/people/sonsdebarcelona/sounds/221950/) by sonsdebarcelona [CC-BY]\n- [5-249937-A.ogg]: clip derived from Crowd clapping fast.wav (http://www.freesound.org/people/IllusiaProductions/sounds/249937/) by IllusiaProductions [CC-BY]\n- [5-254160-A.ogg]: clip derived from Audience Clapping.wav (http://www.freesound.org/people/rivernile7/sounds/254160/) by rivernile7 [CC-BY]\n- [5-262957-A.ogg]: clip derived from Audience_Clapping_Int_1.wav (http://www.freesound.org/people/Cell31_Sound_Productions/sounds/262957/) by Cell31_Sound_Productions [CC-BY]\n\n304 - Breathing:\n- [1-18631-A.ogg]: clip derived from female deep breathing.aif (http://www.freesound.org/people/Walter_Odington/sounds/18631/) by Walter_Odington [CC0]\n- [1-30709-A.ogg]: clip derived from breath.wav (http://www.freesound.org/people/UncleSigmund/sounds/30709/) by UncleSigmund [CC0]\n- [1-30709-B.ogg]: clip derived from breath.wav (http://www.freesound.org/people/UncleSigmund/sounds/30709/) by UncleSigmund [CC0]\n- [1-30709-C.ogg]: clip derived from breath.wav (http://www.freesound.org/people/UncleSigmund/sounds/30709/) by UncleSigmund [CC0]\n- [1-36393-A.ogg]: clip derived from breaths_solo1.aif (http://www.freesound.org/people/thanvannispen/sounds/36393/) by thanvannispen [CC-BY]\n- [1-36397-A.ogg]: clip derived from breaths_solo4.aif (http://www.freesound.org/people/thanvannispen/sounds/36397/) by thanvannispen [CC-BY]\n- [1-36400-A.ogg]: clip derived from breaths_solo7.aif (http://www.freesound.org/people/thanvannispen/sounds/36400/) by thanvannispen [CC-BY]\n- [1-36402-A.ogg]: clip derived from breaths_solo9.aif (http://www.freesound.org/people/thanvannispen/sounds/36402/) by thanvannispen [CC-BY]\n- [2-50774-A.ogg]: clip derived from breath2.ogg (http://www.freesound.org/people/smcameron/sounds/50774/) by smcameron [CC-BY]\n- [2-54961-A.ogg]: clip derived from Breath Human Sci.wav (http://www.freesound.org/people/cidchili/sounds/54961/) by cidchili [CC0]\n- [2-54962-A.ogg]: clip derived from Breath human Sci2.wav (http://www.freesound.org/people/cidchili/sounds/54962/) by cidchili [CC0]\n- [2-66205-A.ogg]: clip derived from Breathe01.wav (http://www.freesound.org/people/JCH321/sounds/66205/) by JCH321 [CC-BY]\n- [2-82455-A.ogg]: clip derived from 01315 tunnel breath.wav (http://www.freesound.org/people/Robinhood76/sounds/82455/) by Robinhood76 [CC-BY-NC]\n- [2-84965-A.ogg]: clip derived from breath ambiance.wav (http://www.freesound.org/people/Ebrown15/sounds/84965/) by Ebrown15 [CC-Sampling+]\n- [2-95567-A.ogg]: clip derived from breathing_female.wav (http://www.freesound.org/people/Kuroseishin/sounds/95567/) by Kuroseishin [CC0]\n- [2-98392-A.ogg]: clip derived from Breath 06.wav (http://www.freesound.org/people/The%20Baron/sounds/98392/) by The Baron [CC-BY]\n- [3-108160-A.ogg]: clip derived from Breath_Woman.wav (http://www.freesound.org/people/audione/sounds/108160/) by audione [CC0]\n- [3-112557-A.ogg]: clip derived from Frightened Breathing.aiff (http://www.freesound.org/people/nickrave/sounds/112557/) by nickrave [CC0]\n- [3-112557-B.ogg]: clip derived from Frightened Breathing.aiff (http://www.freesound.org/people/nickrave/sounds/112557/) by nickrave [CC0]\n- [3-144128-A.ogg]: clip derived from Women breath. Desirable. Dry+Reverb (http://www.freesound.org/people/Pashee/sounds/144128/) by Pashee [CC-BY]\n- [3-144128-B.ogg]: clip derived from Women breath. Desirable. Dry+Reverb (http://www.freesound.org/people/Pashee/sounds/144128/) by Pashee [CC-BY]\n- [3-151206-A.ogg]: clip derived from Female Breathing heavily (http://www.freesound.org/people/OwlStorm/sounds/151206/) by OwlStorm [CC-BY]\n- [3-166125-A.ogg]: clip derived from Breathing_scared.wav (http://www.freesound.org/people/Adam_N/sounds/166125/) by Adam_N [CC0]\n- [3-166125-B.ogg]: clip derived from Breathing_scared.wav (http://www.freesound.org/people/Adam_N/sounds/166125/) by Adam_N [CC0]\n- [4-171706-A.ogg]: clip derived from 03897 breath effort loop.wav (http://www.freesound.org/people/Robinhood76/sounds/171706/) by Robinhood76 [CC-BY-NC]\n- [4-176914-A.ogg]: clip derived from double breath 2.wav (http://www.freesound.org/people/mickmon/sounds/176914/) by mickmon [CC-BY]\n- [4-198025-A.ogg]: clip derived from Breathing (http://www.freesound.org/people/gogo199432/sounds/198025/) by gogo199432 [CC-BY-NC]\n- [4-205526-A.ogg]: clip derived from Breathing.mp3 (http://www.freesound.org/people/movethemoon/sounds/205526/) by movethemoon [CC0]\n- [4-205526-B.ogg]: clip derived from Breathing.mp3 (http://www.freesound.org/people/movethemoon/sounds/205526/) by movethemoon [CC0]\n- [4-207116-A.ogg]: clip derived from man_breathing_asleep.wav (http://www.freesound.org/people/harleto/sounds/207116/) by harleto [CC0]\n- [4-210000-A.ogg]: clip derived from Man breathing.wav (http://www.freesound.org/people/jason130178/sounds/210000/) by jason130178 [CC-BY]\n- [4-210000-B.ogg]: clip derived from Man breathing.wav (http://www.freesound.org/people/jason130178/sounds/210000/) by jason130178 [CC-BY]\n- [5-213293-A.ogg]: clip derived from Shaky breaths - frightened scared young lady girl teenager.wav (http://www.freesound.org/people/AderuMoro/sounds/213293/) by AderuMoro [CC-BY]\n- [5-232816-A.ogg]: clip derived from Contained Breath (http://www.freesound.org/people/xavimuse/sounds/232816/) by xavimuse [CC0]\n- [5-233260-A.ogg]: clip derived from Contained breathing (http://www.freesound.org/people/xavimuse/sounds/233260/) by xavimuse [CC0]\n- [5-234335-A.ogg]: clip derived from Breath (http://www.freesound.org/people/mcroce/sounds/234335/) by mcroce [CC0]\n- [5-235593-A.ogg]: clip derived from Girl_Heavy_Breathing.wav (http://www.freesound.org/people/tcrocker68/sounds/235593/) by tcrocker68 [CC0]\n- [5-238492-A.ogg]: clip derived from 3_Breath sport.wav (http://www.freesound.org/people/mariiao2/sounds/238492/) by mariiao2 [CC0]\n- [5-260164-A.ogg]: clip derived from man breathing 140804_0339.wav (http://www.freesound.org/people/klankbeeld/sounds/260164/) by klankbeeld [CC-BY]\n- [5-261464-A.ogg]: clip derived from Werewolf breathing and sniffing  (http://www.freesound.org/people/kd_jack/sounds/261464/) by kd_jack [CC0]\n\n305 - Coughing:\n- [1-19111-A.ogg]: clip derived from cough1.aiff (http://www.freesound.org/people/Fratz/sounds/19111/) by Fratz [CC-BY]\n- [1-19118-A.ogg]: clip derived from cough3.aiff (http://www.freesound.org/people/Fratz/sounds/19118/) by Fratz [CC-BY]\n- [1-30830-A.ogg]: clip derived from girl_coughing.mp3 (http://www.freesound.org/people/JollySea/sounds/30830/) by JollySea [CC-BY]\n- [1-52266-A.ogg]: clip derived from flemmy cough.wav (http://www.freesound.org/people/dparke4/sounds/52266/) by dparke4 [CC-Sampling+]\n- [1-52323-A.ogg]: clip derived from coughing.wav (http://www.freesound.org/people/stephsinger22/sounds/52323/) by stephsinger22 [CC-Sampling+]\n- [1-53663-A.ogg]: clip derived from cough.wav (http://www.freesound.org/people/crispy527/sounds/53663/) by crispy527 [CC-Sampling+]\n- [1-58792-A.ogg]: clip derived from cough.wav (http://www.freesound.org/people/Syna-Max/sounds/58792/) by Syna-Max [CC-BY-NC]\n- [1-63679-A.ogg]: clip derived from 1125coughing.wav (http://www.freesound.org/people/sagetyrtle/sounds/63679/) by sagetyrtle [CC0]\n- [2-108017-A.ogg]: clip derived from Cough.wav (http://www.freesound.org/people/ErocShadow/sounds/108017/) by ErocShadow [CC-Sampling+]\n- [2-123896-A.ogg]: clip derived from cough.wav (http://www.freesound.org/people/Sempoo/sounds/123896/) by Sempoo [CC0]\n- [2-85292-A.ogg]: clip derived from file0281 cough.mp3 (http://www.freesound.org/people/smokeyvw/sounds/85292/) by smokeyvw [CC-BY]\n- [2-87412-A.ogg]: clip derived from woman cough.wav (http://www.freesound.org/people/bowlingballout/sounds/87412/) by bowlingballout [CC0]\n- [2-87794-A.ogg]: clip derived from tos1(16.01.2009).ogg (http://www.freesound.org/people/aneolus/sounds/87794/) by aneolus [CC0]\n- [2-87795-A.ogg]: clip derived from tos2(16.01.2009).ogg (http://www.freesound.org/people/aneolus/sounds/87795/) by aneolus [CC0]\n- [2-87799-A.ogg]: clip derived from tos6(16.01.2009).ogg (http://www.freesound.org/people/aneolus/sounds/87799/) by aneolus [CC0]\n- [2-98676-A.ogg]: clip derived from distant cough.mp3 (http://www.freesound.org/people/TheWah/sounds/98676/) by TheWah [CC0]\n- [3-125418-A.ogg]: clip derived from Man Coughing.wav (http://www.freesound.org/people/jyoung20/sounds/125418/) by jyoung20 [CC0]\n- [3-132601-A.ogg]: clip derived from Cough.wav (http://www.freesound.org/people/CarlosSolano019/sounds/132601/) by CarlosSolano019 [CC-BY]\n- [3-142604-A.ogg]: clip derived from cough1.mp3 (http://www.freesound.org/people/loisgwenllian/sounds/142604/) by loisgwenllian [CC0]\n- [3-145487-A.ogg]: clip derived from 0005 Cough (http://www.freesound.org/people/bmoreno/sounds/145487/) by bmoreno [CC-BY]\n- [3-146873-A.ogg]: clip derived from Male sick cough (http://www.freesound.org/people/tgfcoder/sounds/146873/) by tgfcoder [CC0]\n- [3-149042-A.ogg]: clip derived from manCough2.wav (http://www.freesound.org/people/Zabuhailo/sounds/149042/) by Zabuhailo [CC-BY]\n- [3-151212-A.ogg]: clip derived from Cough (2) (http://www.freesound.org/people/OwlStorm/sounds/151212/) by OwlStorm [CC-BY]\n- [3-151213-A.ogg]: clip derived from Cough Female (http://www.freesound.org/people/OwlStorm/sounds/151213/) by OwlStorm [CC-BY]\n- [4-152995-A.ogg]: clip derived from Cough.wav (http://www.freesound.org/people/Nmb910/sounds/152995/) by Nmb910 [CC-BY]\n- [4-154443-A.ogg]: clip derived from Cough 2.wav (http://www.freesound.org/people/ecfike/sounds/154443/) by ecfike [CC0]\n- [4-155650-A.ogg]: clip derived from Cough (http://www.freesound.org/people/poorenglishjuggler/sounds/155650/) by poorenglishjuggler [CC0]\n- [4-155650-B.ogg]: clip derived from Cough (http://www.freesound.org/people/poorenglishjuggler/sounds/155650/) by poorenglishjuggler [CC0]\n- [4-157296-A.ogg]: clip derived from Coughing Cough.wav (http://www.freesound.org/people/husky70/sounds/157296/) by husky70 [CC0]\n- [4-169726-A.ogg]: clip derived from Cough! (http://www.freesound.org/people/qubodup/sounds/169726/) by qubodup [CC0]\n- [4-171396-A.ogg]: clip derived from Wet Cough (http://www.freesound.org/people/unfa/sounds/171396/) by unfa [CC-BY]\n- [4-177835-A.ogg]: clip derived from Cough Funny (http://www.freesound.org/people/unfa/sounds/177835/) by unfa [CC-BY]\n- [5-178997-A.ogg]: clip derived from Coughing.wav (http://www.freesound.org/people/BIGTexan7213/sounds/178997/) by BIGTexan7213 [CC-BY]\n- [5-184871-A.ogg]: clip derived from coughing (http://www.freesound.org/people/Eelke/sounds/184871/) by Eelke [CC0]\n- [5-204604-A.ogg]: clip derived from Cough (http://www.freesound.org/people/pikachu09/sounds/204604/) by pikachu09 [CC0]\n- [5-207681-A.ogg]: clip derived from Cough.mp3 (http://www.freesound.org/people/blymannen/sounds/207681/) by blymannen [CC-BY-NC]\n- [5-208761-A.ogg]: clip derived from Cough.wav (http://www.freesound.org/people/Harris85/sounds/208761/) by Harris85 [CC0]\n- [5-209719-A.ogg]: clip derived from Cough (http://www.freesound.org/people/Bramdv/sounds/209719/) by Bramdv [CC0]\n- [5-211197-A.ogg]: clip derived from Real Flu Chest Cold Coughing Male Middle Aged.MP3 (http://www.freesound.org/people/squeakyballoon/sounds/211197/) by squeakyballoon [CC-BY]\n- [5-251489-A.ogg]: clip derived from Cough - young lady.wav (http://www.freesound.org/people/AderuMoro/sounds/251489/) by AderuMoro [CC-BY]\n\n306 - Footsteps:\n- [1-155858-A.ogg]: clip derived from Footsteps in Factory Hall on Wood and Concrete.wav (http://www.freesound.org/people/RutgerMuller/sounds/155858/) by RutgerMuller [CC0]\n- [1-155858-B.ogg]: clip derived from Footsteps in Factory Hall on Wood and Concrete.wav (http://www.freesound.org/people/RutgerMuller/sounds/155858/) by RutgerMuller [CC0]\n- [1-155858-C.ogg]: clip derived from Footsteps in Factory Hall on Wood and Concrete.wav (http://www.freesound.org/people/RutgerMuller/sounds/155858/) by RutgerMuller [CC0]\n- [1-155858-D.ogg]: clip derived from Footsteps in Factory Hall on Wood and Concrete.wav (http://www.freesound.org/people/RutgerMuller/sounds/155858/) by RutgerMuller [CC0]\n- [1-155858-E.ogg]: clip derived from Footsteps in Factory Hall on Wood and Concrete.wav (http://www.freesound.org/people/RutgerMuller/sounds/155858/) by RutgerMuller [CC0]\n- [1-155858-F.ogg]: clip derived from Footsteps in Factory Hall on Wood and Concrete.wav (http://www.freesound.org/people/RutgerMuller/sounds/155858/) by RutgerMuller [CC0]\n- [1-223162-A.ogg]: clip derived from wooden panel footstep 3 (http://www.freesound.org/people/Yoyodaman234/sounds/223162/) by Yoyodaman234 [CC0]\n- [1-51147-A.ogg]: clip derived from Wet Footsteps in Small Shower Cabin.wav (http://www.freesound.org/people/RutgerMuller/sounds/51147/) by RutgerMuller [CC0]\n- [2-209471-A.ogg]: clip derived from footsteps_down_stairs_1.WAV (http://www.freesound.org/people/sinatra314/sounds/209471/) by sinatra314 [CC-BY]\n- [2-209472-A.ogg]: clip derived from footsteps_down_stairs_5.WAV (http://www.freesound.org/people/sinatra314/sounds/209472/) by sinatra314 [CC-BY]\n- [2-209473-A.ogg]: clip derived from footsteps_down_stairs_4.WAV (http://www.freesound.org/people/sinatra314/sounds/209473/) by sinatra314 [CC-BY]\n- [2-209474-A.ogg]: clip derived from footsteps_down_stairs_3.WAV (http://www.freesound.org/people/sinatra314/sounds/209474/) by sinatra314 [CC-BY]\n- [2-209475-A.ogg]: clip derived from footsteps_down_stairs_2.WAV (http://www.freesound.org/people/sinatra314/sounds/209475/) by sinatra314 [CC-BY]\n- [2-209476-A.ogg]: clip derived from footsteps_up_stairs_3.WAV (http://www.freesound.org/people/sinatra314/sounds/209476/) by sinatra314 [CC-BY]\n- [2-209477-A.ogg]: clip derived from footsteps_up_stairs_2.WAV (http://www.freesound.org/people/sinatra314/sounds/209477/) by sinatra314 [CC-BY]\n- [2-209478-A.ogg]: clip derived from footsteps_up_stairs_1.WAV (http://www.freesound.org/people/sinatra314/sounds/209478/) by sinatra314 [CC-BY]\n- [3-103597-A.ogg]: clip derived from walking-in-a-hut-with-stream.mp3 (http://www.freesound.org/people/floating%20tree/sounds/103597/) by floating tree [CC-BY-NC]\n- [3-103598-A.ogg]: clip derived from walking-in-the-lone-shieling.mp3 (http://www.freesound.org/people/floating%20tree/sounds/103598/) by floating tree [CC-BY-NC]\n- [3-103599-A.ogg]: clip derived from walking-lone-shieling-quack.mp3 (http://www.freesound.org/people/floating%20tree/sounds/103599/) by floating tree [CC-BY-NC]\n- [3-103599-B.ogg]: clip derived from walking-lone-shieling-quack.mp3 (http://www.freesound.org/people/floating%20tree/sounds/103599/) by floating tree [CC-BY-NC]\n- [3-249913-A.ogg]: clip derived from walking slow on the street (http://www.freesound.org/people/hintringer/sounds/249913/) by hintringer [CC0]\n- [3-94342-A.ogg]: clip derived from Fotsteg i p\\xe5 v\\xe5ta l\\xf6v.wav (http://www.freesound.org/people/Ryding/sounds/94342/) by Ryding [CC0]\n- [3-94343-A.ogg]: clip derived from Fotsteg p\\xe5 asfalt.wav (http://www.freesound.org/people/Ryding/sounds/94343/) by Ryding [CC0]\n- [3-94344-A.ogg]: clip derived from Fotsteg p\\xe5 grus.wav (http://www.freesound.org/people/Ryding/sounds/94344/) by Ryding [CC0]\n- [4-117627-A.ogg]: clip derived from footsteps - gravel pavement.wav (http://www.freesound.org/people/soundmary/sounds/117627/) by soundmary [CC0]\n- [4-117630-A.ogg]: clip derived from footsteps woodland.wav (http://www.freesound.org/people/soundmary/sounds/117630/) by soundmary [CC0]\n- [4-194979-A.ogg]: clip derived from footsteps in long grass.mp3 (http://www.freesound.org/people/soundmary/sounds/194979/) by soundmary [CC-BY]\n- [4-194981-A.ogg]: clip derived from footsteps on snow .mp3 (http://www.freesound.org/people/soundmary/sounds/194981/) by soundmary [CC-BY]\n- [4-198962-A.ogg]: clip derived from footsteps on wood (http://www.freesound.org/people/Mydo1/sounds/198962/) by Mydo1 [CC0]\n- [4-198962-B.ogg]: clip derived from footsteps on wood (http://www.freesound.org/people/Mydo1/sounds/198962/) by Mydo1 [CC0]\n- [4-218304-A.ogg]: clip derived from 01-21 Footsteps, sneakers on gravel and leaves.wav (http://www.freesound.org/people/MentalSanityOff/sounds/218304/) by MentalSanityOff [CC0]\n- [4-218304-B.ogg]: clip derived from 01-21 Footsteps, sneakers on gravel and leaves.wav (http://www.freesound.org/people/MentalSanityOff/sounds/218304/) by MentalSanityOff [CC0]\n- [5-234263-A.ogg]: clip derived from Footsteps - Wood (http://www.freesound.org/people/Fewes/sounds/234263/) by Fewes [CC-BY]\n- [5-234855-A.ogg]: clip derived from Footsteps_Carpet.wav (http://www.freesound.org/people/mlsulli/sounds/234855/) by mlsulli [CC0]\n- [5-243025-A.ogg]: clip derived from Dirt & Rocks Footsteps SFX #3.wav (http://www.freesound.org/people/AdamWayneGistarb/sounds/243025/) by AdamWayneGistarb [CC-BY]\n- [5-244310-A.ogg]: clip derived from Walking on pavement.wav (http://www.freesound.org/people/JohnsonBrandEditing/sounds/244310/) by JohnsonBrandEditing [CC0]\n- [5-263490-A.ogg]: clip derived from footsteps snow wet crunchy.wav (http://www.freesound.org/people/lwdickens/sounds/263490/) by lwdickens [CC0]\n- [5-263491-A.ogg]: clip derived from footsteps crunchy ice.wav (http://www.freesound.org/people/lwdickens/sounds/263491/) by lwdickens [CC0]\n- [5-263501-A.ogg]: clip derived from footsteps wet snow.wav (http://www.freesound.org/people/lwdickens/sounds/263501/) by lwdickens [CC0]\n- [5-51149-A.ogg]: clip derived from Footsteps Wooden Stairs Squeaking 1.wav (http://www.freesound.org/people/RutgerMuller/sounds/51149/) by RutgerMuller [CC0]\n\n307 - Laughing:\n- [1-1791-A.ogg]: clip derived from stevenClayLaughLoop.wav (http://www.freesound.org/people/nicStage/sounds/1791/) by nicStage [CC-BY]\n- [1-30039-A.ogg]: clip derived from small group laugh 2.flac (http://www.freesound.org/people/Corsica_S/sounds/30039/) by Corsica_S [CC-BY]\n- [1-30043-A.ogg]: clip derived from small group laugh 6.flac (http://www.freesound.org/people/Corsica_S/sounds/30043/) by Corsica_S [CC-BY]\n- [1-33658-A.ogg]: clip derived from laughter.wav (http://www.freesound.org/people/sagetyrtle/sounds/33658/) by sagetyrtle [CC0]\n- [1-36164-A.ogg]: clip derived from laughs2.wav (http://www.freesound.org/people/sagetyrtle/sounds/36164/) by sagetyrtle [CC0]\n- [1-36164-B.ogg]: clip derived from laughs2.wav (http://www.freesound.org/people/sagetyrtle/sounds/36164/) by sagetyrtle [CC0]\n- [1-72695-A.ogg]: clip derived from laugh5_woman.wav (http://www.freesound.org/people/sagetyrtle/sounds/72695/) by sagetyrtle [CC0]\n- [1-73123-A.ogg]: clip derived from 0524laugh.wav (http://www.freesound.org/people/sagetyrtle/sounds/73123/) by sagetyrtle [CC0]\n- [2-109759-A.ogg]: clip derived from girl-laugh.wav (http://www.freesound.org/people/choplin/sounds/109759/) by choplin [CC-BY]\n- [2-109759-B.ogg]: clip derived from girl-laugh.wav (http://www.freesound.org/people/choplin/sounds/109759/) by choplin [CC-BY]\n- [2-60791-A.ogg]: clip derived from Few short Laughs and mighty Laughs.mp3 (http://www.freesound.org/people/J.Zazvurek/sounds/60791/) by J.Zazvurek [CC-BY]\n- [2-60794-A.ogg]: clip derived from Laugh 10.mp3 (http://www.freesound.org/people/J.Zazvurek/sounds/60794/) by J.Zazvurek [CC-BY]\n- [2-60795-A.ogg]: clip derived from Laugh 11.mp3 (http://www.freesound.org/people/J.Zazvurek/sounds/60795/) by J.Zazvurek [CC-BY]\n- [2-62226-A.ogg]: clip derived from 00512 male group laugh 2.wav (http://www.freesound.org/people/Robinhood76/sounds/62226/) by Robinhood76 [CC-BY-NC]\n- [2-79769-A.ogg]: clip derived from CrazyLaugh.wav (http://www.freesound.org/people/engreitz/sounds/79769/) by engreitz [CC-Sampling+]\n- [2-79775-A.ogg]: clip derived from SilentLaugh.wav (http://www.freesound.org/people/engreitz/sounds/79775/) by engreitz [CC-Sampling+]\n- [3-107123-A.ogg]: clip derived from laughing_dasha.wav (http://www.freesound.org/people/tagirov/sounds/107123/) by tagirov [CC-BY]\n- [3-110536-A.ogg]: clip derived from www.soundbyter.com_femalelaugh.wav (http://www.freesound.org/people/soundbyter.com/sounds/110536/) by soundbyter.com [CC-Sampling+]\n- [3-118487-A.ogg]: clip derived from FemaleLaughing.wav (http://www.freesound.org/people/morningfrost/sounds/118487/) by morningfrost [CC0]\n- [3-119459-A.ogg]: clip derived from Male Laughter.wav (http://www.freesound.org/people/Eneasz/sounds/119459/) by Eneasz [CC0]\n- [3-126113-A.ogg]: clip derived from laugh.WAV (http://www.freesound.org/people/klankbeeld/sounds/126113/) by klankbeeld [CC-BY]\n- [3-132747-A.ogg]: clip derived from evil laugh-04.wav (http://www.freesound.org/people/Nanakisan/sounds/132747/) by Nanakisan [CC-BY]\n- [3-152912-A.ogg]: clip derived from female_short_laugh_02 (http://www.freesound.org/people/fmaudio/sounds/152912/) by fmaudio [CC-BY]\n- [3-152997-A.ogg]: clip derived from Female_laugh_04.wav (http://www.freesound.org/people/fmaudio/sounds/152997/) by fmaudio [CC-BY]\n- [4-132803-A.ogg]: clip derived from evil laugh-07.wav (http://www.freesound.org/people/Nanakisan/sounds/132803/) by Nanakisan [CC-BY]\n- [4-132810-A.ogg]: clip derived from evil laugh-14.wav (http://www.freesound.org/people/Nanakisan/sounds/132810/) by Nanakisan [CC-BY]\n- [4-132816-A.ogg]: clip derived from evil laugh-19.wav (http://www.freesound.org/people/Nanakisan/sounds/132816/) by Nanakisan [CC-BY]\n- [4-133674-A.ogg]: clip derived from horror laugh original - 132802__nanakisan__evil-laugh-08.wav (http://www.freesound.org/people/klankbeeld/sounds/133674/) by klankbeeld [CC-BY]\n- [4-155670-A.ogg]: clip derived from children_laugh.mp3 (http://www.freesound.org/people/totya/sounds/155670/) by totya [CC-BY-NC]\n- [4-164243-A.ogg]: clip derived from Young girl laughing uncontrollably.mp3 (http://www.freesound.org/people/caculo/sounds/164243/) by caculo [CC0]\n- [4-167571-A.ogg]: clip derived from evil-laugh.wav (http://www.freesound.org/people/dlovere/sounds/167571/) by dlovere [CC0]\n- [4-181599-A.ogg]: clip derived from Laugh.wav (http://www.freesound.org/people/JulieBaka/sounds/181599/) by JulieBaka [CC0]\n- [5-171118-A.ogg]: clip derived from laugh5.wav (http://www.freesound.org/people/ondrosik/sounds/171118/) by ondrosik [CC-BY-NC]\n- [5-242932-A.ogg]: clip derived from Child Laughing (wav file) (http://www.freesound.org/people/OBXJohn/sounds/242932/) by OBXJohn [CC-BY]\n- [5-242932-B.ogg]: clip derived from Child Laughing (wav file) (http://www.freesound.org/people/OBXJohn/sounds/242932/) by OBXJohn [CC-BY]\n- [5-244526-A.ogg]: clip derived from laugh.mp3 (http://www.freesound.org/people/smartxxd/sounds/244526/) by smartxxd [CC0]\n- [5-253534-A.ogg]: clip derived from Evil Laugh 9.wav (http://www.freesound.org/people/Nanakisan/sounds/253534/) by Nanakisan [CC-BY]\n- [5-259514-A.ogg]: clip derived from Cackling Laugh.mp3 (http://www.freesound.org/people/fattirewhitey/sounds/259514/) by fattirewhitey [CC0]\n- [5-263775-A.ogg]: clip derived from Laughing Man (http://www.freesound.org/people/scottemoil/sounds/263775/) by scottemoil [CC0]\n- [5-263775-B.ogg]: clip derived from Laughing Man (http://www.freesound.org/people/scottemoil/sounds/263775/) by scottemoil [CC0]\n\n308 - Brushing teeth:\n- [1-17092-A.ogg]: clip derived from sw_brushing_teeth.wav (http://www.freesound.org/people/jppi_Stu/sounds/17092/) by jppi_Stu [CC-BY]\n- [1-17092-B.ogg]: clip derived from sw_brushing_teeth.wav (http://www.freesound.org/people/jppi_Stu/sounds/17092/) by jppi_Stu [CC-BY]\n- [1-45641-A.ogg]: clip derived from brushingteeth.mp3 (http://www.freesound.org/people/stomachache/sounds/45641/) by stomachache [CC0]\n- [1-58923-A.ogg]: clip derived from brushingteeth.mp3 (http://www.freesound.org/people/morgantj/sounds/58923/) by morgantj [CC-BY]\n- [1-58923-B.ogg]: clip derived from brushingteeth.mp3 (http://www.freesound.org/people/morgantj/sounds/58923/) by morgantj [CC-BY]\n- [1-61534-A.ogg]: clip derived from brushing teeth.wav (http://www.freesound.org/people/cognito%20perceptu/sounds/61534/) by cognito perceptu [CC0]\n- [1-67432-A.ogg]: clip derived from 00869 brushing teeth 1.wav (http://www.freesound.org/people/Robinhood76/sounds/67432/) by Robinhood76 [CC-BY-NC]\n- [1-68628-A.ogg]: clip derived from brushing teeth with noise in background.wav (http://www.freesound.org/people/bwav/sounds/68628/) by bwav [CC0]\n- [2-73544-A.ogg]: clip derived from toothbrush.aif (http://www.freesound.org/people/sunredcity/sounds/73544/) by sunredcity [CC-BY]\n- [2-82071-A.ogg]: clip derived from 00096_C.wav (http://www.freesound.org/people/kalashnikov_v2/sounds/82071/) by kalashnikov_v2 [CC-BY-NC]\n- [2-83536-A.ogg]: clip derived from Teeth brushing.mp3 (http://www.freesound.org/people/bsumusictech/sounds/83536/) by bsumusictech [CC0]\n- [2-85434-A.ogg]: clip derived from Brushing Teeth.aif (http://www.freesound.org/people/JasonElrod/sounds/85434/) by JasonElrod [CC-BY]\n- [2-86160-A.ogg]: clip derived from Brushing Teeth_1-2.aif (http://www.freesound.org/people/SteveMannella/sounds/86160/) by SteveMannella [CC-Sampling+]\n- [2-92627-A.ogg]: clip derived from Brushing teeth.mp3 (http://www.freesound.org/people/CGEffex/sounds/92627/) by CGEffex [CC-BY]\n- [2-94230-A.ogg]: clip derived from Brushing Teeth.wav (http://www.freesound.org/people/gchand1/sounds/94230/) by gchand1 [CC-Sampling+]\n- [2-96904-A.ogg]: clip derived from Brushing teeth.wav (http://www.freesound.org/people/Ohrwurm/sounds/96904/) by Ohrwurm [CC0]\n- [3-100024-A.ogg]: clip derived from teeth brushing.aif (http://www.freesound.org/people/limwei/sounds/100024/) by limwei [CC-BY]\n- [3-100024-B.ogg]: clip derived from teeth brushing.aif (http://www.freesound.org/people/limwei/sounds/100024/) by limwei [CC-BY]\n- [3-112522-A.ogg]: clip derived from DM House Sounds 001 Brushing Teeth.wav (http://www.freesound.org/people/DaveMatney/sounds/112522/) by DaveMatney [CC-BY]\n- [3-118069-A.ogg]: clip derived from brushteeth.wav (http://www.freesound.org/people/ipears1/sounds/118069/) by ipears1 [CC0]\n- [3-118069-B.ogg]: clip derived from brushteeth.wav (http://www.freesound.org/people/ipears1/sounds/118069/) by ipears1 [CC0]\n- [3-126391-A.ogg]: clip derived from toothbrush-analog-brushing-teeth-spit.aif (http://www.freesound.org/people/Eelke/sounds/126391/) by Eelke [CC-BY-NC]\n- [3-126391-B.ogg]: clip derived from toothbrush-analog-brushing-teeth-spit.aif (http://www.freesound.org/people/Eelke/sounds/126391/) by Eelke [CC-BY-NC]\n- [3-139331-A.ogg]: clip derived from Brushing teeth.wav (http://www.freesound.org/people/FunnyMan374/sounds/139331/) by FunnyMan374 [CC0]\n- [4-144468-A.ogg]: clip derived from Brushing of the Teeth.wav (http://www.freesound.org/people/steviebones/sounds/144468/) by steviebones [CC0]\n- [4-144468-B.ogg]: clip derived from Brushing of the Teeth.wav (http://www.freesound.org/people/steviebones/sounds/144468/) by steviebones [CC0]\n- [4-154405-A.ogg]: clip derived from Brushing Teeth_1-2.aif (http://www.freesound.org/people/lucaslara/sounds/154405/) by lucaslara [CC0]\n- [4-154405-B.ogg]: clip derived from Brushing Teeth_1-2.aif (http://www.freesound.org/people/lucaslara/sounds/154405/) by lucaslara [CC0]\n- [4-172500-A.ogg]: clip derived from Binaural Toothbrush (http://www.freesound.org/people/stomachache/sounds/172500/) by stomachache [CC-BY]\n- [4-172500-B.ogg]: clip derived from Binaural Toothbrush (http://www.freesound.org/people/stomachache/sounds/172500/) by stomachache [CC-BY]\n- [4-172500-C.ogg]: clip derived from Binaural Toothbrush (http://www.freesound.org/people/stomachache/sounds/172500/) by stomachache [CC-BY]\n- [4-172500-D.ogg]: clip derived from Binaural Toothbrush (http://www.freesound.org/people/stomachache/sounds/172500/) by stomachache [CC-BY]\n- [5-147297-A.ogg]: clip derived from Teeth brushing.wav (http://www.freesound.org/people/JeanneCousseau/sounds/147297/) by JeanneCousseau [CC-BY]\n- [5-180229-A.ogg]: clip derived from teethbrushing.WAV (http://www.freesound.org/people/lindhad95/sounds/180229/) by lindhad95 [CC-BY-NC]\n- [5-185516-A.ogg]: clip derived from Brushing Teeth -  Without fallingWater On Sink - Escovando Dentes - Pia sem cair agua.wav (http://www.freesound.org/people/Leossom/sounds/185516/) by Leossom [CC-BY]\n- [5-212050-A.ogg]: clip derived from Brushing_teeth.mp3 (http://www.freesound.org/people/Taira%20Komori/sounds/212050/) by Taira Komori [CC-BY]\n- [5-218196-A.ogg]: clip derived from brushing teeth (http://www.freesound.org/people/edhutschek/sounds/218196/) by edhutschek [CC0]\n- [5-218196-B.ogg]: clip derived from brushing teeth (http://www.freesound.org/people/edhutschek/sounds/218196/) by edhutschek [CC0]\n- [5-220939-A.ogg]: clip derived from Brushing Teeth (http://www.freesound.org/people/nebulousflynn/sounds/220939/) by nebulousflynn [CC-BY]\n- [5-238938-A.ogg]: clip derived from Lavandome los dientes, Grupo 9 Karen y Paula.wav (http://www.freesound.org/people/flosicalcat59/sounds/238938/) by flosicalcat59 [CC-BY]\n\n309 - Snoring:\n- [1-20545-A.ogg]: clip derived from snore.wav (http://www.freesound.org/people/sirplus/sounds/20545/) by sirplus [CC0]\n- [1-27403-A.ogg]: clip derived from snore_okm2.flac (http://www.freesound.org/people/ermine/sounds/27403/) by ermine [CC-BY-NC]\n- [1-27405-A.ogg]: clip derived from snore_okm2_t3.flac (http://www.freesound.org/people/ermine/sounds/27405/) by ermine [CC-BY-NC]\n- [1-39937-A.ogg]: clip derived from 20070816.snoring.duo.00.wav (http://www.freesound.org/people/dobroide/sounds/39937/) by dobroide [CC-BY]\n- [1-40621-A.ogg]: clip derived from SleepingBeauty.wav (http://www.freesound.org/people/acclivity/sounds/40621/) by acclivity [CC-BY-NC]\n- [1-40967-A.ogg]: clip derived from 20070912.snoring.mp3 (http://www.freesound.org/people/dobroide/sounds/40967/) by dobroide [CC-BY]\n- [1-47923-A.ogg]: clip derived from snore.flac (http://www.freesound.org/people/Erdie/sounds/47923/) by Erdie [CC-BY]\n- [1-53444-A.ogg]: clip derived from Snoring.aiff (http://www.freesound.org/people/Brotherprovisional/sounds/53444/) by Brotherprovisional [CC-BY]\n- [2-110417-A.ogg]: clip derived from www.soundbyter.com-male-snore.wav (http://www.freesound.org/people/sound_byter/sounds/110417/) by sound_byter [CC-BY-NC]\n- [2-110417-B.ogg]: clip derived from www.soundbyter.com-male-snore.wav (http://www.freesound.org/people/sound_byter/sounds/110417/) by sound_byter [CC-BY-NC]\n- [2-114609-A.ogg]: clip derived from STE-000.wav (http://www.freesound.org/people/Daxter31/sounds/114609/) by Daxter31 [CC-BY]\n- [2-114609-B.ogg]: clip derived from STE-000.wav (http://www.freesound.org/people/Daxter31/sounds/114609/) by Daxter31 [CC-BY]\n- [2-117330-A.ogg]: clip derived from Ronflements.wav (http://www.freesound.org/people/steph64/sounds/117330/) by steph64 [CC-BY-NC]\n- [2-52001-A.ogg]: clip derived from Snorking#05.aif (http://www.freesound.org/people/scarfield/sounds/52001/) by scarfield [CC0]\n- [2-52001-B.ogg]: clip derived from Snorking#05.aif (http://www.freesound.org/people/scarfield/sounds/52001/) by scarfield [CC0]\n- [2-80313-A.ogg]: clip derived from 20090911.ronca.wav (http://www.freesound.org/people/dobroide/sounds/80313/) by dobroide [CC-BY]\n- [3-123086-A.ogg]: clip derived from russo rocco.wav (http://www.freesound.org/people/extrafonia/sounds/123086/) by extrafonia [CC-BY]\n- [3-124795-A.ogg]: clip derived from 107_snurken.aif (http://www.freesound.org/people/Greencouch/sounds/124795/) by Greencouch [CC-BY]\n- [3-124958-A.ogg]: clip derived from sleeping.wav (http://www.freesound.org/people/Kyster/sounds/124958/) by Kyster [CC-BY]\n- [3-130998-A.ogg]: clip derived from Snoring Symphony part 2 (http://www.freesound.org/people/frederic.font/sounds/130998/) by frederic.font [CC-BY]\n- [3-130998-B.ogg]: clip derived from Snoring Symphony part 2 (http://www.freesound.org/people/frederic.font/sounds/130998/) by frederic.font [CC-BY]\n- [3-151255-A.ogg]: clip derived from Snores / Ronquidos (http://www.freesound.org/people/lebosqui/sounds/151255/) by lebosqui [CC-BY]\n- [3-151557-A.ogg]: clip derived from Snoring (http://www.freesound.org/people/Erik%20Pritzens/sounds/151557/) by Erik Pritzens [CC0]\n- [3-151557-B.ogg]: clip derived from Snoring (http://www.freesound.org/people/Erik%20Pritzens/sounds/151557/) by Erik Pritzens [CC0]\n- [4-180337-A.ogg]: clip derived from Snoring 4 (http://www.freesound.org/people/sankalp/sounds/180337/) by sankalp [CC-BY]\n- [4-183882-A.ogg]: clip derived from snoring 04.01.11.wav (http://www.freesound.org/people/toiletrolltube/sounds/183882/) by toiletrolltube [CC0]\n- [4-183882-B.ogg]: clip derived from snoring 04.01.11.wav (http://www.freesound.org/people/toiletrolltube/sounds/183882/) by toiletrolltube [CC0]\n- [4-184235-A.ogg]: clip derived from Marshall snores.wav (http://www.freesound.org/people/thorvandahl/sounds/184235/) by thorvandahl [CC-BY]\n- [4-184237-A.ogg]: clip derived from Paolo snores.wav (http://www.freesound.org/people/thorvandahl/sounds/184237/) by thorvandahl [CC-BY]\n- [4-191297-A.ogg]: clip derived from Snoring (http://www.freesound.org/people/VlatkoBlazek/sounds/191297/) by VlatkoBlazek [CC-BY]\n- [4-197454-A.ogg]: clip derived from Snoring.wav (http://www.freesound.org/people/everythingsounds/sounds/197454/) by everythingsounds [CC-BY]\n- [4-197454-B.ogg]: clip derived from Snoring.wav (http://www.freesound.org/people/everythingsounds/sounds/197454/) by everythingsounds [CC-BY]\n- [5-216368-A.ogg]: clip derived from Snoring.wav (http://www.freesound.org/people/RSilveira_88/sounds/216368/) by RSilveira_88 [CC-BY]\n- [5-233312-A.ogg]: clip derived from Man Snoring - Freesound.wav (http://www.freesound.org/people/trentbh/sounds/233312/) by trentbh [CC0]\n- [5-234145-A.ogg]: clip derived from Dog Snores (http://www.freesound.org/people/mingusassin/sounds/234145/) by mingusassin [CC0]\n- [5-235874-A.ogg]: clip derived from SJUULKE SNORING 2.WAV (http://www.freesound.org/people/delphidebrain/sounds/235874/) by delphidebrain [CC-BY-NC]\n- [5-235893-A.ogg]: clip derived from SJUULKE SNORING 4.WAV (http://www.freesound.org/people/delphidebrain/sounds/235893/) by delphidebrain [CC-BY-NC]\n- [5-236288-A.ogg]: clip derived from human snore.wav (http://www.freesound.org/people/mffm/sounds/236288/) by mffm [CC0]\n- [5-244459-A.ogg]: clip derived from snoreing.mp3 (http://www.freesound.org/people/milton./sounds/244459/) by milton. [CC0]\n- [5-249748-A.ogg]: clip derived from Loud snoring (http://www.freesound.org/people/samvanzoest/sounds/249748/) by samvanzoest [CC-BY]\n\n310 - Drinking - sipping:\n- [1-17295-A.ogg]: clip derived from luffy_water2.wav (http://www.freesound.org/people/luffy/sounds/17295/) by luffy [CC-BY]\n- [1-32579-A.ogg]: clip derived from Swallowing.wav (http://www.freesound.org/people/redjim/sounds/32579/) by redjim [CC-BY-NC]\n- [1-37226-A.ogg]: clip derived from drinking.wav (http://www.freesound.org/people/sagetyrtle/sounds/37226/) by sagetyrtle [CC0]\n- [1-56270-A.ogg]: clip derived from Water_Drink.wav (http://www.freesound.org/people/Q.K./sounds/56270/) by Q.K. [CC0]\n- [1-67230-A.ogg]: clip derived from slurping.wav (http://www.freesound.org/people/fille1000/sounds/67230/) by fille1000 [CC-Sampling+]\n- [1-79146-A.ogg]: clip derived from Swallowing Water.wav (http://www.freesound.org/people/Razzvio/sounds/79146/) by Razzvio [CC-BY]\n- [1-85909-A.ogg]: clip derived from drinking in big gulps.aiff (http://www.freesound.org/people/mwmarsh/sounds/85909/) by mwmarsh [CC-Sampling+]\n- [1-87565-A.ogg]: clip derived from gulps.wav (http://www.freesound.org/people/SGAK/sounds/87565/) by SGAK [CC0]\n- [2-102581-A.ogg]: clip derived from Gulping liquid.wav (http://www.freesound.org/people/CGEffex/sounds/102581/) by CGEffex [CC-BY]\n- [2-102581-B.ogg]: clip derived from Gulping liquid.wav (http://www.freesound.org/people/CGEffex/sounds/102581/) by CGEffex [CC-BY]\n- [2-121978-A.ogg]: clip derived from Drink Soda.mp3 (http://www.freesound.org/people/isabeldandi/sounds/121978/) by isabeldandi [CC-BY]\n- [2-122763-A.ogg]: clip derived from gurgles.aif (http://www.freesound.org/people/alienistcog/sounds/122763/) by alienistcog [CC0]\n- [2-122763-B.ogg]: clip derived from gurgles.aif (http://www.freesound.org/people/alienistcog/sounds/122763/) by alienistcog [CC0]\n- [2-126756-A.ogg]: clip derived from london_drinking.wav (http://www.freesound.org/people/sbordage/sounds/126756/) by sbordage [CC-BY]\n- [2-92978-A.ogg]: clip derived from 20100322.gulp.wav (http://www.freesound.org/people/dobroide/sounds/92978/) by dobroide [CC-BY]\n- [2-94807-A.ogg]: clip derived from Slurping swallowing and ahh.wav (http://www.freesound.org/people/mwhitcomb83/sounds/94807/) by mwhitcomb83 [CC-Sampling+]\n- [3-133977-A.ogg]: clip derived from Gulp.mp3 (http://www.freesound.org/people/Alvinwhatup2/sounds/133977/) by Alvinwhatup2 [CC0]\n- [3-140323-A.ogg]: clip derived from Gulps (http://www.freesound.org/people/Project_Trident/sounds/140323/) by Project_Trident [CC-BY-NC]\n- [3-144253-A.ogg]: clip derived from Human-Man-Drinking-Soda_Can-01.wav (http://www.freesound.org/people/D%20W/sounds/144253/) by D W [CC-BY]\n- [3-144259-A.ogg]: clip derived from Human-Man-Drinking-Soda_Can-02.wav (http://www.freesound.org/people/D%20W/sounds/144259/) by D W [CC-BY]\n- [3-145387-A.ogg]: clip derived from SwallowingLiquid.aif (http://www.freesound.org/people/timgormly/sounds/145387/) by timgormly [CC0]\n- [3-169907-A.ogg]: clip derived from Eating Oatmeal (http://www.freesound.org/people/qubodup/sounds/169907/) by qubodup [CC0]\n- [3-174866-A.ogg]: clip derived from Throat Swallow Gulp (http://www.freesound.org/people/AshFox/sounds/174866/) by AshFox [CC-BY]\n- [3-185456-A.ogg]: clip derived from drink_water_glass.wav (http://www.freesound.org/people/mshahen/sounds/185456/) by mshahen [CC-BY]\n- [4-188191-A.ogg]: clip derived from Fast food soda with ice, sip slurp straw.wav (http://www.freesound.org/people/MentalSanityOff/sounds/188191/) by MentalSanityOff [CC0]\n- [4-188191-B.ogg]: clip derived from Fast food soda with ice, sip slurp straw.wav (http://www.freesound.org/people/MentalSanityOff/sounds/188191/) by MentalSanityOff [CC0]\n- [4-188191-C.ogg]: clip derived from Fast food soda with ice, sip slurp straw.wav (http://www.freesound.org/people/MentalSanityOff/sounds/188191/) by MentalSanityOff [CC0]\n- [4-188595-A.ogg]: clip derived from [DrinkingCan[Norberd]](http://www.freesound.org/people/alex_audio/sounds/188595/) by alex_audio [CC-BY-NC]\n- [4-194808-A.ogg]: clip derived from Sip of coffee, swallow then "ah" (http://www.freesound.org/people/odditonic/sounds/194808/) by odditonic [CC0]\n- [4-204202-A.ogg]: clip derived from drinking from a cup (sipping) (http://www.freesound.org/people/scottc6115/sounds/204202/) by scottc6115 [CC0]\n- [4-210593-A.ogg]: clip derived from Different Gulps (http://www.freesound.org/people/Mafon2/sounds/210593/) by Mafon2 [CC0]\n- [4-213193-A.ogg]: clip derived from BottleDrinking02.wav (http://www.freesound.org/people/Abolla/sounds/213193/) by Abolla [CC-BY-NC]\n- [5-204114-A.ogg]: clip derived from Man Eating Teriyaki Noodles.wav (http://www.freesound.org/people/marcjunker/sounds/204114/) by marcjunker [CC0]\n- [5-207836-A.ogg]: clip derived from Essen schluerfen schmatzgen ruelpsen trinken Stuhl schieben 2.WAV (http://www.freesound.org/people/diggy/sounds/207836/) by diggy [CC0]\n- [5-207836-B.ogg]: clip derived from Essen schluerfen schmatzgen ruelpsen trinken Stuhl schieben 2.WAV (http://www.freesound.org/people/diggy/sounds/207836/) by diggy [CC0]\n- [5-207836-C.ogg]: clip derived from Essen schluerfen schmatzgen ruelpsen trinken Stuhl schieben 2.WAV (http://www.freesound.org/people/diggy/sounds/207836/) by diggy [CC0]\n- [5-207836-D.ogg]: clip derived from Essen schluerfen schmatzgen ruelpsen trinken Stuhl schieben 2.WAV (http://www.freesound.org/people/diggy/sounds/207836/) by diggy [CC0]\n- [5-243036-A.ogg]: clip derived from sip.wav (http://www.freesound.org/people/testing_player/sounds/243036/) by testing_player [CC0]\n- [5-243635-A.ogg]: clip derived from swallowing.wav (http://www.freesound.org/people/freakinbehemoth/sounds/243635/) by freakinbehemoth [CC0]\n- [5-259640-A.ogg]: clip derived from Drinking Sip Swallow Gasp.wav (http://www.freesound.org/people/Stevious42/sounds/259640/) by Stevious42 [CC-BY]\n\n401 - Door knock:\n- [1-101336-A.ogg]: clip derived from 01801 knocking.wav (http://www.freesound.org/people/Robinhood76/sounds/101336/) by Robinhood76 [CC-BY-NC]\n- [1-103995-A.ogg]: clip derived from Bassdrum_Shell_Ticks.aif (http://www.freesound.org/people/RutgerMuller/sounds/103995/) by RutgerMuller [CC0]\n- [1-103999-A.ogg]: clip derived from Wood_Knocks_2.aif (http://www.freesound.org/people/RutgerMuller/sounds/103999/) by RutgerMuller [CC0]\n- [1-26188-A.ogg]: clip derived from Shop Door Knocks Original.wav (http://www.freesound.org/people/fogma/sounds/26188/) by fogma [CC-BY]\n- [1-52290-A.ogg]: clip derived from door knock.wav (http://www.freesound.org/people/XxBirdoxX/sounds/52290/) by XxBirdoxX [CC0]\n- [1-81001-A.ogg]: clip derived from Knock on door.mp3 (http://www.freesound.org/people/payattention/sounds/81001/) by payattention [CC0]\n- [1-81001-B.ogg]: clip derived from Knock on door.mp3 (http://www.freesound.org/people/payattention/sounds/81001/) by payattention [CC0]\n- [1-82817-A.ogg]: clip derived from 01337 knocking 1.wav (http://www.freesound.org/people/Robinhood76/sounds/82817/) by Robinhood76 [CC-BY-NC]\n- [2-114254-A.ogg]: clip derived from Knocking.wav (http://www.freesound.org/people/GodofPH/sounds/114254/) by GodofPH [CC0]\n- [2-118624-A.ogg]: clip derived from Door Open and Close - Close Mic.wav (http://www.freesound.org/people/E-Audio/sounds/118624/) by E-Audio [CC0]\n- [2-118625-A.ogg]: clip derived from Door Open and Close - Ortf.wav (http://www.freesound.org/people/E-Audio/sounds/118625/) by E-Audio [CC0]\n- [2-120218-A.ogg]: clip derived from Knocking .wav (http://www.freesound.org/people/sportygurl37/sounds/120218/) by sportygurl37 [CC-BY]\n- [2-120218-B.ogg]: clip derived from Knocking .wav (http://www.freesound.org/people/sportygurl37/sounds/120218/) by sportygurl37 [CC-BY]\n- [2-133889-A.ogg]: clip derived from Large wood door Knocking (http://www.freesound.org/people/aroth4/sounds/133889/) by aroth4 [CC0]\n- [2-134915-A.ogg]: clip derived from Knocking 2 (http://www.freesound.org/people/barrygusey/sounds/134915/) by barrygusey [CC0]\n- [2-140841-A.ogg]: clip derived from Knocking on a Door (http://www.freesound.org/people/thecluegeek/sounds/140841/) by thecluegeek [CC0]\n- [3-144510-A.ogg]: clip derived from Door Knock - with urgency.wav (http://www.freesound.org/people/FreqMan/sounds/144510/) by FreqMan [CC0]\n- [3-151089-A.ogg]: clip derived from door knock 2.wav (http://www.freesound.org/people/ripper351/sounds/151089/) by ripper351 [CC0]\n- [3-152594-A.ogg]: clip derived from Door Closing Knock.mp3 (http://www.freesound.org/people/OTBTechno/sounds/152594/) by OTBTechno [CC-BY]\n- [3-154378-A.ogg]: clip derived from door knock.wav (http://www.freesound.org/people/semccab/sounds/154378/) by semccab [CC0]\n- [3-170574-A.ogg]: clip derived from Knocking on a Sliding Glass Door (http://www.freesound.org/people/Ryan_kuke/sounds/170574/) by Ryan_kuke [CC-BY]\n- [3-180147-A.ogg]: clip derived from Knock (http://www.freesound.org/people/TeamKill3r/sounds/180147/) by TeamKill3r [CC0]\n- [3-182023-A.ogg]: clip derived from Eight_knocks_on_bathroom2_door.wav (http://www.freesound.org/people/Omar%20Alvarado/sounds/182023/) by Omar Alvarado [CC-BY]\n- [3-182025-A.ogg]: clip derived from 12_fast_knocks_on_computer_room_door.wav (http://www.freesound.org/people/Omar%20Alvarado/sounds/182025/) by Omar Alvarado [CC-BY]\n- [4-182034-A.ogg]: clip derived from Five_knocks_on_bathroom2_door.wav (http://www.freesound.org/people/Omar%20Alvarado/sounds/182034/) by Omar Alvarado [CC-BY]\n- [4-182039-A.ogg]: clip derived from four_knocks_on_bedroom_door1.wav (http://www.freesound.org/people/Omar%20Alvarado/sounds/182039/) by Omar Alvarado [CC-BY]\n- [4-182041-A.ogg]: clip derived from five_sharp_knocks_on_computer_room_door.wav (http://www.freesound.org/people/Omar%20Alvarado/sounds/182041/) by Omar Alvarado [CC-BY]\n- [4-186518-A.ogg]: clip derived from door-knock-open-close-wooden_close.wav (http://www.freesound.org/people/ragamuffin/sounds/186518/) by ragamuffin [CC0]\n- [4-187284-A.ogg]: clip derived from Parking Garage - Metal Door - Knock & Bang (http://www.freesound.org/people/HazMattt/sounds/187284/) by HazMattt [CC-BY]\n- [4-188878-A.ogg]: clip derived from Door Knocking 01.wav (http://www.freesound.org/people/Suprasummun/sounds/188878/) by Suprasummun [CC-BY]\n- [4-211502-A.ogg]: clip derived from Knocking_iron_door2.mp3 (http://www.freesound.org/people/Taira%20Komori/sounds/211502/) by Taira Komori [CC-BY]\n- [4-261068-A.ogg]: clip derived from Window_Knocking_Door_Interior-perspective_Exterior_Bang_Aggressive.wav (http://www.freesound.org/people/tompallant/sounds/261068/) by tompallant [CC0]\n- [5-218980-A.ogg]: clip derived from knocking GOOD A_2.aif (http://www.freesound.org/people/bcimmet/sounds/218980/) by bcimmet [CC-BY]\n- [5-218981-A.ogg]: clip derived from knocking GOOD A_1.aif (http://www.freesound.org/people/bcimmet/sounds/218981/) by bcimmet [CC-BY]\n- [5-235644-A.ogg]: clip derived from Door Knock.mp3 (http://www.freesound.org/people/Coonick/sounds/235644/) by Coonick [CC0]\n- [5-250026-A.ogg]: clip derived from knocking and throbing at a roomdoor (http://www.freesound.org/people/hintringer/sounds/250026/) by hintringer [CC0]\n- [5-250026-B.ogg]: clip derived from knocking and throbing at a roomdoor (http://www.freesound.org/people/hintringer/sounds/250026/) by hintringer [CC0]\n- [5-251426-A.ogg]: clip derived from pukanie1.wav (http://www.freesound.org/people/colo777/sounds/251426/) by colo777 [CC0]\n- [5-251426-B.ogg]: clip derived from pukanie1.wav (http://www.freesound.org/people/colo777/sounds/251426/) by colo777 [CC0]\n- [5-256512-A.ogg]: clip derived from Knock on the door 2 (http://www.freesound.org/people/HunteR4708/sounds/256512/) by HunteR4708 [CC0]\n\n402 - Mouse click:\n- [1-118206-A.ogg]: clip derived from Mac Book Pro Track Pad Click.aif (http://www.freesound.org/people/MegaPenguin13/sounds/118206/) by MegaPenguin13 [CC0]\n- [1-18655-A.ogg]: clip derived from MOUSE CLICKS.aif (http://www.freesound.org/people/Ch0cchi/sounds/18655/) by Ch0cchi [CC-BY]\n- [1-45645-A.ogg]: clip derived from MouseButtonLeft.wav (http://www.freesound.org/people/firefreak/sounds/45645/) by firefreak [CC-BY]\n- [1-81851-A.ogg]: clip derived from clics.ogg (http://www.freesound.org/people/aneolus/sounds/81851/) by aneolus [CC0]\n- [1-85123-A.ogg]: clip derived from mouse double clicks and clicks.wav (http://www.freesound.org/people/Wrylytucker/sounds/85123/) by Wrylytucker [CC-BY]\n- [1-95563-A.ogg]: clip derived from mouse.wav (http://www.freesound.org/people/j1987/sounds/95563/) by j1987 [CC0]\n- [1-97793-A.ogg]: clip derived from Mouse.wav (http://www.freesound.org/people/CGEffex/sounds/97793/) by CGEffex [CC-BY]\n- [1-99958-A.ogg]: clip derived from Computer mouse.wav (http://www.freesound.org/people/CGEffex/sounds/99958/) by CGEffex [CC-BY]\n- [2-119139-A.ogg]: clip derived from Mouse Clicking.wav (http://www.freesound.org/people/esperri/sounds/119139/) by esperri [CC0]\n- [2-138257-A.ogg]: clip derived from Mouse Click_Sequence.wav (http://www.freesound.org/people/Hupguy/sounds/138257/) by Hupguy [CC0]\n- [2-146877-A.ogg]: clip derived from Mouse clicks and scroll wheel (http://www.freesound.org/people/tgfcoder/sounds/146877/) by tgfcoder [CC0]\n- [2-146877-B.ogg]: clip derived from Mouse clicks and scroll wheel (http://www.freesound.org/people/tgfcoder/sounds/146877/) by tgfcoder [CC0]\n- [2-152895-A.ogg]: clip derived from Sound of a mouse being clicked  (http://www.freesound.org/people/davidkyoku/sounds/152895/) by davidkyoku [CC0]\n- [2-152964-A.ogg]: clip derived from 003_click.wav (http://www.freesound.org/people/JL_barrett/sounds/152964/) by JL_barrett [CC0]\n- [2-153388-A.ogg]: clip derived from Click.wav (http://www.freesound.org/people/pboix/sounds/153388/) by pboix [CC-BY]\n- [2-154688-A.ogg]: clip derived from CLICKING_MOUSE_1-2.aif (http://www.freesound.org/people/mistorcaveman/sounds/154688/) by mistorcaveman [CC0]\n- [3-155556-A.ogg]: clip derived from Mouse Click 7.wav (http://www.freesound.org/people/BigKahuna360/sounds/155556/) by BigKahuna360 [CC0]\n- [3-158056-A.ogg]: clip derived from mouse clicks (2 types) (http://www.freesound.org/people/Eelke/sounds/158056/) by Eelke [CC-BY]\n- [3-158056-B.ogg]: clip derived from mouse clicks (2 types) (http://www.freesound.org/people/Eelke/sounds/158056/) by Eelke [CC-BY]\n- [3-167096-A.ogg]: clip derived from Computer Mouse Clicking (http://www.freesound.org/people/ultradust/sounds/167096/) by ultradust [CC-BY]\n- [3-170312-A.ogg]: clip derived from Mouseclick (http://www.freesound.org/people/ludde44/sounds/170312/) by ludde44 [CC0]\n- [3-170851-A.ogg]: clip derived from mouse.wav (http://www.freesound.org/people/samchitto/sounds/170851/) by samchitto [CC-BY-NC]\n- [3-172179-A.ogg]: clip derived from mouse_click.wav (http://www.freesound.org/people/MathiasR/sounds/172179/) by MathiasR [CC-BY-NC]\n- [3-185313-A.ogg]: clip derived from mouse.wav (http://www.freesound.org/people/ACorcuera/sounds/185313/) by ACorcuera [CC-BY]\n- [4-192068-A.ogg]: clip derived from mouseclicks (http://www.freesound.org/people/OliGon/sounds/192068/) by OliGon [CC0]\n- [4-195305-A.ogg]: clip derived from Computer Mouse Clicks (http://www.freesound.org/people/YOH/sounds/195305/) by YOH [CC0]\n- [4-201300-A.ogg]: clip derived from mouse.wav (http://www.freesound.org/people/Min0512/sounds/201300/) by Min0512 [CC-BY-NC]\n- [4-201800-A.ogg]: clip derived from Mouse Clicks (http://www.freesound.org/people/hykenfreak/sounds/201800/) by hykenfreak [CC-BY]\n- [4-202298-A.ogg]: clip derived from mouse_clicks.wav (http://www.freesound.org/people/gokiburi/sounds/202298/) by gokiburi [CC0]\n- [4-204612-A.ogg]: clip derived from Mouse Click.wav (http://www.freesound.org/people/pikachu09/sounds/204612/) by pikachu09 [CC0]\n- [4-204683-A.ogg]: clip derived from Mouse clicks 2 (http://www.freesound.org/people/pikachu09/sounds/204683/) by pikachu09 [CC0]\n- [4-210309-A.ogg]: clip derived from 20131209_mouse clicks_OlympusLS10_xy.wav (http://www.freesound.org/people/Soundscape_Leuphana/sounds/210309/) by Soundscape_Leuphana [CC0]\n- [5-219318-A.ogg]: clip derived from Mouse Clicks.wav (http://www.freesound.org/people/joebro10/sounds/219318/) by joebro10 [CC0]\n- [5-223103-A.ogg]: clip derived from Mouse clicks.WAV (http://www.freesound.org/people/beansqueso31/sounds/223103/) by beansqueso31 [CC0]\n- [5-223317-A.ogg]: clip derived from mouse clicks.mp3 (http://www.freesound.org/people/philip1789/sounds/223317/) by philip1789 [CC0]\n- [5-232802-A.ogg]: clip derived from Mouse.wav (http://www.freesound.org/people/mariiao2/sounds/232802/) by mariiao2 [CC0]\n- [5-233019-A.ogg]: clip derived from Computer Mouse.mp3 (http://www.freesound.org/people/amorralteixe/sounds/233019/) by amorralteixe [CC-BY]\n- [5-237315-A.ogg]: clip derived from Mouse.wav (http://www.freesound.org/people/nmarciniegasm/sounds/237315/) by nmarciniegasm [CC0]\n- [5-238926-A.ogg]: clip derived from Click-Click.wav (http://www.freesound.org/people/flosicalcat59/sounds/238926/) by flosicalcat59 [CC-BY]\n- [5-244651-A.ogg]: clip derived from Mouse - Clicks & Scroll Wheel.wav (http://www.freesound.org/people/mrfineppine/sounds/244651/) by mrfineppine [CC0]\n\n403 - Keyboard typing:\n- [1-137-A.ogg]: clip derived from keyboard-typing.wav (http://www.freesound.org/people/Anton/sounds/137/) by Anton [CC-BY]\n- [1-53501-A.ogg]: clip derived from typing.wav (http://www.freesound.org/people/jennypendel/sounds/53501/) by jennypendel [CC-Sampling+]\n- [1-61212-A.ogg]: clip derived from 00428 keyboard writing 1.wav (http://www.freesound.org/people/Robinhood76/sounds/61212/) by Robinhood76 [CC-BY-NC]\n- [1-62594-A.ogg]: clip derived from Keyboard.wav (http://www.freesound.org/people/bsumusictech/sounds/62594/) by bsumusictech [CC0]\n- [1-79711-A.ogg]: clip derived from typing_soft.wav (http://www.freesound.org/people/beckmen/sounds/79711/) by beckmen [CC0]\n- [1-84393-A.ogg]: clip derived from keyboard sounf effect.wav (http://www.freesound.org/people/btothedub88/sounds/84393/) by btothedub88 [CC-Sampling+]\n- [1-94231-A.ogg]: clip derived from Computer Type.wav (http://www.freesound.org/people/gchand1/sounds/94231/) by gchand1 [CC-Sampling+]\n- [1-94231-B.ogg]: clip derived from Computer Type.wav (http://www.freesound.org/people/gchand1/sounds/94231/) by gchand1 [CC-Sampling+]\n- [2-104168-A.ogg]: clip derived from keyboard.wav (http://www.freesound.org/people/SpazTastic/sounds/104168/) by SpazTastic [CC-Sampling+]\n- [2-109316-A.ogg]: clip derived from Keyboard, writing.wav (http://www.freesound.org/people/Ryding/sounds/109316/) by Ryding [CC0]\n- [2-118459-A.ogg]: clip derived from person typing on a computer keyboard.mp3 (http://www.freesound.org/people/Omar%20Alvarado/sounds/118459/) by Omar Alvarado [CC-BY]\n- [2-118459-B.ogg]: clip derived from person typing on a computer keyboard.mp3 (http://www.freesound.org/people/Omar%20Alvarado/sounds/118459/) by Omar Alvarado [CC-BY]\n- [2-118817-A.ogg]: clip derived from 25_Typing.wav (http://www.freesound.org/people/lmbubec/sounds/118817/) by lmbubec [CC0]\n- [2-120333-A.ogg]: clip derived from laptop keyboard.wav (http://www.freesound.org/people/cemagar/sounds/120333/) by cemagar [CC-BY]\n- [2-99795-A.ogg]: clip derived from Typing (Desktop).aiff (http://www.freesound.org/people/Zappa_Was_God/sounds/99795/) by Zappa_Was_God [CC-BY-NC]\n- [2-99796-A.ogg]: clip derived from Typing (Laptop).aiff (http://www.freesound.org/people/Zappa_Was_God/sounds/99796/) by Zappa_Was_God [CC-BY-NC]\n- [3-125548-A.ogg]: clip derived from typing.wav (http://www.freesound.org/people/krissyeliot/sounds/125548/) by krissyeliot [CC0]\n- [3-132830-A.ogg]: clip derived from Typing.MP3 (http://www.freesound.org/people/yiannoui/sounds/132830/) by yiannoui [CC0]\n- [3-144106-A.ogg]: clip derived from typing_phrase.wav (http://www.freesound.org/people/jwestbury/sounds/144106/) by jwestbury [CC-BY]\n- [3-144120-A.ogg]: clip derived from Keyboard typing (http://www.freesound.org/people/avakas/sounds/144120/) by avakas [CC-BY]\n- [3-153444-A.ogg]: clip derived from Angry fast Mac keyboard typing.mp3 (http://www.freesound.org/people/orginaljun/sounds/153444/) by orginaljun [CC-BY]\n- [3-154781-A.ogg]: clip derived from Keyboard Typing.wav (http://www.freesound.org/people/jpkweli/sounds/154781/) by jpkweli [CC0]\n- [3-155568-A.ogg]: clip derived from Typing External Keyboard 2.wav (http://www.freesound.org/people/BigKahuna360/sounds/155568/) by BigKahuna360 [CC0]\n- [3-155570-A.ogg]: clip derived from Typing Laptop Keyboard 1.wav (http://www.freesound.org/people/BigKahuna360/sounds/155570/) by BigKahuna360 [CC0]\n- [4-158653-A.ogg]: clip derived from typing_on_compaq_keyboard.wav (http://www.freesound.org/people/nihilyst/sounds/158653/) by nihilyst [CC-BY]\n- [4-167155-A.ogg]: clip derived from Typing (http://www.freesound.org/people/DSPena/sounds/167155/) by DSPena [CC0]\n- [4-172180-A.ogg]: clip derived from keyboard_typing_stream.wav (http://www.freesound.org/people/MathiasR/sounds/172180/) by MathiasR [CC-BY-NC]\n- [4-172742-A.ogg]: clip derived from Apple_Keyboard_2.wav (http://www.freesound.org/people/coldwellw/sounds/172742/) by coldwellw [CC-BY]\n- [4-177243-A.ogg]: clip derived from Binaural keyboard touch-typing.wav (http://www.freesound.org/people/d_mcd/sounds/177243/) by d_mcd [CC0]\n- [4-181707-A.ogg]: clip derived from keyboarding1.wav (http://www.freesound.org/people/keweldog/sounds/181707/) by keweldog [CC0]\n- [4-181708-A.ogg]: clip derived from keyboarding2.wav (http://www.freesound.org/people/keweldog/sounds/181708/) by keweldog [CC0]\n- [4-185613-A.ogg]: clip derived from Computer Keyboard Typing (http://www.freesound.org/people/AldebaranCW/sounds/185613/) by AldebaranCW [CC0]\n- [5-205090-A.ogg]: clip derived from Typing 2.WAV (http://www.freesound.org/people/jadend2/sounds/205090/) by jadend2 [CC0]\n- [5-215005-A.ogg]: clip derived from Computer Dell Keyboard Typing.wav (http://www.freesound.org/people/EthanolProductions/sounds/215005/) by EthanolProductions [CC0]\n- [5-216131-A.ogg]: clip derived from keyboard3.aif (http://www.freesound.org/people/hukleesyr/sounds/216131/) by hukleesyr [CC-BY]\n- [5-222894-A.ogg]: clip derived from Typing (http://www.freesound.org/people/SkepticDosh/sounds/222894/) by SkepticDosh [CC-BY]\n- [5-223099-A.ogg]: clip derived from Keyboard type and mouse click.WAV (http://www.freesound.org/people/beansqueso31/sounds/223099/) by beansqueso31 [CC0]\n- [5-223099-B.ogg]: clip derived from Keyboard type and mouse click.WAV (http://www.freesound.org/people/beansqueso31/sounds/223099/) by beansqueso31 [CC0]\n- [5-234923-A.ogg]: clip derived from Mechanical Keyboard Typing (http://www.freesound.org/people/Nmb910/sounds/234923/) by Nmb910 [CC0]\n- [5-244178-A.ogg]: clip derived from keyboard typing (http://www.freesound.org/people/Snapper4298/sounds/244178/) by Snapper4298 [CC-BY]\n\n404 - Door - wood creaks:\n- [1-51805-A.ogg]: clip derived from door hinge squeak creak o,c.aiff (http://www.freesound.org/people/kyles/sounds/51805/) by kyles [CC0]\n- [1-51805-B.ogg]: clip derived from door hinge squeak creak o,c.aiff (http://www.freesound.org/people/kyles/sounds/51805/) by kyles [CC0]\n- [1-51805-C.ogg]: clip derived from door hinge squeak creak o,c.aiff (http://www.freesound.org/people/kyles/sounds/51805/) by kyles [CC0]\n- [1-51805-D.ogg]: clip derived from door hinge squeak creak o,c.aiff (http://www.freesound.org/people/kyles/sounds/51805/) by kyles [CC0]\n- [1-51805-E.ogg]: clip derived from door hinge squeak creak o,c.aiff (http://www.freesound.org/people/kyles/sounds/51805/) by kyles [CC0]\n- [1-51805-F.ogg]: clip derived from door hinge squeak creak o,c.aiff (http://www.freesound.org/people/kyles/sounds/51805/) by kyles [CC0]\n- [1-51805-G.ogg]: clip derived from door hinge squeak creak o,c.aiff (http://www.freesound.org/people/kyles/sounds/51805/) by kyles [CC0]\n- [1-51805-H.ogg]: clip derived from door hinge squeak creak o,c.aiff (http://www.freesound.org/people/kyles/sounds/51805/) by kyles [CC0]\n- [2-123712-A.ogg]: clip derived from creaking-wood-screws.AIF (http://www.freesound.org/people/alienistcog/sounds/123712/) by alienistcog [CC0]\n- [2-70344-A.ogg]: clip derived from Grince Arm Lo-Lm-Lo.aif (http://www.freesound.org/people/beskhu/sounds/70344/) by beskhu [CC-BY]\n- [2-70366-A.ogg]: clip derived from Grince Plac Lg Lo 2.aif (http://www.freesound.org/people/beskhu/sounds/70366/) by beskhu [CC-BY]\n- [2-70367-A.ogg]: clip derived from Grince Plac Lg Lo-Hm.aif (http://www.freesound.org/people/beskhu/sounds/70367/) by beskhu [CC-BY]\n- [2-7321-A.ogg]: clip derived from treehouse.wav (http://www.freesound.org/people/mystiscool/sounds/7321/) by mystiscool [CC-BY]\n- [2-87780-A.ogg]: clip derived from Creaking Door.wav (http://www.freesound.org/people/Dstruct/sounds/87780/) by Dstruct [CC-BY-NC]\n- [2-91912-A.ogg]: clip derived from Dock,Wood,Creak,Water,Laping.wav (http://www.freesound.org/people/Filipe%20Chagas/sounds/91912/) by Filipe Chagas [CC0]\n- [2-91912-B.ogg]: clip derived from Dock,Wood,Creak,Water,Laping.wav (http://www.freesound.org/people/Filipe%20Chagas/sounds/91912/) by Filipe Chagas [CC0]\n- [3-101381-A.ogg]: clip derived from Creaking Floorboard.wav (http://www.freesound.org/people/Benboncan/sounds/101381/) by Benboncan [CC-BY]\n- [3-101381-B.ogg]: clip derived from Creaking Floorboard.wav (http://www.freesound.org/people/Benboncan/sounds/101381/) by Benboncan [CC-BY]\n- [3-103401-A.ogg]: clip derived from kast_extreem_kraken.mp3 (http://www.freesound.org/people/stijnjobse/sounds/103401/) by stijnjobse [CC0]\n- [3-103401-B.ogg]: clip derived from kast_extreem_kraken.mp3 (http://www.freesound.org/people/stijnjobse/sounds/103401/) by stijnjobse [CC0]\n- [3-103401-C.ogg]: clip derived from kast_extreem_kraken.mp3 (http://www.freesound.org/people/stijnjobse/sounds/103401/) by stijnjobse [CC0]\n- [3-103401-D.ogg]: clip derived from kast_extreem_kraken.mp3 (http://www.freesound.org/people/stijnjobse/sounds/103401/) by stijnjobse [CC0]\n- [3-118194-A.ogg]: clip derived from Wood Chair Creaking.wav (http://www.freesound.org/people/bmcken/sounds/118194/) by bmcken [CC-BY]\n- [3-94355-A.ogg]: clip derived from Knakande d\\xf6rr.wav (http://www.freesound.org/people/Ryding/sounds/94355/) by Ryding [CC0]\n- [4-107117-A.ogg]: clip derived from CrDr_002.wav (http://www.freesound.org/people/tec%20studios/sounds/107117/) by tec studios [CC0]\n- [4-107120-A.ogg]: clip derived from CrDr_005.wav (http://www.freesound.org/people/tec%20studios/sounds/107120/) by tec studios [CC0]\n- [4-107122-A.ogg]: clip derived from CrDr_007.wav (http://www.freesound.org/people/tec%20studios/sounds/107122/) by tec studios [CC0]\n- [4-108352-A.ogg]: clip derived from DoorCreakOpenClose.WAV (http://www.freesound.org/people/pbickh1/sounds/108352/) by pbickh1 [CC-Sampling+]\n- [4-128659-A.ogg]: clip derived from Wood_Creak_ 01 (http://www.freesound.org/people/dheming/sounds/128659/) by dheming [CC-BY]\n- [4-132839-A.ogg]: clip derived from creak.door.wav (http://www.freesound.org/people/UncleSigmund/sounds/132839/) by UncleSigmund [CC0]\n- [4-160036-A.ogg]: clip derived from Creaky Garden Door 1 (http://www.freesound.org/people/Anton/sounds/160036/) by Anton [CC-BY]\n- [4-160036-B.ogg]: clip derived from Creaky Garden Door 1 (http://www.freesound.org/people/Anton/sounds/160036/) by Anton [CC-BY]\n- [5-161270-A.ogg]: clip derived from creaking.wav (http://www.freesound.org/people/smidoid/sounds/161270/) by smidoid [CC-BY-NC]\n- [5-161270-B.ogg]: clip derived from creaking.wav (http://www.freesound.org/people/smidoid/sounds/161270/) by smidoid [CC-BY-NC]\n- [5-173568-A.ogg]: clip derived from creaking door.wav (http://www.freesound.org/people/Bratok999/sounds/173568/) by Bratok999 [CC0]\n- [5-177779-A.ogg]: clip derived from Wood_Creak_02.wav (http://www.freesound.org/people/dheming/sounds/177779/) by dheming [CC-BY]\n- [5-181458-A.ogg]: clip derived from door_thick_creak_medium-perspective_INT (http://www.freesound.org/people/matucha/sounds/181458/) by matucha [CC-BY]\n- [5-187444-A.ogg]: clip derived from Old wooden door creaking (http://www.freesound.org/people/VlatkoBlazek/sounds/187444/) by VlatkoBlazek [CC-BY]\n- [5-188606-A.ogg]: clip derived from Wood Creak.wav (http://www.freesound.org/people/TimPryor/sounds/188606/) by TimPryor [CC-BY]\n- [5-191497-A.ogg]: clip derived from Creaking pier (http://www.freesound.org/people/morosopher/sounds/191497/) by morosopher [CC0]\n\n405 - Can opening:\n- [1-101404-A.ogg]: clip derived from Can Open.WAV (http://www.freesound.org/people/LG/sounds/101404/) by LG [CC-BY]\n- [1-41615-A.ogg]: clip derived from sodacanopen.wav (http://www.freesound.org/people/NoiseCollector/sounds/41615/) by NoiseCollector [CC-BY]\n- [1-43764-A.ogg]: clip derived from open_can.ogg (http://www.freesound.org/people/Staile/sounds/43764/) by Staile [CC-BY]\n- [1-58846-A.ogg]: clip derived from opening_a_soda_can.mp3 (http://www.freesound.org/people/morgantj/sounds/58846/) by morgantj [CC-BY]\n- [1-60676-A.ogg]: clip derived from openingsoda2.wav (http://www.freesound.org/people/K1m218/sounds/60676/) by K1m218 [CC-Sampling+]\n- [1-68670-A.ogg]: clip derived from CokeCan.wav (http://www.freesound.org/people/mikaelfernstrom/sounds/68670/) by mikaelfernstrom [CC-BY]\n- [1-68734-A.ogg]: clip derived from TwoCokeCans.wav (http://www.freesound.org/people/mikaelfernstrom/sounds/68734/) by mikaelfernstrom [CC-BY]\n- [1-69165-A.ogg]: clip derived from cansound.wav (http://www.freesound.org/people/BranRainey/sounds/69165/) by BranRainey [CC0]\n- [2-130245-A.ogg]: clip derived from soda can open (http://www.freesound.org/people/fultron12/sounds/130245/) by fultron12 [CC0]\n- [2-144031-A.ogg]: clip derived from Soda Can Open.aif (http://www.freesound.org/people/timgormly/sounds/144031/) by timgormly [CC0]\n- [2-81112-A.ogg]: clip derived from 12 soda can TLM_05 -AIFF 48:24.aiff (http://www.freesound.org/people/Josh%20Herman/sounds/81112/) by Josh Herman [CC0]\n- [2-81190-A.ogg]: clip derived from soda_can_01.wav (http://www.freesound.org/people/joedeshon/sounds/81190/) by joedeshon [CC-BY]\n- [2-83667-A.ogg]: clip derived from Opening Can Slow.wav (http://www.freesound.org/people/Scott_Kelly/sounds/83667/) by Scott_Kelly [CC-Sampling+]\n- [2-83688-A.ogg]: clip derived from soda can_pop.wav (http://www.freesound.org/people/braffe2/sounds/83688/) by braffe2 [CC-Sampling+]\n- [2-85471-A.ogg]: clip derived from Soda Can Open.aif (http://www.freesound.org/people/JasonElrod/sounds/85471/) by JasonElrod [CC-BY]\n- [2-87282-A.ogg]: clip derived from fizzy-drink-can.aif (http://www.freesound.org/people/Spleencast/sounds/87282/) by Spleencast [CC-BY-NC]\n- [3-147342-A.ogg]: clip derived from open_can2.wav (http://www.freesound.org/people/fuzzpapi/sounds/147342/) by fuzzpapi [CC0]\n- [3-147343-A.ogg]: clip derived from open_can.wav (http://www.freesound.org/people/fuzzpapi/sounds/147343/) by fuzzpapi [CC0]\n- [3-148932-A.ogg]: clip derived from Beer Can Opened.wav (http://www.freesound.org/people/cloudyeyetavern/sounds/148932/) by cloudyeyetavern [CC0]\n- [3-155659-A.ogg]: clip derived from Can opening (http://www.freesound.org/people/Daxyn/sounds/155659/) by Daxyn [CC-BY-NC]\n- [3-166546-A.ogg]: clip derived from openning can.wav (http://www.freesound.org/people/Nic3_one/sounds/166546/) by Nic3_one [CC0]\n- [3-166546-B.ogg]: clip derived from openning can.wav (http://www.freesound.org/people/Nic3_one/sounds/166546/) by Nic3_one [CC0]\n- [3-170002-A.ogg]: clip derived from sodapopcan.mp3 (http://www.freesound.org/people/Omar%20Alvarado/sounds/170002/) by Omar Alvarado [CC-BY-NC]\n- [3-171937-A.ogg]: clip derived from Opening Drink Can (http://www.freesound.org/people/FlameHazeZx/sounds/171937/) by FlameHazeZx [CC-BY-NC]\n- [4-175025-A.ogg]: clip derived from Open can (http://www.freesound.org/people/Cabeeno%20Rossley/sounds/175025/) by Cabeeno Rossley [CC-BY]\n- [4-179836-A.ogg]: clip derived from Opening Soda Can (http://www.freesound.org/people/MrAuralization/sounds/179836/) by MrAuralization [CC0]\n- [4-187384-A.ogg]: clip derived from Open Beer can (http://www.freesound.org/people/jorickhoofd/sounds/187384/) by jorickhoofd [CC-BY]\n- [4-188003-A.ogg]: clip derived from Drinks Can Tab Opening  (http://www.freesound.org/people/Motion_S/sounds/188003/) by Motion_S [CC-BY]\n- [4-212728-A.ogg]: clip derived from Opening_soda_can.mp3 (http://www.freesound.org/people/Taira%20Komori/sounds/212728/) by Taira Komori [CC-BY]\n- [4-215635-A.ogg]: clip derived from Open a can (http://www.freesound.org/people/edhutschek/sounds/215635/) by edhutschek [CC0]\n- [4-216349-A.ogg]: clip derived from Opening_Can_02.wav (http://www.freesound.org/people/RSilveira_88/sounds/216349/) by RSilveira_88 [CC-BY]\n- [4-264453-A.ogg]: clip derived from Opening a Can (http://www.freesound.org/people/Dwums96/sounds/264453/) by Dwums96 [CC0]\n- [5-221878-A.ogg]: clip derived from open_soda.ogg (http://www.freesound.org/people/quiteretarded/sounds/221878/) by quiteretarded [CC0]\n- [5-236299-A.ogg]: clip derived from Opening beer can 1.wav (http://www.freesound.org/people/beansqueso31/sounds/236299/) by beansqueso31 [CC0]\n- [5-237795-A.ogg]: clip derived from opening can.wav (http://www.freesound.org/people/Edenza919/sounds/237795/) by Edenza919 [CC-BY]\n- [5-244327-A.ogg]: clip derived from can open.wav (http://www.freesound.org/people/squashy555/sounds/244327/) by squashy555 [CC0]\n- [5-244933-A.ogg]: clip derived from opening_coke_can.wav (http://www.freesound.org/people/kwahmah_02/sounds/244933/) by kwahmah_02 [CC-BY]\n- [5-250753-A.ogg]: clip derived from Opening_a_can.wav (http://www.freesound.org/people/guilleADRV/sounds/250753/) by guilleADRV [CC0]\n- [5-252248-A.ogg]: clip derived from can opening.wav (http://www.freesound.org/people/Reitanna/sounds/252248/) by Reitanna [CC0]\n- [5-260011-A.ogg]: clip derived from cokecan open.wav (http://www.freesound.org/people/kwahmah_02/sounds/260011/) by kwahmah_02 [CC-BY]\n\n406 - Washing machine:\n- [1-21896-A.ogg]: clip derived from file0100washingmachine.mp3 (http://www.freesound.org/people/ulcered/sounds/21896/) by ulcered [CC-Sampling+]\n- [1-23996-A.ogg]: clip derived from WashingMachine1.flac (http://www.freesound.org/people/acclivity/sounds/23996/) by acclivity [CC-BY-NC]\n- [1-23996-B.ogg]: clip derived from WashingMachine1.flac (http://www.freesound.org/people/acclivity/sounds/23996/) by acclivity [CC-BY-NC]\n- [1-27165-A.ogg]: clip derived from 20061205.washing.machine.pumping.flac (http://www.freesound.org/people/dobroide/sounds/27165/) by dobroide [CC-BY]\n- [1-27166-A.ogg]: clip derived from 20061205.washing.machine.spinning.flac (http://www.freesound.org/people/dobroide/sounds/27166/) by dobroide [CC-BY]\n- [1-32373-A.ogg]: clip derived from Washing Machine.wav (http://www.freesound.org/people/Walter_Odington/sounds/32373/) by Walter_Odington [CC0]\n- [1-32373-B.ogg]: clip derived from Washing Machine.wav (http://www.freesound.org/people/Walter_Odington/sounds/32373/) by Walter_Odington [CC0]\n- [1-49098-A.ogg]: clip derived from washing_machine_valve_open_water_coming_in.mp3 (http://www.freesound.org/people/Greg%20Baumont/sounds/49098/) by Greg Baumont [CC-BY-NC]\n- [2-102567-A.ogg]: clip derived from Washing machine full final spin cycle.wav (http://www.freesound.org/people/earthsounds/sounds/102567/) by earthsounds [CC-BY]\n- [2-102567-B.ogg]: clip derived from Washing machine full final spin cycle.wav (http://www.freesound.org/people/earthsounds/sounds/102567/) by earthsounds [CC-BY]\n- [2-102567-C.ogg]: clip derived from Washing machine full final spin cycle.wav (http://www.freesound.org/people/earthsounds/sounds/102567/) by earthsounds [CC-BY]\n- [2-102567-D.ogg]: clip derived from Washing machine full final spin cycle.wav (http://www.freesound.org/people/earthsounds/sounds/102567/) by earthsounds [CC-BY]\n- [2-102568-A.ogg]: clip derived from Washing machine rinse cycle.wav (http://www.freesound.org/people/earthsounds/sounds/102568/) by earthsounds [CC-BY]\n- [2-121909-A.ogg]: clip derived from Washing machine.wav (http://www.freesound.org/people/Lemodem/sounds/121909/) by Lemodem [CC-BY]\n- [2-51173-A.ogg]: clip derived from Washing Machine.wav (http://www.freesound.org/people/RutgerMuller/sounds/51173/) by RutgerMuller [CC0]\n- [2-59241-A.ogg]: clip derived from wasmachine-centrifugeertkort.mp3 (http://www.freesound.org/people/info@in-tune.nl/sounds/59241/) by info@in-tune.nl [CC-BY]\n- [3-135469-A.ogg]: clip derived from Washing-machine-drying-machine.flac (http://www.freesound.org/people/Erdie/sounds/135469/) by Erdie [CC0]\n- [3-151269-A.ogg]: clip derived from Washing Machine 2.aif (http://www.freesound.org/people/timgormly/sounds/151269/) by timgormly [CC0]\n- [3-151273-A.ogg]: clip derived from Washing Machine Full Cycle.aif (http://www.freesound.org/people/timgormly/sounds/151273/) by timgormly [CC0]\n- [3-156391-A.ogg]: clip derived from washing machine spinning slow (http://www.freesound.org/people/Ama_Dis/sounds/156391/) by Ama_Dis [CC-BY]\n- [3-156393-A.ogg]: clip derived from washing machine spinning fast (http://www.freesound.org/people/Ama_Dis/sounds/156393/) by Ama_Dis [CC-BY]\n- [3-182710-A.ogg]: clip derived from washing machine000.aif (http://www.freesound.org/people/FEAMEIS/sounds/182710/) by FEAMEIS [CC0]\n- [3-182710-B.ogg]: clip derived from washing machine000.aif (http://www.freesound.org/people/FEAMEIS/sounds/182710/) by FEAMEIS [CC0]\n- [3-188726-A.ogg]: clip derived from Washing Machine. close recording. (http://www.freesound.org/people/felix.blume/sounds/188726/) by felix.blume [CC0]\n- [4-218199-A.ogg]: clip derived from Operate a  washing machine (http://www.freesound.org/people/edhutschek/sounds/218199/) by edhutschek [CC0]\n- [4-218199-B.ogg]: clip derived from Operate a  washing machine (http://www.freesound.org/people/edhutschek/sounds/218199/) by edhutschek [CC0]\n- [4-218199-C.ogg]: clip derived from Operate a  washing machine (http://www.freesound.org/people/edhutschek/sounds/218199/) by edhutschek [CC0]\n- [4-218199-D.ogg]: clip derived from Operate a  washing machine (http://www.freesound.org/people/edhutschek/sounds/218199/) by edhutschek [CC0]\n- [4-218199-E.ogg]: clip derived from Operate a  washing machine (http://www.freesound.org/people/edhutschek/sounds/218199/) by edhutschek [CC0]\n- [4-218199-F.ogg]: clip derived from Operate a  washing machine (http://www.freesound.org/people/edhutschek/sounds/218199/) by edhutschek [CC0]\n- [4-218199-G.ogg]: clip derived from Operate a  washing machine (http://www.freesound.org/people/edhutschek/sounds/218199/) by edhutschek [CC0]\n- [4-218199-H.ogg]: clip derived from Operate a  washing machine (http://www.freesound.org/people/edhutschek/sounds/218199/) by edhutschek [CC0]\n- [5-141683-Ad.ogg]: clip derived from Washing Machine Spins.aif (http://www.freesound.org/people/DJ%20Burnham/sounds/141683/) by DJ Burnham [CC0]\n- [5-181977-A.ogg]: clip derived from Washing Machine Working (http://www.freesound.org/people/LanDub/sounds/181977/) by LanDub [CC0]\n- [5-207811-A.ogg]: clip derived from washing machine spin cycle.wav (http://www.freesound.org/people/GJOS/sounds/207811/) by GJOS [CC0]\n- [5-207811-B.ogg]: clip derived from washing machine spin cycle.wav (http://www.freesound.org/people/GJOS/sounds/207811/) by GJOS [CC0]\n- [5-223810-A.ogg]: clip derived from Washing Machine.WAV (http://www.freesound.org/people/poorenglishjuggler/sounds/223810/) by poorenglishjuggler [CC0]\n- [5-238021-A.ogg]: clip derived from Washing machine (http://www.freesound.org/people/eaglechopper/sounds/238021/) by eaglechopper [CC0]\n- [5-245040-A.ogg]: clip derived from Washing Machine- Final Spin.wav (http://www.freesound.org/people/kwahmah_02/sounds/245040/) by kwahmah_02 [CC-BY]\n- [5-260875-A.ogg]: clip derived from Washing machine (similar to heavy machinery) (http://www.freesound.org/people/VlatkoBlazek/sounds/260875/) by VlatkoBlazek [CC-BY]\n\n407 - Vacuum cleaner:\n- [1-100210-A.ogg]: clip derived from Vacuum Cleaner-Samsung Easy 1300.wav (http://www.freesound.org/people/joshfeed/sounds/100210/) by joshfeed [CC-BY-NC]\n- [1-100210-B.ogg]: clip derived from Vacuum Cleaner-Samsung Easy 1300.wav (http://www.freesound.org/people/joshfeed/sounds/100210/) by joshfeed [CC-BY-NC]\n- [1-19840-A.ogg]: clip derived from Hoover Stationary.wav (http://www.freesound.org/people/fogma/sounds/19840/) by fogma [CC-BY]\n- [1-19872-A.ogg]: clip derived from Hoover Moving.wav (http://www.freesound.org/people/fogma/sounds/19872/) by fogma [CC-BY]\n- [1-19872-B.ogg]: clip derived from Hoover Moving.wav (http://www.freesound.org/people/fogma/sounds/19872/) by fogma [CC-BY]\n- [1-46744-A.ogg]: clip derived from 19_VACCL.wav (http://www.freesound.org/people/ReWired/sounds/46744/) by ReWired [CC-Sampling+]\n- [1-60460-A.ogg]: clip derived from vacuum_cleaner_dyson.wav (http://www.freesound.org/people/JimAward/sounds/60460/) by JimAward [CC-BY]\n- [1-79236-A.ogg]: clip derived from bissell_carpet_cleaner_vacuum_01.wav (http://www.freesound.org/people/joedeshon/sounds/79236/) by joedeshon [CC-BY]\n- [2-106072-A.ogg]: clip derived from vacuum1.mp3 (http://www.freesound.org/people/NoiseCollector/sounds/106072/) by NoiseCollector [CC-BY]\n- [2-106073-A.ogg]: clip derived from vacuum2.mp3 (http://www.freesound.org/people/NoiseCollector/sounds/106073/) by NoiseCollector [CC-BY]\n- [2-122820-A.ogg]: clip derived from vacuum_cleaner_moving.mp3 (http://www.freesound.org/people/geodylabs/sounds/122820/) by geodylabs [CC-BY]\n- [2-122820-B.ogg]: clip derived from vacuum_cleaner_moving.mp3 (http://www.freesound.org/people/geodylabs/sounds/122820/) by geodylabs [CC-BY]\n- [2-141681-A.ogg]: clip derived from Vacuum Cleaner in use.aif (http://www.freesound.org/people/DJ%20Burnham/sounds/141681/) by DJ Burnham [CC0]\n- [2-141681-B.ogg]: clip derived from Vacuum Cleaner in use.aif (http://www.freesound.org/people/DJ%20Burnham/sounds/141681/) by DJ Burnham [CC0]\n- [2-141682-A.ogg]: clip derived from Vacuum Cleaner switch on, run, switch off 2.aif (http://www.freesound.org/people/DJ%20Burnham/sounds/141682/) by DJ Burnham [CC0]\n- [2-141682-B.ogg]: clip derived from Vacuum Cleaner switch on, run, switch off 2.aif (http://www.freesound.org/people/DJ%20Burnham/sounds/141682/) by DJ Burnham [CC0]\n- [3-152020-A.ogg]: clip derived from hoover.mp3 (http://www.freesound.org/people/winsx87/sounds/152020/) by winsx87 [CC0]\n- [3-152020-B.ogg]: clip derived from hoover.mp3 (http://www.freesound.org/people/winsx87/sounds/152020/) by winsx87 [CC0]\n- [3-152020-C.ogg]: clip derived from hoover.mp3 (http://www.freesound.org/people/winsx87/sounds/152020/) by winsx87 [CC0]\n- [3-159346-A.ogg]: clip derived from Vacuum Cleaner 02  - short  (http://www.freesound.org/people/Huminaatio/sounds/159346/) by Huminaatio [CC0]\n- [3-159346-B.ogg]: clip derived from Vacuum Cleaner 02  - short  (http://www.freesound.org/people/Huminaatio/sounds/159346/) by Huminaatio [CC0]\n- [3-159347-A.ogg]: clip derived from Vacuum Cleaner 3 - sucking short (http://www.freesound.org/people/Huminaatio/sounds/159347/) by Huminaatio [CC0]\n- [3-159347-B.ogg]: clip derived from Vacuum Cleaner 3 - sucking short (http://www.freesound.org/people/Huminaatio/sounds/159347/) by Huminaatio [CC0]\n- [3-159348-A.ogg]: clip derived from Vacuum Cleaner 01 -  (http://www.freesound.org/people/Huminaatio/sounds/159348/) by Huminaatio [CC0]\n- [4-146200-A.ogg]: clip derived from Vacuum Cleaner.wav (http://www.freesound.org/people/brumbar/sounds/146200/) by brumbar [CC0]\n- [4-172732-A.ogg]: clip derived from 20121230_vacuum.cleaner.start.02.wav (http://www.freesound.org/people/dobroide/sounds/172732/) by dobroide [CC-BY]\n- [4-172733-A.ogg]: clip derived from 20121230_vacuum.cleaner.start.01.wav (http://www.freesound.org/people/dobroide/sounds/172733/) by dobroide [CC-BY]\n- [4-172734-A.ogg]: clip derived from 20121230_vacuum.cleaner.end.02.wav (http://www.freesound.org/people/dobroide/sounds/172734/) by dobroide [CC-BY]\n- [4-172736-A.ogg]: clip derived from 20121230_vacuum.cleaner.working.02.wav (http://www.freesound.org/people/dobroide/sounds/172736/) by dobroide [CC-BY]\n- [4-172736-B.ogg]: clip derived from 20121230_vacuum.cleaner.working.02.wav (http://www.freesound.org/people/dobroide/sounds/172736/) by dobroide [CC-BY]\n- [4-181999-A.ogg]: clip derived from Vacuum cleaner type Bomann 2300 Watt (http://www.freesound.org/people/ohrpilot/sounds/181999/) by ohrpilot [CC0]\n- [4-194680-A.ogg]: clip derived from vacuum cleaner.wav (http://www.freesound.org/people/sundet7/sounds/194680/) by sundet7 [CC0]\n- [5-182007-A.ogg]: clip derived from Vacuum cleaner Bomann 1600 Watt (http://www.freesound.org/people/ohrpilot/sounds/182007/) by ohrpilot [CC0]\n- [5-182010-A.ogg]: clip derived from Vacuum cleaner 3, type "Progress Stuttgart" (http://www.freesound.org/people/ohrpilot/sounds/182010/) by ohrpilot [CC0]\n- [5-182012-A.ogg]: clip derived from Vacuum cleaner 1, type "Progress Stuttgart" (http://www.freesound.org/people/ohrpilot/sounds/182012/) by ohrpilot [CC0]\n- [5-188365-A.ogg]: clip derived from Vacuum Cleaner, with on/off sound (http://www.freesound.org/people/luciolepri/sounds/188365/) by luciolepri [CC-BY-NC]\n- [5-212054-A.ogg]: clip derived from Cleaner_far.mp3 (http://www.freesound.org/people/Taira%20Komori/sounds/212054/) by Taira Komori [CC-BY]\n- [5-212059-A.ogg]: clip derived from Cleaner_nearly.mp3 (http://www.freesound.org/people/Taira%20Komori/sounds/212059/) by Taira Komori [CC-BY]\n- [5-231551-A.ogg]: clip derived from Vacuum cleaner (http://www.freesound.org/people/MrAuralization/sounds/231551/) by MrAuralization [CC0]\n- [5-263902-A.ogg]: clip derived from vacuum.wav (http://www.freesound.org/people/tpd/sounds/263902/) by tpd [CC0]\n\n408 - Clock alarm:\n- [1-13613-A.ogg]: clip derived from alarm clock.aif (http://www.freesound.org/people/hookhead/sounds/13613/) by hookhead [CC0]\n- [1-14262-A.ogg]: clip derived from alarm_clock.wav (http://www.freesound.org/people/xyzr_kx/sounds/14262/) by xyzr_kx [CC0]\n- [1-34853-A.ogg]: clip derived from Alarm Clock.aif (http://www.freesound.org/people/jackstrebor/sounds/34853/) by jackstrebor [CC-Sampling+]\n- [1-67033-A.ogg]: clip derived from AlarmClock.wav (http://www.freesound.org/people/ytiesl2s/sounds/67033/) by ytiesl2s [CC-Sampling+]\n- [1-72195-A.ogg]: clip derived from alarmclock.mp3 (http://www.freesound.org/people/snowflakes/sounds/72195/) by snowflakes [CC0]\n- [1-72195-B.ogg]: clip derived from alarmclock.mp3 (http://www.freesound.org/people/snowflakes/sounds/72195/) by snowflakes [CC0]\n- [1-74517-A.ogg]: clip derived from outro-dia-mais.wav (http://www.freesound.org/people/galeku/sounds/74517/) by galeku [CC0]\n- [1-96890-A.ogg]: clip derived from Alarm Clock.mp3 (http://www.freesound.org/people/TinyJiro/sounds/96890/) by TinyJiro [CC-BY]\n- [2-102435-A.ogg]: clip derived from old alarm clock ringing (http://www.freesound.org/people/tuberatanka/sounds/102435/) by tuberatanka [CC0]\n- [2-104475-A.ogg]: clip derived from Alarm Clock- E.mp3 (http://www.freesound.org/people/rparson/sounds/104475/) by rparson [CC-Sampling+]\n- [2-104476-A.ogg]: clip derived from Alarm Clock- T.mp3 (http://www.freesound.org/people/rparson/sounds/104476/) by rparson [CC-Sampling+]\n- [2-117116-A.ogg]: clip derived from alarmclock.wav (http://www.freesound.org/people/crazy_sam/sounds/117116/) by crazy_sam [CC-BY-NC]\n- [2-78562-A.ogg]: clip derived from alarm_clock_ringing_01.wav (http://www.freesound.org/people/joedeshon/sounds/78562/) by joedeshon [CC-BY]\n- [2-78562-B.ogg]: clip derived from alarm_clock_ringing_01.wav (http://www.freesound.org/people/joedeshon/sounds/78562/) by joedeshon [CC-BY]\n- [2-89516-A.ogg]: clip derived from Bedroom_Electric Alarm Clock Bell Sound.mp3 (http://www.freesound.org/people/CGEffex/sounds/89516/) by CGEffex [CC-BY]\n- [2-96063-A.ogg]: clip derived from alarm beep electronic.wav (http://www.freesound.org/people/reecord2/sounds/96063/) by reecord2 [CC0]\n- [3-117793-A.ogg]: clip derived from Radio Alarm.wav (http://www.freesound.org/people/gerrygregory/sounds/117793/) by gerrygregory [CC0]\n- [3-117883-A.ogg]: clip derived from Alarm Clock.WAV (http://www.freesound.org/people/ArcLegend05/sounds/117883/) by ArcLegend05 [CC-Sampling+]\n- [3-120526-A.ogg]: clip derived from Alarm Clock\'s alarm on.wav (http://www.freesound.org/people/PlayPauseAndRewind/sounds/120526/) by PlayPauseAndRewind [CC0]\n- [3-120526-B.ogg]: clip derived from Alarm Clock\'s alarm on.wav (http://www.freesound.org/people/PlayPauseAndRewind/sounds/120526/) by PlayPauseAndRewind [CC0]\n- [3-131943-A.ogg]: clip derived from Wind up clock with twin bell alarm (http://www.freesound.org/people/AngryAdam/sounds/131943/) by AngryAdam [CC0]\n- [3-132340-A.ogg]: clip derived from Old alarm clock (ticking and ringing) (http://www.freesound.org/people/PlayPauseAndRewind/sounds/132340/) by PlayPauseAndRewind [CC0]\n- [3-139958-A.ogg]: clip derived from Clock.Windup.Bell.Antique.wav (http://www.freesound.org/people/jessepash/sounds/139958/) by jessepash [CC0]\n- [3-148297-A.ogg]: clip derived from Alarm beep.wav (http://www.freesound.org/people/are16ocean/sounds/148297/) by are16ocean [CC0]\n- [4-151242-A.ogg]: clip derived from Travel Alarm.wav (http://www.freesound.org/people/Tempouser/sounds/151242/) by Tempouser [CC-BY-NC]\n- [4-169508-A.ogg]: clip derived from Alarm Clock (High Quality).wav (http://www.freesound.org/people/Olalr/sounds/169508/) by Olalr [CC0]\n- [4-172366-A.ogg]: clip derived from Alarm clock going off (http://www.freesound.org/people/avakas/sounds/172366/) by avakas [CC-BY]\n- [4-189332-A.ogg]: clip derived from Alarm.wav (http://www.freesound.org/people/sangtao/sounds/189332/) by sangtao [CC0]\n- [4-198841-A.ogg]: clip derived from Analog Alarm Clock (http://www.freesound.org/people/bone666138/sounds/198841/) by bone666138 [CC-BY]\n- [4-204684-A.ogg]: clip derived from Alarm.wav (http://www.freesound.org/people/Fredemo/sounds/204684/) by Fredemo [CC-BY-NC]\n- [4-209536-A.ogg]: clip derived from alarm clock.wav (http://www.freesound.org/people/jackjames_tomknevitt/sounds/209536/) by jackjames_tomknevitt [CC0]\n- [4-209698-A.ogg]: clip derived from Analog Timer (http://www.freesound.org/people/EagleStealthTeam/sounds/209698/) by EagleStealthTeam [CC0]\n- [5-210612-A.ogg]: clip derived from Alarm Clock (http://www.freesound.org/people/mefrancis13/sounds/210612/) by mefrancis13 [CC0]\n- [5-211408-A.ogg]: clip derived from Alarm Clock Bedside Clockwork.wav (http://www.freesound.org/people/squeakyballoon/sounds/211408/) by squeakyballoon [CC-BY]\n- [5-219242-A.ogg]: clip derived from alarm clock.wav (http://www.freesound.org/people/ZyryTSounds/sounds/219242/) by ZyryTSounds [CC0]\n- [5-219242-B.ogg]: clip derived from alarm clock.wav (http://www.freesound.org/people/ZyryTSounds/sounds/219242/) by ZyryTSounds [CC0]\n- [5-223176-A.ogg]: clip derived from Generic Alarm Clock (http://www.freesound.org/people/Yoyodaman234/sounds/223176/) by Yoyodaman234 [CC0]\n- [5-233645-A.ogg]: clip derived from Alarm Clock Digital (http://www.freesound.org/people/zanox/sounds/233645/) by zanox [CC-BY-NC]\n- [5-234247-A.ogg]: clip derived from Alarm (http://www.freesound.org/people/rivernile7/sounds/234247/) by rivernile7 [CC-BY]\n- [5-250629-A.ogg]: clip derived from Alarm1.mp3 (http://www.freesound.org/people/kwahmah_02/sounds/250629/) by kwahmah_02 [CC-BY]\n\n409 - Clock tick:\n- [1-21934-A.ogg]: clip derived from 20060810.peters.clock.01.flac (http://www.freesound.org/people/dobroide/sounds/21934/) by dobroide [CC-BY]\n- [1-21935-A.ogg]: clip derived from 20060810.peters.clock.02.flac (http://www.freesound.org/people/dobroide/sounds/21935/) by dobroide [CC-BY]\n- [1-35687-A.ogg]: clip derived from Clock-grandfather-ticks & striking once.MP3 (http://www.freesound.org/people/Bansemer/sounds/35687/) by Bansemer [CC0]\n- [1-42139-A.ogg]: clip derived from Old Scots Clock - Ticking.wav (http://www.freesound.org/people/fauxpress/sounds/42139/) by fauxpress [CC0]\n- [1-48413-A.ogg]: clip derived from clock.wav (http://www.freesound.org/people/cynical/sounds/48413/) by cynical [CC0]\n- [1-57163-A.ogg]: clip derived from delphis CLOCK FX LOOP 5.wav (http://www.freesound.org/people/delphidebrain/sounds/57163/) by delphidebrain [CC-BY]\n- [1-62849-A.ogg]: clip derived from clock_tick_close.wav (http://www.freesound.org/people/life_interconnected/sounds/62849/) by life_interconnected [CC0]\n- [1-62850-A.ogg]: clip derived from clock_tick_realistic.wav (http://www.freesound.org/people/life_interconnected/sounds/62850/) by life_interconnected [CC0]\n- [2-119748-A.ogg]: clip derived from Clock Ticking.wav (http://www.freesound.org/people/jesabat/sounds/119748/) by jesabat [CC-BY]\n- [2-127108-A.ogg]: clip derived from grandfather_clock_ticking_60_seconds.wav (http://www.freesound.org/people/joedeshon/sounds/127108/) by joedeshon [CC-BY]\n- [2-131943-A.ogg]: clip derived from Wind up clock with twin bell alarm (http://www.freesound.org/people/AngryAdam/sounds/131943/) by AngryAdam [CC0]\n- [2-134700-A.ogg]: clip derived from wall clock tw2.WAV (http://www.freesound.org/people/Setuniman/sounds/134700/) by Setuniman [CC-BY]\n- [2-135728-A.ogg]: clip derived from tickende-uhr-fewo-leipzig.wav (http://www.freesound.org/people/massenbachhausen/sounds/135728/) by massenbachhausen [CC-BY]\n- [2-140147-A.ogg]: clip derived from Mantel Clock Ticking.wav (http://www.freesound.org/people/Tewkesound/sounds/140147/) by Tewkesound [CC-BY]\n- [2-141584-A.ogg]: clip derived from Plastic Quartz Clock Tick (http://www.freesound.org/people/MentalSanityOff/sounds/141584/) by MentalSanityOff [CC0]\n- [2-88724-A.ogg]: clip derived from Cuckoo Clock-Twelve Noon.wav (http://www.freesound.org/people/lonemonk/sounds/88724/) by lonemonk [CC-BY]\n- [3-142593-A.ogg]: clip derived from Ticking.aif (http://www.freesound.org/people/timgormly/sounds/142593/) by timgormly [CC0]\n- [3-143933-A.ogg]: clip derived from clock.wav (http://www.freesound.org/people/wentzelj/sounds/143933/) by wentzelj [CC0]\n- [3-150363-A.ogg]: clip derived from GB_wall.flac (http://www.freesound.org/people/Zbylut/sounds/150363/) by Zbylut [CC0]\n- [3-164688-A.ogg]: clip derived from Ticking_clock.wav (http://www.freesound.org/people/Adam_N/sounds/164688/) by Adam_N [CC0]\n- [3-170377-A.ogg]: clip derived from TicTac of a wall clock (http://www.freesound.org/people/felix.blume/sounds/170377/) by felix.blume [CC0]\n- [3-170383-A.ogg]: clip derived from Cuckoo clock (http://www.freesound.org/people/finalCrystine/sounds/170383/) by finalCrystine [CC0]\n- [3-171012-A.ogg]: clip derived from Tick 3.aif (http://www.freesound.org/people/Goup_1/sounds/171012/) by Goup_1 [CC-BY]\n- [3-171041-A.ogg]: clip derived from Old Electromechanical Clock Ticking "Yantar".wav (http://www.freesound.org/people/ST303/sounds/171041/) by ST303 [CC0]\n- [4-175945-A.ogg]: clip derived from TimerRecording.wav (http://www.freesound.org/people/Ovaltine%20Vortex/sounds/175945/) by Ovaltine Vortex [CC0]\n- [4-179984-A.ogg]: clip derived from Ansonia Clock Strikes 11 (http://www.freesound.org/people/kangaroovindaloo/sounds/179984/) by kangaroovindaloo [CC0]\n- [4-181035-A.ogg]: clip derived from Tick Tock Clock - Close Mic.wav (http://www.freesound.org/people/Toby_Crane/sounds/181035/) by Toby_Crane [CC-BY]\n- [4-181865-A.ogg]: clip derived from Ticking Clock (http://www.freesound.org/people/LeoDeSafo/sounds/181865/) by LeoDeSafo [CC0]\n- [4-188033-A.ogg]: clip derived from Ticking Clock (http://www.freesound.org/people/AntumDeluge/sounds/188033/) by AntumDeluge [CC0]\n- [4-191327-A.ogg]: clip derived from TICKTOCK.wav (http://www.freesound.org/people/needforsuv/sounds/191327/) by needforsuv [CC0]\n- [4-194711-A.ogg]: clip derived from Old 60s soviet clock ticking (http://www.freesound.org/people/doxent/sounds/194711/) by doxent [CC-BY]\n- [4-198965-A.ogg]: clip derived from mantelclock.wav (http://www.freesound.org/people/photog123/sounds/198965/) by photog123 [CC0]\n- [5-201194-A.ogg]: clip derived from Mechanical clock ticking (http://www.freesound.org/people/opticalnoise/sounds/201194/) by opticalnoise [CC-BY]\n- [5-208624-A.ogg]: clip derived from Old Clock Ticking (http://www.freesound.org/people/LanDub/sounds/208624/) by LanDub [CC0]\n- [5-209698-A.ogg]: clip derived from Analog Timer (http://www.freesound.org/people/EagleStealthTeam/sounds/209698/) by EagleStealthTeam [CC0]\n- [5-209833-A.ogg]: clip derived from Old Clock Tick Tock (http://www.freesound.org/people/spaceemotion/sounds/209833/) by spaceemotion [CC-BY]\n- [5-210571-A.ogg]: clip derived from wall_clock.wav (http://www.freesound.org/people/Sclolex/sounds/210571/) by Sclolex [CC0]\n- [5-212181-A.ogg]: clip derived from Loopable Ticking Clock (http://www.freesound.org/people/OwlStorm/sounds/212181/) by OwlStorm [CC-BY]\n- [5-219342-A.ogg]: clip derived from clock.wav (http://www.freesound.org/people/fatbass/sounds/219342/) by fatbass [CC0]\n- [5-235671-A.ogg]: clip derived from clock tick.aif (http://www.freesound.org/people/gpenn76/sounds/235671/) by gpenn76 [CC-BY]\n\n410 - Glass breaking:\n- [1-20133-A.ogg]: clip derived from breaking_glass.wav (http://www.freesound.org/people/gezortenplotz/sounds/20133/) by gezortenplotz [CC-BY]\n- [1-84536-A.ogg]: clip derived from glass.wav (http://www.freesound.org/people/cmusounddesign/sounds/84536/) by cmusounddesign [CC-BY]\n- [1-84704-A.ogg]: clip derived from Glass Break 1.wav (http://www.freesound.org/people/cmusounddesign/sounds/84704/) by cmusounddesign [CC-BY]\n- [1-84705-A.ogg]: clip derived from Glass Breaking.wav (http://www.freesound.org/people/cmusounddesign/sounds/84705/) by cmusounddesign [CC-BY]\n- [1-85168-A.ogg]: clip derived from jf Glass Breaking.wav (http://www.freesound.org/people/cmusounddesign/sounds/85168/) by cmusounddesign [CC-BY]\n- [1-85184-A.ogg]: clip derived from mp glass break.wav (http://www.freesound.org/people/cmusounddesign/sounds/85184/) by cmusounddesign [CC-BY]\n- [1-88807-A.ogg]: clip derived from gb11.aif (http://www.freesound.org/people/lsprice/sounds/88807/) by lsprice [CC-BY]\n- [1-977-A.ogg]: clip derived from RBH Glass_Break 01.wav (http://www.freesound.org/people/RHumphries/sounds/977/) by RHumphries [CC-BY]\n- [2-106881-A.ogg]: clip derived from 200 pounds stepping on a glass object.aif (http://www.freesound.org/people/Tomlija/sounds/106881/) by Tomlija [CC-BY]\n- [2-112213-A.ogg]: clip derived from Window breaking.MP3 (http://www.freesound.org/people/m1a2t3z4/sounds/112213/) by m1a2t3z4 [CC0]\n- [2-112213-B.ogg]: clip derived from Window breaking.MP3 (http://www.freesound.org/people/m1a2t3z4/sounds/112213/) by m1a2t3z4 [CC0]\n- [2-141563-A.ogg]: clip derived from Glass Break (http://www.freesound.org/people/avrahamy/sounds/141563/) by avrahamy [CC0]\n- [2-173559-A.ogg]: clip derived from 002.wav (http://www.freesound.org/people/gevaroy/sounds/173559/) by gevaroy [CC0]\n- [2-173607-A.ogg]: clip derived from 009.wav (http://www.freesound.org/people/gevaroy/sounds/173607/) by gevaroy [CC0]\n- [2-173618-A.ogg]: clip derived from 027.wav (http://www.freesound.org/people/gevaroy/sounds/173618/) by gevaroy [CC0]\n- [2-250710-A.ogg]: clip derived from Glass break 1 (http://www.freesound.org/people/Aiwha/sounds/250710/) by Aiwha [CC-BY]\n- [3-203371-A.ogg]: clip derived from glass-explosion_01.ogg (http://www.freesound.org/people/C_Rogers/sounds/203371/) by C_Rogers [CC0]\n- [3-203373-A.ogg]: clip derived from glass-shattering_03.ogg (http://www.freesound.org/people/C_Rogers/sounds/203373/) by C_Rogers [CC0]\n- [3-203374-A.ogg]: clip derived from glass-shattering_02.ogg (http://www.freesound.org/people/C_Rogers/sounds/203374/) by C_Rogers [CC0]\n- [3-203375-A.ogg]: clip derived from glass-shattering_01.ogg (http://www.freesound.org/people/C_Rogers/sounds/203375/) by C_Rogers [CC0]\n- [3-203377-A.ogg]: clip derived from glass-shattering_05.ogg (http://www.freesound.org/people/C_Rogers/sounds/203377/) by C_Rogers [CC0]\n- [3-216280-A.ogg]: clip derived from Breaking_Glass_21.wav (http://www.freesound.org/people/RSilveira_88/sounds/216280/) by RSilveira_88 [CC-BY]\n- [3-216281-A.ogg]: clip derived from Breaking_Glass_13.wav (http://www.freesound.org/people/RSilveira_88/sounds/216281/) by RSilveira_88 [CC-BY]\n- [3-216284-A.ogg]: clip derived from Breaking_Glass_10.wav (http://www.freesound.org/people/RSilveira_88/sounds/216284/) by RSilveira_88 [CC-BY]\n- [4-204115-A.ogg]: clip derived from glass01.flac (http://www.freesound.org/people/Craxic/sounds/204115/) by Craxic [CC-BY]\n- [4-204119-A.ogg]: clip derived from glass03.flac (http://www.freesound.org/people/Craxic/sounds/204119/) by Craxic [CC-BY]\n- [4-204121-A.ogg]: clip derived from glass02.flac (http://www.freesound.org/people/Craxic/sounds/204121/) by Craxic [CC-BY]\n- [4-204123-A.ogg]: clip derived from glass04.flac (http://www.freesound.org/people/Craxic/sounds/204123/) by Craxic [CC-BY]\n- [4-204777-A.ogg]: clip derived from Breaking Glass  (http://www.freesound.org/people/ngruber/sounds/204777/) by ngruber [CC0]\n- [4-204777-B.ogg]: clip derived from Breaking Glass  (http://www.freesound.org/people/ngruber/sounds/204777/) by ngruber [CC0]\n- [4-204777-C.ogg]: clip derived from Breaking Glass  (http://www.freesound.org/people/ngruber/sounds/204777/) by ngruber [CC0]\n- [4-212698-A.ogg]: clip derived from Breaking a glass bottle (http://www.freesound.org/people/dasebr/sounds/212698/) by dasebr [CC0]\n- [5-221528-A.ogg]: clip derived from Glass Break (http://www.freesound.org/people/unfa/sounds/221528/) by unfa [CC-BY]\n- [5-221529-A.ogg]: clip derived from Glass Shards 1 (http://www.freesound.org/people/unfa/sounds/221529/) by unfa [CC-BY]\n- [5-233605-A.ogg]: clip derived from Breaking Glass #3.wav (http://www.freesound.org/people/Agaxly/sounds/233605/) by Agaxly [CC0]\n- [5-233607-A.ogg]: clip derived from Breaking Glass #1.wav (http://www.freesound.org/people/Agaxly/sounds/233607/) by Agaxly [CC0]\n- [5-257642-A.ogg]: clip derived from Glass breaking.wav (http://www.freesound.org/people/speedygonzo/sounds/257642/) by speedygonzo [CC-BY]\n- [5-260432-A.ogg]: clip derived from Glass Break - Small Jar 02 (http://www.freesound.org/people/RoganDerrick/sounds/260432/) by RoganDerrick [CC-BY]\n- [5-260433-A.ogg]: clip derived from Glass Break - Small Jar 01 (http://www.freesound.org/people/RoganDerrick/sounds/260433/) by RoganDerrick [CC-BY]\n- [5-260434-A.ogg]: clip derived from Glass Break - Medium Jar (http://www.freesound.org/people/RoganDerrick/sounds/260434/) by RoganDerrick [CC-BY]\n\n501 - Helicopter:\n- [1-172649-A.ogg]: clip derived from Small Helicopter Takes Off (http://www.freesound.org/people/senorstudy/sounds/172649/) by senorstudy [CC0]\n- [1-172649-B.ogg]: clip derived from Small Helicopter Takes Off (http://www.freesound.org/people/senorstudy/sounds/172649/) by senorstudy [CC0]\n- [1-172649-C.ogg]: clip derived from Small Helicopter Takes Off (http://www.freesound.org/people/senorstudy/sounds/172649/) by senorstudy [CC0]\n- [1-172649-D.ogg]: clip derived from Small Helicopter Takes Off (http://www.freesound.org/people/senorstudy/sounds/172649/) by senorstudy [CC0]\n- [1-172649-E.ogg]: clip derived from Small Helicopter Takes Off (http://www.freesound.org/people/senorstudy/sounds/172649/) by senorstudy [CC0]\n- [1-172649-F.ogg]: clip derived from Small Helicopter Takes Off (http://www.freesound.org/people/senorstudy/sounds/172649/) by senorstudy [CC0]\n- [1-181071-A.ogg]: clip derived from helicopter1.wav (http://www.freesound.org/people/dwareing/sounds/181071/) by dwareing [CC0]\n- [1-181071-B.ogg]: clip derived from helicopter1.wav (http://www.freesound.org/people/dwareing/sounds/181071/) by dwareing [CC0]\n- [2-188822-A.ogg]: clip derived from Helicopter landing (http://www.freesound.org/people/peridactyloptrix/sounds/188822/) by peridactyloptrix [CC0]\n- [2-188822-B.ogg]: clip derived from Helicopter landing (http://www.freesound.org/people/peridactyloptrix/sounds/188822/) by peridactyloptrix [CC0]\n- [2-188822-C.ogg]: clip derived from Helicopter landing (http://www.freesound.org/people/peridactyloptrix/sounds/188822/) by peridactyloptrix [CC0]\n- [2-188822-D.ogg]: clip derived from Helicopter landing (http://www.freesound.org/people/peridactyloptrix/sounds/188822/) by peridactyloptrix [CC0]\n- [2-37806-A.ogg]: clip derived from helicopters2.mp3 (http://www.freesound.org/people/mw_1984/sounds/37806/) by mw_1984 [CC0]\n- [2-37806-B.ogg]: clip derived from helicopters2.mp3 (http://www.freesound.org/people/mw_1984/sounds/37806/) by mw_1984 [CC0]\n- [2-37806-C.ogg]: clip derived from helicopters2.mp3 (http://www.freesound.org/people/mw_1984/sounds/37806/) by mw_1984 [CC0]\n- [2-37806-D.ogg]: clip derived from helicopters2.mp3 (http://www.freesound.org/people/mw_1984/sounds/37806/) by mw_1984 [CC0]\n- [3-150979-A.ogg]: clip derived from helicopter.wav (http://www.freesound.org/people/soundingoffagain/sounds/150979/) by soundingoffagain [CC0]\n- [3-150979-B.ogg]: clip derived from helicopter.wav (http://www.freesound.org/people/soundingoffagain/sounds/150979/) by soundingoffagain [CC0]\n- [3-150979-C.ogg]: clip derived from helicopter.wav (http://www.freesound.org/people/soundingoffagain/sounds/150979/) by soundingoffagain [CC0]\n- [3-154926-A.ogg]: clip derived from London Exterior  helicopter passes overhead.mp3 (http://www.freesound.org/people/kerouac21/sounds/154926/) by kerouac21 [CC0]\n- [3-154926-B.ogg]: clip derived from London Exterior  helicopter passes overhead.mp3 (http://www.freesound.org/people/kerouac21/sounds/154926/) by kerouac21 [CC0]\n- [3-68630-A.ogg]: clip derived from helicopter take off distance.wav (http://www.freesound.org/people/bwav/sounds/68630/) by bwav [CC0]\n- [3-68630-B.ogg]: clip derived from helicopter take off distance.wav (http://www.freesound.org/people/bwav/sounds/68630/) by bwav [CC0]\n- [3-68630-C.ogg]: clip derived from helicopter take off distance.wav (http://www.freesound.org/people/bwav/sounds/68630/) by bwav [CC0]\n- [4-125929-A.ogg]: clip derived from helicopter-construction.aif (http://www.freesound.org/people/alienistcog/sounds/125929/) by alienistcog [CC0]\n- [4-161579-A.ogg]: clip derived from Helicopter Flyover (http://www.freesound.org/people/doctor567/sounds/161579/) by doctor567 [CC0]\n- [4-161579-B.ogg]: clip derived from Helicopter Flyover (http://www.freesound.org/people/doctor567/sounds/161579/) by doctor567 [CC0]\n- [4-175000-A.ogg]: clip derived from Helicopter Fly By (http://www.freesound.org/people/altfuture/sounds/175000/) by altfuture [CC0]\n- [4-175000-B.ogg]: clip derived from Helicopter Fly By (http://www.freesound.org/people/altfuture/sounds/175000/) by altfuture [CC0]\n- [4-175000-C.ogg]: clip derived from Helicopter Fly By (http://www.freesound.org/people/altfuture/sounds/175000/) by altfuture [CC0]\n- [4-193480-A.ogg]: clip derived from Helicopter circles three times overhead (http://www.freesound.org/people/trgame/sounds/193480/) by trgame [CC0]\n- [4-193480-B.ogg]: clip derived from Helicopter circles three times overhead (http://www.freesound.org/people/trgame/sounds/193480/) by trgame [CC0]\n- [5-177957-A.ogg]: clip derived from Helicopter - inside - Takeoff (http://www.freesound.org/people/senorstudy/sounds/177957/) by senorstudy [CC0]\n- [5-177957-B.ogg]: clip derived from Helicopter - inside - Takeoff (http://www.freesound.org/people/senorstudy/sounds/177957/) by senorstudy [CC0]\n- [5-177957-C.ogg]: clip derived from Helicopter - inside - Takeoff (http://www.freesound.org/people/senorstudy/sounds/177957/) by senorstudy [CC0]\n- [5-177957-D.ogg]: clip derived from Helicopter - inside - Takeoff (http://www.freesound.org/people/senorstudy/sounds/177957/) by senorstudy [CC0]\n- [5-177957-E.ogg]: clip derived from Helicopter - inside - Takeoff (http://www.freesound.org/people/senorstudy/sounds/177957/) by senorstudy [CC0]\n- [5-191131-A.ogg]: clip derived from helicopter and thunder (http://www.freesound.org/people/yacou/sounds/191131/) by yacou [CC0]\n- [5-205898-A.ogg]: clip derived from Helicopter Flight recorded from inside.mp3 (http://www.freesound.org/people/Spankous/sounds/205898/) by Spankous [CC0]\n- [5-220955-A.ogg]: clip derived from Helicopter.wav (http://www.freesound.org/people/Islabonita/sounds/220955/) by Islabonita [CC0]\n\n502 - Chainsaw:\n- [1-116765-A.ogg]: clip derived from chainsaw.wav (http://www.freesound.org/people/UncleSigmund/sounds/116765/) by UncleSigmund [CC0]\n- [1-19898-A.ogg]: clip derived from CHAINSAW.wav (http://www.freesound.org/people/JFBSAUVE/sounds/19898/) by JFBSAUVE [CC-BY]\n- [1-19898-B.ogg]: clip derived from CHAINSAW.wav (http://www.freesound.org/people/JFBSAUVE/sounds/19898/) by JFBSAUVE [CC-BY]\n- [1-19898-C.ogg]: clip derived from CHAINSAW.wav (http://www.freesound.org/people/JFBSAUVE/sounds/19898/) by JFBSAUVE [CC-BY]\n- [1-47250-A.ogg]: clip derived from chainsaw.WAV (http://www.freesound.org/people/stomachache/sounds/47250/) by stomachache [CC0]\n- [1-47250-B.ogg]: clip derived from chainsaw.WAV (http://www.freesound.org/people/stomachache/sounds/47250/) by stomachache [CC0]\n- [1-64398-A.ogg]: clip derived from Chainsaw Crosscutting  3.wav (http://www.freesound.org/people/Benboncan/sounds/64398/) by Benboncan [CC-BY]\n- [1-64398-B.ogg]: clip derived from Chainsaw Crosscutting  3.wav (http://www.freesound.org/people/Benboncan/sounds/64398/) by Benboncan [CC-BY]\n- [2-50667-A.ogg]: clip derived from 080320 Chainsaw start, branches falling and engine cut.wav (http://www.freesound.org/people/BoilingSand/sounds/50667/) by BoilingSand [CC-BY]\n- [2-50667-B.ogg]: clip derived from 080320 Chainsaw start, branches falling and engine cut.wav (http://www.freesound.org/people/BoilingSand/sounds/50667/) by BoilingSand [CC-BY]\n- [2-50668-A.ogg]: clip derived from Chainsaw with Bobcat idling and tree bits falling 080320.wav (http://www.freesound.org/people/BoilingSand/sounds/50668/) by BoilingSand [CC-BY]\n- [2-50668-B.ogg]: clip derived from Chainsaw with Bobcat idling and tree bits falling 080320.wav (http://www.freesound.org/people/BoilingSand/sounds/50668/) by BoilingSand [CC-BY]\n- [2-68391-A.ogg]: clip derived from Chainsaw - Tree cases.WAV (http://www.freesound.org/people/Ohrwurm/sounds/68391/) by Ohrwurm [CC0]\n- [2-68391-B.ogg]: clip derived from Chainsaw - Tree cases.WAV (http://www.freesound.org/people/Ohrwurm/sounds/68391/) by Ohrwurm [CC0]\n- [2-77945-A.ogg]: clip derived from chainsaw_Husqvarna_385XPG.wav (http://www.freesound.org/people/theTone/sounds/77945/) by theTone [CC-BY]\n- [2-77945-B.ogg]: clip derived from chainsaw_Husqvarna_385XPG.wav (http://www.freesound.org/people/theTone/sounds/77945/) by theTone [CC-BY]\n- [3-118656-A.ogg]: clip derived from Chainsaw in the garden.wav (http://www.freesound.org/people/Kyster/sounds/118656/) by Kyster [CC-BY]\n- [3-118657-A.ogg]: clip derived from chainsaw and little tree.wav (http://www.freesound.org/people/Kyster/sounds/118657/) by Kyster [CC-BY]\n- [3-118657-B.ogg]: clip derived from chainsaw and little tree.wav (http://www.freesound.org/people/Kyster/sounds/118657/) by Kyster [CC-BY]\n- [3-118658-A.ogg]: clip derived from chainsaw vs chestnut tree.wav (http://www.freesound.org/people/Kyster/sounds/118658/) by Kyster [CC-BY]\n- [3-118658-B.ogg]: clip derived from chainsaw vs chestnut tree.wav (http://www.freesound.org/people/Kyster/sounds/118658/) by Kyster [CC-BY]\n- [3-118972-A.ogg]: clip derived from Chainsaw.wav (http://www.freesound.org/people/esperri/sounds/118972/) by esperri [CC0]\n- [3-118972-B.ogg]: clip derived from Chainsaw.wav (http://www.freesound.org/people/esperri/sounds/118972/) by esperri [CC0]\n- [3-165856-A.ogg]: clip derived from Chainsawing.wav (http://www.freesound.org/people/Puniho/sounds/165856/) by Puniho [CC-BY]\n- [4-149294-A.ogg]: clip derived from chainsaw_cutting01.wav (http://www.freesound.org/people/Vocalphobic/sounds/149294/) by Vocalphobic [CC0]\n- [4-149294-B.ogg]: clip derived from chainsaw_cutting01.wav (http://www.freesound.org/people/Vocalphobic/sounds/149294/) by Vocalphobic [CC0]\n- [4-157611-A.ogg]: clip derived from Exterior_Chainsaw.wav (http://www.freesound.org/people/DJillcom/sounds/157611/) by DJillcom [CC0]\n- [4-157611-B.ogg]: clip derived from Exterior_Chainsaw.wav (http://www.freesound.org/people/DJillcom/sounds/157611/) by DJillcom [CC0]\n- [4-165823-A.ogg]: clip derived from Chainsaw Rampage in the Forest (http://www.freesound.org/people/unfa/sounds/165823/) by unfa [CC-BY]\n- [4-165823-B.ogg]: clip derived from Chainsaw Rampage in the Forest (http://www.freesound.org/people/unfa/sounds/165823/) by unfa [CC-BY]\n- [4-169127-A.ogg]: clip derived from cutting_tree.wav (http://www.freesound.org/people/fkurz/sounds/169127/) by fkurz [CC0]\n- [4-169127-B.ogg]: clip derived from cutting_tree.wav (http://www.freesound.org/people/fkurz/sounds/169127/) by fkurz [CC0]\n- [5-170338-A.ogg]: clip derived from Chainsaws (distant) (http://www.freesound.org/people/micadoe/sounds/170338/) by micadoe [CC0]\n- [5-170338-B.ogg]: clip derived from Chainsaws (distant) (http://www.freesound.org/people/micadoe/sounds/170338/) by micadoe [CC0]\n- [5-171653-A.ogg]: clip derived from Chainsaw cutting through flesh (http://www.freesound.org/people/Audionautics/sounds/171653/) by Audionautics [CC-BY]\n- [5-185579-A.ogg]: clip derived from Chainsaw Cutting.wav (http://www.freesound.org/people/lonemonk/sounds/185579/) by lonemonk [CC-BY]\n- [5-185579-B.ogg]: clip derived from Chainsaw Cutting.wav (http://www.freesound.org/people/lonemonk/sounds/185579/) by lonemonk [CC-BY]\n- [5-216370-A.ogg]: clip derived from Chainsaw.wav (http://www.freesound.org/people/RSilveira_88/sounds/216370/) by RSilveira_88 [CC-BY]\n- [5-216370-B.ogg]: clip derived from Chainsaw.wav (http://www.freesound.org/people/RSilveira_88/sounds/216370/) by RSilveira_88 [CC-BY]\n- [5-222524-A.ogg]: clip derived from wood chipper and chainsaw (http://www.freesound.org/people/Darrenlibrarian/sounds/222524/) by Darrenlibrarian [CC0]\n\n503 - Siren:\n- [1-31482-A.ogg]: clip derived from 20070224.siren.flac (http://www.freesound.org/people/dobroide/sounds/31482/) by dobroide [CC-BY]\n- [1-31482-B.ogg]: clip derived from 20070224.siren.flac (http://www.freesound.org/people/dobroide/sounds/31482/) by dobroide [CC-BY]\n- [1-54084-A.ogg]: clip derived from civildefense.wav (http://www.freesound.org/people/guitarguy1985/sounds/54084/) by guitarguy1985 [CC0]\n- [1-76831-A.ogg]: clip derived from 1000000007.aif (http://www.freesound.org/people/Rodney%20Gilliam/sounds/76831/) by Rodney Gilliam [CC-Sampling+]\n- [1-76831-B.ogg]: clip derived from 1000000007.aif (http://www.freesound.org/people/Rodney%20Gilliam/sounds/76831/) by Rodney Gilliam [CC-Sampling+]\n- [1-76831-C.ogg]: clip derived from 1000000007.aif (http://www.freesound.org/people/Rodney%20Gilliam/sounds/76831/) by Rodney Gilliam [CC-Sampling+]\n- [1-76831-D.ogg]: clip derived from 1000000007.aif (http://www.freesound.org/people/Rodney%20Gilliam/sounds/76831/) by Rodney Gilliam [CC-Sampling+]\n- [1-76831-E.ogg]: clip derived from 1000000007.aif (http://www.freesound.org/people/Rodney%20Gilliam/sounds/76831/) by Rodney Gilliam [CC-Sampling+]\n- [2-43802-A.ogg]: clip derived from OldCarHornsAndSirens.wav (http://www.freesound.org/people/daveincamas/sounds/43802/) by daveincamas [CC-BY]\n- [2-43806-A.ogg]: clip derived from Siren.wav (http://www.freesound.org/people/daveincamas/sounds/43806/) by daveincamas [CC-BY]\n- [2-70052-A.ogg]: clip derived from Sirens.wav (http://www.freesound.org/people/Trinity101/sounds/70052/) by Trinity101 [CC-BY-NC]\n- [2-70052-B.ogg]: clip derived from Sirens.wav (http://www.freesound.org/people/Trinity101/sounds/70052/) by Trinity101 [CC-BY-NC]\n- [2-70936-A.ogg]: clip derived from police.wav (http://www.freesound.org/people/guitarguy1985/sounds/70936/) by guitarguy1985 [CC0]\n- [2-70938-A.ogg]: clip derived from police2.wav (http://www.freesound.org/people/guitarguy1985/sounds/70938/) by guitarguy1985 [CC0]\n- [2-70939-A.ogg]: clip derived from police3.wav (http://www.freesound.org/people/guitarguy1985/sounds/70939/) by guitarguy1985 [CC0]\n- [2-72688-A.ogg]: clip derived from 550AT 3x1M.wav (http://www.freesound.org/people/djscrizzle/sounds/72688/) by djscrizzle [CC-BY]\n- [3-51376-A.ogg]: clip derived from Siren.wav (http://www.freesound.org/people/acclivity/sounds/51376/) by acclivity [CC-BY-NC]\n- [3-51731-A.ogg]: clip derived from ICE passing by, then an ambulance with siren.WAV (http://www.freesound.org/people/Connum/sounds/51731/) by Connum [CC-BY-NC]\n- [3-51731-B.ogg]: clip derived from ICE passing by, then an ambulance with siren.WAV (http://www.freesound.org/people/Connum/sounds/51731/) by Connum [CC-BY-NC]\n- [3-51909-A.ogg]: clip derived from t128.wav (http://www.freesound.org/people/guitarguy1985/sounds/51909/) by guitarguy1985 [CC0]\n- [3-51909-B.ogg]: clip derived from t128.wav (http://www.freesound.org/people/guitarguy1985/sounds/51909/) by guitarguy1985 [CC0]\n- [3-58772-A.ogg]: clip derived from siren on prague castle.wav (http://www.freesound.org/people/auvikogue/sounds/58772/) by auvikogue [CC-BY-NC]\n- [3-62878-A.ogg]: clip derived from police siren2.wav (http://www.freesound.org/people/mungous/sounds/62878/) by mungous [CC-BY-NC]\n- [3-62878-B.ogg]: clip derived from police siren2.wav (http://www.freesound.org/people/mungous/sounds/62878/) by mungous [CC-BY-NC]\n- [4-102871-A.ogg]: clip derived from 20100806.stockholm.sirens.wav (http://www.freesound.org/people/dobroide/sounds/102871/) by dobroide [CC-BY]\n- [4-111671-A.ogg]: clip derived from policeSiren.wav (http://www.freesound.org/people/FatLane/sounds/111671/) by FatLane [CC-BY]\n- [4-111671-B.ogg]: clip derived from policeSiren.wav (http://www.freesound.org/people/FatLane/sounds/111671/) by FatLane [CC-BY]\n- [4-121532-A.ogg]: clip derived from 01052010 Tornado siren test.flac (http://www.freesound.org/people/thaighaudio/sounds/121532/) by thaighaudio [CC0]\n- [4-67358-A.ogg]: clip derived from Siren 2.wav (http://www.freesound.org/people/Benboncan/sounds/67358/) by Benboncan [CC-BY]\n- [4-80761-A.ogg]: clip derived from Siren (http://www.freesound.org/people/fmagrao/sounds/80761/) by fmagrao [CC-BY]\n- [4-90014-A.ogg]: clip derived from police siren perpignan.wav (http://www.freesound.org/people/thfc140491/sounds/90014/) by thfc140491 [CC-BY]\n- [4-90014-B.ogg]: clip derived from police siren perpignan.wav (http://www.freesound.org/people/thfc140491/sounds/90014/) by thfc140491 [CC-BY]\n- [5-117118-A.ogg]: clip derived from siren_low.wav (http://www.freesound.org/people/UncleSigmund/sounds/117118/) by UncleSigmund [CC0]\n- [5-117120-A.ogg]: clip derived from siren_med.wav (http://www.freesound.org/people/UncleSigmund/sounds/117120/) by UncleSigmund [CC0]\n- [5-117122-A.ogg]: clip derived from siren_short_low.wav (http://www.freesound.org/people/UncleSigmund/sounds/117122/) by UncleSigmund [CC0]\n- [5-133989-A.ogg]: clip derived from Whelen WPS-3016 Siren.mp3 (http://www.freesound.org/people/2887679652/sounds/133989/) by 2887679652 [CC0]\n- [5-133989-B.ogg]: clip derived from Whelen WPS-3016 Siren.mp3 (http://www.freesound.org/people/2887679652/sounds/133989/) by 2887679652 [CC0]\n- [5-150409-A.ogg]: clip derived from siren.mp3 (http://www.freesound.org/people/radiopassiveboy/sounds/150409/) by radiopassiveboy [CC0]\n- [5-160551-A.ogg]: clip derived from Thunder, Rain & Siren (http://www.freesound.org/people/GaryEdstrom/sounds/160551/) by GaryEdstrom [CC-BY]\n- [5-184323-A.ogg]: clip derived from alarm_siren.wav (http://www.freesound.org/people/reinsamba/sounds/184323/) by reinsamba [CC-BY]\n\n504 - Car horn:\n- [1-17124-A.ogg]: clip derived from car sharp horn.wav (http://www.freesound.org/people/cognito%20perceptu/sounds/17124/) by cognito perceptu [CC0]\n- [1-19026-A.ogg]: clip derived from claxon.wav (http://www.freesound.org/people/han1/sounds/19026/) by han1 [CC-BY]\n- [1-24074-A.ogg]: clip derived from 405_Car_Horn_Serenade.wav (http://www.freesound.org/people/cinemafia/sounds/24074/) by cinemafia [CC-Sampling+]\n- [1-24076-A.ogg]: clip derived from Hollywood_5am_Bus_and_Car_Horn.wav (http://www.freesound.org/people/cinemafia/sounds/24076/) by cinemafia [CC-Sampling+]\n- [1-254507-A.ogg]: clip derived from Henkil\\xf6auto, ohi asfaltilla, \\xe4\\xe4nimerkki, auton torvi / A car passing by fast, horn honking, doppler, Moskvitsh, a 1962 model (http://www.freesound.org/people/YleArkisto/sounds/254507/) by YleArkisto [CC-BY]\n- [1-26176-A.ogg]: clip derived from toet00.wav (http://www.freesound.org/people/WIM/sounds/26176/) by WIM [CC-BY]\n- [1-26177-A.ogg]: clip derived from toettoet02_.wav (http://www.freesound.org/people/WIM/sounds/26177/) by WIM [CC-BY]\n- [1-28808-A.ogg]: clip derived from wedding cars.wav (http://www.freesound.org/people/ERH/sounds/28808/) by ERH [CC-BY]\n- [2-100648-A.ogg]: clip derived from Shanghai Nanjing Road 20100606_02.mp3 (http://www.freesound.org/people/Unice.ys/sounds/100648/) by Unice.ys [CC0]\n- [2-125520-A.ogg]: clip derived from car-horn.wav (http://www.freesound.org/people/krissyeliot/sounds/125520/) by krissyeliot [CC0]\n- [2-128465-A.ogg]: clip derived from wedding vehicles (http://www.freesound.org/people/saphe/sounds/128465/) by saphe [CC-BY]\n- [2-128465-B.ogg]: clip derived from wedding vehicles (http://www.freesound.org/people/saphe/sounds/128465/) by saphe [CC-BY]\n- [2-138465-A.ogg]: clip derived from Cars_Horns.wav (http://www.freesound.org/people/campl/sounds/138465/) by campl [CC-BY]\n- [2-144137-A.ogg]: clip derived from 2 Car Horns (http://www.freesound.org/people/alukahn/sounds/144137/) by alukahn [CC-BY]\n- [2-42101-A.ogg]: clip derived from N street sounds 287to393.wav (http://www.freesound.org/people/ERH/sounds/42101/) by ERH [CC-BY-NC]\n- [2-54086-A.ogg]: clip derived from horn.wav (http://www.freesound.org/people/guitarguy1985/sounds/54086/) by guitarguy1985 [CC0]\n- [3-145577-A.ogg]: clip derived from 0029 Horn (http://www.freesound.org/people/bmoreno/sounds/145577/) by bmoreno [CC-BY]\n- [3-146697-A.ogg]: clip derived from 0050 Quite_street_and_elements (http://www.freesound.org/people/bmoreno/sounds/146697/) by bmoreno [CC-BY]\n- [3-153057-A.ogg]: clip derived from Car-horn.wav (http://www.freesound.org/people/Xampi8/sounds/153057/) by Xampi8 [CC-BY]\n- [3-155130-A.ogg]: clip derived from 0155 Traffic_street.wav (http://www.freesound.org/people/bmoreno/sounds/155130/) by bmoreno [CC-BY]\n- [3-155234-A.ogg]: clip derived from 0212 Street_light.wav (http://www.freesound.org/people/bmoreno/sounds/155234/) by bmoreno [CC-BY]\n- [3-161010-A.ogg]: clip derived from Rain at night in the city.wav (http://www.freesound.org/people/Elmer_Tom/sounds/161010/) by Elmer_Tom [CC-BY]\n- [3-174840-A.ogg]: clip derived from MITSUBISHI IMIEV electric car HORN LONG.wav (http://www.freesound.org/people/jakobthiesen/sounds/174840/) by jakobthiesen [CC-BY]\n- [3-243726-A.ogg]: clip derived from 140619_Bike_06_R.wav (http://www.freesound.org/people/blaukreuz/sounds/243726/) by blaukreuz [CC0]\n- [4-175845-A.ogg]: clip derived from 110703_17 car horn.WAV (http://www.freesound.org/people/toiletrolltube/sounds/175845/) by toiletrolltube [CC0]\n- [4-175846-A.ogg]: clip derived from 120213_01 car horn (3).wav (http://www.freesound.org/people/toiletrolltube/sounds/175846/) by toiletrolltube [CC0]\n- [4-175855-A.ogg]: clip derived from 120218_01 car horn (2).wav (http://www.freesound.org/people/toiletrolltube/sounds/175855/) by toiletrolltube [CC0]\n- [4-175856-A.ogg]: clip derived from 120216_02 car horn with vehicle noise (4).wav (http://www.freesound.org/people/toiletrolltube/sounds/175856/) by toiletrolltube [CC0]\n- [4-176631-A.ogg]: clip derived from Peugot 206 car interior open window accelerating aggressively racing horn under bridge (http://www.freesound.org/people/jorickhoofd/sounds/176631/) by jorickhoofd [CC-BY]\n- [4-176638-A.ogg]: clip derived from peugot 206 car exterior engine stationary horn claxon reverb (http://www.freesound.org/people/jorickhoofd/sounds/176638/) by jorickhoofd [CC-BY]\n- [4-178402-A.ogg]: clip derived from honk2.mp3 (http://www.freesound.org/people/genel/sounds/178402/) by genel [CC0]\n- [4-191015-A.ogg]: clip derived from descendo_reboucas_dia_sol_130606_01.WAV (http://www.freesound.org/people/sonoplastico/sounds/191015/) by sonoplastico [CC0]\n- [5-179860-A.ogg]: clip derived from car horn 1 - 9.4.11.wav (http://www.freesound.org/people/toiletrolltube/sounds/179860/) by toiletrolltube [CC0]\n- [5-179863-A.ogg]: clip derived from car horn 1 - 11.4.11.wav (http://www.freesound.org/people/toiletrolltube/sounds/179863/) by toiletrolltube [CC0]\n- [5-179865-A.ogg]: clip derived from car horn 2 - 26.3.11.wav (http://www.freesound.org/people/toiletrolltube/sounds/179865/) by toiletrolltube [CC0]\n- [5-179866-A.ogg]: clip derived from car horn 2 - 11.4.11.wav (http://www.freesound.org/people/toiletrolltube/sounds/179866/) by toiletrolltube [CC0]\n- [5-179868-A.ogg]: clip derived from car horn 4 - 26.3.11.wav (http://www.freesound.org/people/toiletrolltube/sounds/179868/) by toiletrolltube [CC0]\n- [5-180156-A.ogg]: clip derived from traffic horns city nervous busy.wav (http://www.freesound.org/people/klankbeeld/sounds/180156/) by klankbeeld [CC-BY]\n- [5-180156-B.ogg]: clip derived from traffic horns city nervous busy.wav (http://www.freesound.org/people/klankbeeld/sounds/180156/) by klankbeeld [CC-BY]\n- [5-180156-C.ogg]: clip derived from traffic horns city nervous busy.wav (http://www.freesound.org/people/klankbeeld/sounds/180156/) by klankbeeld [CC-BY]\n\n505 - Engine:\n- [1-18527-A.ogg]: clip derived from garbage_truck.wav (http://www.freesound.org/people/Corsica_S/sounds/18527/) by Corsica_S [CC-BY]\n- [1-18527-B.ogg]: clip derived from garbage_truck.wav (http://www.freesound.org/people/Corsica_S/sounds/18527/) by Corsica_S [CC-BY]\n- [1-22882-A.ogg]: clip derived from idle truck.flac (http://www.freesound.org/people/Corsica_S/sounds/22882/) by Corsica_S [CC-BY]\n- [1-50454-A.ogg]: clip derived from STE-030.wav (http://www.freesound.org/people/MeltyMcFace/sounds/50454/) by MeltyMcFace [CC-Sampling+]\n- [1-50455-A.ogg]: clip derived from STE-031.wav (http://www.freesound.org/people/MeltyMcFace/sounds/50455/) by MeltyMcFace [CC-Sampling+]\n- [1-50661-A.ogg]: clip derived from Bobcat idling 080320.wav (http://www.freesound.org/people/BoilingSand/sounds/50661/) by BoilingSand [CC-BY]\n- [1-61261-A.ogg]: clip derived from V8_308_ENGINE.mp3 (http://www.freesound.org/people/Hybrid34/sounds/61261/) by Hybrid34 [CC-Sampling+]\n- [1-62565-A.ogg]: clip derived from 1951 Mercury-Turnover and Cold Start.wav (http://www.freesound.org/people/lonemonk/sounds/62565/) by lonemonk [CC-BY]\n- [2-106014-A.ogg]: clip derived from samurai_engine.wav (http://www.freesound.org/people/Corsica_S/sounds/106014/) by Corsica_S [CC-BY]\n- [2-106015-A.ogg]: clip derived from samurai_tailpipe.wav (http://www.freesound.org/people/Corsica_S/sounds/106015/) by Corsica_S [CC-BY]\n- [2-106015-B.ogg]: clip derived from samurai_tailpipe.wav (http://www.freesound.org/people/Corsica_S/sounds/106015/) by Corsica_S [CC-BY]\n- [2-106486-A.ogg]: clip derived from 4banger_engine_outside_1.aiff (http://www.freesound.org/people/KlankOntwerp/sounds/106486/) by KlankOntwerp [CC-BY-NC]\n- [2-106487-A.ogg]: clip derived from 4banger_engine_outside_2.aiff (http://www.freesound.org/people/KlankOntwerp/sounds/106487/) by KlankOntwerp [CC-BY-NC]\n- [2-107228-A.ogg]: clip derived from Triumph start and idle.wav (http://www.freesound.org/people/lonemonk/sounds/107228/) by lonemonk [CC-BY]\n- [2-107228-B.ogg]: clip derived from Triumph start and idle.wav (http://www.freesound.org/people/lonemonk/sounds/107228/) by lonemonk [CC-BY]\n- [2-78651-A.ogg]: clip derived from Moped Motor.wav (http://www.freesound.org/people/JustinBW/sounds/78651/) by JustinBW [CC-BY]\n- [3-115382-A.ogg]: clip derived from Before-after Remix of 11704__alister667__vroom22.mp3 (http://www.freesound.org/people/Timbre/sounds/115382/) by Timbre [CC-BY-NC]\n- [3-119455-A.ogg]: clip derived from Motorbike Idling.wav (http://www.freesound.org/people/lmbubec/sounds/119455/) by lmbubec [CC0]\n- [3-128160-A.ogg]: clip derived from Diesel Truck Idling Front.wav (http://www.freesound.org/people/ibirdfilm/sounds/128160/) by ibirdfilm [CC0]\n- [3-141240-A.ogg]: clip derived from Renault 19 underhood acceleration (http://www.freesound.org/people/lovretta/sounds/141240/) by lovretta [CC0]\n- [3-141240-B.ogg]: clip derived from Renault 19 underhood acceleration (http://www.freesound.org/people/lovretta/sounds/141240/) by lovretta [CC0]\n- [3-146186-A.ogg]: clip derived from DeLorean Engine Idle - 1 (http://www.freesound.org/people/HazMattt/sounds/146186/) by HazMattt [CC-BY-NC]\n- [3-154758-A.ogg]: clip derived from Car Engine Idling.wav (http://www.freesound.org/people/jpkweli/sounds/154758/) by jpkweli [CC0]\n- [3-259622-A.ogg]: clip derived from Diesel Engine Rattle.wav (http://www.freesound.org/people/Stevious42/sounds/259622/) by Stevious42 [CC-BY]\n- [4-171652-A.ogg]: clip derived from Chainsaw Idle loops (http://www.freesound.org/people/Audionautics/sounds/171652/) by Audionautics [CC-BY]\n- [4-186935-A.ogg]: clip derived from engine_idle_loop.wav (http://www.freesound.org/people/ReadeOnly/sounds/186935/) by ReadeOnly [CC0]\n- [4-186936-A.ogg]: clip derived from engine2_idle_loop2.wav (http://www.freesound.org/people/ReadeOnly/sounds/186936/) by ReadeOnly [CC0]\n- [4-186938-A.ogg]: clip derived from engine2_down.wav (http://www.freesound.org/people/ReadeOnly/sounds/186938/) by ReadeOnly [CC0]\n- [4-186940-A.ogg]: clip derived from engine3_idle_loop.wav (http://www.freesound.org/people/ReadeOnly/sounds/186940/) by ReadeOnly [CC0]\n- [4-186962-A.ogg]: clip derived from engine2_up.wav (http://www.freesound.org/people/ReadeOnly/sounds/186962/) by ReadeOnly [CC0]\n- [4-195451-A.ogg]: clip derived from Diesel_start_idle2.wav (http://www.freesound.org/people/phenoxy/sounds/195451/) by phenoxy [CC-BY]\n- [4-201988-A.ogg]: clip derived from 1953 Studebaker-Starts, Rev and Idle (http://www.freesound.org/people/lonemonk/sounds/201988/) by lonemonk [CC-BY]\n- [5-209992-A.ogg]: clip derived from big crane idle  (http://www.freesound.org/people/Yuval/sounds/209992/) by Yuval [CC-BY]\n- [5-209992-B.ogg]: clip derived from big crane idle  (http://www.freesound.org/people/Yuval/sounds/209992/) by Yuval [CC-BY]\n- [5-232272-A.ogg]: clip derived from Idle and Rev - Engine.wav (http://www.freesound.org/people/wikusv/sounds/232272/) by wikusv [CC-BY]\n- [5-235507-A.ogg]: clip derived from Car Engine.wav (http://www.freesound.org/people/ceberation/sounds/235507/) by ceberation [CC-BY]\n- [5-240671-A.ogg]: clip derived from Chrysler Start and Idle (Exhaust perspective) (http://www.freesound.org/people/lonemonk/sounds/240671/) by lonemonk [CC-BY]\n- [5-243773-A.ogg]: clip derived from Diesel-drive2.wav (http://www.freesound.org/people/fastson/sounds/243773/) by fastson [CC0]\n- [5-243773-B.ogg]: clip derived from Diesel-drive2.wav (http://www.freesound.org/people/fastson/sounds/243773/) by fastson [CC0]\n- [5-243783-A.ogg]: clip derived from Diesel-drive.wav (http://www.freesound.org/people/fastson/sounds/243783/) by fastson [CC0]\n\n506 - Train:\n- [1-119125-A.ogg]: clip derived from Steam Train L To R.WAV (http://www.freesound.org/people/Benboncan/sounds/119125/) by Benboncan [CC-BY]\n- [1-54065-A.ogg]: clip derived from trains_on_bridge.wav (http://www.freesound.org/people/reinsamba/sounds/54065/) by reinsamba [CC-BY]\n- [1-54065-B.ogg]: clip derived from trains_on_bridge.wav (http://www.freesound.org/people/reinsamba/sounds/54065/) by reinsamba [CC-BY]\n- [1-62509-A.ogg]: clip derived from CN Freight Train Pass.wav (http://www.freesound.org/people/lonemonk/sounds/62509/) by lonemonk [CC-BY]\n- [1-64473-A.ogg]: clip derived from 20081210FreightTrain2.wav (http://www.freesound.org/people/daveincamas/sounds/64473/) by daveincamas [CC-BY]\n- [1-70300-A.ogg]: clip derived from Diesel Electric.wav (http://www.freesound.org/people/Benboncan/sounds/70300/) by Benboncan [CC-BY]\n- [1-88409-A.ogg]: clip derived from chichibu.wav (http://www.freesound.org/people/Heigh-hoo/sounds/88409/) by Heigh-hoo [CC0]\n- [1-88409-B.ogg]: clip derived from chichibu.wav (http://www.freesound.org/people/Heigh-hoo/sounds/88409/) by Heigh-hoo [CC0]\n- [2-122066-A.ogg]: clip derived from 06-2011 Freight Train Pass #1.wav (http://www.freesound.org/people/lonemonk/sounds/122066/) by lonemonk [CC-BY]\n- [2-122066-B.ogg]: clip derived from 06-2011 Freight Train Pass #1.wav (http://www.freesound.org/people/lonemonk/sounds/122066/) by lonemonk [CC-BY]\n- [2-122067-A.ogg]: clip derived from 06-2011 Freight Train Pass #2.wav (http://www.freesound.org/people/lonemonk/sounds/122067/) by lonemonk [CC-BY]\n- [2-122067-B.ogg]: clip derived from 06-2011 Freight Train Pass #2.wav (http://www.freesound.org/people/lonemonk/sounds/122067/) by lonemonk [CC-BY]\n- [2-135649-A.ogg]: clip derived from Passing trains in Vienna.wav (http://www.freesound.org/people/jgrzinich/sounds/135649/) by jgrzinich [CC-BY-NC]\n- [2-135649-B.ogg]: clip derived from Passing trains in Vienna.wav (http://www.freesound.org/people/jgrzinich/sounds/135649/) by jgrzinich [CC-BY-NC]\n- [2-135649-C.ogg]: clip derived from Passing trains in Vienna.wav (http://www.freesound.org/people/jgrzinich/sounds/135649/) by jgrzinich [CC-BY-NC]\n- [2-262579-A.ogg]: clip derived from Subway1.wav (http://www.freesound.org/people/Trautwein/sounds/262579/) by Trautwein [CC-BY]\n- [3-135650-A.ogg]: clip derived from Train recording.wav (http://www.freesound.org/people/jgrzinich/sounds/135650/) by jgrzinich [CC-BY-NC]\n- [3-135650-B.ogg]: clip derived from Train recording.wav (http://www.freesound.org/people/jgrzinich/sounds/135650/) by jgrzinich [CC-BY-NC]\n- [3-136451-A.ogg]: clip derived from Train passing by in the desert. Close recording. (California, USA) (http://www.freesound.org/people/felix.blume/sounds/136451/) by felix.blume [CC0]\n- [3-138212-A.ogg]: clip derived from train_passing_by_fast.wav (http://www.freesound.org/people/aesqe/sounds/138212/) by aesqe [CC-BY]\n- [3-141559-A.ogg]: clip derived from train-2.wav (http://www.freesound.org/people/xserra/sounds/141559/) by xserra [CC-BY]\n- [3-157492-A.ogg]: clip derived from Inside Train Wagon (http://www.freesound.org/people/caquet/sounds/157492/) by caquet [CC-BY]\n- [3-159445-A.ogg]: clip derived from Brussels Train Station (Gare de Schaerbeek). Train are passing by without stopping. (http://www.freesound.org/people/felix.blume/sounds/159445/) by felix.blume [CC0]\n- [3-159445-B.ogg]: clip derived from Brussels Train Station (Gare de Schaerbeek). Train are passing by without stopping. (http://www.freesound.org/people/felix.blume/sounds/159445/) by felix.blume [CC0]\n- [4-165606-A.ogg]: clip derived from Inside a train in Buenos Aires (regional train inside the city) (http://www.freesound.org/people/felix.blume/sounds/165606/) by felix.blume [CC0]\n- [4-165791-A.ogg]: clip derived from Between two wagons on a train in Argentina (http://www.freesound.org/people/felix.blume/sounds/165791/) by felix.blume [CC0]\n- [4-165791-B.ogg]: clip derived from Between two wagons on a train in Argentina (http://www.freesound.org/people/felix.blume/sounds/165791/) by felix.blume [CC0]\n- [4-165845-A.ogg]: clip derived from On a train in Argentina, between two wagons. Metal pieces are squeaking (http://www.freesound.org/people/felix.blume/sounds/165845/) by felix.blume [CC0]\n- [4-165845-B.ogg]: clip derived from On a train in Argentina, between two wagons. Metal pieces are squeaking (http://www.freesound.org/people/felix.blume/sounds/165845/) by felix.blume [CC0]\n- [4-165933-A.ogg]: clip derived from On a train in Argentina, between two wagons. Door is knocking (http://www.freesound.org/people/felix.blume/sounds/165933/) by felix.blume [CC0]\n- [4-168446-A.ogg]: clip derived from On a train, between the wagon with the door open (Hungary) (http://www.freesound.org/people/felix.blume/sounds/168446/) by felix.blume [CC0]\n- [4-178881-A.ogg]: clip derived from yellow-train-1.wav (http://www.freesound.org/people/xserra/sounds/178881/) by xserra [CC-BY]\n- [5-188796-A.ogg]: clip derived from On a train in Ukraine, going to Chernobyl (http://www.freesound.org/people/felix.blume/sounds/188796/) by felix.blume [CC0]\n- [5-188945-A.ogg]: clip derived from On a train, through the window, going from Slavutich to Chernobyl (http://www.freesound.org/people/felix.blume/sounds/188945/) by felix.blume [CC0]\n- [5-197118-A.ogg]: clip derived from Train coming to a stop (recorded between two cars) (http://www.freesound.org/people/avakas/sounds/197118/) by avakas [CC0]\n- [5-197121-A.ogg]: clip derived from Train moving slowly (recorded between two cars) (http://www.freesound.org/people/avakas/sounds/197121/) by avakas [CC0]\n- [5-197121-B.ogg]: clip derived from Train moving slowly (recorded between two cars) (http://www.freesound.org/people/avakas/sounds/197121/) by avakas [CC0]\n- [5-198600-A.ogg]: clip derived from Train departs from the station (http://www.freesound.org/people/toam/sounds/198600/) by toam [CC-BY]\n- [5-199284-A.ogg]: clip derived from train leaving Sevastopol Ukraine (http://www.freesound.org/people/jgrzinich/sounds/199284/) by jgrzinich [CC-BY-NC]\n- [5-199284-B.ogg]: clip derived from train leaving Sevastopol Ukraine (http://www.freesound.org/people/jgrzinich/sounds/199284/) by jgrzinich [CC-BY-NC]\n\n507 - Church bells:\n- [1-13571-A.ogg]: clip derived from DommeBells.wav (http://www.freesound.org/people/acclivity/sounds/13571/) by acclivity [CC-BY-NC]\n- [1-13572-A.ogg]: clip derived from RocamadorChurch.wav (http://www.freesound.org/people/acclivity/sounds/13572/) by acclivity [CC-BY-NC]\n- [1-21421-A.ogg]: clip derived from 20060730.bells.wav (http://www.freesound.org/people/dobroide/sounds/21421/) by dobroide [CC-BY]\n- [1-22804-A.ogg]: clip derived from CathedralOfTheDowns.mp3 (http://www.freesound.org/people/acclivity/sounds/22804/) by acclivity [CC-BY-NC]\n- [1-40154-A.ogg]: clip derived from Church bells.wav (http://www.freesound.org/people/genghis%20attenborough/sounds/40154/) by genghis attenborough [CC-BY]\n- [1-48298-A.ogg]: clip derived from Bells inside Church.mp3 (http://www.freesound.org/people/melarancida/sounds/48298/) by melarancida [CC0]\n- [1-54747-A.ogg]: clip derived from church bells stolnica.wav (http://www.freesound.org/people/Satoration/sounds/54747/) by Satoration [CC-BY]\n- [1-56907-A.ogg]: clip derived from DentStAndrews ChurchBells Binaural.wav (http://www.freesound.org/people/waterpump/sounds/56907/) by waterpump [CC0]\n- [2-56926-A.ogg]: clip derived from sorana bells_sette_casa.wav (http://www.freesound.org/people/dADDoiT/sounds/56926/) by dADDoiT [CC0]\n- [2-57191-A.ogg]: clip derived from sorana bells_1915satmass.wav (http://www.freesound.org/people/dADDoiT/sounds/57191/) by dADDoiT [CC0]\n- [2-59565-A.ogg]: clip derived from church_bell.mp3 (http://www.freesound.org/people/3bagbrew/sounds/59565/) by 3bagbrew [CC-BY]\n- [2-59566-A.ogg]: clip derived from church_bells2.mp3 (http://www.freesound.org/people/3bagbrew/sounds/59566/) by 3bagbrew [CC-BY]\n- [2-61618-A.ogg]: clip derived from Riga Latvia. Dome church bells.MP3 (http://www.freesound.org/people/ariepaul/sounds/61618/) by ariepaul [CC-BY]\n- [2-77346-A.ogg]: clip derived from campanes-cardedeu-1.wav (http://www.freesound.org/people/xserra/sounds/77346/) by xserra [CC-BY]\n- [2-77347-A.ogg]: clip derived from campanes-cardedeu-2.wav (http://www.freesound.org/people/xserra/sounds/77347/) by xserra [CC-BY]\n- [2-78381-A.ogg]: clip derived from Angelus Bell.aif (http://www.freesound.org/people/Peaveycroquette/sounds/78381/) by Peaveycroquette [CC-BY]\n- [3-111102-A.ogg]: clip derived from 101225_pealing.bells.wav (http://www.freesound.org/people/dobroide/sounds/111102/) by dobroide [CC-BY]\n- [3-111102-B.ogg]: clip derived from 101225_pealing.bells.wav (http://www.freesound.org/people/dobroide/sounds/111102/) by dobroide [CC-BY]\n- [3-122110-A.ogg]: clip derived from bells.wav (http://www.freesound.org/people/ondrosik/sounds/122110/) by ondrosik [CC-BY]\n- [3-139109-A.ogg]: clip derived from All bells ringing at Notre-Dame in Paris (http://www.freesound.org/people/Mxsmanic/sounds/139109/) by Mxsmanic [CC0]\n- [3-87936-A.ogg]: clip derived from Church bells.wav (http://www.freesound.org/people/Ohrwurm/sounds/87936/) by Ohrwurm [CC0]\n- [3-87936-B.ogg]: clip derived from Church bells.wav (http://www.freesound.org/people/Ohrwurm/sounds/87936/) by Ohrwurm [CC0]\n- [3-98193-A.ogg]: clip derived from 01824 church bells.wav (http://www.freesound.org/people/Robinhood76/sounds/98193/) by Robinhood76 [CC-BY-NC]\n- [3-98869-A.ogg]: clip derived from St Dunstans church bells practice.wav (http://www.freesound.org/people/LondonSoundSurvey/sounds/98869/) by LondonSoundSurvey [CC-BY-NC]\n- [4-125825-A.ogg]: clip derived from Bells of St Alkelda\'s Church, Giggleswick, England.flac (http://www.freesound.org/people/thaighaudio/sounds/125825/) by thaighaudio [CC0]\n- [4-125825-B.ogg]: clip derived from Bells of St Alkelda\'s Church, Giggleswick, England.flac (http://www.freesound.org/people/thaighaudio/sounds/125825/) by thaighaudio [CC0]\n- [4-140034-A.ogg]: clip derived from 111227_church_bells.wav (http://www.freesound.org/people/reinsamba/sounds/140034/) by reinsamba [CC-BY]\n- [4-147657-A.ogg]: clip derived from Angelus Mortagne lointain.WAV (http://www.freesound.org/people/jsahab/sounds/147657/) by jsahab [CC-BY-NC]\n- [4-147658-A.ogg]: clip derived from Angelus Mortagne devant (HP).aif (http://www.freesound.org/people/jsahab/sounds/147658/) by jsahab [CC-BY-NC]\n- [4-150364-A.ogg]: clip derived from Church Bells.wav (http://www.freesound.org/people/sarson/sounds/150364/) by sarson [CC0]\n- [4-150364-B.ogg]: clip derived from Church Bells.wav (http://www.freesound.org/people/sarson/sounds/150364/) by sarson [CC0]\n- [4-156827-A.ogg]: clip derived from BRNO Petrov bells.mp3 (http://www.freesound.org/people/JAN69/sounds/156827/) by JAN69 [CC0]\n- [5-179294-A.ogg]: clip derived from Church Bells.wav (http://www.freesound.org/people/jamesrodavidson/sounds/179294/) by jamesrodavidson [CC0]\n- [5-188716-A.ogg]: clip derived from Church Bells of Saint Jacob.wav (http://www.freesound.org/people/TrygveN/sounds/188716/) by TrygveN [CC0]\n- [5-197988-A.ogg]: clip derived from church bells 12pm (http://www.freesound.org/people/Joao_Santos/sounds/197988/) by Joao_Santos [CC-BY-NC]\n- [5-198373-A.ogg]: clip derived from church\'s bells from tower\'s church (http://www.freesound.org/people/organicmanpl/sounds/198373/) by organicmanpl [CC0]\n- [5-201170-A.ogg]: clip derived from Port Sunlight wedding bells.wav (http://www.freesound.org/people/phonoflora/sounds/201170/) by phonoflora [CC-BY]\n- [5-201172-A.ogg]: clip derived from Parish church belfry (2).wav (http://www.freesound.org/people/phonoflora/sounds/201172/) by phonoflora [CC-BY]\n- [5-204741-A.ogg]: clip derived from church bell engelen 110417_04 8am.wav (http://www.freesound.org/people/klankbeeld/sounds/204741/) by klankbeeld [CC-BY]\n- [5-219044-A.ogg]: clip derived from Church Clock Strikes 12.wav (http://www.freesound.org/people/JarredGibb/sounds/219044/) by JarredGibb [CC0]\n\n508 - Airplane:\n- [1-11687-A.ogg]: clip derived from 06-airplane-landing-overhead-zaventem.wav (http://www.freesound.org/people/Bram/sounds/11687/) by Bram [CC-BY]\n- [1-24796-A.ogg]: clip derived from Jet over Rosemont Horizon Parking Lot.wav (http://www.freesound.org/people/Terwelp/sounds/24796/) by Terwelp [CC-BY]\n- [1-36929-A.ogg]: clip derived from plane.wav (http://www.freesound.org/people/UncleSigmund/sounds/36929/) by UncleSigmund [CC0]\n- [1-43807-A.ogg]: clip derived from WW2AirplaneFlyover.wav (http://www.freesound.org/people/daveincamas/sounds/43807/) by daveincamas [CC-BY]\n- [1-43807-B.ogg]: clip derived from WW2AirplaneFlyover.wav (http://www.freesound.org/people/daveincamas/sounds/43807/) by daveincamas [CC-BY]\n- [1-43807-C.ogg]: clip derived from WW2AirplaneFlyover.wav (http://www.freesound.org/people/daveincamas/sounds/43807/) by daveincamas [CC-BY]\n- [1-43807-D.ogg]: clip derived from WW2AirplaneFlyover.wav (http://www.freesound.org/people/daveincamas/sounds/43807/) by daveincamas [CC-BY]\n- [1-53467-A.ogg]: clip derived from airplane.flac (http://www.freesound.org/people/lorenzosu/sounds/53467/) by lorenzosu [CC-BY-NC]\n- [2-105270-A.ogg]: clip derived from single engine plane.WAV (http://www.freesound.org/people/inchadney/sounds/105270/) by inchadney [CC-BY]\n- [2-106849-A.ogg]: clip derived from 20101012.passing.plane.pealing.bell.wav (http://www.freesound.org/people/dobroide/sounds/106849/) by dobroide [CC-BY]\n- [2-160888-A.ogg]: clip derived from air-raid.wav (http://www.freesound.org/people/blaukreuz/sounds/160888/) by blaukreuz [CC0]\n- [2-74361-A.ogg]: clip derived from Airplane Pass.wav (http://www.freesound.org/people/zolopher/sounds/74361/) by zolopher [CC0]\n- [2-78781-A.ogg]: clip derived from Plane pass w_Thunder w_cicadas.wav (http://www.freesound.org/people/patchen/sounds/78781/) by patchen [CC-BY]\n- [2-78799-A.ogg]: clip derived from small plane w_return passby and ambience.wav (http://www.freesound.org/people/patchen/sounds/78799/) by patchen [CC-BY]\n- [2-96654-A.ogg]: clip derived from airplane stereo 24:96 r to l.wav (http://www.freesound.org/people/gvproject/sounds/96654/) by gvproject [CC-Sampling+]\n- [2-98866-A.ogg]: clip derived from fatalbertpassseafair09.mp3 (http://www.freesound.org/people/simav8r/sounds/98866/) by simav8r [CC-Sampling+]\n- [3-115387-A.ogg]: clip derived from OverpassingPlane.wav (http://www.freesound.org/people/adamlhumphreys/sounds/115387/) by adamlhumphreys [CC-BY-NC]\n- [3-115387-B.ogg]: clip derived from OverpassingPlane.wav (http://www.freesound.org/people/adamlhumphreys/sounds/115387/) by adamlhumphreys [CC-BY-NC]\n- [3-115387-C.ogg]: clip derived from OverpassingPlane.wav (http://www.freesound.org/people/adamlhumphreys/sounds/115387/) by adamlhumphreys [CC-BY-NC]\n- [3-128512-A.ogg]: clip derived from HotSummerDay-ColumbiaRiverGorge.mp3 (http://www.freesound.org/people/daveincamas/sounds/128512/) by daveincamas [CC-BY]\n- [3-128512-B.ogg]: clip derived from HotSummerDay-ColumbiaRiverGorge.mp3 (http://www.freesound.org/people/daveincamas/sounds/128512/) by daveincamas [CC-BY]\n- [3-143560-A.ogg]: clip derived from P-47 Flyby (http://www.freesound.org/people/Fight2FlyPhoto/sounds/143560/) by Fight2FlyPhoto [CC-BY]\n- [3-143562-A.ogg]: clip derived from Fw-190 Flyby (http://www.freesound.org/people/Fight2FlyPhoto/sounds/143562/) by Fight2FlyPhoto [CC-BY]\n- [3-193767-A.ogg]: clip derived from VHCL_S_Airplane_Prop_001.wav (http://www.freesound.org/people/conleec/sounds/193767/) by conleec [CC0]\n- [4-161099-A.ogg]: clip derived from F7F Tigercat - High Speed (http://www.freesound.org/people/Fight2FlyPhoto/sounds/161099/) by Fight2FlyPhoto [CC-BY-NC]\n- [4-161099-B.ogg]: clip derived from F7F Tigercat - High Speed (http://www.freesound.org/people/Fight2FlyPhoto/sounds/161099/) by Fight2FlyPhoto [CC-BY-NC]\n- [4-161100-A.ogg]: clip derived from F7F Tigercat & F8F Bearcat Opposing Pass (http://www.freesound.org/people/Fight2FlyPhoto/sounds/161100/) by Fight2FlyPhoto [CC-BY-NC]\n- [4-161103-A.ogg]: clip derived from Hawker Hurricane Flyby (http://www.freesound.org/people/Fight2FlyPhoto/sounds/161103/) by Fight2FlyPhoto [CC-BY-NC]\n- [4-161105-A.ogg]: clip derived from F8F Bearcat Landing Gear Pass (http://www.freesound.org/people/Fight2FlyPhoto/sounds/161105/) by Fight2FlyPhoto [CC-BY-NC]\n- [4-161105-B.ogg]: clip derived from F8F Bearcat Landing Gear Pass (http://www.freesound.org/people/Fight2FlyPhoto/sounds/161105/) by Fight2FlyPhoto [CC-BY-NC]\n- [4-251959-A.ogg]: clip derived from Jet Plane 1 (http://www.freesound.org/people/cliftonmcarlson/sounds/251959/) by cliftonmcarlson [CC-BY]\n- [4-255371-A.ogg]: clip derived from 01 Pistons and Props - Sywell 2014 (http://www.freesound.org/people/konakaboom/sounds/255371/) by konakaboom [CC-BY-NC]\n- [5-215445-A.ogg]: clip derived from ps_flighby_02.wav (http://www.freesound.org/people/laribum/sounds/215445/) by laribum [CC0]\n- [5-215447-A.ogg]: clip derived from ps_fast_flighby_01.wav (http://www.freesound.org/people/laribum/sounds/215447/) by laribum [CC0]\n- [5-215449-A.ogg]: clip derived from ps_flighby_highgain_01.wav (http://www.freesound.org/people/laribum/sounds/215449/) by laribum [CC0]\n- [5-235956-A.ogg]: clip derived from Airliner Flyby.wav (http://www.freesound.org/people/mffm/sounds/235956/) by mffm [CC0]\n- [5-251957-A.ogg]: clip derived from Jet Plane 3 (http://www.freesound.org/people/cliftonmcarlson/sounds/251957/) by cliftonmcarlson [CC-BY]\n- [5-251962-A.ogg]: clip derived from Jet Plane 6 (cleanest recording) (http://www.freesound.org/people/cliftonmcarlson/sounds/251962/) by cliftonmcarlson [CC-BY]\n- [5-251963-A.ogg]: clip derived from Jet Plane 5 (http://www.freesound.org/people/cliftonmcarlson/sounds/251963/) by cliftonmcarlson [CC-BY]\n- [5-251971-A.ogg]: clip derived from Propeller Plane (http://www.freesound.org/people/cliftonmcarlson/sounds/251971/) by cliftonmcarlson [CC-BY]\n\n509 - Fireworks:\n- [1-115545-A.ogg]: clip derived from fireworks_2009_close.wav (http://www.freesound.org/people/matucha/sounds/115545/) by matucha [CC-BY]\n- [1-115545-B.ogg]: clip derived from fireworks_2009_close.wav (http://www.freesound.org/people/matucha/sounds/115545/) by matucha [CC-BY]\n- [1-115545-C.ogg]: clip derived from fireworks_2009_close.wav (http://www.freesound.org/people/matucha/sounds/115545/) by matucha [CC-BY]\n- [1-115546-A.ogg]: clip derived from fireworks_2009_distant.wav (http://www.freesound.org/people/matucha/sounds/115546/) by matucha [CC-BY]\n- [1-160563-A.ogg]: clip derived from Fireworks July 3, 2012 (http://www.freesound.org/people/daveincamas/sounds/160563/) by daveincamas [CC0]\n- [1-160563-B.ogg]: clip derived from Fireworks July 3, 2012 (http://www.freesound.org/people/daveincamas/sounds/160563/) by daveincamas [CC0]\n- [1-25777-A.ogg]: clip derived from multistage_whistling_fireworks_mkh3040_061105.wav (http://www.freesound.org/people/ermine/sounds/25777/) by ermine [CC-BY-NC]\n- [1-25781-A.ogg]: clip derived from whistling_fireworks_mkh3040_061105.wav (http://www.freesound.org/people/ermine/sounds/25781/) by ermine [CC-BY-NC]\n- [2-117615-A.ogg]: clip derived from Fireworks - explode and fizz.wav (http://www.freesound.org/people/soundmary/sounds/117615/) by soundmary [CC0]\n- [2-117615-B.ogg]: clip derived from Fireworks - explode and fizz.wav (http://www.freesound.org/people/soundmary/sounds/117615/) by soundmary [CC0]\n- [2-117615-C.ogg]: clip derived from Fireworks - explode and fizz.wav (http://www.freesound.org/people/soundmary/sounds/117615/) by soundmary [CC0]\n- [2-117615-D.ogg]: clip derived from Fireworks - explode and fizz.wav (http://www.freesound.org/people/soundmary/sounds/117615/) by soundmary [CC0]\n- [2-117615-E.ogg]: clip derived from Fireworks - explode and fizz.wav (http://www.freesound.org/people/soundmary/sounds/117615/) by soundmary [CC0]\n- [2-117616-A.ogg]: clip derived from fireworks exploding 1.wav (http://www.freesound.org/people/soundmary/sounds/117616/) by soundmary [CC0]\n- [2-117617-A.ogg]: clip derived from fireworks.wav (http://www.freesound.org/people/soundmary/sounds/117617/) by soundmary [CC0]\n- [2-165801-A.ogg]: clip derived from fireworks_bin.wav (http://www.freesound.org/people/ramagochi/sounds/165801/) by ramagochi [CC-BY-NC]\n- [3-119120-A.ogg]: clip derived from fireworks.wav (http://www.freesound.org/people/ramagochi/sounds/119120/) by ramagochi [CC-BY]\n- [3-119120-B.ogg]: clip derived from fireworks.wav (http://www.freesound.org/people/ramagochi/sounds/119120/) by ramagochi [CC-BY]\n- [3-119120-C.ogg]: clip derived from fireworks.wav (http://www.freesound.org/people/ramagochi/sounds/119120/) by ramagochi [CC-BY]\n- [3-119120-D.ogg]: clip derived from fireworks.wav (http://www.freesound.org/people/ramagochi/sounds/119120/) by ramagochi [CC-BY]\n- [3-119120-E.ogg]: clip derived from fireworks.wav (http://www.freesound.org/people/ramagochi/sounds/119120/) by ramagochi [CC-BY]\n- [3-172881-A.ogg]: clip derived from Fireworks 2012 Holland 130101_01.flac (http://www.freesound.org/people/klankbeeld/sounds/172881/) by klankbeeld [CC-BY]\n- [3-172922-A.ogg]: clip derived from fireworks_city-periphery_2013.wav (http://www.freesound.org/people/matucha/sounds/172922/) by matucha [CC-BY]\n- [3-251617-A.ogg]: clip derived from Fireworks#1 (http://www.freesound.org/people/bmlake/sounds/251617/) by bmlake [CC-BY]\n- [4-119647-A.ogg]: clip derived from Fireworks 2 16.wav (http://www.freesound.org/people/ramagochi/sounds/119647/) by ramagochi [CC-BY]\n- [4-119647-B.ogg]: clip derived from Fireworks 2 16.wav (http://www.freesound.org/people/ramagochi/sounds/119647/) by ramagochi [CC-BY]\n- [4-119647-C.ogg]: clip derived from Fireworks 2 16.wav (http://www.freesound.org/people/ramagochi/sounds/119647/) by ramagochi [CC-BY]\n- [4-119647-D.ogg]: clip derived from Fireworks 2 16.wav (http://www.freesound.org/people/ramagochi/sounds/119647/) by ramagochi [CC-BY]\n- [4-119648-A.ogg]: clip derived from Fireworks 3 16.wav (http://www.freesound.org/people/ramagochi/sounds/119648/) by ramagochi [CC-BY]\n- [4-119648-B.ogg]: clip derived from Fireworks 3 16.wav (http://www.freesound.org/people/ramagochi/sounds/119648/) by ramagochi [CC-BY]\n- [4-119648-C.ogg]: clip derived from Fireworks 3 16.wav (http://www.freesound.org/people/ramagochi/sounds/119648/) by ramagochi [CC-BY]\n- [4-119648-D.ogg]: clip derived from Fireworks 3 16.wav (http://www.freesound.org/people/ramagochi/sounds/119648/) by ramagochi [CC-BY]\n- [5-160614-A.ogg]: clip derived from Fireworks July 4, 2012 (http://www.freesound.org/people/daveincamas/sounds/160614/) by daveincamas [CC0]\n- [5-160614-B.ogg]: clip derived from Fireworks July 4, 2012 (http://www.freesound.org/people/daveincamas/sounds/160614/) by daveincamas [CC0]\n- [5-160614-C.ogg]: clip derived from Fireworks July 4, 2012 (http://www.freesound.org/people/daveincamas/sounds/160614/) by daveincamas [CC0]\n- [5-160614-D.ogg]: clip derived from Fireworks July 4, 2012 (http://www.freesound.org/people/daveincamas/sounds/160614/) by daveincamas [CC0]\n- [5-160614-E.ogg]: clip derived from Fireworks July 4, 2012 (http://www.freesound.org/people/daveincamas/sounds/160614/) by daveincamas [CC0]\n- [5-160614-F.ogg]: clip derived from Fireworks July 4, 2012 (http://www.freesound.org/people/daveincamas/sounds/160614/) by daveincamas [CC0]\n- [5-160614-G.ogg]: clip derived from Fireworks July 4, 2012 (http://www.freesound.org/people/daveincamas/sounds/160614/) by daveincamas [CC0]\n- [5-160614-H.ogg]: clip derived from Fireworks July 4, 2012 (http://www.freesound.org/people/daveincamas/sounds/160614/) by daveincamas [CC0]\n\n510 - Hand saw:\n- [1-18810-A.ogg]: clip derived from Sawing_02.wav (http://www.freesound.org/people/Koops/sounds/18810/) by Koops [CC-BY]\n- [1-23706-A.ogg]: clip derived from saw.wav (http://www.freesound.org/people/hazure/sounds/23706/) by hazure [CC-BY]\n- [1-46353-A.ogg]: clip derived from little saw petite scie.wav (http://www.freesound.org/people/arnaud%20coutancier/sounds/46353/) by arnaud coutancier [CC-BY-NC]\n- [1-7974-A.ogg]: clip derived from handsaw.wav (http://www.freesound.org/people/dobroide/sounds/7974/) by dobroide [CC-BY]\n- [1-7974-B.ogg]: clip derived from handsaw.wav (http://www.freesound.org/people/dobroide/sounds/7974/) by dobroide [CC-BY]\n- [1-9886-A.ogg]: clip derived from WoodSaw1.wav (http://www.freesound.org/people/Pingel/sounds/9886/) by Pingel [CC0]\n- [1-9887-A.ogg]: clip derived from WoodSaw2.wav (http://www.freesound.org/people/Pingel/sounds/9887/) by Pingel [CC0]\n- [1-9887-B.ogg]: clip derived from WoodSaw2.wav (http://www.freesound.org/people/Pingel/sounds/9887/) by Pingel [CC0]\n- [2-135860-A.ogg]: clip derived from WOOD_SAW_LONG_001.wav (http://www.freesound.org/people/JoelAudio/sounds/135860/) by JoelAudio [CC-BY]\n- [2-184077-A.ogg]: clip derived from Scie.aif (http://www.freesound.org/people/jcarrascovazquez/sounds/184077/) by jcarrascovazquez [CC0]\n- [2-51630-A.ogg]: clip derived from 00135 pilowanie 2.wav (http://www.freesound.org/people/Robinhood76/sounds/51630/) by Robinhood76 [CC-BY-NC]\n- [2-51630-B.ogg]: clip derived from 00135 pilowanie 2.wav (http://www.freesound.org/people/Robinhood76/sounds/51630/) by Robinhood76 [CC-BY-NC]\n- [2-59321-A.ogg]: clip derived from saw_lumber.wav (http://www.freesound.org/people/vibe_crc/sounds/59321/) by vibe_crc [CC0]\n- [2-60180-A.ogg]: clip derived from Coping saw.wav (http://www.freesound.org/people/Benboncan/sounds/60180/) by Benboncan [CC-BY]\n- [2-60180-B.ogg]: clip derived from Coping saw.wav (http://www.freesound.org/people/Benboncan/sounds/60180/) by Benboncan [CC-BY]\n- [2-84693-A.ogg]: clip derived from Wood, Sawed_Dropped.wav (http://www.freesound.org/people/cmusounddesign/sounds/84693/) by cmusounddesign [CC-BY]\n- [3-102583-A.ogg]: clip derived from Sawing 2x4.wav (http://www.freesound.org/people/CGEffex/sounds/102583/) by CGEffex [CC-BY]\n- [3-102583-B.ogg]: clip derived from Sawing 2x4.wav (http://www.freesound.org/people/CGEffex/sounds/102583/) by CGEffex [CC-BY]\n- [3-102583-C.ogg]: clip derived from Sawing 2x4.wav (http://www.freesound.org/people/CGEffex/sounds/102583/) by CGEffex [CC-BY]\n- [3-208820-A.ogg]: clip derived from Music wood.wav (http://www.freesound.org/people/vumseplutten1709/sounds/208820/) by vumseplutten1709 [CC0]\n- [3-96606-A.ogg]: clip derived from Wood saw.wav (http://www.freesound.org/people/Ohrwurm/sounds/96606/) by Ohrwurm [CC0]\n- [3-96606-B.ogg]: clip derived from Wood saw.wav (http://www.freesound.org/people/Ohrwurm/sounds/96606/) by Ohrwurm [CC0]\n- [3-97909-A.ogg]: clip derived from scie.aif (http://www.freesound.org/people/julietteg/sounds/97909/) by julietteg [CC-BY-NC]\n- [3-97909-B.ogg]: clip derived from scie.aif (http://www.freesound.org/people/julietteg/sounds/97909/) by julietteg [CC-BY-NC]\n- [4-102844-A.ogg]: clip derived from 20100802.sawing.wav (http://www.freesound.org/people/dobroide/sounds/102844/) by dobroide [CC-BY]\n- [4-102844-B.ogg]: clip derived from 20100802.sawing.wav (http://www.freesound.org/people/dobroide/sounds/102844/) by dobroide [CC-BY]\n- [4-102844-C.ogg]: clip derived from 20100802.sawing.wav (http://www.freesound.org/people/dobroide/sounds/102844/) by dobroide [CC-BY]\n- [4-198360-A.ogg]: clip derived from Hand sawing.WAV (http://www.freesound.org/people/paulw2k/sounds/198360/) by paulw2k [CC-BY]\n- [4-198360-B.ogg]: clip derived from Hand sawing.WAV (http://www.freesound.org/people/paulw2k/sounds/198360/) by paulw2k [CC-BY]\n- [4-198360-C.ogg]: clip derived from Hand sawing.WAV (http://www.freesound.org/people/paulw2k/sounds/198360/) by paulw2k [CC-BY]\n- [4-251645-A.ogg]: clip derived from Woodshop sawing planing.wav (http://www.freesound.org/people/BitsyMcNics/sounds/251645/) by BitsyMcNics [CC0]\n- [4-251645-B.ogg]: clip derived from Woodshop sawing planing.wav (http://www.freesound.org/people/BitsyMcNics/sounds/251645/) by BitsyMcNics [CC0]\n- [5-250258-A.ogg]: clip derived from Saw 2.wav (http://www.freesound.org/people/kwahmah_02/sounds/250258/) by kwahmah_02 [CC-BY]\n- [5-253094-A.ogg]: clip derived from Saha, halkosaha / Saw, logsaw, man sawing wood manually (http://www.freesound.org/people/YleArkisto/sounds/253094/) by YleArkisto [CC-BY]\n- [5-253094-B.ogg]: clip derived from Saha, halkosaha / Saw, logsaw, man sawing wood manually (http://www.freesound.org/people/YleArkisto/sounds/253094/) by YleArkisto [CC-BY]\n- [5-253094-C.ogg]: clip derived from Saha, halkosaha / Saw, logsaw, man sawing wood manually (http://www.freesound.org/people/YleArkisto/sounds/253094/) by YleArkisto [CC-BY]\n- [5-253094-D.ogg]: clip derived from Saha, halkosaha / Saw, logsaw, man sawing wood manually (http://www.freesound.org/people/YleArkisto/sounds/253094/) by YleArkisto [CC-BY]\n- [5-253101-A.ogg]: clip derived from Saha, justeeri / Saw , a two-man saw, whipsaw, men sawing a stump (http://www.freesound.org/people/YleArkisto/sounds/253101/) by YleArkisto [CC-BY]\n- [5-253101-B.ogg]: clip derived from Saha, justeeri / Saw , a two-man saw, whipsaw, men sawing a stump (http://www.freesound.org/people/YleArkisto/sounds/253101/) by YleArkisto [CC-BY]\n- [5-253101-C.ogg]: clip derived from Saha, justeeri / Saw , a two-man saw, whipsaw, men sawing a stump (http://www.freesound.org/people/YleArkisto/sounds/253101/) by YleArkisto [CC-BY]\n', 'licenses/esc-metadata.csv': 'filename,fold,target,category,esc10,src_file,take\n1-100032-A-0.wav,1,0,dog,True,100032,A\n1-100038-A-14.wav,1,14,chirping_birds,False,100038,A\n1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A\n1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B\n1-101296-A-19.wav,1,19,thunderstorm,False,101296,A\n1-101296-B-19.wav,1,19,thunderstorm,False,101296,B\n1-101336-A-30.wav,1,30,door_wood_knock,False,101336,A\n1-101404-A-34.wav,1,34,can_opening,False,101404,A\n1-103298-A-9.wav,1,9,crow,False,103298,A\n1-103995-A-30.wav,1,30,door_wood_knock,False,103995,A\n1-103999-A-30.wav,1,30,door_wood_knock,False,103999,A\n1-104089-A-22.wav,1,22,clapping,False,104089,A\n1-104089-B-22.wav,1,22,clapping,False,104089,B\n1-105224-A-22.wav,1,22,clapping,False,105224,A\n1-110389-A-0.wav,1,0,dog,True,110389,A\n1-110537-A-22.wav,1,22,clapping,False,110537,A\n1-115521-A-19.wav,1,19,thunderstorm,False,115521,A\n1-115545-A-48.wav,1,48,fireworks,False,115545,A\n1-115545-B-48.wav,1,48,fireworks,False,115545,B\n1-115545-C-48.wav,1,48,fireworks,False,115545,C\n1-115546-A-48.wav,1,48,fireworks,False,115546,A\n1-115920-A-22.wav,1,22,clapping,False,115920,A\n1-115920-B-22.wav,1,22,clapping,False,115920,B\n1-115921-A-22.wav,1,22,clapping,False,115921,A\n1-116765-A-41.wav,1,41,chainsaw,True,116765,A\n1-11687-A-47.wav,1,47,airplane,False,11687,A\n1-118206-A-31.wav,1,31,mouse_click,False,118206,A\n1-118559-A-17.wav,1,17,pouring_water,False,118559,A\n1-119125-A-45.wav,1,45,train,False,119125,A\n1-121951-A-8.wav,1,8,sheep,False,121951,A\n1-12653-A-15.wav,1,15,water_drops,False,12653,A\n1-12654-A-15.wav,1,15,water_drops,False,12654,A\n1-12654-B-15.wav,1,15,water_drops,False,12654,B\n1-13571-A-46.wav,1,46,church_bells,False,13571,A\n1-13572-A-46.wav,1,46,church_bells,False,13572,A\n1-13613-A-37.wav,1,37,clock_alarm,False,13613,A\n1-137-A-32.wav,1,32,keyboard_typing,False,137,A\n1-137296-A-16.wav,1,16,wind,False,137296,A\n1-14262-A-37.wav,1,37,clock_alarm,False,14262,A\n1-155858-A-25.wav,1,25,footsteps,False,155858,A\n1-155858-B-25.wav,1,25,footsteps,False,155858,B\n1-155858-C-25.wav,1,25,footsteps,False,155858,C\n1-155858-D-25.wav,1,25,footsteps,False,155858,D\n1-155858-E-25.wav,1,25,footsteps,False,155858,E\n1-155858-F-25.wav,1,25,footsteps,False,155858,F\n1-15689-A-4.wav,1,4,frog,False,15689,A\n1-15689-B-4.wav,1,4,frog,False,15689,B\n1-160563-A-48.wav,1,48,fireworks,False,160563,A\n1-160563-B-48.wav,1,48,fireworks,False,160563,B\n1-16568-A-3.wav,1,3,cow,False,16568,A\n1-16746-A-15.wav,1,15,water_drops,False,16746,A\n1-17092-A-27.wav,1,27,brushing_teeth,False,17092,A\n1-17092-B-27.wav,1,27,brushing_teeth,False,17092,B\n1-17124-A-43.wav,1,43,car_horn,False,17124,A\n1-17150-A-12.wav,1,12,crackling_fire,True,17150,A\n1-172649-A-40.wav,1,40,helicopter,True,172649,A\n1-172649-B-40.wav,1,40,helicopter,True,172649,B\n1-172649-C-40.wav,1,40,helicopter,True,172649,C\n1-172649-D-40.wav,1,40,helicopter,True,172649,D\n1-172649-E-40.wav,1,40,helicopter,True,172649,E\n1-172649-F-40.wav,1,40,helicopter,True,172649,F\n1-17295-A-29.wav,1,29,drinking_sipping,False,17295,A\n1-17367-A-10.wav,1,10,rain,True,17367,A\n1-17565-A-12.wav,1,12,crackling_fire,True,17565,A\n1-17585-A-7.wav,1,7,insects,False,17585,A\n1-17742-A-12.wav,1,12,crackling_fire,True,17742,A\n1-17808-A-12.wav,1,12,crackling_fire,True,17808,A\n1-17808-B-12.wav,1,12,crackling_fire,True,17808,B\n1-1791-A-26.wav,1,26,laughing,False,1791,A\n1-17970-A-4.wav,1,4,frog,False,17970,A\n1-18074-A-6.wav,1,6,hen,False,18074,A\n1-18074-B-6.wav,1,6,hen,False,18074,B\n1-181071-A-40.wav,1,40,helicopter,True,181071,A\n1-181071-B-40.wav,1,40,helicopter,True,181071,B\n1-18527-A-44.wav,1,44,engine,False,18527,A\n1-18527-B-44.wav,1,44,engine,False,18527,B\n1-18631-A-23.wav,1,23,breathing,False,18631,A\n1-18655-A-31.wav,1,31,mouse_click,False,18655,A\n1-187207-A-20.wav,1,20,crying_baby,True,187207,A\n1-18755-A-4.wav,1,4,frog,False,18755,A\n1-18755-B-4.wav,1,4,frog,False,18755,B\n1-18757-A-4.wav,1,4,frog,False,18757,A\n1-18810-A-49.wav,1,49,hand_saw,False,18810,A\n1-19026-A-43.wav,1,43,car_horn,False,19026,A\n1-19111-A-24.wav,1,24,coughing,False,19111,A\n1-19118-A-24.wav,1,24,coughing,False,19118,A\n1-19501-A-7.wav,1,7,insects,False,19501,A\n1-196660-A-8.wav,1,8,sheep,False,196660,A\n1-196660-B-8.wav,1,8,sheep,False,196660,B\n1-19840-A-36.wav,1,36,vacuum_cleaner,False,19840,A\n1-19872-A-36.wav,1,36,vacuum_cleaner,False,19872,A\n1-19872-B-36.wav,1,36,vacuum_cleaner,False,19872,B\n1-19898-A-41.wav,1,41,chainsaw,True,19898,A\n1-19898-B-41.wav,1,41,chainsaw,True,19898,B\n1-19898-C-41.wav,1,41,chainsaw,True,19898,C\n1-20133-A-39.wav,1,39,glass_breaking,False,20133,A\n1-202111-A-3.wav,1,3,cow,False,202111,A\n1-20545-A-28.wav,1,28,snoring,False,20545,A\n1-20736-A-18.wav,1,18,toilet_flush,False,20736,A\n1-208757-A-2.wav,1,2,pig,False,208757,A\n1-208757-B-2.wav,1,2,pig,False,208757,B\n1-208757-C-2.wav,1,2,pig,False,208757,C\n1-208757-D-2.wav,1,2,pig,False,208757,D\n1-208757-E-2.wav,1,2,pig,False,208757,E\n1-211527-A-20.wav,1,20,crying_baby,True,211527,A\n1-211527-B-20.wav,1,20,crying_baby,True,211527,B\n1-211527-C-20.wav,1,20,crying_baby,True,211527,C\n1-21189-A-10.wav,1,10,rain,True,21189,A\n1-21421-A-46.wav,1,46,church_bells,False,21421,A\n1-21896-A-35.wav,1,35,washing_machine,False,21896,A\n1-21934-A-38.wav,1,38,clock_tick,True,21934,A\n1-21935-A-38.wav,1,38,clock_tick,True,21935,A\n1-223162-A-25.wav,1,25,footsteps,False,223162,A\n1-22694-A-20.wav,1,20,crying_baby,True,22694,A\n1-22694-B-20.wav,1,20,crying_baby,True,22694,B\n1-22804-A-46.wav,1,46,church_bells,False,22804,A\n1-22882-A-44.wav,1,44,engine,False,22882,A\n1-23094-A-15.wav,1,15,water_drops,False,23094,A\n1-23094-B-15.wav,1,15,water_drops,False,23094,B\n1-23222-A-19.wav,1,19,thunderstorm,False,23222,A\n1-23222-B-19.wav,1,19,thunderstorm,False,23222,B\n1-23706-A-49.wav,1,49,hand_saw,False,23706,A\n1-23996-A-35.wav,1,35,washing_machine,False,23996,A\n1-23996-B-35.wav,1,35,washing_machine,False,23996,B\n1-24074-A-43.wav,1,43,car_horn,False,24074,A\n1-24076-A-43.wav,1,43,car_horn,False,24076,A\n1-24524-A-19.wav,1,19,thunderstorm,False,24524,A\n1-24524-B-19.wav,1,19,thunderstorm,False,24524,B\n1-24524-C-19.wav,1,19,thunderstorm,False,24524,C\n1-24796-A-47.wav,1,47,airplane,False,24796,A\n1-254507-A-43.wav,1,43,car_horn,False,254507,A\n1-25777-A-48.wav,1,48,fireworks,False,25777,A\n1-25781-A-48.wav,1,48,fireworks,False,25781,A\n1-260640-A-2.wav,1,2,pig,False,260640,A\n1-260640-B-2.wav,1,2,pig,False,260640,B\n1-260640-C-2.wav,1,2,pig,False,260640,C\n1-26143-A-21.wav,1,21,sneezing,True,26143,A\n1-26176-A-43.wav,1,43,car_horn,False,26176,A\n1-26177-A-43.wav,1,43,car_horn,False,26177,A\n1-26188-A-30.wav,1,30,door_wood_knock,False,26188,A\n1-26222-A-10.wav,1,10,rain,True,26222,A\n1-26806-A-1.wav,1,1,rooster,True,26806,A\n1-27165-A-35.wav,1,35,washing_machine,False,27165,A\n1-27166-A-35.wav,1,35,washing_machine,False,27166,A\n1-27403-A-28.wav,1,28,snoring,False,27403,A\n1-27405-A-28.wav,1,28,snoring,False,27405,A\n1-27724-A-1.wav,1,1,rooster,True,27724,A\n1-28005-A-18.wav,1,18,toilet_flush,False,28005,A\n1-28135-A-11.wav,1,11,sea_waves,True,28135,A\n1-28135-B-11.wav,1,11,sea_waves,True,28135,B\n1-28808-A-43.wav,1,43,car_horn,False,28808,A\n1-29532-A-16.wav,1,16,wind,False,29532,A\n1-29561-A-10.wav,1,10,rain,True,29561,A\n1-29680-A-21.wav,1,21,sneezing,True,29680,A\n1-30039-A-26.wav,1,26,laughing,False,30039,A\n1-30043-A-26.wav,1,26,laughing,False,30043,A\n1-30214-A-18.wav,1,18,toilet_flush,False,30214,A\n1-30226-A-0.wav,1,0,dog,True,30226,A\n1-30344-A-0.wav,1,0,dog,True,30344,A\n1-30709-A-23.wav,1,23,breathing,False,30709,A\n1-30709-B-23.wav,1,23,breathing,False,30709,B\n1-30709-C-23.wav,1,23,breathing,False,30709,C\n1-30830-A-24.wav,1,24,coughing,False,30830,A\n1-31251-A-6.wav,1,6,hen,False,31251,A\n1-31251-B-6.wav,1,6,hen,False,31251,B\n1-31482-A-42.wav,1,42,siren,False,31482,A\n1-31482-B-42.wav,1,42,siren,False,31482,B\n1-31748-A-21.wav,1,21,sneezing,True,31748,A\n1-31836-A-4.wav,1,4,frog,False,31836,A\n1-31836-B-4.wav,1,4,frog,False,31836,B\n1-32318-A-0.wav,1,0,dog,True,32318,A\n1-32373-A-35.wav,1,35,washing_machine,False,32373,A\n1-32373-B-35.wav,1,35,washing_machine,False,32373,B\n1-32579-A-29.wav,1,29,drinking_sipping,False,32579,A\n1-33658-A-26.wav,1,26,laughing,False,33658,A\n1-34094-A-5.wav,1,5,cat,False,34094,A\n1-34094-B-5.wav,1,5,cat,False,34094,B\n1-34119-A-1.wav,1,1,rooster,True,34119,A\n1-34119-B-1.wav,1,1,rooster,True,34119,B\n1-34495-A-14.wav,1,14,chirping_birds,False,34495,A\n1-34497-A-14.wav,1,14,chirping_birds,False,34497,A\n1-34853-A-37.wav,1,37,clock_alarm,False,34853,A\n1-35687-A-38.wav,1,38,clock_tick,True,35687,A\n1-36164-A-26.wav,1,26,laughing,False,36164,A\n1-36164-B-26.wav,1,26,laughing,False,36164,B\n1-36393-A-23.wav,1,23,breathing,False,36393,A\n1-36397-A-23.wav,1,23,breathing,False,36397,A\n1-36400-A-23.wav,1,23,breathing,False,36400,A\n1-36402-A-23.wav,1,23,breathing,False,36402,A\n1-36929-A-47.wav,1,47,airplane,False,36929,A\n1-37226-A-29.wav,1,29,drinking_sipping,False,37226,A\n1-38559-A-14.wav,1,14,chirping_birds,False,38559,A\n1-38560-A-14.wav,1,14,chirping_birds,False,38560,A\n1-39835-A-9.wav,1,9,crow,False,39835,A\n1-39835-B-9.wav,1,9,crow,False,39835,B\n1-39901-A-11.wav,1,11,sea_waves,True,39901,A\n1-39901-B-11.wav,1,11,sea_waves,True,39901,B\n1-39923-A-1.wav,1,1,rooster,True,39923,A\n1-39937-A-28.wav,1,28,snoring,False,39937,A\n1-40154-A-46.wav,1,46,church_bells,False,40154,A\n1-40621-A-28.wav,1,28,snoring,False,40621,A\n1-40730-A-1.wav,1,1,rooster,True,40730,A\n1-40967-A-28.wav,1,28,snoring,False,40967,A\n1-41615-A-34.wav,1,34,can_opening,False,41615,A\n1-4211-A-12.wav,1,12,crackling_fire,True,4211,A\n1-42139-A-38.wav,1,38,clock_tick,True,42139,A\n1-43382-A-1.wav,1,1,rooster,True,43382,A\n1-43760-A-11.wav,1,11,sea_waves,True,43760,A\n1-43764-A-34.wav,1,34,can_opening,False,43764,A\n1-43807-A-47.wav,1,47,airplane,False,43807,A\n1-43807-B-47.wav,1,47,airplane,False,43807,B\n1-43807-C-47.wav,1,47,airplane,False,43807,C\n1-43807-D-47.wav,1,47,airplane,False,43807,D\n1-44831-A-1.wav,1,1,rooster,True,44831,A\n1-45641-A-27.wav,1,27,brushing_teeth,False,45641,A\n1-45645-A-31.wav,1,31,mouse_click,False,45645,A\n1-46040-A-14.wav,1,14,chirping_birds,False,46040,A\n1-46272-A-12.wav,1,12,crackling_fire,True,46272,A\n1-46274-A-18.wav,1,18,toilet_flush,False,46274,A\n1-46353-A-49.wav,1,49,hand_saw,False,46353,A\n1-46744-A-36.wav,1,36,vacuum_cleaner,False,46744,A\n1-46938-A-7.wav,1,7,insects,False,46938,A\n1-46938-B-7.wav,1,7,insects,False,46938,B\n1-47250-A-41.wav,1,41,chainsaw,True,47250,A\n1-47250-B-41.wav,1,41,chainsaw,True,47250,B\n1-47273-A-21.wav,1,21,sneezing,True,47273,A\n1-47274-A-21.wav,1,21,sneezing,True,47274,A\n1-47709-A-16.wav,1,16,wind,False,47709,A\n1-47714-A-16.wav,1,16,wind,False,47714,A\n1-47819-A-5.wav,1,5,cat,False,47819,A\n1-47819-B-5.wav,1,5,cat,False,47819,B\n1-47819-C-5.wav,1,5,cat,False,47819,C\n1-47923-A-28.wav,1,28,snoring,False,47923,A\n1-48298-A-46.wav,1,46,church_bells,False,48298,A\n1-48413-A-38.wav,1,38,clock_tick,True,48413,A\n1-49098-A-35.wav,1,35,washing_machine,False,49098,A\n1-49409-A-8.wav,1,8,sheep,False,49409,A\n1-49409-B-8.wav,1,8,sheep,False,49409,B\n1-50060-A-10.wav,1,10,rain,True,50060,A\n1-50454-A-44.wav,1,44,engine,False,50454,A\n1-50455-A-44.wav,1,44,engine,False,50455,A\n1-50623-A-15.wav,1,15,water_drops,False,50623,A\n1-50625-A-17.wav,1,17,pouring_water,False,50625,A\n1-50661-A-44.wav,1,44,engine,False,50661,A\n1-50688-A-17.wav,1,17,pouring_water,False,50688,A\n1-51035-A-16.wav,1,16,wind,False,51035,A\n1-51036-A-16.wav,1,16,wind,False,51036,A\n1-51037-A-16.wav,1,16,wind,False,51037,A\n1-51147-A-25.wav,1,25,footsteps,False,51147,A\n1-51170-A-18.wav,1,18,toilet_flush,False,51170,A\n1-51433-A-17.wav,1,17,pouring_water,False,51433,A\n1-51436-A-17.wav,1,17,pouring_water,False,51436,A\n1-51805-A-33.wav,1,33,door_wood_creaks,False,51805,A\n1-51805-B-33.wav,1,33,door_wood_creaks,False,51805,B\n1-51805-C-33.wav,1,33,door_wood_creaks,False,51805,C\n1-51805-D-33.wav,1,33,door_wood_creaks,False,51805,D\n1-51805-E-33.wav,1,33,door_wood_creaks,False,51805,E\n1-51805-F-33.wav,1,33,door_wood_creaks,False,51805,F\n1-51805-G-33.wav,1,33,door_wood_creaks,False,51805,G\n1-51805-H-33.wav,1,33,door_wood_creaks,False,51805,H\n1-52266-A-24.wav,1,24,coughing,False,52266,A\n1-52290-A-30.wav,1,30,door_wood_knock,False,52290,A\n1-52323-A-24.wav,1,24,coughing,False,52323,A\n1-53444-A-28.wav,1,28,snoring,False,53444,A\n1-53467-A-47.wav,1,47,airplane,False,53467,A\n1-53501-A-32.wav,1,32,keyboard_typing,False,53501,A\n1-53663-A-24.wav,1,24,coughing,False,53663,A\n1-53670-A-18.wav,1,18,toilet_flush,False,53670,A\n1-54065-A-45.wav,1,45,train,False,54065,A\n1-54065-B-45.wav,1,45,train,False,54065,B\n1-54084-A-42.wav,1,42,siren,False,54084,A\n1-54505-A-21.wav,1,21,sneezing,True,54505,A\n1-54747-A-46.wav,1,46,church_bells,False,54747,A\n1-54752-A-18.wav,1,18,toilet_flush,False,54752,A\n1-54918-A-14.wav,1,14,chirping_birds,False,54918,A\n1-54918-B-14.wav,1,14,chirping_birds,False,54918,B\n1-54958-A-10.wav,1,10,rain,True,54958,A\n1-56233-A-9.wav,1,9,crow,False,56233,A\n1-56234-A-9.wav,1,9,crow,False,56234,A\n1-56269-A-18.wav,1,18,toilet_flush,False,56269,A\n1-56270-A-29.wav,1,29,drinking_sipping,False,56270,A\n1-56311-A-10.wav,1,10,rain,True,56311,A\n1-56380-A-5.wav,1,5,cat,False,56380,A\n1-56380-B-5.wav,1,5,cat,False,56380,B\n1-56907-A-46.wav,1,46,church_bells,False,56907,A\n1-57163-A-38.wav,1,38,clock_tick,True,57163,A\n1-57316-A-13.wav,1,13,crickets,False,57316,A\n1-57318-A-13.wav,1,13,crickets,False,57318,A\n1-57795-A-8.wav,1,8,sheep,False,57795,A\n1-58277-A-3.wav,1,3,cow,False,58277,A\n1-58792-A-24.wav,1,24,coughing,False,58792,A\n1-58846-A-34.wav,1,34,can_opening,False,58846,A\n1-58923-A-27.wav,1,27,brushing_teeth,False,58923,A\n1-58923-B-27.wav,1,27,brushing_teeth,False,58923,B\n1-59102-A-13.wav,1,13,crickets,False,59102,A\n1-59324-A-21.wav,1,21,sneezing,True,59324,A\n1-59513-A-0.wav,1,0,dog,True,59513,A\n1-5996-A-6.wav,1,6,hen,False,5996,A\n1-60460-A-36.wav,1,36,vacuum_cleaner,False,60460,A\n1-60676-A-34.wav,1,34,can_opening,False,60676,A\n1-60997-A-20.wav,1,20,crying_baby,True,60997,A\n1-60997-B-20.wav,1,20,crying_baby,True,60997,B\n1-61212-A-32.wav,1,32,keyboard_typing,False,61212,A\n1-61221-A-17.wav,1,17,pouring_water,False,61221,A\n1-61252-A-11.wav,1,11,sea_waves,True,61252,A\n1-61261-A-44.wav,1,44,engine,False,61261,A\n1-61534-A-27.wav,1,27,brushing_teeth,False,61534,A\n1-62509-A-45.wav,1,45,train,False,62509,A\n1-62565-A-44.wav,1,44,engine,False,62565,A\n1-62594-A-32.wav,1,32,keyboard_typing,False,62594,A\n1-62849-A-38.wav,1,38,clock_tick,True,62849,A\n1-62850-A-38.wav,1,38,clock_tick,True,62850,A\n1-63679-A-24.wav,1,24,coughing,False,63679,A\n1-63871-A-10.wav,1,10,rain,True,63871,A\n1-64398-A-41.wav,1,41,chainsaw,True,64398,A\n1-64398-B-41.wav,1,41,chainsaw,True,64398,B\n1-64473-A-45.wav,1,45,train,False,64473,A\n1-65483-A-13.wav,1,13,crickets,False,65483,A\n1-67033-A-37.wav,1,37,clock_alarm,False,67033,A\n1-67152-A-17.wav,1,17,pouring_water,False,67152,A\n1-67230-A-29.wav,1,29,drinking_sipping,False,67230,A\n1-67432-A-27.wav,1,27,brushing_teeth,False,67432,A\n1-68628-A-27.wav,1,27,brushing_teeth,False,68628,A\n1-68670-A-34.wav,1,34,can_opening,False,68670,A\n1-68734-A-34.wav,1,34,can_opening,False,68734,A\n1-69165-A-34.wav,1,34,can_opening,False,69165,A\n1-69422-A-3.wav,1,3,cow,False,69422,A\n1-69641-A-3.wav,1,3,cow,False,69641,A\n1-69760-A-16.wav,1,16,wind,False,69760,A\n1-70300-A-45.wav,1,45,train,False,70300,A\n1-7057-A-12.wav,1,12,crackling_fire,True,7057,A\n1-71030-A-6.wav,1,6,hen,False,71030,A\n1-72195-A-37.wav,1,37,clock_alarm,False,72195,A\n1-72195-B-37.wav,1,37,clock_alarm,False,72195,B\n1-72229-A-6.wav,1,6,hen,False,72229,A\n1-72229-B-6.wav,1,6,hen,False,72229,B\n1-72695-A-26.wav,1,26,laughing,False,72695,A\n1-73123-A-26.wav,1,26,laughing,False,73123,A\n1-73585-A-7.wav,1,7,insects,False,73585,A\n1-74517-A-37.wav,1,37,clock_alarm,False,74517,A\n1-7456-A-13.wav,1,13,crickets,False,7456,A\n1-75162-A-9.wav,1,9,crow,False,75162,A\n1-75189-A-7.wav,1,7,insects,False,75189,A\n1-75190-A-8.wav,1,8,sheep,False,75190,A\n1-76831-A-42.wav,1,42,siren,False,76831,A\n1-76831-B-42.wav,1,42,siren,False,76831,B\n1-76831-C-42.wav,1,42,siren,False,76831,C\n1-76831-D-42.wav,1,42,siren,False,76831,D\n1-76831-E-42.wav,1,42,siren,False,76831,E\n1-77160-A-3.wav,1,3,cow,False,77160,A\n1-77241-A-3.wav,1,3,cow,False,77241,A\n1-79113-A-5.wav,1,5,cat,False,79113,A\n1-79146-A-29.wav,1,29,drinking_sipping,False,79146,A\n1-79220-A-17.wav,1,17,pouring_water,False,79220,A\n1-79236-A-36.wav,1,36,vacuum_cleaner,False,79236,A\n1-79711-A-32.wav,1,32,keyboard_typing,False,79711,A\n1-7973-A-7.wav,1,7,insects,False,7973,A\n1-7974-A-49.wav,1,49,hand_saw,False,7974,A\n1-7974-B-49.wav,1,49,hand_saw,False,7974,B\n1-80785-A-7.wav,1,7,insects,False,80785,A\n1-80840-A-13.wav,1,13,crickets,False,80840,A\n1-80841-A-13.wav,1,13,crickets,False,80841,A\n1-81001-A-30.wav,1,30,door_wood_knock,False,81001,A\n1-81001-B-30.wav,1,30,door_wood_knock,False,81001,B\n1-81269-A-3.wav,1,3,cow,False,81269,A\n1-81851-A-31.wav,1,31,mouse_click,False,81851,A\n1-81883-A-21.wav,1,21,sneezing,True,81883,A\n1-82817-A-30.wav,1,30,door_wood_knock,False,82817,A\n1-84393-A-32.wav,1,32,keyboard_typing,False,84393,A\n1-84536-A-39.wav,1,39,glass_breaking,False,84536,A\n1-84704-A-39.wav,1,39,glass_breaking,False,84704,A\n1-84705-A-39.wav,1,39,glass_breaking,False,84705,A\n1-85123-A-31.wav,1,31,mouse_click,False,85123,A\n1-85168-A-39.wav,1,39,glass_breaking,False,85168,A\n1-85184-A-39.wav,1,39,glass_breaking,False,85184,A\n1-85362-A-0.wav,1,0,dog,True,85362,A\n1-85909-A-29.wav,1,29,drinking_sipping,False,85909,A\n1-87565-A-29.wav,1,29,drinking_sipping,False,87565,A\n1-88409-A-45.wav,1,45,train,False,88409,A\n1-88409-B-45.wav,1,45,train,False,88409,B\n1-88574-A-8.wav,1,8,sheep,False,88574,A\n1-88807-A-39.wav,1,39,glass_breaking,False,88807,A\n1-90797-A-15.wav,1,15,water_drops,False,90797,A\n1-91359-A-11.wav,1,11,sea_waves,True,91359,A\n1-91359-B-11.wav,1,11,sea_waves,True,91359,B\n1-94036-A-22.wav,1,22,clapping,False,94036,A\n1-94231-A-32.wav,1,32,keyboard_typing,False,94231,A\n1-94231-B-32.wav,1,32,keyboard_typing,False,94231,B\n1-95563-A-31.wav,1,31,mouse_click,False,95563,A\n1-96890-A-37.wav,1,37,clock_alarm,False,96890,A\n1-96950-A-9.wav,1,9,crow,False,96950,A\n1-96950-B-9.wav,1,9,crow,False,96950,B\n1-97392-A-0.wav,1,0,dog,True,97392,A\n1-977-A-39.wav,1,39,glass_breaking,False,977,A\n1-97793-A-31.wav,1,31,mouse_click,False,97793,A\n1-9841-A-13.wav,1,13,crickets,False,9841,A\n1-9886-A-49.wav,1,49,hand_saw,False,9886,A\n1-9887-A-49.wav,1,49,hand_saw,False,9887,A\n1-9887-B-49.wav,1,49,hand_saw,False,9887,B\n1-99958-A-31.wav,1,31,mouse_click,False,99958,A\n2-100648-A-43.wav,2,43,car_horn,False,100648,A\n2-100786-A-1.wav,2,1,rooster,True,100786,A\n2-101676-A-10.wav,2,10,rain,True,101676,A\n2-102414-A-17.wav,2,17,pouring_water,False,102414,A\n2-102414-B-17.wav,2,17,pouring_water,False,102414,B\n2-102414-C-17.wav,2,17,pouring_water,False,102414,C\n2-102414-D-17.wav,2,17,pouring_water,False,102414,D\n2-102414-E-17.wav,2,17,pouring_water,False,102414,E\n2-102414-F-17.wav,2,17,pouring_water,False,102414,F\n2-102414-G-17.wav,2,17,pouring_water,False,102414,G\n2-102435-A-37.wav,2,37,clock_alarm,False,102435,A\n2-102567-A-35.wav,2,35,washing_machine,False,102567,A\n2-102567-B-35.wav,2,35,washing_machine,False,102567,B\n2-102567-C-35.wav,2,35,washing_machine,False,102567,C\n2-102567-D-35.wav,2,35,washing_machine,False,102567,D\n2-102568-A-35.wav,2,35,washing_machine,False,102568,A\n2-102581-A-29.wav,2,29,drinking_sipping,False,102581,A\n2-102581-B-29.wav,2,29,drinking_sipping,False,102581,B\n2-102852-A-11.wav,2,11,sea_waves,True,102852,A\n2-103423-A-3.wav,2,3,cow,False,103423,A\n2-103424-A-3.wav,2,3,cow,False,103424,A\n2-103426-A-3.wav,2,3,cow,False,103426,A\n2-103427-A-3.wav,2,3,cow,False,103427,A\n2-103428-A-3.wav,2,3,cow,False,103428,A\n2-104105-A-19.wav,2,19,thunderstorm,False,104105,A\n2-104105-B-19.wav,2,19,thunderstorm,False,104105,B\n2-104168-A-32.wav,2,32,keyboard_typing,False,104168,A\n2-104475-A-37.wav,2,37,clock_alarm,False,104475,A\n2-104476-A-37.wav,2,37,clock_alarm,False,104476,A\n2-104877-A-3.wav,2,3,cow,False,104877,A\n2-104952-A-16.wav,2,16,wind,False,104952,A\n2-104952-B-16.wav,2,16,wind,False,104952,B\n2-105270-A-47.wav,2,47,airplane,False,105270,A\n2-106014-A-44.wav,2,44,engine,False,106014,A\n2-106015-A-44.wav,2,44,engine,False,106015,A\n2-106015-B-44.wav,2,44,engine,False,106015,B\n2-106019-A-13.wav,2,13,crickets,False,106019,A\n2-106072-A-36.wav,2,36,vacuum_cleaner,False,106072,A\n2-106073-A-36.wav,2,36,vacuum_cleaner,False,106073,A\n2-106486-A-44.wav,2,44,engine,False,106486,A\n2-106487-A-44.wav,2,44,engine,False,106487,A\n2-106849-A-47.wav,2,47,airplane,False,106849,A\n2-106881-A-39.wav,2,39,glass_breaking,False,106881,A\n2-107228-A-44.wav,2,44,engine,False,107228,A\n2-107228-B-44.wav,2,44,engine,False,107228,B\n2-107351-A-20.wav,2,20,crying_baby,True,107351,A\n2-107351-B-20.wav,2,20,crying_baby,True,107351,B\n2-108017-A-24.wav,2,24,coughing,False,108017,A\n2-108760-A-14.wav,2,14,chirping_birds,False,108760,A\n2-108760-B-14.wav,2,14,chirping_birds,False,108760,B\n2-108761-A-14.wav,2,14,chirping_birds,False,108761,A\n2-108763-A-9.wav,2,9,crow,False,108763,A\n2-108766-A-9.wav,2,9,crow,False,108766,A\n2-108767-A-9.wav,2,9,crow,False,108767,A\n2-108767-B-9.wav,2,9,crow,False,108767,B\n2-108767-C-9.wav,2,9,crow,False,108767,C\n2-109231-A-9.wav,2,9,crow,False,109231,A\n2-109231-B-9.wav,2,9,crow,False,109231,B\n2-109231-C-9.wav,2,9,crow,False,109231,C\n2-109316-A-32.wav,2,32,keyboard_typing,False,109316,A\n2-109371-A-16.wav,2,16,wind,False,109371,A\n2-109371-B-16.wav,2,16,wind,False,109371,B\n2-109371-C-16.wav,2,16,wind,False,109371,C\n2-109371-D-16.wav,2,16,wind,False,109371,D\n2-109374-A-16.wav,2,16,wind,False,109374,A\n2-109505-A-21.wav,2,21,sneezing,True,109505,A\n2-109759-A-26.wav,2,26,laughing,False,109759,A\n2-109759-B-26.wav,2,26,laughing,False,109759,B\n2-110010-A-5.wav,2,5,cat,False,110010,A\n2-110011-A-5.wav,2,5,cat,False,110011,A\n2-110417-A-28.wav,2,28,snoring,False,110417,A\n2-110417-B-28.wav,2,28,snoring,False,110417,B\n2-110613-A-13.wav,2,13,crickets,False,110613,A\n2-110614-A-8.wav,2,8,sheep,False,110614,A\n2-110614-B-8.wav,2,8,sheep,False,110614,B\n2-112213-A-39.wav,2,39,glass_breaking,False,112213,A\n2-112213-B-39.wav,2,39,glass_breaking,False,112213,B\n2-114254-A-30.wav,2,30,door_wood_knock,False,114254,A\n2-114280-A-0.wav,2,0,dog,True,114280,A\n2-114587-A-0.wav,2,0,dog,True,114587,A\n2-114609-A-28.wav,2,28,snoring,False,114609,A\n2-114609-B-28.wav,2,28,snoring,False,114609,B\n2-116400-A-0.wav,2,0,dog,True,116400,A\n2-117116-A-37.wav,2,37,clock_alarm,False,117116,A\n2-117271-A-0.wav,2,0,dog,True,117271,A\n2-117330-A-28.wav,2,28,snoring,False,117330,A\n2-117615-A-48.wav,2,48,fireworks,False,117615,A\n2-117615-B-48.wav,2,48,fireworks,False,117615,B\n2-117615-C-48.wav,2,48,fireworks,False,117615,C\n2-117615-D-48.wav,2,48,fireworks,False,117615,D\n2-117615-E-48.wav,2,48,fireworks,False,117615,E\n2-117616-A-48.wav,2,48,fireworks,False,117616,A\n2-117617-A-48.wav,2,48,fireworks,False,117617,A\n2-117625-A-10.wav,2,10,rain,True,117625,A\n2-117795-A-3.wav,2,3,cow,False,117795,A\n2-117795-B-3.wav,2,3,cow,False,117795,B\n2-118072-A-0.wav,2,0,dog,True,118072,A\n2-118104-A-21.wav,2,21,sneezing,True,118104,A\n2-118459-A-32.wav,2,32,keyboard_typing,False,118459,A\n2-118459-B-32.wav,2,32,keyboard_typing,False,118459,B\n2-118624-A-30.wav,2,30,door_wood_knock,False,118624,A\n2-118625-A-30.wav,2,30,door_wood_knock,False,118625,A\n2-118817-A-32.wav,2,32,keyboard_typing,False,118817,A\n2-118964-A-0.wav,2,0,dog,True,118964,A\n2-119102-A-21.wav,2,21,sneezing,True,119102,A\n2-119139-A-31.wav,2,31,mouse_click,False,119139,A\n2-119161-A-8.wav,2,8,sheep,False,119161,A\n2-119161-B-8.wav,2,8,sheep,False,119161,B\n2-119161-C-8.wav,2,8,sheep,False,119161,C\n2-119748-A-38.wav,2,38,clock_tick,True,119748,A\n2-120218-A-30.wav,2,30,door_wood_knock,False,120218,A\n2-120218-B-30.wav,2,30,door_wood_knock,False,120218,B\n2-120333-A-32.wav,2,32,keyboard_typing,False,120333,A\n2-120586-A-6.wav,2,6,hen,False,120586,A\n2-120587-A-6.wav,2,6,hen,False,120587,A\n2-121909-A-35.wav,2,35,washing_machine,False,121909,A\n2-121978-A-29.wav,2,29,drinking_sipping,False,121978,A\n2-122066-A-45.wav,2,45,train,False,122066,A\n2-122066-B-45.wav,2,45,train,False,122066,B\n2-122067-A-45.wav,2,45,train,False,122067,A\n2-122067-B-45.wav,2,45,train,False,122067,B\n2-122104-A-0.wav,2,0,dog,True,122104,A\n2-122104-B-0.wav,2,0,dog,True,122104,B\n2-122616-A-14.wav,2,14,chirping_birds,False,122616,A\n2-122763-A-29.wav,2,29,drinking_sipping,False,122763,A\n2-122763-B-29.wav,2,29,drinking_sipping,False,122763,B\n2-122820-A-36.wav,2,36,vacuum_cleaner,False,122820,A\n2-122820-B-36.wav,2,36,vacuum_cleaner,False,122820,B\n2-123712-A-33.wav,2,33,door_wood_creaks,False,123712,A\n2-123896-A-24.wav,2,24,coughing,False,123896,A\n2-124564-A-15.wav,2,15,water_drops,False,124564,A\n2-124662-A-11.wav,2,11,sea_waves,True,124662,A\n2-125520-A-43.wav,2,43,car_horn,False,125520,A\n2-125821-A-13.wav,2,13,crickets,False,125821,A\n2-125875-A-13.wav,2,13,crickets,False,125875,A\n2-125966-A-11.wav,2,11,sea_waves,True,125966,A\n2-126433-A-17.wav,2,17,pouring_water,False,126433,A\n2-126756-A-29.wav,2,29,drinking_sipping,False,126756,A\n2-127108-A-38.wav,2,38,clock_tick,True,127108,A\n2-127109-A-6.wav,2,6,hen,False,127109,A\n2-128465-A-43.wav,2,43,car_horn,False,128465,A\n2-128465-B-43.wav,2,43,car_horn,False,128465,B\n2-128631-A-21.wav,2,21,sneezing,True,128631,A\n2-130245-A-34.wav,2,34,can_opening,False,130245,A\n2-130978-A-21.wav,2,21,sneezing,True,130978,A\n2-130979-A-21.wav,2,21,sneezing,True,130979,A\n2-131943-A-38.wav,2,38,clock_tick,True,131943,A\n2-132157-A-11.wav,2,11,sea_waves,True,132157,A\n2-132157-B-11.wav,2,11,sea_waves,True,132157,B\n2-133863-A-11.wav,2,11,sea_waves,True,133863,A\n2-133889-A-30.wav,2,30,door_wood_knock,False,133889,A\n2-134049-A-6.wav,2,6,hen,False,134049,A\n2-134700-A-38.wav,2,38,clock_tick,True,134700,A\n2-134915-A-30.wav,2,30,door_wood_knock,False,134915,A\n2-135649-A-45.wav,2,45,train,False,135649,A\n2-135649-B-45.wav,2,45,train,False,135649,B\n2-135649-C-45.wav,2,45,train,False,135649,C\n2-135728-A-38.wav,2,38,clock_tick,True,135728,A\n2-135860-A-49.wav,2,49,hand_saw,False,135860,A\n2-137162-A-11.wav,2,11,sea_waves,True,137162,A\n2-138257-A-31.wav,2,31,mouse_click,False,138257,A\n2-138465-A-43.wav,2,43,car_horn,False,138465,A\n2-139748-A-15.wav,2,15,water_drops,False,139748,A\n2-139748-B-15.wav,2,15,water_drops,False,139748,B\n2-139749-A-15.wav,2,15,water_drops,False,139749,A\n2-140147-A-38.wav,2,38,clock_tick,True,140147,A\n2-140841-A-30.wav,2,30,door_wood_knock,False,140841,A\n2-141563-A-39.wav,2,39,glass_breaking,False,141563,A\n2-141584-A-38.wav,2,38,clock_tick,True,141584,A\n2-141681-A-36.wav,2,36,vacuum_cleaner,False,141681,A\n2-141681-B-36.wav,2,36,vacuum_cleaner,False,141681,B\n2-141682-A-36.wav,2,36,vacuum_cleaner,False,141682,A\n2-141682-B-36.wav,2,36,vacuum_cleaner,False,141682,B\n2-144031-A-34.wav,2,34,can_opening,False,144031,A\n2-144137-A-43.wav,2,43,car_horn,False,144137,A\n2-146877-A-31.wav,2,31,mouse_click,False,146877,A\n2-146877-B-31.wav,2,31,mouse_click,False,146877,B\n2-151079-A-20.wav,2,20,crying_baby,True,151079,A\n2-152895-A-31.wav,2,31,mouse_click,False,152895,A\n2-152964-A-31.wav,2,31,mouse_click,False,152964,A\n2-153388-A-31.wav,2,31,mouse_click,False,153388,A\n2-154688-A-31.wav,2,31,mouse_click,False,154688,A\n2-155801-A-11.wav,2,11,sea_waves,True,155801,A\n2-157488-A-6.wav,2,6,hen,False,157488,A\n2-158746-A-2.wav,2,2,pig,False,158746,A\n2-158746-B-2.wav,2,2,pig,False,158746,B\n2-158746-C-2.wav,2,2,pig,False,158746,C\n2-158746-D-2.wav,2,2,pig,False,158746,D\n2-160128-A-7.wav,2,7,insects,False,160128,A\n2-160888-A-47.wav,2,47,airplane,False,160888,A\n2-165801-A-48.wav,2,48,fireworks,False,165801,A\n2-166644-A-2.wav,2,2,pig,False,166644,A\n2-166644-B-2.wav,2,2,pig,False,166644,B\n2-166644-C-2.wav,2,2,pig,False,166644,C\n2-173559-A-39.wav,2,39,glass_breaking,False,173559,A\n2-173607-A-39.wav,2,39,glass_breaking,False,173607,A\n2-173618-A-39.wav,2,39,glass_breaking,False,173618,A\n2-182508-A-8.wav,2,8,sheep,False,182508,A\n2-182508-B-8.wav,2,8,sheep,False,182508,B\n2-184077-A-49.wav,2,49,hand_saw,False,184077,A\n2-18766-A-12.wav,2,12,crackling_fire,True,18766,A\n2-18766-B-12.wav,2,12,crackling_fire,True,18766,B\n2-188822-A-40.wav,2,40,helicopter,True,188822,A\n2-188822-B-40.wav,2,40,helicopter,True,188822,B\n2-188822-C-40.wav,2,40,helicopter,True,188822,C\n2-188822-D-40.wav,2,40,helicopter,True,188822,D\n2-196688-A-8.wav,2,8,sheep,False,196688,A\n2-205966-A-16.wav,2,16,wind,False,205966,A\n2-209471-A-25.wav,2,25,footsteps,False,209471,A\n2-209472-A-25.wav,2,25,footsteps,False,209472,A\n2-209473-A-25.wav,2,25,footsteps,False,209473,A\n2-209474-A-25.wav,2,25,footsteps,False,209474,A\n2-209475-A-25.wav,2,25,footsteps,False,209475,A\n2-209476-A-25.wav,2,25,footsteps,False,209476,A\n2-209477-A-25.wav,2,25,footsteps,False,209477,A\n2-209478-A-25.wav,2,25,footsteps,False,209478,A\n2-250710-A-39.wav,2,39,glass_breaking,False,250710,A\n2-25292-A-22.wav,2,22,clapping,False,25292,A\n2-25293-A-22.wav,2,22,clapping,False,25293,A\n2-262579-A-45.wav,2,45,train,False,262579,A\n2-28314-A-12.wav,2,12,crackling_fire,True,28314,A\n2-28314-B-12.wav,2,12,crackling_fire,True,28314,B\n2-30322-A-12.wav,2,12,crackling_fire,True,30322,A\n2-30322-B-12.wav,2,12,crackling_fire,True,30322,B\n2-32515-A-4.wav,2,4,frog,False,32515,A\n2-32515-B-4.wav,2,4,frog,False,32515,B\n2-32515-C-4.wav,2,4,frog,False,32515,C\n2-32515-D-4.wav,2,4,frog,False,32515,D\n2-32834-A-4.wav,2,4,frog,False,32834,A\n2-37806-A-40.wav,2,40,helicopter,True,37806,A\n2-37806-B-40.wav,2,40,helicopter,True,37806,B\n2-37806-C-40.wav,2,40,helicopter,True,37806,C\n2-37806-D-40.wav,2,40,helicopter,True,37806,D\n2-37870-A-2.wav,2,2,pig,False,37870,A\n2-39441-A-19.wav,2,19,thunderstorm,False,39441,A\n2-39443-A-19.wav,2,19,thunderstorm,False,39443,A\n2-39443-B-19.wav,2,19,thunderstorm,False,39443,B\n2-39945-A-19.wav,2,19,thunderstorm,False,39945,A\n2-39945-B-19.wav,2,19,thunderstorm,False,39945,B\n2-39945-C-19.wav,2,19,thunderstorm,False,39945,C\n2-42101-A-43.wav,2,43,car_horn,False,42101,A\n2-43802-A-42.wav,2,42,siren,False,43802,A\n2-43806-A-42.wav,2,42,siren,False,43806,A\n2-50665-A-20.wav,2,20,crying_baby,True,50665,A\n2-50666-A-20.wav,2,20,crying_baby,True,50666,A\n2-50667-A-41.wav,2,41,chainsaw,True,50667,A\n2-50667-B-41.wav,2,41,chainsaw,True,50667,B\n2-50668-A-41.wav,2,41,chainsaw,True,50668,A\n2-50668-B-41.wav,2,41,chainsaw,True,50668,B\n2-50774-A-23.wav,2,23,breathing,False,50774,A\n2-51173-A-35.wav,2,35,washing_machine,False,51173,A\n2-51630-A-49.wav,2,49,hand_saw,False,51630,A\n2-51630-B-49.wav,2,49,hand_saw,False,51630,B\n2-52001-A-28.wav,2,28,snoring,False,52001,A\n2-52001-B-28.wav,2,28,snoring,False,52001,B\n2-52085-A-4.wav,2,4,frog,False,52085,A\n2-52085-B-4.wav,2,4,frog,False,52085,B\n2-52789-A-4.wav,2,4,frog,False,52789,A\n2-54086-A-43.wav,2,43,car_horn,False,54086,A\n2-54961-A-23.wav,2,23,breathing,False,54961,A\n2-54962-A-23.wav,2,23,breathing,False,54962,A\n2-56926-A-46.wav,2,46,church_bells,False,56926,A\n2-57191-A-46.wav,2,46,church_bells,False,57191,A\n2-57733-A-22.wav,2,22,clapping,False,57733,A\n2-59241-A-35.wav,2,35,washing_machine,False,59241,A\n2-59321-A-49.wav,2,49,hand_saw,False,59321,A\n2-59565-A-46.wav,2,46,church_bells,False,59565,A\n2-59566-A-46.wav,2,46,church_bells,False,59566,A\n2-60180-A-49.wav,2,49,hand_saw,False,60180,A\n2-60180-B-49.wav,2,49,hand_saw,False,60180,B\n2-60791-A-26.wav,2,26,laughing,False,60791,A\n2-60794-A-26.wav,2,26,laughing,False,60794,A\n2-60795-A-26.wav,2,26,laughing,False,60795,A\n2-60900-A-22.wav,2,22,clapping,False,60900,A\n2-61311-A-12.wav,2,12,crackling_fire,True,61311,A\n2-61618-A-46.wav,2,46,church_bells,False,61618,A\n2-62226-A-26.wav,2,26,laughing,False,62226,A\n2-64332-A-18.wav,2,18,toilet_flush,False,64332,A\n2-64962-A-15.wav,2,15,water_drops,False,64962,A\n2-64963-A-15.wav,2,15,water_drops,False,64963,A\n2-65484-A-18.wav,2,18,toilet_flush,False,65484,A\n2-65747-A-12.wav,2,12,crackling_fire,True,65747,A\n2-65750-A-1.wav,2,1,rooster,True,65750,A\n2-66205-A-23.wav,2,23,breathing,False,66205,A\n2-66637-A-20.wav,2,20,crying_baby,True,66637,A\n2-66637-B-20.wav,2,20,crying_baby,True,66637,B\n2-67422-A-18.wav,2,18,toilet_flush,False,67422,A\n2-68391-A-41.wav,2,41,chainsaw,True,68391,A\n2-68391-B-41.wav,2,41,chainsaw,True,68391,B\n2-68595-A-15.wav,2,15,water_drops,False,68595,A\n2-68595-B-15.wav,2,15,water_drops,False,68595,B\n2-69131-A-5.wav,2,5,cat,False,69131,A\n2-69131-B-5.wav,2,5,cat,False,69131,B\n2-70052-A-42.wav,2,42,siren,False,70052,A\n2-70052-B-42.wav,2,42,siren,False,70052,B\n2-70280-A-18.wav,2,18,toilet_flush,False,70280,A\n2-70344-A-33.wav,2,33,door_wood_creaks,False,70344,A\n2-70366-A-33.wav,2,33,door_wood_creaks,False,70366,A\n2-70367-A-33.wav,2,33,door_wood_creaks,False,70367,A\n2-70936-A-42.wav,2,42,siren,False,70936,A\n2-70938-A-42.wav,2,42,siren,False,70938,A\n2-70939-A-42.wav,2,42,siren,False,70939,A\n2-71162-A-1.wav,2,1,rooster,True,71162,A\n2-72268-A-6.wav,2,6,hen,False,72268,A\n2-72547-A-14.wav,2,14,chirping_birds,False,72547,A\n2-72547-B-14.wav,2,14,chirping_birds,False,72547,B\n2-72547-C-14.wav,2,14,chirping_birds,False,72547,C\n2-72547-D-14.wav,2,14,chirping_birds,False,72547,D\n2-72677-A-18.wav,2,18,toilet_flush,False,72677,A\n2-72688-A-42.wav,2,42,siren,False,72688,A\n2-72970-A-10.wav,2,10,rain,True,72970,A\n2-73027-A-10.wav,2,10,rain,True,73027,A\n2-7321-A-33.wav,2,33,door_wood_creaks,False,7321,A\n2-73260-A-10.wav,2,10,rain,True,73260,A\n2-73544-A-27.wav,2,27,brushing_teeth,False,73544,A\n2-74361-A-47.wav,2,47,airplane,False,74361,A\n2-74977-A-18.wav,2,18,toilet_flush,False,74977,A\n2-75726-A-6.wav,2,6,hen,False,75726,A\n2-76408-A-22.wav,2,22,clapping,False,76408,A\n2-76408-B-22.wav,2,22,clapping,False,76408,B\n2-76408-C-22.wav,2,22,clapping,False,76408,C\n2-76408-D-22.wav,2,22,clapping,False,76408,D\n2-76868-A-6.wav,2,6,hen,False,76868,A\n2-77346-A-46.wav,2,46,church_bells,False,77346,A\n2-77347-A-46.wav,2,46,church_bells,False,77347,A\n2-77945-A-41.wav,2,41,chainsaw,True,77945,A\n2-77945-B-41.wav,2,41,chainsaw,True,77945,B\n2-78381-A-46.wav,2,46,church_bells,False,78381,A\n2-78562-A-37.wav,2,37,clock_alarm,False,78562,A\n2-78562-B-37.wav,2,37,clock_alarm,False,78562,B\n2-78651-A-44.wav,2,44,engine,False,78651,A\n2-78781-A-47.wav,2,47,airplane,False,78781,A\n2-78799-A-47.wav,2,47,airplane,False,78799,A\n2-79769-A-26.wav,2,26,laughing,False,79769,A\n2-79775-A-26.wav,2,26,laughing,False,79775,A\n2-80313-A-28.wav,2,28,snoring,False,80313,A\n2-80482-A-20.wav,2,20,crying_baby,True,80482,A\n2-80844-A-13.wav,2,13,crickets,False,80844,A\n2-81112-A-34.wav,2,34,can_opening,False,81112,A\n2-81190-A-34.wav,2,34,can_opening,False,81190,A\n2-81270-A-1.wav,2,1,rooster,True,81270,A\n2-81731-A-10.wav,2,10,rain,True,81731,A\n2-81970-A-7.wav,2,7,insects,False,81970,A\n2-81970-B-7.wav,2,7,insects,False,81970,B\n2-81970-C-7.wav,2,7,insects,False,81970,C\n2-82071-A-27.wav,2,27,brushing_teeth,False,82071,A\n2-82077-A-7.wav,2,7,insects,False,82077,A\n2-82274-A-5.wav,2,5,cat,False,82274,A\n2-82274-B-5.wav,2,5,cat,False,82274,B\n2-82367-A-10.wav,2,10,rain,True,82367,A\n2-82455-A-23.wav,2,23,breathing,False,82455,A\n2-82538-A-21.wav,2,21,sneezing,True,82538,A\n2-83270-A-13.wav,2,13,crickets,False,83270,A\n2-83536-A-27.wav,2,27,brushing_teeth,False,83536,A\n2-83667-A-34.wav,2,34,can_opening,False,83667,A\n2-83688-A-34.wav,2,34,can_opening,False,83688,A\n2-83934-A-5.wav,2,5,cat,False,83934,A\n2-83934-B-5.wav,2,5,cat,False,83934,B\n2-84693-A-49.wav,2,49,hand_saw,False,84693,A\n2-84943-A-18.wav,2,18,toilet_flush,False,84943,A\n2-84965-A-23.wav,2,23,breathing,False,84965,A\n2-85139-A-13.wav,2,13,crickets,False,85139,A\n2-85292-A-24.wav,2,24,coughing,False,85292,A\n2-85434-A-27.wav,2,27,brushing_teeth,False,85434,A\n2-85471-A-34.wav,2,34,can_opening,False,85471,A\n2-85945-A-18.wav,2,18,toilet_flush,False,85945,A\n2-86160-A-27.wav,2,27,brushing_teeth,False,86160,A\n2-87282-A-34.wav,2,34,can_opening,False,87282,A\n2-87412-A-24.wav,2,24,coughing,False,87412,A\n2-87780-A-33.wav,2,33,door_wood_creaks,False,87780,A\n2-87781-A-10.wav,2,10,rain,True,87781,A\n2-87794-A-24.wav,2,24,coughing,False,87794,A\n2-87795-A-24.wav,2,24,coughing,False,87795,A\n2-87799-A-24.wav,2,24,coughing,False,87799,A\n2-88724-A-38.wav,2,38,clock_tick,True,88724,A\n2-89516-A-37.wav,2,37,clock_alarm,False,89516,A\n2-91912-A-33.wav,2,33,door_wood_creaks,False,91912,A\n2-91912-B-33.wav,2,33,door_wood_creaks,False,91912,B\n2-92627-A-27.wav,2,27,brushing_teeth,False,92627,A\n2-92978-A-29.wav,2,29,drinking_sipping,False,92978,A\n2-93030-A-21.wav,2,21,sneezing,True,93030,A\n2-94230-A-27.wav,2,27,brushing_teeth,False,94230,A\n2-94807-A-29.wav,2,29,drinking_sipping,False,94807,A\n2-95035-A-1.wav,2,1,rooster,True,95035,A\n2-95258-A-1.wav,2,1,rooster,True,95258,A\n2-95258-B-1.wav,2,1,rooster,True,95258,B\n2-95567-A-23.wav,2,23,breathing,False,95567,A\n2-96033-A-13.wav,2,13,crickets,False,96033,A\n2-96063-A-37.wav,2,37,clock_alarm,False,96063,A\n2-96460-A-1.wav,2,1,rooster,True,96460,A\n2-96654-A-47.wav,2,47,airplane,False,96654,A\n2-96904-A-27.wav,2,27,brushing_teeth,False,96904,A\n2-98392-A-23.wav,2,23,breathing,False,98392,A\n2-98676-A-24.wav,2,24,coughing,False,98676,A\n2-98866-A-47.wav,2,47,airplane,False,98866,A\n2-99795-A-32.wav,2,32,keyboard_typing,False,99795,A\n2-99796-A-32.wav,2,32,keyboard_typing,False,99796,A\n2-99955-A-7.wav,2,7,insects,False,99955,A\n2-99955-B-7.wav,2,7,insects,False,99955,B\n2-99955-C-7.wav,2,7,insects,False,99955,C\n3-100018-A-18.wav,3,18,toilet_flush,False,100018,A\n3-100024-A-27.wav,3,27,brushing_teeth,False,100024,A\n3-100024-B-27.wav,3,27,brushing_teeth,False,100024,B\n3-101381-A-33.wav,3,33,door_wood_creaks,False,101381,A\n3-101381-B-33.wav,3,33,door_wood_creaks,False,101381,B\n3-102583-A-49.wav,3,49,hand_saw,False,102583,A\n3-102583-B-49.wav,3,49,hand_saw,False,102583,B\n3-102583-C-49.wav,3,49,hand_saw,False,102583,C\n3-102908-A-4.wav,3,4,frog,False,102908,A\n3-103050-A-19.wav,3,19,thunderstorm,False,103050,A\n3-103051-A-19.wav,3,19,thunderstorm,False,103051,A\n3-103051-B-19.wav,3,19,thunderstorm,False,103051,B\n3-103051-C-19.wav,3,19,thunderstorm,False,103051,C\n3-103401-A-33.wav,3,33,door_wood_creaks,False,103401,A\n3-103401-B-33.wav,3,33,door_wood_creaks,False,103401,B\n3-103401-C-33.wav,3,33,door_wood_creaks,False,103401,C\n3-103401-D-33.wav,3,33,door_wood_creaks,False,103401,D\n3-103597-A-25.wav,3,25,footsteps,False,103597,A\n3-103598-A-25.wav,3,25,footsteps,False,103598,A\n3-103599-A-25.wav,3,25,footsteps,False,103599,A\n3-103599-B-25.wav,3,25,footsteps,False,103599,B\n3-104632-A-12.wav,3,12,crackling_fire,True,104632,A\n3-104761-A-7.wav,3,7,insects,False,104761,A\n3-104761-B-7.wav,3,7,insects,False,104761,B\n3-104958-A-12.wav,3,12,crackling_fire,True,104958,A\n3-105235-A-7.wav,3,7,insects,False,105235,A\n3-105236-A-7.wav,3,7,insects,False,105236,A\n3-107123-A-26.wav,3,26,laughing,False,107123,A\n3-107219-A-1.wav,3,1,rooster,True,107219,A\n3-108160-A-23.wav,3,23,breathing,False,108160,A\n3-108451-A-17.wav,3,17,pouring_water,False,108451,A\n3-108451-B-17.wav,3,17,pouring_water,False,108451,B\n3-108677-A-18.wav,3,18,toilet_flush,False,108677,A\n3-108791-A-18.wav,3,18,toilet_flush,False,108791,A\n3-110536-A-26.wav,3,26,laughing,False,110536,A\n3-110913-A-7.wav,3,7,insects,False,110913,A\n3-110913-B-7.wav,3,7,insects,False,110913,B\n3-110913-C-7.wav,3,7,insects,False,110913,C\n3-110913-D-7.wav,3,7,insects,False,110913,D\n3-111102-A-46.wav,3,46,church_bells,False,111102,A\n3-111102-B-46.wav,3,46,church_bells,False,111102,B\n3-112356-A-18.wav,3,18,toilet_flush,False,112356,A\n3-112397-A-9.wav,3,9,crow,False,112397,A\n3-112522-A-27.wav,3,27,brushing_teeth,False,112522,A\n3-112557-A-23.wav,3,23,breathing,False,112557,A\n3-112557-B-23.wav,3,23,breathing,False,112557,B\n3-115382-A-44.wav,3,44,engine,False,115382,A\n3-115387-A-47.wav,3,47,airplane,False,115387,A\n3-115387-B-47.wav,3,47,airplane,False,115387,B\n3-115387-C-47.wav,3,47,airplane,False,115387,C\n3-116135-A-1.wav,3,1,rooster,True,116135,A\n3-117293-A-9.wav,3,9,crow,False,117293,A\n3-117504-A-16.wav,3,16,wind,False,117504,A\n3-117504-B-16.wav,3,16,wind,False,117504,B\n3-117793-A-37.wav,3,37,clock_alarm,False,117793,A\n3-117883-A-37.wav,3,37,clock_alarm,False,117883,A\n3-118059-A-18.wav,3,18,toilet_flush,False,118059,A\n3-118069-A-27.wav,3,27,brushing_teeth,False,118069,A\n3-118069-B-27.wav,3,27,brushing_teeth,False,118069,B\n3-118194-A-33.wav,3,33,door_wood_creaks,False,118194,A\n3-118487-A-26.wav,3,26,laughing,False,118487,A\n3-118656-A-41.wav,3,41,chainsaw,True,118656,A\n3-118657-A-41.wav,3,41,chainsaw,True,118657,A\n3-118657-B-41.wav,3,41,chainsaw,True,118657,B\n3-118658-A-41.wav,3,41,chainsaw,True,118658,A\n3-118658-B-41.wav,3,41,chainsaw,True,118658,B\n3-118972-A-41.wav,3,41,chainsaw,True,118972,A\n3-118972-B-41.wav,3,41,chainsaw,True,118972,B\n3-119120-A-48.wav,3,48,fireworks,False,119120,A\n3-119120-B-48.wav,3,48,fireworks,False,119120,B\n3-119120-C-48.wav,3,48,fireworks,False,119120,C\n3-119120-D-48.wav,3,48,fireworks,False,119120,D\n3-119120-E-48.wav,3,48,fireworks,False,119120,E\n3-119455-A-44.wav,3,44,engine,False,119455,A\n3-119459-A-26.wav,3,26,laughing,False,119459,A\n3-120526-A-37.wav,3,37,clock_alarm,False,120526,A\n3-120526-B-37.wav,3,37,clock_alarm,False,120526,B\n3-120644-A-12.wav,3,12,crackling_fire,True,120644,A\n3-121348-A-9.wav,3,9,crow,False,121348,A\n3-122110-A-46.wav,3,46,church_bells,False,122110,A\n3-123086-A-28.wav,3,28,snoring,False,123086,A\n3-123224-A-19.wav,3,19,thunderstorm,False,123224,A\n3-124376-A-3.wav,3,3,cow,False,124376,A\n3-124376-B-3.wav,3,3,cow,False,124376,B\n3-124600-A-19.wav,3,19,thunderstorm,False,124600,A\n3-124795-A-28.wav,3,28,snoring,False,124795,A\n3-124925-A-9.wav,3,9,crow,False,124925,A\n3-124958-A-28.wav,3,28,snoring,False,124958,A\n3-125418-A-24.wav,3,24,coughing,False,125418,A\n3-125548-A-32.wav,3,32,keyboard_typing,False,125548,A\n3-126113-A-26.wav,3,26,laughing,False,126113,A\n3-126358-A-3.wav,3,3,cow,False,126358,A\n3-126358-B-3.wav,3,3,cow,False,126358,B\n3-126391-A-27.wav,3,27,brushing_teeth,False,126391,A\n3-126391-B-27.wav,3,27,brushing_teeth,False,126391,B\n3-127874-A-17.wav,3,17,pouring_water,False,127874,A\n3-127890-A-9.wav,3,9,crow,False,127890,A\n3-127890-B-9.wav,3,9,crow,False,127890,B\n3-127890-C-9.wav,3,9,crow,False,127890,C\n3-128160-A-44.wav,3,44,engine,False,128160,A\n3-128512-A-47.wav,3,47,airplane,False,128512,A\n3-128512-B-47.wav,3,47,airplane,False,128512,B\n3-129264-A-9.wav,3,9,crow,False,129264,A\n3-129338-A-13.wav,3,13,crickets,False,129338,A\n3-129678-A-13.wav,3,13,crickets,False,129678,A\n3-130330-A-22.wav,3,22,clapping,False,130330,A\n3-130998-A-28.wav,3,28,snoring,False,130998,A\n3-130998-B-28.wav,3,28,snoring,False,130998,B\n3-131943-A-37.wav,3,37,clock_alarm,False,131943,A\n3-132340-A-37.wav,3,37,clock_alarm,False,132340,A\n3-132601-A-24.wav,3,24,coughing,False,132601,A\n3-132747-A-26.wav,3,26,laughing,False,132747,A\n3-132830-A-32.wav,3,32,keyboard_typing,False,132830,A\n3-132852-A-10.wav,3,10,rain,True,132852,A\n3-133977-A-29.wav,3,29,drinking_sipping,False,133977,A\n3-134049-A-1.wav,3,1,rooster,True,134049,A\n3-134699-A-16.wav,3,16,wind,False,134699,A\n3-134699-B-16.wav,3,16,wind,False,134699,B\n3-134699-C-16.wav,3,16,wind,False,134699,C\n3-134802-A-13.wav,3,13,crickets,False,134802,A\n3-135469-A-35.wav,3,35,washing_machine,False,135469,A\n3-135650-A-45.wav,3,45,train,False,135650,A\n3-135650-B-45.wav,3,45,train,False,135650,B\n3-136288-A-0.wav,3,0,dog,True,136288,A\n3-136451-A-45.wav,3,45,train,False,136451,A\n3-136608-A-16.wav,3,16,wind,False,136608,A\n3-137152-A-1.wav,3,1,rooster,True,137152,A\n3-138114-A-22.wav,3,22,clapping,False,138114,A\n3-138212-A-45.wav,3,45,train,False,138212,A\n3-139109-A-46.wav,3,46,church_bells,False,139109,A\n3-139331-A-27.wav,3,27,brushing_teeth,False,139331,A\n3-139958-A-37.wav,3,37,clock_alarm,False,139958,A\n3-140199-A-8.wav,3,8,sheep,False,140199,A\n3-140199-B-8.wav,3,8,sheep,False,140199,B\n3-140199-C-8.wav,3,8,sheep,False,140199,C\n3-140199-D-8.wav,3,8,sheep,False,140199,D\n3-140323-A-29.wav,3,29,drinking_sipping,False,140323,A\n3-140774-A-10.wav,3,10,rain,True,140774,A\n3-141240-A-44.wav,3,44,engine,False,141240,A\n3-141240-B-44.wav,3,44,engine,False,141240,B\n3-141559-A-45.wav,3,45,train,False,141559,A\n3-141684-A-21.wav,3,21,sneezing,True,141684,A\n3-142005-A-10.wav,3,10,rain,True,142005,A\n3-142006-A-10.wav,3,10,rain,True,142006,A\n3-142349-A-17.wav,3,17,pouring_water,False,142349,A\n3-142593-A-38.wav,3,38,clock_tick,True,142593,A\n3-142601-A-21.wav,3,21,sneezing,True,142601,A\n3-142604-A-24.wav,3,24,coughing,False,142604,A\n3-142605-A-21.wav,3,21,sneezing,True,142605,A\n3-143119-A-21.wav,3,21,sneezing,True,143119,A\n3-143560-A-47.wav,3,47,airplane,False,143560,A\n3-143562-A-47.wav,3,47,airplane,False,143562,A\n3-143929-A-10.wav,3,10,rain,True,143929,A\n3-143933-A-38.wav,3,38,clock_tick,True,143933,A\n3-144028-A-0.wav,3,0,dog,True,144028,A\n3-144106-A-32.wav,3,32,keyboard_typing,False,144106,A\n3-144120-A-32.wav,3,32,keyboard_typing,False,144120,A\n3-144128-A-23.wav,3,23,breathing,False,144128,A\n3-144128-B-23.wav,3,23,breathing,False,144128,B\n3-144253-A-29.wav,3,29,drinking_sipping,False,144253,A\n3-144259-A-29.wav,3,29,drinking_sipping,False,144259,A\n3-144510-A-30.wav,3,30,door_wood_knock,False,144510,A\n3-144692-A-21.wav,3,21,sneezing,True,144692,A\n3-144827-A-11.wav,3,11,sea_waves,True,144827,A\n3-144827-B-11.wav,3,11,sea_waves,True,144827,B\n3-144891-A-19.wav,3,19,thunderstorm,False,144891,A\n3-144891-B-19.wav,3,19,thunderstorm,False,144891,B\n3-145382-A-1.wav,3,1,rooster,True,145382,A\n3-145387-A-29.wav,3,29,drinking_sipping,False,145387,A\n3-145487-A-24.wav,3,24,coughing,False,145487,A\n3-145577-A-43.wav,3,43,car_horn,False,145577,A\n3-145719-A-17.wav,3,17,pouring_water,False,145719,A\n3-145774-A-12.wav,3,12,crackling_fire,True,145774,A\n3-146033-A-13.wav,3,13,crickets,False,146033,A\n3-146186-A-44.wav,3,44,engine,False,146186,A\n3-146697-A-43.wav,3,43,car_horn,False,146697,A\n3-146873-A-24.wav,3,24,coughing,False,146873,A\n3-146964-A-5.wav,3,5,cat,False,146964,A\n3-146965-A-5.wav,3,5,cat,False,146965,A\n3-146972-A-5.wav,3,5,cat,False,146972,A\n3-147342-A-34.wav,3,34,can_opening,False,147342,A\n3-147343-A-34.wav,3,34,can_opening,False,147343,A\n3-147965-A-12.wav,3,12,crackling_fire,True,147965,A\n3-148297-A-37.wav,3,37,clock_alarm,False,148297,A\n3-148330-A-21.wav,3,21,sneezing,True,148330,A\n3-148932-A-34.wav,3,34,can_opening,False,148932,A\n3-149042-A-24.wav,3,24,coughing,False,149042,A\n3-149189-A-1.wav,3,1,rooster,True,149189,A\n3-149448-A-17.wav,3,17,pouring_water,False,149448,A\n3-149465-A-22.wav,3,22,clapping,False,149465,A\n3-150231-A-21.wav,3,21,sneezing,True,150231,A\n3-150363-A-38.wav,3,38,clock_tick,True,150363,A\n3-150979-A-40.wav,3,40,helicopter,True,150979,A\n3-150979-B-40.wav,3,40,helicopter,True,150979,B\n3-150979-C-40.wav,3,40,helicopter,True,150979,C\n3-151080-A-20.wav,3,20,crying_baby,True,151080,A\n3-151081-A-20.wav,3,20,crying_baby,True,151081,A\n3-151081-B-20.wav,3,20,crying_baby,True,151081,B\n3-151089-A-30.wav,3,30,door_wood_knock,False,151089,A\n3-151206-A-23.wav,3,23,breathing,False,151206,A\n3-151212-A-24.wav,3,24,coughing,False,151212,A\n3-151213-A-24.wav,3,24,coughing,False,151213,A\n3-151255-A-28.wav,3,28,snoring,False,151255,A\n3-151269-A-35.wav,3,35,washing_machine,False,151269,A\n3-151273-A-35.wav,3,35,washing_machine,False,151273,A\n3-151557-A-28.wav,3,28,snoring,False,151557,A\n3-151557-B-28.wav,3,28,snoring,False,151557,B\n3-152007-A-20.wav,3,20,crying_baby,True,152007,A\n3-152007-B-20.wav,3,20,crying_baby,True,152007,B\n3-152007-C-20.wav,3,20,crying_baby,True,152007,C\n3-152007-D-20.wav,3,20,crying_baby,True,152007,D\n3-152007-E-20.wav,3,20,crying_baby,True,152007,E\n3-152020-A-36.wav,3,36,vacuum_cleaner,False,152020,A\n3-152020-B-36.wav,3,36,vacuum_cleaner,False,152020,B\n3-152020-C-36.wav,3,36,vacuum_cleaner,False,152020,C\n3-152039-A-3.wav,3,3,cow,False,152039,A\n3-152039-B-3.wav,3,3,cow,False,152039,B\n3-152594-A-30.wav,3,30,door_wood_knock,False,152594,A\n3-152912-A-26.wav,3,26,laughing,False,152912,A\n3-152997-A-26.wav,3,26,laughing,False,152997,A\n3-153057-A-43.wav,3,43,car_horn,False,153057,A\n3-153444-A-32.wav,3,32,keyboard_typing,False,153444,A\n3-154378-A-30.wav,3,30,door_wood_knock,False,154378,A\n3-154439-A-17.wav,3,17,pouring_water,False,154439,A\n3-154758-A-44.wav,3,44,engine,False,154758,A\n3-154781-A-32.wav,3,32,keyboard_typing,False,154781,A\n3-154926-A-40.wav,3,40,helicopter,True,154926,A\n3-154926-B-40.wav,3,40,helicopter,True,154926,B\n3-154957-A-1.wav,3,1,rooster,True,154957,A\n3-155130-A-43.wav,3,43,car_horn,False,155130,A\n3-155234-A-43.wav,3,43,car_horn,False,155234,A\n3-155312-A-0.wav,3,0,dog,True,155312,A\n3-155556-A-31.wav,3,31,mouse_click,False,155556,A\n3-155568-A-32.wav,3,32,keyboard_typing,False,155568,A\n3-155570-A-32.wav,3,32,keyboard_typing,False,155570,A\n3-155577-A-14.wav,3,14,chirping_birds,False,155577,A\n3-155578-A-14.wav,3,14,chirping_birds,False,155578,A\n3-155579-A-14.wav,3,14,chirping_birds,False,155579,A\n3-155583-A-14.wav,3,14,chirping_birds,False,155583,A\n3-155584-A-14.wav,3,14,chirping_birds,False,155584,A\n3-155642-A-11.wav,3,11,sea_waves,True,155642,A\n3-155642-B-11.wav,3,11,sea_waves,True,155642,B\n3-155659-A-34.wav,3,34,can_opening,False,155659,A\n3-155766-A-13.wav,3,13,crickets,False,155766,A\n3-156391-A-35.wav,3,35,washing_machine,False,156391,A\n3-156393-A-35.wav,3,35,washing_machine,False,156393,A\n3-156558-A-21.wav,3,21,sneezing,True,156558,A\n3-156581-A-14.wav,3,14,chirping_birds,False,156581,A\n3-156581-B-14.wav,3,14,chirping_birds,False,156581,B\n3-156907-A-15.wav,3,15,water_drops,False,156907,A\n3-157149-A-10.wav,3,10,rain,True,157149,A\n3-157187-A-12.wav,3,12,crackling_fire,True,157187,A\n3-157487-A-10.wav,3,10,rain,True,157487,A\n3-157492-A-45.wav,3,45,train,False,157492,A\n3-157615-A-10.wav,3,10,rain,True,157615,A\n3-157695-A-0.wav,3,0,dog,True,157695,A\n3-158056-A-31.wav,3,31,mouse_click,False,158056,A\n3-158056-B-31.wav,3,31,mouse_click,False,158056,B\n3-158476-A-12.wav,3,12,crackling_fire,True,158476,A\n3-159346-A-36.wav,3,36,vacuum_cleaner,False,159346,A\n3-159346-B-36.wav,3,36,vacuum_cleaner,False,159346,B\n3-159347-A-36.wav,3,36,vacuum_cleaner,False,159347,A\n3-159347-B-36.wav,3,36,vacuum_cleaner,False,159347,B\n3-159348-A-36.wav,3,36,vacuum_cleaner,False,159348,A\n3-159445-A-45.wav,3,45,train,False,159445,A\n3-159445-B-45.wav,3,45,train,False,159445,B\n3-160119-A-15.wav,3,15,water_drops,False,160119,A\n3-160993-A-3.wav,3,3,cow,False,160993,A\n3-161010-A-43.wav,3,43,car_horn,False,161010,A\n3-161500-A-17.wav,3,17,pouring_water,False,161500,A\n3-162786-A-13.wav,3,13,crickets,False,162786,A\n3-163288-A-1.wav,3,1,rooster,True,163288,A\n3-163459-A-0.wav,3,0,dog,True,163459,A\n3-163607-A-13.wav,3,13,crickets,False,163607,A\n3-163607-B-13.wav,3,13,crickets,False,163607,B\n3-163727-A-3.wav,3,3,cow,False,163727,A\n3-164120-A-11.wav,3,11,sea_waves,True,164120,A\n3-164216-A-6.wav,3,6,hen,False,164216,A\n3-164216-B-6.wav,3,6,hen,False,164216,B\n3-164216-C-6.wav,3,6,hen,False,164216,C\n3-164592-A-15.wav,3,15,water_drops,False,164592,A\n3-164593-A-15.wav,3,15,water_drops,False,164593,A\n3-164594-A-15.wav,3,15,water_drops,False,164594,A\n3-164595-A-15.wav,3,15,water_drops,False,164595,A\n3-164630-A-11.wav,3,11,sea_waves,True,164630,A\n3-164688-A-38.wav,3,38,clock_tick,True,164688,A\n3-165856-A-41.wav,3,41,chainsaw,True,165856,A\n3-166125-A-23.wav,3,23,breathing,False,166125,A\n3-166125-B-23.wav,3,23,breathing,False,166125,B\n3-166324-A-15.wav,3,15,water_drops,False,166324,A\n3-166326-A-15.wav,3,15,water_drops,False,166326,A\n3-166422-A-11.wav,3,11,sea_waves,True,166422,A\n3-166546-A-34.wav,3,34,can_opening,False,166546,A\n3-166546-B-34.wav,3,34,can_opening,False,166546,B\n3-167096-A-31.wav,3,31,mouse_click,False,167096,A\n3-169907-A-29.wav,3,29,drinking_sipping,False,169907,A\n3-170002-A-34.wav,3,34,can_opening,False,170002,A\n3-170015-A-0.wav,3,0,dog,True,170015,A\n3-170312-A-31.wav,3,31,mouse_click,False,170312,A\n3-170377-A-38.wav,3,38,clock_tick,True,170377,A\n3-170383-A-38.wav,3,38,clock_tick,True,170383,A\n3-170574-A-30.wav,3,30,door_wood_knock,False,170574,A\n3-170851-A-31.wav,3,31,mouse_click,False,170851,A\n3-171012-A-38.wav,3,38,clock_tick,True,171012,A\n3-171041-A-38.wav,3,38,clock_tick,True,171041,A\n3-171281-A-6.wav,3,6,hen,False,171281,A\n3-171937-A-34.wav,3,34,can_opening,False,171937,A\n3-172179-A-31.wav,3,31,mouse_click,False,172179,A\n3-172881-A-48.wav,3,48,fireworks,False,172881,A\n3-172922-A-48.wav,3,48,fireworks,False,172922,A\n3-174840-A-43.wav,3,43,car_horn,False,174840,A\n3-174866-A-29.wav,3,29,drinking_sipping,False,174866,A\n3-177082-A-22.wav,3,22,clapping,False,177082,A\n3-177083-A-22.wav,3,22,clapping,False,177083,A\n3-178096-A-6.wav,3,6,hen,False,178096,A\n3-180147-A-30.wav,3,30,door_wood_knock,False,180147,A\n3-180256-A-0.wav,3,0,dog,True,180256,A\n3-180977-A-0.wav,3,0,dog,True,180977,A\n3-181132-A-14.wav,3,14,chirping_birds,False,181132,A\n3-181278-A-22.wav,3,22,clapping,False,181278,A\n3-182023-A-30.wav,3,30,door_wood_knock,False,182023,A\n3-182025-A-30.wav,3,30,door_wood_knock,False,182025,A\n3-182710-A-35.wav,3,35,washing_machine,False,182710,A\n3-182710-B-35.wav,3,35,washing_machine,False,182710,B\n3-185313-A-31.wav,3,31,mouse_click,False,185313,A\n3-185456-A-29.wav,3,29,drinking_sipping,False,185456,A\n3-187549-A-6.wav,3,6,hen,False,187549,A\n3-187549-B-6.wav,3,6,hen,False,187549,B\n3-187710-A-11.wav,3,11,sea_waves,True,187710,A\n3-188390-A-6.wav,3,6,hen,False,188390,A\n3-188726-A-35.wav,3,35,washing_machine,False,188726,A\n3-193767-A-47.wav,3,47,airplane,False,193767,A\n3-197408-A-8.wav,3,8,sheep,False,197408,A\n3-197408-B-8.wav,3,8,sheep,False,197408,B\n3-197408-C-8.wav,3,8,sheep,False,197408,C\n3-197435-A-22.wav,3,22,clapping,False,197435,A\n3-197435-B-22.wav,3,22,clapping,False,197435,B\n3-203371-A-39.wav,3,39,glass_breaking,False,203371,A\n3-203373-A-39.wav,3,39,glass_breaking,False,203373,A\n3-203374-A-39.wav,3,39,glass_breaking,False,203374,A\n3-203375-A-39.wav,3,39,glass_breaking,False,203375,A\n3-203377-A-39.wav,3,39,glass_breaking,False,203377,A\n3-20861-A-8.wav,3,8,sheep,False,20861,A\n3-208820-A-49.wav,3,49,hand_saw,False,208820,A\n3-216280-A-39.wav,3,39,glass_breaking,False,216280,A\n3-216281-A-39.wav,3,39,glass_breaking,False,216281,A\n3-216284-A-39.wav,3,39,glass_breaking,False,216284,A\n3-233151-A-2.wav,3,2,pig,False,233151,A\n3-243726-A-43.wav,3,43,car_horn,False,243726,A\n3-246513-A-16.wav,3,16,wind,False,246513,A\n3-246513-B-16.wav,3,16,wind,False,246513,B\n3-249913-A-25.wav,3,25,footsteps,False,249913,A\n3-251617-A-48.wav,3,48,fireworks,False,251617,A\n3-253081-A-2.wav,3,2,pig,False,253081,A\n3-253084-A-2.wav,3,2,pig,False,253084,A\n3-253084-B-2.wav,3,2,pig,False,253084,B\n3-253084-C-2.wav,3,2,pig,False,253084,C\n3-253084-D-2.wav,3,2,pig,False,253084,D\n3-253084-E-2.wav,3,2,pig,False,253084,E\n3-257858-A-2.wav,3,2,pig,False,257858,A\n3-259622-A-44.wav,3,44,engine,False,259622,A\n3-51376-A-42.wav,3,42,siren,False,51376,A\n3-51731-A-42.wav,3,42,siren,False,51731,A\n3-51731-B-42.wav,3,42,siren,False,51731,B\n3-51909-A-42.wav,3,42,siren,False,51909,A\n3-51909-B-42.wav,3,42,siren,False,51909,B\n3-58772-A-42.wav,3,42,siren,False,58772,A\n3-62878-A-42.wav,3,42,siren,False,62878,A\n3-62878-B-42.wav,3,42,siren,False,62878,B\n3-65748-A-12.wav,3,12,crackling_fire,True,65748,A\n3-68630-A-40.wav,3,40,helicopter,True,68630,A\n3-68630-B-40.wav,3,40,helicopter,True,68630,B\n3-68630-C-40.wav,3,40,helicopter,True,68630,C\n3-70962-A-4.wav,3,4,frog,False,70962,A\n3-70962-B-4.wav,3,4,frog,False,70962,B\n3-70962-C-4.wav,3,4,frog,False,70962,C\n3-71964-A-4.wav,3,4,frog,False,71964,A\n3-71964-B-4.wav,3,4,frog,False,71964,B\n3-71964-C-4.wav,3,4,frog,False,71964,C\n3-83527-A-4.wav,3,4,frog,False,83527,A\n3-87936-A-46.wav,3,46,church_bells,False,87936,A\n3-87936-B-46.wav,3,46,church_bells,False,87936,B\n3-92637-A-18.wav,3,18,toilet_flush,False,92637,A\n3-93010-A-18.wav,3,18,toilet_flush,False,93010,A\n3-94342-A-25.wav,3,25,footsteps,False,94342,A\n3-94343-A-25.wav,3,25,footsteps,False,94343,A\n3-94344-A-25.wav,3,25,footsteps,False,94344,A\n3-94355-A-33.wav,3,33,door_wood_creaks,False,94355,A\n3-95694-A-5.wav,3,5,cat,False,95694,A\n3-95695-A-5.wav,3,5,cat,False,95695,A\n3-95695-B-5.wav,3,5,cat,False,95695,B\n3-95697-A-5.wav,3,5,cat,False,95697,A\n3-95698-A-5.wav,3,5,cat,False,95698,A\n3-96606-A-49.wav,3,49,hand_saw,False,96606,A\n3-96606-B-49.wav,3,49,hand_saw,False,96606,B\n3-97909-A-49.wav,3,49,hand_saw,False,97909,A\n3-97909-B-49.wav,3,49,hand_saw,False,97909,B\n3-98193-A-46.wav,3,46,church_bells,False,98193,A\n3-98771-A-18.wav,3,18,toilet_flush,False,98771,A\n3-98869-A-46.wav,3,46,church_bells,False,98869,A\n4-102844-A-49.wav,4,49,hand_saw,False,102844,A\n4-102844-B-49.wav,4,49,hand_saw,False,102844,B\n4-102844-C-49.wav,4,49,hand_saw,False,102844,C\n4-102871-A-42.wav,4,42,siren,False,102871,A\n4-107117-A-33.wav,4,33,door_wood_creaks,False,107117,A\n4-107120-A-33.wav,4,33,door_wood_creaks,False,107120,A\n4-107122-A-33.wav,4,33,door_wood_creaks,False,107122,A\n4-108352-A-33.wav,4,33,door_wood_creaks,False,108352,A\n4-111671-A-42.wav,4,42,siren,False,111671,A\n4-111671-B-42.wav,4,42,siren,False,111671,B\n4-117627-A-25.wav,4,25,footsteps,False,117627,A\n4-117630-A-25.wav,4,25,footsteps,False,117630,A\n4-119647-A-48.wav,4,48,fireworks,False,119647,A\n4-119647-B-48.wav,4,48,fireworks,False,119647,B\n4-119647-C-48.wav,4,48,fireworks,False,119647,C\n4-119647-D-48.wav,4,48,fireworks,False,119647,D\n4-119648-A-48.wav,4,48,fireworks,False,119648,A\n4-119648-B-48.wav,4,48,fireworks,False,119648,B\n4-119648-C-48.wav,4,48,fireworks,False,119648,C\n4-119648-D-48.wav,4,48,fireworks,False,119648,D\n4-119720-A-18.wav,4,18,toilet_flush,False,119720,A\n4-120160-A-5.wav,4,5,cat,False,120160,A\n4-121532-A-42.wav,4,42,siren,False,121532,A\n4-123680-A-18.wav,4,18,toilet_flush,False,123680,A\n4-125070-A-19.wav,4,19,thunderstorm,False,125070,A\n4-125071-A-19.wav,4,19,thunderstorm,False,125071,A\n4-125072-A-19.wav,4,19,thunderstorm,False,125072,A\n4-125072-B-19.wav,4,19,thunderstorm,False,125072,B\n4-125825-A-46.wav,4,46,church_bells,False,125825,A\n4-125825-B-46.wav,4,46,church_bells,False,125825,B\n4-125929-A-40.wav,4,40,helicopter,True,125929,A\n4-126046-A-18.wav,4,18,toilet_flush,False,126046,A\n4-126532-A-18.wav,4,18,toilet_flush,False,126532,A\n4-128659-A-33.wav,4,33,door_wood_creaks,False,128659,A\n4-130584-A-4.wav,4,4,frog,False,130584,A\n4-130891-A-7.wav,4,7,insects,False,130891,A\n4-132383-A-2.wav,4,2,pig,False,132383,A\n4-132383-B-2.wav,4,2,pig,False,132383,B\n4-132803-A-26.wav,4,26,laughing,False,132803,A\n4-132810-A-26.wav,4,26,laughing,False,132810,A\n4-132816-A-26.wav,4,26,laughing,False,132816,A\n4-132839-A-33.wav,4,33,door_wood_creaks,False,132839,A\n4-133047-A-5.wav,4,5,cat,False,133047,A\n4-133047-B-5.wav,4,5,cat,False,133047,B\n4-133047-C-5.wav,4,5,cat,False,133047,C\n4-133674-A-26.wav,4,26,laughing,False,133674,A\n4-133895-A-7.wav,4,7,insects,False,133895,A\n4-135439-A-18.wav,4,18,toilet_flush,False,135439,A\n4-136381-A-9.wav,4,9,crow,False,136381,A\n4-138344-A-9.wav,4,9,crow,False,138344,A\n4-140034-A-46.wav,4,46,church_bells,False,140034,A\n4-141365-A-18.wav,4,18,toilet_flush,False,141365,A\n4-143118-A-7.wav,4,7,insects,False,143118,A\n4-143118-B-7.wav,4,7,insects,False,143118,B\n4-144083-A-16.wav,4,16,wind,False,144083,A\n4-144083-B-16.wav,4,16,wind,False,144083,B\n4-144083-C-16.wav,4,16,wind,False,144083,C\n4-144085-A-16.wav,4,16,wind,False,144085,A\n4-144468-A-27.wav,4,27,brushing_teeth,False,144468,A\n4-144468-B-27.wav,4,27,brushing_teeth,False,144468,B\n4-145006-A-18.wav,4,18,toilet_flush,False,145006,A\n4-145081-A-9.wav,4,9,crow,False,145081,A\n4-146200-A-36.wav,4,36,vacuum_cleaner,False,146200,A\n4-147240-A-2.wav,4,2,pig,False,147240,A\n4-147240-B-2.wav,4,2,pig,False,147240,B\n4-147657-A-46.wav,4,46,church_bells,False,147657,A\n4-147658-A-46.wav,4,46,church_bells,False,147658,A\n4-149294-A-41.wav,4,41,chainsaw,True,149294,A\n4-149294-B-41.wav,4,41,chainsaw,True,149294,B\n4-149940-A-5.wav,4,5,cat,False,149940,A\n4-149940-B-5.wav,4,5,cat,False,149940,B\n4-150364-A-46.wav,4,46,church_bells,False,150364,A\n4-150364-B-46.wav,4,46,church_bells,False,150364,B\n4-151242-A-37.wav,4,37,clock_alarm,False,151242,A\n4-152958-A-18.wav,4,18,toilet_flush,False,152958,A\n4-152995-A-24.wav,4,24,coughing,False,152995,A\n4-154405-A-27.wav,4,27,brushing_teeth,False,154405,A\n4-154405-B-27.wav,4,27,brushing_teeth,False,154405,B\n4-154443-A-24.wav,4,24,coughing,False,154443,A\n4-154793-A-4.wav,4,4,frog,False,154793,A\n4-155650-A-24.wav,4,24,coughing,False,155650,A\n4-155650-B-24.wav,4,24,coughing,False,155650,B\n4-155670-A-26.wav,4,26,laughing,False,155670,A\n4-156227-A-9.wav,4,9,crow,False,156227,A\n4-156827-A-46.wav,4,46,church_bells,False,156827,A\n4-156843-A-21.wav,4,21,sneezing,True,156843,A\n4-156844-A-21.wav,4,21,sneezing,True,156844,A\n4-156993-A-19.wav,4,19,thunderstorm,False,156993,A\n4-157296-A-24.wav,4,24,coughing,False,157296,A\n4-157297-A-21.wav,4,21,sneezing,True,157297,A\n4-157611-A-41.wav,4,41,chainsaw,True,157611,A\n4-157611-B-41.wav,4,41,chainsaw,True,157611,B\n4-158653-A-32.wav,4,32,keyboard_typing,False,158653,A\n4-159426-A-9.wav,4,9,crow,False,159426,A\n4-159609-A-14.wav,4,14,chirping_birds,False,159609,A\n4-159609-B-14.wav,4,14,chirping_birds,False,159609,B\n4-160036-A-33.wav,4,33,door_wood_creaks,False,160036,A\n4-160036-B-33.wav,4,33,door_wood_creaks,False,160036,B\n4-160999-A-10.wav,4,10,rain,True,160999,A\n4-161099-A-47.wav,4,47,airplane,False,161099,A\n4-161099-B-47.wav,4,47,airplane,False,161099,B\n4-161100-A-47.wav,4,47,airplane,False,161100,A\n4-161103-A-47.wav,4,47,airplane,False,161103,A\n4-161105-A-47.wav,4,47,airplane,False,161105,A\n4-161105-B-47.wav,4,47,airplane,False,161105,B\n4-161127-A-10.wav,4,10,rain,True,161127,A\n4-161303-A-5.wav,4,5,cat,False,161303,A\n4-161303-B-5.wav,4,5,cat,False,161303,B\n4-161519-A-19.wav,4,19,thunderstorm,False,161519,A\n4-161579-A-40.wav,4,40,helicopter,True,161579,A\n4-161579-B-40.wav,4,40,helicopter,True,161579,B\n4-163264-A-10.wav,4,10,rain,True,163264,A\n4-163606-A-16.wav,4,16,wind,False,163606,A\n4-163608-B-16.wav,4,16,wind,False,163608,B\n4-163609-A-16.wav,4,16,wind,False,163609,A\n4-163609-B-16.wav,4,16,wind,False,163609,B\n4-163697-A-13.wav,4,13,crickets,False,163697,A\n4-164021-A-1.wav,4,1,rooster,True,164021,A\n4-164064-A-1.wav,4,1,rooster,True,164064,A\n4-164064-B-1.wav,4,1,rooster,True,164064,B\n4-164064-C-1.wav,4,1,rooster,True,164064,C\n4-164206-A-10.wav,4,10,rain,True,164206,A\n4-164243-A-26.wav,4,26,laughing,False,164243,A\n4-164661-A-12.wav,4,12,crackling_fire,True,164661,A\n4-164661-B-12.wav,4,12,crackling_fire,True,164661,B\n4-164859-A-1.wav,4,1,rooster,True,164859,A\n4-165606-A-45.wav,4,45,train,False,165606,A\n4-165791-A-45.wav,4,45,train,False,165791,A\n4-165791-B-45.wav,4,45,train,False,165791,B\n4-165813-A-19.wav,4,19,thunderstorm,False,165813,A\n4-165813-B-19.wav,4,19,thunderstorm,False,165813,B\n4-165823-A-41.wav,4,41,chainsaw,True,165823,A\n4-165823-B-41.wav,4,41,chainsaw,True,165823,B\n4-165845-A-45.wav,4,45,train,False,165845,A\n4-165845-B-45.wav,4,45,train,False,165845,B\n4-165933-A-45.wav,4,45,train,False,165933,A\n4-166661-A-10.wav,4,10,rain,True,166661,A\n4-167063-A-11.wav,4,11,sea_waves,True,167063,A\n4-167063-B-11.wav,4,11,sea_waves,True,167063,B\n4-167063-C-11.wav,4,11,sea_waves,True,167063,C\n4-167077-A-20.wav,4,20,crying_baby,True,167077,A\n4-167077-B-20.wav,4,20,crying_baby,True,167077,B\n4-167077-C-20.wav,4,20,crying_baby,True,167077,C\n4-167155-A-32.wav,4,32,keyboard_typing,False,167155,A\n4-167571-A-26.wav,4,26,laughing,False,167571,A\n4-167642-A-21.wav,4,21,sneezing,True,167642,A\n4-168155-A-15.wav,4,15,water_drops,False,168155,A\n4-168446-A-45.wav,4,45,train,False,168446,A\n4-168868-A-17.wav,4,17,pouring_water,False,168868,A\n4-169127-A-41.wav,4,41,chainsaw,True,169127,A\n4-169127-B-41.wav,4,41,chainsaw,True,169127,B\n4-169508-A-37.wav,4,37,clock_alarm,False,169508,A\n4-169726-A-24.wav,4,24,coughing,False,169726,A\n4-170078-A-1.wav,4,1,rooster,True,170078,A\n4-170247-A-12.wav,4,12,crackling_fire,True,170247,A\n4-170247-B-12.wav,4,12,crackling_fire,True,170247,B\n4-171207-A-12.wav,4,12,crackling_fire,True,171207,A\n4-171396-A-24.wav,4,24,coughing,False,171396,A\n4-171519-A-21.wav,4,21,sneezing,True,171519,A\n4-171652-A-44.wav,4,44,engine,False,171652,A\n4-171706-A-23.wav,4,23,breathing,False,171706,A\n4-171823-A-13.wav,4,13,crickets,False,171823,A\n4-172143-A-13.wav,4,13,crickets,False,172143,A\n4-172180-A-32.wav,4,32,keyboard_typing,False,172180,A\n4-172366-A-37.wav,4,37,clock_alarm,False,172366,A\n4-172377-A-17.wav,4,17,pouring_water,False,172377,A\n4-172500-A-27.wav,4,27,brushing_teeth,False,172500,A\n4-172500-B-27.wav,4,27,brushing_teeth,False,172500,B\n4-172500-C-27.wav,4,27,brushing_teeth,False,172500,C\n4-172500-D-27.wav,4,27,brushing_teeth,False,172500,D\n4-172732-A-36.wav,4,36,vacuum_cleaner,False,172732,A\n4-172733-A-36.wav,4,36,vacuum_cleaner,False,172733,A\n4-172734-A-36.wav,4,36,vacuum_cleaner,False,172734,A\n4-172736-A-36.wav,4,36,vacuum_cleaner,False,172736,A\n4-172736-B-36.wav,4,36,vacuum_cleaner,False,172736,B\n4-172742-A-32.wav,4,32,keyboard_typing,False,172742,A\n4-173865-A-9.wav,4,9,crow,False,173865,A\n4-173865-B-9.wav,4,9,crow,False,173865,B\n4-174797-A-15.wav,4,15,water_drops,False,174797,A\n4-174860-A-3.wav,4,3,cow,False,174860,A\n4-174860-B-3.wav,4,3,cow,False,174860,B\n4-175000-A-40.wav,4,40,helicopter,True,175000,A\n4-175000-B-40.wav,4,40,helicopter,True,175000,B\n4-175000-C-40.wav,4,40,helicopter,True,175000,C\n4-175025-A-34.wav,4,34,can_opening,False,175025,A\n4-175845-A-43.wav,4,43,car_horn,False,175845,A\n4-175846-A-43.wav,4,43,car_horn,False,175846,A\n4-175855-A-43.wav,4,43,car_horn,False,175855,A\n4-175856-A-43.wav,4,43,car_horn,False,175856,A\n4-175945-A-38.wav,4,38,clock_tick,True,175945,A\n4-176631-A-43.wav,4,43,car_horn,False,176631,A\n4-176638-A-43.wav,4,43,car_horn,False,176638,A\n4-176914-A-23.wav,4,23,breathing,False,176914,A\n4-177243-A-32.wav,4,32,keyboard_typing,False,177243,A\n4-177250-A-10.wav,4,10,rain,True,177250,A\n4-177835-A-24.wav,4,24,coughing,False,177835,A\n4-178402-A-43.wav,4,43,car_horn,False,178402,A\n4-178881-A-45.wav,4,45,train,False,178881,A\n4-179836-A-34.wav,4,34,can_opening,False,179836,A\n4-179984-A-38.wav,4,38,clock_tick,True,179984,A\n4-180337-A-28.wav,4,28,snoring,False,180337,A\n4-180380-A-10.wav,4,10,rain,True,180380,A\n4-180453-A-17.wav,4,17,pouring_water,False,180453,A\n4-181035-A-38.wav,4,38,clock_tick,True,181035,A\n4-181286-A-10.wav,4,10,rain,True,181286,A\n4-181362-A-13.wav,4,13,crickets,False,181362,A\n4-181563-A-12.wav,4,12,crackling_fire,True,181563,A\n4-181599-A-26.wav,4,26,laughing,False,181599,A\n4-181628-A-17.wav,4,17,pouring_water,False,181628,A\n4-181707-A-32.wav,4,32,keyboard_typing,False,181707,A\n4-181708-A-32.wav,4,32,keyboard_typing,False,181708,A\n4-181865-A-38.wav,4,38,clock_tick,True,181865,A\n4-181955-A-3.wav,4,3,cow,False,181955,A\n4-181955-B-3.wav,4,3,cow,False,181955,B\n4-181955-C-3.wav,4,3,cow,False,181955,C\n4-181999-A-36.wav,4,36,vacuum_cleaner,False,181999,A\n4-182034-A-30.wav,4,30,door_wood_knock,False,182034,A\n4-182039-A-30.wav,4,30,door_wood_knock,False,182039,A\n4-182041-A-30.wav,4,30,door_wood_knock,False,182041,A\n4-182314-A-7.wav,4,7,insects,False,182314,A\n4-182314-B-7.wav,4,7,insects,False,182314,B\n4-182368-A-12.wav,4,12,crackling_fire,True,182368,A\n4-182369-A-12.wav,4,12,crackling_fire,True,182369,A\n4-182395-A-0.wav,4,0,dog,True,182395,A\n4-182613-A-11.wav,4,11,sea_waves,True,182613,A\n4-182613-B-11.wav,4,11,sea_waves,True,182613,B\n4-182795-A-14.wav,4,14,chirping_birds,False,182795,A\n4-182839-A-17.wav,4,17,pouring_water,False,182839,A\n4-183487-A-1.wav,4,1,rooster,True,183487,A\n4-183882-A-28.wav,4,28,snoring,False,183882,A\n4-183882-B-28.wav,4,28,snoring,False,183882,B\n4-183992-A-0.wav,4,0,dog,True,183992,A\n4-184235-A-28.wav,4,28,snoring,False,184235,A\n4-184237-A-28.wav,4,28,snoring,False,184237,A\n4-184434-A-21.wav,4,21,sneezing,True,184434,A\n4-184575-A-0.wav,4,0,dog,True,184575,A\n4-185415-A-21.wav,4,21,sneezing,True,185415,A\n4-185575-A-20.wav,4,20,crying_baby,True,185575,A\n4-185575-B-20.wav,4,20,crying_baby,True,185575,B\n4-185575-C-20.wav,4,20,crying_baby,True,185575,C\n4-185613-A-32.wav,4,32,keyboard_typing,False,185613,A\n4-185619-A-21.wav,4,21,sneezing,True,185619,A\n4-186518-A-30.wav,4,30,door_wood_knock,False,186518,A\n4-186693-A-17.wav,4,17,pouring_water,False,186693,A\n4-186935-A-44.wav,4,44,engine,False,186935,A\n4-186936-A-44.wav,4,44,engine,False,186936,A\n4-186938-A-44.wav,4,44,engine,False,186938,A\n4-186940-A-44.wav,4,44,engine,False,186940,A\n4-186962-A-44.wav,4,44,engine,False,186962,A\n4-187284-A-30.wav,4,30,door_wood_knock,False,187284,A\n4-187384-A-34.wav,4,34,can_opening,False,187384,A\n4-187504-A-17.wav,4,17,pouring_water,False,187504,A\n4-187504-B-17.wav,4,17,pouring_water,False,187504,B\n4-187769-A-14.wav,4,14,chirping_birds,False,187769,A\n4-187769-B-14.wav,4,14,chirping_birds,False,187769,B\n4-188003-A-34.wav,4,34,can_opening,False,188003,A\n4-188033-A-38.wav,4,38,clock_tick,True,188033,A\n4-188191-A-29.wav,4,29,drinking_sipping,False,188191,A\n4-188191-B-29.wav,4,29,drinking_sipping,False,188191,B\n4-188191-C-29.wav,4,29,drinking_sipping,False,188191,C\n4-188287-A-9.wav,4,9,crow,False,188287,A\n4-188293-A-15.wav,4,15,water_drops,False,188293,A\n4-188293-B-15.wav,4,15,water_drops,False,188293,B\n4-188595-A-29.wav,4,29,drinking_sipping,False,188595,A\n4-188703-A-8.wav,4,8,sheep,False,188703,A\n4-188703-B-8.wav,4,8,sheep,False,188703,B\n4-188703-C-8.wav,4,8,sheep,False,188703,C\n4-188703-D-8.wav,4,8,sheep,False,188703,D\n4-188878-A-30.wav,4,30,door_wood_knock,False,188878,A\n4-189332-A-37.wav,4,37,clock_alarm,False,189332,A\n4-189828-A-22.wav,4,22,clapping,False,189828,A\n4-189830-A-22.wav,4,22,clapping,False,189830,A\n4-189832-A-22.wav,4,22,clapping,False,189832,A\n4-189832-B-22.wav,4,22,clapping,False,189832,B\n4-189833-A-22.wav,4,22,clapping,False,189833,A\n4-189836-A-22.wav,4,22,clapping,False,189836,A\n4-189838-A-22.wav,4,22,clapping,False,189838,A\n4-191015-A-43.wav,4,43,car_horn,False,191015,A\n4-191246-A-7.wav,4,7,insects,False,191246,A\n4-191246-B-7.wav,4,7,insects,False,191246,B\n4-191297-A-28.wav,4,28,snoring,False,191297,A\n4-191327-A-38.wav,4,38,clock_tick,True,191327,A\n4-191687-A-0.wav,4,0,dog,True,191687,A\n4-192068-A-31.wav,4,31,mouse_click,False,192068,A\n4-192236-A-0.wav,4,0,dog,True,192236,A\n4-193480-A-40.wav,4,40,helicopter,True,193480,A\n4-193480-B-40.wav,4,40,helicopter,True,193480,B\n4-194246-A-13.wav,4,13,crickets,False,194246,A\n4-194680-A-36.wav,4,36,vacuum_cleaner,False,194680,A\n4-194711-A-38.wav,4,38,clock_tick,True,194711,A\n4-194754-A-0.wav,4,0,dog,True,194754,A\n4-194808-A-29.wav,4,29,drinking_sipping,False,194808,A\n4-194979-A-25.wav,4,25,footsteps,False,194979,A\n4-194981-A-25.wav,4,25,footsteps,False,194981,A\n4-195305-A-31.wav,4,31,mouse_click,False,195305,A\n4-195451-A-44.wav,4,44,engine,False,195451,A\n4-195497-A-11.wav,4,11,sea_waves,True,195497,A\n4-195497-B-11.wav,4,11,sea_waves,True,195497,B\n4-195707-A-13.wav,4,13,crickets,False,195707,A\n4-195805-A-13.wav,4,13,crickets,False,195805,A\n4-196671-A-8.wav,4,8,sheep,False,196671,A\n4-196671-B-8.wav,4,8,sheep,False,196671,B\n4-196672-A-8.wav,4,8,sheep,False,196672,A\n4-197103-A-6.wav,4,6,hen,False,197103,A\n4-197454-A-28.wav,4,28,snoring,False,197454,A\n4-197454-B-28.wav,4,28,snoring,False,197454,B\n4-197871-A-15.wav,4,15,water_drops,False,197871,A\n4-198025-A-23.wav,4,23,breathing,False,198025,A\n4-198360-A-49.wav,4,49,hand_saw,False,198360,A\n4-198360-B-49.wav,4,49,hand_saw,False,198360,B\n4-198360-C-49.wav,4,49,hand_saw,False,198360,C\n4-198841-A-37.wav,4,37,clock_alarm,False,198841,A\n4-198962-A-25.wav,4,25,footsteps,False,198962,A\n4-198962-B-25.wav,4,25,footsteps,False,198962,B\n4-198965-A-38.wav,4,38,clock_tick,True,198965,A\n4-199261-A-0.wav,4,0,dog,True,199261,A\n4-200330-A-6.wav,4,6,hen,False,200330,A\n4-200330-B-6.wav,4,6,hen,False,200330,B\n4-201300-A-31.wav,4,31,mouse_click,False,201300,A\n4-201800-A-31.wav,4,31,mouse_click,False,201800,A\n4-201988-A-44.wav,4,44,engine,False,201988,A\n4-202298-A-31.wav,4,31,mouse_click,False,202298,A\n4-202749-A-13.wav,4,13,crickets,False,202749,A\n4-204115-A-39.wav,4,39,glass_breaking,False,204115,A\n4-204119-A-39.wav,4,39,glass_breaking,False,204119,A\n4-204121-A-39.wav,4,39,glass_breaking,False,204121,A\n4-204123-A-39.wav,4,39,glass_breaking,False,204123,A\n4-204202-A-29.wav,4,29,drinking_sipping,False,204202,A\n4-204612-A-31.wav,4,31,mouse_click,False,204612,A\n4-204618-A-11.wav,4,11,sea_waves,True,204618,A\n4-204683-A-31.wav,4,31,mouse_click,False,204683,A\n4-204684-A-37.wav,4,37,clock_alarm,False,204684,A\n4-204777-A-39.wav,4,39,glass_breaking,False,204777,A\n4-204777-B-39.wav,4,39,glass_breaking,False,204777,B\n4-204777-C-39.wav,4,39,glass_breaking,False,204777,C\n4-204830-A-6.wav,4,6,hen,False,204830,A\n4-205526-A-23.wav,4,23,breathing,False,205526,A\n4-205526-B-23.wav,4,23,breathing,False,205526,B\n4-205738-A-22.wav,4,22,clapping,False,205738,A\n4-207116-A-23.wav,4,23,breathing,False,207116,A\n4-207124-A-0.wav,4,0,dog,True,207124,A\n4-208021-A-1.wav,4,1,rooster,True,208021,A\n4-209536-A-37.wav,4,37,clock_alarm,False,209536,A\n4-209698-A-37.wav,4,37,clock_alarm,False,209698,A\n4-210000-A-23.wav,4,23,breathing,False,210000,A\n4-210000-B-23.wav,4,23,breathing,False,210000,B\n4-210309-A-31.wav,4,31,mouse_click,False,210309,A\n4-210593-A-29.wav,4,29,drinking_sipping,False,210593,A\n4-211502-A-30.wav,4,30,door_wood_knock,False,211502,A\n4-212604-A-15.wav,4,15,water_drops,False,212604,A\n4-212604-B-15.wav,4,15,water_drops,False,212604,B\n4-212604-C-15.wav,4,15,water_drops,False,212604,C\n4-212698-A-39.wav,4,39,glass_breaking,False,212698,A\n4-212728-A-34.wav,4,34,can_opening,False,212728,A\n4-213193-A-29.wav,4,29,drinking_sipping,False,213193,A\n4-213915-A-3.wav,4,3,cow,False,213915,A\n4-213915-B-3.wav,4,3,cow,False,213915,B\n4-213915-C-3.wav,4,3,cow,False,213915,C\n4-215635-A-34.wav,4,34,can_opening,False,215635,A\n4-216211-A-14.wav,4,14,chirping_birds,False,216211,A\n4-216349-A-34.wav,4,34,can_opening,False,216349,A\n4-218199-A-35.wav,4,35,washing_machine,False,218199,A\n4-218199-B-35.wav,4,35,washing_machine,False,218199,B\n4-218199-C-35.wav,4,35,washing_machine,False,218199,C\n4-218199-D-35.wav,4,35,washing_machine,False,218199,D\n4-218199-E-35.wav,4,35,washing_machine,False,218199,E\n4-218199-F-35.wav,4,35,washing_machine,False,218199,F\n4-218199-G-35.wav,4,35,washing_machine,False,218199,G\n4-218199-H-35.wav,4,35,washing_machine,False,218199,H\n4-218304-A-25.wav,4,25,footsteps,False,218304,A\n4-218304-B-25.wav,4,25,footsteps,False,218304,B\n4-223125-A-14.wav,4,14,chirping_birds,False,223125,A\n4-223127-A-14.wav,4,14,chirping_birds,False,223127,A\n4-232495-A-6.wav,4,6,hen,False,232495,A\n4-234644-A-2.wav,4,2,pig,False,234644,A\n4-234879-A-6.wav,4,6,hen,False,234879,A\n4-244318-A-6.wav,4,6,hen,False,244318,A\n4-250864-A-8.wav,4,8,sheep,False,250864,A\n4-250869-A-2.wav,4,2,pig,False,250869,A\n4-250869-B-2.wav,4,2,pig,False,250869,B\n4-250869-C-2.wav,4,2,pig,False,250869,C\n4-251645-A-49.wav,4,49,hand_saw,False,251645,A\n4-251645-B-49.wav,4,49,hand_saw,False,251645,B\n4-251959-A-47.wav,4,47,airplane,False,251959,A\n4-253649-A-6.wav,4,6,hen,False,253649,A\n4-255371-A-47.wav,4,47,airplane,False,255371,A\n4-261068-A-30.wav,4,30,door_wood_knock,False,261068,A\n4-264453-A-34.wav,4,34,can_opening,False,264453,A\n4-59579-A-20.wav,4,20,crying_baby,True,59579,A\n4-59579-B-20.wav,4,20,crying_baby,True,59579,B\n4-67358-A-42.wav,4,42,siren,False,67358,A\n4-80761-A-42.wav,4,42,siren,False,80761,A\n4-90014-A-42.wav,4,42,siren,False,90014,A\n4-90014-B-42.wav,4,42,siren,False,90014,B\n4-99193-A-4.wav,4,4,frog,False,99193,A\n4-99193-B-4.wav,4,4,frog,False,99193,B\n4-99644-A-4.wav,4,4,frog,False,99644,A\n4-99644-B-4.wav,4,4,frog,False,99644,B\n4-99644-C-4.wav,4,4,frog,False,99644,C\n4-99644-D-4.wav,4,4,frog,False,99644,D\n5-103415-A-2.wav,5,2,pig,False,103415,A\n5-103416-A-2.wav,5,2,pig,False,103416,A\n5-103418-A-2.wav,5,2,pig,False,103418,A\n5-103420-A-2.wav,5,2,pig,False,103420,A\n5-103421-A-2.wav,5,2,pig,False,103421,A\n5-103422-A-2.wav,5,2,pig,False,103422,A\n5-117118-A-42.wav,5,42,siren,False,117118,A\n5-117120-A-42.wav,5,42,siren,False,117120,A\n5-117122-A-42.wav,5,42,siren,False,117122,A\n5-117250-A-2.wav,5,2,pig,False,117250,A\n5-117773-A-16.wav,5,16,wind,False,117773,A\n5-127990-A-2.wav,5,2,pig,False,127990,A\n5-133989-A-42.wav,5,42,siren,False,133989,A\n5-133989-B-42.wav,5,42,siren,False,133989,B\n5-141683-A-35.wav,5,35,washing_machine,False,141683,A\n5-147297-A-27.wav,5,27,brushing_teeth,False,147297,A\n5-150409-A-42.wav,5,42,siren,False,150409,A\n5-151085-A-20.wav,5,20,crying_baby,True,151085,A\n5-156026-A-4.wav,5,4,frog,False,156026,A\n5-156026-B-4.wav,5,4,frog,False,156026,B\n5-156026-C-4.wav,5,4,frog,False,156026,C\n5-156026-D-4.wav,5,4,frog,False,156026,D\n5-156698-A-18.wav,5,18,toilet_flush,False,156698,A\n5-156999-A-19.wav,5,19,thunderstorm,False,156999,A\n5-156999-B-19.wav,5,19,thunderstorm,False,156999,B\n5-156999-C-19.wav,5,19,thunderstorm,False,156999,C\n5-156999-D-19.wav,5,19,thunderstorm,False,156999,D\n5-156999-E-19.wav,5,19,thunderstorm,False,156999,E\n5-157204-A-16.wav,5,16,wind,False,157204,A\n5-157204-B-16.wav,5,16,wind,False,157204,B\n5-160551-A-42.wav,5,42,siren,False,160551,A\n5-160614-A-48.wav,5,48,fireworks,False,160614,A\n5-160614-B-48.wav,5,48,fireworks,False,160614,B\n5-160614-C-48.wav,5,48,fireworks,False,160614,C\n5-160614-D-48.wav,5,48,fireworks,False,160614,D\n5-160614-E-48.wav,5,48,fireworks,False,160614,E\n5-160614-F-48.wav,5,48,fireworks,False,160614,F\n5-160614-G-48.wav,5,48,fireworks,False,160614,G\n5-160614-H-48.wav,5,48,fireworks,False,160614,H\n5-161270-A-33.wav,5,33,door_wood_creaks,False,161270,A\n5-161270-B-33.wav,5,33,door_wood_creaks,False,161270,B\n5-169983-A-5.wav,5,5,cat,False,169983,A\n5-170338-A-41.wav,5,41,chainsaw,True,170338,A\n5-170338-B-41.wav,5,41,chainsaw,True,170338,B\n5-171118-A-26.wav,5,26,laughing,False,171118,A\n5-171653-A-41.wav,5,41,chainsaw,True,171653,A\n5-172299-A-5.wav,5,5,cat,False,172299,A\n5-172639-A-5.wav,5,5,cat,False,172639,A\n5-173568-A-33.wav,5,33,door_wood_creaks,False,173568,A\n5-177034-A-18.wav,5,18,toilet_flush,False,177034,A\n5-177614-A-5.wav,5,5,cat,False,177614,A\n5-177779-A-33.wav,5,33,door_wood_creaks,False,177779,A\n5-177957-A-40.wav,5,40,helicopter,True,177957,A\n5-177957-B-40.wav,5,40,helicopter,True,177957,B\n5-177957-C-40.wav,5,40,helicopter,True,177957,C\n5-177957-D-40.wav,5,40,helicopter,True,177957,D\n5-177957-E-40.wav,5,40,helicopter,True,177957,E\n5-178997-A-24.wav,5,24,coughing,False,178997,A\n5-179294-A-46.wav,5,46,church_bells,False,179294,A\n5-179496-A-16.wav,5,16,wind,False,179496,A\n5-179496-B-16.wav,5,16,wind,False,179496,B\n5-179860-A-43.wav,5,43,car_horn,False,179860,A\n5-179863-A-43.wav,5,43,car_horn,False,179863,A\n5-179865-A-43.wav,5,43,car_horn,False,179865,A\n5-179866-A-43.wav,5,43,car_horn,False,179866,A\n5-179868-A-43.wav,5,43,car_horn,False,179868,A\n5-180156-A-43.wav,5,43,car_horn,False,180156,A\n5-180156-B-43.wav,5,43,car_horn,False,180156,B\n5-180156-C-43.wav,5,43,car_horn,False,180156,C\n5-180229-A-27.wav,5,27,brushing_teeth,False,180229,A\n5-181458-A-33.wav,5,33,door_wood_creaks,False,181458,A\n5-181766-A-10.wav,5,10,rain,True,181766,A\n5-181977-A-35.wav,5,35,washing_machine,False,181977,A\n5-182007-A-36.wav,5,36,vacuum_cleaner,False,182007,A\n5-182010-A-36.wav,5,36,vacuum_cleaner,False,182010,A\n5-182012-A-36.wav,5,36,vacuum_cleaner,False,182012,A\n5-182404-A-18.wav,5,18,toilet_flush,False,182404,A\n5-184323-A-42.wav,5,42,siren,False,184323,A\n5-184871-A-24.wav,5,24,coughing,False,184871,A\n5-185516-A-27.wav,5,27,brushing_teeth,False,185516,A\n5-185579-A-41.wav,5,41,chainsaw,True,185579,A\n5-185579-B-41.wav,5,41,chainsaw,True,185579,B\n5-185908-A-18.wav,5,18,toilet_flush,False,185908,A\n5-186924-A-12.wav,5,12,crackling_fire,True,186924,A\n5-187201-A-4.wav,5,4,frog,False,187201,A\n5-187201-B-4.wav,5,4,frog,False,187201,B\n5-187444-A-33.wav,5,33,door_wood_creaks,False,187444,A\n5-187979-A-21.wav,5,21,sneezing,True,187979,A\n5-188365-A-36.wav,5,36,vacuum_cleaner,False,188365,A\n5-188495-A-19.wav,5,19,thunderstorm,False,188495,A\n5-188606-A-33.wav,5,33,door_wood_creaks,False,188606,A\n5-188655-A-10.wav,5,10,rain,True,188655,A\n5-188716-A-46.wav,5,46,church_bells,False,188716,A\n5-188796-A-45.wav,5,45,train,False,188796,A\n5-188945-A-45.wav,5,45,train,False,188945,A\n5-189212-A-12.wav,5,12,crackling_fire,True,189212,A\n5-189237-A-12.wav,5,12,crackling_fire,True,189237,A\n5-189795-A-4.wav,5,4,frog,False,189795,A\n5-191131-A-40.wav,5,40,helicopter,True,191131,A\n5-191497-A-33.wav,5,33,door_wood_creaks,False,191497,A\n5-192191-A-19.wav,5,19,thunderstorm,False,192191,A\n5-193339-A-10.wav,5,10,rain,True,193339,A\n5-193473-A-12.wav,5,12,crackling_fire,True,193473,A\n5-193473-B-12.wav,5,12,crackling_fire,True,193473,B\n5-194533-A-21.wav,5,21,sneezing,True,194533,A\n5-194892-A-10.wav,5,10,rain,True,194892,A\n5-194899-A-3.wav,5,3,cow,False,194899,A\n5-194899-B-3.wav,5,3,cow,False,194899,B\n5-194899-C-3.wav,5,3,cow,False,194899,C\n5-194899-D-3.wav,5,3,cow,False,194899,D\n5-194930-A-1.wav,5,1,rooster,True,194930,A\n5-194930-B-1.wav,5,1,rooster,True,194930,B\n5-194932-A-7.wav,5,7,insects,False,194932,A\n5-195517-A-7.wav,5,7,insects,False,195517,A\n5-195518-A-7.wav,5,7,insects,False,195518,A\n5-195557-A-19.wav,5,19,thunderstorm,False,195557,A\n5-195710-A-10.wav,5,10,rain,True,195710,A\n5-197118-A-45.wav,5,45,train,False,197118,A\n5-197121-A-45.wav,5,45,train,False,197121,A\n5-197121-B-45.wav,5,45,train,False,197121,B\n5-197446-A-7.wav,5,7,insects,False,197446,A\n5-197913-A-18.wav,5,18,toilet_flush,False,197913,A\n5-197988-A-46.wav,5,46,church_bells,False,197988,A\n5-198278-A-7.wav,5,7,insects,False,198278,A\n5-198278-B-7.wav,5,7,insects,False,198278,B\n5-198278-C-7.wav,5,7,insects,False,198278,C\n5-198321-A-10.wav,5,10,rain,True,198321,A\n5-198373-A-46.wav,5,46,church_bells,False,198373,A\n5-198411-A-20.wav,5,20,crying_baby,True,198411,A\n5-198411-B-20.wav,5,20,crying_baby,True,198411,B\n5-198411-C-20.wav,5,20,crying_baby,True,198411,C\n5-198411-D-20.wav,5,20,crying_baby,True,198411,D\n5-198411-E-20.wav,5,20,crying_baby,True,198411,E\n5-198411-F-20.wav,5,20,crying_baby,True,198411,F\n5-198411-G-20.wav,5,20,crying_baby,True,198411,G\n5-198600-A-45.wav,5,45,train,False,198600,A\n5-198891-A-8.wav,5,8,sheep,False,198891,A\n5-198891-B-8.wav,5,8,sheep,False,198891,B\n5-198891-C-8.wav,5,8,sheep,False,198891,C\n5-198891-D-8.wav,5,8,sheep,False,198891,D\n5-199284-A-45.wav,5,45,train,False,199284,A\n5-199284-B-45.wav,5,45,train,False,199284,B\n5-200329-A-8.wav,5,8,sheep,False,200329,A\n5-200329-B-8.wav,5,8,sheep,False,200329,B\n5-200329-C-8.wav,5,8,sheep,False,200329,C\n5-200334-A-1.wav,5,1,rooster,True,200334,A\n5-200334-B-1.wav,5,1,rooster,True,200334,B\n5-200339-A-1.wav,5,1,rooster,True,200339,A\n5-200461-A-11.wav,5,11,sea_waves,True,200461,A\n5-200461-B-11.wav,5,11,sea_waves,True,200461,B\n5-201170-A-46.wav,5,46,church_bells,False,201170,A\n5-201172-A-46.wav,5,46,church_bells,False,201172,A\n5-201194-A-38.wav,5,38,clock_tick,True,201194,A\n5-201274-A-21.wav,5,21,sneezing,True,201274,A\n5-201664-A-18.wav,5,18,toilet_flush,False,201664,A\n5-202020-A-18.wav,5,18,toilet_flush,False,202020,A\n5-202220-A-21.wav,5,21,sneezing,True,202220,A\n5-202540-A-18.wav,5,18,toilet_flush,False,202540,A\n5-202795-A-3.wav,5,3,cow,False,202795,A\n5-202898-A-10.wav,5,10,rain,True,202898,A\n5-203128-A-0.wav,5,0,dog,True,203128,A\n5-203128-B-0.wav,5,0,dog,True,203128,B\n5-203739-A-10.wav,5,10,rain,True,203739,A\n5-204114-A-29.wav,5,29,drinking_sipping,False,204114,A\n5-204352-A-13.wav,5,13,crickets,False,204352,A\n5-204352-B-13.wav,5,13,crickets,False,204352,B\n5-204604-A-24.wav,5,24,coughing,False,204604,A\n5-204741-A-46.wav,5,46,church_bells,False,204741,A\n5-205090-A-32.wav,5,32,keyboard_typing,False,205090,A\n5-205589-A-17.wav,5,17,pouring_water,False,205589,A\n5-205589-B-17.wav,5,17,pouring_water,False,205589,B\n5-205898-A-40.wav,5,40,helicopter,True,205898,A\n5-207681-A-24.wav,5,24,coughing,False,207681,A\n5-207781-A-17.wav,5,17,pouring_water,False,207781,A\n5-207811-A-35.wav,5,35,washing_machine,False,207811,A\n5-207811-B-35.wav,5,35,washing_machine,False,207811,B\n5-207836-A-29.wav,5,29,drinking_sipping,False,207836,A\n5-207836-B-29.wav,5,29,drinking_sipping,False,207836,B\n5-207836-C-29.wav,5,29,drinking_sipping,False,207836,C\n5-207836-D-29.wav,5,29,drinking_sipping,False,207836,D\n5-208030-A-0.wav,5,0,dog,True,208030,A\n5-208624-A-38.wav,5,38,clock_tick,True,208624,A\n5-208761-A-24.wav,5,24,coughing,False,208761,A\n5-208810-A-11.wav,5,11,sea_waves,True,208810,A\n5-208810-B-11.wav,5,11,sea_waves,True,208810,B\n5-209698-A-38.wav,5,38,clock_tick,True,209698,A\n5-209719-A-24.wav,5,24,coughing,False,209719,A\n5-209833-A-38.wav,5,38,clock_tick,True,209833,A\n5-209989-A-22.wav,5,22,clapping,False,209989,A\n5-209992-A-44.wav,5,44,engine,False,209992,A\n5-209992-B-44.wav,5,44,engine,False,209992,B\n5-210540-A-13.wav,5,13,crickets,False,210540,A\n5-210571-A-38.wav,5,38,clock_tick,True,210571,A\n5-210612-A-37.wav,5,37,clock_alarm,False,210612,A\n5-211197-A-24.wav,5,24,coughing,False,211197,A\n5-211408-A-37.wav,5,37,clock_alarm,False,211408,A\n5-212050-A-27.wav,5,27,brushing_teeth,False,212050,A\n5-212054-A-36.wav,5,36,vacuum_cleaner,False,212054,A\n5-212059-A-36.wav,5,36,vacuum_cleaner,False,212059,A\n5-212181-A-38.wav,5,38,clock_tick,True,212181,A\n5-212454-A-0.wav,5,0,dog,True,212454,A\n5-212730-A-17.wav,5,17,pouring_water,False,212730,A\n5-212734-A-17.wav,5,17,pouring_water,False,212734,A\n5-212736-A-17.wav,5,17,pouring_water,False,212736,A\n5-213077-A-11.wav,5,11,sea_waves,True,213077,A\n5-213293-A-23.wav,5,23,breathing,False,213293,A\n5-213802-A-12.wav,5,12,crackling_fire,True,213802,A\n5-213836-A-9.wav,5,9,crow,False,213836,A\n5-213836-B-9.wav,5,9,crow,False,213836,B\n5-213836-C-9.wav,5,9,crow,False,213836,C\n5-213836-D-9.wav,5,9,crow,False,213836,D\n5-213855-A-0.wav,5,0,dog,True,213855,A\n5-214362-A-17.wav,5,17,pouring_water,False,214362,A\n5-214759-A-5.wav,5,5,cat,False,214759,A\n5-214759-B-5.wav,5,5,cat,False,214759,B\n5-214869-A-9.wav,5,9,crow,False,214869,A\n5-215005-A-32.wav,5,32,keyboard_typing,False,215005,A\n5-215172-A-13.wav,5,13,crickets,False,215172,A\n5-215179-A-13.wav,5,13,crickets,False,215179,A\n5-215445-A-47.wav,5,47,airplane,False,215445,A\n5-215447-A-47.wav,5,47,airplane,False,215447,A\n5-215449-A-47.wav,5,47,airplane,False,215449,A\n5-215634-A-17.wav,5,17,pouring_water,False,215634,A\n5-215658-A-12.wav,5,12,crackling_fire,True,215658,A\n5-215658-B-12.wav,5,12,crackling_fire,True,215658,B\n5-216131-A-32.wav,5,32,keyboard_typing,False,216131,A\n5-216213-A-13.wav,5,13,crickets,False,216213,A\n5-216214-A-13.wav,5,13,crickets,False,216214,A\n5-216216-A-13.wav,5,13,crickets,False,216216,A\n5-216368-A-28.wav,5,28,snoring,False,216368,A\n5-216370-A-41.wav,5,41,chainsaw,True,216370,A\n5-216370-B-41.wav,5,41,chainsaw,True,216370,B\n5-217158-A-0.wav,5,0,dog,True,217158,A\n5-217186-A-16.wav,5,16,wind,False,217186,A\n5-217186-B-16.wav,5,16,wind,False,217186,B\n5-217186-C-16.wav,5,16,wind,False,217186,C\n5-218196-A-27.wav,5,27,brushing_teeth,False,218196,A\n5-218196-B-27.wav,5,27,brushing_teeth,False,218196,B\n5-218494-A-22.wav,5,22,clapping,False,218494,A\n5-218980-A-30.wav,5,30,door_wood_knock,False,218980,A\n5-218981-A-30.wav,5,30,door_wood_knock,False,218981,A\n5-219044-A-46.wav,5,46,church_bells,False,219044,A\n5-219242-A-37.wav,5,37,clock_alarm,False,219242,A\n5-219242-B-37.wav,5,37,clock_alarm,False,219242,B\n5-219318-A-31.wav,5,31,mouse_click,False,219318,A\n5-219342-A-38.wav,5,38,clock_tick,True,219342,A\n5-219379-A-11.wav,5,11,sea_waves,True,219379,A\n5-219379-B-11.wav,5,11,sea_waves,True,219379,B\n5-219379-C-11.wav,5,11,sea_waves,True,219379,C\n5-220026-A-21.wav,5,21,sneezing,True,220026,A\n5-220027-A-21.wav,5,21,sneezing,True,220027,A\n5-220939-A-27.wav,5,27,brushing_teeth,False,220939,A\n5-220955-A-40.wav,5,40,helicopter,True,220955,A\n5-221518-A-21.wav,5,21,sneezing,True,221518,A\n5-221528-A-39.wav,5,39,glass_breaking,False,221528,A\n5-221529-A-39.wav,5,39,glass_breaking,False,221529,A\n5-221567-A-22.wav,5,22,clapping,False,221567,A\n5-221568-A-22.wav,5,22,clapping,False,221568,A\n5-221593-A-21.wav,5,21,sneezing,True,221593,A\n5-221878-A-34.wav,5,34,can_opening,False,221878,A\n5-221950-A-22.wav,5,22,clapping,False,221950,A\n5-222041-A-9.wav,5,9,crow,False,222041,A\n5-222524-A-41.wav,5,41,chainsaw,True,222524,A\n5-222894-A-32.wav,5,32,keyboard_typing,False,222894,A\n5-223099-A-32.wav,5,32,keyboard_typing,False,223099,A\n5-223099-B-32.wav,5,32,keyboard_typing,False,223099,B\n5-223103-A-31.wav,5,31,mouse_click,False,223103,A\n5-223176-A-37.wav,5,37,clock_alarm,False,223176,A\n5-223317-A-31.wav,5,31,mouse_click,False,223317,A\n5-223810-A-35.wav,5,35,washing_machine,False,223810,A\n5-231551-A-36.wav,5,36,vacuum_cleaner,False,231551,A\n5-231762-A-0.wav,5,0,dog,True,231762,A\n5-232272-A-44.wav,5,44,engine,False,232272,A\n5-232802-A-31.wav,5,31,mouse_click,False,232802,A\n5-232816-A-23.wav,5,23,breathing,False,232816,A\n5-233019-A-31.wav,5,31,mouse_click,False,233019,A\n5-233160-A-1.wav,5,1,rooster,True,233160,A\n5-233260-A-23.wav,5,23,breathing,False,233260,A\n5-233312-A-28.wav,5,28,snoring,False,233312,A\n5-233605-A-39.wav,5,39,glass_breaking,False,233605,A\n5-233607-A-39.wav,5,39,glass_breaking,False,233607,A\n5-233645-A-37.wav,5,37,clock_alarm,False,233645,A\n5-233787-A-7.wav,5,7,insects,False,233787,A\n5-234145-A-28.wav,5,28,snoring,False,234145,A\n5-234247-A-37.wav,5,37,clock_alarm,False,234247,A\n5-234263-A-25.wav,5,25,footsteps,False,234263,A\n5-234335-A-23.wav,5,23,breathing,False,234335,A\n5-234855-A-25.wav,5,25,footsteps,False,234855,A\n5-234879-A-1.wav,5,1,rooster,True,234879,A\n5-234879-B-1.wav,5,1,rooster,True,234879,B\n5-234923-A-32.wav,5,32,keyboard_typing,False,234923,A\n5-235507-A-44.wav,5,44,engine,False,235507,A\n5-235593-A-23.wav,5,23,breathing,False,235593,A\n5-235644-A-30.wav,5,30,door_wood_knock,False,235644,A\n5-235671-A-38.wav,5,38,clock_tick,True,235671,A\n5-235874-A-28.wav,5,28,snoring,False,235874,A\n5-235893-A-28.wav,5,28,snoring,False,235893,A\n5-235956-A-47.wav,5,47,airplane,False,235956,A\n5-236288-A-28.wav,5,28,snoring,False,236288,A\n5-236299-A-34.wav,5,34,can_opening,False,236299,A\n5-237315-A-31.wav,5,31,mouse_click,False,237315,A\n5-237499-A-4.wav,5,4,frog,False,237499,A\n5-237795-A-34.wav,5,34,can_opening,False,237795,A\n5-238021-A-35.wav,5,35,washing_machine,False,238021,A\n5-238492-A-23.wav,5,23,breathing,False,238492,A\n5-238926-A-31.wav,5,31,mouse_click,False,238926,A\n5-238938-A-27.wav,5,27,brushing_teeth,False,238938,A\n5-240671-A-44.wav,5,44,engine,False,240671,A\n5-241846-A-15.wav,5,15,water_drops,False,241846,A\n5-242490-A-14.wav,5,14,chirping_birds,False,242490,A\n5-242491-A-14.wav,5,14,chirping_birds,False,242491,A\n5-242492-A-3.wav,5,3,cow,False,242492,A\n5-242711-A-9.wav,5,9,crow,False,242711,A\n5-242932-A-26.wav,5,26,laughing,False,242932,A\n5-242932-B-26.wav,5,26,laughing,False,242932,B\n5-243025-A-25.wav,5,25,footsteps,False,243025,A\n5-243036-A-29.wav,5,29,drinking_sipping,False,243036,A\n5-243448-A-14.wav,5,14,chirping_birds,False,243448,A\n5-243449-A-14.wav,5,14,chirping_birds,False,243449,A\n5-243450-A-14.wav,5,14,chirping_birds,False,243450,A\n5-243459-A-14.wav,5,14,chirping_birds,False,243459,A\n5-243459-B-14.wav,5,14,chirping_birds,False,243459,B\n5-243635-A-29.wav,5,29,drinking_sipping,False,243635,A\n5-243773-A-44.wav,5,44,engine,False,243773,A\n5-243773-B-44.wav,5,44,engine,False,243773,B\n5-243783-A-44.wav,5,44,engine,False,243783,A\n5-244178-A-32.wav,5,32,keyboard_typing,False,244178,A\n5-244310-A-25.wav,5,25,footsteps,False,244310,A\n5-244315-A-6.wav,5,6,hen,False,244315,A\n5-244315-B-6.wav,5,6,hen,False,244315,B\n5-244315-C-6.wav,5,6,hen,False,244315,C\n5-244327-A-34.wav,5,34,can_opening,False,244327,A\n5-244459-A-28.wav,5,28,snoring,False,244459,A\n5-244526-A-26.wav,5,26,laughing,False,244526,A\n5-244651-A-31.wav,5,31,mouse_click,False,244651,A\n5-244933-A-34.wav,5,34,can_opening,False,244933,A\n5-245040-A-35.wav,5,35,washing_machine,False,245040,A\n5-248341-A-6.wav,5,6,hen,False,248341,A\n5-248341-B-6.wav,5,6,hen,False,248341,B\n5-248341-C-6.wav,5,6,hen,False,248341,C\n5-249748-A-28.wav,5,28,snoring,False,249748,A\n5-249937-A-22.wav,5,22,clapping,False,249937,A\n5-250026-A-30.wav,5,30,door_wood_knock,False,250026,A\n5-250026-B-30.wav,5,30,door_wood_knock,False,250026,B\n5-250258-A-49.wav,5,49,hand_saw,False,250258,A\n5-250629-A-37.wav,5,37,clock_alarm,False,250629,A\n5-250753-A-34.wav,5,34,can_opening,False,250753,A\n5-251426-A-30.wav,5,30,door_wood_knock,False,251426,A\n5-251426-B-30.wav,5,30,door_wood_knock,False,251426,B\n5-251489-A-24.wav,5,24,coughing,False,251489,A\n5-251957-A-47.wav,5,47,airplane,False,251957,A\n5-251962-A-47.wav,5,47,airplane,False,251962,A\n5-251963-A-47.wav,5,47,airplane,False,251963,A\n5-251971-A-47.wav,5,47,airplane,False,251971,A\n5-252248-A-34.wav,5,34,can_opening,False,252248,A\n5-253085-A-3.wav,5,3,cow,False,253085,A\n5-253085-B-3.wav,5,3,cow,False,253085,B\n5-253094-A-49.wav,5,49,hand_saw,False,253094,A\n5-253094-B-49.wav,5,49,hand_saw,False,253094,B\n5-253094-C-49.wav,5,49,hand_saw,False,253094,C\n5-253094-D-49.wav,5,49,hand_saw,False,253094,D\n5-253101-A-49.wav,5,49,hand_saw,False,253101,A\n5-253101-B-49.wav,5,49,hand_saw,False,253101,B\n5-253101-C-49.wav,5,49,hand_saw,False,253101,C\n5-253534-A-26.wav,5,26,laughing,False,253534,A\n5-254160-A-22.wav,5,22,clapping,False,254160,A\n5-254832-A-15.wav,5,15,water_drops,False,254832,A\n5-254832-B-15.wav,5,15,water_drops,False,254832,B\n5-256452-A-5.wav,5,5,cat,False,256452,A\n5-256512-A-30.wav,5,30,door_wood_knock,False,256512,A\n5-257349-A-15.wav,5,15,water_drops,False,257349,A\n5-257642-A-39.wav,5,39,glass_breaking,False,257642,A\n5-257839-A-14.wav,5,14,chirping_birds,False,257839,A\n5-259169-A-5.wav,5,5,cat,False,259169,A\n5-259180-A-15.wav,5,15,water_drops,False,259180,A\n5-259514-A-26.wav,5,26,laughing,False,259514,A\n5-259640-A-29.wav,5,29,drinking_sipping,False,259640,A\n5-260011-A-34.wav,5,34,can_opening,False,260011,A\n5-260164-A-23.wav,5,23,breathing,False,260164,A\n5-260432-A-39.wav,5,39,glass_breaking,False,260432,A\n5-260433-A-39.wav,5,39,glass_breaking,False,260433,A\n5-260434-A-39.wav,5,39,glass_breaking,False,260434,A\n5-260875-A-35.wav,5,35,washing_machine,False,260875,A\n5-261325-A-9.wav,5,9,crow,False,261325,A\n5-261433-A-15.wav,5,15,water_drops,False,261433,A\n5-261439-A-15.wav,5,15,water_drops,False,261439,A\n5-261464-A-23.wav,5,23,breathing,False,261464,A\n5-262641-A-15.wav,5,15,water_drops,False,262641,A\n5-262957-A-22.wav,5,22,clapping,False,262957,A\n5-263490-A-25.wav,5,25,footsteps,False,263490,A\n5-263491-A-25.wav,5,25,footsteps,False,263491,A\n5-263501-A-25.wav,5,25,footsteps,False,263501,A\n5-263775-A-26.wav,5,26,laughing,False,263775,A\n5-263775-B-26.wav,5,26,laughing,False,263775,B\n5-263831-A-6.wav,5,6,hen,False,263831,A\n5-263831-B-6.wav,5,6,hen,False,263831,B\n5-263902-A-36.wav,5,36,vacuum_cleaner,False,263902,A\n5-51149-A-25.wav,5,25,footsteps,False,51149,A\n5-61635-A-8.wav,5,8,sheep,False,61635,A\n5-9032-A-0.wav,5,0,dog,True,9032,A\n', 'weights/ast/dimer-base-manifest.json': '{\n  "format": "dimer_hf_snapshot",\n  "formatVersion": 1,\n  "modelKey": "ast-audioset",\n  "modelId": "MIT/ast-finetuned-audioset-10-10-0.4593",\n  "revision": "f826b80d28226b62986cc218e5cec390b1096902",\n  "files": [\n    {\n      "path": "README.md",\n      "bytes": 1165,\n      "sha256": "dbc8ce1fc5abd1635d073640d8cb032901bfe264a5e06507f363f10a6112cfc8"\n    },\n    {\n      "path": "config.json",\n      "bytes": 26763,\n      "sha256": "a93d525511d77e8ecc933d09674b85099815bbbb417c228a4edd655e252fb9ff"\n    },\n    {\n      "path": "model.safetensors",\n      "bytes": 346404948,\n      "sha256": "ae0c1e2ad4e1381d851fa9bf298ba13ebc9c5a914cdee2dbe427a6583869924d"\n    },\n    {\n      "path": "preprocessor_config.json",\n      "bytes": 297,\n      "sha256": "8d04ba5a9c6fca5d39d0de2b1fd05ecf79deb589fbba279728bbebac39934231"\n    }\n  ],\n  "totalBytes": 346433173\n}', 'licenses/ast-code.txt': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or Derivative\n          Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2026 Kurt Valcorza\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'licenses/weight-provenance.md': '# Weight provenance and DIMER hosting\n\n- Upstream: `MIT/ast-finetuned-audioset-10-10-0.4593`\n- Immutable revision: `f826b80d28226b62986cc218e5cec390b1096902`\n- Weight format: SafeTensors (`model.safetensors`, 346,404,948 bytes)\n- Upstream weight license: BSD-3-Clause (`license: bsd-3-clause` in the pinned upstream README)\n- Local layout: `weights/ast-audioset/` holds `README.md`, `config.json`, `model.safetensors`, `preprocessor_config.json` and `dimer-base-manifest.json`. The manifest lists every file with its byte size and SHA-256; `verify_snapshot()` in `src/ast_audio_classification_pipeline/pipeline.py` checks all four before any load. `.safetensors` files are git-ignored; the Git repository does not vendor the checkpoint.\n- DIMER hosting: BSD-3-Clause permits use, modification, redistribution and commercial use provided the copyright notice, the conditions list and the disclaimer are retained and the MIT name is not used to endorse derived products without permission. DIMER may mirror the pinned checkpoint in its model store under those terms.\n- Loader trust boundary: Transformers `ASTForAudioClassification` + `ASTFeatureExtractor` with `trust_remote_code=False`; the loader reads only the verified local directory (`local_files_only=True`) and falls back to the Hub at the pinned revision only when `allow_download=True` is passed explicitly.\n', 'source.json': '{\n  "repository": "kurtvalcorza/ast-audio-classification-pipeline",\n  "base_revision": "1988bd6cdda5fb3d1d8619f866f53f7ed9c56f27",\n  "generator": "build_sound_workshop.py/1",\n  "generator_sha256": "337c129c65acdd0305a279f1db3f1168c1f28f5496389d16bd55c40f0b51181d",\n  "files": {\n    "ast_reference/__init__.py": "1472ec08cfdaaabfb1dc14a14c526d89d52416a927836332300eb234a54ddae2",\n    "ast_reference/metrics.py": "07c293514f0f5078b9da8b5a26804c7c13d8b36dbb98f0cd1e3be98dd3147e6a",\n    "ast_reference/pipeline.py": "35099702eda941543b5f7d66db72cdc2f4663853e9e435df6eddde7141d2c54b",\n    "ast_reference/samples.py": "23157c35d2682af659be15cdf4f94cfba57b8918e9aafc25a105a266ae7557dd",\n    "workshop.py": "dc7c97d7e7fdb523e91ac98bdc5343535676164a0cc2ed20ef9e080776298781",\n    "requirements.txt": "0e98dd5f6fb7843b31e640fd9e776ae59d85cca189a30dde1eb3febfe76319d4",\n    "sample.json": "d743a570b16a3d1047abb4bf1f996b5dc956bed3bc4e824619731fe48d4d3024",\n    "licenses/esc-attribution.txt": "f6aec3a3283a2efdcbb1feff334adad7ab17c097bdec0cbb4eebdb9e16886074",\n    "licenses/esc-metadata.csv": "ca660da60191a97de289983a05821c9382d852a38a2ba8428980816b68cf6246",\n    "weights/ast/dimer-base-manifest.json": "13c8248aa95f5c6cee3eccb512248e96797d31c2a54d0cf477cbb0bbf49b91a9",\n    "licenses/ast-code.txt": "f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138",\n    "licenses/weight-provenance.md": "83b825a5c0ebebc95854c4e475335ba0c2579f28b90ac085558a0c1d7567c571"\n  }\n}'}
CARRIED_HASHES = {'ast_reference/__init__.py': '1472ec08cfdaaabfb1dc14a14c526d89d52416a927836332300eb234a54ddae2', 'ast_reference/metrics.py': '07c293514f0f5078b9da8b5a26804c7c13d8b36dbb98f0cd1e3be98dd3147e6a', 'ast_reference/pipeline.py': '35099702eda941543b5f7d66db72cdc2f4663853e9e435df6eddde7141d2c54b', 'ast_reference/samples.py': '23157c35d2682af659be15cdf4f94cfba57b8918e9aafc25a105a266ae7557dd', 'workshop.py': 'dc7c97d7e7fdb523e91ac98bdc5343535676164a0cc2ed20ef9e080776298781', 'requirements.txt': '0e98dd5f6fb7843b31e640fd9e776ae59d85cca189a30dde1eb3febfe76319d4', 'sample.json': 'd743a570b16a3d1047abb4bf1f996b5dc956bed3bc4e824619731fe48d4d3024', 'licenses/esc-attribution.txt': 'f6aec3a3283a2efdcbb1feff334adad7ab17c097bdec0cbb4eebdb9e16886074', 'licenses/esc-metadata.csv': 'ca660da60191a97de289983a05821c9382d852a38a2ba8428980816b68cf6246', 'weights/ast/dimer-base-manifest.json': '13c8248aa95f5c6cee3eccb512248e96797d31c2a54d0cf477cbb0bbf49b91a9', 'licenses/ast-code.txt': 'f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138', 'licenses/weight-provenance.md': '83b825a5c0ebebc95854c4e475335ba0c2579f28b90ac085558a0c1d7567c571', 'source.json': '0cea560d76ee65ba74a880e024ff90002a7f04d36f1d2719854351e48b77f6c0'}
for name, text in CARRIED_FILES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8', newline='\n')
    if hashlib.sha256(path.read_bytes()).hexdigest() != CARRIED_HASHES[name]:
        raise RuntimeError('Carried file integrity failure: ' + name)
print('Embedded reference code and immutable manifests verified.')

In [ ]:
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
for attempt in range(3):
    try:
        with urllib.request.urlopen(UV_URL, timeout=90) as response:
            wheel = response.read(20081405)
        break
    except (urllib.error.URLError, TimeoutError, ConnectionError):
        if attempt == 2:
            raise
        time.sleep(2 ** attempt)
if len(wheel) != 20081404 or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
    raise RuntimeError('uv wheel size/hash mismatch')
import io
with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
    member = next(n for n in archive.namelist() if n.endswith('.data/scripts/uv'))
    UV = ROOT / 'uv'
    UV.write_bytes(archive.read(member))
UV.chmod(0o700)
ENV = dict(os.environ, HF_HUB_DISABLE_IMPLICIT_TOKEN='1', HF_HUB_DISABLE_TELEMETRY='1', DO_NOT_TRACK='1')
ENV.pop('HF_TOKEN', None)
ENV.pop('HUGGING_FACE_HUB_TOKEN', None)
# Colab's kernel exports MPLBACKEND=module://matplotlib_inline.backend_inline; the isolated worker does not
# install matplotlib_inline, so importing matplotlib there would fail. The worker only writes figures to files.
ENV['MPLBACKEND'] = 'Agg'
subprocess.run([str(UV), 'venv', '--managed-python', '--python', '3.12.12', str(ROOT / 'env')], env=ENV, check=True)
PYTHON = ROOT / 'env/bin/python'
subprocess.run([str(UV), 'pip', 'install', '--python', str(PYTHON), '--require-hashes', '--only-binary', ':all:',
                '--index-url', 'https://pypi.org/simple', '-r', str(ROOT / 'requirements.txt')], env=ENV, check=True)
subprocess.run([str(PYTHON), '-c', 'import sys,torch; print(sys.version); print(torch.__version__); assert torch.cuda.is_available()'], env=ENV, check=True)
BOOTSTRAP_SECONDS = time.perf_counter() - SESSION_START

def run_stage(stage, *options, root=None):
    selected_root = ROOT if root is None else root
    log = selected_root / (stage + '.log')
    print('Running', stage, '— one model per process; log:', log, flush=True)
    with log.open('w', encoding='utf-8') as output:
        result = subprocess.Popen([str(PYTHON), '-u', str(ROOT / 'workshop.py'), '--root', str(selected_root), '--stage', stage, *options],
                                  env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in result.stdout:
            output.write(line)
            output.flush()
            if line.strip() and len(line) < 2000:
                print(line.rstrip(), flush=True)
        result.wait()
    print(log.read_text(encoding='utf-8')[-7000:])
    if result.returncode:
        raise RuntimeError(f'{stage} failed with exit {result.returncode}. Read the diagnostics above.')

from IPython.display import Audio, FileLink, Image, Markdown, display

def read_output(name):
    return json.loads((ROOT / 'outputs' / name).read_text())

def table(rows, columns):
    def cell(value):
        if value is None:
            return 'undefined'
        if isinstance(value, float):
            return f'{value:.4f}'
        return str(value).replace('|', '/').replace('\n', ' ')
    lines = ['| ' + ' | '.join(title for key, title in columns) + ' |',
             '| ' + ' | '.join('---' for _ in columns) + ' |']
    lines += ['| ' + ' | '.join(cell(row.get(key)) for key, title in columns) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))

def show_metrics():
    rows = []
    for model, roles in read_output('metrics.json').items():
        for role, values in roles.items():
            rows.append(dict(model=model, role=role, accuracy=values['accuracy'], macro_f1=values['macro_f1']))
    table(rows, [('model', 'System'), ('role', 'Role'), ('accuracy', 'Accuracy'), ('macro_f1', 'Macro-F1')])

def show_figures(pattern, limit=4):
    paths = sorted((ROOT / 'outputs').glob(pattern), key=lambda p: ({'clean': 0, '20': 1, '10': 2, '0': 3}.get(p.stem.rsplit('_', 1)[-1], 4), p.name))[:limit]
    if not paths:
        raise RuntimeError('Expected learning figures were not produced: ' + pattern)
    for path in paths:
        print(path.name)
        display(Image(filename=str(path)))

def play_examples(pattern, limit=2):
    paths = sorted((ROOT / 'outputs').glob(pattern), key=lambda p: ({'clean': 0, '20': 1, '10': 2, '0': 3}.get(p.stem.rsplit('_', 1)[-1], 4), p.name))[:limit]
    if not paths:
        raise RuntimeError('Expected listening examples were not produced: ' + pattern)
    print('Optional listening: start at low volume. Captions and figures convey the same comparisons.')
    for path in paths:
        print(path.stem)
        display(Audio(filename=str(path), normalize=False, autoplay=False))

## 2. Inspect real recordings and their roles

**Input:** all **400 real ESC-10 clips**, selected from pinned ESC metadata. **System:** verified WAV decoding, deterministic mono float32 resampling from 44.1 kHz to 16 kHz, duplicate and original-recording checks. **Output:** an immutable split and listening examples.

| Role | Official folds | Clips | Per class | Purpose |
|---|---|---:|---:|---|
| Train | 1–3 | 240 | 24 | Fit the head and training-only baseline |
| Validation | 4 | 80 | 8 | Choose the epoch; later descriptive noise activity |
| Test | 5 | 80 | 8 | Evaluate the locked choice |

An original recording can supply several clips. Related `src_file` groups must stay in one role; 400 clips are not 400 independent recording sessions. This fixed split is a teaching protocol, not the official five-fold benchmark average. The fixed label order is **chainsaw, clock_tick, crackling_fire, crying_baby, dog, helicopter, rain, rooster, sea_waves, sneezing**.

A **waveform** shows amplitude over time. **Sample rate** counts observations per second: 16 kHz means 16,000 samples/second. A **spectrogram** shows how frequency content changes over time. Resampling changes the sampling grid; it does not make a low-quality recording contain more information. We do not peak-normalize, trim quiet regions or stretch time silently. A **fixed gain of 0.5** is applied to every default and BYOD waveform before resampling, including 16 kHz input. This provides consistent headroom for sinc-resampling overshoot without per-clip normalization or clipping. Original amplitudes and this gain remain recorded. The later noise experiment applies its separately recorded common gain after this preprocessing.

**Predict:** which categories might share similar rhythms or frequency patterns? Why could two excerpts of the same original recording make a random split misleading?

In [ ]:
run_stage('prepare')
prepared = read_output('prepare.json')
print('Class vocabulary:', prepared['class_names'])
print('Split counts:', prepared.get('roles', prepared.get('counts')))
print('Original-recording groups:', prepared['groups'])
show_figures('examples/*.png', limit=2)
play_examples('examples/*.wav', limit=2)

**Notice:** real clip labels, seconds on the waveform and frequency/time axes on the spectrogram. Playback is optional, starts only when clicked and uses `normalize=False`; amplitude differences remain visible and audible. Begin at low volume. Plots and captions support participation without listening.

A log-mel spectrogram is a frequency summary, not the model's answer. AST receives a normalized filterbank tensor padded to **1,024 frames**. The model-input plot later marks the valid-time/padding boundary: padding is not newly observed silence.

## 3. Distinguish the original model from our new task

The pinned AST checkpoint was trained for **527 AudioSet labels**, using independent sigmoid scores: several events may be present together. This notebook attaches a new ten-class head to the frozen backbone and uses **softmax** for a single-label task. Its ten scores sum to one; this does not make them calibrated probabilities of correctness.

The randomly initialized head is an **untrained task head**, not pretrained ESC-10 classification. We never match AudioSet names to ESC-10 using fuzzy strings or compare sigmoid scores directly with softmax scores. A closed vocabulary forces a choice even for an unfamiliar sound.

Three directly comparable systems use the same ten labels: a training-majority predictor, the seeded initial head, and the validation-selected trained head. Majority ties use fixed class order; the balanced training set gives no informed reason to favor the winning class.

## 4. Learn the metric before evaluating the model

Accuracy is the fraction of correct clips. For each class, **precision** is `TP/(TP+FP)`, **recall** is `TP/(TP+FN)`, and **F1** is `2TP/(2TP+FP+FN)`. Macro-F1 averages class F1 over the fixed vocabulary, giving each category equal weight. Undefined precision/recall are null with a reason, not perfect scores.

**Predict:** in a hypothetical sample with 90 dogs and 10 sneezes, what happens if a system always predicts dog? Is 90% accuracy enough?

In [ ]:
accuracy = 90 / 100
recall_sneezing = 0 / 10
f1_dog = 2 * 90 / (2 * 90 + 10 + 0)
f1_sneezing = 0.0
print({'accuracy': accuracy, 'sneezing_recall': recall_sneezing,
       'macro_f1': (f1_dog + f1_sneezing) / 2})

<details><summary>Worked interpretation</summary>Accuracy is 0.90, sneezing recall is 0, and macro-F1 is about 0.474. High accuracy can conceal complete failure on a minority class. This is toy arithmetic, not an ESC-10 result; our default sample is balanced.</details>

## 5. Adapt the head and lock the checkpoint

**Input:** training clips and validation clips in disjoint original-recording groups. **Model:** frozen AST backbone; trainable classifier LayerNorm and linear layer. **Output:** initial and selected predictions, training history, measured updates and an exported head.

The frozen recipe is seed 42, ten epochs, batch four, AdamW learning rate `1e-3`, weight decay `0.01`, float32 cross-entropy, no augmentation or scheduler. Features are extracted in evaluation mode in batches of at most four. Detached feature caches bind inputs, processor, model, source and dtype. One optimizer continues across all ten epochs: the default expects **600 actual updates**.

Evaluate epoch zero, then each epoch. Select the highest **unrounded validation macro-F1**; earliest ties win, including epoch zero. Restore the chosen head before final test evaluation. Test labels never choose a checkpoint, learning rate or noise recipe.

**Predict:** must ten training epochs improve held-out accuracy? If epoch zero wins, how can we distinguish actual training from a skipped training stage?

In [ ]:
run_stage('train')
history = read_output('training_history.json')
table(history, [('epoch', 'Epoch'), ('train_loss', 'Training loss'), ('validation_macro_f1', 'Validation macro-F1')])
selection = read_output('selection.json')
print('Selected epoch:', selection['selected_epoch'])
show_metrics()
per_class = read_output('metrics.json')['selected']['test']['per_class']
table([dict(category=name, **values) for name, values in per_class.items()],
      [('category', 'Class'), ('precision', 'Precision'), ('recall', 'Recall'), ('f1', 'F1'), ('support', 'Test clips')])
print('Undefined precision means no predicted positives; see exported reasons.')
print('Actual updates:', selection['optimizer_steps'], 'Maximum head change during training:', selection['max_head_delta_during_training'])
show_figures('model_input.png', limit=1)

**Notice:** validation changes, per-class errors and initial-versus-selected test outcomes. The model-input figure separates measured sound from padding. Confusion rows are true labels and columns are predictions; normalized rows show where each true class goes. Raw counts retain support.

Training evidence records exact trainable names/counts, actual updates, finite losses, pre-selection head changes and unchanged backbone hashes. Epoch-zero selection can have zero final delta even when training genuinely updated parameters. Improvement is not required for an honest result.

<details><summary>Interpretation hint</summary>Optimization fits training data; validation selects; test evaluates the fixed decision. Correlated clips from a source are not independent observations. Report corrected errors and new errors together, and never tune after seeing final test performance.</details>

## 6. Reload the selected artifact in a fresh process

**Input:** selected classifier `.pt`, SHA-256 manifest, exact base identity and ordered vocabulary. **System:** a new process using `weights_only=True`, schema/finiteness checks and fresh waveform-to-feature inference. **Output:** prediction and metric parity checks.

This is the existing classifier-only PyTorch artifact format, **not safetensors**. Base weights are not included. Wrong label order, base, activation, tensor schema or digest must be refused. No unrestricted pickle fallback is allowed.

Fixed tolerances: logits `atol=1e-5, rtol=1e-4`; probabilities `atol=1e-6, rtol=1e-4`; identical top-one labels/confusion counts; accuracy and macro-F1 drift at most `1e-12`.

**Predict:** does successful reload establish accurate sound recognition, or reproducibility of the saved behavior?

In [ ]:
run_stage('reload')
parity = read_output('reload_parity.json')
print({key: value for key, value in parity.items() if isinstance(value, (bool, int, float))})
print('Per-probe deviations and identities are retained in reload_parity.json.')

**Notice:** raw-logit and probability differences as well as metric parity. A saved model can reproduce wrong answers perfectly. Preserve fixed tolerances and investigate a mismatch before qualification.

## 7. Change one thing: synthetic additive noise

**Input:** 20 validation clips, two per class selected by fixed filename order with distinct source groups where available. **Model:** the locked selected head. **Change:** clean / 20 / 10 / 0 dB signal-to-noise ratio (SNR).

A stable digest-derived seed creates one zero-mean Gaussian noise realization per clip, reused at all levels. With whole-clip signal RMS `r`, noise RMS is `r * 10**(-SNR/20)`. At 20 dB, noise RMS is one tenth of signal RMS; at 0 dB they are equal. This includes quiet regions and is not speech-active SNR.

One common gain per clip prevents clipping across all four conditions. The control is **gain-matched clean**, not original-amplitude clean. Noise level is the only intended changing factor: labels, duration, preprocessing, model and noise realization stay fixed. Each waveform gets newly computed features.

**Predict:** will macro-F1 necessarily decline at every noise level? Which confusion pairs might change? Write your prediction before running.

In [ ]:
run_stage('activity')
table(read_output('noise_summary.json'), [('condition', 'Condition'), ('accuracy', 'Accuracy'),
    ('macro_f1', 'Macro-F1'), ('changed_predictions', 'Changed predictions vs matched clean')])
show_figures('noise_examples/*.png', limit=4)
play_examples('noise_examples/*.wav', limit=4)

**Notice:** measured SNR, common gain, zero clipped samples and paired predictions against matched clean. Playback uses the actual scored signal and does not normalize each condition independently. The exported records preserve these controls.

<details><summary>Interpretation hint</summary>A noisy condition may improve some predictions. Keep that observation instead of choosing a different seed. This is one synthetic additive-noise experiment on reused validation clips; it supplies neither new held-out evidence nor proof of street, microphone or room-noise robustness. Selection stays locked.</details>

## 8. Export evidence and draw a bounded conclusion

The report contains full class logits/probabilities, per-class metrics, confusion counts, paired errors, deterministic examples, selected artifact, reload evidence, data attribution and measured time/memory. Tables are reloaded and archive checksums verified. Hashes and detailed traces live in files, keeping the learning flow readable.

A successful execution remains **Candidate** until the exact fresh hosted run and real labelled BYOD evidence are reviewed. It does not automatically approve a release.

In [ ]:
run_stage('report')
display(Markdown((ROOT / 'outputs/summary.md').read_text()))
show_figures('confusion.png', limit=1)
display(FileLink(str(ROOT / 'outputs/sound_results.zip')))
session = {'bootstrap_seconds': BOOTSTRAP_SECONDS, 'elapsed_seconds': time.perf_counter() - SESSION_START,
           'source_files': CARRIED_HASHES, 'qualification': 'Candidate pending exact hosted and real BYOD review'}
(ROOT / 'session_summary.json').write_text(json.dumps(session, indent=2))
print('Setup seconds:', round(BOOTSTRAP_SECONDS, 1), 'Total elapsed seconds:', round(session['elapsed_seconds'], 1))

Use your measured results in this optional personal note:

> On [clips and independent source groups], the selected head obtained [accuracy and macro-F1] versus [baseline]. It corrected [count] initial errors and introduced [count] new errors. At [noise level], [paired result] changed. An error worth inspecting is [actual example]. These results do not establish [broader claim].

**Self-check:** explain a waveform versus a spectrogram; sigmoid versus softmax; training versus checkpoint selection; clip count versus independent groups; actual updates versus selected state; matched-clean control versus original amplitude; and reload parity versus predictive quality.

## 9. Optional: your own labelled recordings

Use only recordings you may process in third-party Colab. Provide a directory with `manifest.json`, ordered `class_names` and records with `id`, relative WAV `audio`, class `label`, `group_id`, and explicit `train`/`validation`/`test` role. Use **2–10 classes**, each with at least **4/2/2 clips** and **two independent groups per role**. Related excerpts share a group and stay in one role.

Limits: at most 400 clips; WAV PCM16 or IEEE float32; 8–48 kHz; one or two channels; 0.5–10 seconds; 8 MiB/file; 1 MiB manifest; 4,000 seconds total. Stereo is downmixed by arithmetic mean. Longer-than-window audio is refused, not silently truncated. Headers, allocation bounds, finite normalized amplitudes, vocabulary, group isolation and decoded/resampled duplicates are checked before feature extraction. All-silent clips are refused.

This repeats **validation → baselines → training → selection → test → reload → noise → export** in a separate directory, preserving the canonical run. Private waveforms and spectrograms are excluded from shareable ZIPs by default; IDs, labels and metrics may remain. Review exports before sharing. Default Run all never opens an upload dialog.

In [ ]:
RUN_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
if RUN_BYOD:
    if not BYOD_PATH.strip():
        raise ValueError('Set BYOD_PATH to the labelled directory containing manifest.json.')
    run_stage('byod', '--byod', str(Path(BYOD_PATH).resolve()))
    byod_root = Path(read_output('byod_output.json')['root'])
    display(FileLink(str(byod_root / 'outputs/sound_results.zip')))
else:
    print('Optional labelled BYOD disabled.')

## 10. Optional: one recording without a label

The selected head can score one bounded WAV without a reference label. Its scores use the trained closed vocabulary. Unknown-sound detection is not provided; accuracy and F1 are **null / not measurable**. This inference option supplements the full labelled workflow and performs no new adaptation.

In [ ]:
RUN_UNLABELLED = False  # @param {type:"boolean"}
AUDIO_PATH = ''  # @param {type:"string"}
if RUN_UNLABELLED:
    if not AUDIO_PATH.strip():
        raise ValueError('Set AUDIO_PATH to a WAV within the documented limits.')
    run_stage('infer', '--audio', str(Path(AUDIO_PATH).resolve()))
    result = read_output('inference.json')
    table([{'class': name, 'probability': value} for name, value in zip(result['class_names'], result['probabilities'], strict=True)],
          [('class', 'Known class'), ('probability', 'Softmax score')])
    print('Predicted:', result['predicted'], 'Accuracy/F1:', result['accuracy'], result['macro_f1'])
    print(result['reason'])
else:
    print('Optional unlabelled inference disabled.')

## Troubleshooting, limitations and attribution

| Observation | Action |
|---|---|
| GPU missing or out of memory | Select a fresh T4 runtime. Do not silently skip feature extraction or training. |
| Download or package failure | Inspect the stage log; retry transient downloads while preserving pins. Never bypass integrity checks. |
| Group or duplicate refusal | Correct recording ownership/splits; do not silently discard records to retain target counts. |
| Epoch zero wins or test worsens | Preserve the result and investigate; do not tune on test labels. |
| Reload mismatch | Keep fixed tolerances and inspect raw outputs/schema before qualification. |
| Noise activity looks better | Report the counterexample; do not reroll the noise seed. |
| Own recordings refused | Fix WAV format, duration, amplitudes, grouping or class support before model execution. |

ESC-10 is a small, curated balanced dataset with correlated excerpts and known class-related preprocessing artifacts. Upstream pretraining overlap has not been ruled out. The fixed split does not reproduce the official five-fold benchmark. These results do not establish ecological monitoring, calibrated confidence, unknown-event detection or deployment robustness. Recordings can contain overlapping events despite one dataset label.

**Attribution:** ESC-10 (Karol J. Piczak, 2015), derived from Freesound recordings; pinned metadata and per-clip original-source attribution accompany the sample manifest. Upstream describes ESC-10 as CC BY 3.0, but its original per-clip attribution tags include **233 CC0, 165 CC BY and two CC Sampling+ recordings**. These original terms remain visible; their relationship to the subset statement requires maintainer attribution review before release. Do not assume all recordings have identical terms or extend the subset statement to all ESC-50. AST uses the pinned MIT AudioSet checkpoint with its recorded BSD-3-Clause weight license; carried source retains its own license. Base weights are downloaded and are not bundled in the classifier export.

**AI Assistance Disclosure:** This notebook's code and explanations were developed with generative AI assistance under maintainer direction. The maintainer remains responsible for reviewing implementation, validating results and making release decisions. AI assistance does not constitute independent verification, provider endorsement or release approval.

References: [AST model card](https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593), [ESC dataset and attribution](https://github.com/karolpiczak/ESC-50/tree/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6), [DIMER notebook specification](https://github.com/kurtvalcorza/ml-worker/blob/main/integrations/dimer/fleet-specs/NOTEBOOK_SPEC.md). Exact source and dependency identities accompany the exported evidence.